# 辨識人並裁切

### 加速之前

In [ ]:
# -*- coding: utf-8 -*-
from collections import defaultdict, deque
import cv2
import numpy as np
import torch
from ultralytics import YOLO
import os
from scipy.optimize import linear_sum_assignment
from PIL import Image
import datetime # 用於時間戳轉換

# --- 導入 Re-ID 和圖像轉換函式庫 ---
import torchreid
from torchvision import transforms

# --- 全域變數 for ROI ---
roi_points_manual = []
roi_defined_manual = False
roi_mask_manual = None
img_for_roi_selection_manual = None

# --- 參數設定 ---
YOLO_MODEL_PATH = 'yolov12x.pt' # 已修正回存在的模型
PERSON_CLASS_ID = 0
YOLO_CONFIDENCE_THRESHOLD = 0.3
NMS_IOU_THRESHOLD = 0.4
TARGET_INFERENCE_SIZE = 1280

# --- Re-ID 模型配置 ---
REID_MODEL_NAME = 'osnet_x1_0'
REID_MODEL_WEIGHTS = 'msmt17'

# --- 追蹤與匹配參數 ---
MAX_LOST_FRAMES = 1500
IOU_MATCHING_THRESHOLD = 0.3
FEATURE_MATCHING_THRESHOLD = 0.45
LAMBDA_WEIGHT = 0.95


MAX_TRACKS_IN_ROI = 20
# --- 全域變數 for ROI ---
roi_points_manual = []
# ... 其他全域變數 ...

# --- 【新功能】循環ID池 ---
available_ids = set()


# --- 裁切參數 ---
ENABLE_CROPPING = True
CROP_INTERVAL_SECONDS = 0.7
OUTPUT_CROP_DIR = "ttt"

# --- 班級快照參數 ---
ENABLE_CLASS_SNAPSHOT = True
OUTPUT_SNAPSHOT_DIR = "ttt123"

# --- 【新功能】班級快照儲存選項 ---
# True: 儲存帶有標註框的快照; False: 儲存乾淨的原始快照
SAVE_ANNOTATED_SNAPSHOT = False 

# --- 繪圖參數 ---
BOX_THICKNESS = 2
FONT_SCALE_LABEL = 0.5
FONT_THICKNESS_LABEL = 1
TRACK_LINE_THICKNESS = 3
FIXED_BOX_COLOR_BGR = (255, 191, 0)
ROI_POLYGON_COLOR = (0, 0, 255)

# --- 輔助函式 ---
def manual_roi_mouse_click(event, x, y, flags, param):
    global roi_points_manual, img_for_roi_selection_manual
    if event == cv2.EVENT_LBUTTONDOWN:
        if len(roi_points_manual) < 4:
            roi_points_manual.append((x, y))
            cv2.circle(img_for_roi_selection_manual, (x, y), 5, (0, 255, 0), -1)
            if len(roi_points_manual) > 1:
                cv2.line(img_for_roi_selection_manual, roi_points_manual[-2], roi_points_manual[-1], (0, 255, 0), 2)
            if len(roi_points_manual) == 4:
                cv2.line(img_for_roi_selection_manual, roi_points_manual[3], roi_points_manual[0], (0, 255, 0), 2)
            cv2.imshow("Select ROI - 4 points, then press 'c' or 's'", img_for_roi_selection_manual)

def calculate_iou(box1, box2):
    x1_1, y1_1, x2_1, y2_1 = box1
    x1_2, y1_2, x2_2, y2_2 = box2
    xi1, yi1, xi2, yi2 = max(x1_1, x1_2), max(y1_1, y1_2), min(x2_1, x2_2), min(y2_1, y2_2)
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    box1_area = (x2_1 - x1_1) * (y2_1 - y1_1)
    box2_area = (x2_2 - x1_2) * (y2_2 - y1_2)
    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0

def format_time_from_ms(milliseconds):
    if milliseconds < 0:
        return "00-00-00-000"
    td = datetime.timedelta(milliseconds=milliseconds)
    hours, remainder = divmod(td.seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    milliseconds_part = td.microseconds // 1000
    return f"{hours:02d}-{minutes:02d}-{seconds:02d}-{milliseconds_part:03d}"

def extract_features_from_rois(frame, bboxes, reid_model, reid_transform, device):
    features = []
    for bbox in bboxes:
        x1, y1, x2, y2 = bbox
        roi = frame[max(0, y1):min(frame.shape[0], y2), max(0, x1):min(frame.shape[1], x2)]
        if roi.size == 0:
            features.append(np.zeros(512))
            continue
        
        img_pil = Image.fromarray(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB))
        img_tensor = reid_transform(img_pil).unsqueeze(0).to(device)
        
        with torch.no_grad():
            feature = reid_model(img_tensor)
            feature /= feature.norm(dim=-1, keepdim=True)
        features.append(feature.cpu().numpy().flatten())
    return np.array(features)

# --- 主函數 ---
def main():
    global roi_points_manual, roi_defined_manual, roi_mask_manual, img_for_roi_selection_manual
    global available_ids # 【ID池修改】引入全域ID池

    # --- 【ID池修改】初始化ID池，包含1到MAX_TRACKS_IN_ROI的所有ID ---
    available_ids = set(range(1, MAX_TRACKS_IN_ROI + 1))

    FEATURE_BANK_SIZE = 50 

    try:
        # ... (後續的初始化程式碼保持不變，直到 while 迴圈之前) ...
        if torch.cuda.is_available():
            device = torch.device('cuda'); print(f"CUDA 可用。GPU: {torch.cuda.get_device_name(0)}")
        else:
            device = torch.device('cpu'); print("CUDA 不可用。將使用 CPU。")
    except Exception as e:
        device = torch.device('cpu'); print(f"CUDA 檢測出錯: {e}, 使用 CPU.")

    print(f"正在載入 YOLO 模型: {YOLO_MODEL_PATH}...")
    try:
        yolo_model = YOLO(YOLO_MODEL_PATH).to(device)
        print(f"YOLO 模型 '{YOLO_MODEL_PATH}' 已載入到: {device}")
    except Exception as e:
        print(f"載入 YOLO 模型 '{YOLO_MODEL_PATH}' 失敗: {e}"); exit()

    print(f"正在載入 Re-ID 模型: {REID_MODEL_NAME}...")
    try:
        reid_model = torchreid.models.build_model(name=REID_MODEL_NAME, num_classes=1, loss='triplet', pretrained=False)
        weights_path = f"{REID_MODEL_NAME}_{REID_MODEL_WEIGHTS}.pth"
        torchreid.utils.load_pretrained_weights(reid_model, weights_path)
        reid_model = reid_model.to(device)
        reid_model.eval()
        reid_transform = transforms.Compose([
            transforms.Resize((256, 128)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        print(f"Re-ID 模型已載入到 {device}")
    except Exception as e:
        print(f"載入 Re-ID 模型失敗: {e}"); exit()

    video_path = input("請輸入影片檔案路徑 (.mp4 或 .mov): ").strip().strip('"')
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): print(f"錯誤：無法開啟影片檔案 {video_path}"); return
    
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    crop_frame_interval = int(fps * CROP_INTERVAL_SECONDS)
    
    if ENABLE_CROPPING:
        os.makedirs(OUTPUT_CROP_DIR, exist_ok=True)
        print(f"個人裁切功能已啟用。")
    if ENABLE_CLASS_SNAPSHOT:
        os.makedirs(OUTPUT_SNAPSHOT_DIR, exist_ok=True)
        print(f"班級快照功能已啟用。")

    ret_roi, first_frame_for_roi = cap.read()
    if not ret_roi: cap.release(); return
    img_for_roi_selection_manual = first_frame_for_roi.copy()
    roi_select_window_name = "Select ROI - 4 points, then press 'c' or 's'"
    cv2.namedWindow(roi_select_window_name, cv2.WINDOW_NORMAL)
    cv2.imshow(roi_select_window_name, img_for_roi_selection_manual)
    cv2.setMouseCallback(roi_select_window_name, manual_roi_mouse_click)
    roi_selection_active = True
    while roi_selection_active:
        key_roi = cv2.waitKey(20) & 0xFF
        if cv2.getWindowProperty(roi_select_window_name, cv2.WND_PROP_VISIBLE) < 1 or key_roi in [ord('c'), ord('s'), ord('q')]:
            if len(roi_points_manual) == 4 and (key_roi == ord('c') or cv2.getWindowProperty(roi_select_window_name, cv2.WND_PROP_VISIBLE) < 1):
                roi_defined_manual = True; print("ROI 已定義。")
            elif key_roi == ord('s'): print("已跳過 ROI 定義。")
            if key_roi == ord('q'): cap.release(); cv2.destroyAllWindows(); return
            roi_selection_active = False
    if roi_defined_manual:
        roi_mask_manual = np.zeros(first_frame_for_roi.shape[:2], dtype=np.uint8)
        cv2.fillPoly(roi_mask_manual, [np.array(roi_points_manual, dtype=np.int32)], 255)
    cv2.destroyAllWindows()
    
    active_tracks = {}
    lost_tracks = {}
    # 【ID池修改】不再需要 next_stable_id
    # next_stable_id = 1
    track_history_for_drawing = defaultdict(lambda: deque(maxlen=30))
    last_crop_frame = defaultdict(int)
    last_snapshot_frame = 0

    frame_count = 0
    tracking_window_name = "Advanced Tracking with Snapshots"
    cv2.namedWindow(tracking_window_name, cv2.WINDOW_NORMAL)
    print("\n開始處理影片幀 (按 'q' 鍵退出)...")
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame_count += 1
        annotated_frame = frame.copy()
        
        input_frame_for_yolo = frame
        if roi_defined_manual and roi_mask_manual is not None:
            input_frame_for_yolo = cv2.bitwise_and(frame, frame, mask=roi_mask_manual)

        results = yolo_model.predict(
            source=input_frame_for_yolo, classes=[PERSON_CLASS_ID], 
            conf=YOLO_CONFIDENCE_THRESHOLD, iou=NMS_IOU_THRESHOLD,
            imgsz=TARGET_INFERENCE_SIZE, verbose=False, device=device
        )
        current_detections_bboxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        
        if len(current_detections_bboxes) > 0:
            current_detections_features = extract_features_from_rois(
                frame, current_detections_bboxes, reid_model, reid_transform, device
            )
        else:
            current_detections_features = np.array([])
        
        # --- (匹配邏輯，包括特徵庫的部分，都保持不變) ---
        active_ids_list = list(active_tracks.keys())
        active_bboxes = [track['bbox'] for track in active_tracks.values()]
        
        cost_matrix = np.ones((len(active_bboxes), len(current_detections_bboxes)))
        if len(active_bboxes) > 0 and len(current_detections_bboxes) > 0:
            for i, stable_id in enumerate(active_ids_list):
                for j in range(len(current_detections_bboxes)):
                    iou = calculate_iou(active_bboxes[i], current_detections_bboxes[j])
                    if iou < IOU_MATCHING_THRESHOLD: continue
                    track_feature_bank = active_tracks[stable_id]['feature_bank']
                    similarities = [np.dot(old_feat, current_detections_features[j]) for old_feat in track_feature_bank]
                    max_similarity = max(similarities) if similarities else 0.0
                    if max_similarity < FEATURE_MATCHING_THRESHOLD: continue
                    motion_cost, appearance_cost = 1 - iou, 1 - max_similarity
                    cost_matrix[i, j] = LAMBDA_WEIGHT * motion_cost + (1 - LAMBDA_WEIGHT) * appearance_cost

        row_ind, col_ind = linear_sum_assignment(cost_matrix)
        matched_active_indices, matched_detection_indices = set(), set()
        for r, c in zip(row_ind, col_ind):
            if cost_matrix[r, c] < 1:
                stable_id = active_ids_list[r]
                active_tracks[stable_id]['bbox'] = current_detections_bboxes[c]
                active_tracks[stable_id]['feature_bank'].append(current_detections_features[c])
                active_tracks[stable_id]['last_seen'] = frame_count
                matched_active_indices.add(r)
                matched_detection_indices.add(c)

        unmatched_detections_indices = [i for i, _ in enumerate(current_detections_bboxes) if i not in matched_detection_indices]
        lost_ids_list = list(lost_tracks.keys())
        if len(unmatched_detections_indices) > 0 and len(lost_ids_list) > 0:
            if not (roi_defined_manual and len(active_tracks) >= MAX_TRACKS_IN_ROI):
                cost_matrix_lost = np.ones((len(lost_ids_list), len(unmatched_detections_indices)))
                for i, stable_id in enumerate(lost_ids_list):
                    for j, u_idx in enumerate(unmatched_detections_indices):
                        track_feature_bank = lost_tracks[stable_id]['feature_bank']
                        similarities = [np.dot(old_feat, current_detections_features[u_idx]) for old_feat in track_feature_bank]
                        max_similarity = max(similarities) if similarities else 0.0
                        if max_similarity > FEATURE_MATCHING_THRESHOLD:
                            cost_matrix_lost[i, j] = 1 - max_similarity
                row_ind_lost, col_ind_lost = linear_sum_assignment(cost_matrix_lost)
                for r, c in zip(row_ind_lost, col_ind_lost):
                    if roi_defined_manual and len(active_tracks) >= MAX_TRACKS_IN_ROI: break
                    if cost_matrix_lost[r, c] < 1:
                        stable_id = lost_ids_list[r]
                        original_det_idx = unmatched_detections_indices[c]
                        resurrected_track = lost_tracks[stable_id]
                        resurrected_track['bbox'] = current_detections_bboxes[original_det_idx]
                        resurrected_track['feature_bank'].append(current_detections_features[original_det_idx])
                        resurrected_track['last_seen'] = frame_count
                        active_tracks[stable_id] = resurrected_track
                        del lost_tracks[stable_id]
                        matched_detection_indices.add(original_det_idx)
        
        # 第三級：建立新軌跡
        unmatched_new_indices = [i for i, _ in enumerate(current_detections_bboxes) if i not in matched_detection_indices]
        for i in unmatched_new_indices:
            # 【ID池修改】數量限制現在由ID池的可用性決定
            if roi_defined_manual and not available_ids:
                # 如果啟用了ROI，並且ID池已空，則不再分配新ID
                continue

            # 【ID池修改】從ID池中獲取一個新ID
            if available_ids:
                new_id = min(available_ids) # 取出最小的可用ID
                available_ids.remove(new_id) # 從池中移除

                new_feature_bank = deque(maxlen=FEATURE_BANK_SIZE)
                new_feature_bank.append(current_detections_features[i])
                active_tracks[new_id] = {
                    'bbox': current_detections_bboxes[i],
                    'feature_bank': new_feature_bank,
                    'last_seen': frame_count
                }
        
        # 更新軌跡狀態：將未匹配的活躍軌跡移至丟失列表
        ids_to_move_to_lost = [active_ids_list[i] for i, _ in enumerate(active_ids_list) if i not in matched_active_indices]
        for stable_id in ids_to_move_to_lost:
            if stable_id in active_tracks:
                lost_tracks[stable_id] = active_tracks[stable_id]
                lost_tracks[stable_id]['lost_for'] = 0 
                del active_tracks[stable_id]
        
        # 清理長期丟失的軌跡
        ids_to_remove_from_lost = []
        for sid, data in lost_tracks.items():
            data['lost_for'] += 1
            if data['lost_for'] > MAX_LOST_FRAMES:
                ids_to_remove_from_lost.append(sid)
        
        for sid in ids_to_remove_from_lost:
            # 【ID池修改】當一個ID被永久刪除時，將其歸還ID池
            if sid in lost_tracks:
                del lost_tracks[sid]
                available_ids.add(sid) # <-- 核心：回收ID
                print(f"ID {sid} 已回收。可用ID池: {sorted(list(available_ids))}")

            if sid in track_history_for_drawing: 
                del track_history_for_drawing[sid]

        # --- (繪圖和儲存部分保持不變) ---
        for stable_id, track_data in active_tracks.items():
            x1, y1, x2, y2 = track_data['bbox']
            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), FIXED_BOX_COLOR_BGR, BOX_THICKNESS)
            label_text = f"ID:{stable_id}"
            (text_w, text_h), baseline = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE_LABEL, FONT_THICKNESS_LABEL)
            text_y_pos = y1 - 7 if y1 - text_h - 7 > 0 else y1 + text_h + baseline + 7
            cv2.rectangle(annotated_frame, (x1, text_y_pos - text_h - baseline), (x1 + text_w, text_y_pos + baseline), FIXED_BOX_COLOR_BGR, -1)
            cv2.putText(annotated_frame, label_text, (x1, text_y_pos), cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE_LABEL, (0,0,0), FONT_THICKNESS_LABEL, cv2.LINE_AA)
            center_x, center_y = (x1 + x2) // 2, (y1 + y2) // 2
            track_history_for_drawing[stable_id].append((center_x, center_y))
            points = np.array(track_history_for_drawing[stable_id], dtype=np.int32).reshape((-1, 1, 2))
            if len(points) > 1:
                cv2.polylines(annotated_frame, [points], isClosed=False, color=(230, 230, 230), thickness=TRACK_LINE_THICKNESS)
            
            if ENABLE_CROPPING and (frame_count - last_crop_frame[stable_id]) >= crop_frame_interval:
                last_crop_frame[stable_id] = frame_count
                current_time_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
                time_str = format_time_from_ms(current_time_ms)
                id_folder_path = os.path.join(OUTPUT_CROP_DIR, f"ID_{stable_id}")
                os.makedirs(id_folder_path, exist_ok=True)
                cropped_image = frame[max(0, y1):min(frame.shape[0], y2), max(0, x1):min(frame.shape[1], x2)]
                if cropped_image.size > 0:
                    filename = f"{time_str}.jpg"
                    cv2.imwrite(os.path.join(id_folder_path, filename), cropped_image)

        if ENABLE_CLASS_SNAPSHOT and (frame_count - last_snapshot_frame) >= crop_frame_interval:
            last_snapshot_frame = frame_count
            current_time_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
            time_str = format_time_from_ms(current_time_ms)
            if SAVE_ANNOTATED_SNAPSHOT: image_to_save = annotated_frame
            else: image_to_save = frame
            snapshot_filename = f"snapshot_{time_str}.jpg"
            snapshot_path = os.path.join(OUTPUT_SNAPSHOT_DIR, snapshot_filename)
            cv2.imwrite(snapshot_path, image_to_save)

        if roi_defined_manual and roi_mask_manual is not None:
            cv2.polylines(annotated_frame, [np.array(roi_points_manual, dtype=np.int32)], True, ROI_POLYGON_COLOR, 2)
        
        cv2.imshow(tracking_window_name, annotated_frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("處理完成。")
if __name__ == "__main__":
    main()

### test 3 分層提取

In [ ]:
# -*- coding: utf-8 -*-
from collections import defaultdict, deque
import cv2
import numpy as np
import torch
from ultralytics import YOLO
import os
from scipy.optimize import linear_sum_assignment
from PIL import Image
import datetime # 用於時間戳轉換

# --- 導入 Re-ID 和圖像轉換函式庫 ---
import torchreid
from torchvision import transforms

# --- 全域變數 for ROI ---
roi_points_manual = []
roi_defined_manual = False
roi_mask_manual = None
img_for_roi_selection_manual = None

# --- 參數設定 ---
YOLO_MODEL_PATH = 'yolov12x.pt' # 已修正回存在的模型
PERSON_CLASS_ID = 0
YOLO_CONFIDENCE_THRESHOLD = 0.3
NMS_IOU_THRESHOLD = 0.4
TARGET_INFERENCE_SIZE = 1024

# --- Re-ID 模型配置 ---
REID_MODEL_NAME = 'osnet_x1_0'
REID_MODEL_WEIGHTS = 'msmt17'

# --- 追蹤與匹配參數 ---
MAX_LOST_FRAMES = 1500
IOU_MATCHING_THRESHOLD = 0.4
FEATURE_MATCHING_THRESHOLD = 0.5
LAMBDA_WEIGHT = 0.8
HIGH_CONF_IOU_THRESHOLD = 0.7 # 【新增】高信度 IoU 匹配的閾值
EMA_ALPHA = 0.95 # 【新增】EMA 更新的平滑因子，值越高越穩定
ABSOLUTE_FEATURE_THRESHOLD = 0.3 #新增】絕對外觀門檻，低於此值強制拒絕匹配


MAX_TRACKS_IN_ROI = 17
# --- 全域變數 for ROI ---
roi_points_manual = []
# ... 其他全域變數 ...

# --- 【新功能】循環ID池 ---
available_ids = set()


# --- 裁切參數 ---
ENABLE_CROPPING = True
CROP_INTERVAL_SECONDS = 0.7
OUTPUT_CROP_DIR = "0824_english_mid"

# --- 班級快照參數 ---
ENABLE_CLASS_SNAPSHOT = True
OUTPUT_SNAPSHOT_DIR = "0824_english_class"

# --- 【新功能】班級快照儲存選項 ---
# True: 儲存帶有標註框的快照; False: 儲存乾淨的原始快照
SAVE_ANNOTATED_SNAPSHOT = False 

# --- 繪圖參數 ---
BOX_THICKNESS = 2
FONT_SCALE_LABEL = 0.5
FONT_THICKNESS_LABEL = 1
TRACK_LINE_THICKNESS = 3
FIXED_BOX_COLOR_BGR = (255, 191, 0)
ROI_POLYGON_COLOR = (0, 0, 255)

# --- 輔助函式 ---
def manual_roi_mouse_click(event, x, y, flags, param):
    global roi_points_manual, img_for_roi_selection_manual
    if event == cv2.EVENT_LBUTTONDOWN:
        if len(roi_points_manual) < 4:
            roi_points_manual.append((x, y))
            cv2.circle(img_for_roi_selection_manual, (x, y), 5, (0, 255, 0), -1)
            if len(roi_points_manual) > 1:
                cv2.line(img_for_roi_selection_manual, roi_points_manual[-2], roi_points_manual[-1], (0, 255, 0), 2)
            if len(roi_points_manual) == 4:
                cv2.line(img_for_roi_selection_manual, roi_points_manual[3], roi_points_manual[0], (0, 255, 0), 2)
            cv2.imshow("Select ROI - 4 points, then press 'c' or 's'", img_for_roi_selection_manual)

def calculate_iou(box1, box2):
    x1_1, y1_1, x2_1, y2_1 = box1
    x1_2, y1_2, x2_2, y2_2 = box2
    xi1, yi1, xi2, yi2 = max(x1_1, x1_2), max(y1_1, y1_2), min(x2_1, x2_2), min(y2_1, y2_2)
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    box1_area = (x2_1 - x1_1) * (y2_1 - y1_1)
    box2_area = (x2_2 - x1_2) * (y2_2 - y1_2)
    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0

def format_time_from_ms(milliseconds):
    if milliseconds < 0:
        return "00-00-00-000"
    td = datetime.timedelta(milliseconds=milliseconds)
    hours, remainder = divmod(td.seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    milliseconds_part = td.microseconds // 1000
    return f"{hours:02d}-{minutes:02d}-{seconds:02d}-{milliseconds_part:03d}"

def extract_features_batch(frame: np.ndarray, 
                           bboxes: np.ndarray, 
                           reid_model: torch.nn.Module, 
                           reid_transform: transforms.Compose, 
                           device: torch.device) -> np.ndarray:
    """
    從多個邊界框中以批次方式提取 Re-ID 特徵。

    Args:
        frame (np.ndarray): 原始影像幀 (BGR格式)。
        bboxes (np.ndarray): 邊界框的 NumPy 陣列，形狀為 (N, 4)，格式為 [x1, y1, x2, y2]。
        reid_model (torch.nn.Module): 已載入並設為 eval 模式的 Re-ID 模型。
        reid_transform (transforms.Compose): 用於預處理圖像的轉換流程。
        device (torch.device): 運行模型的設備 (例如 'cuda' 或 'cpu')。

    Returns:
        np.ndarray: 提取出的特徵陣列，形狀為 (N, feature_dim)。如果輸入的 bboxes 為空，則返回一個空的陣列。
    """
    # 如果沒有傳入任何邊界框，直接返回一個空的 NumPy 陣列
    if bboxes.shape[0] == 0:
        return np.array([])

    # 準備一個列表來存放經過預處理的 ROI Tensors
    rois_tensors = []
    
    # 將 BGR 幀一次性轉換為 RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    for bbox in bboxes:
        x1, y1, x2, y2 = bbox
        
        # 進行邊界檢查，確保裁切區域不超出圖像範圍
        roi_rgb = frame_rgb[max(0, y1):min(frame.shape[0], y2), max(0, x1):min(frame.shape[1], x2)]

        # 如果裁切後的 ROI 大小為 0 (例如，一個無效的邊界框)，則跳過
        if roi_rgb.size == 0:
            # 為了保持特徵與 bboxes 的一一對應，我們可以填充一個零向量
            # Re-ID 模型的輸出維度通常是固定的，例如 osnet_x1_0 是 512
            # 這裡我們硬編碼 512，更好的做法是從模型配置中獲取
            feature_dim = 512 # 根據你的 Re-ID 模型調整
            rois_tensors.append(torch.zeros(3, 256, 128)) # 填充一個符合轉換後尺寸的零 Tensor
            print(f"警告：偵測到一個無效的空邊界框 {bbox}，已用零向量填充。")
            continue
            
        # 將 NumPy 陣列 (ROI) 轉換為 PIL Image，以符合 reid_transform 的輸入格式
        img_pil = Image.fromarray(roi_rgb)
        
        # 應用轉換流程 (Resize, ToTensor, Normalize)
        img_tensor = reid_transform(img_pil)
        
        rois_tensors.append(img_tensor)
    
    # 如果經過濾後沒有任何有效的 ROI，也返回空陣列
    if not rois_tensors:
        return np.array([])
        
    # 將 Tensor 列表堆疊成一個批次 (batch)
    # torch.stack 會增加一個新的維度在最前面，從 [T, C, H, W] 變成 [B, C, H, W]
    # 其中 B 是批次大小 (即有效的 ROI 數量), T 是 Tensor 數量
    batch_tensor = torch.stack(rois_tensors).to(device)
    
    # 使用 torch.no_grad() 進行推論，以節省記憶體並加速計算
    with torch.no_grad():
        # 將整個批次送入 Re-ID 模型
        features_batch = reid_model(batch_tensor)
        
        # 進行 L2 標準化，這在度量學習中是標準步驟
        features_batch /= features_batch.norm(dim=-1, keepdim=True)
    
    # 將結果從 GPU 移回 CPU，並轉換為 NumPy 陣列以便後續計算 (如計算 cost_matrix)
    return features_batch.cpu().numpy()

# --- 主函數 ---
def main():
    global roi_points_manual, roi_defined_manual, roi_mask_manual, img_for_roi_selection_manual
    global available_ids

    # --- ID池初始化 ---
    available_ids = set(range(1, MAX_TRACKS_IN_ROI + 1))

    try:
        if torch.cuda.is_available():
            device = torch.device('cuda'); print(f"CUDA 可用。GPU: {torch.cuda.get_device_name(0)}")
        else:
            device = torch.device('cpu'); print("CUDA 不可用。將使用 CPU。")
    except Exception as e:
        device = torch.device('cpu'); print(f"CUDA 檢測出錯: {e}, 使用 CPU.")

    # --- YOLO 模型載入 (帶錯誤處理) ---
    print(f"正在載入 YOLO 模型: {YOLO_MODEL_PATH}...")
    try:
        yolo_model = None 
        if not os.path.exists(YOLO_MODEL_PATH):
            raise FileNotFoundError(f"YOLO 模型檔案不存在於指定路徑: {YOLO_MODEL_PATH}")
        yolo_model = YOLO(YOLO_MODEL_PATH)
        yolo_model.to(device)
        print(f"YOLO 模型 '{YOLO_MODEL_PATH}' 已成功載入到: {device}")
    except Exception as e:
        print(f"--- 嚴重錯誤：載入 YOLO 模型 '{YOLO_MODEL_PATH}' 失敗 ---")
        import traceback
        traceback.print_exc()
        print("程式即將終止。"); exit()

    # --- Re-ID 模型載入 (帶錯誤處理) ---
    print(f"正在載入 Re-ID 模型: {REID_MODEL_NAME}...")
    try:
        reid_model = None
        reid_transform = None
        reid_model = torchreid.models.build_model(
            name=REID_MODEL_NAME, num_classes=1000, pretrained=True
        )
        reid_model = reid_model.to(device)
        reid_model.eval()
        reid_transform = transforms.Compose([
            transforms.Resize((256, 128)), transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        print(f"Re-ID 模型與 Transform 已成功載入到 {device}")
    except Exception as e:
        print("\n" + "="*50)
        print("--- 嚴重錯誤：Re-ID 模型初始化失敗 ---")
        import traceback
        traceback.print_exc()
        print("程式即將終止。"); exit()

    # --- 影片讀取與 ROI 設定 ---
    video_path = input("請輸入影片檔案路徑 (.mp4 或 .mov): ").strip().strip('"')
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): print(f"錯誤：無法開啟影片檔案 {video_path}"); return
    
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    crop_frame_interval = int(fps * CROP_INTERVAL_SECONDS)
    
    if ENABLE_CROPPING: os.makedirs(OUTPUT_CROP_DIR, exist_ok=True); print(f"個人裁切功能已啟用。")
    if ENABLE_CLASS_SNAPSHOT: os.makedirs(OUTPUT_SNAPSHOT_DIR, exist_ok=True); print(f"班級快照功能已啟用。")

    ret_roi, first_frame_for_roi = cap.read()
    if not ret_roi: cap.release(); return
    img_for_roi_selection_manual = first_frame_for_roi.copy()
    roi_select_window_name = "Select ROI - 4 points, then press 'c' or 's'"
    cv2.namedWindow(roi_select_window_name, cv2.WINDOW_NORMAL)
    cv2.imshow(roi_select_window_name, img_for_roi_selection_manual)
    cv2.setMouseCallback(roi_select_window_name, manual_roi_mouse_click)
    roi_selection_active = True
    while roi_selection_active:
        key_roi = cv2.waitKey(20) & 0xFF
        if cv2.getWindowProperty(roi_select_window_name, cv2.WND_PROP_VISIBLE) < 1 or key_roi in [ord('c'), ord('s'), ord('q')]:
            if len(roi_points_manual) == 4 and (key_roi == ord('c') or cv2.getWindowProperty(roi_select_window_name, cv2.WND_PROP_VISIBLE) < 1):
                roi_defined_manual = True; print("ROI 已定義。")
            elif key_roi == ord('s'): print("已跳過 ROI 定義。")
            if key_roi == ord('q'): cap.release(); cv2.destroyAllWindows(); return
            roi_selection_active = False
    if roi_defined_manual:
        roi_mask_manual = np.zeros(first_frame_for_roi.shape[:2], dtype=np.uint8)
        cv2.fillPoly(roi_mask_manual, [np.array(roi_points_manual, dtype=np.int32)], 255)
    cv2.destroyAllWindows()
    
    # --- 初始化追蹤相關變數 ---
    active_tracks = {}
    lost_tracks = {}
    track_history_for_drawing = defaultdict(lambda: deque(maxlen=30))
    last_crop_frame = defaultdict(int)
    last_snapshot_frame = 0

    frame_count = 0
    tracking_window_name = "Advanced Tracking with Snapshots"
    cv2.namedWindow(tracking_window_name, cv2.WINDOW_NORMAL)
    print("\n開始處理影片幀 (按 'q' 鍵退出)...")
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    
    import time
    time_stats = defaultdict(float)
    profile_interval = 100

    # --- 主處理迴圈 ---
    while cap.isOpened():
        loop_start_time = time.time()
        
        t1 = time.time()
        ret, frame = cap.read()
        if not ret: break
        time_stats["read"] += time.time() - t1
        
        frame_count += 1
        annotated_frame = frame.copy()
        
        input_frame_for_yolo = frame
        if roi_defined_manual and roi_mask_manual is not None:
            input_frame_for_yolo = cv2.bitwise_and(frame, frame, mask=roi_mask_manual)

        t3 = time.time()
        results = yolo_model.predict(
            source=input_frame_for_yolo, classes=[PERSON_CLASS_ID], 
            conf=YOLO_CONFIDENCE_THRESHOLD, iou=NMS_IOU_THRESHOLD,
            imgsz=TARGET_INFERENCE_SIZE, verbose=False, device=device
        )
        current_detections_bboxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        time_stats["yolo"] += time.time() - t3
        
        reid_extraction_time = 0

        # --- 4. 追蹤與匹配邏輯 (引入硬性否決權) ---
        t7 = time.time()
        
        active_ids_list = list(active_tracks.keys())
        active_bboxes = np.array([track['bbox'] for track in active_tracks.values()]) if active_tracks else np.array([])
        
        unmatched_active_indices = set(range(len(active_bboxes)))
        unmatched_detection_indices = set(range(len(current_detections_bboxes)))
        
        # --- 階段 1: 高速 IoU 預匹配 ---
        if len(active_bboxes) > 0 and len(current_detections_bboxes) > 0:
            iou_cost_matrix = np.ones((len(active_bboxes), len(current_detections_bboxes)))
            for i in range(len(active_bboxes)):
                for j in range(len(current_detections_bboxes)):
                    iou = calculate_iou(active_bboxes[i], current_detections_bboxes[j])
                    if iou > IOU_MATCHING_THRESHOLD:
                        iou_cost_matrix[i, j] = 1 - iou
            
            row_ind, col_ind = linear_sum_assignment(iou_cost_matrix)
            
            for r, c in zip(row_ind, col_ind):
                # 僅對 IoU 非常高的進行預匹配
                if iou_cost_matrix[r, c] < (1 - HIGH_CONF_IOU_THRESHOLD):
                    stable_id = active_ids_list[r]
                    active_tracks[stable_id]['bbox'] = current_detections_bboxes[c]
                    active_tracks[stable_id]['last_seen'] = frame_count
                    unmatched_active_indices.discard(r)
                    unmatched_detection_indices.discard(c)

        # --- 階段 2: 針對性 Re-ID 特徵提取 ---
        det_indices_for_reid = list(unmatched_detection_indices)
        if det_indices_for_reid:
            t_reid_start = time.time()
            bboxes_for_reid = current_detections_bboxes[det_indices_for_reid]
            features_for_reid = extract_features_batch(
                frame, bboxes_for_reid, reid_model, reid_transform, device)
            reid_extraction_time += time.time() - t_reid_start
        else:
            features_for_reid = np.array([])
        
        # --- 階段 3: 基於 EMA 特徵的二次匹配 ---
        unmatched_track_ids = [active_ids_list[i] for i in unmatched_active_indices] + list(lost_tracks.keys())
        reid_matched_det_indices_in_reid_list = set()
        
        if unmatched_track_ids and det_indices_for_reid:
            reid_cost_matrix = np.ones((len(unmatched_track_ids), len(det_indices_for_reid)))
            
            for i, stable_id in enumerate(unmatched_track_ids):
                track_data = active_tracks.get(stable_id) or lost_tracks.get(stable_id)
                if not track_data: continue
                
                is_lost_track = stable_id in lost_tracks
                current_feature_threshold = FEATURE_MATCHING_THRESHOLD - 0.1 if is_lost_track else FEATURE_MATCHING_THRESHOLD
                current_iou_threshold = IOU_MATCHING_THRESHOLD - 0.1 if is_lost_track else IOU_MATCHING_THRESHOLD

                for j, det_idx in enumerate(det_indices_for_reid):
                    iou = calculate_iou(track_data['bbox'], current_detections_bboxes[det_idx])
                    if iou < current_iou_threshold: continue
                        
                    similarity = np.dot(track_data['ema_feature'], features_for_reid[j])

                    # ##################################################################
                    # ### 【唯一的修改點】引入硬性否決權 ###
                    # ##################################################################
                    # 如果外觀相似度低於絕對門檻，則無論位置多近都強制拒絕匹配
                    # 這是防止特徵污染的最後一道防線
                    if similarity < ABSOLUTE_FEATURE_THRESHOLD:
                        continue
                    # ##################################################################

                    if similarity > current_feature_threshold:
                        motion_cost, appearance_cost = 1 - iou, 1 - similarity
                        reid_cost_matrix[i, j] = LAMBDA_WEIGHT * motion_cost + (1 - LAMBDA_WEIGHT) * appearance_cost

            row_ind_reid, col_ind_reid = linear_sum_assignment(reid_cost_matrix)
            
            for r, c in zip(row_ind_reid, col_ind_reid):
                if reid_cost_matrix[r, c] < 1:
                    stable_id = unmatched_track_ids[r]
                    original_det_idx = det_indices_for_reid[c]
                    new_feature = features_for_reid[c]
                    
                    if stable_id in lost_tracks:
                        active_tracks[stable_id] = lost_tracks.pop(stable_id)
                    
                    track_data = active_tracks[stable_id]
                    track_data['bbox'] = current_detections_bboxes[original_det_idx]
                    track_data['ema_feature'] = (1 - EMA_ALPHA) * new_feature + EMA_ALPHA * track_data['ema_feature']
                    track_data['last_seen'] = frame_count
                    
                    if stable_id in active_ids_list:
                        idx_in_active_list = active_ids_list.index(stable_id)
                        unmatched_active_indices.discard(idx_in_active_list)
                    reid_matched_det_indices_in_reid_list.add(c)

        # --- 階段 4: 軌跡狀態管理 (及後續邏輯保持不變) ---
        final_unmatched_det_reid_indices = [c for c in range(len(det_indices_for_reid)) if c not in reid_matched_det_indices_in_reid_list]

        for i in final_unmatched_det_reid_indices:
            if roi_defined_manual and not available_ids: continue
            if available_ids:
                new_id = min(available_ids)
                available_ids.remove(new_id)
                new_feature = features_for_reid[i]
                original_det_idx = det_indices_for_reid[i]
                
                active_tracks[new_id] = {
                    'bbox': current_detections_bboxes[original_det_idx],
                    'ema_feature': new_feature,
                    'last_seen': frame_count,
                }

        ids_to_move_to_lost = [active_ids_list[i] for i in unmatched_active_indices]
        for stable_id in ids_to_move_to_lost:
            if stable_id in active_tracks:
                lost_tracks[stable_id] = active_tracks.pop(stable_id)
                lost_tracks[stable_id]['lost_for'] = 0

        ids_to_remove_from_lost = []
        for sid, data in lost_tracks.items():
            data['lost_for'] += 1
            if data['lost_for'] > MAX_LOST_FRAMES:
                ids_to_remove_from_lost.append(sid)
        
        for sid in ids_to_remove_from_lost:
            if sid in lost_tracks: del lost_tracks[sid]
            if sid in track_history_for_drawing: del track_history_for_drawing[sid]
            available_ids.add(sid)
        
        time_stats["tracking"] += time.time() - t7
        time_stats["reid"] += reid_extraction_time

        # --- 5. 繪圖與存檔 ---
        # (此部分及後續 profiling 邏輯完全不變)
    # --- 5. 繪圖與存檔 ---
        t9 = time.time()
        for stable_id, track_data in active_tracks.items():
            x1, y1, x2, y2 = track_data['bbox']
            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), FIXED_BOX_COLOR_BGR, BOX_THICKNESS)
            label_text = f"ID:{stable_id}"
            (text_w, text_h), baseline = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE_LABEL, FONT_THICKNESS_LABEL)
            text_y_pos = y1 - 7 if y1 - text_h - 7 > 0 else y1 + text_h + baseline + 7
            cv2.rectangle(annotated_frame, (x1, text_y_pos - text_h - baseline), (x1 + text_w, text_y_pos + baseline), FIXED_BOX_COLOR_BGR, -1)
            cv2.putText(annotated_frame, label_text, (x1, text_y_pos), cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE_LABEL, (0,0,0), FONT_THICKNESS_LABEL, cv2.LINE_AA)
            center_x, center_y = (x1 + x2) // 2, (y1 + y2) // 2
            track_history_for_drawing[stable_id].append((center_x, center_y))
            points = np.array(list(track_history_for_drawing[stable_id]), dtype=np.int32).reshape((-1, 1, 2))
            if len(points) > 1:
                cv2.polylines(annotated_frame, [points], isClosed=False, color=(230, 230, 230), thickness=TRACK_LINE_THICKNESS)
            
            # ##################################################################
            # ### 【修改點 1】將個人裁切邏輯加回來 ###
            # ##################################################################
            if ENABLE_CROPPING and (frame_count - last_crop_frame.get(stable_id, -crop_frame_interval)) >= crop_frame_interval:
                last_crop_frame[stable_id] = frame_count
                current_time_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
                time_str = format_time_from_ms(current_time_ms)
                
                id_folder_path = os.path.join(OUTPUT_CROP_DIR, f"ID_{stable_id}")
                os.makedirs(id_folder_path, exist_ok=True)
                
                # 從原始、乾淨的 frame 中裁切，而不是從 annotated_frame
                cropped_image = frame[max(0, y1):min(frame.shape[0], y2), max(0, x1):min(frame.shape[1], x2)]
                
                if cropped_image.size > 0:
                    filename = f"{time_str}.jpg"
                    cv2.imwrite(os.path.join(id_folder_path, filename), cropped_image)
            # ##################################################################

        # ##################################################################
        # ### 【修改點 2】將班級快照邏輯加回來 ###
        # ##################################################################
        if ENABLE_CLASS_SNAPSHOT and (frame_count - last_snapshot_frame) >= crop_frame_interval:
            last_snapshot_frame = frame_count
            current_time_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
            time_str = format_time_from_ms(current_time_ms)

            # 根據 SAVE_ANNOTATED_SNAPSHOT 的設定來決定要儲存哪張圖
            image_to_save = annotated_frame if SAVE_ANNOTATED_SNAPSHOT else frame
            
            snapshot_filename = f"snapshot_{time_str}.jpg"
            snapshot_path = os.path.join(OUTPUT_SNAPSHOT_DIR, snapshot_filename)
            cv2.imwrite(snapshot_path, image_to_save)
        # ##################################################################
        
        if roi_defined_manual and roi_mask_manual is not None:
            cv2.polylines(annotated_frame, [np.array(roi_points_manual, dtype=np.int32)], True, ROI_POLYGON_COLOR, 2)
        
        time_stats["draw_save"] += time.time() - t9

        time_stats["total"] += time.time() - loop_start_time
        
        cv2.imshow(tracking_window_name, annotated_frame)
        
        if frame_count % profile_interval == 0:
            print("\n--- 效能分析報告 (過去 100 幀平均耗時) ---")
            total_time = time_stats["total"]
            if total_time > 0:
                avg_fps = profile_interval / total_time
                print(f"平均 FPS: {avg_fps:.2f}")
                categories = ["read", "yolo", "reid", "tracking", "draw_save"]
                cat_names = ["影片讀取", "YOLO 推論", "Re-ID 提取", "追蹤匹配", "繪圖與存檔"]
                total_categorized = sum(time_stats[cat] for cat in categories)
                for i, cat in enumerate(categories):
                    print(f"{i+1}. {cat_names[i]:<10}: {(time_stats[cat] / total_time) * 100:6.2f}% ({time_stats[cat]:.3f} s)")
                other_time = total_time - total_categorized
                print(f"{len(categories)+1}. {'其他 (顯示/延遲)':<10}: {(other_time / total_time) * 100:6.2f}% ({other_time:.3f} s)")
                print("-------------------------------------------------")
            
            time_stats = defaultdict(float)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("處理完成。")


if __name__ == "__main__":
    main()

### Yolo+Clip

In [ ]:
# -*- coding: utf-8 -*-
from collections import defaultdict, deque
import cv2
import numpy as np
import torch
from ultralytics import YOLO
import os
from scipy.optimize import linear_sum_assignment
from PIL import Image
import datetime

from transformers import CLIPProcessor, CLIPModel

# --- 全域變數 for ROI ---
roi_points_manual = []
roi_defined_manual = False
roi_mask_manual = None
img_for_roi_selection_manual = None

# --- 參數設定 ---
YOLO_MODEL_PATH = 'yolov8l.pt'
PERSON_CLASS_ID = 0
YOLO_CONFIDENCE_THRESHOLD = 0.4
NMS_IOU_THRESHOLD = 0.3
TARGET_INFERENCE_SIZE = 1920
MODEL_ID = "openai/clip-vit-base-patch32"

# --- 【新功能】影片開始處理時間 ---
# 以 "時:分:秒" 格式設定影片開始處理的時間點。例如 "00:03:13"。
# 若設為 "" 或 None，則從頭開始處理。
START_TIME_STR = "00:15:51" 

# --- 追蹤與匹配參數 ---
MAX_LOST_FRAMES = 20000
IOU_MATCHING_THRESHOLD = 0.4
# 【優化三】稍微提高活躍軌跡的門檻，讓匹配更嚴格
FEATURE_MATCHING_THRESHOLD = 0.84 
ABSOLUTE_FEATURE_THRESHOLD = 0.70
LAMBDA_WEIGHT = 0.8
HIGH_CONF_IOU_THRESHOLD = 0.7 
EMA_ALPHA = 0.95 

# --- 優化參數 ---
LOST_TRACK_FEATURE_THRESHOLD = 0.75 
LOST_TRACK_IOU_THRESHOLD = 0.3      
REID_CONFIRMATION_FRAMES = 5        
# 【優化一】如果目標中心點移動小於 N 像素，則視為靜止，不更新其外觀特徵
STATIC_MOVEMENT_THRESHOLD = 5 
# 【優化二】在計算相似度時，初始特徵(錨點)的權重
INITIAL_FEATURE_WEIGHT = 0.2

# --- 效能優化參數 ---
FRAME_SKIP_INTERVAL = 5  # 每 N 幀處理一次，跳過中間的 N-1 幀。設為 1 則不跳過。

MAX_TRACKS_IN_ROI = 20
available_ids = set()

# --- 裁切參數 ---
ENABLE_CROPPING = True
CROP_INTERVAL_SECONDS = 0.7
OUTPUT_CROP_DIR = "1019_english_mid"

# --- 班級快照參數 ---
ENABLE_CLASS_SNAPSHOT = True
OUTPUT_SNAPSHOT_DIR = "1019_english_class"

# --- 【新功能】班級快照儲存選項 ---
# True: 儲存帶有標註框的快照; False: 儲存乾淨的原始快照
SAVE_ANNOTATED_SNAPSHOT = False 


# --- 繪圖參數 ---
BOX_THICKNESS = 2
FONT_SCALE_LABEL = 0.5
FONT_THICKNESS_LABEL = 2
TRACK_LINE_THICKNESS = 3
FIXED_BOX_COLOR_BGR = (255, 191, 0)
ROI_POLYGON_COLOR = (0, 0, 255)

# --- 輔助函式 ---
def manual_roi_mouse_click(event, x, y, flags, param):
    global roi_points_manual, img_for_roi_selection_manual
    if event == cv2.EVENT_LBUTTONDOWN and len(roi_points_manual) < 4:
        roi_points_manual.append((x, y))
        cv2.circle(img_for_roi_selection_manual, (x, y), 5, (0, 255, 0), -1)
        if len(roi_points_manual) > 1:
            cv2.line(img_for_roi_selection_manual, roi_points_manual[-2], roi_points_manual[-1], (0, 255, 0), 2)
        if len(roi_points_manual) == 4:
            cv2.line(img_for_roi_selection_manual, roi_points_manual[3], roi_points_manual[0], (0, 255, 0), 2)
        cv2.imshow("Select ROI - 4 points, then press 'c' or 's'", img_for_roi_selection_manual)

def calculate_iou(box1, box2):
    x1_1, y1_1, x2_1, y2_1 = box1; x1_2, y1_2, x2_2, y2_2 = box2
    xi1, yi1, xi2, yi2 = max(x1_1, x1_2), max(y1_1, y1_2), min(x2_1, x2_2), min(y2_1, y2_2)
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    box1_area, box2_area = (x2_1 - x1_1) * (y2_1 - y1_1), (x2_2 - x1_2) * (y2_2 - y1_2)
    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0

def format_time_from_ms(milliseconds):
    if milliseconds < 0: return "00-00-00-000"
    td = datetime.timedelta(milliseconds=milliseconds)
    hours, remainder = divmod(td.seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}-{minutes:02d}-{seconds:02d}-{td.microseconds // 1000:03d}"

# --- 【新功能】新增的輔助函式 ---
def parse_time_to_ms(time_str):
    """將 HH:MM:SS 格式的時間字串轉換為毫秒"""
    if not time_str or not isinstance(time_str, str):
        return 0
    try:
        parts = time_str.split(':')
        if len(parts) != 3:
            print(f"警告：時間格式錯誤 '{time_str}'，應為 HH:MM:SS。將從頭開始處理。")
            return 0
        h, m, s = map(int, parts)
        milliseconds = (h * 3600 + m * 60 + s) * 1000
        return milliseconds
    except (ValueError, TypeError):
        print(f"警告：無法解析時間 '{time_str}'。將從頭開始處理。")
        return 0

# --- 特徵提取函式 (保持不變) ---
def extract_features_batch(frame, bboxes, reid_model, reid_processor, device):
    if bboxes.shape[0] == 0: return np.array([])
    pil_images = []
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    valid_indices = []
    for i, bbox in enumerate(bboxes):
        x1, y1, x2, y2 = bbox
        roi_rgb = frame_rgb[max(0, y1):min(frame.shape[0], y2), max(0, x1):min(frame.shape[1], x2)]
        if roi_rgb.size > 0:
            pil_images.append(Image.fromarray(roi_rgb))
            valid_indices.append(i)
    if not pil_images: return np.array([]), []
    inputs = reid_processor(images=pil_images, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        features_batch = reid_model.get_image_features(**inputs)
        features_batch /= features_batch.norm(dim=-1, keepdim=True)
    return features_batch.cpu().numpy(), valid_indices

# --- 主函數 ---
def main():
    global roi_points_manual, roi_defined_manual, roi_mask_manual, img_for_roi_selection_manual, available_ids
    available_ids = set(range(1, MAX_TRACKS_IN_ROI + 1))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"正在使用設備: {device}")

    # --- 模型載入 (保持不變) ---
    print(f"正在載入 YOLO 模型: {YOLO_MODEL_PATH}...")
    try: yolo_model = YOLO(YOLO_MODEL_PATH).to(device)
    except Exception as e: print(f"YOLO 載入失敗: {e}"); exit()
    print(f"正在從 Hugging Face 載入 CLIP 模型: {MODEL_ID}...")
    reid_model, reid_processor = None, None
    try:
        reid_model = CLIPModel.from_pretrained(MODEL_ID, use_safetensors=True).to(device).eval()
        reid_processor = CLIPProcessor.from_pretrained(MODEL_ID)
    except Exception as e: print(f"CLIP 載入失敗: {e}"); exit()
    if reid_model is None: print("CLIP 未能初始化，程式終止。"); return

    # --- 影片讀取與 ROI 設定 (有修改) ---
    video_path = input("請輸入影片檔案路徑 (.mp4 或 .mov): ").strip().strip('"')
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): print(f"錯誤：無法開啟影片檔案 {video_path}"); return

    start_ms = 0
    # --- 【新功能】處理影片開始時間 ---
    if START_TIME_STR:
        start_ms = parse_time_to_ms(START_TIME_STR)
        if start_ms > 0:
            success = cap.set(cv2.CAP_PROP_POS_MSEC, start_ms)
            if success:
                print(f"成功將影片跳轉至 {START_TIME_STR} ({start_ms} ms) 開始處理。")
            else:
                print(f"警告：無法將影片跳轉至指定時間 {START_TIME_STR}。將從頭開始處理。")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    crop_frame_interval = int(fps * CROP_INTERVAL_SECONDS)
    if ENABLE_CROPPING: os.makedirs(OUTPUT_CROP_DIR, exist_ok=True); print(f"個人裁切功能已啟用。")
    if ENABLE_CLASS_SNAPSHOT: os.makedirs(OUTPUT_SNAPSHOT_DIR, exist_ok=True); print(f"班級快照功能已啟用。")
    
    # 讀取用於 ROI 設定的幀（現在會是跳轉後的幀）
    ret_roi, first_frame_for_roi = cap.read()
    if not ret_roi: 
        print("錯誤：無法從指定的時間點讀取影片幀。")
        cap.release()
        return
        
    img_for_roi_selection_manual = first_frame_for_roi.copy()
    roi_select_window_name = "Select ROI - 4 points, then press 'c' or 's'"
    cv2.namedWindow(roi_select_window_name, cv2.WINDOW_NORMAL)
    cv2.imshow(roi_select_window_name, img_for_roi_selection_manual)
    cv2.setMouseCallback(roi_select_window_name, manual_roi_mouse_click)
    while True:
        key_roi = cv2.waitKey(20) & 0xFF
        if cv2.getWindowProperty(roi_select_window_name, cv2.WND_PROP_VISIBLE) < 1 or key_roi in [ord('c'), ord('s'), ord('q')]:
            if len(roi_points_manual) == 4 and (key_roi == ord('c') or cv2.getWindowProperty(roi_select_window_name, cv2.WND_PROP_VISIBLE) < 1):
                roi_defined_manual = True; print("ROI 已定義。")
            elif key_roi == ord('s'): print("已跳過 ROI 定義。")
            if key_roi == ord('q'): cap.release(); cv2.destroyAllWindows(); return
            break
    if roi_defined_manual:
        roi_mask_manual = np.zeros(first_frame_for_roi.shape[:2], dtype=np.uint8)
        cv2.fillPoly(roi_mask_manual, [np.array(roi_points_manual, dtype=np.int32)], 255)
    cv2.destroyAllWindows()
    
    # --- 初始化追蹤變數 (保持不變) ---
    active_tracks, lost_tracks = {}, {}
    last_crop_frame = defaultdict(int)
    last_snapshot_frame = 0
    frame_count = 0
    tracking_window_name = "Optimized Long-Term Tracking"
    cv2.namedWindow(tracking_window_name, cv2.WINDOW_NORMAL)
    print("\n開始處理影片幀 (按 'q' 鍵退出)...")

    # 將指針重設回我們想要開始的位置，因為 ROI 設定時已經讀取了一幀
    if START_TIME_STR and start_ms > 0:
        cap.set(cv2.CAP_PROP_POS_MSEC, start_ms)

    # --- 主處理迴圈 ---
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame_count += 1
        
        # 【優化】幀跳過邏輯
        # 如果不是要處理的幀 (例如第2, 3, 5, 6...幀)，則只繪製上一幀的結果，然後跳到下一輪迴圈
        if frame_count % FRAME_SKIP_INTERVAL != 1 and FRAME_SKIP_INTERVAL > 1:
            # 即使跳過計算，我們仍然需要繪製上一幀的結果以保持畫面流暢
            annotated_frame = frame.copy()
            for sid, track in active_tracks.items():
                x1, y1, x2, y2 = track['bbox']
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), FIXED_BOX_COLOR_BGR, BOX_THICKNESS)
                label = f"ID:{sid}"; cv2.putText(annotated_frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE_LABEL, FIXED_BOX_COLOR_BGR, FONT_THICKNESS_LABEL)
            
            if roi_defined_manual: cv2.polylines(annotated_frame, [np.array(roi_points_manual, dtype=np.int32)], True, ROI_POLYGON_COLOR, 2)
            cv2.imshow(tracking_window_name, annotated_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'): break
            continue # 直接跳到下一個 while 迴圈

        # --- 以下是完整的處理流程，只在需要處理的幀 (例如第1, 4, 7...幀) 上運行 ---

        # 1. 偵測
        input_frame_for_yolo = cv2.bitwise_and(frame, frame, mask=roi_mask_manual) if roi_defined_manual else frame
        results = yolo_model.predict(input_frame_for_yolo, classes=[PERSON_CLASS_ID], conf=YOLO_CONFIDENCE_THRESHOLD, iou=NMS_IOU_THRESHOLD, imgsz=TARGET_INFERENCE_SIZE, verbose=False)
        current_detections_bboxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        
        # 2. 追蹤與匹配邏輯 (保持您的長期優化版本)
        active_ids_list = list(active_tracks.keys())
        unmatched_active_indices = set(range(len(active_ids_list)))
        unmatched_detection_indices = set(range(len(current_detections_bboxes)))
        if len(active_ids_list) > 0 and len(current_detections_bboxes) > 0:
            active_bboxes = np.array([t['bbox'] for t in active_tracks.values()])
            iou_cost = 1 - np.array([[calculate_iou(b1, b2) for b2 in current_detections_bboxes] for b1 in active_bboxes])
            row_ind, col_ind = linear_sum_assignment(iou_cost)
            for r, c in zip(row_ind, col_ind):
                if iou_cost[r, c] < (1 - HIGH_CONF_IOU_THRESHOLD):
                    stable_id = active_ids_list[r]
                    active_tracks[stable_id].update({'bbox': current_detections_bboxes[c], 'last_seen': frame_count})
                    unmatched_active_indices.discard(r)
                    unmatched_detection_indices.discard(c)
        det_indices_for_reid = list(unmatched_detection_indices)
        features_for_reid, valid_feature_indices = np.array([]), []
        if det_indices_for_reid:
            bboxes_for_reid = current_detections_bboxes[det_indices_for_reid]
            features_for_reid, valid_feature_indices = extract_features_batch(frame, bboxes_for_reid, reid_model, reid_processor, device)
        unmatched_track_ids = [active_ids_list[i] for i in unmatched_active_indices] + list(lost_tracks.keys())
        if unmatched_track_ids and features_for_reid.shape[0] > 0:
            cost_matrix = np.full((len(unmatched_track_ids), len(features_for_reid)), 1.0)
            for i, sid in enumerate(unmatched_track_ids):
                track = active_tracks.get(sid) or lost_tracks.get(sid)
                is_lost = sid in lost_tracks
                current_feature_thresh = LOST_TRACK_FEATURE_THRESHOLD if is_lost else FEATURE_MATCHING_THRESHOLD
                current_iou_thresh = LOST_TRACK_IOU_THRESHOLD if is_lost else IOU_MATCHING_THRESHOLD
                for j in range(features_for_reid.shape[0]):
                    original_det_idx = det_indices_for_reid[valid_feature_indices[j]]
                    iou = calculate_iou(track['bbox'], current_detections_bboxes[original_det_idx])
                    if iou < current_iou_thresh: continue
                    sim_ema = np.dot(track['ema_feature'], features_for_reid[j])
                    sim_initial = np.dot(track['initial_feature'], features_for_reid[j])
                    combined_sim = (1 - INITIAL_FEATURE_WEIGHT) * sim_ema + INITIAL_FEATURE_WEIGHT * sim_initial
                    if combined_sim > current_feature_thresh and combined_sim > ABSOLUTE_FEATURE_THRESHOLD:
                        cost_matrix[i, j] = LAMBDA_WEIGHT * (1 - iou) + (1 - LAMBDA_WEIGHT) * (1 - combined_sim)
            row_ind, col_ind = linear_sum_assignment(cost_matrix)
            for r, c in zip(row_ind, col_ind):
                if cost_matrix[r, c] < 1:
                    sid = unmatched_track_ids[r]
                    original_det_idx = det_indices_for_reid[valid_feature_indices[c]]
                    old_bbox = (active_tracks.get(sid) or lost_tracks.get(sid))['bbox']
                    old_center = ((old_bbox[0] + old_bbox[2]) / 2, (old_bbox[1] + old_bbox[3]) / 2)
                    was_lost = sid in lost_tracks
                    if was_lost:
                        active_tracks[sid] = lost_tracks.pop(sid)
                        active_tracks[sid]['reid_confirmed_frames'] = 0 
                    track = active_tracks[sid]
                    new_bbox = current_detections_bboxes[original_det_idx]
                    track.update({'bbox': new_bbox, 'last_seen': frame_count})
                    new_center = ((new_bbox[0] + new_bbox[2]) / 2, (new_bbox[1] + new_bbox[3]) / 2)
                    movement = np.linalg.norm(np.array(old_center) - np.array(new_center))
                    if track.get('reid_confirmed_frames', REID_CONFIRMATION_FRAMES) < REID_CONFIRMATION_FRAMES:
                        track['reid_confirmed_frames'] += 1
                    elif movement > STATIC_MOVEMENT_THRESHOLD:
                        new_feature = features_for_reid[c]
                        track['ema_feature'] = (1 - EMA_ALPHA) * new_feature + EMA_ALPHA * track['ema_feature']
                    if not was_lost:
                        unmatched_active_indices.discard(active_ids_list.index(sid))
                    unmatched_detection_indices.discard(original_det_idx)

        # 3. 軌跡狀態管理 (保持不變)
        for idx in unmatched_active_indices:
            sid = active_ids_list[idx]
            if sid in active_tracks: lost_tracks[sid] = active_tracks.pop(sid)
        for sid in list(lost_tracks.keys()):
            if frame_count - lost_tracks[sid]['last_seen'] > MAX_LOST_FRAMES:
                available_ids.add(sid); del lost_tracks[sid]
        for det_idx in unmatched_detection_indices:
            if available_ids:
                new_id = min(available_ids)
                available_ids.remove(new_id)
                bbox_for_new = np.array([current_detections_bboxes[det_idx]])
                feature_for_new, _ = extract_features_batch(frame, bbox_for_new, reid_model, reid_processor, device)
                if feature_for_new.shape[0] > 0:
                    active_tracks[new_id] = {'bbox': bbox_for_new[0], 'ema_feature': feature_for_new[0], 'initial_feature': feature_for_new[0].copy(), 'last_seen': frame_count}

        # 4. 繪圖與存檔 (保持不變)
        annotated_frame = frame.copy()
        for sid, track in active_tracks.items():
            x1, y1, x2, y2 = track['bbox']
            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), FIXED_BOX_COLOR_BGR, BOX_THICKNESS)
            label = f"ID:{sid}"; cv2.putText(annotated_frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE_LABEL, FIXED_BOX_COLOR_BGR, FONT_THICKNESS_LABEL)
            if ENABLE_CROPPING and (frame_count - last_crop_frame.get(sid, -crop_frame_interval)) >= crop_frame_interval:
                last_crop_frame[sid] = frame_count
                relative_time_ms = cap.get(cv2.CAP_PROP_POS_MSEC) - start_ms
                time_str = format_time_from_ms(relative_time_ms)
                id_folder_path = os.path.join(OUTPUT_CROP_DIR, f"ID_{sid}"); os.makedirs(id_folder_path, exist_ok=True)
                cropped_image = frame[max(0, y1):min(frame.shape[0], y2), max(0, x1):min(frame.shape[1], x2)]
                if cropped_image.size > 0:
                     cv2.imwrite(os.path.join(id_folder_path, f"{time_str}.jpg"), cropped_image, [cv2.IMWRITE_JPEG_QUALITY, 95])
        if ENABLE_CLASS_SNAPSHOT and (frame_count - last_snapshot_frame) >= crop_frame_interval:
            last_snapshot_frame = frame_count
            relative_time_ms = cap.get(cv2.CAP_PROP_POS_MSEC) - start_ms
            time_str = format_time_from_ms(relative_time_ms)
            image_to_save = annotated_frame if SAVE_ANNOTATED_SNAPSHOT else frame
            cv2.imwrite(os.path.join(OUTPUT_SNAPSHOT_DIR, f"snapshot_{time_str}.jpg"), image_to_save)
        
        if roi_defined_manual: cv2.polylines(annotated_frame, [np.array(roi_points_manual, dtype=np.int32)], True, ROI_POLYGON_COLOR, 2)
        cv2.imshow(tracking_window_name, annotated_frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

    cap.release()
    cv2.destroyAllWindows()
    print("處理完成。")

if __name__ == "__main__":
    main()

### TEST AI加強圖片

In [ ]:
# -*- coding: utf-8 -*-
from collections import defaultdict, deque
import cv2
import numpy as np
import torch
from ultralytics import YOLO
import os
from scipy.optimize import linear_sum_assignment
from PIL import Image
import datetime

from transformers import CLIPProcessor, CLIPModel

# --- 全域變數 for ROI ---
roi_points_manual = []
roi_defined_manual = False
roi_mask_manual = None
img_for_roi_selection_manual = None

# --- 參數設定 ---
YOLO_MODEL_PATH = 'yolov8x.pt'
PERSON_CLASS_ID = 0
YOLO_CONFIDENCE_THRESHOLD = 0.4
NMS_IOU_THRESHOLD = 0.3
TARGET_INFERENCE_SIZE = 640
MODEL_ID = "openai/clip-vit-base-patch32"

# --- 【新功能】影片開始處理時間 ---
# 以 "時:分:秒" 格式設定影片開始處理的時間點。例如 "00:03:13"。
# 若設為 "" 或 None，則從頭開始處理。
START_TIME_STR = "00:01:40" 

# --- 追蹤與匹配參數 ---
MAX_LOST_FRAMES = 20000
IOU_MATCHING_THRESHOLD = 0.4
# 【優化三】稍微提高活躍軌跡的門檻，讓匹配更嚴格
FEATURE_MATCHING_THRESHOLD = 0.84 
ABSOLUTE_FEATURE_THRESHOLD = 0.70
LAMBDA_WEIGHT = 0.8
HIGH_CONF_IOU_THRESHOLD = 0.7 
EMA_ALPHA = 0.95 

# --- 優化參數 ---
LOST_TRACK_FEATURE_THRESHOLD = 0.75 
LOST_TRACK_IOU_THRESHOLD = 0.3      
REID_CONFIRMATION_FRAMES = 5        
# 【優化一】如果目標中心點移動小於 N 像素，則視為靜止，不更新其外觀特徵
STATIC_MOVEMENT_THRESHOLD = 5 
# 【優化二】在計算相似度時，初始特徵(錨點)的權重
INITIAL_FEATURE_WEIGHT = 0.2

# --- 效能優化參數 ---
FRAME_SKIP_INTERVAL = 3  # 每 N 幀處理一次，跳過中間的 N-1 幀。設為 1 則不跳過。

MAX_TRACKS_IN_ROI = 17
available_ids = set()

# --- 裁切參數 ---
ENABLE_CROPPING = True
CROP_INTERVAL_SECONDS = 0.7
OUTPUT_CROP_DIR = "0907_english_mid"

# --- 班級快照參數 ---
ENABLE_CLASS_SNAPSHOT = True
OUTPUT_SNAPSHOT_DIR = "0907_english_class"

# --- 【新功能】班級快照儲存選項 ---
# True: 儲存帶有標註框的快照; False: 儲存乾淨的原始快照
SAVE_ANNOTATED_SNAPSHOT = False 


# --- 繪圖參數 ---
BOX_THICKNESS = 2
FONT_SCALE_LABEL = 0.5
FONT_THICKNESS_LABEL = 2
TRACK_LINE_THICKNESS = 3
FIXED_BOX_COLOR_BGR = (255, 191, 0)
ROI_POLYGON_COLOR = (0, 0, 255)

# --- 輔助函式 ---
def manual_roi_mouse_click(event, x, y, flags, param):
    global roi_points_manual, img_for_roi_selection_manual
    if event == cv2.EVENT_LBUTTONDOWN and len(roi_points_manual) < 4:
        roi_points_manual.append((x, y))
        cv2.circle(img_for_roi_selection_manual, (x, y), 5, (0, 255, 0), -1)
        if len(roi_points_manual) > 1:
            cv2.line(img_for_roi_selection_manual, roi_points_manual[-2], roi_points_manual[-1], (0, 255, 0), 2)
        if len(roi_points_manual) == 4:
            cv2.line(img_for_roi_selection_manual, roi_points_manual[3], roi_points_manual[0], (0, 255, 0), 2)
        cv2.imshow("Select ROI - 4 points, then press 'c' or 's'", img_for_roi_selection_manual)

def calculate_iou(box1, box2):
    x1_1, y1_1, x2_1, y2_1 = box1; x1_2, y1_2, x2_2, y2_2 = box2
    xi1, yi1, xi2, yi2 = max(x1_1, x1_2), max(y1_1, y1_2), min(x2_1, x2_2), min(y2_1, y2_2)
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    box1_area, box2_area = (x2_1 - x1_1) * (y2_1 - y1_1), (x2_2 - x1_2) * (y2_2 - y1_2)
    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0

def format_time_from_ms(milliseconds):
    if milliseconds < 0: return "00-00-00-000"
    td = datetime.timedelta(milliseconds=milliseconds)
    hours, remainder = divmod(td.seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}-{minutes:02d}-{seconds:02d}-{td.microseconds // 1000:03d}"

# --- 【新功能】新增的輔助函式 ---
def parse_time_to_ms(time_str):
    """將 HH:MM:SS 格式的時間字串轉換為毫秒"""
    if not time_str or not isinstance(time_str, str):
        return 0
    try:
        parts = time_str.split(':')
        if len(parts) != 3:
            print(f"警告：時間格式錯誤 '{time_str}'，應為 HH:MM:SS。將從頭開始處理。")
            return 0
        h, m, s = map(int, parts)
        milliseconds = (h * 3600 + m * 60 + s) * 1000
        return milliseconds
    except (ValueError, TypeError):
        print(f"警告：無法解析時間 '{time_str}'。將從頭開始處理。")
        return 0

# --- 特徵提取函式 (保持不變) ---
def extract_features_batch(frame, bboxes, reid_model, reid_processor, device):
    if bboxes.shape[0] == 0: return np.array([])
    pil_images = []
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    valid_indices = []
    for i, bbox in enumerate(bboxes):
        x1, y1, x2, y2 = bbox
        roi_rgb = frame_rgb[max(0, y1):min(frame.shape[0], y2), max(0, x1):min(frame.shape[1], x2)]
        if roi_rgb.size > 0:
            pil_images.append(Image.fromarray(roi_rgb))
            valid_indices.append(i)
    if not pil_images: return np.array([]), []
    inputs = reid_processor(images=pil_images, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        features_batch = reid_model.get_image_features(**inputs)
        features_batch /= features_batch.norm(dim=-1, keepdim=True)
    return features_batch.cpu().numpy(), valid_indices

# --- 主函數 ---
def main():
    global roi_points_manual, roi_defined_manual, roi_mask_manual, img_for_roi_selection_manual, available_ids
    # 初始化可用 ID 集合
    available_ids = set(range(1, MAX_TRACKS_IN_ROI + 1))
    
    # =========================================================================
    # --- 步驟 1: GUI 互動階段 - 設定影片路徑與 ROI
    #   (此階段只處理影片讀取和使用者介面，完全不載入大型 AI 模型)
    # =========================================================================
    
    # --- 影片讀取 ---
    video_path = input("請輸入影片檔案路徑 (.mp4 或 .mov): ").strip().strip('"')
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): 
        print(f"錯誤：無法開啟影片檔案 {video_path}")
        return

    # --- 處理影片開始時間 ---
    start_ms = 0 # 初始化 start_ms
    if START_TIME_STR:
        start_ms = parse_time_to_ms(START_TIME_STR)
        if start_ms > 0:
            # 嘗試跳轉到指定時間點
            success = cap.set(cv2.CAP_PROP_POS_MSEC, start_ms)
            if not success:
                print(f"警告：無法將影片跳轉至指定時間 {START_TIME_STR}。將從頭開始處理。")
                # 如果跳轉失敗，確保指針回到開頭
                cap.set(cv2.CAP_PROP_POS_MSEC, 0)

    # 讀取用於 ROI 設定的幀
    ret_roi, first_frame_for_roi = cap.read()
    if not ret_roi: 
        print("錯誤：無法從影片中讀取第一幀以設定 ROI。")
        cap.release()
        return
        
    # --- ROI 手動設定 GUI ---
    img_for_roi_selection_manual = first_frame_for_roi.copy()
    roi_select_window_name = "Select ROI - 4 points, then press 'c' or 's'"
    # 使用 WINDOW_NORMAL 讓使用者可以縮放視窗
    cv2.namedWindow(roi_select_window_name, cv2.WINDOW_NORMAL) 
    cv2.imshow(roi_select_window_name, img_for_roi_selection_manual)
    cv2.setMouseCallback(roi_select_window_name, manual_roi_mouse_click)
    print("請在彈出的視窗中點擊4個點定義ROI，完成後按 'c' 確認，或按 's' 跳過。")
    
    # 等待使用者完成 ROI 設定
    while True:
        key_roi = cv2.waitKey(20) & 0xFF
        # 檢查視窗是否被手動關閉，或是否按下了指定按鍵
        if cv2.getWindowProperty(roi_select_window_name, cv2.WND_PROP_VISIBLE) < 1 or key_roi in [ord('c'), ord('s'), ord('q')]:
            if len(roi_points_manual) == 4 and key_roi == ord('c'):
                roi_defined_manual = True
                print("ROI 已成功定義。")
            elif key_roi == ord('s'): 
                print("已跳過 ROI 定義，將處理整個畫面。")
            elif key_roi == ord('q'): 
                print("使用者中斷操作。")
                cap.release()
                cv2.destroyAllWindows()
                return
            break
            
    if roi_defined_manual:
        roi_mask_manual = np.zeros(first_frame_for_roi.shape[:2], dtype=np.uint8)
        cv2.fillPoly(roi_mask_manual, [np.array(roi_points_manual, dtype=np.int32)], 255)
    
    # 【關鍵修復】
    # 在載入任何大型模型 (特別是使用 CUDA 的 PyTorch 模型) 之前，
    # 必須徹底關閉並銷毀所有由 OpenCV 創建的 GUI 視窗。
    # 這可以防止因 GUI 執行緒與 CUDA 初始化之間的衝突而導致的核心崩潰。
    print("\nROI 設定完成。正在關閉 GUI 視窗...")
    cv2.destroyAllWindows()
    # 增加一個極短的延遲，確保作業系統有足夠時間完全釋放視窗資源
    # time.sleep(0.1) 

    # =========================================================================
    # --- 步驟 2: 模型載入階段
    #   (此階段在所有 GUI 視窗都已關閉後執行，專心載入模型到 GPU)
    # =========================================================================
    print("現在開始載入 AI 模型，此過程可能需要一些時間，請稍候...")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"正在使用設備: {device}")

    # --- 載入 YOLO 模型 ---
    try: 
        print(f"正在載入 YOLO 模型: {YOLO_MODEL_PATH}...")
        yolo_model = YOLO(YOLO_MODEL_PATH).to(device)
        print("YOLO 模型載入成功。")
    except Exception as e: 
        print(f"錯誤：YOLO 模型載入失敗: {e}")
        cap.release()
        return
        
    # --- 載入 CLIP Re-ID 模型 ---
    try:
        print(f"正在從 Hugging Face 載入 CLIP 模型: {MODEL_ID}...")
        reid_model = CLIPModel.from_pretrained(MODEL_ID, use_safetensors=True).to(device).eval()
        reid_processor = CLIPProcessor.from_pretrained(MODEL_ID)
        print("CLIP 模型載入成功。")
    except Exception as e: 
        print(f"錯誤：CLIP 模型載入失敗: {e}")
        cap.release()
        return

    # --- 載入 AI 超解析度模型 (可選) ---
    use_sr = False
    sr = None
    if ENABLE_CROPPING:
        sr = cv2.dnn_superres.DnnSuperResImpl_create()
        model_path = "EDSR_x4.pb" # 確保此模型檔存在於您的專案目錄中
        if os.path.exists(model_path):
            try:
                sr.readModel(model_path)
                sr.setModel("edsr", 4)
                print(f"成功載入 AI 超解析度模型 ({model_path})。")
                use_sr = True
            except Exception as e:
                print(f"警告：載入 AI 超解析度模型失敗: {e}。將儲存原始解析度裁切圖。")
        else:
            print(f"警告：找不到 AI 超解析度模型檔案 '{model_path}'。將儲存原始解析度裁切圖。")

    # =========================================================================
    # --- 步驟 3: 主處理迴圈階段
    #   (所有模型都準備就緒，開始逐幀處理影片)
    # =========================================================================

    # --- 初始化追蹤相關變數 ---
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    crop_frame_interval = int(fps * CROP_INTERVAL_SECONDS)
    if ENABLE_CROPPING: os.makedirs(OUTPUT_CROP_DIR, exist_ok=True); print(f"個人裁切圖將儲存至: {OUTPUT_CROP_DIR}")
    if ENABLE_CLASS_SNAPSHOT: os.makedirs(OUTPUT_SNAPSHOT_DIR, exist_ok=True); print(f"班級快照將儲存至: {OUTPUT_SNAPSHOT_DIR}")
    
    active_tracks, lost_tracks = {}, {}
    last_crop_frame = defaultdict(int)
    last_snapshot_frame = 0
    frame_count = 0
    
    # 建立最終顯示結果的視窗
    tracking_window_name = "Optimized Long-Term Tracking (Press 'q' to exit)"
    cv2.namedWindow(tracking_window_name, cv2.WINDOW_NORMAL)
    print("\n模型載入完成，開始處理影片幀...")

    # 將影片指針重設回我們想要開始的確切位置，以防讀取第一幀時影響了它
    if start_ms > 0:
        cap.set(cv2.CAP_PROP_POS_MSEC, start_ms)

    # --- 主處理迴圈 ---
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: 
            print("影片處理完畢或無法讀取下一幀。")
            break
        frame_count += 1
        
        # --- 幀跳過邏輯 ---
        if frame_count % FRAME_SKIP_INTERVAL != 1 and FRAME_SKIP_INTERVAL > 1:
            # 即使跳過處理，也應該顯示畫面並更新舊的標註框，讓畫面看起來流暢
            annotated_frame = frame.copy()
            for sid, track in active_tracks.items():
                x1, y1, x2, y2 = track['bbox']
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), FIXED_BOX_COLOR_BGR, BOX_THICKNESS)
                label = f"ID:{sid}"; cv2.putText(annotated_frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE_LABEL, FIXED_BOX_COLOR_BGR, FONT_THICKNESS_LABEL)
            if roi_defined_manual: cv2.polylines(annotated_frame, [np.array(roi_points_manual, dtype=np.int32)], True, ROI_POLYGON_COLOR, 2)
            cv2.imshow(tracking_window_name, annotated_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'): break
            continue

        # --- 完整的偵測與追蹤流程 ---

        # 1. 偵測 (在 ROI 內進行)
        input_frame_for_yolo = cv2.bitwise_and(frame, frame, mask=roi_mask_manual) if roi_defined_manual else frame
        results = yolo_model.predict(input_frame_for_yolo, classes=[PERSON_CLASS_ID], conf=YOLO_CONFIDENCE_THRESHOLD, iou=NMS_IOU_THRESHOLD, imgsz=TARGET_INFERENCE_SIZE, verbose=False)
        current_detections_bboxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        
        # (為了簡潔，這裡省略了您原有的追蹤匹配等程式碼，它們保持不變)
        # 2. 追蹤與匹配邏輯 (保持您的長期優化版本)
        active_ids_list = list(active_tracks.keys())
        unmatched_active_indices = set(range(len(active_ids_list)))
        unmatched_detection_indices = set(range(len(current_detections_bboxes)))
        if len(active_ids_list) > 0 and len(current_detections_bboxes) > 0:
            active_bboxes = np.array([t['bbox'] for t in active_tracks.values()])
            iou_cost = 1 - np.array([[calculate_iou(b1, b2) for b2 in current_detections_bboxes] for b1 in active_bboxes])
            row_ind, col_ind = linear_sum_assignment(iou_cost)
            for r, c in zip(row_ind, col_ind):
                if iou_cost[r, c] < (1 - HIGH_CONF_IOU_THRESHOLD):
                    stable_id = active_ids_list[r]
                    active_tracks[stable_id].update({'bbox': current_detections_bboxes[c], 'last_seen': frame_count})
                    unmatched_active_indices.discard(r)
                    unmatched_detection_indices.discard(c)
        det_indices_for_reid = list(unmatched_detection_indices)
        features_for_reid, valid_feature_indices = np.array([]), []
        if det_indices_for_reid:
            bboxes_for_reid = current_detections_bboxes[det_indices_for_reid]
            features_for_reid, valid_feature_indices = extract_features_batch(frame, bboxes_for_reid, reid_model, reid_processor, device)
        unmatched_track_ids = [active_ids_list[i] for i in unmatched_active_indices] + list(lost_tracks.keys())
        if unmatched_track_ids and features_for_reid.shape[0] > 0:
            cost_matrix = np.full((len(unmatched_track_ids), len(features_for_reid)), 1.0)
            for i, sid in enumerate(unmatched_track_ids):
                track = active_tracks.get(sid) or lost_tracks.get(sid)
                is_lost = sid in lost_tracks
                current_feature_thresh = LOST_TRACK_FEATURE_THRESHOLD if is_lost else FEATURE_MATCHING_THRESHOLD
                current_iou_thresh = LOST_TRACK_IOU_THRESHOLD if is_lost else IOU_MATCHING_THRESHOLD
                for j in range(features_for_reid.shape[0]):
                    original_det_idx = det_indices_for_reid[valid_feature_indices[j]]
                    iou = calculate_iou(track['bbox'], current_detections_bboxes[original_det_idx])
                    if iou < current_iou_thresh: continue
                    sim_ema = np.dot(track['ema_feature'], features_for_reid[j])
                    sim_initial = np.dot(track['initial_feature'], features_for_reid[j])
                    combined_sim = (1 - INITIAL_FEATURE_WEIGHT) * sim_ema + INITIAL_FEATURE_WEIGHT * sim_initial
                    if combined_sim > current_feature_thresh and combined_sim > ABSOLUTE_FEATURE_THRESHOLD:
                        cost_matrix[i, j] = LAMBDA_WEIGHT * (1 - iou) + (1 - LAMBDA_WEIGHT) * (1 - combined_sim)
            row_ind, col_ind = linear_sum_assignment(cost_matrix)
            for r, c in zip(row_ind, col_ind):
                if cost_matrix[r, c] < 1:
                    sid = unmatched_track_ids[r]
                    original_det_idx = det_indices_for_reid[valid_feature_indices[c]]
                    old_bbox = (active_tracks.get(sid) or lost_tracks.get(sid))['bbox']
                    old_center = ((old_bbox[0] + old_bbox[2]) / 2, (old_bbox[1] + old_bbox[3]) / 2)
                    was_lost = sid in lost_tracks
                    if was_lost:
                        active_tracks[sid] = lost_tracks.pop(sid)
                        active_tracks[sid]['reid_confirmed_frames'] = 0 
                    track = active_tracks[sid]
                    new_bbox = current_detections_bboxes[original_det_idx]
                    track.update({'bbox': new_bbox, 'last_seen': frame_count})
                    new_center = ((new_bbox[0] + new_bbox[2]) / 2, (new_bbox[1] + new_bbox[3]) / 2)
                    movement = np.linalg.norm(np.array(old_center) - np.array(new_center))
                    if track.get('reid_confirmed_frames', REID_CONFIRMATION_FRAMES) < REID_CONFIRMATION_FRAMES:
                        track['reid_confirmed_frames'] += 1
                    elif movement > STATIC_MOVEMENT_THRESHOLD:
                        new_feature = features_for_reid[c]
                        track['ema_feature'] = (1 - EMA_ALPHA) * new_feature + EMA_ALPHA * track['ema_feature']
                    if not was_lost:
                        unmatched_active_indices.discard(active_ids_list.index(sid))
                    unmatched_detection_indices.discard(original_det_idx)
        # 3. 軌跡狀態管理
        for idx in unmatched_active_indices:
            sid = active_ids_list[idx]
            if sid in active_tracks: lost_tracks[sid] = active_tracks.pop(sid)
        for sid in list(lost_tracks.keys()):
            if frame_count - lost_tracks[sid]['last_seen'] > MAX_LOST_FRAMES:
                available_ids.add(sid); del lost_tracks[sid]
        for det_idx in unmatched_detection_indices:
            if available_ids:
                new_id = min(available_ids)
                available_ids.remove(new_id)
                bbox_for_new = np.array([current_detections_bboxes[det_idx]])
                feature_for_new, _ = extract_features_batch(frame, bbox_for_new, reid_model, reid_processor, device)
                if feature_for_new.shape[0] > 0:
                    active_tracks[new_id] = {'bbox': bbox_for_new[0], 'ema_feature': feature_for_new[0], 'initial_feature': feature_for_new[0].copy(), 'last_seen': frame_count}

        # 4. 繪圖與存檔
        annotated_frame = frame.copy()
        for sid, track in active_tracks.items():
            x1, y1, x2, y2 = track['bbox']
            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), FIXED_BOX_COLOR_BGR, BOX_THICKNESS)
            label = f"ID:{sid}"; cv2.putText(annotated_frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE_LABEL, FIXED_BOX_COLOR_BGR, FONT_THICKNESS_LABEL)
            
            # --- 裁切與儲存邏輯 ---
            if ENABLE_CROPPING and (frame_count - last_crop_frame.get(sid, -crop_frame_interval)) >= crop_frame_interval:
                last_crop_frame[sid] = frame_count
                time_str = format_time_from_ms(cap.get(cv2.CAP_PROP_POS_MSEC))
                id_folder_path = os.path.join(OUTPUT_CROP_DIR, f"ID_{sid}"); os.makedirs(id_folder_path, exist_ok=True)
                
                cropped_image = frame[max(0, y1):min(frame.shape[0], y2), max(0, x1):min(frame.shape[1], x2)]
                
                if cropped_image.size > 0:
                    if use_sr:
                        try:
                            upscaled_image = sr.upsample(cropped_image)
                            cv2.imwrite(os.path.join(id_folder_path, f"{time_str}.jpg"), upscaled_image)
                        except Exception as e:
                            # 如果放大失敗，則儲存原圖
                            print(f"警告：ID {sid} 的影像 AI 放大失敗 ({e})，儲存原圖。")
                            cv2.imwrite(os.path.join(id_folder_path, f"{time_str}_orig.jpg"), cropped_image)
                    else:
                        cv2.imwrite(os.path.join(id_folder_path, f"{time_str}.jpg"), cropped_image)

        # --- 班級快照邏輯 ---
        if ENABLE_CLASS_SNAPSHOT and (frame_count - last_snapshot_frame) >= crop_frame_interval:
            last_snapshot_frame = frame_count
            time_str = format_time_from_ms(cap.get(cv2.CAP_PROP_POS_MSEC))
            image_to_save = annotated_frame if SAVE_ANNOTATED_SNAPSHOT else frame
            cv2.imwrite(os.path.join(OUTPUT_SNAPSHOT_DIR, f"snapshot_{time_str}.jpg"), image_to_save)
        
        if roi_defined_manual: cv2.polylines(annotated_frame, [np.array(roi_points_manual, dtype=np.int32)], True, ROI_POLYGON_COLOR, 2)
        cv2.imshow(tracking_window_name, annotated_frame)
        
        # 按 'q' 鍵退出迴圈
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("使用者手動停止處理。")
            break

    # --- 清理資源 ---
    cap.release()
    cv2.destroyAllWindows()
    print("處理完成，資源已釋放。")

if __name__ == "__main__":
    main()

# API分析

### 行為分析

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import base64
import json
import datetime
import time
from collections import Counter, defaultdict
from openai import OpenAI, APIError, RateLimitError, AuthenticationError
from PIL import Image, UnidentifiedImageError
import io
from tqdm import tqdm
import math
from dotenv import load_dotenv 
from concurrent.futures import ThreadPoolExecutor, as_completed
import bisect

# --- Configuration ---


# 班級整體照片資料夾路徑 
CLASSROOM_IMAGES_FOLDER = r'C:\Users\User\Desktop\test\student_full_classroom\0713_english_class'

# 【新】老師視角照片資料夾路徑 (從學生視角拍攝老師在黑板前的照片) - 可選
TEACHER_POSITION_JSON = r'C:\Users\User\Desktop\test\teacher_position\0706test.json'

# 【新】時間軸對齊設定 (格式: "HH:MM:SS")
CLASSROOM_START_TIME_OFFSET = "00:04:15"  # 班級照片的起始時間，設定為空字串 "" 表示不篩選
TEACHER_JSON_START_TIME_OFFSET = "00:04:15" # 老師位置資料的起始時間，設定為空字串 "" 表示不篩選

# ... (您原有的其他設定) ...

JSON_OUTPUT_FOLDER = "SynologyDrive\json_behavior"


JSON_FILENAME_TEMPLATE = "student_{student_id}_behavior_report_{timestamp}.json"
API_RETRY_DELAY_SECONDS = 10
MODEL_NAME_VISION = "gpt-4.1" # 推薦使用最新多模態模型，如 gpt-4o
MODEL_NAME_TEXT = "gpt-4.1"   # 推薦使用最新文本模型，如 gpt-4o 或 gpt-4-turbo
MAX_TOKENS_VISION_COMPLETION = 2500 # 序列分析可能需要更多 token
MAX_TOKENS_SUMMARY_COMPLETION = 1000 # 個性化總結的 token
BEHAVIOR_CONFIDENCE_THRESHOLD_FILTER = 0.9 # 行為信度過濾閾值 (例如 70%)
IMAGES_PER_API_CALL = 10 # 一次API調用處理的圖片數量
BEHAVIOR_SIMILARITY_CONFIDENCE_THRESHOLD = 0.15 # 過濾相似連續行為的信度差異閾值

# --- 標準行為分類與定義 (核心辭典，來自圖一) ---
STANDARD_BEHAVIOR_CATEGORIES = {
    "視線": [
        {"label": "目視教師", "definition": "學生的面部主要朝向教室前方教師區域。"},
        {"label": "目視黑板", "definition": "學生的面部主要朝向教室前方黑板區域。"},
        {"label": "目視書本/筆記","definition": "學生的視線朝向自己的桌面、書本或筆記本。注意：只要學生頭部低垂且沒有進行其他可識別的非學習行為（如玩弄物品、趴睡），就應優先歸類於此。"},
        {"label": "目視同學", "definition": "學生的面部明確轉向另一位或多位同學。"},
        {"label": "目視他處", "definition": "學生的面部朝向非前方、非桌面/教材、非同學的方向。"}
    ],
    "肢體(手部)": [
        {"label": "做筆記", "definition": "在『目視書本/筆記』的基礎上，學生手持筆，並在紙張上進行明確的書寫、繪圖動作。此標籤優先級高於單純的『目視書本/筆記』。"},
        {"label": "翻閱書本", "definition": "學生手部正在主動翻閱、移動書本。"},
        {"label": "玩弄物品", "definition": "學生手部在進行與當前學習任務無關的重複性、無目的性的小動作。注意：喝水、整理書包不屬於玩弄物品。"},
    ],
    "身體姿態": [
        {"label": "坐姿直立", "definition": "學生上半身基本保持直立，或略微前傾。"},
        {"label": "身體前傾", "definition": "學生上半身明顯向前傾斜，靠近桌面或前方。"},
        {"label": "身體後靠", "definition": "學生背部倚靠在椅背上，可能呈現較放鬆的姿態。"},
        {"label": "低頭(非學習)", "definition": "學生頭部明顯低垂，但視線並非看向書本，且無任何手部學習動作，可能在發呆或打瞌睡。**僅在能明確排除看書和筆記時使用。**"},
        {"label": "趴睡", "definition": "學生將頭部枕於手臂或桌面，或以其他方式呈現明顯的睡眠或休息狀態。"}
    ],
    "互動": [
        {"label": "舉手", "definition": "學生舉起一隻手（通常高於頭部）。"}
    ],
    "其他狀態": [
        {"label": "喝水/飲食", "definition": "學生正在使用杯子、水壺等容器飲用液體，或食用固體食物。"},
        {"label": "整理個人物品", "definition": "學生正在整理書包、桌面抽屜、衣物等與當前直接學習任務無關的個人物品。"},
        {"label": "無明顯特定行為", "definition": "學生處於一種相對靜止、無上述明確行為的狀態，視線和手部無特定指向或動作。"},
        {"label": "被遮擋/無法判斷", "definition": "因遮擋、模糊或角度問題，無法清晰識別學生的主要行為。"}
    ]
}
# 自動從新結構生成有效的標籤列表
VALID_BEHAVIOR_LABELS = [item['label'] for category in STANDARD_BEHAVIOR_CATEGORIES.values() for item in category]

# AI返回標籤到標準標籤的映射規則 (與新標籤對齊)
BEHAVIOR_MAPPING_RULES = {
    "目視桌面/教材": "目視書本", "目視桌面": "目視書本", "看書": "目視書本",
    "書寫/做筆記": "筆記", "動手操作-書寫/做筆記": "筆記",
    "視覺專注-閱讀書本/講義": "目視書本",
    "視覺專注-看老師/黑板方向": "目視教師", "目視黑板/老師": "目視教師",
    "看老師": "目視教師", "看黑板": "目視黑板",
    "非任務相關動作-玩弄物品(筆等)": "玩弄物品", "非任務相關動作-觸摸臉部/頭髮": "玩弄物品",
    "非任務相關動作-整理物品": "整理個人物品",
    "社交互動-與同學互動": "目視同學",
    "趴睡/休息": "趴睡",
    "低頭/伏案(非睡)": "低頭",
}

BEHAVIOR_VALENCE_MAP = {
    # 正向 (Active Engagement): 明顯投入當前學習任務的行為
    "正向": [
        "做筆記",         # 最高級別的投入指標，結合了視覺和動手操作
        "舉手",             # 主動參與互動
        "目視教師",         # 專注於教學者
        "目視黑板",         # 專注於教學內容
        "目視書本/筆記",    # 可能是閱讀，也可能是發呆，單從視線難以100%確定
        "翻閱書本",         # 可能是找資料（正向），也可能是隨意翻動（中性）
        "身體前傾"
    ],
    # 負向 (Clear Disengagement): 明顯脫離學習任務、可能影響自己或他人的行為
    "負向": [
        "趴睡",             # 完全脫離學習狀態
        "玩弄物品",         # 明顯的非任務行為，分散注意力
        "目視他處",          # 視線脫離學習區域（老師、黑板、課本）
        "目視同學"         
    ],
    # 中性 (Neutral / Context-Dependent): 基本姿態、生理需求或情境依賴的行為
    "中性": [
        "坐姿直立",         # 標準姿態，無法直接判斷投入度
        "身體後靠",         # 可能是放鬆，也可能是疲倦，無法直接判斷
        "低頭(非學習)",     # 雖然傾向負向，但可能是短暫疲勞，歸為中性以避免過度解讀
        "喝水/飲食",        # 基本生理需求，除非頻繁發生，否則不應標為負向
        "整理個人物品",     
        "無明顯特定行為",   # 預設的中性狀態
        "被遮擋/無法判斷" 
    ]
}


LABEL_TO_VALENCE = {
    label: valence 
    for valence, labels in BEHAVIOR_VALENCE_MAP.items() 
    for label in labels
}

# ---------------------------
#  API 金鑰配置
# ---------------------------
load_dotenv() # 加載 .env 文件中的環境變數
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("警告：未找到環境變數 'OPENAI_API_KEY' (已嘗試從 .env 加載)。")
    api_key = input("請貼上你的 OpenAI API 金鑰 (sk-...) 並按 Enter：").strip()
    if not api_key: print("錯誤：未提供 API 金鑰。程式即將終止。"); exit()
    else: print("已使用手動輸入的 API 金鑰。")

try:
    client = OpenAI(api_key=api_key)
    client.models.list() # 嘗試調用一個簡單的API來驗證金鑰
    print("OpenAI client 初始化並驗證成功。")
except AuthenticationError: print("錯誤：OpenAI API 金鑰無效或認證失敗。"); exit()
except RateLimitError: print("錯誤：API金鑰已達到速率限制。請稍後再試或檢查您的配額。"); exit()
except APIError as e: print(f"初始化 OpenAI Client 時發生 API 錯誤: {e}"); exit()
except Exception as e: print(f"初始化 OpenAI Client 時發生未知錯誤: {e}"); exit()

# ---------------------------
# 輔助函數
# ---------------------------
def parse_time_offset(time_str):
    """將 "HH:MM:SS" 格式的字串轉換為 timedelta 物件"""
    if not time_str or not isinstance(time_str, str):
        return None
    try:
        h, m, s = map(int, time_str.split(':'))
        return datetime.timedelta(hours=h, minutes=m, seconds=s)
    except (ValueError, TypeError):
        print(f"警告：時間偏移量 '{time_str}' 格式不正確，應為 'HH:MM:SS'。將忽略此設定。")
        return None

def get_valid_input(prompt_message):
    while True:
        user_input = input(prompt_message).strip()
        if user_input: return user_input
        print("錯誤：輸入不能為空，請重新輸入。")

def get_valid_folder_path(prompt_message, is_optional=False): # 新增 is_optional 參數
    while True:
        folder_path = input(prompt_message).strip().strip('"')
        if not folder_path and is_optional:
            return None # 如果是可選的且用戶未輸入，返回 None
        if os.path.isdir(folder_path):
            return folder_path
        print(f"錯誤：路徑 '{folder_path}' 不是一個有效的資料夾，請重新輸入。")

def get_timestamp_from_filename(filename):
    """
    【升級版】從檔名解析時間戳。
    能同時處理兩種格式:
    1. HH-MM-SS-ms (例如 'snapshot_10-20-30-123.jpg')
    2. ...HHhMMmSSs... (例如 'blackboard_000h00m50s_f0001501.png')
    """
    # 模式一：優先嘗試匹配 HH-MM-SS-ms 格式
    match_format1 = re.search(r'(\d{2})-(\d{2})-(\d{2})-(\d{3})', filename)
    if match_format1:
        try:
            h, m, s, ms = map(int, match_format1.groups())
            return datetime.timedelta(hours=h, minutes=m, seconds=s, milliseconds=ms)
        except (ValueError, IndexError):
            # 如果解析失敗，就繼續嘗試下一個格式，而不是直接返回
            pass 

    # 模式二：如果模式一失敗，則嘗試匹配 ...h...m...s 格式
    match_format2 = re.search(r'(\d+)h(\d{2})m(\d{2})s', filename)
    if match_format2:
        try:
            # 注意：這種格式沒有毫秒，我們將其設為 0
            h, m, s = map(int, match_format2.groups())
            return datetime.timedelta(hours=h, minutes=m, seconds=s, milliseconds=0)
        except (ValueError, IndexError):
            # 如果解析失敗，同樣略過
            pass

    # 如果兩種格式都沒匹配成功，最終返回 None
    return None

def encode_image_to_base64(image_path, max_size_kb=512, target_quality=75):
    try:
        with Image.open(image_path) as img: img.verify()
        with Image.open(image_path) as img:
            if img.mode == 'RGBA' or img.mode == 'P': img = img.convert('RGB')
            current_quality = target_quality
            output_buffer = io.BytesIO()
            temp_img = img.copy()
            temp_img.save(output_buffer, format="JPEG", quality=current_quality)
            current_size_kb = output_buffer.tell() / 1024

            if current_size_kb > max_size_kb:
                scale_factor = math.sqrt(max_size_kb / current_size_kb)
                new_width = int(temp_img.width * scale_factor * 0.9)
                new_height = int(temp_img.height * scale_factor * 0.9)
                if new_width >= 50 and new_height >= 50: # 最小尺寸限制
                    print(f"    圖片 {os.path.basename(image_path)} ({current_size_kb:.1f} KB) 過大，縮放並調整質量...")
                    temp_img = temp_img.resize((new_width, new_height), Image.Resampling.LANCZOS)
                    output_buffer = io.BytesIO()
                    temp_img.save(output_buffer, format="JPEG", quality=max(current_quality - 15, 40)) # 質量可以降更多
                else:
                    print(f"    警告: 圖片 {os.path.basename(image_path)} 縮放後過小，可能影響質量。使用較低質量。")
                    output_buffer = io.BytesIO()
                    img.save(output_buffer, format="JPEG", quality=max(current_quality // 2, 30) )


            output_buffer.seek(0)
            binary_data = output_buffer.getvalue()
            base64_encoded_data = base64.b64encode(binary_data)
            return f"data:image/jpeg;base64,{base64_encoded_data.decode('utf-8')}"
    except Exception as e: print(f"錯誤：處理圖片 '{image_path}': {e}"); return None

def get_behavior_sequence_analysis_system_prompt(student_position, has_teacher_context):
    """
    【v5.1 - 最終精簡版】為序列分析任務生成動態的系統提示 (System Prompt)。
    此版本移除了在新流程下已不再必要的 head_pose_rules 表格，使指令更清晰、更高效。
    """
    # --- 1. 動態生成行為分類表 (此部分邏輯不變) ---
    behavior_table_for_prompt = "\n\n**【學習行為分類與定義表】**\n你 **必須** 且 **只能** 從以下列表的「標籤」中選擇行為進行標註。**絕對不允許** 創造、組合或使用任何不在列表中的標籤。\n"
    for main_cat, sub_cats in STANDARD_BEHAVIOR_CATEGORIES.items():
        behavior_table_for_prompt += f"\n**{main_cat}**\n"
        for item in sub_cats:
            behavior_table_for_prompt += f"*   **{item['label']}**: {item['definition']}\n"
    behavior_table_for_prompt += "\n*如果行為因任何原因無法清晰對應到上述任何一項，請 **必須** 使用 **'被遮擋/無法判斷'** 標籤。*\n"

    # --- 2. 根據是否有老師情境，動態生成指令片段 ---
    teacher_context_header = ""
    teacher_context_workflow = f"""*   **學生座位**: 該學生的固定座位是 **'{student_position}'**。"""
    teacher_context_cross_validation = "則根據學生頭部朝向和常理進行推斷（例如，直視前方是「目視黑板」，明顯轉向一側可能是「目視同學」）。"
    
    if has_teacher_context:
        teacher_context_header = """2.  **【老師位置文字情境】(最高優先級情境)**: 你會收到一段關於老師位置的文字描述（例如：「老師在左側」）。這是判斷「目視教師」的**決定性依據**。"""
        
        teacher_context_workflow = f"""*   **老師位置**: 根據使用者提供的【老師位置文字情境】，你已知曉老師在教室前方的「左側」、「中間」或「右側」。
*   **學生座位**: 該學生的固定座位是 **'{student_position}'**。"""
        
        teacher_context_cross_validation = """**根據【老師位置文字情境】**: 將學生的頭部朝向與文字描述的老師**位置**進行比對。
            *   **若匹配**: 標記為 **"目視教師"**。（例如：學生頭部朝向右側，且情境文字是「右側」，則匹配）
            *   **若不匹配**: 標記為 **"目視黑板"** (如果視線是朝向前方區域) 或 **"目視他處"**。"""

    # --- 3. 組合所有指令到一個連貫的 Prompt 中 ---
    return f"""
「你是一名精通多視角影像分析的課堂行為專家，如同偵探一樣細緻。」

**【你的核心任務：多源交叉驗證】**

你將收到以下三種資料，請將它們視為一個完整的案件檔案來分析：
1.  **【學生特寫照片序列】**: 這是近距離觀察學生的主要視角。
{teacher_context_header}
3.  **【班級整體情境照片】**: 這是上帝視角，提供最全面的情境，也是在特寫照片不清晰時的**權威備援**。

**【分析流程與決策樹】**

對於序列中的**每一組**對應時間的照片，請嚴格遵循以下決策流程來判斷行為：

**步驟一：主要分析與初步判斷**
1.  **分析【學生特寫照片序列】**:
    *   首先，仔細觀察特寫照片。如果照片清晰且無遮擋，請根據它進行初步的行為判斷。
    *   **如果特寫照片中有多人**，請專注於與前幾張照片中學生特徵最一致的那一位。

**步驟二：交叉驗證與最終標註 (最關鍵)**
2.  **使用【班級整體情境照片】進行驗證或補充**:
    *   **定位學生**: 在班級照片中，根據使用者提供的座位提示 **'{student_position}'**，精確找到目標學生。
    *   **比對行為**:
        *   **情況 A (特寫清晰):** 將你在特寫照片中的初步判斷，與班級照片中該學生的行為進行比對。若兩者一致，則確認該行為標籤。若不一致，**請以班級整體照片中的行為為準**，因為它提供了更完整的上下文。
        *   **情況 B (特寫被遮擋、模糊或無法判斷):** 在這種情況下，**【班級整體情境照片】將成為你判斷的主要依據**。直接分析你在班級照片中定位到的學生的行為，並以此進行標註。

**步驟三：判斷視線方向 (目視教師 vs. 目視黑板)**
*   在完成上述步驟確定學生視線朝向前方後，再結合【老師位置文字情境】進行最終判斷。
*   {teacher_context_cross_validation}

**步驟四：判斷其他細節行為**
*   根據你最終確認的學生姿態，參考下方的【學習行為分類與定義表】來標註如 "做筆記"、"玩弄物品"、"趴睡" 等具體行為。

{behavior_table_for_prompt}

**【輸出 JSON 格式要求與範例】**
*   你的回答**必須**是一個結構完整的 JSON 物件。
*   在 `per_image_highlights` 的 `context_description` 欄位中，**必須**說明你是主要依據特寫照片還是班級照片得出的結論。

**【輸出 JSON 格式範例】**
```json
{{
  "sequence_analysis_confidence": 0.95,
  "sequence_summary": "學生在大部分時間能保持專注，但在中段部分因特寫鏡頭被遮擋，通過班級照片觀察到其有短暫的低頭休息行為。",
  "per_image_highlights": [
    {{
      "image_index_in_sequence": 0,
      "context_description": "主要依據：特寫照片清晰。學生頭部朝向與老師位置『右側』匹配。",
      "behavior_category": "目視教師",
      "confidence": 0.98,
      "description": "特寫照片和班級照片均顯示學生頭部明確轉向右側。"
    }},
    {{
      "image_index_in_sequence": 1,
      "context_description": "主要依據：特寫照片被遮擋，改用班級照片分析。學生位於第四排左側。",
      "behavior_category": "低頭(非學習)",
      "confidence": 0.90,
      "description": "雖然特寫照片無法判斷，但在班級照片中清晰可見該學生頭部低垂，無手部動作。"
    }}
  ]
}}
```"""

def normalize_behavior_label(label_from_ai):
    if not isinstance(label_from_ai, str): return "未知標籤"
    label_from_ai_trimmed = label_from_ai.strip()
    if label_from_ai_trimmed in BEHAVIOR_MAPPING_RULES: return BEHAVIOR_MAPPING_RULES[label_from_ai_trimmed]
    if label_from_ai_trimmed in VALID_BEHAVIOR_LABELS: return label_from_ai_trimmed
    # 更寬鬆的匹配 (謹慎，可能需要更多規則)
    for valid_label in VALID_BEHAVIOR_LABELS:
        if valid_label in label_from_ai_trimmed or label_from_ai_trimmed in valid_label : # 雙向包含檢查
            # 特殊處理組合標籤，例如 "筆記/翻閱書本" -> "筆記"
            if "/" in label_from_ai_trimmed:
                parts = label_from_ai_trimmed.split('/')
                for part in parts:
                    if part.strip() in VALID_BEHAVIOR_LABELS:
                        return part.strip() # 取第一個匹配的標準標籤
            return valid_label # 如果是部分包含標準標籤
    print(f"    歸一化警告：無法將AI標籤 '{label_from_ai_trimmed}' 映射到標準分類，將使用原始標籤（可能導致統計問題）。")
    return label_from_ai_trimmed

def default_error_result_structure():
    return { "error": "分析失敗或無有效數據", "sequence_analysis_confidence": 0.0, "sequence_summary": "未能生成序列總結。", "dominant_sustained_behaviors": [], "significant_behavior_shifts": [], "per_image_highlights": [], "general_sequence_atmosphere_hint": "未知" }

def analyze_student_behavior_from_images_sequence(student_image_paths, teacher_position_text, classroom_view_image_path, image_filenames_batch, openai_client, student_id, student_position):
    if not student_image_paths:
        return {"error": "沒有提供學生圖片路徑", "analysis": default_error_result_structure()}
    
    user_message_content = []
    
    # 1. 編碼學生個人圖片序列 (主要分析對象)
    encoded_student_images = []
    for img_path in student_image_paths:
        b64_img = encode_image_to_base64(img_path)
        if b64_img:
            encoded_student_images.append({"type": "image_url", "image_url": {"url": b64_img, "detail": "low"}})

    if not encoded_student_images:
        return {"error": "所有學生圖片編碼失敗", "analysis": default_error_result_structure()}
        
    # 2. 【已移除】不再編碼老師視角照片
    
    # 3. 編碼班級整體照片 (用於定位和周圍情境)
    encoded_classroom_view_image = None
    if classroom_view_image_path:
        b64_img = encode_image_to_base64(classroom_view_image_path)
        if b64_img:
            encoded_classroom_view_image = {"type": "image_url", "image_url": {"url": b64_img, "detail": "auto"}}

    # 4. 按照新情境組合請求體
    user_message_content.append({"type": "text", "text": f"請根據系統提示中的偵探任務，分析學生「{student_id}」的行為。學生照片序列的文件名（供您參考）為: {', '.join(image_filenames_batch)}。"})
    
    # 【新】直接將老師位置文字加入 Prompt
    teacher_context_prompt = f"【老師位置情境】根據預先分析，在此時間段，老師的位置在教室前方的「{teacher_position_text}」。請以此作為判斷『目視教師』的核心依據。"
    user_message_content.append({"type": "text", "text": teacher_context_prompt})
        
    if encoded_classroom_view_image:
        user_message_content.append({"type": "text", "text": "【班級整體照片】(用於定位學生和觀察整體氛圍)"})
        user_message_content.append(encoded_classroom_view_image)
    
    user_message_content.append({"type": "text", "text": "【學生個人照片序列】(主要分析對象)"})
    user_message_content.extend(encoded_student_images)

    # 【新】修改 System Prompt，移除對老師照片的引用
    system_prompt_content = get_behavior_sequence_analysis_system_prompt(student_position, bool(teacher_position_text and teacher_position_text != "未知"))

    # --- 後續的 API 呼叫和錯誤處理邏輯保持不變 ---
    retry_attempts = 2
    for attempt in range(retry_attempts + 1):
        try:
            has_teacher_context = bool(teacher_position_text and teacher_position_text != "未知")
            has_classroom_image = bool(encoded_classroom_view_image)
            print(f"  正在向 {MODEL_NAME_VISION} 發送請求 (學生: {len(encoded_student_images)}, 老師位置數據: {'有' if has_teacher_context else '無'}, 班級整體照片: {'有' if has_classroom_image else '無'})...")
            
            response = openai_client.chat.completions.create(
                model=MODEL_NAME_VISION, response_format={"type": "json_object"},
                messages=[{"role": "system", "content": system_prompt_content}, {"role": "user", "content": user_message_content}],
                max_tokens=MAX_TOKENS_VISION_COMPLETION, temperature=0.05
            )
            raw_content = response.choices[0].message.content
            analysis_result = json.loads(raw_content)
            
            if "per_image_highlights" in analysis_result and isinstance(analysis_result["per_image_highlights"], list):
                for hl in analysis_result["per_image_highlights"]:
                    if isinstance(hl, dict) and "behavior_category" in hl:
                        hl["behavior_category"] = normalize_behavior_label(hl.get("behavior_category"))
            return analysis_result

        except json.JSONDecodeError: print(f"    錯誤：無法解析JSON (第 {attempt+1} 次嘗試)。回應: {raw_content[:500]}...");
        except RateLimitError: print(f"  警告：API速率限制，等待 {API_RETRY_DELAY_SECONDS}s 後重試..."); time.sleep(API_RETRY_DELAY_SECONDS)
        except APIError as e: print(f"  錯誤：API錯誤 (第 {attempt+1} 次嘗試): {e}"); time.sleep(5)
        except Exception as e: print(f"  錯誤：未知錯誤 (第 {attempt+1} 次嘗試): {e}"); time.sleep(5)
    
    print(f"  錯誤：圖片序列分析在多次重試後失敗。")
    return default_error_result_structure()

def generate_personalized_summary_notes(student_id, overall_stats, non_task_highlights, openai_client):
    if not openai_client: return {"error": "OpenAI client not available"}
    stats_summary_for_ai = "\n".join([f"- {s['behavior_category']}: {s['percentage']:.1f}% ({s['count']}次)" for s in overall_stats[:7]]) # 確保百分比格式
    non_task_prompt_part = "該生在本堂課中，未觀察到明顯或頻繁的非任務相關行為。"
    if non_task_highlights:
        highlights_str = "\n".join([f"  - 圖 '{img_fn}' (~{ts}): '{beh}' (描述: {desc})" for img_fn, ts, beh, desc in non_task_highlights[:3]])
        if highlights_str: non_task_prompt_part = f"在本堂課中，觀察到一些非任務相關行為，例如：\n{highlights_str}\n這可能影響了學習專注度。"
    prompt_for_summary = f"""
    「你是一位專業且富有同理心的學習行為教練。你的目標不是批評，而是透過客觀數據，引導學生發現自己的學習模式，並提供能立即實踐的策略，以激發他們『自我反思』的動力。」

    **任務：** 為學生「{student_id}」撰寫一份「AI學習夥伴的觀察與建議」。

    **學生的課堂行為數據：**
    *   **主要行為分佈:**
    {stats_summary_for_ai}
    *   **值得注意的行為片段:**
    {non_task_prompt_part}

    **撰寫指引與 JSON 格式要求：**
    請嚴格遵循以下指引，產生一個結構完整的 JSON 物件。

    *   **`greeting` (問候語):**
        *   用親切、個人化的方式稱呼學生，例如：「嗨，{student_id} 同學，一起來看看這次課堂的學習足跡吧！」

    *   **`positive_feedback` (亮點觀察):**
        *   **必須**從數據中找出最值得肯定的行為（例如「目視教師」或「筆記」佔比最高），並給予具體、真誠的讚美。
        *   **範例**：「我發現你在這堂課有超過一半的時間都在『目視教師』，這代表你非常努力地跟上老師的節奏，非常棒！」

    *   **`observation_points_summary` (行為模式提醒):**
        *   客觀、中性地指出一個或兩個最主要的、可能影響學習的行為模式。避免使用負面詞彙。
        *   **範例**：「數據也顯示，大約有 15% 的時間出現了『玩弄物品』或『目視他處』的狀況，這些時刻可能讓我們不小心錯過了一些重點喔。」

    *   **`reflection_points` (反思引導提問):**
        *   **【此項最為關鍵】** 根據前面的觀察點，設計 2-3 個**開放式問題**，引導學生思考行為背後的原因，而不是直接給答案。
        *   **問題範例 1**：「我們可以一起回想看看，當出現『玩弄物品』的時候，通常是在課程的哪個階段呢？是覺得內容太簡單、太難，還是剛好有點疲倦了呢？」
        *   **問題範例 2**：「當視線看向其他地方時，是想到了什麼有趣的事，還是被教室裡的其他動靜吸引了呢？了解這些原因，能幫助我們找到最適合自己的專注方法。」

    *   **`suggestions` (可實踐的小建議):**
        *   提供 1-2 個**具體、微小、且容易執行**的行動建議。不要說「要專心」，而是給出方法。
        *   **建議範例 1**：「下次當你發現自己開始無意識地轉筆時，可以試著把它輕輕放下，然後做一個深呼吸，再重新將目光移回老師或課本上。」
        *   **建議範例 2**：「如果感覺到疲倦或分心，可以試試看『筆記專注法』：在筆記本上寫下老師說的任何一個關鍵字，這個小動作能幫助我們的大腦重新連線！」

    *   **`encouragement` (鼓勵與結語):**
        *   用一句溫暖、有力的話作結，強調這份報告是幫助他成長的工具。
        *   **範例**：「每一次的觀察都是為了讓我們更了解自己。相信你透過這些小小的調整，一定能發揮出自己最大的潛力，加油！」
    """
    # ... (API調用和結果解析邏輯與您上一版本相同，確保 try-except 完整) ...
    retry_attempts = 2
    for attempt in range(retry_attempts):
        try:
            print(f"  正在為學生 {student_id} 生成個性化總結...")
            response = client.chat.completions.create(
                model=MODEL_NAME_TEXT, response_format={"type": "json_object"},
                messages=[{"role": "system", "content": "你是一位富有同理心和洞察力的教育顧問。"},
                          {"role": "user", "content": prompt_for_summary}],
                max_tokens=MAX_TOKENS_SUMMARY_COMPLETION, temperature=0.7
            )
            summary_data = json.loads(response.choices[0].message.content)
            expected_keys = ["greeting", "positive_feedback", "observation_points_summary", "reflection_points", "suggestions", "encouragement"]
            if all(key in summary_data for key in expected_keys): print(f"  成功為學生 {student_id} 生成個性化總結。"); return summary_data
            else: print(f"    警告：個性化總結JSON缺少鍵。返回: {summary_data}"); return {key: summary_data.get(key, f"AI未能生成 ({key})") for key in expected_keys}
        except Exception as e: print(f"  生成個性化總結錯誤 ({attempt + 1}): {e}"); time.sleep(API_RETRY_DELAY_SECONDS if isinstance(e, RateLimitError) else 5)
        if attempt == retry_attempts - 1: print(f"  錯誤：無法為學生 {student_id} 生成個性化總結。"); return {"greeting": f"親愛的 {student_id},", "positive_feedback": "總結生成遇到問題。", "observation_points_summary": "請參考統計數據。", "reflection_points": "未能生成。", "suggestions": "請自行評估。", "encouragement": "加油！"}
    return {}

def find_closest_image_path(representative_timestamp, sorted_photo_list, max_time_diff_seconds=5):
    """
    一個可重用的輔助函數，用二分查找法在排序好的照片列表中找到時間最接近的照片路徑。
    """
    if not sorted_photo_list:
        return None

    all_timestamps = [item[0] for item in sorted_photo_list]
    # bisect_left 找到應該插入的位置
    insertion_point = bisect.bisect_left(all_timestamps, representative_timestamp)

    closest_path = None
    min_diff = datetime.timedelta.max

    # 只檢查插入點及其前後的幾個候選照片，效率極高
    # 檢查範圍設為 insertion_point-2 到 insertion_point+2 以增加容錯
    start_index = max(0, insertion_point - 2)
    end_index = min(len(sorted_photo_list), insertion_point + 2)

    for i in range(start_index, end_index):
        candidate_ts, candidate_path = sorted_photo_list[i]
        diff = abs(candidate_ts - representative_timestamp)
        if diff < min_diff:
            min_diff = diff
            closest_path = candidate_path

    # 只有在時間差在容許範圍內才返回路徑
    if closest_path and min_diff.total_seconds() <= max_time_diff_seconds:
        return closest_path
    
    return None

def find_closest_position(representative_timestamp, sorted_position_list, max_time_diff_seconds=10):
    """
    用二分查找法在排序好的位置列表中找到時間最接近的老師位置。
    """
    if not sorted_position_list:
        return "未知"

    all_timestamps = [item[0] for item in sorted_position_list]
    insertion_point = bisect.bisect_left(all_timestamps, representative_timestamp)

    closest_position = "未知"
    min_diff = datetime.timedelta.max

    start_index = max(0, insertion_point - 2)
    end_index = min(len(sorted_position_list), insertion_point + 2)

    for i in range(start_index, end_index):
        candidate_ts, candidate_pos = sorted_position_list[i]
        diff = abs(candidate_ts - representative_timestamp)
        if diff < min_diff:
            min_diff = diff
            closest_position = candidate_pos
    
    if min_diff.total_seconds() <= max_time_diff_seconds:
        return closest_position
    
    return "未知"

def process_single_batch(batch_idx, image_batch_info, teacher_positions_data, sorted_classroom_photos, client, student_id, student_position):
    """
    【升級版】處理單一圖片批次，使用老師位置文字數據。
    """
    batch_student_paths = [info["path"] for info in image_batch_info]
    batch_image_filenames = [info["filename"] for info in image_batch_info]
    
    representative_timestamp = image_batch_info[0]["timestamp_obj"]

    # --- 為當前批次查找匹配的「老師位置文字」和「班級整體照片」 ---
    teacher_position_text = find_closest_position(representative_timestamp, teacher_positions_data)
    classroom_view_path = find_closest_image_path(representative_timestamp, sorted_classroom_photos)

    # --- 呼叫【新版】分析函數 ---
    sequence_analysis_data = analyze_student_behavior_from_images_sequence(
        student_image_paths=batch_student_paths,
        teacher_position_text=teacher_position_text, # <-- 傳遞文字
        classroom_view_image_path=classroom_view_path,
        image_filenames_batch=batch_image_filenames,
        openai_client=client,
        student_id=student_id,
        student_position=student_position
    )

    return {
        "batch_index": batch_idx,
        "image_batch_info": image_batch_info,
        "matched_teacher_position_text": teacher_position_text, # <-- 修改回傳的鍵名
        "matched_classroom_view_image": os.path.basename(classroom_view_path) if classroom_view_path else None,
        "analysis": sequence_analysis_data
    }

def main():
    print("--- 學生課堂學習行為分析報告生成 (JSON) v4.3 - 最終修正版 ---")
    print(f"視覺模型: {MODEL_NAME_VISION}, 文本模型: {MODEL_NAME_TEXT}")
    print(f"圖片批次大小: {IMAGES_PER_API_CALL}, 行為信度閾值: {BEHAVIOR_CONFIDENCE_THRESHOLD_FILTER*100:.0f}%")
    print("-" * 40)

    # --- 使用者輸入部分 ---
    student_id = get_valid_input("請輸入學生姓名 (可中文): ")
    student_number = get_valid_input("請輸入學生座號 (例如: 1): ") 
    student_images_folder = get_valid_folder_path(f"請輸入 '{student_id}' 的個人影像資料夾: ")
    print("\n--- 請輸入課堂情境資訊 ---")
    student_position = get_valid_input("請輸入學生座位 (例如: '第3排中間', '第1排左側'): ")
    print("-" * 50)

    # --- 準備輸出資料夾 ---
    safe_student_id_for_folder = re.sub(r'[\\/*?:"<>|]',"", student_id.replace(" ", "_"))
    student_specific_folder_base = os.path.join(JSON_OUTPUT_FOLDER, safe_student_id_for_folder)
    os.makedirs(student_specific_folder_base, exist_ok=True)

    # --- 準備學生照片資料 ---
    image_files_with_timestamps = []
    valid_extensions = ('.png', '.jpg', '.jpeg', '.webp')
    print(f"正在掃描學生個人照片資料夾: {student_images_folder}...")
    for filename in os.listdir(student_images_folder):
        if filename.lower().endswith(valid_extensions):
            timestamp = get_timestamp_from_filename(filename)
            if timestamp:
                image_files_with_timestamps.append({
                    "path": os.path.join(student_images_folder, filename), "filename": filename,
                    "timestamp_obj": timestamp,
                    "timestamp_str": str(timestamp).split('.')[0]
                })
    image_files_with_timestamps.sort(key=lambda x: x["timestamp_obj"])
    if not image_files_with_timestamps: 
        print("錯誤：學生資料夾中未找到有效時間格式的圖片。")
        return
    print(f"成功找到 {len(image_files_with_timestamps)} 張學生個人照片。")

    # --- 定義一個可重用的函數來加載和排序照片 ---
    def load_teacher_positions(json_path, start_time_offset_str=""): # 新增 start_time_offset_str 參數
        if not json_path or not os.path.isfile(json_path):
            print(f"提示：未提供或找不到老師位置 JSON 檔案 ({json_path})。將不使用老師位置情境。")
            return []
        
        # 新增：解析時間偏移量
        start_offset = parse_time_offset(start_time_offset_str)
        
        print(f"正在讀取老師位置數據: {json_path}...")
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            position_map = []
            original_count = len(data) # 新增：記錄原始數量
            
            for item in data:
                try:
                    h, m, s = map(int, item['timestamp'].split(':'))
                    td = datetime.timedelta(hours=h, minutes=m, seconds=s)
                    
                    # 新增：時間過濾邏輯
                    if start_offset and td < start_offset:
                        continue # 如果時間早於設定的起始點，則跳過此筆資料
                        
                    position_map.append((td, item['position']))
                except (ValueError, KeyError):
                    continue
            
            position_map.sort(key=lambda x: x[0])

            # 新增：輸出過濾訊息
            if start_offset:
                print(f"成功加載老師位置數據。根據起始時間 '{start_time_offset_str}' 進行過濾，保留 {len(position_map)} 筆 (原 {original_count} 筆)。")
            else:
                print(f"成功加載並索引了 {len(position_map)} 筆老師位置數據。")
                
            return position_map
        except Exception as e:
            print(f"錯誤：讀取或解析老師位置 JSON 時發生問題: {e}")
            return []

    # --- 定義一個可重用的函數來加載和排序班級照片 ---
    def load_and_sort_photos(folder_path, photo_type_name, start_time_offset_str=""): # 新增 start_time_offset_str 參數
        if not folder_path or not os.path.isdir(folder_path):
            print(f"提示：未提供或找不到 {photo_type_name} 照片資料夾 ({folder_path})。")
            return []
        
        # 新增：解析時間偏移量
        start_offset = parse_time_offset(start_time_offset_str)
        
        print(f"正在掃描 {photo_type_name} 照片資料夾: {folder_path}...")
        photo_list = []
        original_count = 0 # 新增：記錄原始數量

        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                original_count += 1
                timestamp = get_timestamp_from_filename(filename)
                if timestamp:
                    # 新增：時間過濾邏輯
                    if start_offset and timestamp < start_offset:
                        continue # 如果時間早於設定的起始點，則跳過此張照片
                    
                    photo_list.append((timestamp, os.path.join(folder_path, filename)))
        
        if photo_list:
            sorted_photos = sorted(photo_list)
            # 新增：輸出過濾訊息
            if start_offset:
                print(f"成功加載 {photo_type_name} 照片。根據起始時間 '{start_time_offset_str}' 進行過濾，保留 {len(sorted_photos)} 張 (原 {original_count} 張有效格式照片)。")
            else:
                print(f"成功加載並索引了 {len(sorted_photos)} 張 {photo_type_name} 照片。")
            return sorted_photos
        else:
            print(f"警告：在 {photo_type_name} 照片資料夾 '{folder_path}' 中未找到符合條件的圖片。")
            return []

    # --- 分別加載兩種情境資料 ---
    teacher_positions_data = load_teacher_positions(TEACHER_POSITION_JSON, TEACHER_JSON_START_TIME_OFFSET)
    sorted_classroom_photos = load_and_sort_photos(CLASSROOM_IMAGES_FOLDER, "班級整體", CLASSROOM_START_TIME_OFFSET)

    # --- 圖片分批 ---
    image_batches = [image_files_with_timestamps[i:i + IMAGES_PER_API_CALL]
                     for i in range(0, len(image_files_with_timestamps), IMAGES_PER_API_CALL)]
    print(f"學生圖片將被分為 {len(image_batches)} 個批次進行分析。")

    # --- 開始平行處理 ---
    MAX_WORKERS = 8
    print(f"將使用最多 {MAX_WORKERS} 個執行緒進行平行分析...")

    all_results_from_threads = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_batch_idx = {
            executor.submit(
                process_single_batch,
                idx, batch_info,
                teacher_positions_data,
                sorted_classroom_photos,
                client, student_id, student_position
            ): idx for idx, batch_info in enumerate(image_batches)
        }
        for future in tqdm(as_completed(future_to_batch_idx), total=len(image_batches), desc=f"分析學生 {student_id} 的圖片批次"):
            try:
                result = future.result()
                all_results_from_threads.append(result)
            except Exception as exc:
                batch_idx = future_to_batch_idx[future]
                print(f'\n批次 {batch_idx + 1} 執行時產生錯誤: {exc}')
                all_results_from_threads.append({"batch_index": batch_idx, "error": str(exc)})

    # --- 結果排序與匯總 ---
    all_results_from_threads.sort(key=lambda x: x['batch_index'])
    print("\n所有批次分析完成，開始匯總數據...")

    # ... (您原有的匯總邏輯，這部分是正確的，無需修改) ...
    all_sequence_analysis_results = []
    overall_behavior_summary = {
        "total_images_processed_in_batches": 0, "behavior_counts": Counter(),
        "behavior_confidence_sum": defaultdict(float), "non_task_behavior_examples_for_summary": []
    }
    
    for result in tqdm(all_results_from_threads, desc="匯總分析結果"):
        if "error" in result:
            sequence_result_to_store = {"batch_index": result['batch_index'] + 1, "analysis": default_error_result_structure()}
            all_sequence_analysis_results.append(sequence_result_to_store)
            continue
        
        image_batch_info = result["image_batch_info"]
        sequence_analysis_data = result["analysis"]
        batch_image_filenames = [info["filename"] for info in image_batch_info]

        sequence_result_to_store = {
            "batch_index": result['batch_index'] + 1,
            "image_filenames_in_batch": batch_image_filenames,
            "matched_teacher_position_text": result.get("matched_teacher_position_text"),
            "matched_classroom_view_image": result.get("matched_classroom_view_image"),
            "analysis": sequence_analysis_data
        }
        all_sequence_analysis_results.append(sequence_result_to_store)

        if "error" not in sequence_analysis_data:
            overall_behavior_summary["total_images_processed_in_batches"] += len(batch_image_filenames)
            if "per_image_highlights" in sequence_analysis_data:
                for hl_item in sequence_analysis_data.get("per_image_highlights", []):
                    cat, conf, desc = hl_item.get("behavior_category"), hl_item.get("confidence", 0.0), hl_item.get("description")
                    if cat:
                        overall_behavior_summary["behavior_counts"][cat] += 1
                        overall_behavior_summary["behavior_confidence_sum"][cat] += float(conf)
                        
                        non_task_keywords = ["玩弄物品", "目視他處", "趴睡", "喝水/飲食", "整理個人物品"]
                        if cat in non_task_keywords and desc and len(overall_behavior_summary["non_task_behavior_examples_for_summary"]) < 3:
                            try:
                                student_img_idx = hl_item.get("image_index_in_sequence", -1)
                                if 0 <= student_img_idx < len(batch_image_filenames):
                                    hl_filename = batch_image_filenames[student_img_idx]
                                    hl_timestamp = next((info["timestamp_str"] for info in image_batch_info if info["filename"] == hl_filename), "未知時間")
                                    overall_behavior_summary["non_task_behavior_examples_for_summary"].append(
                                        (hl_filename, hl_timestamp, cat, desc)
                                    )
                            except Exception as e_idx:
                                print(f"提取非任務示例時出錯: {e_idx}")

    # ... (計算統計和生成總結的部分，都是正確的，無需修改) ...
    overall_behavior_stats_list = []
    total_highlight_instances = sum(overall_behavior_summary["behavior_counts"].values())
    
    # 新增：用於統計正負向行為總數的字典
    valence_summary = {"正向": 0, "負向": 0, "中性": 0}

    for behavior, count in overall_behavior_summary["behavior_counts"].items():
        percentage = (count / total_highlight_instances * 100) if total_highlight_instances > 0 else 0
        avg_confidence = (overall_behavior_summary["behavior_confidence_sum"][behavior] / count) if count > 0 else 0
        
        # 【新增】從我們建立的字典中查找該行為的價值分類
        valence = LABEL_TO_VALENCE.get(behavior, "未分類") # 使用 .get() 避免找不到時出錯

        # 【新增】將計數加入到價值分類統計中
        if valence in valence_summary:
            valence_summary[valence] += count

        overall_behavior_stats_list.append({
            "behavior_category": behavior,
            "valence": valence,  # <-- 新增的欄位
            "count": count,
            "percentage": round(percentage, 1),
            "average_confidence": round(avg_confidence, 2)
        })

    overall_behavior_stats_list.sort(key=lambda x: x["count"], reverse=True)
    personalized_notes = generate_personalized_summary_notes(student_id, overall_behavior_stats_list, overall_behavior_summary["non_task_behavior_examples_for_summary"], client)
    behavior_to_images_map = defaultdict(list)
    for result in all_results_from_threads:
        if "error" in result or "analysis" not in result or "error" in result["analysis"]: continue
        sequence_analysis = result["analysis"]
        image_batch_info = result.get("image_batch_info", [])
        filenames_in_batch = [info.get("filename") for info in image_batch_info]
        if "per_image_highlights" in sequence_analysis and isinstance(sequence_analysis["per_image_highlights"], list):
            for highlight in sequence_analysis["per_image_highlights"]:
                behavior_category = highlight.get("behavior_category")
                image_index = highlight.get("image_index_in_sequence")
                if (behavior_category and isinstance(image_index, int) and 0 <= image_index < len(filenames_in_batch)):
                    image_filename = filenames_in_batch[image_index]
                    if image_filename and image_filename not in behavior_to_images_map[behavior_category]:
                        behavior_to_images_map[behavior_category].append(image_filename)
    for behavior in behavior_to_images_map:
        behavior_to_images_map[behavior].sort()
    print("行為索引建立完成。")

    # ==============================================================================
    #  ↓↓↓ 【關鍵修正】替換掉會引發 NameError 的舊程式碼 ↓↓↓
    # ==============================================================================
    
    # 計算正負向行為的總佔比
    total_classified_instances = sum(valence_summary.values())
    valence_summary_with_percentage = {
        valence: {
            "count": count,
            "percentage": round((count / total_classified_instances * 100), 1) if total_classified_instances > 0 else 0
        } for valence, count in valence_summary.items()
    }


    # --- 組合最終 JSON 報告 ---
    final_json_output = {
        "report_metadata": {
            "student_id": student_id,
            "student_number": student_number,
            "report_generation_time": "07/13",
            "student_image_source_folder": "英文",
            "teacher_position_source_json": os.path.basename(TEACHER_POSITION_JSON) if TEACHER_POSITION_JSON and os.path.isfile(TEACHER_POSITION_JSON) else "N/A",
            "classroom_view_source_folder": os.path.basename(CLASSROOM_IMAGES_FOLDER) if CLASSROOM_IMAGES_FOLDER and os.path.isdir(CLASSROOM_IMAGES_FOLDER) else "N/A",            "classroom_context": { "student_position": student_position },
            "analysis_settings": {
                "vision_model": MODEL_NAME_VISION,
                "text_model": MODEL_NAME_TEXT,
                "images_per_batch": IMAGES_PER_API_CALL,
                "context_images_per_batch_desc": "動態匹配老師和班級照片各一張",
                "confidence_threshold": BEHAVIOR_CONFIDENCE_THRESHOLD_FILTER
            }
        },
        "overall_summary": {
            "total_images_found": len(image_files_with_timestamps),
            "total_images_analyzed": overall_behavior_summary["total_images_processed_in_batches"],
            "total_batches": len(image_batches),
            "valence_summary": valence_summary_with_percentage, # <-- 新增的整體統計
            "behavior_statistics": overall_behavior_stats_list, # <-- 現在每一項裡面都包含了 valence 標籤
            "behavior_to_images_index": behavior_to_images_map,
            "ai_summary_notes": personalized_notes
        },
        "detailed_sequence_analysis": all_sequence_analysis_results
    }

    # --- 儲存檔案 ---
    current_timestamp_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_student_id_for_filename = re.sub(r'[\\/*?:"<>|]',"", student_id.replace(" ", "_"))
    json_filename = JSON_FILENAME_TEMPLATE.format(student_id=safe_student_id_for_filename, timestamp=current_timestamp_str)
    json_filepath = os.path.join(student_specific_folder_base, json_filename)
    try:
        with open(json_filepath, 'w', encoding='utf-8') as f:
            json.dump(final_json_output, f, ensure_ascii=False, indent=4)
        print(f"\n✅ 學生 '{student_id}' 的序列行為分析報告已成功儲存至: {json_filepath}")
    except Exception as e:
        print(f"❌ 錯誤：儲存 JSON 檔案時發生問題：{e}")
    
    print("\n--- 處理完成 ---")

if __name__ == "__main__":
    main()

### 節省成本版

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import base64
import json
import datetime
import time
from collections import Counter, defaultdict
from openai import OpenAI, APIError, RateLimitError, AuthenticationError
from PIL import Image, UnidentifiedImageError
import io
from tqdm import tqdm
import math
from dotenv import load_dotenv 
from concurrent.futures import ThreadPoolExecutor, as_completed
import bisect

# --- Configuration ---


# 班級整體照片資料夾路徑 
CLASSROOM_IMAGES_FOLDER = r'C:\Users\User\Desktop\test\student_full_classroom\0713_english_class'

# 【新】老師視角照片資料夾路徑 (從學生視角拍攝老師在黑板前的照片) - 可選
TEACHER_POSITION_JSON = r'C:\Users\User\Desktop\test\teacher_position\0706test.json'

# 【新】時間軸對齊設定 (格式: "HH:MM:SS")
CLASSROOM_START_TIME_OFFSET = "00:04:15"  # 班級照片的起始時間，設定為空字串 "" 表示不篩選
TEACHER_JSON_START_TIME_OFFSET = "00:04:15" # 老師位置資料的起始時間，設定為空字串 "" 表示不篩選

# ... (您原有的其他設定) ...

JSON_OUTPUT_FOLDER = "SynologyDrive\json_behavior"


JSON_FILENAME_TEMPLATE = "student_{student_id}_behavior_report_{timestamp}.json"
SAMPLING_RATE = 5  # 圖片取樣率 (每 5 張分析 1 張，設為 1 則分析全部)
IMAGE_DETAIL_LEVEL = "low" # 圖片解析度 ('low' 或 'high')，low 可大幅降低成本
MODEL_NAME_VISION = "gpt-4.1" # 推薦使用最新多模態模型，如 gpt-4o
MODEL_NAME_TEXT = "gpt-4-turbo"   # 推薦使用最新文本模型，如 gpt-4o 或 gpt-4-turbo
MODEL_NAME_SUMMARY = "gpt-3.5-turbo" # 【新】用於生成個人化總結的經濟型模型MAX_TOKENS_VISION_COMPLETION = 2500 # 序列分析可能需要更多 token
MAX_TOKENS_SUMMARY_COMPLETION = 1000 # 個性化總結的 token
BEHAVIOR_CONFIDENCE_THRESHOLD_FILTER = 0.9 # 行為信度過濾閾值 (例如 70%)
IMAGES_PER_API_CALL = 10 # 一次API調用處理的圖片數量
BEHAVIOR_SIMILARITY_CONFIDENCE_THRESHOLD = 0.15 # 過濾相似連續行為的信度差異閾值

# ==========================================================

# --- 標準行為分類與定義 (核心辭典，來自圖一) ---
STANDARD_BEHAVIOR_CATEGORIES = {
    "視線": [
        {"label": "目視教師", "definition": "學生的面部主要朝向教室前方教師區域。"},
        {"label": "目視黑板", "definition": "學生的面部主要朝向教室前方黑板區域。"},
        {"label": "目視書本/筆記","definition": "學生的視線朝向自己的桌面、書本或筆記本。注意：只要學生頭部低垂且沒有進行其他可識別的非學習行為（如玩弄物品、趴睡），就應優先歸類於此。"},
        {"label": "目視同學", "definition": "學生的面部明確轉向另一位或多位同學。"},
        {"label": "目視他處", "definition": "學生的面部朝向非前方、非桌面/教材、非同學的方向。"}
    ],
    "肢體(手部)": [
        {"label": "做筆記", "definition": "在『目視書本/筆記』的基礎上，學生手持筆，並在紙張上進行明確的書寫、繪圖動作。此標籤優先級高於單純的『目視書本/筆記』。"},
        {"label": "翻閱書本", "definition": "學生手部正在主動翻閱、移動書本。"},
        {"label": "玩弄物品", "definition": "學生手部在進行與當前學習任務無關的重複性、無目的性的小動作。注意：喝水、整理書包不屬於玩弄物品。"},
    ],
    "身體姿態": [
        {"label": "坐姿直立", "definition": "學生上半身基本保持直立，或略微前傾。"},
        {"label": "身體前傾", "definition": "學生上半身明顯向前傾斜，靠近桌面或前方。"},
        {"label": "身體後靠", "definition": "學生背部倚靠在椅背上，可能呈現較放鬆的姿態。"},
        {"label": "低頭(非學習)", "definition": "學生頭部明顯低垂，但視線並非看向書本，且無任何手部學習動作，可能在發呆或打瞌睡。**僅在能明確排除看書和筆記時使用。**"},
        {"label": "趴睡", "definition": "學生將頭部枕於手臂或桌面，或以其他方式呈現明顯的睡眠或休息狀態。"}
    ],
    "互動": [
        {"label": "舉手", "definition": "學生舉起一隻手（通常高於頭部）。"}
    ],
    "其他狀態": [
        {"label": "喝水/飲食", "definition": "學生正在使用杯子、水壺等容器飲用液體，或食用固體食物。"},
        {"label": "整理個人物品", "definition": "學生正在整理書包、桌面抽屜、衣物等與當前直接學習任務無關的個人物品。"},
        {"label": "無明顯特定行為", "definition": "學生處於一種相對靜止、無上述明確行為的狀態，視線和手部無特定指向或動作。"},
        {"label": "被遮擋/無法判斷", "definition": "因遮擋、模糊或角度問題，無法清晰識別學生的主要行為。"}
    ]
}
# 自動從新結構生成有效的標籤列表
VALID_BEHAVIOR_LABELS = [item['label'] for category in STANDARD_BEHAVIOR_CATEGORIES.values() for item in category]

BEHAVIOR_CODES = {
    # 視線
    "V_TCH": "目視教師", "V_BRD": "目視黑板", "V_BOK": "目視書本/筆記", 
    "V_CLS": "目視同學", "V_ELS": "目視他處",
    # 肢體(手部)
    "H_NOT": "做筆記", "H_FLIP": "翻閱書本", "H_PLAY": "玩弄物品",
    # 身體姿態
    "P_STR": "坐姿直立", "P_LEAN": "身體前傾", "P_BACK": "身體後靠", 
    "P_DWN": "低頭(非學習)", "P_SLP": "趴睡",
    # 互動
    "I_HND": "舉手",
    # 其他狀態
    "O_DRK": "喝水/飲食", "O_TDY": "整理個人物品", 
    "O_NON": "無明顯特定行為", "O_UNK": "被遮擋/無法判斷"
}

# 自動生成反向查找字典，用於本地解碼
CODE_TO_BEHAVIOR = {code: label for code, label in BEHAVIOR_CODES.items()}
BEHAVIOR_TO_CODE = {label: code for code, label in BEHAVIOR_CODES.items()}

# AI返回標籤到標準標籤的映射規則 (與新標籤對齊)
BEHAVIOR_MAPPING_RULES = {
    "目視桌面/教材": "目視書本", "目視桌面": "目視書本", "看書": "目視書本",
    "書寫/做筆記": "筆記", "動手操作-書寫/做筆記": "筆記",
    "視覺專注-閱讀書本/講義": "目視書本",
    "視覺專注-看老師/黑板方向": "目視教師", "目視黑板/老師": "目視教師",
    "看老師": "目視教師", "看黑板": "目視黑板",
    "非任務相關動作-玩弄物品(筆等)": "玩弄物品", "非任務相關動作-觸摸臉部/頭髮": "玩弄物品",
    "非任務相關動作-整理物品": "整理個人物品",
    "社交互動-與同學互動": "目視同學",
    "趴睡/休息": "趴睡",
    "低頭/伏案(非睡)": "低頭",
}

BEHAVIOR_VALENCE_MAP = {
    # 正向 (Active Engagement): 明顯投入當前學習任務的行為
    "正向": [
        "做筆記",         # 最高級別的投入指標，結合了視覺和動手操作
        "舉手",             # 主動參與互動
        "目視教師",         # 專注於教學者
        "目視黑板",         # 專注於教學內容
        "目視書本/筆記",    # 可能是閱讀，也可能是發呆，單從視線難以100%確定
        "翻閱書本"         # 可能是找資料（正向），也可能是隨意翻動（中性）
    ],
    # 負向 (Clear Disengagement): 明顯脫離學習任務、可能影響自己或他人的行為
    "負向": [
        "趴睡",             # 完全脫離學習狀態
        "玩弄物品",         # 明顯的非任務行為，分散注意力
        "目視他處",          # 視線脫離學習區域（老師、黑板、課本）
        "目視同學"         # 在小組討論中是正向，在聽講時可能是負向，故歸為中性    
    ],
    # 中性 (Neutral / Context-Dependent): 基本姿態、生理需求或情境依賴的行為
    "中性": [
        "身體前傾",          # 通常與高度專注相關
        "坐姿直立",         # 標準姿態，無法直接判斷投入度
        "身體後靠",         # 可能是放鬆，也可能是疲倦，無法直接判斷
        "低頭(非學習)",     # 雖然傾向負向，但可能是短暫疲勞，歸為中性以避免過度解讀
        "喝水/飲食",        # 基本生理需求，除非頻繁發生，否則不應標為負向
        "整理個人物品",     
        "無明顯特定行為",   # 預設的中性狀態
        "被遮擋/無法判斷" 
    ]
}

LABEL_TO_VALENCE = {
    label: valence 
    for valence, labels in BEHAVIOR_VALENCE_MAP.items() 
    for label in labels
}

# ---------------------------
#  API 金鑰配置
# ---------------------------
load_dotenv() # 加載 .env 文件中的環境變數
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("警告：未找到環境變數 'OPENAI_API_KEY' (已嘗試從 .env 加載)。")
    api_key = input("請貼上你的 OpenAI API 金鑰 (sk-...) 並按 Enter：").strip()
    if not api_key: print("錯誤：未提供 API 金鑰。程式即將終止。"); exit()
    else: print("已使用手動輸入的 API 金鑰。")

try:
    client = OpenAI(api_key=api_key)
    client.models.list() # 嘗試調用一個簡單的API來驗證金鑰
    print("OpenAI client 初始化並驗證成功。")
except AuthenticationError: print("錯誤：OpenAI API 金鑰無效或認證失敗。"); exit()
except RateLimitError: print("錯誤：API金鑰已達到速率限制。請稍後再試或檢查您的配額。"); exit()
except APIError as e: print(f"初始化 OpenAI Client 時發生 API 錯誤: {e}"); exit()
except Exception as e: print(f"初始化 OpenAI Client 時發生未知錯誤: {e}"); exit()

# ---------------------------
# 輔助函數
# ---------------------------
def parse_time_offset(time_str):
    """將 "HH:MM:SS" 格式的字串轉換為 timedelta 物件"""
    if not time_str or not isinstance(time_str, str):
        return None
    try:
        h, m, s = map(int, time_str.split(':'))
        return datetime.timedelta(hours=h, minutes=m, seconds=s)
    except (ValueError, TypeError):
        print(f"警告：時間偏移量 '{time_str}' 格式不正確，應為 'HH:MM:SS'。將忽略此設定。")
        return None

def get_valid_input(prompt_message):
    while True:
        user_input = input(prompt_message).strip()
        if user_input: return user_input
        print("錯誤：輸入不能為空，請重新輸入。")

def get_valid_folder_path(prompt_message, is_optional=False): # 新增 is_optional 參數
    while True:
        folder_path = input(prompt_message).strip().strip('"')
        if not folder_path and is_optional:
            return None # 如果是可選的且用戶未輸入，返回 None
        if os.path.isdir(folder_path):
            return folder_path
        print(f"錯誤：路徑 '{folder_path}' 不是一個有效的資料夾，請重新輸入。")

def get_timestamp_from_filename(filename):
    """
    【升級版】從檔名解析時間戳。
    能同時處理兩種格式:
    1. HH-MM-SS-ms (例如 'snapshot_10-20-30-123.jpg')
    2. ...HHhMMmSSs... (例如 'blackboard_000h00m50s_f0001501.png')
    """
    # 模式一：優先嘗試匹配 HH-MM-SS-ms 格式
    match_format1 = re.search(r'(\d{2})-(\d{2})-(\d{2})-(\d{3})', filename)
    if match_format1:
        try:
            h, m, s, ms = map(int, match_format1.groups())
            return datetime.timedelta(hours=h, minutes=m, seconds=s, milliseconds=ms)
        except (ValueError, IndexError):
            # 如果解析失敗，就繼續嘗試下一個格式，而不是直接返回
            pass 

    # 模式二：如果模式一失敗，則嘗試匹配 ...h...m...s 格式
    match_format2 = re.search(r'(\d+)h(\d{2})m(\d{2})s', filename)
    if match_format2:
        try:
            # 注意：這種格式沒有毫秒，我們將其設為 0
            h, m, s = map(int, match_format2.groups())
            return datetime.timedelta(hours=h, minutes=m, seconds=s, milliseconds=0)
        except (ValueError, IndexError):
            # 如果解析失敗，同樣略過
            pass

    # 如果兩種格式都沒匹配成功，最終返回 None
    return None

def encode_image_to_base64(image_path, max_size_kb=512, target_quality=75):
    try:
        with Image.open(image_path) as img: img.verify()
        with Image.open(image_path) as img:
            if img.mode == 'RGBA' or img.mode == 'P': img = img.convert('RGB')
            current_quality = target_quality
            output_buffer = io.BytesIO()
            temp_img = img.copy()
            temp_img.save(output_buffer, format="JPEG", quality=current_quality)
            current_size_kb = output_buffer.tell() / 1024

            if current_size_kb > max_size_kb:
                scale_factor = math.sqrt(max_size_kb / current_size_kb)
                new_width = int(temp_img.width * scale_factor * 0.9)
                new_height = int(temp_img.height * scale_factor * 0.9)
                if new_width >= 50 and new_height >= 50: # 最小尺寸限制
                    print(f"    圖片 {os.path.basename(image_path)} ({current_size_kb:.1f} KB) 過大，縮放並調整質量...")
                    temp_img = temp_img.resize((new_width, new_height), Image.Resampling.LANCZOS)
                    output_buffer = io.BytesIO()
                    temp_img.save(output_buffer, format="JPEG", quality=max(current_quality - 15, 40)) # 質量可以降更多
                else:
                    print(f"    警告: 圖片 {os.path.basename(image_path)} 縮放後過小，可能影響質量。使用較低質量。")
                    output_buffer = io.BytesIO()
                    img.save(output_buffer, format="JPEG", quality=max(current_quality // 2, 30) )


            output_buffer.seek(0)
            binary_data = output_buffer.getvalue()
            base64_encoded_data = base64.b64encode(binary_data)
            return f"data:image/jpeg;base64,{base64_encoded_data.decode('utf-8')}"
    except Exception as e: print(f"錯誤：處理圖片 '{image_path}': {e}"); return None

def get_behavior_sequence_analysis_system_prompt(student_position, has_teacher_context):
    """
    【v6.0 - 成本優化版】為序列分析任務生成動態的系統提示。
    此版本使用「行為編碼」並要求精簡的 JSON 輸出以大幅降低 Token 成本。
    """
    # --- 1. 動態生成使用「編碼」的行為表 ---
    behavior_table_for_prompt = "\n\n**【學習行為編碼表】**\n你 **必須** 且 **只能** 從以下列表的「編碼 (Code)」中選擇行為進行標註。**絕對不允許** 使用任何不在列表中的編碼。\n"
    for code, label in BEHAVIOR_CODES.items():
        # 從原始定義中查找對應的描述
        definition = ""
        for category in STANDARD_BEHAVIOR_CATEGORIES.values():
            for item in category:
                if item['label'] == label:
                    definition = item['definition']
                    break
        behavior_table_for_prompt += f"*   **`{code}`**: {label} - {definition}\n"
    behavior_table_for_prompt += "\n*如果行為因任何原因無法清晰判斷，請 **必須** 使用 **`O_UNK`** 編碼。*\n"

    # --- 2. 根據是否有老師情境，動態生成指令片段 (邏輯不變) ---
    teacher_context_header = ""
    teacher_context_workflow = f"""*   **學生座位**: 該學生的固定座位是 **'{student_position}'**。"""
    teacher_context_cross_validation = "則根據學生頭部朝向和常理進行推斷（例如，直視前方是「目視黑板 (`V_BRD`)」，明顯轉向一側可能是「目視同學 (`V_CLS`)」）。"
    
    if has_teacher_context:
        teacher_context_header = """2.  **【老師位置文字情境】(最高優先級情境)**: 你會收到一段關於老師位置的文字描述（例如：「老師在左側」）。這是判斷「目視教師 (`V_TCH`)」的**決定性依據**。"""
        teacher_context_workflow = f"""*   **老師位置**: 根據使用者提供的【老師位置文字情境】，你已知曉老師在教室前方的「左側」、「中間」或「右側」。
*   **學生座位**: 該學生的固定座位是 **'{student_position}'**。"""
        teacher_context_cross_validation = """**根據【老師位置文字情境】**: 將學生的頭部朝向與文字描述的老師**位置**進行比對。
            *   **若匹配**: 標記為 **`V_TCH`**。（例如：學生頭部朝向右側，且情境文字是「右側」，則匹配）
            *   **若不匹配**: 標記為 **`V_BRD`** (如果視線是朝向前方區域) 或 **`V_ELS`**。"""

    # --- 3. 組合所有指令到一個連貫的 Prompt 中 ---
    return f"""
「你是一位高效的課堂行為分析AI。你的任務是快速、準確地根據多源信息，輸出結構化的行為編碼。」

**【你的核心任務：多源交叉驗證與編碼輸出】**

你將收到以下資料，請整合分析：
1.  **【學生特寫照片序列】**: 主要觀察視角。
{teacher_context_header}
3.  **【班級整體情境照片】**: 上帝視角，用於驗證和補充。

**【分析流程】**

對於序列中的每一組照片，嚴格遵循以下流程：

**步驟一：主要分析**
*   觀察【學生特寫照片】。如果特寫照片有多人，專注於與前序照片特徵一致的學生。

**步驟二：交叉驗證 (最關鍵)**
*   在【班級整體情境照片】中，根據座位提示 **'{student_position}'** 找到目標學生。
*   **若特寫照片清晰**，與班級照片中的行為比對。若不一致，**以班級照片為準**。
*   **若特寫照片模糊或被遮擋**，**直接依據班級照片**進行判斷。

**步驟三：判斷視線**
*   在確定視線朝向前後，結合【老師位置文字情境】來決定使用 `V_TCH` (目視教師), `V_BRD` (目視黑板), 或其他視線編碼。
*   {teacher_context_cross_validation}

**步驟四：標註行為編碼**
*   參考下方的【學習行為編碼表】，為每個圖像選擇最合適的**一個**編碼。

{behavior_table_for_prompt}

**【輸出 JSON 格式要求 - 極簡版】**
*   你的回答**必須**是一個結構完整的 JSON 物件。
*   **不要**包含 `sequence_summary` 和 `description` 欄位。
*   `behavior_category` 欄位的值**必須**是【學習行為編碼表】中的**編碼** (例如: `H_NOT`, `V_TCH`)。

**【輸出 JSON 格式範例】**
```json
{{
  "sequence_analysis_confidence": 0.95,
  "per_image_highlights": [
    {{
      "image_index_in_sequence": 0,
      "context_description": "依據特寫，頭部朝向與老師位置匹配。",
      "behavior_category": "V_TCH",
      "confidence": 0.98
    }},
    {{
      "image_index_in_sequence": 1,
      "context_description": "特寫被遮擋，依據班級照片分析。",
      "behavior_category": "P_DWN",
      "confidence": 0.90
    }}
  ]
}}
```"""

def default_error_result_structure():
    return { "error": "分析失敗或無有效數據", "sequence_analysis_confidence": 0.0, "sequence_summary": "未能生成序列總結。", "dominant_sustained_behaviors": [], "significant_behavior_shifts": [], "per_image_highlights": [], "general_sequence_atmosphere_hint": "未知" }

def analyze_student_behavior_from_images_sequence(student_image_paths, teacher_position_text, classroom_view_image_path, image_filenames_batch, openai_client, student_id, student_position):
    if not student_image_paths:
        return {"error": "沒有提供學生圖片路徑", "analysis": default_error_result_structure()}
    
    user_message_content = []
    
    # 1. 編碼學生個人圖片序列 (使用配置中的解析度)
    encoded_student_images = []
    for img_path in student_image_paths:
        b64_img = encode_image_to_base64(img_path)
        if b64_img:
            encoded_student_images.append({
                "type": "image_url", 
                "image_url": {"url": b64_img, "detail": IMAGE_DETAIL_LEVEL} # 使用全局配置
            })

    if not encoded_student_images:
        return {"error": "所有學生圖片編碼失敗", "analysis": default_error_result_structure()}
        
    # 2. 編碼班級整體照片 (使用配置中的解析度)
    encoded_classroom_view_image = None
    if classroom_view_image_path:
        b64_img = encode_image_to_base64(classroom_view_image_path)
        if b64_img:
            encoded_classroom_view_image = {
                "type": "image_url", 
                "image_url": {"url": b64_img, "detail": IMAGE_DETAIL_LEVEL} # 使用全局配置
            }

    # 3. 組合請求體 (邏輯不變)
    user_message_content.append({"type": "text", "text": f"請根據系統提示中的偵探任務，分析學生「{student_id}」的行為。學生照片序列的文件名（供您參考）為: {', '.join(image_filenames_batch)}。"})
    teacher_context_prompt = f"【老師位置情境】根據預先分析，在此時間段，老師的位置在教室前方的「{teacher_position_text}」。請以此作為判斷『目視教師』的核心依據。"
    user_message_content.append({"type": "text", "text": teacher_context_prompt})
    if encoded_classroom_view_image:
        user_message_content.append({"type": "text", "text": "【班級整體照片】(用於定位學生和觀察整體氛圍)"})
        user_message_content.append(encoded_classroom_view_image)
    user_message_content.append({"type": "text", "text": "【學生個人照片序列】(主要分析對象)"})
    user_message_content.extend(encoded_student_images)

    # 4. 獲取新的系統提示
    system_prompt_content = get_behavior_sequence_analysis_system_prompt(student_position, bool(teacher_position_text and teacher_position_text != "未知"))

    # 5. API 呼叫與錯誤處理
    retry_attempts = 2
    for attempt in range(retry_attempts + 1):
        try:
            has_teacher_context = bool(teacher_position_text and teacher_position_text != "未知")
            has_classroom_image = bool(encoded_classroom_view_image)
            print(f"  正在向 {MODEL_NAME_VISION} 發送請求 (學生: {len(encoded_student_images)}, 老師位置數據: {'有' if has_teacher_context else '無'}, 班級整體照片: {'有' if has_classroom_image else '無'})...")
            
            response = openai_client.chat.completions.create(
                model=MODEL_NAME_VISION, response_format={"type": "json_object"},
                messages=[{"role": "system", "content": system_prompt_content}, {"role": "user", "content": user_message_content}],
                max_tokens=MAX_TOKENS_VISION_COMPLETION, temperature=0.05
            )
            raw_content = response.choices[0].message.content
            analysis_result = json.loads(raw_content)
            
            # ==========================================================
            # ↓↓↓ 【關鍵新增】本地解碼，將行為編碼轉換回中文標籤 ↓↓↓
            # ==========================================================
            if "per_image_highlights" in analysis_result and isinstance(analysis_result["per_image_highlights"], list):
                for hl in analysis_result["per_image_highlights"]:
                    if isinstance(hl, dict) and "behavior_category" in hl:
                        code = hl.get("behavior_category")
                        # 使用我們建立的字典進行解碼
                        hl["behavior_category"] = CODE_TO_BEHAVIOR.get(code, f"未知編碼({code})") # 如果找不到代碼，保留原始代碼以便除錯
            
            # 【新增】由於AI不再生成，我們手動補上空欄位以保持格式統一
            if "sequence_summary" not in analysis_result:
                analysis_result["sequence_summary"] = "已設定為精簡模式，此欄位由本地生成。"

            return analysis_result

        except json.JSONDecodeError: print(f"    錯誤：無法解析JSON (第 {attempt+1} 次嘗試)。回應: {raw_content[:500]}...");
        except RateLimitError: print(f"  警告：API速率限制，等待 {API_RETRY_DELAY_SECONDS}s 後重試..."); time.sleep(API_RETRY_DELAY_SECONDS)
        except APIError as e: print(f"  錯誤：API錯誤 (第 {attempt+1} 次嘗試): {e}"); time.sleep(5)
        except Exception as e: print(f"  錯誤：未知錯誤 (第 {attempt+1} 次嘗試): {e}"); time.sleep(5)
    
    print(f"  錯誤：圖片序列分析在多次重試後失敗。")
    return default_error_result_structure()

def generate_personalized_summary_notes(student_id, overall_stats, non_task_highlights, openai_client):
    if not openai_client: return {"error": "OpenAI client not available"}
    # ... (此函數內部的 Prompt 內容完全不需要修改) ...
    stats_summary_for_ai = "\n".join([f"- {s['behavior_category']}: {s['percentage']:.1f}% ({s['count']}次)" for s in overall_stats[:7]]) # 確保百分比格式
    non_task_prompt_part = "該生在本堂課中，未觀察到明顯或頻繁的非任務相關行為。"
    if non_task_highlights:
        highlights_str = "\n".join([f"  - 圖 '{img_fn}' (~{ts}): '{beh}' (描述: {desc})" for img_fn, ts, beh, desc in non_task_highlights[:3]])
        if highlights_str: non_task_prompt_part = f"在本堂課中，觀察到一些非任務相關行為，例如：\n{highlights_str}\n這可能影響了學習專注度。"
    prompt_for_summary = f"""
    「你是一位專業且富有同理心的學習行為教練。你的目標不是批評，而是透過客觀數據，引導學生發現自己的學習模式，並提供能立即實踐的策略，以激發他們『自我反思』的動力。」

    **任務：** 為學生「{student_id}」撰寫一份「AI學習夥伴的觀察與建議」。

    **學生的課堂行為數據：**
    *   **主要行為分佈:**
    {stats_summary_for_ai}
    *   **值得注意的行為片段:**
    {non_task_prompt_part}

    **撰寫指引與 JSON 格式要求：**
    請嚴格遵循以下指引，產生一個結構完整的 JSON 物件。

    *   **`greeting` (問候語):**
        *   用親切、個人化的方式稱呼學生，例如：「嗨，{student_id} 同學，一起來看看這次課堂的學習足跡吧！」

    *   **`positive_feedback` (亮點觀察):**
        *   **必須**從數據中找出最值得肯定的行為（例如「目視教師」或「筆記」佔比最高），並給予具體、真誠的讚美。
        *   **範例**：「我發現你在這堂課有超過一半的時間都在『目視教師』，這代表你非常努力地跟上老師的節奏，非常棒！」

    *   **`observation_points_summary` (行為模式提醒):**
        *   客觀、中性地指出一個或兩個最主要的、可能影響學習的行為模式。避免使用負面詞彙。
        *   **範例**：「數據也顯示，大約有 15% 的時間出現了『玩弄物品』或『目視他處』的狀況，這些時刻可能讓我們不小心錯過了一些重點喔。」

    *   **`reflection_points` (反思引導提問):**
        *   **【此項最為關鍵】** 根據前面的觀察點，設計 2-3 個**開放式問題**，引導學生思考行為背後的原因，而不是直接給答案。
        *   **問題範例 1**：「我們可以一起回想看看，當出現『玩弄物品』的時候，通常是在課程的哪個階段呢？是覺得內容太簡單、太難，還是剛好有點疲倦了呢？」
        *   **問題範例 2**：「當視線看向其他地方時，是想到了什麼有趣的事，還是被教室裡的其他動靜吸引了呢？了解這些原因，能幫助我們找到最適合自己的專注方法。」

    *   **`suggestions` (可實踐的小建議):**
        *   提供 1-2 個**具體、微小、且容易執行**的行動建議。不要說「要專心」，而是給出方法。
        *   **建議範例 1**：「下次當你發現自己開始無意識地轉筆時，可以試著把它輕輕放下，然後做一個深呼吸，再重新將目光移回老師或課本上。」
        *   **建議範例 2**：「如果感覺到疲倦或分心，可以試試看『筆記專注法』：在筆記本上寫下老師說的任何一個關鍵字，這個小動作能幫助我們的大腦重新連線！」

    *   **`encouragement` (鼓勵與結語):**
        *   用一句溫暖、有力的話作結，強調這份報告是幫助他成長的工具。
        *   **範例**：「每一次的觀察都是為了讓我們更了解自己。相信你透過這些小小的調整，一定能發揮出自己最大的潛力，加油！」
    """
    retry_attempts = 2
    for attempt in range(retry_attempts):
        try:
            print(f"  正在為學生 {student_id} 使用 {MODEL_NAME_SUMMARY} 生成個性化總結...") # 修改了日誌輸出
            response = client.chat.completions.create(
                # ==========================================================
                # ↓↓↓ 【關鍵修改】使用我們新設定的經濟型模型 ↓↓↓
                # ==========================================================
                model=MODEL_NAME_SUMMARY, 
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": "你是一位富有同理心和洞察力的教育顧問。"},
                          {"role": "user", "content": prompt_for_summary}],
                max_tokens=MAX_TOKENS_SUMMARY_COMPLETION, temperature=0.7
            )
            summary_data = json.loads(response.choices[0].message.content)
            expected_keys = ["greeting", "positive_feedback", "observation_points_summary", "reflection_points", "suggestions", "encouragement"]
            if all(key in summary_data for key in expected_keys): 
                print(f"  成功為學生 {student_id} 生成個性化總結。")
                return summary_data
            else: 
                print(f"    警告：個性化總結JSON缺少鍵。返回: {summary_data}")
                return {key: summary_data.get(key, f"AI未能生成 ({key})") for key in expected_keys}
        except Exception as e: 
            print(f"  生成個性化總結錯誤 ({attempt + 1}): {e}")
            time.sleep(API_RETRY_DELAY_SECONDS if isinstance(e, RateLimitError) else 5)
        if attempt == retry_attempts - 1: 
            print(f"  錯誤：無法為學生 {student_id} 生成個性化總結。")
            return {"greeting": f"親愛的 {student_id},", "positive_feedback": "總結生成遇到問題。", "observation_points_summary": "請參考統計數據。", "reflection_points": "未能生成。", "suggestions": "請自行評估。", "encouragement": "加油！"}
    return {}

def find_closest_image_path(representative_timestamp, sorted_photo_list, max_time_diff_seconds=5):
    """
    一個可重用的輔助函數，用二分查找法在排序好的照片列表中找到時間最接近的照片路徑。
    """
    if not sorted_photo_list:
        return None

    all_timestamps = [item[0] for item in sorted_photo_list]
    # bisect_left 找到應該插入的位置
    insertion_point = bisect.bisect_left(all_timestamps, representative_timestamp)

    closest_path = None
    min_diff = datetime.timedelta.max

    # 只檢查插入點及其前後的幾個候選照片，效率極高
    # 檢查範圍設為 insertion_point-2 到 insertion_point+2 以增加容錯
    start_index = max(0, insertion_point - 2)
    end_index = min(len(sorted_photo_list), insertion_point + 2)

    for i in range(start_index, end_index):
        candidate_ts, candidate_path = sorted_photo_list[i]
        diff = abs(candidate_ts - representative_timestamp)
        if diff < min_diff:
            min_diff = diff
            closest_path = candidate_path

    # 只有在時間差在容許範圍內才返回路徑
    if closest_path and min_diff.total_seconds() <= max_time_diff_seconds:
        return closest_path
    
    return None

def find_closest_position(representative_timestamp, sorted_position_list, max_time_diff_seconds=10):
    """
    用二分查找法在排序好的位置列表中找到時間最接近的老師位置。
    """
    if not sorted_position_list:
        return "未知"

    all_timestamps = [item[0] for item in sorted_position_list]
    insertion_point = bisect.bisect_left(all_timestamps, representative_timestamp)

    closest_position = "未知"
    min_diff = datetime.timedelta.max

    start_index = max(0, insertion_point - 2)
    end_index = min(len(sorted_position_list), insertion_point + 2)

    for i in range(start_index, end_index):
        candidate_ts, candidate_pos = sorted_position_list[i]
        diff = abs(candidate_ts - representative_timestamp)
        if diff < min_diff:
            min_diff = diff
            closest_position = candidate_pos
    
    if min_diff.total_seconds() <= max_time_diff_seconds:
        return closest_position
    
    return "未知"

def process_single_batch(batch_idx, image_batch_info, teacher_positions_data, sorted_classroom_photos, client, student_id, student_position):
    """
    【升級版】處理單一圖片批次，使用老師位置文字數據。
    """
    batch_student_paths = [info["path"] for info in image_batch_info]
    batch_image_filenames = [info["filename"] for info in image_batch_info]
    
    representative_timestamp = image_batch_info[0]["timestamp_obj"]

    # --- 為當前批次查找匹配的「老師位置文字」和「班級整體照片」 ---
    teacher_position_text = find_closest_position(representative_timestamp, teacher_positions_data)
    classroom_view_path = find_closest_image_path(representative_timestamp, sorted_classroom_photos)

    # --- 呼叫【新版】分析函數 ---
    sequence_analysis_data = analyze_student_behavior_from_images_sequence(
        student_image_paths=batch_student_paths,
        teacher_position_text=teacher_position_text, # <-- 傳遞文字
        classroom_view_image_path=classroom_view_path,
        image_filenames_batch=batch_image_filenames,
        openai_client=client,
        student_id=student_id,
        student_position=student_position
    )

    return {
        "batch_index": batch_idx,
        "image_batch_info": image_batch_info,
        "matched_teacher_position_text": teacher_position_text, # <-- 修改回傳的鍵名
        "matched_classroom_view_image": os.path.basename(classroom_view_path) if classroom_view_path else None,
        "analysis": sequence_analysis_data
    }

def main():
    print("--- 學生課堂學習行為分析報告生成 (JSON) v5.0 - 成本優化版 ---")
    print(f"視覺模型: {MODEL_NAME_VISION}, 文本模型: {MODEL_NAME_TEXT}, 總結模型: {MODEL_NAME_SUMMARY}")
    print(f"圖片批次大小: {IMAGES_PER_API_CALL}, 圖片取樣率: 1/{SAMPLING_RATE}, 圖片解析度: {IMAGE_DETAIL_LEVEL}")
    print("-" * 40)

    # --- 使用者輸入部分 ---
    student_id = get_valid_input("請輸入學生姓名 (可中文): ")
    student_number = get_valid_input("請輸入學生座號 (例如: 1): ") 
    student_images_folder = get_valid_folder_path(f"請輸入 '{student_id}' 的個人影像資料夾: ")
    print("\n--- 請輸入課堂情境資訊 ---")
    student_position = get_valid_input("請輸入學生座位 (例如: '第3排中間', '第1排左側'): ")
    print("-" * 50)

    # --- 準備輸出資料夾 ---
    safe_student_id_for_folder = re.sub(r'[\\/*?:"<>|]',"", student_id.replace(" ", "_"))
    student_specific_folder_base = os.path.join(JSON_OUTPUT_FOLDER, safe_student_id_for_folder)
    os.makedirs(student_specific_folder_base, exist_ok=True)

    # --- 準備學生照片資料 ---
    image_files_with_timestamps = []
    valid_extensions = ('.png', '.jpg', '.jpeg', '.webp')
    print(f"正在掃描學生個人照片資料夾: {student_images_folder}...")
    for filename in os.listdir(student_images_folder):
        if filename.lower().endswith(valid_extensions):
            timestamp = get_timestamp_from_filename(filename)
            if timestamp:
                image_files_with_timestamps.append({
                    "path": os.path.join(student_images_folder, filename), "filename": filename,
                    "timestamp_obj": timestamp,
                    "timestamp_str": str(timestamp).split('.')[0]
                })
    image_files_with_timestamps.sort(key=lambda x: x["timestamp_obj"])
    
    # ==========================================================
    # ↓↓↓ 【新增】圖片取樣以降低成本 ↓↓↓
    # ==========================================================
    if SAMPLING_RATE > 1:
        original_count = len(image_files_with_timestamps)
        image_files_with_timestamps = image_files_with_timestamps[::SAMPLING_RATE]
        print(f"已執行圖片取樣：從 {original_count} 張原始照片中，每 {SAMPLING_RATE} 張取 1 張，共 {len(image_files_with_timestamps)} 張照片將被分析。")
    else:
        print(f"成功找到 {len(image_files_with_timestamps)} 張學生個人照片 (未取樣)。")

    if not image_files_with_timestamps: 
        print("錯誤：學生資料夾中未找到有效時間格式的圖片，或取樣後為空。")
        return
    # ==========================================================

    # ... 後續的程式碼，從 `load_teacher_positions` 開始，到整個 `main` 函數結束，
    # 都不需要再做任何修改。您可以直接使用您原有的版本。
    # 我將剩餘部分貼在下方以保證完整性。
    
    def load_teacher_positions(json_path, start_time_offset_str=""): # 新增 start_time_offset_str 參數
        if not json_path or not os.path.isfile(json_path):
            print(f"提示：未提供或找不到老師位置 JSON 檔案 ({json_path})。將不使用老師位置情境。")
            return []
        start_offset = parse_time_offset(start_time_offset_str)
        print(f"正在讀取老師位置數據: {json_path}...")
        try:
            with open(json_path, 'r', encoding='utf-8') as f: data = json.load(f)
            position_map, original_count = [], len(data)
            for item in data:
                try:
                    h, m, s = map(int, item['timestamp'].split(':'))
                    td = datetime.timedelta(hours=h, minutes=m, seconds=s)
                    if start_offset and td < start_offset: continue
                    position_map.append((td, item['position']))
                except (ValueError, KeyError): continue
            position_map.sort(key=lambda x: x[0])
            if start_offset: print(f"成功加載老師位置數據。根據起始時間 '{start_time_offset_str}' 進行過濾，保留 {len(position_map)} 筆 (原 {original_count} 筆)。")
            else: print(f"成功加載並索引了 {len(position_map)} 筆老師位置數據。")
            return position_map
        except Exception as e:
            print(f"錯誤：讀取或解析老師位置 JSON 時發生問題: {e}"); return []

    def load_and_sort_photos(folder_path, photo_type_name, start_time_offset_str=""): # 新增 start_time_offset_str 參數
        if not folder_path or not os.path.isdir(folder_path):
            print(f"提示：未提供或找不到 {photo_type_name} 照片資料夾 ({folder_path})。"); return []
        start_offset = parse_time_offset(start_time_offset_str)
        print(f"正在掃描 {photo_type_name} 照片資料夾: {folder_path}...")
        photo_list, original_count = [], 0
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                original_count += 1
                timestamp = get_timestamp_from_filename(filename)
                if timestamp:
                    if start_offset and timestamp < start_offset: continue
                    photo_list.append((timestamp, os.path.join(folder_path, filename)))
        if photo_list:
            sorted_photos = sorted(photo_list)
            if start_offset: print(f"成功加載 {photo_type_name} 照片。根據起始時間 '{start_time_offset_str}' 進行過濾，保留 {len(sorted_photos)} 張 (原 {original_count} 張有效格式照片)。")
            else: print(f"成功加載並索引了 {len(sorted_photos)} 張 {photo_type_name} 照片。")
            return sorted_photos
        else:
            print(f"警告：在 {photo_type_name} 照片資料夾 '{folder_path}' 中未找到符合條件的圖片。"); return []

    teacher_positions_data = load_teacher_positions(TEACHER_POSITION_JSON, TEACHER_JSON_START_TIME_OFFSET)
    sorted_classroom_photos = load_and_sort_photos(CLASSROOM_IMAGES_FOLDER, "班級整體", CLASSROOM_START_TIME_OFFSET)
    image_batches = [image_files_with_timestamps[i:i + IMAGES_PER_API_CALL] for i in range(0, len(image_files_with_timestamps), IMAGES_PER_API_CALL)]
    print(f"學生圖片將被分為 {len(image_batches)} 個批次進行分析。")

    MAX_WORKERS = 8
    print(f"將使用最多 {MAX_WORKERS} 個執行緒進行平行分析...")
    all_results_from_threads = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_batch_idx = { executor.submit(process_single_batch, idx, batch_info, teacher_positions_data, sorted_classroom_photos, client, student_id, student_position): idx for idx, batch_info in enumerate(image_batches) }
        for future in tqdm(as_completed(future_to_batch_idx), total=len(image_batches), desc=f"分析學生 {student_id} 的圖片批次"):
            try:
                result = future.result()
                all_results_from_threads.append(result)
            except Exception as exc:
                batch_idx = future_to_batch_idx[future]
                print(f'\n批次 {batch_idx + 1} 執行時產生錯誤: {exc}')
                all_results_from_threads.append({"batch_index": batch_idx, "error": str(exc)})

    all_results_from_threads.sort(key=lambda x: x['batch_index'])
    print("\n所有批次分析完成，開始匯總數據...")
    all_sequence_analysis_results = []
    overall_behavior_summary = { "total_images_processed_in_batches": 0, "behavior_counts": Counter(), "behavior_confidence_sum": defaultdict(float), "non_task_behavior_examples_for_summary": [] }
    for result in tqdm(all_results_from_threads, desc="匯總分析結果"):
        if "error" in result:
            all_sequence_analysis_results.append({"batch_index": result['batch_index'] + 1, "analysis": default_error_result_structure()})
            continue
        image_batch_info, sequence_analysis_data = result["image_batch_info"], result["analysis"]
        batch_image_filenames = [info["filename"] for info in image_batch_info]
        all_sequence_analysis_results.append({ "batch_index": result['batch_index'] + 1, "image_filenames_in_batch": batch_image_filenames, "matched_teacher_position_text": result.get("matched_teacher_position_text"), "matched_classroom_view_image": result.get("matched_classroom_view_image"), "analysis": sequence_analysis_data })
        if "error" not in sequence_analysis_data:
            overall_behavior_summary["total_images_processed_in_batches"] += len(batch_image_filenames)
            if "per_image_highlights" in sequence_analysis_data:
                for hl_item in sequence_analysis_data.get("per_image_highlights", []):
                    cat, conf, desc = hl_item.get("behavior_category"), hl_item.get("confidence", 0.0), hl_item.get("description")
                    if cat:
                        overall_behavior_summary["behavior_counts"][cat] += 1
                        overall_behavior_summary["behavior_confidence_sum"][cat] += float(conf)
                        non_task_keywords = ["玩弄物品", "目視他處", "趴睡", "喝水/飲食", "整理個人物品"]
                        if cat in non_task_keywords and desc and len(overall_behavior_summary["non_task_behavior_examples_for_summary"]) < 3:
                            try:
                                student_img_idx = hl_item.get("image_index_in_sequence", -1)
                                if 0 <= student_img_idx < len(batch_image_filenames):
                                    hl_filename = batch_image_filenames[student_img_idx]
                                    hl_timestamp = next((info["timestamp_str"] for info in image_batch_info if info["filename"] == hl_filename), "未知時間")
                                    overall_behavior_summary["non_task_behavior_examples_for_summary"].append( (hl_filename, hl_timestamp, cat, desc) )
                            except Exception as e_idx: print(f"提取非任務示例時出錯: {e_idx}")
    
    overall_behavior_stats_list, total_highlight_instances = [], sum(overall_behavior_summary["behavior_counts"].values())
    valence_summary = {"正向": 0, "負向": 0, "中性": 0}
    for behavior, count in overall_behavior_summary["behavior_counts"].items():
        percentage, avg_confidence = (count / total_highlight_instances * 100) if total_highlight_instances > 0 else 0, (overall_behavior_summary["behavior_confidence_sum"][behavior] / count) if count > 0 else 0
        valence = LABEL_TO_VALENCE.get(behavior, "未分類")
        if valence in valence_summary: valence_summary[valence] += count
        overall_behavior_stats_list.append({ "behavior_category": behavior, "valence": valence, "count": count, "percentage": round(percentage, 1), "average_confidence": round(avg_confidence, 2) })
    overall_behavior_stats_list.sort(key=lambda x: x["count"], reverse=True)
    personalized_notes = generate_personalized_summary_notes(student_id, overall_behavior_stats_list, overall_behavior_summary["non_task_behavior_examples_for_summary"], client)
    behavior_to_images_map = defaultdict(list)
    for result in all_results_from_threads:
        if "error" in result or "analysis" not in result or "error" in result["analysis"]: continue
        sequence_analysis, image_batch_info, filenames_in_batch = result["analysis"], result.get("image_batch_info", []), [info.get("filename") for info in result.get("image_batch_info", [])]
        if "per_image_highlights" in sequence_analysis and isinstance(sequence_analysis["per_image_highlights"], list):
            for highlight in sequence_analysis["per_image_highlights"]:
                behavior_category, image_index = highlight.get("behavior_category"), highlight.get("image_index_in_sequence")
                if (behavior_category and isinstance(image_index, int) and 0 <= image_index < len(filenames_in_batch)):
                    image_filename = filenames_in_batch[image_index]
                    if image_filename and image_filename not in behavior_to_images_map[behavior_category]: behavior_to_images_map[behavior_category].append(image_filename)
    for behavior in behavior_to_images_map: behavior_to_images_map[behavior].sort()
    print("行為索引建立完成。")

    total_classified_instances = sum(valence_summary.values())
    valence_summary_with_percentage = { valence: { "count": count, "percentage": round((count / total_classified_instances * 100), 1) if total_classified_instances > 0 else 0 } for valence, count in valence_summary.items() }

    final_json_output = {
        "report_metadata": {
            "student_id": student_id, "student_number": student_number, "report_generation_time": "07/13", "student_image_source_folder": "英文",
            "teacher_position_source_json": os.path.basename(TEACHER_POSITION_JSON) if TEACHER_POSITION_JSON and os.path.isfile(TEACHER_POSITION_JSON) else "N/A",
            "classroom_view_source_folder": os.path.basename(CLASSROOM_IMAGES_FOLDER) if CLASSROOM_IMAGES_FOLDER and os.path.isdir(CLASSROOM_IMAGES_FOLDER) else "N/A",            "classroom_context": { "student_position": student_position },
            "analysis_settings": {
                "vision_model": MODEL_NAME_VISION, "text_model": MODEL_NAME_TEXT, "images_per_batch": IMAGES_PER_API_CALL, "context_images_per_batch_desc": "動態匹配老師和班級照片各一張", "confidence_threshold": BEHAVIOR_CONFIDENCE_THRESHOLD_FILTER,
                "cost_optimization": { "sampling_rate": SAMPLING_RATE, "image_detail": IMAGE_DETAIL_LEVEL } # 新增成本優化資訊
            }
        },
        "overall_summary": {
            "total_images_found": len(image_files_with_timestamps) * SAMPLING_RATE if SAMPLING_RATE > 1 else len(image_files_with_timestamps), # 顯示原始數量
            "total_images_analyzed": overall_behavior_summary["total_images_processed_in_batches"],
            "total_batches": len(image_batches), "valence_summary": valence_summary_with_percentage, "behavior_statistics": overall_behavior_stats_list, "behavior_to_images_index": behavior_to_images_map, "ai_summary_notes": personalized_notes
        },
        "detailed_sequence_analysis": all_sequence_analysis_results
    }

    current_timestamp_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_student_id_for_filename = re.sub(r'[\\/*?:"<>|]',"", student_id.replace(" ", "_"))
    json_filename = JSON_FILENAME_TEMPLATE.format(student_id=safe_student_id_for_filename, timestamp=current_timestamp_str)
    json_filepath = os.path.join(student_specific_folder_base, json_filename)
    try:
        with open(json_filepath, 'w', encoding='utf-8') as f: json.dump(final_json_output, f, ensure_ascii=False, indent=4)
        print(f"\n✅ 學生 '{student_id}' 的序列行為分析報告已成功儲存至: {json_filepath}")
    except Exception as e: print(f"❌ 錯誤：儲存 JSON 檔案時發生問題：{e}")
    
    print("\n--- 處理完成 ---")

if __name__ == "__main__":
    main()

### AZURE  GPT-4.1

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import base64
import json
import datetime
import time
from collections import Counter, defaultdict
from openai import AzureOpenAI, APIError, RateLimitError, AuthenticationError 
from PIL import Image, UnidentifiedImageError
import io
from tqdm import tqdm
import math
from dotenv import load_dotenv 
from concurrent.futures import ThreadPoolExecutor, as_completed
import bisect

# --- Configuration ---


# 班級整體照片資料夾路徑 
CLASSROOM_IMAGES_FOLDER = r'C:\Users\User\Desktop\test\student_full_classroom\0817_english_class'

# 【新】老師視角照片資料夾路徑 (從學生視角拍攝老師在黑板前的照片) - 可選
TEACHER_POSITION_JSON = r'C:\Users\User\Desktop\test\teacher_position\0817_position.json'

#課堂情況json
CLASSROOM_STATE_JSON_PATH = r'C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0817\老師上課\TXT\0817_processed.json'

# 【新】時間軸對齊設定 (格式: "HH:MM:SS")
CLASSROOM_START_TIME_OFFSET = "00:00:00"  # 班級照片的起始時間，設定為空字串 "" 表示不篩選
TEACHER_JSON_START_TIME_OFFSET ="00:00:00" # 老師位置資料的起始時間，設定為空字串 "" 表示不篩選

# ... (您原有的其他設定) ...

JSON_OUTPUT_FOLDER = "SynologyDrive\json_behavior"


JSON_FILENAME_TEMPLATE = "student_{student_id}_behavior_report_{timestamp}.json"
SAMPLING_RATE = 3  
IMAGE_DETAIL_LEVEL = "low" # 圖片解析度 ('low' 或 'high')，low 可大幅降低成本
MAX_TOKENS_VISION_COMPLETION = 2500
MAX_TOKENS_SUMMARY_COMPLETION = 1000 # 個性化總結的 token
BEHAVIOR_CONFIDENCE_THRESHOLD_FILTER = 0.96 # 行為信度過濾閾值 (例如 70%)
IMAGES_PER_API_CALL = 10 # 一次API調用處理的圖片數量
BEHAVIOR_SIMILARITY_CONFIDENCE_THRESHOLD = 0.15 # 過濾相似連續行為的信度差異閾值
API_RETRY_DELAY_SECONDS = 10

# ==========================================================

# --- 標準行為分類與定義 (核心辭典，來自圖一) ---
STANDARD_BEHAVIOR_CATEGORIES = {
    "視線": [
        {"label": "目視教師", "definition": "學生的頭部與視線明確聚焦於教室【前方】的【老師所在】區域。當老師位置已知且學生視線方向與之匹配時，此標籤的優先級最高。"},
        {"label": "目視黑板", "definition": "學生的頭部與視線明確聚焦於教室【前方】的【非老師所在】的黑板或螢幕區域。當老師位置未知，或學生視線明確未朝向老師時，這是面向前方的預設專注行為。"}, 
        {"label": "目視書本/筆記", "definition": "學生的視線向量【主要朝下】，聚焦於其個人桌面工作區內的書本、講義或筆記本。...（保留黃金準則）... **【注意】**：如果學生同時在進行『做筆記』，根據『動作優先』原則，你應將『做筆記』作為主要標籤。"},
        {"label": "目視同學", "definition": "學生的頭部與視線【脫離朝向前方或桌面的軸線】，明確轉向教室中的側面或後方，與其他同學處於同一視線水平。**【核心情境判斷規則】**：此行為的性質【必須】根據【課堂狀態】來決定：(1) **在需要安靜聽講的狀態下** (如『文法講解』、『閱讀分析』)，此行為應被理解為【非任務相關】的社交互動或注意力分散。 (2) **在允許互動的狀態下** (如『習題檢討』、『分組討論』)，此行為可能涉及【任務相關】的交流（如交換考卷、討論問題），你應在描述中體現這一點。"},
        {"label": "目視他處", "definition": "【備用排除性標籤】。當你已確認學生的視線【不】符合『目視教師』、『目視黑板』、『目視書本/筆記』或『目視同學』的任何一項明確定義時，才使用此標籤。它捕捉的是失去焦點的狀態，例如：【抬頭看天花板】、【轉頭看沒有同學及老師的地方】。"}     
    ],
    "肢體(手部)": [
        {"label": "做筆記", "definition": "【核心證據：手持筆且筆尖接觸紙張】。只有當你清晰地看到學生【手持筆】並在紙上進行書寫時，才可以使用此標籤。**【最高優先級】**：只要滿足此行為的核心證據，其優先級就【高於】所有靜態的視線行為，例如『目視書本/筆記』。在這種情況下，『做筆記』應作為主要行為被標註。"},
        {"label": "翻書", "definition": "學生手部正在主動翻閱、移動書本。"},
        {"label": "觸摸臉部", "definition": "學生【非支撐性地】用手短暫觸碰或摩擦自己的臉部、鼻子、嘴巴或眼睛。此行為區別於『托腮』的持續性支撐動作。"},
        {"label": "觸摸頭髮", "definition": "學生用手觸摸、撥弄或整理自己的頭髮。注意：即使手臂抬得較高，只要手部的主要動作是與頭髮互動，就【必須】使用此標籤，而不是『舉手』類標籤。"}
    ],
    "身體姿態": [
        {"label": "坐姿直立", "definition": "學生上半身軀幹基本垂直於地面，或輕微前傾。"},
        {"label": "身體前傾", "definition": "學生上半身軀幹明顯向前彎曲，靠近桌面。此行為描述的是一種【清醒狀態下】的姿態。如果學生頭部接觸桌面或手臂，應【優先使用】『趴睡』標籤。"},
        {"label": "身體後靠", "definition": "學生背部倚靠在椅背上。"},
        {"label": "低頭(非學習)", "definition": "【極其嚴格的排除性標籤】。僅在你能夠【極度確信地】觀察到以下【全部】條件時才可使用：1. 學生頭部明顯低垂。2. 其視線【明確沒有】朝向任何學習材料（例如，看向地面、自己的懷中或空無一物的桌面）。3. 其手部沒有在進行任何學習相關操作。"},
        {"label": "趴睡", "definition": "學生將頭部【枕於】手臂或桌面上，呈現明確的休息或睡眠狀態。**【最高優先級】**：只要觀察到頭部接觸桌面或手臂的休息姿態，此標籤的優先級【高於】所有其他學習相關標籤（如『做筆記』、『目視書本』）。"},
        {"label": "托腮", "definition": "【核心定義：手部對頭部提供持續性支撐】。學生使用一隻或兩隻手的手掌、拳頭或手臂，支撐其下巴、臉頰或頭部的重量。這是一個純粹的物理姿態描述，不包含任何意圖推斷。"}
    ],
    "互動": [
        {"label": "主動舉手", "definition": "學生舉起一隻手，意圖提問或回答問題。**【三大核心物理證據，必須同時滿足】**：(1) 手臂向上伸展，手部明顯高於肩膀。(2) 手掌形態為張開朝前或中性放鬆，【嚴禁】手指指向特定方向或揮舞。(3) 該動作具有一定的持續性（非瞬間劃過）。**【情境觸發】**：此行為最常發生在【老師單向授課】的狀態下（如『文法/句型講解』、『閱讀/文章分析』），代表學生的自發性提問或補充。**【絕對排除】**：任何手部接觸頭部、與同學互動的手勢、指向性的動作，都【不允許】標記為此行為。"},
        {"label": "被動舉手", "definition": "學生舉手以回應老師的群體性指令（如投票、調查）。**【核心視覺證據】**：通常是多數學生同時舉手，姿態可能較為放鬆，手臂不必完全伸直。**【情境觸發】**：此行為最常發生在【老師與學生互動】的狀態下（如『課堂問答/互動』、『習題/考卷檢討』），代表學生回應老師的指令或提問。**【關鍵區分】**：此標籤的判斷【高度依賴】課堂情境和群體性動作。如果情境不匹配，應避免使用此標籤。"}
    ],
    "其他狀態": [
        {"label": "喝水", "definition": "學生正在使用容器飲用液體。"},
        {"label": "飲食", "definition": "學生正在食用固體食物。"},
        {"label": "玩弄手部／文具", "definition": "學生手部在進行與學習無關的重複性小動作，例如玩手指、轉筆、玩弄橡皮擦等"},
        # {"label": "整理書包", "definition": "學生正在整理書包、桌面文具。"},
        {"label": "被遮擋/無法判斷", "definition": "【最終備用標籤】。因遮擋、模糊或角度問題，無法清晰識別學生的主要行為時，【必須】使用此標籤。"}
    ]
}
# 自動從新結構生成有效的標籤列表
VALID_BEHAVIOR_LABELS = [item['label'] for category in STANDARD_BEHAVIOR_CATEGORIES.values() for item in category]

BEHAVIOR_CODES = {
    # 視線 (不變)
    "V_TCH": "目視教師", "V_BRD": "目視黑板", "V_BOK": "目視書本/筆記",
    "V_CLS": "目視同學", "V_ELS": "目視他處",
    
    # 肢體(手部) (移除 H_FLIP)
    "H_NOT": "做筆記",
    "H_PLAY_HW": "玩弄手部/文具",
    "H_TOUCH_F": "觸摸臉部",
    "H_TOUCH_H": "觸摸頭髮",
    
    # 身體姿態 (新增 P_THK - P for Posture, THK for Thinking)
    "P_STR": "坐姿直立", "P_LEAN": "身體前傾", "P_BACK": "身體後靠",
    "P_DWN": "低頭(非學習)", "P_SLP": "趴睡",
    "P_THK": "托腮", # <--- 新增
    
    # 互動 (將 I_HND 拆分為主動/被動)
    "I_HND_A": "主動舉手", # A for Active
    "I_HND_P": "被動舉手", # P for Passive
    
    # 其他狀態 (不變)
    "O_DRK_W": "喝水",
    "O_EAT_S": "飲食",
    # "O_TDY": "整理個人物品",
    "O_UNK": "被遮擋/無法判斷"
}

# 自動生成反向查找字典，用於本地解碼
CODE_TO_BEHAVIOR = {code: label for code, label in BEHAVIOR_CODES.items()}
BEHAVIOR_TO_CODE = {label: code for code, label in BEHAVIOR_CODES.items()}

# AI返回標籤到標準標籤的映射規則 (與新標籤對齊)
BEHAVIOR_MAPPING_RULES = {
    "目視桌面/教材": "目視書本", "目視桌面": "目視書本", "看書": "目視書本",
    "書寫/做筆記": "筆記", "動手操作-書寫/做筆記": "筆記",
    "視覺專注-閱讀書本/講義": "目視書本",
    "視覺專注-看老師/黑板方向": "目視教師", "目視黑板/老師": "目視教師",
    "看老師": "目視教師", "看黑板": "目視黑板",
    # --- ↓↓↓ 【修改點】更新映射規則以對應新標籤 ↓↓↓ ---
    "玩弄物品": "玩弄手部/文具", # 將模糊的舊標籤對應到最可能的新標籤
    "非任務相關動作-玩弄物品(筆等)": "玩弄手部/文具",
    # 移除了"非任務相關動作-觸摸臉部/頭髮"，鼓勵AI直接使用更精確的新標籤
    # "非任務相關動作-整理物品": "整理個人物品",
    "社交互動-與同學互動": "目視同學",
    "趴睡/休息": "趴睡",
    "低頭/伏案(非睡)": "低頭",
}

BEHAVIOR_VALENCE_MAP = {
    "正向": [
        "做筆記",
        "主動舉手", # <-- 修改
        "目視教師",
        "目視黑板",
        "目視書本/筆記",
    ],
    "負向": [
        "趴睡",
        "玩弄手部/文具",
        "觸摸臉部",
        "觸摸頭髮",
        "目視他處",
        "目視同學"
    ],
    "中性": [
        "身體前傾",
        "坐姿直立",
        "身體後靠",
        "喝水",
        "飲食",       
        "玩弄手部／文具",
        "翻書",
        "低頭(非學習)",
        # "整理個人物品",     
        "被遮擋/無法判斷",
        "托腮", # <-- 新增
        "被動舉手"  # <-- 新增
    ]
}

LABEL_TO_VALENCE = {
    label: valence 
    for valence, labels in BEHAVIOR_VALENCE_MAP.items() 
    for label in labels
}

# ---------------------------
#  API 金鑰配置
# ---------------------------
load_dotenv()

AZURE_API_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
# 從環境變數讀取部署名稱
VISION_DEPLOYMENT_NAME = os.getenv("CHAT_COMPLETION_NAME")
TEXT_DEPLOYMENT_NAME = os.getenv("CHAT_COMPLETION_NAME")
SUMMARY_DEPLOYMENT_NAME = os.getenv("CHAT_COMPLETION_NAME")

# 檢查必要的 Azure 配置是否存在
if not all([AZURE_API_KEY, AZURE_ENDPOINT, VISION_DEPLOYMENT_NAME, SUMMARY_DEPLOYMENT_NAME]):
    print("錯誤：缺少必要的 Azure OpenAI 環境變數。")
    print("請檢查您的 .env 檔案是否包含 AZURE_OPENAI_KEY, AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_VISION_DEPLOYMENT_NAME, 和 AZURE_OPENAI_SUMMARY_DEPLOYMENT_NAME。")
    exit()

try:
    # ### 【修改 3】: 初始化 AzureOpenAI Client ###
    client = AzureOpenAI(
        api_key=AZURE_API_KEY,
        azure_endpoint=AZURE_ENDPOINT,
        api_version="2024-02-01"  # 使用一個穩定的 API 版本
    )
    client.models.list() # 嘗試調用一個簡單的API來驗證配置
    print("Azure OpenAI client 初始化並驗證成功。")
    print(f"  - 視覺分析將使用部署: '{VISION_DEPLOYMENT_NAME}'")
    print(f"  - 個性化總結將使用部署: '{SUMMARY_DEPLOYMENT_NAME}'")
except AuthenticationError: print("錯誤：Azure OpenAI API 金鑰或端點無效，認證失敗。"); exit()
except Exception as e: print(f"初始化 Azure OpenAI Client 時發生錯誤: {e}"); exit()

# ---------------------------
# 輔助函數
# ---------------------------
def find_state_for_timestamp(target_timestamp, classroom_states):
    """
    【v4.0 穩定版】根據時間戳，查找對應的課堂狀態標籤。
    使用線性查找以確保最高的準確性和可靠性。
    """
    if not classroom_states:
        return "未知"
    
    # 為了除錯，只打印一次查找的詳細信息
    if not hasattr(find_state_for_timestamp, "has_logged_first_search"):
        print("\n" + "-"*20 + " [除錯日誌 - 首次查找課堂狀態] " + "-"*20)
        print(f"  - 正在用第一張照片的時間戳进行匹配...")
        print(f"  - 目標時間戳 (Target Timestamp): {target_timestamp}")
        if classroom_states:
            first_state = classroom_states[0]
            print(f"  - 正在检查第一個時間區間: 從 {first_state.get('start_time_td')} 到 {first_state.get('end_time_td')}")
        print("-" * 69 + "\n")
        find_state_for_timestamp.has_logged_first_search = True

    # 遍歷每一個課堂狀態的時間區間
    for state in classroom_states:
        start_td = state.get("start_time_td")
        end_td = state.get("end_time_td")
        
        # 確保這個區間的時間數據是有效的
        if start_td and end_td:
            # 檢查目標時間戳是否落在 [開始時間, 結束時間] 這個閉区间内
            if start_td <= target_timestamp <= end_td:
                return state["classroom_state"]
    
    # 如果遍歷完所有區間都沒找到，說明時間戳確實超出了範圍
    return "未知"

def parse_time_offset(time_str):
    """將 "HH:MM:SS" 格式的字串轉換為 timedelta 物件"""
    if not time_str or not isinstance(time_str, str):
        return None
    try:
        h, m, s = map(int, time_str.split(':'))
        return datetime.timedelta(hours=h, minutes=m, seconds=s)
    except (ValueError, TypeError):
        print(f"警告：時間偏移量 '{time_str}' 格式不正確，應為 'HH:MM:SS'。將忽略此設定。")
        return None

def get_valid_input(prompt_message):
    while True:
        user_input = input(prompt_message).strip()
        if user_input: return user_input
        print("錯誤：輸入不能為空，請重新輸入。")

def get_valid_folder_path(prompt_message, is_optional=False): # 新增 is_optional 參數
    while True:
        folder_path = input(prompt_message).strip().strip('"')
        if not folder_path and is_optional:
            return None # 如果是可選的且用戶未輸入，返回 None
        if os.path.isdir(folder_path):
            return folder_path
        print(f"錯誤：路徑 '{folder_path}' 不是一個有效的資料夾，請重新輸入。")

def get_timestamp_from_filename(filename):
    """
    【v3.0 兼容版】从档名解析时间戳。
    能同时处理多种常见格式。
    """
    # 模式一：最优先，精确匹配 HH-MM-SS-ms.jpg 格式 (常见于学生照片)
    match = re.search(r'^(\d{2})-(\d{2})-(\d{2})-(\d{3})\.(jpg|jpeg|png|webp)$', filename, re.IGNORECASE)
    if match:
        try:
            h, m, s, ms, ext = match.groups()
            return datetime.timedelta(hours=int(h), minutes=int(m), seconds=int(s), milliseconds=int(ms))
        except (ValueError, IndexError):
            pass

    # 模式二：匹配包含 ...HH-MM-SS-ms... 的通用格式
    match = re.search(r'(\d{2})-(\d{2})-(\d{2})-(\d{3})', filename)
    if match:
        try:
            h, m, s, ms = map(int, match.groups())
            return datetime.timedelta(hours=h, minutes=m, seconds=s, milliseconds=ms)
        except (ValueError, IndexError):
            pass

    # 模式三：匹配包含 ...h...m...s 的通用格式
    match = re.search(r'(\d+)h(\d{2})m(\d{2})s', filename)
    if match:
        try:
            h, m, s = map(int, match.groups())
            return datetime.timedelta(hours=h, minutes=m, seconds=s)
        except (ValueError, IndexError):
            pass

    # 如果所有格式都沒匹配成功，最终返回 None
    return None

def encode_image_to_base64(image_path, max_size_kb=512, target_quality=75):
    try:
        with Image.open(image_path) as img: img.verify()
        with Image.open(image_path) as img:
            if img.mode == 'RGBA' or img.mode == 'P': img = img.convert('RGB')
            current_quality = target_quality
            output_buffer = io.BytesIO()
            temp_img = img.copy()
            temp_img.save(output_buffer, format="JPEG", quality=current_quality)
            current_size_kb = output_buffer.tell() / 1024

            if current_size_kb > max_size_kb:
                scale_factor = math.sqrt(max_size_kb / current_size_kb)
                new_width = int(temp_img.width * scale_factor * 0.9)
                new_height = int(temp_img.height * scale_factor * 0.9)
                if new_width >= 50 and new_height >= 50: # 最小尺寸限制
                    print(f"    圖片 {os.path.basename(image_path)} ({current_size_kb:.1f} KB) 過大，縮放並調整質量...")
                    temp_img = temp_img.resize((new_width, new_height), Image.Resampling.LANCZOS)
                    output_buffer = io.BytesIO()
                    temp_img.save(output_buffer, format="JPEG", quality=max(current_quality - 15, 40)) # 質量可以降更多
                else:
                    print(f"    警告: 圖片 {os.path.basename(image_path)} 縮放後過小，可能影響質量。使用較低質量。")
                    output_buffer = io.BytesIO()
                    img.save(output_buffer, format="JPEG", quality=max(current_quality // 2, 30) )


            output_buffer.seek(0)
            binary_data = output_buffer.getvalue()
            base64_encoded_data = base64.b64encode(binary_data)
            return f"data:image/jpeg;base64,{base64_encoded_data.decode('utf-8')}"
    except Exception as e: print(f"錯誤：處理圖片 '{image_path}': {e}"); return None

def load_classroom_states(json_path):
    """【全新】讀取預處理好的課堂狀態時間軸 JSON 檔案。"""
    if not json_path or not os.path.isfile(json_path):
        print("提示：未提供或找不到課堂狀態 JSON 檔案。將不使用課堂情境。")
        return None  # 返回 None 以便更明確地判斷失敗
    
    print(f"正在讀取課堂狀態時間軸: {json_path}...")
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f) # <--- 關鍵修正：將 f 作為參數傳入
            
            # 將時間字串預先轉換為 timedelta 物件以便快速比較
            timeline = data.get("timeline", [])
            for state in timeline:
                state["start_time_td"] = parse_time_offset(state["start_time"])
                state["end_time_td"] = parse_time_offset(state["end_time"])
            print(f"成功載入 {len(timeline)} 個課堂狀態階段。")
            return timeline
    except Exception as e:
        print(f"❌ 錯誤：讀取或解析課堂狀態 JSON 時發生問題: {e}")
        return None # 返回 None

def get_behavior_sequence_analysis_system_prompt(student_position, has_teacher_context):
    """
    【v14.0 - 前後排動態協議版】為序列分析任務生成動-態的、具備情境感知能力的系統提示。
    此版本的核心是能夠根據學生的【前後排位置】，動態生成完全不同的分析協議。
    """
    # --- 1. 預先生成靜態的行為編碼表 (此部分邏輯不變) ---
    behavior_table_for_prompt = "\n\n**【學習行為編碼表】**\n你 **必須** 且 **只能** 從以下列表的「編碼 (Code)」中選擇行為進行標註。...\n"
    for code, label in BEHAVIOR_CODES.items():
        definition = ""
        for category in STANDARD_BEHAVIOR_CATEGORIES.values():
            for item in category:
                if item['label'] == label:
                    definition = item['definition']
                    break
            if definition:
                break
        behavior_table_for_prompt += f"*   **`{code}`**: {label} - {definition}\n"
    behavior_table_for_prompt += "\n*如果行為因任何原因無法清晰判斷，請 **必須** 使用 **`O_UNK`** 編碼。*\n"

    # --- 2. 【核心升級】根據學生位置，動態生成【主攝影機規則】 ---
    # (此部分邏輯不變，但作為後續推理的基礎)
    primary_camera_rules = ""
    if "左" in student_position:
        primary_camera_rules = """
**準則 1A: 主要分析視角 (左側攝影機)**
*   【學生個人照片序列】來自教室前方的【左側】攝影機。
*   對於這位坐在左側的學生，【正面朝向鏡頭】僅代表他在看教室的【左前方】。為了看向【教室正前方中心】，他的頭部必須輕微地朝向他自己的【右側】轉動。
"""
    elif "右" in student_position:
        primary_camera_rules = """
**準則 1A: 主要分析視角 (右側攝影機)**
*   【學生個人照片序列】來自教室前方的【右側】攝影機。
*   對於這位坐在右側的學生，【正面朝向鏡頭】僅代表他在看教室的【右前方】。為了看向【教室正前方中心】，他的頭部必須輕微地朝向他自己的【左側】轉動。
"""
    else: # 預設為中間
        primary_camera_rules = """
**準則 1A: 主要分析視角 (中間攝影機)**
*   【學生個人照片序列】來自教室前方的【中間】攝影機。
*   當這位學生臉部【正面朝向鏡頭】時，即代表其視線正朝向教室的【正前方中心】。
"""

    # --- 3. 【全新架構】根據學生【排數】，動態生成【分析協議】 ---
    row_specific_protocol = ""
    # 檢查學生是在前排還是後排
    is_front_row = any(keyword in student_position for keyword in ["第一", "第二"])
    is_rear_row = any(keyword in student_position for keyword in ["第三", "第四"])

    if is_rear_row:
        # --- 這是為後排學生設計的、基於「邏輯推斷」的協議 ---
        row_specific_protocol = """
**【第二部分：後排學生分析協議 (絕對推斷模式)】**

由於該學生 (`{student_position}`) 位於教室後方，視覺證據極其有限，你必須作為一名頂級偵探，遵循以下不可動搖的推斷鐵則。

*   **鐵則 A：低頭行為的「絕對學習推定」原則 (The Absolute Presumption of Learning)**
    *   對於後排學生，其桌面學習材料【總是】被假定為存在，即使你看不清。
    *   因此，只要你觀察到學生【頭部低垂】，你就**必須、無條件地、最優先地**將其主要行為判定為 `V_BOK` (目視書本/筆記)。
    *   **【疲勞姿態豁免條款】**：學生在低頭看書時，常常會伴隨各種旨在維持專注或對抗疲勞的姿態，例如**『揉眼睛』、『搔頭』、『手扶額頭』、『托腮』**。這些動作【絕對不允許】被用來推翻『目視書本/筆記』的核心判斷。你應將它們視為【學習過程的一部分】，並可將其作為次要標籤疊加。
    *   **【推翻條件】**：只有在你擁有【極其明確的、不可辯駁的】視覺證據，證明他正在進行以下**兩種**活動之一時，才能推翻此推定：(1) **明確的社交互動** (`V_CLS`)；(2) **明確的隱蔽手部動作** (`H_PLAY_HW`)。

*   **鐵則 B：前方視線的「教師優先」原則**
    *   由於後排學生視野開闊，一個朝向教室前方的視線，其目標是老師的概率極高。
    *   因此，只要學生的視線是朝向【教室前方】的任何區域，你就應**優先**將其標註為 `V_TCH` (目視教師)，其次才是 `V_BRD` (目視黑板)。
"""
    else: # 預設為前排規則
        # --- 這是為前排學生設計的、基於「視覺證據」的協議 ---
        row_specific_protocol = """
**【第二部分：前排學生分析協議 (高證據模式)】**

由於該學生 (`{student_position}`) 位於教室前方，視覺證據清晰，你的所有判斷都必須基於【嚴格的視覺證據】。

*   **證據準則 A：低頭行為的分診決策樹**
    *   當你觀察到學生【頭部低垂】時，你必須遵循以下檢查順序：
        1.  **第一優先：** 桌上是否有可見的學習材料，且視線朝向它？ -> **是** -> `V_BOK` (目視書本/筆記)。
        2.  **第二優先：** 身體或視線是否明確轉向鄰座同學？ -> **是** -> `V_CLS` (目視同學)。
        3.  **第三優先：** 手部是否有隱蔽動作（如操作手機）？ -> **是** -> `H_PLAY_HW` (玩弄手部/文具)。
        4.  **最後選項：** 僅在以上情況都明確排除後，才使用 `P_DWN` (低頭非學習)。

*   **證據準則 B：前方視線的精確匹配**
    *   當學生視線朝向【教室前方】時，你必須嚴格結合【老師位置情境】進行幾何匹配：
        *   視線方向與老師位置**高度匹配**？ -> `V_TCH` (目視教師)。
        *   視線方向在前方，但與老師位置有**偏差**？ -> `V_BRD` (目視黑板)。

"""

    # --- 4. 準備其他通用模塊和資訊 ---
    teacher_context_header = "5.  **【老師位置文字情境】**: 用於交叉驗證你視線判斷的輔助數據。" if has_teacher_context else ""
    universal_rules = """
---
**【第三部分：通用行為判斷準則 (全體適用)】**

在你通過【第二部分】的協議確定了學生的主要視線方向或狀態後，你必須遵循以下通用規則來最終確定標籤組合。

*   **準則 A：【舉手行為的嚴格過-濾器】**
    *   當觀察到【手臂抬起】時，**嚴禁**直接標註為舉手。必須先通過以下排除法檢查：
        1.  手是在【指向】、【揮舞】或與同學手勢互動嗎？ -> **是** -> 標註為 `V_CLS`，過濾結束。
        2.  手是在【觸摸頭部/頭髮】嗎？ -> **是** -> 標註為 `H_TOUCH_H` 或 `H_TOUCH_F`，過濾結束。
    *   只有通過了上述過濾，才能根據【課堂狀態】判斷是 `I_ND_A` (主動) 還是 `I_HND_P` (被動)。

*   **準則 B：【標註主要行為與輔助姿態 (動作優先原則)】**
    *   這是在確定最終標籤組合時的**核心決策流程**：
    1.  **首先，尋找最核心的【主動動作】**：
        *   學生是否在**做筆記 (`H_NOT`)**？ -> **是** -> 將 `H_NOT` 作為**第一個、最主要**的標籤。
        *   學生是否在**玩弄手部/文具 (`H_PLAY_HW`)**？ -> **是** -> 將 `H_PLAY_HW` 作為**第一個、最主要**的標籤。
    2.  **然後，標註對應的【輔助視線】（如果沒有主動動作）**：
        *   如果**沒有**檢測到 `H_NOT` 或 `H_PLAY_HW`，那麼**才把**你在【第二部分】中判斷的視線行為（`V_TCH`, `V_BOK` 等）作為**主要標籤**。
        *   如果**已經**標註了 `H_NOT`，那麼 `V_BOK` (目視書本) 就是一個不言而喻的次要行為，可以省略或作為第二個標籤。
    3.  **最後，疊加一個【通用身體姿態】**：
        *   在確定了主要行為後，再疊加一個最能描述學生整體狀態的姿態標籤，例如 `P_STR`(坐姿直立), `P_LEAN`(身體前傾), `P_THK`(托腮) 等。

*   **準則 C：【臉部表情輔助驗證】**
    *   如果面部可見，可觀察表情（微笑、困惑等）來驗證你的判斷。
"""
    # --- 5. 生成最終的、完整的 Prompt ---
    return f"""
「你是一位世界頂尖的教育分析師，精通電腦視覺、空間幾何推理與心理學。你的任務是綜合所有給定的物理與情境資訊，對學生的學習行為進行最精準、最客觀的標註。」

**【你收到的資訊來源】**
1.  **【學生個人照片序列】**: 你的主要分析對象，其來源攝影機是動態的。
2.  **【班級整體照片】**: 你的情境驗證工具，其來源攝影機是固定的。
3.  **【課堂狀態】**: 判斷行為動機的核心上下文。
4.  **【學生座位】**: `{student_position}`。
{teacher_context_header}

---
**【第一部分：多機位攝影機系統與物理準則】**

你必須理解你正在處理一個【多機位系統】。這是所有判斷的基礎。

{primary_camera_rules}

**準則 1B: 情境驗證視角 (固定為中間攝影機)**
*   用於情境驗證的【班級整體照片】**永遠**來自教室前方的【中間】攝影機。
*   當你需要分析【班級整體照片】時，你必須使用【中間攝影機】的視角規則來進行判斷。

**準則 2: 影像鏡像轉換 (全域適用)**
*   所有攝影機畫面都是鏡像的：照片中的**左側** => 教室中的**右側**；照片中的**右側** => 教室中的**左側**。

---
{row_specific_protocol}
---
{universal_rules}
---
{behavior_table_for_prompt}
---
**【第四部分：輸出格式要求 - 嚴格版】**
*   你的回答**必須**是一個結構完整的、單一的 JSON 物件。
*   `behavior_category` 欄位的值**必須**是一個包含 1 到 2 個編碼字串的**陣列 (Array)**。
*   你的最外層輸出**只能包含** `sequence_analysis_confidence` 和 `per_image_highlights` 這兩個鍵。
*   在 `per_image_highlights` 的每個物件中，**只能包含** `image_index_in_sequence`, `context_description`, `behavior_category`, 和 `confidence` 這四個鍵。
*   **絕對不允許**在任何層級輸出任何額外的、未經請求的鍵或註解。

**【輸出 JSON 格式範例】**
```json
{{
  "sequence_analysis_confidence": 0.97,
  "per_image_highlights": [
    {{
      "image_index_in_sequence": 0,
      "context_description": "學生座位在第四排右側，應用後排分析協議。頭部低垂，根據『學習優先』原則，判定為目視書本。",
      "behavior_category": ["V_BOK", "P_STR"],
      "confidence": 0.95
    }},
    {{
      "image_index_in_sequence": 1,
      "context_description": "學生座位在第一排中間，應用前排分析協議。學生臉部正對鏡頭，老師位置在中間，根據視覺證據匹配為目視教師。",
      "behavior_category": ["V_TCH"],
      "confidence": 0.99
    }}
  ]
}}
```"""

def default_error_result_structure():
    return { "error": "分析失敗或無有效數據", "sequence_analysis_confidence": 0.0, "sequence_summary": "未能生成序列總結。", "dominant_sustained_behaviors": [], "significant_behavior_shifts": [], "per_image_highlights": [], "general_sequence_atmosphere_hint": "未知" }

def clean_analysis_json(raw_analysis):
    """
    清理從 API 返回的分析 JSON，只保留我們需要的欄位。
    這是一個防禦性措施，用來處理微調模型的「幻覺」問題。
    """
    if not isinstance(raw_analysis, dict):
        return default_error_result_structure()

    # 定義合法的鍵
    allowed_top_level_keys = {"sequence_analysis_confidence", "per_image_highlights"}
    allowed_highlight_keys = {"image_index_in_sequence", "context_description", "behavior_category", "confidence"}

    cleaned_analysis = {}
    
    # 1. 清理最外層的鍵
    for key, value in raw_analysis.items():
        if key in allowed_top_level_keys:
            cleaned_analysis[key] = value

    # 2. 檢查並清理 per_image_highlights 列表
    if "per_image_highlights" in cleaned_analysis and isinstance(cleaned_analysis["per_image_highlights"], list):
        cleaned_highlights = []
        for raw_highlight in cleaned_analysis["per_image_highlights"]:
            if not isinstance(raw_highlight, dict):
                continue # 如果列表中的元素不是字典，就跳過它
            
            cleaned_highlight = {}
            for key, value in raw_highlight.items():
                if key in allowed_highlight_keys:
                    cleaned_highlight[key] = value
            
            # 確保必要的鍵存在，即使為空
            for required_key in allowed_highlight_keys:
                if required_key not in cleaned_highlight:
                    cleaned_highlight[required_key] = None

            cleaned_highlights.append(cleaned_highlight)
        
        cleaned_analysis["per_image_highlights"] = cleaned_highlights

    # 3. 如果最外層缺少必要的鍵，補上預設值
    if "sequence_analysis_confidence" not in cleaned_analysis:
        cleaned_analysis["sequence_analysis_confidence"] = 0.0
    if "per_image_highlights" not in cleaned_analysis:
        cleaned_analysis["per_image_highlights"] = []

    return cleaned_analysis

def analyze_student_behavior_from_images_sequence(student_image_paths, teacher_position_text, classroom_view_image_path, classroom_state, image_filenames_batch, openai_client, student_id, student_position):
    """
    【v8.1 穩定版】分析單一圖片批次，使用預處理好的「課堂狀態」標籤。
    此版本包含了完整的錯誤處理、JSON清理和穩健性增強。
    """
    if not student_image_paths:
        return default_error_result_structure() # 直接返回錯誤結構，而不是字典
    
    user_message_content = []
    
    # 1. 編碼學生個人圖片序列 (使用配置中的解析度)
    encoded_student_images = []
    for img_path in student_image_paths:
        b64_img = encode_image_to_base64(img_path)
        if b64_img:
            encoded_student_images.append({
                "type": "image_url", 
                "image_url": {"url": b64_img, "detail": IMAGE_DETAIL_LEVEL}
            })

    if not encoded_student_images:
        return default_error_result_structure() # 直接返回錯誤結構
        
    # 2. 編碼班級整體照片 (使用配置中的解析度)
    encoded_classroom_view_image = None
    if classroom_view_image_path:
        b64_img = encode_image_to_base64(classroom_view_image_path)
        if b64_img:
            encoded_classroom_view_image = {
                "type": "image_url", 
                "image_url": {"url": b64_img, "detail": IMAGE_DETAIL_LEVEL}
            }

    # 3. 組合完整的請求體，包含所有情境信息
    user_message_content.append({"type": "text", "text": f"請根據系統提示中的偵探任務，分析學生「{student_id}」的行為。學生照片序列的文件名（供您參考）為: {', '.join(image_filenames_batch)}。"})
    
    # 加入課堂狀態標籤
    state_context_prompt = f"【課堂狀態】: {classroom_state}"
    user_message_content.append({"type": "text", "text": state_context_prompt})

    # 加入老師位置情境
    teacher_context_prompt = f"【老師位置情境】根據預先分析，在此時間段，老師的位置在教室前方的「{teacher_position_text}」。請以此作為判斷『目視教師』的核心依據。"
    user_message_content.append({"type": "text", "text": teacher_context_prompt})
    
    # 依序加入班級照片和學生照片
    if encoded_classroom_view_image:
        user_message_content.append({"type": "text", "text": "【班級整體照片】(用於定位學生和觀察整體氛圍)"})
        user_message_content.append(encoded_classroom_view_image)
    user_message_content.append({"type": "text", "text": "【學生個人照片序列】(主要分析對象)"})
    user_message_content.extend(encoded_student_images)

    # 4. 獲取新的、具備情境感知能力的系統提示
    system_prompt_content = get_behavior_sequence_analysis_system_prompt(student_position, bool(teacher_position_text and teacher_position_text != "未知"))

    # 5. API 呼叫與錯誤處理
    retry_attempts = 2
    for attempt in range(retry_attempts + 1):
        raw_content = "" # 初始化 raw_content 以免在 except 區塊中引用未定義變數
        try:
            has_teacher_context = bool(teacher_position_text and teacher_position_text != "未知")
            has_classroom_image = bool(encoded_classroom_view_image)
            has_state = classroom_state != "未知"

            print(f"  正在向 {VISION_DEPLOYMENT_NAME} 發送請求 (學生: {len(encoded_student_images)}, 課堂狀態: {'有' if has_state else '無'}, 老師位置: {'有' if has_teacher_context else '無'}, 班級照片: {'有' if has_classroom_image else '無'})...")
            
            response = openai_client.chat.completions.create(
                model=VISION_DEPLOYMENT_NAME, 
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": system_prompt_content},
                    {"role": "user", "content": user_message_content}
                ],
                max_tokens=MAX_TOKENS_VISION_COMPLETION,  # gpt-4.1
                temperature=0.05 # gpt-4.1
                # max_completion_tokens=MAX_TOKENS_VISION_COMPLETION, #gpt-5
                
            )
            raw_content = response.choices[0].message.content
            
            # --- ✅【核心修正點】---
            # 步驟 1: 解析原始回應
            analysis_result_raw = json.loads(raw_content)
            
            # 步驟 2: 清理可能包含幻覺的 JSON，確保格式正確
            analysis_result = clean_analysis_json(analysis_result_raw)
            # --- ✅【修正結束】---

            # 本地解碼，將行為編碼轉換回中文標籤列表
            if "per_image_highlights" in analysis_result and isinstance(analysis_result["per_image_highlights"], list):
                for hl in analysis_result["per_image_highlights"]:
                    # 增加穩健性檢查，防止因清理後的 None 值導致錯誤
                    if not hl or not isinstance(hl, dict) or "behavior_category" not in hl:
                        continue
                    
                    codes = hl.get("behavior_category")
                    if not codes: # 處理 behavior_category 為 None 的情況
                        continue

                    if not isinstance(codes, list):
                        codes = [codes]
                    
                    # 過濾掉可能是 None 的 code
                    decoded_behaviors = [CODE_TO_BEHAVIOR.get(code, f"未知編碼({code})") for code in codes if code]
                    hl["behavior_category"] = decoded_behaviors
            
            # 補上空的摘要欄位以保持格式統一 (這一步可以保留，也可以移除，因為 clean_analysis_json 會確保結構)
            if "sequence_summary" not in analysis_result:
                analysis_result["sequence_summary"] = "已設定為精簡模式，此欄位由本地生成。"

            return analysis_result

        except json.JSONDecodeError:
            # 這裡的邏輯是正確的：如果 JSON 本身語法錯誤，就記錄並重試
            print(f"    錯誤：無法解析JSON (第 {attempt+1} 次嘗試)。回應: {raw_content[:500]}...");
            time.sleep(5) # 增加短暫延遲
        except RateLimitError:
            print(f"  警告：API速率限制，等待 {API_RETRY_DELAY_SECONDS}s 後重試...");
            time.sleep(API_RETRY_DELAY_SECONDS)
        except APIError as e:
            wait_time = 10 + attempt * 5 
            print(f"  錯誤：API錯誤 (第 {attempt+1} 次嘗試): {e}. 將在 {wait_time} 秒後重試...");
            time.sleep(wait_time)
        except Exception as e:
            # 處理 NameError 和其他所有未知錯誤
            wait_time = 5 + attempt * 5
            print(f"  錯誤：未知錯誤 (第 {attempt+1} 次嘗試): {e}. 將在 {wait_time} 秒後重試...");
            time.sleep(wait_time)
    
    print(f"  錯誤：圖片序列分析在多次重試後失敗。")
    return default_error_result_structure()

def generate_personalized_summary_notes(student_id, overall_stats, non_task_highlights, openai_client):
    if not openai_client: return {"error": "OpenAI client not available"}
    # ... (此函數內部的 Prompt 內容完全不需要修改) ...
    stats_summary_for_ai = "\n".join([f"- {s['behavior_category']}: {s['percentage']:.1f}% ({s['count']}次)" for s in overall_stats[:7]]) # 確保百分比格式
    non_task_prompt_part = "該生在本堂課中，未觀察到明顯或頻繁的非任務相關行為。"
    if non_task_highlights:
        highlights_str = "\n".join([f"  - 圖 '{img_fn}' (~{ts}): '{beh}' (描述: {desc})" for img_fn, ts, beh, desc in non_task_highlights[:3]])
        if highlights_str: non_task_prompt_part = f"在本堂課中，觀察到一些非任務相關行為，例如：\n{highlights_str}\n這可能影響了學習專注度。"
    prompt_for_summary = f"""
    「你是一位專業且富有同理心的學習行為教練。你的目標不是批評，而是透過客觀數據，引導學生發現自己的學習模式，並提供能立即實踐的策略，以激發他們『自我反思』的動力。」

    **任務：** 為學生「{student_id}」撰寫一份「AI學習夥伴的觀察與建議」。

    **學生的課堂行為數據：**
    *   **主要行為分佈:**
    {stats_summary_for_ai}
    *   **值得注意的行為片段:**
    {non_task_prompt_part}

    **撰寫指引與 JSON 格式要求：**
    請嚴格遵循以下指引，產生一個結構完整的 JSON 物件。

    *   **`greeting` (問候語):**
        *   用親切、個人化的方式稱呼學生，例如：「嗨，{student_id} 同學，一起來看看這次課堂的學習足跡吧！」

    *   **`positive_feedback` (亮點觀察):**
        *   **必須**從數據中找出最值得肯定的行為（例如「目視教師」或「筆記」佔比最高），並給予具體、真誠的讚美。
        *   **範例**：「我發現你在這堂課有超過一半的時間都在『目視教師』，這代表你非常努力地跟上老師的節奏，非常棒！」

    *   **`observation_points_summary` (行為模式提醒):**
        *   客觀、中性地指出一個或兩個最主要的、可能影響學習的行為模式。避免使用負面詞彙。
        *   **範例**：「數據也顯示，大約有 15% 的時間出現了『玩弄物品』或『目視他處』的狀況，這些時刻可能讓我們不小心錯過了一些重點喔。」

    *   **`reflection_points` (反思引導提問):**
        *   **【此項最為關鍵】** 根據前面的觀察點，設計 2-3 個**開放式問題**，引導學生思考行為背後的原因，而不是直接給答案。
        *   **問題範例 1**：「我們可以一起回想看看，當出現『玩弄物品』的時候，通常是在課程的哪個階段呢？是覺得內容太簡單、太難，還是剛好有點疲倦了呢？」
        *   **問題範例 2**：「當視線看向其他地方時，是想到了什麼有趣的事，還是被教室裡的其他動靜吸引了呢？了解這些原因，能幫助我們找到最適合自己的專注方法。」

    *   **`suggestions` (可實踐的小建議):**
        *   提供 1-2 個**具體、微小、且容易執行**的行動建議。不要說「要專心」，而是給出方法。
        *   **建議範例 1**：「下次當你發現自己開始無意識地轉筆時，可以試著把它輕輕放下，然後做一個深呼吸，再重新將目光移回老師或課本上。」
        *   **建議範例 2**：「如果感覺到疲倦或分心，可以試試看『筆記專注法』：在筆記本上寫下老師說的任何一個關鍵字，這個小動作能幫助我們的大腦重新連線！」

    *   **`encouragement` (鼓勵與結語):**
        *   用一句溫暖、有力的話作結，強調這份報告是幫助他成長的工具。
        *   **範例**：「每一次的觀察都是為了讓我們更了解自己。相信你透過這些小小的調整，一定能發揮出自己最大的潛力，加油！」
    """
    retry_attempts = 2
    for attempt in range(retry_attempts):
        try:
            print(f"  正在為學生 {student_id} 使用 {SUMMARY_DEPLOYMENT_NAME} 生成個性化總結...") # 修改了日誌輸出
            response = client.chat.completions.create(
                # ==========================================================
                # ↓↓↓ 【關鍵修改】使用我們新設定的經濟型模型 ↓↓↓
                # ==========================================================
                model=SUMMARY_DEPLOYMENT_NAME, 
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": "你是一位富有同理心和洞察力的教育顧問。"},
                          {"role": "user", "content": prompt_for_summary}],
                max_tokens=MAX_TOKENS_SUMMARY_COMPLETION, #gpt-4.1
                temperature=0.7 #gpt-4.1
                # max_completion_tokens=MAX_TOKENS_SUMMARY_COMPLETION, # gpt-5        
                
            )
            summary_data = json.loads(response.choices[0].message.content)
            expected_keys = ["greeting", "positive_feedback", "observation_points_summary", "reflection_points", "suggestions", "encouragement"]
            if all(key in summary_data for key in expected_keys): 
                print(f"  成功為學生 {student_id} 生成個性化總結。")
                return summary_data
            else: 
                print(f"    警告：個性化總結JSON缺少鍵。返回: {summary_data}")
                return {key: summary_data.get(key, f"AI未能生成 ({key})") for key in expected_keys}
        except Exception as e: 
            print(f"  生成個性化總結錯誤 ({attempt + 1}): {e}")
            time.sleep(API_RETRY_DELAY_SECONDS if isinstance(e, RateLimitError) else 5)
        if attempt == retry_attempts - 1: 
            print(f"  錯誤：無法為學生 {student_id} 生成個性化總結。")
            return {"greeting": f"親愛的 {student_id},", "positive_feedback": "總結生成遇到問題。", "observation_points_summary": "請參考統計數據。", "reflection_points": "未能生成。", "suggestions": "請自行評估。", "encouragement": "加油！"}
    return {}

def find_closest_image_path(representative_timestamp, sorted_photo_list, max_time_diff_seconds=5):
    """
    一個可重用的輔助函數，用二分查找法在排序好的照片列表中找到時間最接近的照片路徑。
    """
    if not sorted_photo_list:
        return None

    all_timestamps = [item[0] for item in sorted_photo_list]
    # bisect_left 找到應該插入的位置
    insertion_point = bisect.bisect_left(all_timestamps, representative_timestamp)

    closest_path = None
    min_diff = datetime.timedelta.max

    # 只檢查插入點及其前後的幾個候選照片，效率極高
    # 檢查範圍設為 insertion_point-2 到 insertion_point+2 以增加容錯
    start_index = max(0, insertion_point - 2)
    end_index = min(len(sorted_photo_list), insertion_point + 2)

    for i in range(start_index, end_index):
        candidate_ts, candidate_path = sorted_photo_list[i]
        diff = abs(candidate_ts - representative_timestamp)
        if diff < min_diff:
            min_diff = diff
            closest_path = candidate_path

    # 只有在時間差在容許範圍內才返回路徑
    if closest_path and min_diff.total_seconds() <= max_time_diff_seconds:
        return closest_path
    
    return None

def find_closest_position(representative_timestamp, sorted_position_list, max_time_diff_seconds=10):
    """
    用二分查找法在排序好的位置列表中找到時間最接近的老師位置。
    """
    if not sorted_position_list:
        return "未知"

    all_timestamps = [item[0] for item in sorted_position_list]
    insertion_point = bisect.bisect_left(all_timestamps, representative_timestamp)

    closest_position = "未知"
    min_diff = datetime.timedelta.max

    start_index = max(0, insertion_point - 2)
    end_index = min(len(sorted_position_list), insertion_point + 2)

    for i in range(start_index, end_index):
        candidate_ts, candidate_pos = sorted_position_list[i]
        diff = abs(candidate_ts - representative_timestamp)
        if diff < min_diff:
            min_diff = diff
            closest_position = candidate_pos
    
    if min_diff.total_seconds() <= max_time_diff_seconds:
        return closest_position
    
    return "未知"

def process_single_batch(batch_idx, image_batch_info, teacher_positions_data, sorted_classroom_photos, classroom_states, client, student_id, student_position):
    """
    【升級版】處理單一圖片批次，使用老師位置和預處理好的課堂狀態。
    """
    batch_student_paths = [info["path"] for info in image_batch_info]
    batch_image_filenames = [info["filename"] for info in image_batch_info]
    
    representative_timestamp = image_batch_info[0]["timestamp_obj"]

    # --- 為當前批次查找所有情境數據 ---
    teacher_position_text = find_closest_position(representative_timestamp, teacher_positions_data)
    classroom_view_path = find_closest_image_path(representative_timestamp, sorted_classroom_photos)
    # 【關鍵修正】使用正確的變數名稱 classroom_states
    classroom_state = find_state_for_timestamp(representative_timestamp, classroom_states)

    # --- 呼叫【新版】分析函數 ---
    sequence_analysis_data = analyze_student_behavior_from_images_sequence(
        student_image_paths=batch_student_paths,
        teacher_position_text=teacher_position_text,
        classroom_view_image_path=classroom_view_path,
        classroom_state=classroom_state, # <-- 傳入狀態標籤
        image_filenames_batch=batch_image_filenames,
        openai_client=client,
        student_id=student_id,
        student_position=student_position
    )

    return {
        "batch_index": batch_idx,
        "image_batch_info": image_batch_info,
        "matched_teacher_position_text": teacher_position_text,
        "matched_classroom_view_image": os.path.basename(classroom_view_path) if classroom_view_path else None,
        "matched_classroom_state": classroom_state,
        "analysis": sequence_analysis_data
    }

def main():
    print("--- 學生課堂學習行為分析報告生成 (JSON) v5.0 - 成本優化版 ---")
    print(f"視覺模型: {VISION_DEPLOYMENT_NAME}, 文本模型: {VISION_DEPLOYMENT_NAME}, 總結模型: {VISION_DEPLOYMENT_NAME}")
    print(f"圖片批次大小: {IMAGES_PER_API_CALL}, 圖片取樣率: 1/{SAMPLING_RATE}, 圖片解析度: {IMAGE_DETAIL_LEVEL}")
    print("-" * 40)

    # --- 使用者輸入部分 ---
    student_id = get_valid_input("請輸入學生姓名 (可中文): ")
    student_number = get_valid_input("請輸入學生座號 (例如: 1): ") 
    student_images_folder = get_valid_folder_path(f"請輸入 '{student_id}' 的個人影像資料夾: ")
    print("\n--- 請輸入課堂情境資訊 ---")
    student_position = get_valid_input("請輸入學生座位 (例如: '第3排中間', '第1排左側'): ")
    print("-" * 50)

    # --- 準備輸出資料夾 ---
    safe_student_id_for_folder = re.sub(r'[\\/*?:"<>|]',"", student_id.replace(" ", "_"))
    student_specific_folder_base = os.path.join(JSON_OUTPUT_FOLDER, safe_student_id_for_folder)
    os.makedirs(student_specific_folder_base, exist_ok=True)

    # --- 準備學生照片資料 ---
    image_files_with_timestamps = []
    valid_extensions = ('.png', '.jpg', '.jpeg', '.webp')
    print(f"正在掃描學生個人照片資料夾: {student_images_folder}...")
    for filename in os.listdir(student_images_folder):
        if filename.lower().endswith(valid_extensions):
            timestamp = get_timestamp_from_filename(filename)
            if timestamp:
                image_files_with_timestamps.append({
                    "path": os.path.join(student_images_folder, filename), "filename": filename,
                    "timestamp_obj": timestamp, # <-- 直接使用原始時間戳，不做任何校準
                    "timestamp_str": str(timestamp).split('.')[0]
                })
    image_files_with_timestamps.sort(key=lambda x: x["timestamp_obj"])
    
    # ==========================================================
    # ↓↓↓ 【新增】圖片取樣以降低成本 ↓↓↓
    # ==========================================================
    if SAMPLING_RATE > 1:
        original_count = len(image_files_with_timestamps)
        image_files_with_timestamps = image_files_with_timestamps[::SAMPLING_RATE]
        print(f"已執行圖片取樣：從 {original_count} 張原始照片中，每 {SAMPLING_RATE} 張取 1 張，共 {len(image_files_with_timestamps)} 張照片將被分析。")
    else:
        print(f"成功找到 {len(image_files_with_timestamps)} 張學生個人照片 (未取樣)。")

    if not image_files_with_timestamps: 
        print("錯誤：學生資料夾中未找到有效時間格式的圖片，或取樣後為空。")
        return
    # ==========================================================

    # ... 後續的程式碼，從 `load_teacher_positions` 開始，到整個 `main` 函數結束，
    # 都不需要再做任何修改。您可以直接使用您原有的版本。
    # 我將剩餘部分貼在下方以保證完整性。
    
    def load_teacher_positions(json_path, start_time_offset_str=""): # 新增 start_time_offset_str 參數
        if not json_path or not os.path.isfile(json_path):
            print(f"提示：未提供或找不到老師位置 JSON 檔案 ({json_path})。將不使用老師位置情境。")
            return []
        start_offset = parse_time_offset(start_time_offset_str)
        print(f"正在讀取老師位置數據: {json_path}...")
        try:
            with open(json_path, 'r', encoding='utf-8') as f: data = json.load(f)
            position_map, original_count = [], len(data)
            for item in data:
                try:
                    h, m, s = map(int, item['timestamp'].split(':'))
                    td = datetime.timedelta(hours=h, minutes=m, seconds=s)
                    if start_offset and td < start_offset: continue
                    position_map.append((td, item['position']))
                except (ValueError, KeyError): continue
            position_map.sort(key=lambda x: x[0])
            if start_offset: print(f"成功加載老師位置數據。根據起始時間 '{start_time_offset_str}' 進行過濾，保留 {len(position_map)} 筆 (原 {original_count} 筆)。")
            else: print(f"成功加載並索引了 {len(position_map)} 筆老師位置數據。")
            return position_map
        except Exception as e:
            print(f"錯誤：讀取或解析老師位置 JSON 時發生問題: {e}"); return []

    def load_and_sort_photos(folder_path, photo_type_name, start_time_offset_str=""): # 新增 start_time_offset_str 參數
        if not folder_path or not os.path.isdir(folder_path):
            print(f"提示：未提供或找不到 {photo_type_name} 照片資料夾 ({folder_path})。"); return []
        start_offset = parse_time_offset(start_time_offset_str)
        print(f"正在掃描 {photo_type_name} 照片資料夾: {folder_path}...")
        photo_list, original_count = [], 0
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                original_count += 1
                timestamp = get_timestamp_from_filename(filename)
                if timestamp:
                    if start_offset and timestamp < start_offset: continue
                    photo_list.append((timestamp, os.path.join(folder_path, filename)))
        if photo_list:
            sorted_photos = sorted(photo_list)
            if start_offset: print(f"成功加載 {photo_type_name} 照片。根據起始時間 '{start_time_offset_str}' 進行過濾，保留 {len(sorted_photos)} 張 (原 {original_count} 張有效格式照片)。")
            else: print(f"成功加載並索引了 {len(sorted_photos)} 張 {photo_type_name} 照片。")
            return sorted_photos
        else:
            print(f"警告：在 {photo_type_name} 照片資料夾 '{folder_path}' 中未找到符合條件的圖片。"); return []

    teacher_positions_data = load_teacher_positions(TEACHER_POSITION_JSON, TEACHER_JSON_START_TIME_OFFSET)
    sorted_classroom_photos = load_and_sort_photos(CLASSROOM_IMAGES_FOLDER, "班級整體", CLASSROOM_START_TIME_OFFSET)
    classroom_states = load_classroom_states(CLASSROOM_STATE_JSON_PATH)

    image_batches = [image_files_with_timestamps[i:i + IMAGES_PER_API_CALL] for i in range(0, len(image_files_with_timestamps), IMAGES_PER_API_CALL)]
    print(f"學生圖片將被分為 {len(image_batches)} 個批次進行分析。")

    MAX_WORKERS = 4
    print(f"將使用最多 {MAX_WORKERS} 個執行緒進行平行分析...")
    all_results_from_threads = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_batch_idx = { 
            executor.submit(process_single_batch, idx, batch_info, teacher_positions_data, sorted_classroom_photos, classroom_states, client, student_id, student_position): idx 
            for idx, batch_info in enumerate(image_batches) 
        }
        for future in tqdm(as_completed(future_to_batch_idx), total=len(image_batches), desc=f"分析學生 {student_id} 的圖片批次"):
            try:
                result = future.result()
                all_results_from_threads.append(result)
            except Exception as exc:
                batch_idx = future_to_batch_idx[future]
                print(f'\n批次 {batch_idx + 1} 執行時產生錯誤: {exc}')
                all_results_from_threads.append({"batch_index": batch_idx, "error": str(exc)})

    all_results_from_threads.sort(key=lambda x: x['batch_index'])
    print("\n所有批次分析完成，開始匯總數據...")
    all_sequence_analysis_results = []
    overall_behavior_summary = { "total_images_processed_in_batches": 0, "behavior_counts": Counter(), "behavior_confidence_sum": defaultdict(float), "non_task_behavior_examples_for_summary": [] }
    for result in tqdm(all_results_from_threads, desc="匯總分析結果"):
        if "error" in result:
            all_sequence_analysis_results.append({"batch_index": result['batch_index'] + 1, "analysis": default_error_result_structure()})
            continue
        image_batch_info, sequence_analysis_data = result["image_batch_info"], result["analysis"]
        batch_image_filenames = [info["filename"] for info in image_batch_info]
        all_sequence_analysis_results.append({ "batch_index": result['batch_index'] + 1, "image_filenames_in_batch": batch_image_filenames, "matched_teacher_position_text": result.get("matched_teacher_position_text"), "matched_classroom_view_image": result.get("matched_classroom_view_image"), "analysis": sequence_analysis_data })
        if "error" not in sequence_analysis_data:
            overall_behavior_summary["total_images_processed_in_batches"] += len(batch_image_filenames)
            if "per_image_highlights" in sequence_analysis_data:
                for hl_item in sequence_analysis_data.get("per_image_highlights", []):
                    # 【修改點 3】修改匯總邏輯以處理行為列表
                    behavior_list, conf, desc = hl_item.get("behavior_category"), hl_item.get("confidence", 0.0), hl_item.get("context_description") # AI prompt 改為輸出 context_description
                    
                    if not behavior_list: continue

                    # 確保是列表
                    if not isinstance(behavior_list, list):
                        behavior_list = [behavior_list]
                    
                    # 遍歷列表中的每一個行為進行統計
                    for cat in behavior_list:
                        overall_behavior_summary["behavior_counts"][cat] += 1
                        overall_behavior_summary["behavior_confidence_sum"][cat] += float(conf)
                    
                    # 處理非任務行為範例（選擇列表中的第一個作為代表）
                    primary_behavior_for_example = behavior_list[0]
                    non_task_keywords = [
                        "玩弄手部/文具", "觸摸臉部", "觸摸頭髮",
                        "目視他處", "趴睡", "喝水", "飲食" #, "整理個人物品"
                    ]
                    if primary_behavior_for_example in non_task_keywords and desc and len(overall_behavior_summary["non_task_behavior_examples_for_summary"]) < 3:
                        try:
                            student_img_idx = hl_item.get("image_index_in_sequence", -1)
                            if 0 <= student_img_idx < len(batch_image_filenames):
                                hl_filename = batch_image_filenames[student_img_idx]
                                hl_timestamp = next((info["timestamp_str"] for info in image_batch_info if info["filename"] == hl_filename), "未知時間")
                                # 將整個行為列表存入範例，讓報告更詳細
                                overall_behavior_summary["non_task_behavior_examples_for_summary"].append( (hl_filename, hl_timestamp, ", ".join(behavior_list), desc) )
                        except Exception as e_idx: print(f"提取非任務示例時出錯: {e_idx}")
    
    overall_behavior_stats_list, total_highlight_instances = [], sum(overall_behavior_summary["behavior_counts"].values())
    valence_summary = {"正向": 0, "負向": 0, "中性": 0}
    for behavior, count in overall_behavior_summary["behavior_counts"].items():
        percentage, avg_confidence = (count / total_highlight_instances * 100) if total_highlight_instances > 0 else 0, (overall_behavior_summary["behavior_confidence_sum"][behavior] / count) if count > 0 else 0
        valence = LABEL_TO_VALENCE.get(behavior, "未分類")
        if valence in valence_summary: valence_summary[valence] += count
        overall_behavior_stats_list.append({ "behavior_category": behavior, "valence": valence, "count": count, "percentage": round(percentage, 1), "average_confidence": round(avg_confidence, 2) })
    overall_behavior_stats_list.sort(key=lambda x: x["count"], reverse=True)
    personalized_notes = generate_personalized_summary_notes(student_id, overall_behavior_stats_list, overall_behavior_summary["non_task_behavior_examples_for_summary"], client)
    behavior_to_images_map = defaultdict(list)
    for result in all_results_from_threads:
        if "error" in result or "analysis" not in result or "error" in result["analysis"]: continue
        sequence_analysis, image_batch_info, filenames_in_batch = result["analysis"], result.get("image_batch_info", []), [info.get("filename") for info in result.get("image_batch_info", [])]
        if "per_image_highlights" in sequence_analysis and isinstance(sequence_analysis["per_image_highlights"], list):
            for highlight in sequence_analysis["per_image_highlights"]:
                # 【修改點 4】修改索引建立邏輯以處理行為列表
                behavior_list, image_index = highlight.get("behavior_category"), highlight.get("image_index_in_sequence")
                
                if not behavior_list or not isinstance(image_index, int) or not (0 <= image_index < len(filenames_in_batch)):
                    continue

                if not isinstance(behavior_list, list):
                    behavior_list = [behavior_list]
                
                image_filename = filenames_in_batch[image_index]
                if image_filename:
                    # 為列表中的每一個行為都建立索引
                    for behavior_category in behavior_list:
                        if image_filename not in behavior_to_images_map[behavior_category]:
                            behavior_to_images_map[behavior_category].append(image_filename)
    for behavior in behavior_to_images_map: behavior_to_images_map[behavior].sort()
    print("行為索引建立完成。")

    total_classified_instances = sum(valence_summary.values())
    valence_summary_with_percentage = { valence: { "count": count, "percentage": round((count / total_classified_instances * 100), 1) if total_classified_instances > 0 else 0 } for valence, count in valence_summary.items() }

    final_json_output = {
        "report_metadata": {
            "student_id": student_id, "student_number": student_number, "report_generation_time": "08/17", "student_image_source_folder": "英文",
            "teacher_position_source_json": os.path.basename(TEACHER_POSITION_JSON) if TEACHER_POSITION_JSON and os.path.isfile(TEACHER_POSITION_JSON) else "N/A",
            "classroom_view_source_folder": os.path.basename(CLASSROOM_IMAGES_FOLDER) if CLASSROOM_IMAGES_FOLDER and os.path.isdir(CLASSROOM_IMAGES_FOLDER) else "N/A",            "classroom_context": { "student_position": student_position },
            "analysis_settings": {
                "vision_model": VISION_DEPLOYMENT_NAME, "text_model": TEXT_DEPLOYMENT_NAME, "images_per_batch": IMAGES_PER_API_CALL, "context_images_per_batch_desc": "動態匹配老師和班級照片各一張", "confidence_threshold": BEHAVIOR_CONFIDENCE_THRESHOLD_FILTER,
                "cost_optimization": { "sampling_rate": SAMPLING_RATE, "image_detail": IMAGE_DETAIL_LEVEL } # 新增成本優化資訊
            }
        },
        "overall_summary": {
            "total_images_found": len(image_files_with_timestamps) * SAMPLING_RATE if SAMPLING_RATE > 1 else len(image_files_with_timestamps), # 顯示原始數量
            "total_images_analyzed": overall_behavior_summary["total_images_processed_in_batches"],
            "total_batches": len(image_batches), "valence_summary": valence_summary_with_percentage, "behavior_statistics": overall_behavior_stats_list, "behavior_to_images_index": behavior_to_images_map, "ai_summary_notes": personalized_notes
        },
        "detailed_sequence_analysis": all_sequence_analysis_results
    }

    current_timestamp_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_student_id_for_filename = re.sub(r'[\\/*?:"<>|]',"", student_id.replace(" ", "_"))
    json_filename = JSON_FILENAME_TEMPLATE.format(student_id=safe_student_id_for_filename, timestamp=current_timestamp_str)
    json_filepath = os.path.join(student_specific_folder_base, json_filename)
    try:
        with open(json_filepath, 'w', encoding='utf-8') as f: json.dump(final_json_output, f, ensure_ascii=False, indent=4)
        print(f"\n✅ 學生 '{student_id}' 的序列行為分析報告已成功儲存至: {json_filepath}")
    except Exception as e: print(f"❌ 錯誤：儲存 JSON 檔案時發生問題：{e}")
    
    print("\n--- 處理完成 ---")

if __name__ == "__main__":
    main()

### Azure GPT-4.1 test

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import base64
import json
import datetime
import time
from collections import Counter, defaultdict
from openai import AzureOpenAI, APIError, RateLimitError, AuthenticationError 
from PIL import Image, UnidentifiedImageError
import io
from tqdm import tqdm
import math
from dotenv import load_dotenv 
from concurrent.futures import ThreadPoolExecutor, as_completed
import bisect

# --- Configuration ---


# 班級整體照片資料夾路徑 
CLASSROOM_IMAGES_FOLDER = r'C:\Users\User\Desktop\test\student_full_classroom\0817_english_class'

# 【新】老師視角照片資料夾路徑 (從學生視角拍攝老師在黑板前的照片) - 可選
TEACHER_POSITION_JSON = r'C:\Users\User\Desktop\test\teacher_position\0817_position.json'

#課堂情況json
CLASSROOM_STATE_JSON_PATH = r'C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0817\老師上課\TXT\0817_processed.json'

# 【新】時間軸對齊設定 (格式: "HH:MM:SS")
CLASSROOM_START_TIME_OFFSET = "00:00:00"  # 班級照片的起始時間，設定為空字串 "" 表示不篩選
TEACHER_JSON_START_TIME_OFFSET ="00:00:00" # 老師位置資料的起始時間，設定為空字串 "" 表示不篩選

# ... (您原有的其他設定) ...

JSON_OUTPUT_FOLDER = "SynologyDrive\json_behavior"


JSON_FILENAME_TEMPLATE = "student_{student_id}_behavior_report_{timestamp}.json"
SAMPLING_RATE = 3  
IMAGE_DETAIL_LEVEL = "low" # 圖片解析度 ('low' 或 'high')，low 可大幅降低成本
MAX_TOKENS_VISION_COMPLETION = 2500
MAX_TOKENS_SUMMARY_COMPLETION = 1000 # 個性化總結的 token
BEHAVIOR_CONFIDENCE_THRESHOLD_FILTER = 0.96 # 行為信度過濾閾值 (例如 70%)
IMAGES_PER_API_CALL = 10 # 一次API調用處理的圖片數量
BEHAVIOR_SIMILARITY_CONFIDENCE_THRESHOLD = 0.15 # 過濾相似連續行為的信度差異閾值
API_RETRY_DELAY_SECONDS = 10

# ==========================================================

# --- 標準行為分類與定義 (核心辭典，來自圖一) ---
STANDARD_BEHAVIOR_CATEGORIES = {
    "視線": [
        {"label": "目視教師", "definition": "學生的頭部與視線明確聚焦於教室【前方】的【老師所在】區域，且其頭部旋轉角度處於一個【合理的朝前弧度】內（通常不超過45度）。**【絕對排除條款】**: 任何導致學生視線與其身體朝向構成【接近90度或更大角度】的頭部大幅度轉動，【絕對不允許】被標註為『目視教師』。這種姿態應被優先考慮為『目視同學』(`V_CLS`)或『目視他處』(`V_ELS`)。"},
        {"label": "目視黑板", "definition": "學生的頭部與視線明確聚焦於教室【前方】的【非老師所在】的黑板或螢幕區域。當老師位置未知，或學生視線明確未朝向老師時，這是面向前方的預設專注行為。"}, 
        {"label": "目視書本/筆記", "definition": "學生的頭部與視線向量【主要朝下】，且頭部的水平旋轉角度【沒有明顯偏離】其身體所朝向的【個人桌面工作區軸線】。此標籤捕捉的是在個人學習材料上的視覺專注狀態。**【嚴格邊界】**: 一旦頭部/視線明確地、持續地轉向側面（例如，足以與鄰座同學進行眼神交流），即使視線仍然略微朝下，也【必須優先考慮】標註為『目視同學』。**【注意】**：如果學生同時在進行『做筆記』，根據『動作優先』原則，你應將『做筆記』作為主要標籤。"},
        {"label": "目視同學", "definition": "學生的【主要身體朝向與視線】明確脫離前方或個人桌面，轉向側面或後方的同學。**【最高優先級的情境標籤】**: 一旦觀察到這種明確的身體轉向，此標籤的優先級就高於大多數獨立的個人動作。它涵蓋了從純粹的視覺交流到【涉及物件的協作行為】（如共同看書、討論問題）。**【核心情境判斷規則】**: 此行為的性質根據【課堂狀態】決定..."},
        {"label": "目視他處", "definition": "【備用排除性標籤】。當你已確認學生的視線【不】符合『目視教師』、『目視黑板』、『目視書本/筆記』或『目視同學』的任何一項明確定義時，才使用此標籤。它捕捉的是失去焦點的狀態，例如：【抬頭看天花板】、【轉頭看沒有同學及老師的地方】。"}     
    ],
    "肢體(手部)": [
        {"label": "做筆記", "definition": "【核心證據：觀察到一個動態的書寫過程】。你必須能明確看到學生手持筆，且筆尖正在紙張上進行【有意義的移動或書寫/繪製動作】。**【最高優先級】**：這是一個高優先級的動作標籤。**【嚴格排除條款】**：以下情況【絕對不允許】標註為『做筆記』：(1) **靜態持筆**：僅僅手持筆，或筆尖靜止停留在紙上，應標註為『目視書本/筆記』 (`V_BOK`)。(2) **動作中斷**：當學生手部的主要動作變為其他行為（如『觸摸頭髮』、『托腮』、『與同學互動』），即使手中仍持有筆，也必須以該【瞬時動作】為主要標籤。"},
        {"label": "翻書", "definition": "學生手部正在主動翻閱、移動書本。"},
        {"label": "觸摸臉部", "definition": "學生【非支撐性地】用手短暫觸碰或摩擦自己的臉部、鼻子、嘴巴或眼睛。此行為區別於『托腮』的持續性支撐動作。"},
        {"label": "觸摸頭髮", "definition": "學生用手觸摸、撥弄或整理自己的頭髮。注意：即使手臂抬得較高，只要手部的主要動作是與頭髮互動，就【必須】使用此標籤，而不是『舉手』類標籤。"}
    ],
    "身體姿態": [
        {"label": "坐姿直立", "definition": "學生上半身軀幹基本垂直於地面，或輕微前傾。"},
        {"label": "身體前傾", "definition": "學生上半身軀幹明顯向前彎曲，靠近桌面。此行為描述的是一種【清醒狀態下】的姿態。如果學生頭部接觸桌面或手臂，應【優先使用】『趴睡』標籤。"},
        {"label": "身體後靠", "definition": "學生背部倚靠在椅背上。"},
        {"label": "低頭(非學習)", "definition": "【極其嚴格的排除性標籤】。僅在你能夠【極度確信地】觀察到以下【全部】條件時才可使用：1. 學生頭部明顯低垂。2. 其視線【明確沒有】朝向任何學習材料（例如，看向地面、自己的懷中或空無一物的桌面）。3. 其手部沒有在進行任何學習相關操作。"},
        {"label": "趴睡", "definition": "學生將頭部【枕於】手臂或桌面上，呈現明確的休息或睡眠狀態。**【最高優先級】**：只要觀察到頭部接觸桌面或手臂的休息姿態，此標籤的優先級【高於】所有其他學習相關標籤（如『做筆記』、『目視書本』）。"},
        {"label": "托腮", "definition": "【核心定義：手部對頭部提供持續性支撐】。學生使用一隻或兩隻手的手掌、拳頭或手臂，支撐其下巴、臉頰或頭部的重量。這是一個純粹的物理姿態描述，不包含任何意圖推斷。"}
    ],
    "互動": [
        {"label": "主動舉手", "definition": "學生舉起一隻手，意圖提問或回答問題。**【三大核心物理證據，必須同時滿足】**：(1) 手臂向上伸展，手部明顯高於肩膀。(2) 手掌形態為張開朝前或中性放鬆，【嚴禁】手指指向特定方向或揮舞。(3) 該動作具有一定的持續性（非瞬間劃過）。**【情境觸發】**：此行為最常發生在【老師單向授課】的狀態下（如『文法/句型講解』、『閱讀/文章分析』），代表學生的自發性提問或補充。**【絕對排除】**：任何手部接觸頭部、與同學互動的手勢、指向性的動作，都【不允許】標記為此行為。"},
        {"label": "被動舉手", "definition": "學生舉手以回應老師的群體性指令（如投票、調查）。**【核心視覺證據】**：通常是多數學生同時舉手，姿態可能較為放鬆，手臂不必完全伸直。**【情境觸發】**：此行為最常發生在【老師與學生互動】的狀態下（如『課堂問答/互動』、『習題/考卷檢討』），代表學生回應老師的指令或提問。**【關鍵區分】**：此標籤的判斷【高度依賴】課堂情境和群體性動作。如果情境不匹配，應避免使用此標籤。"}
    ],
    "其他狀態": [
        {"label": "喝水", "definition": "【核心證據：清晰可見的容器】。只有當你能夠【明確地】看到學生手持水瓶、杯子或其他容器，並將其送至嘴邊時，才可以使用此標籤。**【嚴格排除】**: 任何僅有低頭姿態、手部靠近臉部但【沒有可見容器】的場景，都【嚴禁】標註為此行為。在此情況下，應優先考慮『目視書本/筆記』(`V_BOK`)或『趴睡』(`P_SLP`)。"},
        {"label": "飲食", "definition": "學生正在食用固體食物。"},
        {"label": "玩弄手部／文具", "definition": "學生手部在進行與學習無關的重複性小動作，例如玩手指、轉筆、玩弄橡皮擦等"},
        # {"label": "整理書包", "definition": "學生正在整理書包、桌面文具。"},
        {"label": "被遮擋/無法判斷", "definition": "【最終備用標籤】。因遮擋、模糊或角度問題，無法清晰識別學生的主要行為時，【必須】使用此標籤。"}
    ]
}
# 自動從新結構生成有效的標籤列表
VALID_BEHAVIOR_LABELS = [item['label'] for category in STANDARD_BEHAVIOR_CATEGORIES.values() for item in category]

BEHAVIOR_CODES = {
    # 視線 (不變)
    "V_TCH": "目視教師", "V_BRD": "目視黑板", "V_BOK": "目視書本/筆記",
    "V_CLS": "目視同學", "V_ELS": "目視他處",
    
    # 肢體(手部) (移除 H_FLIP)
    "H_NOT": "做筆記",
    "H_PLAY_HW": "玩弄手部/文具",
    "H_TOUCH_F": "觸摸臉部",
    "H_TOUCH_H": "觸摸頭髮",
    
    # 身體姿態 (新增 P_THK - P for Posture, THK for Thinking)
    "P_STR": "坐姿直立", "P_LEAN": "身體前傾", "P_BACK": "身體後靠",
    "P_DWN": "低頭(非學習)", "P_SLP": "趴睡",
    "P_THK": "托腮", # <--- 新增
    
    # 互動 (將 I_HND 拆分為主動/被動)
    "I_HND_A": "主動舉手", # A for Active
    "I_HND_P": "被動舉手", # P for Passive
    
    # 其他狀態 (不變)
    "O_DRK_W": "喝水",
    "O_EAT_S": "飲食",
    # "O_TDY": "整理個人物品",
    "O_UNK": "被遮擋/無法判斷"
}

# 自動生成反向查找字典，用於本地解碼
CODE_TO_BEHAVIOR = {code: label for code, label in BEHAVIOR_CODES.items()}
BEHAVIOR_TO_CODE = {label: code for code, label in BEHAVIOR_CODES.items()}

# AI返回標籤到標準標籤的映射規則 (與新標籤對齊)
BEHAVIOR_MAPPING_RULES = {
    "目視桌面/教材": "目視書本", "目視桌面": "目視書本", "看書": "目視書本",
    "書寫/做筆記": "筆記", "動手操作-書寫/做筆記": "筆記",
    "視覺專注-閱讀書本/講義": "目視書本",
    "視覺專注-看老師/黑板方向": "目視教師", "目視黑板/老師": "目視教師",
    "看老師": "目視教師", "看黑板": "目視黑板",
    # --- ↓↓↓ 【修改點】更新映射規則以對應新標籤 ↓↓↓ ---
    "玩弄物品": "玩弄手部/文具", # 將模糊的舊標籤對應到最可能的新標籤
    "非任務相關動作-玩弄物品(筆等)": "玩弄手部/文具",
    # 移除了"非任務相關動作-觸摸臉部/頭髮"，鼓勵AI直接使用更精確的新標籤
    # "非任務相關動作-整理物品": "整理個人物品",
    "社交互動-與同學互動": "目視同學",
    "趴睡/休息": "趴睡",
    "低頭/伏案(非睡)": "低頭",
}

BEHAVIOR_VALENCE_MAP = {
    "正向": [
        "做筆記",
        "主動舉手", # <-- 修改
        "目視教師",
        "目視黑板",
        "目視書本/筆記",
    ],
    "負向": [
        "趴睡",
        "玩弄手部/文具",
        "觸摸臉部",
        "觸摸頭髮",
        "目視他處",
        "目視同學"
    ],
    "中性": [
        "身體前傾",
        "坐姿直立",
        "身體後靠",
        "喝水",
        "飲食",       
        "玩弄手部／文具",
        "翻書",
        "低頭(非學習)",
        # "整理個人物品",     
        "被遮擋/無法判斷",
        "托腮", # <-- 新增
        "被動舉手"  # <-- 新增
    ]
}

LABEL_TO_VALENCE = {
    label: valence 
    for valence, labels in BEHAVIOR_VALENCE_MAP.items() 
    for label in labels
}

# ---------------------------
#  API 金鑰配置
# ---------------------------
load_dotenv()

AZURE_API_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
# 從環境變數讀取部署名稱
VISION_DEPLOYMENT_NAME = os.getenv("CHAT_COMPLETION_NAME")
TEXT_DEPLOYMENT_NAME = os.getenv("CHAT_COMPLETION_NAME")
SUMMARY_DEPLOYMENT_NAME = os.getenv("CHAT_COMPLETION_NAME")

# 檢查必要的 Azure 配置是否存在
if not all([AZURE_API_KEY, AZURE_ENDPOINT, VISION_DEPLOYMENT_NAME, SUMMARY_DEPLOYMENT_NAME]):
    print("錯誤：缺少必要的 Azure OpenAI 環境變數。")
    print("請檢查您的 .env 檔案是否包含 AZURE_OPENAI_KEY, AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_VISION_DEPLOYMENT_NAME, 和 AZURE_OPENAI_SUMMARY_DEPLOYMENT_NAME。")
    exit()

try:
    # ### 【修改 3】: 初始化 AzureOpenAI Client ###
    client = AzureOpenAI(
        api_key=AZURE_API_KEY,
        azure_endpoint=AZURE_ENDPOINT,
        api_version="2024-02-01"  # 使用一個穩定的 API 版本
    )
    client.models.list() # 嘗試調用一個簡單的API來驗證配置
    print("Azure OpenAI client 初始化並驗證成功。")
    print(f"  - 視覺分析將使用部署: '{VISION_DEPLOYMENT_NAME}'")
    print(f"  - 個性化總結將使用部署: '{SUMMARY_DEPLOYMENT_NAME}'")
except AuthenticationError: print("錯誤：Azure OpenAI API 金鑰或端點無效，認證失敗。"); exit()
except Exception as e: print(f"初始化 Azure OpenAI Client 時發生錯誤: {e}"); exit()

# ---------------------------
# 輔助函數
# ---------------------------
def find_state_for_timestamp(target_timestamp, classroom_states):
    """
    【v4.0 穩定版】根據時間戳，查找對應的課堂狀態標籤。
    使用線性查找以確保最高的準確性和可靠性。
    """
    if not classroom_states:
        return "未知"
    
    # 為了除錯，只打印一次查找的詳細信息
    if not hasattr(find_state_for_timestamp, "has_logged_first_search"):
        print("\n" + "-"*20 + " [除錯日誌 - 首次查找課堂狀態] " + "-"*20)
        print(f"  - 正在用第一張照片的時間戳进行匹配...")
        print(f"  - 目標時間戳 (Target Timestamp): {target_timestamp}")
        if classroom_states:
            first_state = classroom_states[0]
            print(f"  - 正在检查第一個時間區間: 從 {first_state.get('start_time_td')} 到 {first_state.get('end_time_td')}")
        print("-" * 69 + "\n")
        find_state_for_timestamp.has_logged_first_search = True

    # 遍歷每一個課堂狀態的時間區間
    for state in classroom_states:
        start_td = state.get("start_time_td")
        end_td = state.get("end_time_td")
        
        # 確保這個區間的時間數據是有效的
        if start_td and end_td:
            # 檢查目標時間戳是否落在 [開始時間, 結束時間] 這個閉区间内
            if start_td <= target_timestamp <= end_td:
                return state["classroom_state"]
    
    # 如果遍歷完所有區間都沒找到，說明時間戳確實超出了範圍
    return "未知"

def parse_time_offset(time_str):
    """將 "HH:MM:SS" 格式的字串轉換為 timedelta 物件"""
    if not time_str or not isinstance(time_str, str):
        return None
    try:
        h, m, s = map(int, time_str.split(':'))
        return datetime.timedelta(hours=h, minutes=m, seconds=s)
    except (ValueError, TypeError):
        print(f"警告：時間偏移量 '{time_str}' 格式不正確，應為 'HH:MM:SS'。將忽略此設定。")
        return None

def get_valid_input(prompt_message):
    while True:
        user_input = input(prompt_message).strip()
        if user_input: return user_input
        print("錯誤：輸入不能為空，請重新輸入。")

def get_valid_folder_path(prompt_message, is_optional=False): # 新增 is_optional 參數
    while True:
        folder_path = input(prompt_message).strip().strip('"')
        if not folder_path and is_optional:
            return None # 如果是可選的且用戶未輸入，返回 None
        if os.path.isdir(folder_path):
            return folder_path
        print(f"錯誤：路徑 '{folder_path}' 不是一個有效的資料夾，請重新輸入。")

def get_timestamp_from_filename(filename):
    """
    【v3.0 兼容版】从档名解析时间戳。
    能同时处理多种常见格式。
    """
    # 模式一：最优先，精确匹配 HH-MM-SS-ms.jpg 格式 (常见于学生照片)
    match = re.search(r'^(\d{2})-(\d{2})-(\d{2})-(\d{3})\.(jpg|jpeg|png|webp)$', filename, re.IGNORECASE)
    if match:
        try:
            h, m, s, ms, ext = match.groups()
            return datetime.timedelta(hours=int(h), minutes=int(m), seconds=int(s), milliseconds=int(ms))
        except (ValueError, IndexError):
            pass

    # 模式二：匹配包含 ...HH-MM-SS-ms... 的通用格式
    match = re.search(r'(\d{2})-(\d{2})-(\d{2})-(\d{3})', filename)
    if match:
        try:
            h, m, s, ms = map(int, match.groups())
            return datetime.timedelta(hours=h, minutes=m, seconds=s, milliseconds=ms)
        except (ValueError, IndexError):
            pass

    # 模式三：匹配包含 ...h...m...s 的通用格式
    match = re.search(r'(\d+)h(\d{2})m(\d{2})s', filename)
    if match:
        try:
            h, m, s = map(int, match.groups())
            return datetime.timedelta(hours=h, minutes=m, seconds=s)
        except (ValueError, IndexError):
            pass

    # 如果所有格式都沒匹配成功，最终返回 None
    return None

def encode_image_to_base64(image_path, max_size_kb=512, target_quality=75):
    try:
        with Image.open(image_path) as img: img.verify()
        with Image.open(image_path) as img:
            if img.mode == 'RGBA' or img.mode == 'P': img = img.convert('RGB')
            current_quality = target_quality
            output_buffer = io.BytesIO()
            temp_img = img.copy()
            temp_img.save(output_buffer, format="JPEG", quality=current_quality)
            current_size_kb = output_buffer.tell() / 1024

            if current_size_kb > max_size_kb:
                scale_factor = math.sqrt(max_size_kb / current_size_kb)
                new_width = int(temp_img.width * scale_factor * 0.9)
                new_height = int(temp_img.height * scale_factor * 0.9)
                if new_width >= 50 and new_height >= 50: # 最小尺寸限制
                    print(f"    圖片 {os.path.basename(image_path)} ({current_size_kb:.1f} KB) 過大，縮放並調整質量...")
                    temp_img = temp_img.resize((new_width, new_height), Image.Resampling.LANCZOS)
                    output_buffer = io.BytesIO()
                    temp_img.save(output_buffer, format="JPEG", quality=max(current_quality - 15, 40)) # 質量可以降更多
                else:
                    print(f"    警告: 圖片 {os.path.basename(image_path)} 縮放後過小，可能影響質量。使用較低質量。")
                    output_buffer = io.BytesIO()
                    img.save(output_buffer, format="JPEG", quality=max(current_quality // 2, 30) )


            output_buffer.seek(0)
            binary_data = output_buffer.getvalue()
            base64_encoded_data = base64.b64encode(binary_data)
            return f"data:image/jpeg;base64,{base64_encoded_data.decode('utf-8')}"
    except Exception as e: print(f"錯誤：處理圖片 '{image_path}': {e}"); return None

def load_classroom_states(json_path):
    """【全新】讀取預處理好的課堂狀態時間軸 JSON 檔案。"""
    if not json_path or not os.path.isfile(json_path):
        print("提示：未提供或找不到課堂狀態 JSON 檔案。將不使用課堂情境。")
        return None  # 返回 None 以便更明確地判斷失敗
    
    print(f"正在讀取課堂狀態時間軸: {json_path}...")
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f) # <--- 關鍵修正：將 f 作為參數傳入
            
            # 將時間字串預先轉換為 timedelta 物件以便快速比較
            timeline = data.get("timeline", [])
            for state in timeline:
                state["start_time_td"] = parse_time_offset(state["start_time"])
                state["end_time_td"] = parse_time_offset(state["end_time"])
            print(f"成功載入 {len(timeline)} 個課堂狀態階段。")
            return timeline
    except Exception as e:
        print(f"❌ 錯誤：讀取或解析課堂狀態 JSON 時發生問題: {e}")
        return None # 返回 None

def get_behavior_sequence_analysis_system_prompt(student_position, has_teacher_context):
    """
    【v14.4 - 終極重構版】
    此版本在 v14.3 的基礎上進行了結構性重構，將所有規則模塊化，
    消除了邏輯冗餘，並將決策樹統一整合，達到了最高的清晰度與可維護性。
    """
    # --- 1. 行為編碼表 (靜態模塊) ---
    behavior_table_for_prompt = "\n\n**【學習行為編碼表】**\n你 **必須** 且 **只能** 從以下列表的「編碼 (Code)」中選擇行為進行標註。...\n"
    for code, label in BEHAVIOR_CODES.items():
        definition = ""
        for category in STANDARD_BEHAVIOR_CATEGORIES.values():
            for item in category:
                if item['label'] == label:
                    definition = item['definition']
                    break
            if definition:
                break
        behavior_table_for_prompt += f"*   **`{code}`**: {label} - {definition}\n"
    behavior_table_for_prompt += "\n*如果行為因任何原因無法清晰判斷，請 **必須** 使用 **`O_UNK`** 編碼。*\n"

    # --- 2. 空間與攝影機規則 (動態模塊) ---
    analysis_view_rules = ""
    if "左" in student_position:
        analysis_view_rules = "*   **分析視角**: 拍攝該學生的【學生個人照片序列】來自教室前方的【左側】攝影機。\n*   **視線基準**: 對於這位學生，【正面朝向鏡頭】僅代表他在看教室的【左前方】。為了看向【教室正前方中心】，他的頭部必須輕微地朝向他自己的【右側】轉動。"
    elif "右" in student_position:
        analysis_view_rules = "*   **分析視角**: 拍攝該學生的【學生個人照片序列】來自教室前方的【右側】攝影機。\n*   **視線基準**: 對於這位學生，【正面朝向鏡頭】僅代表他在看教室的【右前方】。為了看向【教室正前方中心】，他的頭部必須輕微地朝向他自己的【左側】轉動。"
    else: # 預設為中間
        analysis_view_rules = "*   **分析視角**: 拍攝該學生的【學生個人照片序列】來自教室前方的【中間】攝影機。\n*   **視線基準**: 對於這位學生，當他臉部【正面朝向鏡頭】時，即代表其視線正朝向教室的【正前方中心】。"

    camera_and_spatial_rules = f"""
---
**【第一部分：多機位系統與空間推理框架 (最高優先級)】**
你的所有空間判斷都必須基於一個【兩步推理】的框架：首先建立宏觀地圖，然後應用微觀視角。

*   **準則 1A: 宏觀情境視角 (固定中央廣角鏡頭)**
    *   **定義**: 用於情境感知的【班級整體照片】**永遠**來自教室前方的**中央廣角鏡頭**。
    *   **任務**: 你必須使用這張照片來建立一個視覺座標系，將教室劃分為**左側、中央、右側**三個區域，並在其中定位目標學生。

*   **準則 1B: 微觀分析視角 (動態學生鏡頭)**
    *   {analysis_view_rules}

*   **通用空間準則**:
    *   **朝前弧度原則**: 對教室前方的專注行為（`V_TCH`, `V_BRD`）必須發生在一個合理的「朝前弧度」內。一個使學生頭部與其肩膀構成【接近90度】的**大幅度轉頭**，是其注意力【脫離前方】的明確證據，你**必須**將其優先判定為 `V_CLS`。
    *   **影像鏡像轉換**: 所有攝影機畫面都是鏡像的：照片中的**左側** => 教室中的**右側**；照片中的**右側** => 教室中的**左側**。

**【核心執行協議】**:
你的視線分析**必須**結合宏觀與微觀資訊。例如：在班級照片中，你確認學生位於**畫面左側區域** (準則 1A)。同時，`student_position` 文字告訴你拍攝他的鏡頭在**左側** (準則 1B)。綜合這兩點，你才能最準確地判斷他為了看向老師而需要轉動的角度。
"""

    # --- 3. 行為決策協議 (動態模塊) ---
    # 將前後排規則與統一決策協議整合成一個大的、連貫的決策流程
    is_front_row = any(keyword in student_position for keyword in ["第一", "第二"])
    is_rear_row = any(keyword in student_position for keyword in ["第三", "第四"])

    # 3.1 定義前後排獨有的規則
    front_row_rules = """
*   **低頭行為的分診決策樹 (軸線檢查版)**:
    1.  **軸線檢查**: 判斷頭部是【正下方】（判定 `V_BOK`）還是【側下方】（判定 `V_CLS`）。
    2.  **手部動作覆寫**: 檢查是否有隱蔽的非學習動作 (`H_PLAY_HW`) 來覆寫視線判斷。
    3.  **最終備用**: 都不是才使用 `P_DWN`。
*   **前方視線的精確匹配**: 嚴格結合【老師位置情境】進行幾何匹配，判定 `V_TCH` 或 `V_BRD`。
"""
    rear_row_rules = """
*   **低可視度下的姿態優先原則**: 當細節無法辨認時，必須忽略猜測，嚴格依賴姿態鐵則。
*   **低頭行為的「絕對學習推定」原則**: 只要頭部低垂，就**必須、無條件地**判定為 `V_BOK`。
*   **前方視線的「教師優先」原則**: 只要視線朝前，就**優先**標註為 `V_TCH`。
"""

    # 3.2 根據學生位置選擇對應規則
    if is_rear_row:
        position_specific_rules = f"""
**【第二部分：後排學生分析協議 (絕對推斷模式)】**
你必須遵循以下不可動搖的推斷鐵則：
{rear_row_rules}
"""
    else: # 預設為前排
        position_specific_rules = f"""
**【第二部分：前排學生分析協議 (高證據模式)】**
你的所有判斷都必須基於【嚴格的視覺證據】：
{front_row_rules}
"""

    # 3.3 定義統一的、後續的行為決策流程
    universal_decision_flow = """
---
**【第三部分：統一行為決策流程 (全體適用)】**
**【核心原則】**: 你的判斷【必須】基於每一張獨立圖片捕捉到的**瞬時物理證據**。在通過【第二部分】獲得初步的視線判斷後，你必須嚴格遵循以下【單一、統一】的分層決策樹來確定最終的主要行為標籤。

*   **第一層：檢查【壓倒性的身體姿態】(最高優先級)**
    *   **情況 A (向前協作姿態):** 身體【向前或向側前方】探出，進入鄰座同學的桌面空間 -> 主要行為判定為 **`V_BOK`**。**決策結束。**
    *   **情況 B (向後/側面社交姿態):** 身體向【側面或後方】旋轉，與同學進行面對面交流 -> 主要行為判定為 **`V_CLS`**。**決策結束。**

*   **第二層：檢查【定義明確的主動動作】(僅在身體朝前時評估)**
    *   是否在進行**動態的書寫 (`H_NOT`)**？ -> 是 -> `H_NOT` 是主要標籤。**決策結束。**
    *   是否在**玩弄手部/文具 (`H_PLAY_HW`)**？ -> 是 -> `H_PLAY_HW` 是主要標籤。**決策結束。**
    *   手臂是否抬起？ -> 是 -> 執行**【舉手行為過濾器】**（排除 `V_CLS`, `H_TOUCH_H/F` 後，判斷 `I_HND_A/P`）。**決策結束。**

*   **第三層：檢查【其他細微姿態與動作】**
    *   學生是否將頭**枕於**手臂或桌面？ -> 是 -> `P_SLP` (趴睡)。**決策結束。**
    *   學生是否在**托腮 (`P_THK`)**？ -> 是 -> `P_THK` 作為主要標籤。
    *   學生是否在進行**其他非任務相關動作**（如觸摸臉部 `H_TOUCH_F`）？ -> 是 -> 標註對應動作。

*   **第四層：標註【靜態視覺行為】(最終備用選項)**
    *   僅在**所有**上述檢查項都不適用的情況下，才使用在【第二部分】中獲得的**初步視線判斷**（如 `V_TCH`, `V_BOK`）作為最終的主要行為標籤。

*   **第五層：疊加【通用輔助姿態】**
    *   在確定了主要行為後，可疊加一個輔助的姿態標籤（如 `P_LEAN`）。
"""

    # --- 4. 生成最終的、完整的 Prompt ---
    teacher_context_header = "5.  **【老師位置文字情境】**: 用於交叉驗證你視線判斷的輔助數據。" if has_teacher_context else ""

    return f"""
「你是一位世界頂尖的教育分析師，精通電腦視覺、空間幾何推理與心理學。你的任務是綜合所有給定的物理與情境資訊，對學生的學習行為進行最精準、最客觀的標註。」

**【你收到的資訊來源】**
1.  **【班級整體照片】**: 你的【空間座標系】基準。
2.  **【學生個人照片序列】**: 你的主要分析對象。
3.  **【課堂狀態】**: 判斷行為動機的核心上下文。
4.  **【學生座位文字描述】**: `{student_position}`，用於輔助定位。
{teacher_context_header}

{camera_and_spatial_rules}

{position_specific_rules}

{universal_decision_flow}

{behavior_table_for_prompt}

---
**【第四部分：輸出格式要求 - 嚴謹推理與簡明輸出】**
*   你的回答**必須**是一個結構完整的、單一的 JSON 物件。
*   `behavior_category` 的值**必須**是一個包含 1 到 2 個編碼字串的**陣列 (Array)**。
*   `per_image_highlights` 的每個物件中**必須包含** `image_index_in_sequence`, `context_description`, `behavior_category`, 和 `confidence` 這四個鍵。
*   **【關鍵指令】**: `context_description` 內容**必須極度簡潔**，例如："後排協議 -> 低頭推定" 或 "左側區域規則 -> 視線匹配"。

**【輸出 JSON 格式範例】**
```json
{{
  "sequence_analysis_confidence": 0.97,
  "per_image_highlights": [
    {{
      "image_index_in_sequence": 0,
      "context_description": "後排協議：低頭推定。",
      "behavior_category": ["V_BOK", "P_STR"],
      "confidence": 0.95
    }},
    {{
      "image_index_in_sequence": 1,
      "context_description": "左側區域規則：頭部右轉，匹配老師位置。",
      "behavior_category": ["V_TCH"],
      "confidence": 0.99
    }},
    {{
      "image_index_in_sequence": 2,
      "context_description": "社交優先：向前協作，判定為 V_BOK。",
      "behavior_category": ["V_BOK", "P_LEAN"],
      "confidence": 0.98
    }}
  ]
}}
```"""

def default_error_result_structure():
    return { "error": "分析失敗或無有效數據", "sequence_analysis_confidence": 0.0, "sequence_summary": "未能生成序列總結。", "dominant_sustained_behaviors": [], "significant_behavior_shifts": [], "per_image_highlights": [], "general_sequence_atmosphere_hint": "未知" }

def clean_analysis_json(raw_analysis):
    """
    清理從 API 返回的分析 JSON，只保留我們需要的欄位。
    這是一個防禦性措施，用來處理微調模型的「幻覺」問題。
    """
    if not isinstance(raw_analysis, dict):
        return default_error_result_structure()

    # 定義合法的鍵
    allowed_top_level_keys = {"sequence_analysis_confidence", "per_image_highlights"}
    allowed_highlight_keys = {"image_index_in_sequence", "context_description", "behavior_category", "confidence"}

    cleaned_analysis = {}
    
    # 1. 清理最外層的鍵
    for key, value in raw_analysis.items():
        if key in allowed_top_level_keys:
            cleaned_analysis[key] = value

    # 2. 檢查並清理 per_image_highlights 列表
    if "per_image_highlights" in cleaned_analysis and isinstance(cleaned_analysis["per_image_highlights"], list):
        cleaned_highlights = []
        for raw_highlight in cleaned_analysis["per_image_highlights"]:
            if not isinstance(raw_highlight, dict):
                continue # 如果列表中的元素不是字典，就跳過它
            
            cleaned_highlight = {}
            for key, value in raw_highlight.items():
                if key in allowed_highlight_keys:
                    cleaned_highlight[key] = value
            
            # 確保必要的鍵存在，即使為空
            for required_key in allowed_highlight_keys:
                if required_key not in cleaned_highlight:
                    cleaned_highlight[required_key] = None

            cleaned_highlights.append(cleaned_highlight)
        
        cleaned_analysis["per_image_highlights"] = cleaned_highlights

    # 3. 如果最外層缺少必要的鍵，補上預設值
    if "sequence_analysis_confidence" not in cleaned_analysis:
        cleaned_analysis["sequence_analysis_confidence"] = 0.0
    if "per_image_highlights" not in cleaned_analysis:
        cleaned_analysis["per_image_highlights"] = []

    return cleaned_analysis

def analyze_student_behavior_from_images_sequence(student_image_paths, teacher_position_text, classroom_view_image_path, classroom_state, image_filenames_batch, openai_client, student_id, student_position):
    """
    【v8.1 穩定版】分析單一圖片批次，使用預處理好的「課堂狀態」標籤。
    此版本包含了完整的錯誤處理、JSON清理和穩健性增強。
    """
    if not student_image_paths:
        return default_error_result_structure() # 直接返回錯誤結構，而不是字典
    
    user_message_content = []
    
    # 1. 編碼學生個人圖片序列 (使用配置中的解析度)
    encoded_student_images = []
    for img_path in student_image_paths:
        b64_img = encode_image_to_base64(img_path)
        if b64_img:
            encoded_student_images.append({
                "type": "image_url", 
                "image_url": {"url": b64_img, "detail": IMAGE_DETAIL_LEVEL}
            })

    if not encoded_student_images:
        return default_error_result_structure() # 直接返回錯誤結構
        
    # 2. 編碼班級整體照片 (使用配置中的解析度)
    encoded_classroom_view_image = None
    if classroom_view_image_path:
        b64_img = encode_image_to_base64(classroom_view_image_path)
        if b64_img:
            encoded_classroom_view_image = {
                "type": "image_url", 
                "image_url": {"url": b64_img, "detail": IMAGE_DETAIL_LEVEL}
            }

    # 3. 組合完整的請求體，包含所有情境信息
    user_message_content.append({"type": "text", "text": f"請根據系統提示中的偵探任務，分析學生「{student_id}」的行為。學生照片序列的文件名（供您參考）為: {', '.join(image_filenames_batch)}。"})
    
    # 加入課堂狀態標籤
    state_context_prompt = f"【課堂狀態】: {classroom_state}"
    user_message_content.append({"type": "text", "text": state_context_prompt})

    # 加入老師位置情境
    teacher_context_prompt = f"【老師位置情境】根據預先分析，在此時間段，老師的位置在教室前方的「{teacher_position_text}」。請以此作為判斷『目視教師』的核心依據。"
    user_message_content.append({"type": "text", "text": teacher_context_prompt})
    
    # 依序加入班級照片和學生照片
    if encoded_classroom_view_image:
        user_message_content.append({"type": "text", "text": "【班級整體照片】(用於定位學生和觀察整體氛圍)"})
        user_message_content.append(encoded_classroom_view_image)
    user_message_content.append({"type": "text", "text": "【學生個人照片序列】(主要分析對象)"})
    user_message_content.extend(encoded_student_images)

    # 4. 獲取新的、具備情境感知能力的系統提示
    system_prompt_content = get_behavior_sequence_analysis_system_prompt(student_position, bool(teacher_position_text and teacher_position_text != "未知"))

    # 5. API 呼叫與錯誤處理
    retry_attempts = 2
    for attempt in range(retry_attempts + 1):
        raw_content = "" # 初始化 raw_content 以免在 except 區塊中引用未定義變數
        try:
            has_teacher_context = bool(teacher_position_text and teacher_position_text != "未知")
            has_classroom_image = bool(encoded_classroom_view_image)
            has_state = classroom_state != "未知"

            print(f"  正在向 {VISION_DEPLOYMENT_NAME} 發送請求 (學生: {len(encoded_student_images)}, 課堂狀態: {'有' if has_state else '無'}, 老師位置: {'有' if has_teacher_context else '無'}, 班級照片: {'有' if has_classroom_image else '無'})...")
            
            response = openai_client.chat.completions.create(
                model=VISION_DEPLOYMENT_NAME, 
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": system_prompt_content},
                    {"role": "user", "content": user_message_content}
                ],
                max_tokens=MAX_TOKENS_VISION_COMPLETION,  # gpt-4.1
                temperature=0.05 # gpt-4.1
                # max_completion_tokens=MAX_TOKENS_VISION_COMPLETION, #gpt-5
                
            )
            raw_content = response.choices[0].message.content
            
            # --- ✅【核心修正點】---
            # 步驟 1: 解析原始回應
            analysis_result_raw = json.loads(raw_content)
            
            # 步驟 2: 清理可能包含幻覺的 JSON，確保格式正確
            analysis_result = clean_analysis_json(analysis_result_raw)
            # --- ✅【修正結束】---

            # 本地解碼，將行為編碼轉換回中文標籤列表
            if "per_image_highlights" in analysis_result and isinstance(analysis_result["per_image_highlights"], list):
                for hl in analysis_result["per_image_highlights"]:
                    # 增加穩健性檢查，防止因清理後的 None 值導致錯誤
                    if not hl or not isinstance(hl, dict) or "behavior_category" not in hl:
                        continue
                    
                    codes = hl.get("behavior_category")
                    if not codes: # 處理 behavior_category 為 None 的情況
                        continue

                    if not isinstance(codes, list):
                        codes = [codes]
                    
                    # 過濾掉可能是 None 的 code
                    decoded_behaviors = [CODE_TO_BEHAVIOR.get(code, f"未知編碼({code})") for code in codes if code]
                    hl["behavior_category"] = decoded_behaviors
            
            # 補上空的摘要欄位以保持格式統一 (這一步可以保留，也可以移除，因為 clean_analysis_json 會確保結構)
            if "sequence_summary" not in analysis_result:
                analysis_result["sequence_summary"] = "已設定為精簡模式，此欄位由本地生成。"

            return analysis_result

        except json.JSONDecodeError:
            # 這裡的邏輯是正確的：如果 JSON 本身語法錯誤，就記錄並重試
            print(f"    錯誤：無法解析JSON (第 {attempt+1} 次嘗試)。回應: {raw_content[:500]}...");
            time.sleep(5) # 增加短暫延遲
        except RateLimitError:
            print(f"  警告：API速率限制，等待 {API_RETRY_DELAY_SECONDS}s 後重試...");
            time.sleep(API_RETRY_DELAY_SECONDS)
        except APIError as e:
            wait_time = 10 + attempt * 5 
            print(f"  錯誤：API錯誤 (第 {attempt+1} 次嘗試): {e}. 將在 {wait_time} 秒後重試...");
            time.sleep(wait_time)
        except Exception as e:
            # 處理 NameError 和其他所有未知錯誤
            wait_time = 5 + attempt * 5
            print(f"  錯誤：未知錯誤 (第 {attempt+1} 次嘗試): {e}. 將在 {wait_time} 秒後重試...");
            time.sleep(wait_time)
    
    print(f"  錯誤：圖片序列分析在多次重試後失敗。")
    return default_error_result_structure()

def generate_personalized_summary_notes(student_id, overall_stats, non_task_highlights, openai_client):
    if not openai_client: return {"error": "OpenAI client not available"}
    # ... (此函數內部的 Prompt 內容完全不需要修改) ...
    stats_summary_for_ai = "\n".join([f"- {s['behavior_category']}: {s['percentage']:.1f}% ({s['count']}次)" for s in overall_stats[:7]]) # 確保百分比格式
    non_task_prompt_part = "該生在本堂課中，未觀察到明顯或頻繁的非任務相關行為。"
    if non_task_highlights:
        highlights_str = "\n".join([f"  - 圖 '{img_fn}' (~{ts}): '{beh}' (描述: {desc})" for img_fn, ts, beh, desc in non_task_highlights[:3]])
        if highlights_str: non_task_prompt_part = f"在本堂課中，觀察到一些非任務相關行為，例如：\n{highlights_str}\n這可能影響了學習專注度。"
    prompt_for_summary = f"""
    「你是一位專業且富有同理心的學習行為教練。你的目標不是批評，而是透過客觀數據，引導學生發現自己的學習模式，並提供能立即實踐的策略，以激發他們『自我反思』的動力。」

    **任務：** 為學生「{student_id}」撰寫一份「AI學習夥伴的觀察與建議」。

    **學生的課堂行為數據：**
    *   **主要行為分佈:**
    {stats_summary_for_ai}
    *   **值得注意的行為片段:**
    {non_task_prompt_part}

    **撰寫指引與 JSON 格式要求：**
    請嚴格遵循以下指引，產生一個結構完整的 JSON 物件。

    *   **`greeting` (問候語):**
        *   用親切、個人化的方式稱呼學生，例如：「嗨，{student_id} 同學，一起來看看這次課堂的學習足跡吧！」

    *   **`positive_feedback` (亮點觀察):**
        *   **必須**從數據中找出最值得肯定的行為（例如「目視教師」或「筆記」佔比最高），並給予具體、真誠的讚美。
        *   **範例**：「我發現你在這堂課有超過一半的時間都在『目視教師』，這代表你非常努力地跟上老師的節奏，非常棒！」

    *   **`observation_points_summary` (行為模式提醒):**
        *   客觀、中性地指出一個或兩個最主要的、可能影響學習的行為模式。避免使用負面詞彙。
        *   **範例**：「數據也顯示，大約有 15% 的時間出現了『玩弄物品』或『目視他處』的狀況，這些時刻可能讓我們不小心錯過了一些重點喔。」

    *   **`reflection_points` (反思引導提問):**
        *   **【此項最為關鍵】** 根據前面的觀察點，設計 2-3 個**開放式問題**，引導學生思考行為背後的原因，而不是直接給答案。
        *   **問題範例 1**：「我們可以一起回想看看，當出現『玩弄物品』的時候，通常是在課程的哪個階段呢？是覺得內容太簡單、太難，還是剛好有點疲倦了呢？」
        *   **問題範例 2**：「當視線看向其他地方時，是想到了什麼有趣的事，還是被教室裡的其他動靜吸引了呢？了解這些原因，能幫助我們找到最適合自己的專注方法。」

    *   **`suggestions` (可實踐的小建議):**
        *   提供 1-2 個**具體、微小、且容易執行**的行動建議。不要說「要專心」，而是給出方法。
        *   **建議範例 1**：「下次當你發現自己開始無意識地轉筆時，可以試著把它輕輕放下，然後做一個深呼吸，再重新將目光移回老師或課本上。」
        *   **建議範例 2**：「如果感覺到疲倦或分心，可以試試看『筆記專注法』：在筆記本上寫下老師說的任何一個關鍵字，這個小動作能幫助我們的大腦重新連線！」

    *   **`encouragement` (鼓勵與結語):**
        *   用一句溫暖、有力的話作結，強調這份報告是幫助他成長的工具。
        *   **範例**：「每一次的觀察都是為了讓我們更了解自己。相信你透過這些小小的調整，一定能發揮出自己最大的潛力，加油！」
    """
    retry_attempts = 2
    for attempt in range(retry_attempts):
        try:
            print(f"  正在為學生 {student_id} 使用 {SUMMARY_DEPLOYMENT_NAME} 生成個性化總結...") # 修改了日誌輸出
            response = client.chat.completions.create(
                # ==========================================================
                # ↓↓↓ 【關鍵修改】使用我們新設定的經濟型模型 ↓↓↓
                # ==========================================================
                model=SUMMARY_DEPLOYMENT_NAME, 
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": "你是一位富有同理心和洞察力的教育顧問。"},
                          {"role": "user", "content": prompt_for_summary}],
                max_tokens=MAX_TOKENS_SUMMARY_COMPLETION, #gpt-4.1
                temperature=0.7 #gpt-4.1
                # max_completion_tokens=MAX_TOKENS_SUMMARY_COMPLETION, # gpt-5        
                
            )
            summary_data = json.loads(response.choices[0].message.content)
            expected_keys = ["greeting", "positive_feedback", "observation_points_summary", "reflection_points", "suggestions", "encouragement"]
            if all(key in summary_data for key in expected_keys): 
                print(f"  成功為學生 {student_id} 生成個性化總結。")
                return summary_data
            else: 
                print(f"    警告：個性化總結JSON缺少鍵。返回: {summary_data}")
                return {key: summary_data.get(key, f"AI未能生成 ({key})") for key in expected_keys}
        except Exception as e: 
            print(f"  生成個性化總結錯誤 ({attempt + 1}): {e}")
            time.sleep(API_RETRY_DELAY_SECONDS if isinstance(e, RateLimitError) else 5)
        if attempt == retry_attempts - 1: 
            print(f"  錯誤：無法為學生 {student_id} 生成個性化總結。")
            return {"greeting": f"親愛的 {student_id},", "positive_feedback": "總結生成遇到問題。", "observation_points_summary": "請參考統計數據。", "reflection_points": "未能生成。", "suggestions": "請自行評估。", "encouragement": "加油！"}
    return {}

def find_closest_image_path(representative_timestamp, sorted_photo_list, max_time_diff_seconds=5):
    """
    一個可重用的輔助函數，用二分查找法在排序好的照片列表中找到時間最接近的照片路徑。
    """
    if not sorted_photo_list:
        return None

    all_timestamps = [item[0] for item in sorted_photo_list]
    # bisect_left 找到應該插入的位置
    insertion_point = bisect.bisect_left(all_timestamps, representative_timestamp)

    closest_path = None
    min_diff = datetime.timedelta.max

    # 只檢查插入點及其前後的幾個候選照片，效率極高
    # 檢查範圍設為 insertion_point-2 到 insertion_point+2 以增加容錯
    start_index = max(0, insertion_point - 2)
    end_index = min(len(sorted_photo_list), insertion_point + 2)

    for i in range(start_index, end_index):
        candidate_ts, candidate_path = sorted_photo_list[i]
        diff = abs(candidate_ts - representative_timestamp)
        if diff < min_diff:
            min_diff = diff
            closest_path = candidate_path

    # 只有在時間差在容許範圍內才返回路徑
    if closest_path and min_diff.total_seconds() <= max_time_diff_seconds:
        return closest_path
    
    return None

def find_closest_position(representative_timestamp, sorted_position_list, max_time_diff_seconds=10):
    """
    用二分查找法在排序好的位置列表中找到時間最接近的老師位置。
    """
    if not sorted_position_list:
        return "未知"

    all_timestamps = [item[0] for item in sorted_position_list]
    insertion_point = bisect.bisect_left(all_timestamps, representative_timestamp)

    closest_position = "未知"
    min_diff = datetime.timedelta.max

    start_index = max(0, insertion_point - 2)
    end_index = min(len(sorted_position_list), insertion_point + 2)

    for i in range(start_index, end_index):
        candidate_ts, candidate_pos = sorted_position_list[i]
        diff = abs(candidate_ts - representative_timestamp)
        if diff < min_diff:
            min_diff = diff
            closest_position = candidate_pos
    
    if min_diff.total_seconds() <= max_time_diff_seconds:
        return closest_position
    
    return "未知"

def process_single_batch(batch_idx, image_batch_info, teacher_positions_data, sorted_classroom_photos, classroom_states, client, student_id, student_position):
    """
    【升級版】處理單一圖片批次，使用老師位置和預處理好的課堂狀態。
    """
    batch_student_paths = [info["path"] for info in image_batch_info]
    batch_image_filenames = [info["filename"] for info in image_batch_info]
    
    representative_timestamp = image_batch_info[0]["timestamp_obj"]

    # --- 為當前批次查找所有情境數據 ---
    teacher_position_text = find_closest_position(representative_timestamp, teacher_positions_data)
    classroom_view_path = find_closest_image_path(representative_timestamp, sorted_classroom_photos)
    # 【關鍵修正】使用正確的變數名稱 classroom_states
    classroom_state = find_state_for_timestamp(representative_timestamp, classroom_states)

    # --- 呼叫【新版】分析函數 ---
    sequence_analysis_data = analyze_student_behavior_from_images_sequence(
        student_image_paths=batch_student_paths,
        teacher_position_text=teacher_position_text,
        classroom_view_image_path=classroom_view_path,
        classroom_state=classroom_state, # <-- 傳入狀態標籤
        image_filenames_batch=batch_image_filenames,
        openai_client=client,
        student_id=student_id,
        student_position=student_position
    )

    return {
        "batch_index": batch_idx,
        "image_batch_info": image_batch_info,
        "matched_teacher_position_text": teacher_position_text,
        "matched_classroom_view_image": os.path.basename(classroom_view_path) if classroom_view_path else None,
        "matched_classroom_state": classroom_state,
        "analysis": sequence_analysis_data
    }

def main():
    print("--- 學生課堂學習行為分析報告生成 (JSON) v5.0 - 成本優化版 ---")
    print(f"視覺模型: {VISION_DEPLOYMENT_NAME}, 文本模型: {VISION_DEPLOYMENT_NAME}, 總結模型: {VISION_DEPLOYMENT_NAME}")
    print(f"圖片批次大小: {IMAGES_PER_API_CALL}, 圖片取樣率: 1/{SAMPLING_RATE}, 圖片解析度: {IMAGE_DETAIL_LEVEL}")
    print("-" * 40)

    # --- 使用者輸入部分 ---
    student_id = get_valid_input("請輸入學生姓名 (可中文): ")
    student_number = get_valid_input("請輸入學生座號 (例如: 1): ") 
    student_images_folder = get_valid_folder_path(f"請輸入 '{student_id}' 的個人影像資料夾: ")
    print("\n--- 請輸入課堂情境資訊 ---")
    student_position = get_valid_input("請輸入學生座位 (例如: '第3排中間', '第1排左側'): ")
    print("-" * 50)

    # --- 準備輸出資料夾 ---
    safe_student_id_for_folder = re.sub(r'[\\/*?:"<>|]',"", student_id.replace(" ", "_"))
    student_specific_folder_base = os.path.join(JSON_OUTPUT_FOLDER, safe_student_id_for_folder)
    os.makedirs(student_specific_folder_base, exist_ok=True)

    # --- 準備學生照片資料 ---
    image_files_with_timestamps = []
    valid_extensions = ('.png', '.jpg', '.jpeg', '.webp')
    print(f"正在掃描學生個人照片資料夾: {student_images_folder}...")
    for filename in os.listdir(student_images_folder):
        if filename.lower().endswith(valid_extensions):
            timestamp = get_timestamp_from_filename(filename)
            if timestamp:
                image_files_with_timestamps.append({
                    "path": os.path.join(student_images_folder, filename), "filename": filename,
                    "timestamp_obj": timestamp, # <-- 直接使用原始時間戳，不做任何校準
                    "timestamp_str": str(timestamp).split('.')[0]
                })
    image_files_with_timestamps.sort(key=lambda x: x["timestamp_obj"])
    
    # ==========================================================
    # ↓↓↓ 【新增】圖片取樣以降低成本 ↓↓↓
    # ==========================================================
    if SAMPLING_RATE > 1:
        original_count = len(image_files_with_timestamps)
        image_files_with_timestamps = image_files_with_timestamps[::SAMPLING_RATE]
        print(f"已執行圖片取樣：從 {original_count} 張原始照片中，每 {SAMPLING_RATE} 張取 1 張，共 {len(image_files_with_timestamps)} 張照片將被分析。")
    else:
        print(f"成功找到 {len(image_files_with_timestamps)} 張學生個人照片 (未取樣)。")

    if not image_files_with_timestamps: 
        print("錯誤：學生資料夾中未找到有效時間格式的圖片，或取樣後為空。")
        return
    # ==========================================================

    # ... 後續的程式碼，從 `load_teacher_positions` 開始，到整個 `main` 函數結束，
    # 都不需要再做任何修改。您可以直接使用您原有的版本。
    # 我將剩餘部分貼在下方以保證完整性。
    
    def load_teacher_positions(json_path, start_time_offset_str=""): # 新增 start_time_offset_str 參數
        if not json_path or not os.path.isfile(json_path):
            print(f"提示：未提供或找不到老師位置 JSON 檔案 ({json_path})。將不使用老師位置情境。")
            return []
        start_offset = parse_time_offset(start_time_offset_str)
        print(f"正在讀取老師位置數據: {json_path}...")
        try:
            with open(json_path, 'r', encoding='utf-8') as f: data = json.load(f)
            position_map, original_count = [], len(data)
            for item in data:
                try:
                    h, m, s = map(int, item['timestamp'].split(':'))
                    td = datetime.timedelta(hours=h, minutes=m, seconds=s)
                    if start_offset and td < start_offset: continue
                    position_map.append((td, item['position']))
                except (ValueError, KeyError): continue
            position_map.sort(key=lambda x: x[0])
            if start_offset: print(f"成功加載老師位置數據。根據起始時間 '{start_time_offset_str}' 進行過濾，保留 {len(position_map)} 筆 (原 {original_count} 筆)。")
            else: print(f"成功加載並索引了 {len(position_map)} 筆老師位置數據。")
            return position_map
        except Exception as e:
            print(f"錯誤：讀取或解析老師位置 JSON 時發生問題: {e}"); return []

    def load_and_sort_photos(folder_path, photo_type_name, start_time_offset_str=""): # 新增 start_time_offset_str 參數
        if not folder_path or not os.path.isdir(folder_path):
            print(f"提示：未提供或找不到 {photo_type_name} 照片資料夾 ({folder_path})。"); return []
        start_offset = parse_time_offset(start_time_offset_str)
        print(f"正在掃描 {photo_type_name} 照片資料夾: {folder_path}...")
        photo_list, original_count = [], 0
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                original_count += 1
                timestamp = get_timestamp_from_filename(filename)
                if timestamp:
                    if start_offset and timestamp < start_offset: continue
                    photo_list.append((timestamp, os.path.join(folder_path, filename)))
        if photo_list:
            sorted_photos = sorted(photo_list)
            if start_offset: print(f"成功加載 {photo_type_name} 照片。根據起始時間 '{start_time_offset_str}' 進行過濾，保留 {len(sorted_photos)} 張 (原 {original_count} 張有效格式照片)。")
            else: print(f"成功加載並索引了 {len(sorted_photos)} 張 {photo_type_name} 照片。")
            return sorted_photos
        else:
            print(f"警告：在 {photo_type_name} 照片資料夾 '{folder_path}' 中未找到符合條件的圖片。"); return []

    teacher_positions_data = load_teacher_positions(TEACHER_POSITION_JSON, TEACHER_JSON_START_TIME_OFFSET)
    sorted_classroom_photos = load_and_sort_photos(CLASSROOM_IMAGES_FOLDER, "班級整體", CLASSROOM_START_TIME_OFFSET)
    classroom_states = load_classroom_states(CLASSROOM_STATE_JSON_PATH)

    image_batches = [image_files_with_timestamps[i:i + IMAGES_PER_API_CALL] for i in range(0, len(image_files_with_timestamps), IMAGES_PER_API_CALL)]
    print(f"學生圖片將被分為 {len(image_batches)} 個批次進行分析。")

    MAX_WORKERS = 4
    print(f"將使用最多 {MAX_WORKERS} 個執行緒進行平行分析...")
    all_results_from_threads = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_batch_idx = { 
            executor.submit(process_single_batch, idx, batch_info, teacher_positions_data, sorted_classroom_photos, classroom_states, client, student_id, student_position): idx 
            for idx, batch_info in enumerate(image_batches) 
        }
        for future in tqdm(as_completed(future_to_batch_idx), total=len(image_batches), desc=f"分析學生 {student_id} 的圖片批次"):
            try:
                result = future.result()
                all_results_from_threads.append(result)
            except Exception as exc:
                batch_idx = future_to_batch_idx[future]
                print(f'\n批次 {batch_idx + 1} 執行時產生錯誤: {exc}')
                all_results_from_threads.append({"batch_index": batch_idx, "error": str(exc)})

    all_results_from_threads.sort(key=lambda x: x['batch_index'])
    print("\n所有批次分析完成，開始匯總數據...")
    all_sequence_analysis_results = []
    overall_behavior_summary = { "total_images_processed_in_batches": 0, "behavior_counts": Counter(), "behavior_confidence_sum": defaultdict(float), "non_task_behavior_examples_for_summary": [] }
    for result in tqdm(all_results_from_threads, desc="匯總分析結果"):
        if "error" in result:
            all_sequence_analysis_results.append({"batch_index": result['batch_index'] + 1, "analysis": default_error_result_structure()})
            continue
        image_batch_info, sequence_analysis_data = result["image_batch_info"], result["analysis"]
        batch_image_filenames = [info["filename"] for info in image_batch_info]
        all_sequence_analysis_results.append({ "batch_index": result['batch_index'] + 1, "image_filenames_in_batch": batch_image_filenames, "matched_teacher_position_text": result.get("matched_teacher_position_text"), "matched_classroom_view_image": result.get("matched_classroom_view_image"), "analysis": sequence_analysis_data })
        if "error" not in sequence_analysis_data:
            overall_behavior_summary["total_images_processed_in_batches"] += len(batch_image_filenames)
            if "per_image_highlights" in sequence_analysis_data:
                for hl_item in sequence_analysis_data.get("per_image_highlights", []):
                    
                    behavior_list_original = hl_item.get("behavior_category")
                    conf = hl_item.get("confidence", 0.0)
                    
                    UNKNOWN_LABEL = "被遮擋/無法判斷"
                    
                    # 複製一份原始行為列表，用於後續的非任務行為判斷
                    behavior_list_for_stats = behavior_list_original
                    
                    # 判斷條件 1: 信度是否低於設定的閾值
                    is_low_confidence = conf is not None and conf < BEHAVIOR_CONFIDENCE_THRESHOLD_FILTER
                    # 判斷條件 2: AI 是否沒有返回任何有效的行為標籤
                    is_empty_behavior = not behavior_list_original

                    # 如果信度過低或行為為空，則強制將此筆紀錄歸類為 "無法判斷"
                    if is_low_confidence or is_empty_behavior:
                        behavior_list_for_stats = [UNKNOWN_LABEL]
                        
                    if not behavior_list_for_stats: continue # 如果處理後仍然為空，則跳過

                    # 確保用於統計的行為列表是 list 格式
                    if not isinstance(behavior_list_for_stats, list):
                        behavior_list_for_stats = [behavior_list_for_stats]
                    
                    # 遍歷列表中的每一個行為進行統計 (此處使用的是可能被覆寫過的 behavior_list_for_stats)
                    for cat in behavior_list_for_stats:
                        overall_behavior_summary["behavior_counts"][cat] += 1
                        overall_behavior_summary["behavior_confidence_sum"][cat] += float(conf if conf is not None else 0.0)
                    
                    # 處理非任務行為範例（使用未經過濾的原始行為）
                    if behavior_list_original and isinstance(behavior_list_original, list):
                        primary_behavior_for_example = behavior_list_original[0]
                        non_task_keywords = [
                            "玩弄手部/文具", "觸摸臉部", "觸摸頭髮",
                            "目視他處", "趴睡", "喝水", "飲食"
                        ]
                        if primary_behavior_for_example in non_task_keywords and len(overall_behavior_summary["non_task_behavior_examples_for_summary"]) < 3:
                            try:
                                student_img_idx = hl_item.get("image_index_in_sequence", -1)
                                if 0 <= student_img_idx < len(batch_image_filenames):
                                    hl_filename = batch_image_filenames[student_img_idx]
                                    hl_timestamp = next((info["timestamp_str"] for info in image_batch_info if info["filename"] == hl_filename), "未知時間")
                                    context_desc = hl_item.get("context_description", "")
                                    
                                    overall_behavior_summary["non_task_behavior_examples_for_summary"].append( 
                                        (hl_filename, hl_timestamp, ", ".join(behavior_list_original), context_desc) 
                                    )
                            except Exception as e_idx: print(f"提取非任務示例時出錯: {e_idx}")
    
    overall_behavior_stats_list, total_highlight_instances = [], sum(overall_behavior_summary["behavior_counts"].values())
    valence_summary = {"正向": 0, "負向": 0, "中性": 0}
    for behavior, count in overall_behavior_summary["behavior_counts"].items():
        percentage, avg_confidence = (count / total_highlight_instances * 100) if total_highlight_instances > 0 else 0, (overall_behavior_summary["behavior_confidence_sum"][behavior] / count) if count > 0 else 0
        valence = LABEL_TO_VALENCE.get(behavior, "未分類")
        if valence in valence_summary: valence_summary[valence] += count
        overall_behavior_stats_list.append({ "behavior_category": behavior, "valence": valence, "count": count, "percentage": round(percentage, 1), "average_confidence": round(avg_confidence, 2) })
    overall_behavior_stats_list.sort(key=lambda x: x["count"], reverse=True)
    personalized_notes = generate_personalized_summary_notes(student_id, overall_behavior_stats_list, overall_behavior_summary["non_task_behavior_examples_for_summary"], client)
    behavior_to_images_map = defaultdict(list)
    for result in all_results_from_threads:
        if "error" in result or "analysis" not in result or "error" in result["analysis"]: continue
        sequence_analysis, image_batch_info, filenames_in_batch = result["analysis"], result.get("image_batch_info", []), [info.get("filename") for info in result.get("image_batch_info", [])]
        if "per_image_highlights" in sequence_analysis and isinstance(sequence_analysis["per_image_highlights"], list):
            for highlight in sequence_analysis["per_image_highlights"]:
                # 【修改點 4】修改索引建立邏輯以處理行為列表
                behavior_list, image_index = highlight.get("behavior_category"), highlight.get("image_index_in_sequence")
                
                if not behavior_list or not isinstance(image_index, int) or not (0 <= image_index < len(filenames_in_batch)):
                    continue

                if not isinstance(behavior_list, list):
                    behavior_list = [behavior_list]
                
                image_filename = filenames_in_batch[image_index]
                if image_filename:
                    # 為列表中的每一個行為都建立索引
                    for behavior_category in behavior_list:
                        if image_filename not in behavior_to_images_map[behavior_category]:
                            behavior_to_images_map[behavior_category].append(image_filename)
    for behavior in behavior_to_images_map: behavior_to_images_map[behavior].sort()
    print("行為索引建立完成。")

    total_classified_instances = sum(valence_summary.values())
    valence_summary_with_percentage = { valence: { "count": count, "percentage": round((count / total_classified_instances * 100), 1) if total_classified_instances > 0 else 0 } for valence, count in valence_summary.items() }

    final_json_output = {
        "report_metadata": {
            "student_id": student_id, "student_number": student_number, "report_generation_time": "08/17", "student_image_source_folder": "英文",
            "teacher_position_source_json": os.path.basename(TEACHER_POSITION_JSON) if TEACHER_POSITION_JSON and os.path.isfile(TEACHER_POSITION_JSON) else "N/A",
            "classroom_view_source_folder": os.path.basename(CLASSROOM_IMAGES_FOLDER) if CLASSROOM_IMAGES_FOLDER and os.path.isdir(CLASSROOM_IMAGES_FOLDER) else "N/A",            "classroom_context": { "student_position": student_position },
            "analysis_settings": {
                "vision_model": VISION_DEPLOYMENT_NAME, "text_model": TEXT_DEPLOYMENT_NAME, "images_per_batch": IMAGES_PER_API_CALL, "context_images_per_batch_desc": "動態匹配老師和班級照片各一張", "confidence_threshold": BEHAVIOR_CONFIDENCE_THRESHOLD_FILTER,
                "cost_optimization": { "sampling_rate": SAMPLING_RATE, "image_detail": IMAGE_DETAIL_LEVEL } # 新增成本優化資訊
            }
        },
        "overall_summary": {
            "total_images_found": len(image_files_with_timestamps) * SAMPLING_RATE if SAMPLING_RATE > 1 else len(image_files_with_timestamps), # 顯示原始數量
            "total_images_analyzed": overall_behavior_summary["total_images_processed_in_batches"],
            "total_batches": len(image_batches), "valence_summary": valence_summary_with_percentage, "behavior_statistics": overall_behavior_stats_list, "behavior_to_images_index": behavior_to_images_map, "ai_summary_notes": personalized_notes
        },
        "detailed_sequence_analysis": all_sequence_analysis_results
    }

    current_timestamp_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_student_id_for_filename = re.sub(r'[\\/*?:"<>|]',"", student_id.replace(" ", "_"))
    json_filename = JSON_FILENAME_TEMPLATE.format(student_id=safe_student_id_for_filename, timestamp=current_timestamp_str)
    json_filepath = os.path.join(student_specific_folder_base, json_filename)
    try:
        with open(json_filepath, 'w', encoding='utf-8') as f: json.dump(final_json_output, f, ensure_ascii=False, indent=4)
        print(f"\n✅ 學生 '{student_id}' 的序列行為分析報告已成功儲存至: {json_filepath}")
    except Exception as e: print(f"❌ 錯誤：儲存 JSON 檔案時發生問題：{e}")
    
    print("\n--- 處理完成 ---")

if __name__ == "__main__":
    main()

### Azure GPT-4.1 加上優化行為

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import base64
import json
import datetime
import time
from collections import Counter, defaultdict
from openai import AzureOpenAI, APIError, RateLimitError, AuthenticationError 
from PIL import Image, UnidentifiedImageError
import io
from tqdm import tqdm
import math
from dotenv import load_dotenv 
from concurrent.futures import ThreadPoolExecutor, as_completed
import bisect

# --- Configuration ---


# 班級整體照片資料夾路徑 
CLASSROOM_IMAGES_FOLDER = r'C:\Users\User\Desktop\test\student_full_classroom\0824_english_class'

# 【新】老師視角照片資料夾路徑 (從學生視角拍攝老師在黑板前的照片) - 可選
TEACHER_POSITION_JSON = r'C:\Users\User\Desktop\test\teacher_position\0824_position.json'

#課堂情況json
CLASSROOM_STATE_JSON_PATH = r'C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0824\老師\TXT\0824.json'

# 【新】時間軸對齊設定 (格式: "HH:MM:SS")
CLASSROOM_START_TIME_OFFSET = "00:13:47"  # 班級照片的起始時間，設定為空字串 "" 表示不篩選
TEACHER_JSON_START_TIME_OFFSET ="00:00:00" # 老師位置資料的起始時間，設定為空字串 "" 表示不篩選

# ... (您原有的其他設定) ...

JSON_OUTPUT_FOLDER = "SynologyDrive\json_behavior"


JSON_FILENAME_TEMPLATE = "student_{student_id}_behavior_report_{timestamp}.json"
SAMPLING_RATE = 3  
IMAGE_DETAIL_LEVEL = "low" # 圖片解析度 ('low' 或 'high')，low 可大幅降低成本
MAX_TOKENS_VISION_COMPLETION = 2500
MAX_TOKENS_SUMMARY_COMPLETION = 1000 # 個性化總結的 token
BEHAVIOR_CONFIDENCE_THRESHOLD_FILTER = 0.96 # 行為信度過濾閾值 (例如 70%)
IMAGES_PER_API_CALL = 10 # 一次API調用處理的圖片數量
BEHAVIOR_SIMILARITY_CONFIDENCE_THRESHOLD = 0.15 # 過濾相似連續行為的信度差異閾值
API_RETRY_DELAY_SECONDS = 10

# ==========================================================

# --- 標準行為分類與定義 (核心辭典，來自圖一) ---
STANDARD_BEHAVIOR_CATEGORIES = {
    "視線": [
        {"label": "目視教師", "definition": "學生的頭部與視線明確聚焦於教室【前方】的【老師所在】區域，且其頭部旋轉角度處於一個【合理的朝前弧度】內（通常不超過45度）。**【絕對排除條款】**: 任何導致學生視線與其身體朝向構成【接近90度或更大角度】的頭部大幅度轉動，【絕對不允許】被標註為『目視教師』。這種姿態應被優先考慮為『目視同學』(`V_CLS`)或『目視他處』(`V_ELS`)。"},
        {"label": "目視黑板", "definition": "學生的頭部與視線明確聚焦於教室【前方】的【非老師所在】的黑板或螢幕區域。當老師位置未知，或學生視線明確未朝向老師時，這是面向前方的預設專注行為。"}, 
        {"label": "目視書本/筆記", "definition": "學生的頭部與視線向量【主要朝下】，且頭部的水平旋轉角度【沒有明顯偏離】其身體所朝向的【個人桌面工作區軸線】。此標籤捕捉的是在個人學習材料上的視覺專注狀態。**【嚴格邊界】**: 一旦頭部/視線明確地、持續地轉向側面（例如，足以與鄰座同學進行眼神交流），即使視線仍然略微朝下，也【必須優先考慮】標註為『目視同學』。**【注意】**：如果學生同時在進行『做筆記』，根據『動作優先』原則，你應將『做筆記』作為主要標籤。"},
        {"label": "目視同學", "definition": "學生的【主要身體朝向與視線】明確脫離前方或個人桌面，轉向側面或後方的同學。**【最高優先級的情境標籤】**: 一旦觀察到這種明確的身體轉向，此標籤的優先級就高於大多數獨立的個人動作。它涵蓋了從純粹的視覺交流到【涉及物件的協作行為】（如共同看書、討論問題）。**【核心情境判斷規則】**: 此行為的性質根據【課堂狀態】決定..."},
        {"label": "目視他處", "definition": "【備用排除性標籤】。當你已確認學生的視線【不】符合『目視教師』、『目視黑板』、『目視書本/筆記』或『目視同學』的任何一項明確定義時，才使用此標籤。它捕捉的是失去焦點的狀態，例如：【抬頭看天花板】、【轉頭看沒有同學及老師的地方】。"}     
    ],
    "肢體(手部)": [
        {"label": "做筆記", "definition": "【核心證據：觀察到一個動態的書寫過程】。你必須能明確看到學生手持筆，且筆尖正在紙張上進行【有意義的移動或書寫/繪製動作】。**【最高優先級】**：這是一個高優先級的動作標籤。**【嚴格排除條款】**：以下情況【絕對不允許】標註為『做筆記』：(1) **靜態持筆**：僅僅手持筆，或筆尖靜止停留在紙上，應標註為『目視書本/筆記』 (`V_BOK`)。(2) **動作中斷**：當學生手部的主要動作變為其他行為（如『觸摸頭髮』、『托腮』、『與同學互動』），即使手中仍持有筆，也必須以該【瞬時動作】為主要標籤。"},
        {"label": "翻書", "definition": "學生手部正在主動翻閱、移動書本。"},
        {"label": "觸摸臉部", "definition": "學生【非支撐性地】用手短暫觸碰或摩擦自己的臉部、鼻子、嘴巴或眼睛。此行為區別於『托腮』的持續性支撐動作。"},
        {"label": "觸摸頭髮", "definition": "學生用手觸摸、撥弄或整理自己的頭髮。注意：即使手臂抬得較高，只要手部的主要動作是與頭髮互動，就【必須】使用此標籤，而不是『舉手』類標籤。"}
    ],
    "身體姿態": [
        {"label": "坐姿直立", "definition": "學生上半身軀幹基本垂直於地面，或輕微前傾。"},
        {"label": "身體前傾", "definition": "學生上半身軀幹明顯向前彎曲，靠近桌面。此行為描述的是一種【清醒狀態下】的姿態。如果學生頭部接觸桌面或手臂，應【優先使用】『趴睡』標籤。"},
        {"label": "身體後靠", "definition": "學生背部倚靠在椅背上。"},
        {"label": "低頭(非學習)", "definition": "【極其嚴格的排除性標籤】。僅在你能夠【極度確信地】觀察到以下【全部】條件時才可使用：1. 學生頭部明顯低垂。2. 其視線【明確沒有】朝向任何學習材料（例如，看向地面、自己的懷中或空無一物的桌面）。3. 其手部沒有在進行任何學習相關操作。"},
        {"label": "趴睡", "definition": "學生將頭部【枕於】手臂或桌面上，呈現明確的休息或睡眠狀態。**【最高優先級】**：只要觀察到頭部接觸桌面或手臂的休息姿態，此標籤的優先級【高於】所有其他學習相關標籤（如『做筆記』、『目視書本』）。"},
        {"label": "托腮", "definition": "【核心定義：手部對頭部提供持續性支撐】。學生使用一隻或兩隻手的手掌、拳頭或手臂，支撐其下巴、臉頰或頭部的重量。這是一個純粹的物理姿態描述，不包含任何意圖推斷。"}
    ],
    "互動": [
        {"label": "主動舉手", "definition": "學生舉起一隻手，意圖提問或回答問題。**【三大核心物理證據，必須同時滿足】**：(1) 手臂向上伸展，手部明顯高於肩膀。(2) 手掌形態為張開朝前或中性放鬆，【嚴禁】手指指向特定方向或揮舞。(3) 該動作具有一定的持續性（非瞬間劃過）。**【情境觸發】**：此行為最常發生在【老師單向授課】的狀態下（如『文法/句型講解』、『閱讀/文章分析』），代表學生的自發性提問或補充。**【絕對排除】**：任何手部接觸頭部、與同學互動的手勢、指向性的動作，都【不允許】標記為此行為。"},
        {"label": "被動舉手", "definition": "學生舉手以回應老師的群體性指令（如投票、調查）。**【核心視覺證據】**：通常是多數學生同時舉手，姿態可能較為放鬆，手臂不必完全伸直。**【情境觸發】**：此行為最常發生在【老師與學生互動】的狀態下（如『課堂問答/互動』、『習題/考卷檢討』），代表學生回應老師的指令或提問。**【關鍵區分】**：此標籤的判斷【高度依賴】課堂情境和群體性動作。如果情境不匹配，應避免使用此標籤。"}
    ],
    "其他狀態": [
        {"label": "喝水", "definition": "【核心證據：清晰可見的容器】。只有當你能夠【明確地】看到學生手持水瓶、杯子或其他容器，並將其送至嘴邊時，才可以使用此標籤。**【嚴格排除】**: 任何僅有低頭姿態、手部靠近臉部但【沒有可見容器】的場景，都【嚴禁】標註為此行為。在此情況下，應優先考慮『目視書本/筆記』(`V_BOK`)或『趴睡』(`P_SLP`)。"},
        {"label": "飲食", "definition": "學生正在食用固體食物。"},
        {"label": "玩弄手部／文具", "definition": "學生手部在進行與學習無關的重複性小動作，例如玩手指、轉筆、玩弄橡皮擦等"},
        # {"label": "整理書包", "definition": "學生正在整理書包、桌面文具。"},
        {"label": "被遮擋/無法判斷", "definition": "【最終備用標籤】。因遮擋、模糊或角度問題，無法清晰識別學生的主要行為時，【必須】使用此標籤。"}
    ]
}
# 自動從新結構生成有效的標籤列表
VALID_BEHAVIOR_LABELS = [item['label'] for category in STANDARD_BEHAVIOR_CATEGORIES.values() for item in category]

BEHAVIOR_CODES = {
    # 視線 (不變)
    "V_TCH": "目視教師", "V_BRD": "目視黑板", "V_BOK": "目視書本/筆記",
    "V_CLS": "目視同學", "V_ELS": "目視他處",
    
    # 肢體(手部) (移除 H_FLIP)
    "H_NOT": "做筆記",
    "H_PLAY_HW": "玩弄手部/文具",
    "H_TOUCH_F": "觸摸臉部",
    "H_TOUCH_H": "觸摸頭髮",
    
    # 身體姿態 (新增 P_THK - P for Posture, THK for Thinking)
    "P_STR": "坐姿直立", "P_LEAN": "身體前傾", "P_BACK": "身體後靠",
    "P_DWN": "低頭(非學習)", "P_SLP": "趴睡",
    "P_THK": "托腮", # <--- 新增
    
    # 互動 (將 I_HND 拆分為主動/被動)
    "I_HND_A": "主動舉手", # A for Active
    "I_HND_P": "被動舉手", # P for Passive
    
    # 其他狀態 (不變)
    "O_DRK_W": "喝水",
    "O_EAT_S": "飲食",
    # "O_TDY": "整理個人物品",
    "O_UNK": "被遮擋/無法判斷"
}

# 自動生成反向查找字典，用於本地解碼
CODE_TO_BEHAVIOR = {code: label for code, label in BEHAVIOR_CODES.items()}
BEHAVIOR_TO_CODE = {label: code for code, label in BEHAVIOR_CODES.items()}

# AI返回標籤到標準標籤的映射規則 (與新標籤對齊)
BEHAVIOR_MAPPING_RULES = {
    "目視桌面/教材": "目視書本", "目視桌面": "目視書本", "看書": "目視書本",
    "書寫/做筆記": "筆記", "動手操作-書寫/做筆記": "筆記",
    "視覺專注-閱讀書本/講義": "目視書本",
    "視覺專注-看老師/黑板方向": "目視教師", "目視黑板/老師": "目視教師",
    "看老師": "目視教師", "看黑板": "目視黑板",
    # --- ↓↓↓ 【修改點】更新映射規則以對應新標籤 ↓↓↓ ---
    "玩弄物品": "玩弄手部/文具", # 將模糊的舊標籤對應到最可能的新標籤
    "非任務相關動作-玩弄物品(筆等)": "玩弄手部/文具",
    # 移除了"非任務相關動作-觸摸臉部/頭髮"，鼓勵AI直接使用更精確的新標籤
    # "非任務相關動作-整理物品": "整理個人物品",
    "社交互動-與同學互動": "目視同學",
    "趴睡/休息": "趴睡",
    "低頭/伏案(非睡)": "低頭",
}

BEHAVIOR_VALENCE_MAP = {
    "正向": [
        "做筆記",
        "主動舉手", # <-- 修改
        "目視教師",
        "目視黑板",
        "目視書本/筆記",
    ],
    "負向": [
        "趴睡",
        "玩弄手部/文具",
        "觸摸臉部",
        "觸摸頭髮",
        "目視他處",
        "目視同學"
    ],
    "中性": [
        "身體前傾",
        "坐姿直立",
        "身體後靠",
        "喝水",
        "飲食",       
        "玩弄手部／文具",
        "翻書",
        "低頭(非學習)",
        # "整理個人物品",     
        "被遮擋/無法判斷",
        "托腮", # <-- 新增
        "被動舉手"  # <-- 新增
    ]
}

LABEL_TO_VALENCE = {
    label: valence 
    for valence, labels in BEHAVIOR_VALENCE_MAP.items() 
    for label in labels
}

# ---------------------------
#  API 金鑰配置
# ---------------------------
load_dotenv()

AZURE_API_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
# 從環境變數讀取部署名稱
VISION_DEPLOYMENT_NAME = os.getenv("CHAT_COMPLETION_NAME")
TEXT_DEPLOYMENT_NAME = os.getenv("CHAT_COMPLETION_NAME")
SUMMARY_DEPLOYMENT_NAME = os.getenv("CHAT_COMPLETION_NAME")

# 檢查必要的 Azure 配置是否存在
if not all([AZURE_API_KEY, AZURE_ENDPOINT, VISION_DEPLOYMENT_NAME, SUMMARY_DEPLOYMENT_NAME]):
    print("錯誤：缺少必要的 Azure OpenAI 環境變數。")
    print("請檢查您的 .env 檔案是否包含 AZURE_OPENAI_KEY, AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_VISION_DEPLOYMENT_NAME, 和 AZURE_OPENAI_SUMMARY_DEPLOYMENT_NAME。")
    exit()

try:
    # ### 【修改 3】: 初始化 AzureOpenAI Client ###
    client = AzureOpenAI(
        api_key=AZURE_API_KEY,
        azure_endpoint=AZURE_ENDPOINT,
        api_version="2024-02-01"  # 使用一個穩定的 API 版本
    )
    client.models.list() # 嘗試調用一個簡單的API來驗證配置
    print("Azure OpenAI client 初始化並驗證成功。")
    print(f"  - 視覺分析將使用部署: '{VISION_DEPLOYMENT_NAME}'")
    print(f"  - 個性化總結將使用部署: '{SUMMARY_DEPLOYMENT_NAME}'")
except AuthenticationError: print("錯誤：Azure OpenAI API 金鑰或端點無效，認證失敗。"); exit()
except Exception as e: print(f"初始化 Azure OpenAI Client 時發生錯誤: {e}"); exit()

# ---------------------------
# 輔助函數
# ---------------------------
def load_and_preprocess_calibrations(json_path):
    """
    【全新】讀取、索引並排序 calibration_export.json。
    這一步是將扁平的數據列表，轉換成一個高效的、以錯誤為索引的知識庫。
    """
    if not os.path.isfile(json_path):
        print("提示：未找到校準檔案 (calibration_export.json)，將不使用 Few-Shot 修正。")
        return {}

    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        indexed_calibrations = defaultdict(list)
        for record in data:
            original_behavior = record.get("original_behavior")
            if original_behavior:
                indexed_calibrations[original_behavior].append(record)
        
        for behavior in indexed_calibrations:
            indexed_calibrations[behavior].sort(key=lambda x: x.get('error_rating', 5))
            
        print(f"✅ 成功載入並索引 {len(data)} 筆校準紀錄，涵蓋 {len(indexed_calibrations)} 種錯誤類型。")
        return indexed_calibrations

    except Exception as e:
        print(f"❌ 錯誤：處理校準檔案 {json_path} 時失敗: {e}")
        return {}

def select_relevant_examples(indexed_calibrations, previous_batch_results=None, num_examples=1):
    """
    【全新】從索引好的知識庫中，動態選擇最相關的範例。
    """
    if not indexed_calibrations:
        return []

    if previous_batch_results and "analysis" in previous_batch_results:
        highlights = previous_batch_results["analysis"].get("per_image_highlights", [])
        if highlights:
            behavior_counts = Counter(
                behavior
                for hl in highlights
                for behavior in hl.get("behavior_category", []) if isinstance(behavior, str) # 確保是字串
            )
            if behavior_counts:
                most_common_error, _ = behavior_counts.most_common(1)[0]
                if most_common_error in indexed_calibrations:
                    print(f"  [動態Few-Shot]: 偵測到近期潛在錯誤 '{most_common_error}'，選取針對性修正範例。")
                    return indexed_calibrations[most_common_error][:num_examples]

    print("  [動態Few-Shot]: 未找到針對性範例，選取全域最嚴重錯誤範例。")
    all_examples = [ex for sublist in indexed_calibrations.values() for ex in sublist]
    all_examples.sort(key=lambda x: x.get('error_rating', 5))
    return all_examples[:num_examples]

def format_examples_for_prompt(examples_list):
    """
    【全新 & 強化版】將挑選出的範例物件，格式化為可以注入 Prompt 的文字字串。
    """
    if not examples_list:
        return ""

    prompt_parts = ["\n---", "**【第五部分：錯誤案例學習與校準 (Few-Shot Examples)】**", "你必須從以下真人校準的案例中學習，以修正你的判斷模型。\n"]
    
    for i, ex in enumerate(examples_list, 1):
        context = ex.get("context", {})
        reasoning = context.get("original_ai_reasoning", "無紀錄")
        confidence = context.get("original_ai_confidence", 0.0)
        seating = context.get("seating_position", "未知座位")
        
        example_context_desc = f"當時學生坐在 **{seating}**，AI 在該次分析中對此圖的信心度為 **{confidence:.2f}**。"
        
        part = f"""
**[學習案例 #{i}]**
*   **情境描述**: {example_context_desc}
*   **AI 錯誤推理 (應避免)**: "{reasoning}"
*   **AI 錯誤輸出 (應避免)**: "{ex.get('original_behavior')}"
*   **專家糾正的正確輸出**: "{ex.get('corrected_behavior')}"
"""
        prompt_parts.append(part)
    
    return "\n".join(prompt_parts)

def find_state_for_timestamp(target_timestamp, classroom_states):
    """
    【v4.0 穩定版】根據時間戳，查找對應的課堂狀態標籤。
    使用線性查找以確保最高的準確性和可靠性。
    """
    if not classroom_states:
        return "未知"
    
    # 為了除錯，只打印一次查找的詳細信息
    if not hasattr(find_state_for_timestamp, "has_logged_first_search"):
        print("\n" + "-"*20 + " [除錯日誌 - 首次查找課堂狀態] " + "-"*20)
        print(f"  - 正在用第一張照片的時間戳进行匹配...")
        print(f"  - 目標時間戳 (Target Timestamp): {target_timestamp}")
        if classroom_states:
            first_state = classroom_states[0]
            print(f"  - 正在检查第一個時間區間: 從 {first_state.get('start_time_td')} 到 {first_state.get('end_time_td')}")
        print("-" * 69 + "\n")
        find_state_for_timestamp.has_logged_first_search = True

    # 遍歷每一個課堂狀態的時間區間
    for state in classroom_states:
        start_td = state.get("start_time_td")
        end_td = state.get("end_time_td")
        
        # 確保這個區間的時間數據是有效的
        if start_td and end_td:
            # 檢查目標時間戳是否落在 [開始時間, 結束時間] 這個閉区间内
            if start_td <= target_timestamp <= end_td:
                return state["classroom_state"]
    
    # 如果遍歷完所有區間都沒找到，說明時間戳確實超出了範圍
    return "未知"

def parse_time_offset(time_str):
    """將 "HH:MM:SS" 格式的字串轉換為 timedelta 物件"""
    if not time_str or not isinstance(time_str, str):
        return None
    try:
        h, m, s = map(int, time_str.split(':'))
        return datetime.timedelta(hours=h, minutes=m, seconds=s)
    except (ValueError, TypeError):
        print(f"警告：時間偏移量 '{time_str}' 格式不正確，應為 'HH:MM:SS'。將忽略此設定。")
        return None

def get_valid_input(prompt_message):
    while True:
        user_input = input(prompt_message).strip()
        if user_input: return user_input
        print("錯誤：輸入不能為空，請重新輸入。")

def get_valid_folder_path(prompt_message, is_optional=False): # 新增 is_optional 參數
    while True:
        folder_path = input(prompt_message).strip().strip('"')
        if not folder_path and is_optional:
            return None # 如果是可選的且用戶未輸入，返回 None
        if os.path.isdir(folder_path):
            return folder_path
        print(f"錯誤：路徑 '{folder_path}' 不是一個有效的資料夾，請重新輸入。")

def get_timestamp_from_filename(filename):
    """
    【v3.0 兼容版】从档名解析时间戳。
    能同时处理多种常见格式。
    """
    # 模式一：最优先，精确匹配 HH-MM-SS-ms.jpg 格式 (常见于学生照片)
    match = re.search(r'^(\d{2})-(\d{2})-(\d{2})-(\d{3})\.(jpg|jpeg|png|webp)$', filename, re.IGNORECASE)
    if match:
        try:
            h, m, s, ms, ext = match.groups()
            return datetime.timedelta(hours=int(h), minutes=int(m), seconds=int(s), milliseconds=int(ms))
        except (ValueError, IndexError):
            pass

    # 模式二：匹配包含 ...HH-MM-SS-ms... 的通用格式
    match = re.search(r'(\d{2})-(\d{2})-(\d{2})-(\d{3})', filename)
    if match:
        try:
            h, m, s, ms = map(int, match.groups())
            return datetime.timedelta(hours=h, minutes=m, seconds=s, milliseconds=ms)
        except (ValueError, IndexError):
            pass

    # 模式三：匹配包含 ...h...m...s 的通用格式
    match = re.search(r'(\d+)h(\d{2})m(\d{2})s', filename)
    if match:
        try:
            h, m, s = map(int, match.groups())
            return datetime.timedelta(hours=h, minutes=m, seconds=s)
        except (ValueError, IndexError):
            pass

    # 如果所有格式都沒匹配成功，最终返回 None
    return None

def encode_image_to_base64(image_path, max_size_kb=512, target_quality=75):
    try:
        with Image.open(image_path) as img: img.verify()
        with Image.open(image_path) as img:
            if img.mode == 'RGBA' or img.mode == 'P': img = img.convert('RGB')
            current_quality = target_quality
            output_buffer = io.BytesIO()
            temp_img = img.copy()
            temp_img.save(output_buffer, format="JPEG", quality=current_quality)
            current_size_kb = output_buffer.tell() / 1024

            if current_size_kb > max_size_kb:
                scale_factor = math.sqrt(max_size_kb / current_size_kb)
                new_width = int(temp_img.width * scale_factor * 0.9)
                new_height = int(temp_img.height * scale_factor * 0.9)
                if new_width >= 50 and new_height >= 50: # 最小尺寸限制
                    print(f"    圖片 {os.path.basename(image_path)} ({current_size_kb:.1f} KB) 過大，縮放並調整質量...")
                    temp_img = temp_img.resize((new_width, new_height), Image.Resampling.LANCZOS)
                    output_buffer = io.BytesIO()
                    temp_img.save(output_buffer, format="JPEG", quality=max(current_quality - 15, 40)) # 質量可以降更多
                else:
                    print(f"    警告: 圖片 {os.path.basename(image_path)} 縮放後過小，可能影響質量。使用較低質量。")
                    output_buffer = io.BytesIO()
                    img.save(output_buffer, format="JPEG", quality=max(current_quality // 2, 30) )


            output_buffer.seek(0)
            binary_data = output_buffer.getvalue()
            base64_encoded_data = base64.b64encode(binary_data)
            return f"data:image/jpeg;base64,{base64_encoded_data.decode('utf-8')}"
    except Exception as e: print(f"錯誤：處理圖片 '{image_path}': {e}"); return None

def load_classroom_states(json_path):
    """【全新】讀取預處理好的課堂狀態時間軸 JSON 檔案。"""
    if not json_path or not os.path.isfile(json_path):
        print("提示：未提供或找不到課堂狀態 JSON 檔案。將不使用課堂情境。")
        return None  # 返回 None 以便更明確地判斷失敗
    
    print(f"正在讀取課堂狀態時間軸: {json_path}...")
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f) # <--- 關鍵修正：將 f 作為參數傳入
            
            # 將時間字串預先轉換為 timedelta 物件以便快速比較
            timeline = data.get("timeline", [])
            for state in timeline:
                state["start_time_td"] = parse_time_offset(state["start_time"])
                state["end_time_td"] = parse_time_offset(state["end_time"])
            print(f"成功載入 {len(timeline)} 個課堂狀態階段。")
            return timeline
    except Exception as e:
        print(f"❌ 錯誤：讀取或解析課堂狀態 JSON 時發生問題: {e}")
        return None # 返回 None

def get_behavior_sequence_analysis_system_prompt(student_position, has_teacher_context , few_shot_prompt_str=""):
    """
    【v14.4 - 終極重構版】
    此版本在 v14.3 的基礎上進行了結構性重構，將所有規則模塊化，
    消除了邏輯冗餘，並將決策樹統一整合，達到了最高的清晰度與可維護性。
    """
    # --- 1. 行為編碼表 (靜態模塊) ---
    behavior_table_for_prompt = "\n\n**【學習行為編碼表】**\n你 **必須** 且 **只能** 從以下列表的「編碼 (Code)」中選擇行為進行標註。...\n"
    for code, label in BEHAVIOR_CODES.items():
        definition = ""
        for category in STANDARD_BEHAVIOR_CATEGORIES.values():
            for item in category:
                if item['label'] == label:
                    definition = item['definition']
                    break
            if definition:
                break
        behavior_table_for_prompt += f"*   **`{code}`**: {label} - {definition}\n"
    behavior_table_for_prompt += "\n*如果行為因任何原因無法清晰判斷，請 **必須** 使用 **`O_UNK`** 編碼。*\n"

    # --- 2. 空間與攝影機規則 (動態模塊) ---
    analysis_view_rules = ""
    if "左" in student_position:
        analysis_view_rules = "*   **分析視角**: 拍攝該學生的【學生個人照片序列】來自教室前方的【左側】攝影機。\n*   **視線基準**: 對於這位學生，【正面朝向鏡頭】僅代表他在看教室的【左前方】。為了看向【教室正前方中心】，他的頭部必須輕微地朝向他自己的【右側】轉動。"
    elif "右" in student_position:
        analysis_view_rules = "*   **分析視角**: 拍攝該學生的【學生個人照片序列】來自教室前方的【右側】攝影機。\n*   **視線基準**: 對於這位學生，【正面朝向鏡頭】僅代表他在看教室的【右前方】。為了看向【教室正前方中心】，他的頭部必須輕微地朝向他自己的【左側】轉動。"
    else: # 預設為中間
        analysis_view_rules = "*   **分析視角**: 拍攝該學生的【學生個人照片序列】來自教室前方的【中間】攝影機。\n*   **視線基準**: 對於這位學生，當他臉部【正面朝向鏡頭】時，即代表其視線正朝向教室的【正前方中心】。"

    camera_and_spatial_rules = f"""
---
**【第一部分：多機位系統與空間推理框架 (最高優先級)】**
你的所有空間判斷都必須基於一個【兩步推理】的框架：首先建立宏觀地圖，然後應用微觀視角。

*   **準則 1A: 宏觀情境視角 (固定中央廣角鏡頭)**
    *   **定義**: 用於情境感知的【班級整體照片】**永遠**來自教室前方的**中央廣角鏡頭**。
    *   **任務**: 你必須使用這張照片來建立一個視覺座標系，將教室劃分為**左側、中央、右側**三個區域，並在其中定位目標學生。

*   **準則 1B: 微觀分析視角 (動態學生鏡頭)**
    *   {analysis_view_rules}

*   **通用空間準則**:
    *   **朝前弧度原則**: 對教室前方的專注行為（`V_TCH`, `V_BRD`）必須發生在一個合理的「朝前弧度」內。一個使學生頭部與其肩膀構成【接近90度】的**大幅度轉頭**，是其注意力【脫離前方】的明確證據，你**必須**將其優先判定為 `V_CLS`。
    *   **影像鏡像轉換**: 所有攝影機畫面都是鏡像的：照片中的**左側** => 教室中的**右側**；照片中的**右側** => 教室中的**左側**。

**【核心執行協議】**:
你的視線分析**必須**結合宏觀與微觀資訊。例如：在班級照片中，你確認學生位於**畫面左側區域** (準則 1A)。同時，`student_position` 文字告訴你拍攝他的鏡頭在**左側** (準則 1B)。綜合這兩點，你才能最準確地判斷他為了看向老師而需要轉動的角度。
"""

    # --- 3. 行為決策協議 (動態模塊) ---
    # 將前後排規則與統一決策協議整合成一個大的、連貫的決策流程
    is_front_row = any(keyword in student_position for keyword in ["第一", "第二"])
    is_rear_row = any(keyword in student_position for keyword in ["第三", "第四"])

    # 3.1 定義前後排獨有的規則
    front_row_rules = """
*   **低頭行為的分診決策樹 (軸線檢查版)**:
    1.  **軸線檢查**: 判斷頭部是【正下方】（判定 `V_BOK`）還是【側下方】（判定 `V_CLS`）。
    2.  **手部動作覆寫**: 檢查是否有隱蔽的非學習動作 (`H_PLAY_HW`) 來覆寫視線判斷。
    3.  **最終備用**: 都不是才使用 `P_DWN`。
*   **前方視線的精確匹配**: 嚴格結合【老師位置情境】進行幾何匹配，判定 `V_TCH` 或 `V_BRD`。
"""
    rear_row_rules = """
*   **低可視度下的姿態優先原則**: 當細節無法辨認時，必須忽略猜測，嚴格依賴姿態鐵則。
*   **低頭行為的「絕對學習推定」原則**: 只要頭部低垂，就**必須、無條件地**判定為 `V_BOK`。
*   **前方視線的「教師優先」原則**: 只要視線朝前，就**優先**標註為 `V_TCH`。
"""

    # 3.2 根據學生位置選擇對應規則
    if is_rear_row:
        position_specific_rules = f"""
**【第二部分：後排學生分析協議 (絕對推斷模式)】**
你必須遵循以下不可動搖的推斷鐵則：
{rear_row_rules}
"""
    else: # 預設為前排
        position_specific_rules = f"""
**【第二部分：前排學生分析協議 (高證據模式)】**
你的所有判斷都必須基於【嚴格的視覺證據】：
{front_row_rules}
"""

    # 3.3 定義統一的、後續的行為決策流程
    universal_decision_flow = """
---
**【第三部分：統一行為決策流程 (全體適用)】**
**【核心原則】**: 你的判斷【必須】基於每一張獨立圖片捕捉到的**瞬時物理證據**。在通過【第二部分】獲得初步的視線判斷後，你必須嚴格遵循以下【單一、統一】的分層決策樹來確定最終的主要行為標籤。

*   **第一層：檢查【壓倒性的身體姿態】(最高優先級)**
    *   **情況 A (向前協作姿態):** 身體【向前或向側前方】探出，進入鄰座同學的桌面空間 -> 主要行為判定為 **`V_BOK`**。**決策結束。**
    *   **情況 B (向後/側面社交姿態):** 身體向【側面或後方】旋轉，與同學進行面對面交流 -> 主要行為判定為 **`V_CLS`**。**決策結束。**

*   **第二層：檢查【定義明確的主動動作】(僅在身體朝前時評估)**
    *   是否在進行**動態的書寫 (`H_NOT`)**？ -> 是 -> `H_NOT` 是主要標籤。**決策結束。**
    *   是否在**玩弄手部/文具 (`H_PLAY_HW`)**？ -> 是 -> `H_PLAY_HW` 是主要標籤。**決策結束。**
    *   手臂是否抬起？ -> 是 -> 執行**【舉手行為過濾器】**（排除 `V_CLS`, `H_TOUCH_H/F` 後，判斷 `I_HND_A/P`）。**決策結束。**

*   **第三層：檢查【其他細微姿態與動作】**
    *   學生是否將頭**枕於**手臂或桌面？ -> 是 -> `P_SLP` (趴睡)。**決策結束。**
    *   學生是否在**托腮 (`P_THK`)**？ -> 是 -> `P_THK` 作為主要標籤。
    *   學生是否在進行**其他非任務相關動作**（如觸摸臉部 `H_TOUCH_F`）？ -> 是 -> 標註對應動作。

*   **第四層：標註【靜態視覺行為】(最終備用選項)**
    *   僅在**所有**上述檢查項都不適用的情況下，才使用在【第二部分】中獲得的**初步視線判斷**（如 `V_TCH`, `V_BOK`）作為最終的主要行為標籤。

*   **第五層：疊加【通用輔助姿態】**
    *   在確定了主要行為後，可疊加一個輔助的姿態標籤（如 `P_LEAN`）。
"""

    # --- 4. 生成最終的、完整的 Prompt ---
    teacher_context_header = "5.  **【老師位置文字情境】**: 用於交叉驗證你視線判斷的輔助數據。" if has_teacher_context else ""

    return f"""
「你是一位世界頂尖的教育分析師，精通電腦視覺、空間幾何推理與心理學。你的任務是綜合所有給定的物理與情境資訊，對學生的學習行為進行最精準、最客觀的標註。」

**【你收到的資訊來源】**
1.  **【班級整體照片】**: 你的【空間座標系】基準。
2.  **【學生個人照片序列】**: 你的主要分析對象。
3.  **【課堂狀態】**: 判斷行為動機的核心上下文。
4.  **【學生座位文字描述】**: `{student_position}`，用於輔助定位。
{teacher_context_header}

{camera_and_spatial_rules}

{position_specific_rules}

{universal_decision_flow}

{behavior_table_for_prompt}

{few_shot_prompt_str} 

---
**【第四部分：輸出格式要求 - 嚴謹推理與簡明輸出】**
*   你的回答**必須**是一個結構完整的、單一的 JSON 物件。
*   `behavior_category` 的值**必須**是一個包含 1 到 2 個編碼字串的**陣列 (Array)**。
*   `per_image_highlights` 的每個物件中**必須包含** `image_index_in_sequence`, `context_description`, `behavior_category`, 和 `confidence` 這四個鍵。
*   **【關鍵指令】**: `context_description` 內容**必須極度簡潔**，例如："後排協議 -> 低頭推定" 或 "左側區域規則 -> 視線匹配"。

**【輸出 JSON 格式範例】**
```json
{{
  "sequence_analysis_confidence": 0.97,
  "per_image_highlights": [
    {{
      "image_index_in_sequence": 0,
      "context_description": "後排協議：低頭推定。",
      "behavior_category": ["V_BOK", "P_STR"],
      "confidence": 0.95
    }},
    {{
      "image_index_in_sequence": 1,
      "context_description": "左側區域規則：頭部右轉，匹配老師位置。",
      "behavior_category": ["V_TCH"],
      "confidence": 0.99
    }},
    {{
      "image_index_in_sequence": 2,
      "context_description": "社交優先：向前協作，判定為 V_BOK。",
      "behavior_category": ["V_BOK", "P_LEAN"],
      "confidence": 0.98
    }}
  ]
}}
```"""

def default_error_result_structure():
    return { "error": "分析失敗或無有效數據", "sequence_analysis_confidence": 0.0, "sequence_summary": "未能生成序列總結。", "dominant_sustained_behaviors": [], "significant_behavior_shifts": [], "per_image_highlights": [], "general_sequence_atmosphere_hint": "未知" }

def clean_analysis_json(raw_analysis):
    """
    清理從 API 返回的分析 JSON，只保留我們需要的欄位。
    這是一個防禦性措施，用來處理微調模型的「幻覺」問題。
    """
    if not isinstance(raw_analysis, dict):
        return default_error_result_structure()

    # 定義合法的鍵
    allowed_top_level_keys = {"sequence_analysis_confidence", "per_image_highlights"}
    allowed_highlight_keys = {"image_index_in_sequence", "context_description", "behavior_category", "confidence"}

    cleaned_analysis = {}
    
    # 1. 清理最外層的鍵
    for key, value in raw_analysis.items():
        if key in allowed_top_level_keys:
            cleaned_analysis[key] = value

    # 2. 檢查並清理 per_image_highlights 列表
    if "per_image_highlights" in cleaned_analysis and isinstance(cleaned_analysis["per_image_highlights"], list):
        cleaned_highlights = []
        for raw_highlight in cleaned_analysis["per_image_highlights"]:
            if not isinstance(raw_highlight, dict):
                continue # 如果列表中的元素不是字典，就跳過它
            
            cleaned_highlight = {}
            for key, value in raw_highlight.items():
                if key in allowed_highlight_keys:
                    cleaned_highlight[key] = value
            
            # 確保必要的鍵存在，即使為空
            for required_key in allowed_highlight_keys:
                if required_key not in cleaned_highlight:
                    cleaned_highlight[required_key] = None

            cleaned_highlights.append(cleaned_highlight)
        
        cleaned_analysis["per_image_highlights"] = cleaned_highlights

    # 3. 如果最外層缺少必要的鍵，補上預設值
    if "sequence_analysis_confidence" not in cleaned_analysis:
        cleaned_analysis["sequence_analysis_confidence"] = 0.0
    if "per_image_highlights" not in cleaned_analysis:
        cleaned_analysis["per_image_highlights"] = []

    return cleaned_analysis

def analyze_student_behavior_from_images_sequence(student_image_paths, teacher_position_text, classroom_view_image_path, classroom_state, image_filenames_batch, openai_client, student_id, student_position , few_shot_prompt_str=""):
    """
    【v8.1 穩定版】分析單一圖片批次，使用預處理好的「課堂狀態」標籤。
    此版本包含了完整的錯誤處理、JSON清理和穩健性增強。
    """
    if not student_image_paths:
        return default_error_result_structure() # 直接返回錯誤結構，而不是字典
    
    user_message_content = []
    
    # 1. 編碼學生個人圖片序列 (使用配置中的解析度)
    encoded_student_images = []
    for img_path in student_image_paths:
        b64_img = encode_image_to_base64(img_path)
        if b64_img:
            encoded_student_images.append({
                "type": "image_url", 
                "image_url": {"url": b64_img, "detail": IMAGE_DETAIL_LEVEL}
            })

    if not encoded_student_images:
        return default_error_result_structure() # 直接返回錯誤結構
        
    # 2. 編碼班級整體照片 (使用配置中的解析度)
    encoded_classroom_view_image = None
    if classroom_view_image_path:
        b64_img = encode_image_to_base64(classroom_view_image_path)
        if b64_img:
            encoded_classroom_view_image = {
                "type": "image_url", 
                "image_url": {"url": b64_img, "detail": IMAGE_DETAIL_LEVEL}
            }

    # 3. 組合完整的請求體，包含所有情境信息
    user_message_content.append({"type": "text", "text": f"請根據系統提示中的偵探任務，分析學生「{student_id}」的行為。學生照片序列的文件名（供您參考）為: {', '.join(image_filenames_batch)}。"})
    
    # 加入課堂狀態標籤
    state_context_prompt = f"【課堂狀態】: {classroom_state}"
    user_message_content.append({"type": "text", "text": state_context_prompt})

    # 加入老師位置情境
    teacher_context_prompt = f"【老師位置情境】根據預先分析，在此時間段，老師的位置在教室前方的「{teacher_position_text}」。請以此作為判斷『目視教師』的核心依據。"
    user_message_content.append({"type": "text", "text": teacher_context_prompt})
    
    # 依序加入班級照片和學生照片
    if encoded_classroom_view_image:
        user_message_content.append({"type": "text", "text": "【班級整體照片】(用於定位學生和觀察整體氛圍)"})
        user_message_content.append(encoded_classroom_view_image)
    user_message_content.append({"type": "text", "text": "【學生個人照片序列】(主要分析對象)"})
    user_message_content.extend(encoded_student_images)

    # 4. 獲取新的、具備情境感知能力的系統提示
    system_prompt_content = get_behavior_sequence_analysis_system_prompt(student_position, bool(teacher_position_text and teacher_position_text != "未知"),few_shot_prompt_str )

    # 5. API 呼叫與錯誤處理
    retry_attempts = 2
    for attempt in range(retry_attempts + 1):
        raw_content = "" # 初始化 raw_content 以免在 except 區塊中引用未定義變數
        try:
            has_teacher_context = bool(teacher_position_text and teacher_position_text != "未知")
            has_classroom_image = bool(encoded_classroom_view_image)
            has_state = classroom_state != "未知"

            print(f"  正在向 {VISION_DEPLOYMENT_NAME} 發送請求 (學生: {len(encoded_student_images)}, 課堂狀態: {'有' if has_state else '無'}, 老師位置: {'有' if has_teacher_context else '無'}, 班級照片: {'有' if has_classroom_image else '無'})...")
            
            response = openai_client.chat.completions.create(
                model=VISION_DEPLOYMENT_NAME, 
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": system_prompt_content},
                    {"role": "user", "content": user_message_content}
                ],
                max_tokens=MAX_TOKENS_VISION_COMPLETION,  # gpt-4.1
                temperature=0.05 # gpt-4.1
                # max_completion_tokens=MAX_TOKENS_VISION_COMPLETION, #gpt-5
                
            )
            raw_content = response.choices[0].message.content
            
            # --- ✅【核心修正點】---
            # 步驟 1: 解析原始回應
            analysis_result_raw = json.loads(raw_content)
            
            # 步驟 2: 清理可能包含幻覺的 JSON，確保格式正確
            analysis_result = clean_analysis_json(analysis_result_raw)
            # --- ✅【修正結束】---

            # 本地解碼，將行為編碼轉換回中文標籤列表
            if "per_image_highlights" in analysis_result and isinstance(analysis_result["per_image_highlights"], list):
                for hl in analysis_result["per_image_highlights"]:
                    # 增加穩健性檢查，防止因清理後的 None 值導致錯誤
                    if not hl or not isinstance(hl, dict) or "behavior_category" not in hl:
                        continue
                    
                    codes = hl.get("behavior_category")
                    if not codes: # 處理 behavior_category 為 None 的情況
                        continue

                    if not isinstance(codes, list):
                        codes = [codes]
                    
                    # 過濾掉可能是 None 的 code
                    decoded_behaviors = [CODE_TO_BEHAVIOR.get(code, f"未知編碼({code})") for code in codes if code]
                    hl["behavior_category"] = decoded_behaviors
            
            # 補上空的摘要欄位以保持格式統一 (這一步可以保留，也可以移除，因為 clean_analysis_json 會確保結構)
            if "sequence_summary" not in analysis_result:
                analysis_result["sequence_summary"] = "已設定為精簡模式，此欄位由本地生成。"

            return analysis_result

        except json.JSONDecodeError:
            # 這裡的邏輯是正確的：如果 JSON 本身語法錯誤，就記錄並重試
            print(f"    錯誤：無法解析JSON (第 {attempt+1} 次嘗試)。回應: {raw_content[:500]}...");
            time.sleep(5) # 增加短暫延遲
        except RateLimitError:
            print(f"  警告：API速率限制，等待 {API_RETRY_DELAY_SECONDS}s 後重試...");
            time.sleep(API_RETRY_DELAY_SECONDS)
        except APIError as e:
            wait_time = 10 + attempt * 5 
            print(f"  錯誤：API錯誤 (第 {attempt+1} 次嘗試): {e}. 將在 {wait_time} 秒後重試...");
            time.sleep(wait_time)
        except Exception as e:
            # 處理 NameError 和其他所有未知錯誤
            wait_time = 5 + attempt * 5
            print(f"  錯誤：未知錯誤 (第 {attempt+1} 次嘗試): {e}. 將在 {wait_time} 秒後重試...");
            time.sleep(wait_time)
    
    print(f"  錯誤：圖片序列分析在多次重試後失敗。")
    return default_error_result_structure()

def generate_personalized_summary_notes(student_id, overall_stats, non_task_highlights, openai_client):
    if not openai_client: return {"error": "OpenAI client not available"}
    # ... (此函數內部的 Prompt 內容完全不需要修改) ...
    stats_summary_for_ai = "\n".join([f"- {s['behavior_category']}: {s['percentage']:.1f}% ({s['count']}次)" for s in overall_stats[:7]]) # 確保百分比格式
    non_task_prompt_part = "該生在本堂課中，未觀察到明顯或頻繁的非任務相關行為。"
    if non_task_highlights:
        highlights_str = "\n".join([f"  - 圖 '{img_fn}' (~{ts}): '{beh}' (描述: {desc})" for img_fn, ts, beh, desc in non_task_highlights[:3]])
        if highlights_str: non_task_prompt_part = f"在本堂課中，觀察到一些非任務相關行為，例如：\n{highlights_str}\n這可能影響了學習專注度。"
    prompt_for_summary = f"""
    「你是一位專業且富有同理心的學習行為教練。你的目標不是批評，而是透過客觀數據，引導學生發現自己的學習模式，並提供能立即實踐的策略，以激發他們『自我反思』的動力。」

    **任務：** 為學生「{student_id}」撰寫一份「AI學習夥伴的觀察與建議」。

    **學生的課堂行為數據：**
    *   **主要行為分佈:**
    {stats_summary_for_ai}
    *   **值得注意的行為片段:**
    {non_task_prompt_part}

    **撰寫指引與 JSON 格式要求：**
    請嚴格遵循以下指引，產生一個結構完整的 JSON 物件。

    *   **`greeting` (問候語):**
        *   用親切、個人化的方式稱呼學生，例如：「嗨，{student_id} 同學，一起來看看這次課堂的學習足跡吧！」

    *   **`positive_feedback` (亮點觀察):**
        *   **必須**從數據中找出最值得肯定的行為（例如「目視教師」或「筆記」佔比最高），並給予具體、真誠的讚美。
        *   **範例**：「我發現你在這堂課有超過一半的時間都在『目視教師』，這代表你非常努力地跟上老師的節奏，非常棒！」

    *   **`observation_points_summary` (行為模式提醒):**
        *   客觀、中性地指出一個或兩個最主要的、可能影響學習的行為模式。避免使用負面詞彙。
        *   **範例**：「數據也顯示，大約有 15% 的時間出現了『玩弄物品』或『目視他處』的狀況，這些時刻可能讓我們不小心錯過了一些重點喔。」

    *   **`reflection_points` (反思引導提問):**
        *   **【此項最為關鍵】** 根據前面的觀察點，設計 2-3 個**開放式問題**，引導學生思考行為背後的原因，而不是直接給答案。
        *   **問題範例 1**：「我們可以一起回想看看，當出現『玩弄物品』的時候，通常是在課程的哪個階段呢？是覺得內容太簡單、太難，還是剛好有點疲倦了呢？」
        *   **問題範例 2**：「當視線看向其他地方時，是想到了什麼有趣的事，還是被教室裡的其他動靜吸引了呢？了解這些原因，能幫助我們找到最適合自己的專注方法。」

    *   **`suggestions` (可實踐的小建議):**
        *   提供 1-2 個**具體、微小、且容易執行**的行動建議。不要說「要專心」，而是給出方法。
        *   **建議範例 1**：「下次當你發現自己開始無意識地轉筆時，可以試著把它輕輕放下，然後做一個深呼吸，再重新將目光移回老師或課本上。」
        *   **建議範例 2**：「如果感覺到疲倦或分心，可以試試看『筆記專注法』：在筆記本上寫下老師說的任何一個關鍵字，這個小動作能幫助我們的大腦重新連線！」

    *   **`encouragement` (鼓勵與結語):**
        *   用一句溫暖、有力的話作結，強調這份報告是幫助他成長的工具。
        *   **範例**：「每一次的觀察都是為了讓我們更了解自己。相信你透過這些小小的調整，一定能發揮出自己最大的潛力，加油！」
    """
    retry_attempts = 2
    for attempt in range(retry_attempts):
        try:
            print(f"  正在為學生 {student_id} 使用 {SUMMARY_DEPLOYMENT_NAME} 生成個性化總結...") # 修改了日誌輸出
            response = client.chat.completions.create(
                # ==========================================================
                # ↓↓↓ 【關鍵修改】使用我們新設定的經濟型模型 ↓↓↓
                # ==========================================================
                model=SUMMARY_DEPLOYMENT_NAME, 
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": "你是一位富有同理心和洞察力的教育顧問。"},
                          {"role": "user", "content": prompt_for_summary}],
                max_tokens=MAX_TOKENS_SUMMARY_COMPLETION, #gpt-4.1
                temperature=0.7 #gpt-4.1
                # max_completion_tokens=MAX_TOKENS_SUMMARY_COMPLETION, # gpt-5        
                
            )
            summary_data = json.loads(response.choices[0].message.content)
            expected_keys = ["greeting", "positive_feedback", "observation_points_summary", "reflection_points", "suggestions", "encouragement"]
            if all(key in summary_data for key in expected_keys): 
                print(f"  成功為學生 {student_id} 生成個性化總結。")
                return summary_data
            else: 
                print(f"    警告：個性化總結JSON缺少鍵。返回: {summary_data}")
                return {key: summary_data.get(key, f"AI未能生成 ({key})") for key in expected_keys}
        except Exception as e: 
            print(f"  生成個性化總結錯誤 ({attempt + 1}): {e}")
            time.sleep(API_RETRY_DELAY_SECONDS if isinstance(e, RateLimitError) else 5)
        if attempt == retry_attempts - 1: 
            print(f"  錯誤：無法為學生 {student_id} 生成個性化總結。")
            return {"greeting": f"親愛的 {student_id},", "positive_feedback": "總結生成遇到問題。", "observation_points_summary": "請參考統計數據。", "reflection_points": "未能生成。", "suggestions": "請自行評估。", "encouragement": "加油！"}
    return {}

def find_closest_image_path(representative_timestamp, sorted_photo_list, max_time_diff_seconds=5):
    """
    一個可重用的輔助函數，用二分查找法在排序好的照片列表中找到時間最接近的照片路徑。
    """
    if not sorted_photo_list:
        return None

    all_timestamps = [item[0] for item in sorted_photo_list]
    # bisect_left 找到應該插入的位置
    insertion_point = bisect.bisect_left(all_timestamps, representative_timestamp)

    closest_path = None
    min_diff = datetime.timedelta.max

    # 只檢查插入點及其前後的幾個候選照片，效率極高
    # 檢查範圍設為 insertion_point-2 到 insertion_point+2 以增加容錯
    start_index = max(0, insertion_point - 2)
    end_index = min(len(sorted_photo_list), insertion_point + 2)

    for i in range(start_index, end_index):
        candidate_ts, candidate_path = sorted_photo_list[i]
        diff = abs(candidate_ts - representative_timestamp)
        if diff < min_diff:
            min_diff = diff
            closest_path = candidate_path

    # 只有在時間差在容許範圍內才返回路徑
    if closest_path and min_diff.total_seconds() <= max_time_diff_seconds:
        return closest_path
    
    return None

def find_closest_position(representative_timestamp, sorted_position_list, max_time_diff_seconds=10):
    """
    用二分查找法在排序好的位置列表中找到時間最接近的老師位置。
    """
    if not sorted_position_list:
        return "未知"

    all_timestamps = [item[0] for item in sorted_position_list]
    insertion_point = bisect.bisect_left(all_timestamps, representative_timestamp)

    closest_position = "未知"
    min_diff = datetime.timedelta.max

    start_index = max(0, insertion_point - 2)
    end_index = min(len(sorted_position_list), insertion_point + 2)

    for i in range(start_index, end_index):
        candidate_ts, candidate_pos = sorted_position_list[i]
        diff = abs(candidate_ts - representative_timestamp)
        if diff < min_diff:
            min_diff = diff
            closest_position = candidate_pos
    
    if min_diff.total_seconds() <= max_time_diff_seconds:
        return closest_position
    
    return "未知"

def process_single_batch(batch_idx, image_batch_info, teacher_positions_data, sorted_classroom_photos, classroom_states, client, student_id, student_position, indexed_calibrations, previous_batch_results=None):
    """
    【升級版】處理單一圖片批次，並動態注入 Few-Shot 範例。
    """
    batch_student_paths = [info["path"] for info in image_batch_info]
    batch_image_filenames = [info["filename"] for info in image_batch_info]
    
    representative_timestamp = image_batch_info[0]["timestamp_obj"]

    # --- 【新增】動態選擇並格式化 Few-Shot 範例 ---
    selected_examples = select_relevant_examples(indexed_calibrations, previous_batch_results)
    few_shot_prompt_str = format_examples_for_prompt(selected_examples)

    # --- 為當前批次查找所有情境數據 (不變) ---
    teacher_position_text = find_closest_position(representative_timestamp, teacher_positions_data)
    classroom_view_path = find_closest_image_path(representative_timestamp, sorted_classroom_photos)
    classroom_state = find_state_for_timestamp(representative_timestamp, classroom_states)

    # --- 呼叫【新版】分析函數 ---
    sequence_analysis_data = analyze_student_behavior_from_images_sequence(
        student_image_paths=batch_student_paths,
        teacher_position_text=teacher_position_text,
        classroom_view_image_path=classroom_view_path,
        classroom_state=classroom_state,
        image_filenames_batch=batch_image_filenames,
        openai_client=client,
        student_id=student_id,
        student_position=student_position,
        few_shot_prompt_str=few_shot_prompt_str # <--- 傳入新參數
    )

    # 返回結果的結構保持不變
    return {
        "batch_index": batch_idx,
        "image_batch_info": image_batch_info,
        "matched_teacher_position_text": teacher_position_text,
        "matched_classroom_view_image": os.path.basename(classroom_view_path) if classroom_view_path else None,
        "matched_classroom_state": classroom_state,
        "analysis": sequence_analysis_data
    }

def main():
    print("--- 學生課堂學習行為分析報告生成 (JSON) v5.0 - 成本優化版 ---")
    print(f"視覺模型: {VISION_DEPLOYMENT_NAME}, 文本模型: {VISION_DEPLOYMENT_NAME}, 總結模型: {VISION_DEPLOYMENT_NAME}")
    print(f"圖片批次大小: {IMAGES_PER_API_CALL}, 圖片取樣率: 1/{SAMPLING_RATE}, 圖片解析度: {IMAGE_DETAIL_LEVEL}")

    calibration_db_path = r'C:\Users\User\Desktop\test\training_json\calibration_export.json'
    indexed_calibrations = load_and_preprocess_calibrations(calibration_db_path)
    print("-" * 40)

    # --- 使用者輸入部分 ---
    student_id = get_valid_input("請輸入學生姓名 (可中文): ")
    student_number = get_valid_input("請輸入學生座號 (例如: 1): ") 
    student_images_folder = get_valid_folder_path(f"請輸入 '{student_id}' 的個人影像資料夾: ")
    print("\n--- 請輸入課堂情境資訊 ---")
    student_position = get_valid_input("請輸入學生座位 (例如: '第3排中間', '第1排左側'): ")
    print("-" * 50)

    # --- 準備輸出資料夾 ---
    safe_student_id_for_folder = re.sub(r'[\\/*?:"<>|]',"", student_id.replace(" ", "_"))
    student_specific_folder_base = os.path.join(JSON_OUTPUT_FOLDER, safe_student_id_for_folder)
    os.makedirs(student_specific_folder_base, exist_ok=True)

    # --- 準備學生照片資料 ---
    image_files_with_timestamps = []
    valid_extensions = ('.png', '.jpg', '.jpeg', '.webp')
    print(f"正在掃描學生個人照片資料夾: {student_images_folder}...")
    for filename in os.listdir(student_images_folder):
        if filename.lower().endswith(valid_extensions):
            timestamp = get_timestamp_from_filename(filename)
            if timestamp:
                image_files_with_timestamps.append({
                    "path": os.path.join(student_images_folder, filename), "filename": filename,
                    "timestamp_obj": timestamp, # <-- 直接使用原始時間戳，不做任何校準
                    "timestamp_str": str(timestamp).split('.')[0]
                })
    image_files_with_timestamps.sort(key=lambda x: x["timestamp_obj"])
    
    # ==========================================================
    # ↓↓↓ 【新增】圖片取樣以降低成本 ↓↓↓
    # ==========================================================
    if SAMPLING_RATE > 1:
        original_count = len(image_files_with_timestamps)
        image_files_with_timestamps = image_files_with_timestamps[::SAMPLING_RATE]
        print(f"已執行圖片取樣：從 {original_count} 張原始照片中，每 {SAMPLING_RATE} 張取 1 張，共 {len(image_files_with_timestamps)} 張照片將被分析。")
    else:
        print(f"成功找到 {len(image_files_with_timestamps)} 張學生個人照片 (未取樣)。")

    if not image_files_with_timestamps: 
        print("錯誤：學生資料夾中未找到有效時間格式的圖片，或取樣後為空。")
        return
    # ==========================================================

    # ... 後續的程式碼，從 `load_teacher_positions` 開始，到整個 `main` 函數結束，
    # 都不需要再做任何修改。您可以直接使用您原有的版本。
    # 我將剩餘部分貼在下方以保證完整性。
    
    def load_teacher_positions(json_path, start_time_offset_str=""): # 新增 start_time_offset_str 參數
        if not json_path or not os.path.isfile(json_path):
            print(f"提示：未提供或找不到老師位置 JSON 檔案 ({json_path})。將不使用老師位置情境。")
            return []
        start_offset = parse_time_offset(start_time_offset_str)
        print(f"正在讀取老師位置數據: {json_path}...")
        try:
            with open(json_path, 'r', encoding='utf-8') as f: data = json.load(f)
            position_map, original_count = [], len(data)
            for item in data:
                try:
                    h, m, s = map(int, item['timestamp'].split(':'))
                    td = datetime.timedelta(hours=h, minutes=m, seconds=s)
                    if start_offset and td < start_offset: continue
                    position_map.append((td, item['position']))
                except (ValueError, KeyError): continue
            position_map.sort(key=lambda x: x[0])
            if start_offset: print(f"成功加載老師位置數據。根據起始時間 '{start_time_offset_str}' 進行過濾，保留 {len(position_map)} 筆 (原 {original_count} 筆)。")
            else: print(f"成功加載並索引了 {len(position_map)} 筆老師位置數據。")
            return position_map
        except Exception as e:
            print(f"錯誤：讀取或解析老師位置 JSON 時發生問題: {e}"); return []

    def load_and_sort_photos(folder_path, photo_type_name, start_time_offset_str=""): # 新增 start_time_offset_str 參數
        if not folder_path or not os.path.isdir(folder_path):
            print(f"提示：未提供或找不到 {photo_type_name} 照片資料夾 ({folder_path})。"); return []
        start_offset = parse_time_offset(start_time_offset_str)
        print(f"正在掃描 {photo_type_name} 照片資料夾: {folder_path}...")
        photo_list, original_count = [], 0
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                original_count += 1
                timestamp = get_timestamp_from_filename(filename)
                if timestamp:
                    if start_offset and timestamp < start_offset: continue
                    photo_list.append((timestamp, os.path.join(folder_path, filename)))
        if photo_list:
            sorted_photos = sorted(photo_list)
            if start_offset: print(f"成功加載 {photo_type_name} 照片。根據起始時間 '{start_time_offset_str}' 進行過濾，保留 {len(sorted_photos)} 張 (原 {original_count} 張有效格式照片)。")
            else: print(f"成功加載並索引了 {len(sorted_photos)} 張 {photo_type_name} 照片。")
            return sorted_photos
        else:
            print(f"警告：在 {photo_type_name} 照片資料夾 '{folder_path}' 中未找到符合條件的圖片。"); return []

    teacher_positions_data = load_teacher_positions(TEACHER_POSITION_JSON, TEACHER_JSON_START_TIME_OFFSET)
    sorted_classroom_photos = load_and_sort_photos(CLASSROOM_IMAGES_FOLDER, "班級整體", CLASSROOM_START_TIME_OFFSET)
    classroom_states = load_classroom_states(CLASSROOM_STATE_JSON_PATH)

    image_batches = [image_files_with_timestamps[i:i + IMAGES_PER_API_CALL] for i in range(0, len(image_files_with_timestamps), IMAGES_PER_API_CALL)]
    print(f"學生圖片將被分為 {len(image_batches)} 個批次進行分析。")

    print(f"將以序列化方式處理 {len(image_batches)} 個批次，以啟用動態Few-Shot學習...")
    all_results_from_threads = []
    previous_batch_result = None # 初始化，用於儲存上一個批次的結果

    # 使用 tqdm 顯示進度條
    for idx, batch_info in enumerate(tqdm(image_batches, desc=f"分析學生 {student_id} 的圖片批次")):
        try:
            result = process_single_batch(
                batch_idx=idx,
                image_batch_info=batch_info,
                teacher_positions_data=teacher_positions_data,
                sorted_classroom_photos=sorted_classroom_photos,
                classroom_states=classroom_states,
                client=client,
                student_id=student_id,
                student_position=student_position,
                indexed_calibrations=indexed_calibrations,      # <--- 傳入知識庫
                previous_batch_results=previous_batch_result  # <--- 傳入上個批次的結果
            )
            all_results_from_threads.append(result)
            previous_batch_result = result # 更新，為下一次迭代做準備
        except Exception as exc:
            print(f'\n批次 {idx + 1} 執行時產生錯誤: {exc}')
            all_results_from_threads.append({"batch_index": idx, "error": str(exc)})
            previous_batch_result = None # 如果出錯，重置

    all_results_from_threads.sort(key=lambda x: x['batch_index'])
    print("\n所有批次分析完成，開始匯總數據...")
    all_sequence_analysis_results = []
    overall_behavior_summary = { "total_images_processed_in_batches": 0, "behavior_counts": Counter(), "behavior_confidence_sum": defaultdict(float), "non_task_behavior_examples_for_summary": [] }
    for result in tqdm(all_results_from_threads, desc="匯總分析結果"):
        if "error" in result:
            all_sequence_analysis_results.append({"batch_index": result['batch_index'] + 1, "analysis": default_error_result_structure()})
            continue
        image_batch_info, sequence_analysis_data = result["image_batch_info"], result["analysis"]
        batch_image_filenames = [info["filename"] for info in image_batch_info]
        all_sequence_analysis_results.append({ "batch_index": result['batch_index'] + 1, "image_filenames_in_batch": batch_image_filenames, "matched_teacher_position_text": result.get("matched_teacher_position_text"), "matched_classroom_view_image": result.get("matched_classroom_view_image"), "analysis": sequence_analysis_data })
        if "error" not in sequence_analysis_data:
            overall_behavior_summary["total_images_processed_in_batches"] += len(batch_image_filenames)
            if "per_image_highlights" in sequence_analysis_data:
                for hl_item in sequence_analysis_data.get("per_image_highlights", []):
                    
                    behavior_list_original = hl_item.get("behavior_category")
                    conf = hl_item.get("confidence", 0.0)
                    
                    UNKNOWN_LABEL = "被遮擋/無法判斷"
                    
                    # 複製一份原始行為列表，用於後續的非任務行為判斷
                    behavior_list_for_stats = behavior_list_original
                    
                    # 判斷條件 1: 信度是否低於設定的閾值
                    is_low_confidence = conf is not None and conf < BEHAVIOR_CONFIDENCE_THRESHOLD_FILTER
                    # 判斷條件 2: AI 是否沒有返回任何有效的行為標籤
                    is_empty_behavior = not behavior_list_original

                    # 如果信度過低或行為為空，則強制將此筆紀錄歸類為 "無法判斷"
                    if is_low_confidence or is_empty_behavior:
                        behavior_list_for_stats = [UNKNOWN_LABEL]
                        
                    if not behavior_list_for_stats: continue # 如果處理後仍然為空，則跳過

                    # 確保用於統計的行為列表是 list 格式
                    if not isinstance(behavior_list_for_stats, list):
                        behavior_list_for_stats = [behavior_list_for_stats]
                    
                    # 遍歷列表中的每一個行為進行統計 (此處使用的是可能被覆寫過的 behavior_list_for_stats)
                    for cat in behavior_list_for_stats:
                        overall_behavior_summary["behavior_counts"][cat] += 1
                        overall_behavior_summary["behavior_confidence_sum"][cat] += float(conf if conf is not None else 0.0)
                    
                    # 處理非任務行為範例（使用未經過濾的原始行為）
                    if behavior_list_original and isinstance(behavior_list_original, list):
                        primary_behavior_for_example = behavior_list_original[0]
                        non_task_keywords = [
                            "玩弄手部/文具", "觸摸臉部", "觸摸頭髮",
                            "目視他處", "趴睡", "喝水", "飲食"
                        ]
                        if primary_behavior_for_example in non_task_keywords and len(overall_behavior_summary["non_task_behavior_examples_for_summary"]) < 3:
                            try:
                                student_img_idx = hl_item.get("image_index_in_sequence", -1)
                                if 0 <= student_img_idx < len(batch_image_filenames):
                                    hl_filename = batch_image_filenames[student_img_idx]
                                    hl_timestamp = next((info["timestamp_str"] for info in image_batch_info if info["filename"] == hl_filename), "未知時間")
                                    context_desc = hl_item.get("context_description", "")
                                    
                                    overall_behavior_summary["non_task_behavior_examples_for_summary"].append( 
                                        (hl_filename, hl_timestamp, ", ".join(behavior_list_original), context_desc) 
                                    )
                            except Exception as e_idx: print(f"提取非任務示例時出錯: {e_idx}")
    
    overall_behavior_stats_list, total_highlight_instances = [], sum(overall_behavior_summary["behavior_counts"].values())
    valence_summary = {"正向": 0, "負向": 0, "中性": 0}
    for behavior, count in overall_behavior_summary["behavior_counts"].items():
        percentage, avg_confidence = (count / total_highlight_instances * 100) if total_highlight_instances > 0 else 0, (overall_behavior_summary["behavior_confidence_sum"][behavior] / count) if count > 0 else 0
        valence = LABEL_TO_VALENCE.get(behavior, "未分類")
        if valence in valence_summary: valence_summary[valence] += count
        overall_behavior_stats_list.append({ "behavior_category": behavior, "valence": valence, "count": count, "percentage": round(percentage, 1), "average_confidence": round(avg_confidence, 2) })
    overall_behavior_stats_list.sort(key=lambda x: x["count"], reverse=True)
    personalized_notes = generate_personalized_summary_notes(student_id, overall_behavior_stats_list, overall_behavior_summary["non_task_behavior_examples_for_summary"], client)
    behavior_to_images_map = defaultdict(list)
    for result in all_results_from_threads:
        if "error" in result or "analysis" not in result or "error" in result["analysis"]: continue
        sequence_analysis, image_batch_info, filenames_in_batch = result["analysis"], result.get("image_batch_info", []), [info.get("filename") for info in result.get("image_batch_info", [])]
        if "per_image_highlights" in sequence_analysis and isinstance(sequence_analysis["per_image_highlights"], list):
            for highlight in sequence_analysis["per_image_highlights"]:
                # 【修改點 4】修改索引建立邏輯以處理行為列表
                behavior_list, image_index = highlight.get("behavior_category"), highlight.get("image_index_in_sequence")
                
                if not behavior_list or not isinstance(image_index, int) or not (0 <= image_index < len(filenames_in_batch)):
                    continue

                if not isinstance(behavior_list, list):
                    behavior_list = [behavior_list]
                
                image_filename = filenames_in_batch[image_index]
                if image_filename:
                    # 為列表中的每一個行為都建立索引
                    for behavior_category in behavior_list:
                        if image_filename not in behavior_to_images_map[behavior_category]:
                            behavior_to_images_map[behavior_category].append(image_filename)
    for behavior in behavior_to_images_map: behavior_to_images_map[behavior].sort()
    print("行為索引建立完成。")

    total_classified_instances = sum(valence_summary.values())
    valence_summary_with_percentage = { valence: { "count": count, "percentage": round((count / total_classified_instances * 100), 1) if total_classified_instances > 0 else 0 } for valence, count in valence_summary.items() }

    final_json_output = {
        "report_metadata": {
            "student_id": student_id, "student_number": student_number, "report_generation_time": "08/24", "student_image_source_folder": "英文",
            "teacher_position_source_json": os.path.basename(TEACHER_POSITION_JSON) if TEACHER_POSITION_JSON and os.path.isfile(TEACHER_POSITION_JSON) else "N/A",
            "classroom_view_source_folder": os.path.basename(CLASSROOM_IMAGES_FOLDER) if CLASSROOM_IMAGES_FOLDER and os.path.isdir(CLASSROOM_IMAGES_FOLDER) else "N/A",            "classroom_context": { "student_position": student_position },
            "analysis_settings": {
                "vision_model": VISION_DEPLOYMENT_NAME, "text_model": TEXT_DEPLOYMENT_NAME, "images_per_batch": IMAGES_PER_API_CALL, "context_images_per_batch_desc": "動態匹配老師和班級照片各一張", "confidence_threshold": BEHAVIOR_CONFIDENCE_THRESHOLD_FILTER,
                "cost_optimization": { "sampling_rate": SAMPLING_RATE, "image_detail": IMAGE_DETAIL_LEVEL } # 新增成本優化資訊
            }
        },
        "overall_summary": {
            "total_images_found": len(image_files_with_timestamps) * SAMPLING_RATE if SAMPLING_RATE > 1 else len(image_files_with_timestamps), # 顯示原始數量
            "total_images_analyzed": overall_behavior_summary["total_images_processed_in_batches"],
            "total_batches": len(image_batches), "valence_summary": valence_summary_with_percentage, "behavior_statistics": overall_behavior_stats_list, "behavior_to_images_index": behavior_to_images_map, "ai_summary_notes": personalized_notes
        },
        "detailed_sequence_analysis": all_sequence_analysis_results
    }

    current_timestamp_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_student_id_for_filename = re.sub(r'[\\/*?:"<>|]',"", student_id.replace(" ", "_"))
    json_filename = JSON_FILENAME_TEMPLATE.format(student_id=safe_student_id_for_filename, timestamp=current_timestamp_str)
    json_filepath = os.path.join(student_specific_folder_base, json_filename)
    try:
        with open(json_filepath, 'w', encoding='utf-8') as f: json.dump(final_json_output, f, ensure_ascii=False, indent=4)
        print(f"\n✅ 學生 '{student_id}' 的序列行為分析報告已成功儲存至: {json_filepath}")
    except Exception as e: print(f"❌ 錯誤：儲存 JSON 檔案時發生問題：{e}")
    
    print("\n--- 處理完成 ---")

if __name__ == "__main__":
    main()

### azure gpt-5

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import base64
import json
import datetime
import time
from collections import Counter, defaultdict
from openai import AzureOpenAI, APIError, RateLimitError, AuthenticationError 
from PIL import Image, UnidentifiedImageError
import io
from tqdm import tqdm
import math
from dotenv import load_dotenv 
from concurrent.futures import ThreadPoolExecutor, as_completed
import bisect

# --- Configuration ---


# 班級整體照片資料夾路徑 
CLASSROOM_IMAGES_FOLDER = r'C:\Users\User\Desktop\test\student_full_classroom\0817_english_class'

# 【新】老師視角照片資料夾路徑 (從學生視角拍攝老師在黑板前的照片) - 可選
TEACHER_POSITION_JSON = r'C:\Users\User\Desktop\test\teacher_position\0817_position.json'

#課堂情況json
CLASSROOM_STATE_JSON_PATH = r'C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0817\老師上課\TXT\0817_processed.json'

# 【新】時間軸對齊設定 (格式: "HH:MM:SS")
CLASSROOM_START_TIME_OFFSET = "00:00:00"  # 班級照片的起始時間，設定為空字串 "" 表示不篩選
TEACHER_JSON_START_TIME_OFFSET ="00:00:00" # 老師位置資料的起始時間，設定為空字串 "" 表示不篩選

# ... (您原有的其他設定) ...

JSON_OUTPUT_FOLDER = "SynologyDrive\json_behavior"


JSON_FILENAME_TEMPLATE = "student_{student_id}_behavior_report_{timestamp}.json"
SAMPLING_RATE = 3  
IMAGE_DETAIL_LEVEL = "low" # 圖片解析度 ('low' 或 'high')，low 可大幅降低成本
MAX_TOKENS_VISION_COMPLETION = 3500
MAX_TOKENS_SUMMARY_COMPLETION = 1000 # 個性化總結的 token
BEHAVIOR_CONFIDENCE_THRESHOLD_FILTER = 0.97 # 行為信度過濾閾值 (例如 70%)
IMAGES_PER_API_CALL = 6 # 一次API調用處理的圖片數量
BEHAVIOR_SIMILARITY_CONFIDENCE_THRESHOLD = 0.15 # 過濾相似連續行為的信度差異閾值
API_RETRY_DELAY_SECONDS = 10

# ==========================================================

# --- 標準行為分類與定義 (核心辭典，來自圖一) ---
STANDARD_BEHAVIOR_CATEGORIES = {
    "視線": [
        {"label": "目視教師", "definition": "學生的頭部與視線明確聚焦於教室【前方】的【老師所在】區域，且其頭部旋轉角度處於一個【合理的朝前弧度】內（通常不超過45度）。**【絕對排除條款】**: 任何導致學生視線與其身體朝向構成【接近90度或更大角度】的頭部大幅度轉動，【絕對不允許】被標註為『目視教師』。這種姿態應被優先考慮為『目視同學』(`V_CLS`)或『目視他處』(`V_ELS`)。"},
        {"label": "目視黑板", "definition": "學生的頭部與視線明確聚焦於教室【前方】的【非老師所在】的黑板或螢幕區域。當老師位置未知，或學生視線明確未朝向老師時，這是面向前方的預設專注行為。"}, 
        {"label": "目視書本/筆記", "definition": "學生的頭部與視線向量【主要朝下】，且頭部的水平旋轉角度【沒有明顯偏離】其身體所朝向的【個人桌面工作區軸線】。此標籤捕捉的是在個人學習材料上的視覺專注狀態。**【嚴格邊界】**: 一旦頭部/視線明確地、持續地轉向側面（例如，足以與鄰座同學進行眼神交流），即使視線仍然略微朝下，也【必須優先考慮】標註為『目視同學』。**【注意】**：如果學生同時在進行『做筆記』，根據『動作優先』原則，你應將『做筆記』作為主要標籤。"},
        {"label": "目視同學", "definition": "學生的【主要身體朝向與視線】明確脫離前方或個人桌面，轉向側面或後方的同學。**【最高優先級的情境標籤】**: 一旦觀察到這種明確的身體轉向，此標籤的優先級就高於大多數獨立的個人動作。它涵蓋了從純粹的視覺交流到【涉及物件的協作行為】（如共同看書、討論問題）。**【核心情境判斷規則】**: 此行為的性質根據【課堂狀態】決定..."},
        {"label": "目視他處", "definition": "【備用排除性標籤】。當你已確認學生的視線【不】符合『目視教師』、『目視黑板』、『目視書本/筆記』或『目視同學』的任何一項明確定義時，才使用此標籤。它捕捉的是失去焦點的狀態，例如：【抬頭看天花板】、【轉頭看沒有同學及老師的地方】。"}     
    ],
    "肢體(手部)": [
        {"label": "做筆記", "definition": "【核心證據：觀察到一個動態的書寫過程】。你必須能明確看到學生手持筆，且筆尖正在紙張上進行【有意義的移動或書寫/繪製動作】。**【最高優先級】**：這是一個高優先級的動作標籤。**【嚴格排除條款】**：以下情況【絕對不允許】標註為『做筆記』：(1) **靜態持筆**：僅僅手持筆，或筆尖靜止停留在紙上，應標註為『目視書本/筆記』 (`V_BOK`)。(2) **動作中斷**：當學生手部的主要動作變為其他行為（如『觸摸頭髮』、『托腮』、『與同學互動』），即使手中仍持有筆，也必須以該【瞬時動作】為主要標籤。"},
        {"label": "翻書", "definition": "學生手部正在主動翻閱、移動書本。"},
        {"label": "觸摸臉部", "definition": "學生【非支撐性地】用手短暫觸碰或摩擦自己的臉部、鼻子、嘴巴或眼睛。此行為區別於『托腮』的持續性支撐動作。"},
        {"label": "觸摸頭髮", "definition": "學生用手觸摸、撥弄或整理自己的頭髮。注意：即使手臂抬得較高，只要手部的主要動作是與頭髮互動，就【必須】使用此標籤，而不是『舉手』類標籤。"}
    ],
    "身體姿態": [
        {"label": "坐姿直立", "definition": "學生上半身軀幹基本垂直於地面，或輕微前傾。"},
        {"label": "身體前傾", "definition": "學生上半身軀幹明顯向前彎曲，靠近桌面。此行為描述的是一種【清醒狀態下】的姿態。如果學生頭部接觸桌面或手臂，應【優先使用】『趴睡』標籤。"},
        {"label": "身體後靠", "definition": "學生背部倚靠在椅背上。"},
        {"label": "低頭(非學習)", "definition": "【極其嚴格的排除性標籤】。僅在你能夠【極度確信地】觀察到以下【全部】條件時才可使用：1. 學生頭部明顯低垂。2. 其視線【明確沒有】朝向任何學習材料（例如，看向地面、自己的懷中或空無一物的桌面）。3. 其手部沒有在進行任何學習相關操作。"},
        {"label": "趴睡", "definition": "學生將頭部【枕於】手臂或桌面上，呈現明確的休息或睡眠狀態。**【最高優先級】**：只要觀察到頭部接觸桌面或手臂的休息姿態，此標籤的優先級【高於】所有其他學習相關標籤（如『做筆記』、『目視書本』）。"},
        {"label": "托腮", "definition": "【核心定義：手部對頭部提供持續性支撐】。學生使用一隻或兩隻手的手掌、拳頭或手臂，支撐其下巴、臉頰或頭部的重量。這是一個純粹的物理姿態描述，不包含任何意圖推斷。"}
    ],
    "互動": [
        {"label": "主動舉手", "definition": "學生舉起一隻手，意圖提問或回答問題。**【三大核心物理證據，必須同時滿足】**：(1) 手臂向上伸展，手部明顯高於肩膀。(2) 手掌形態為張開朝前或中性放鬆，【嚴禁】手指指向特定方向或揮舞。(3) 該動作具有一定的持續性（非瞬間劃過）。**【情境觸發】**：此行為最常發生在【老師單向授課】的狀態下（如『文法/句型講解』、『閱讀/文章分析』），代表學生的自發性提問或補充。**【絕對排除】**：任何手部接觸頭部、與同學互動的手勢、指向性的動作，都【不允許】標記為此行為。"},
        {"label": "被動舉手", "definition": "學生舉手以回應老師的群體性指令（如投票、調查）。**【核心視覺證據】**：通常是多數學生同時舉手，姿態可能較為放鬆，手臂不必完全伸直。**【情境觸發】**：此行為最常發生在【老師與學生互動】的狀態下（如『課堂問答/互動』、『習題/考卷檢討』），代表學生回應老師的指令或提問。**【關鍵區分】**：此標籤的判斷【高度依賴】課堂情境和群體性動作。如果情境不匹配，應避免使用此標籤。"}
    ],
    "其他狀態": [
        {"label": "喝水", "definition": "【核心證據：清晰可見的容器】。只有當你能夠【明確地】看到學生手持水瓶、杯子或其他容器，並將其送至嘴邊時，才可以使用此標籤。**【嚴格排除】**: 任何僅有低頭姿態、手部靠近臉部但【沒有可見容器】的場景，都【嚴禁】標註為此行為。在此情況下，應優先考慮『目視書本/筆記』(`V_BOK`)或『趴睡』(`P_SLP`)。"},
        {"label": "飲食", "definition": "學生正在食用固體食物。"},
        {"label": "玩弄手部／文具", "definition": "學生手部在進行與學習無關的重複性小動作，例如玩手指、轉筆、玩弄橡皮擦等"},
        # {"label": "整理書包", "definition": "學生正在整理書包、桌面文具。"},
        {"label": "被遮擋/無法判斷", "definition": "【最終備用標籤】。因遮擋、模糊或角度問題，無法清晰識別學生的主要行為時，【必須】使用此標籤。"}
    ]
}
# 自動從新結構生成有效的標籤列表
VALID_BEHAVIOR_LABELS = [item['label'] for category in STANDARD_BEHAVIOR_CATEGORIES.values() for item in category]

BEHAVIOR_CODES = {
    # 視線 (不變)
    "V_TCH": "目視教師", "V_BRD": "目視黑板", "V_BOK": "目視書本/筆記",
    "V_CLS": "目視同學", "V_ELS": "目視他處",
    
    # 肢體(手部) (移除 H_FLIP)
    "H_NOT": "做筆記",
    "H_PLAY_HW": "玩弄手部/文具",
    "H_TOUCH_F": "觸摸臉部",
    "H_TOUCH_H": "觸摸頭髮",
    
    # 身體姿態 (新增 P_THK - P for Posture, THK for Thinking)
    "P_STR": "坐姿直立", "P_LEAN": "身體前傾", "P_BACK": "身體後靠",
    "P_DWN": "低頭(非學習)", "P_SLP": "趴睡",
    "P_THK": "托腮", # <--- 新增
    
    # 互動 (將 I_HND 拆分為主動/被動)
    "I_HND_A": "主動舉手", # A for Active
    "I_HND_P": "被動舉手", # P for Passive
    
    # 其他狀態 (不變)
    "O_DRK_W": "喝水",
    "O_EAT_S": "飲食",
    # "O_TDY": "整理個人物品",
    "O_UNK": "被遮擋/無法判斷"
}

# 自動生成反向查找字典，用於本地解碼
CODE_TO_BEHAVIOR = {code: label for code, label in BEHAVIOR_CODES.items()}
BEHAVIOR_TO_CODE = {label: code for code, label in BEHAVIOR_CODES.items()}

# AI返回標籤到標準標籤的映射規則 (與新標籤對齊)
BEHAVIOR_MAPPING_RULES = {
    "目視桌面/教材": "目視書本", "目視桌面": "目視書本", "看書": "目視書本",
    "書寫/做筆記": "筆記", "動手操作-書寫/做筆記": "筆記",
    "視覺專注-閱讀書本/講義": "目視書本",
    "視覺專注-看老師/黑板方向": "目視教師", "目視黑板/老師": "目視教師",
    "看老師": "目視教師", "看黑板": "目視黑板",
    # --- ↓↓↓ 【修改點】更新映射規則以對應新標籤 ↓↓↓ ---
    "玩弄物品": "玩弄手部/文具", # 將模糊的舊標籤對應到最可能的新標籤
    "非任務相關動作-玩弄物品(筆等)": "玩弄手部/文具",
    # 移除了"非任務相關動作-觸摸臉部/頭髮"，鼓勵AI直接使用更精確的新標籤
    # "非任務相關動作-整理物品": "整理個人物品",
    "社交互動-與同學互動": "目視同學",
    "趴睡/休息": "趴睡",
    "低頭/伏案(非睡)": "低頭",
}

BEHAVIOR_VALENCE_MAP = {
    "正向": [
        "做筆記",
        "主動舉手", # <-- 修改
        "目視教師",
        "目視黑板",
        "目視書本/筆記",
    ],
    "負向": [
        "趴睡",
        "玩弄手部/文具",
        "觸摸臉部",
        "觸摸頭髮",
        "目視他處",
        "目視同學"
    ],
    "中性": [
        "身體前傾",
        "坐姿直立",
        "身體後靠",
        "喝水",
        "飲食",       
        "玩弄手部／文具",
        "翻書",
        "低頭(非學習)",
        # "整理個人物品",     
        "被遮擋/無法判斷",
        "托腮", # <-- 新增
        "被動舉手"  # <-- 新增
    ]
}

LABEL_TO_VALENCE = {
    label: valence 
    for valence, labels in BEHAVIOR_VALENCE_MAP.items() 
    for label in labels
}

# ---------------------------
#  API 金鑰配置
# ---------------------------
load_dotenv()

AZURE_API_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
# 從環境變數讀取部署名稱
VISION_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_VISION_DEPLOYMENT")
SUMMARY_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_SUMMARY_DEPLOYMENT")

# 為了兼容，讓 TEXT_DEPLOYMENT_NAME 跟隨視覺模型 (雖然目前沒用到)
TEXT_DEPLOYMENT_NAME = VISION_DEPLOYMENT_NAME 

# 檢查必要的 Azure 配置是否存在
if not all([AZURE_API_KEY, AZURE_ENDPOINT, VISION_DEPLOYMENT_NAME, SUMMARY_DEPLOYMENT_NAME]):
    print("錯誤：缺少必要的 Azure OpenAI 環境變數。")
    print("請檢查您的 .env 檔案是否包含 AZURE_OPENAI_KEY, AZURE_ENDPOINT, AZURE_OPENAI_VISION_DEPLOYMENT, 和 AZURE_OPENAI_SUMMARY_DEPLOYMENT。")
    exit()

try:
    # ### 【修改 3】: 初始化 AzureOpenAI Client ###
    client = AzureOpenAI(
        api_key=AZURE_API_KEY,
        azure_endpoint=AZURE_ENDPOINT,
        api_version="2024-02-01"  # 使用一個穩定的 API 版本
    )
    client.models.list() # 嘗試調用一個簡單的API來驗證配置
    print("Azure OpenAI client 初始化並驗證成功。")
    print(f"  - 視覺分析將使用部署: '{VISION_DEPLOYMENT_NAME}'")
    print(f"  - 個性化總結將使用部署: '{SUMMARY_DEPLOYMENT_NAME}'")
except AuthenticationError: print("錯誤：Azure OpenAI API 金鑰或端點無效，認證失敗。"); exit()
except Exception as e: print(f"初始化 Azure OpenAI Client 時發生錯誤: {e}"); exit()

# ---------------------------
# 輔助函數
# ---------------------------
def get_behavior_sequence_analysis_system_prompt(student_position, has_teacher_context):
    """
    【v14.4 - 終極重構版】
    此版本在 v14.3 的基礎上進行了結構性重構，將所有規則模塊化，
    消除了邏輯冗餘，並將決策樹統一整合，達到了最高的清晰度與可維護性。
    """
    # --- 1. 行為編碼表 (靜態模塊) ---
    behavior_table_for_prompt = "\n\n**【學習行為編碼表】**\n你 **必須** 且 **只能** 從以下列表的「編碼 (Code)」中選擇行為進行標註。...\n"
    for code, label in BEHAVIOR_CODES.items():
        definition = ""
        for category in STANDARD_BEHAVIOR_CATEGORIES.values():
            for item in category:
                if item['label'] == label:
                    definition = item['definition']
                    break
            if definition:
                break
        behavior_table_for_prompt += f"*   **`{code}`**: {label} - {definition}\n"
    behavior_table_for_prompt += "\n*如果行為因任何原因無法清晰判斷，請 **必須** 使用 **`O_UNK`** 編碼。*\n"

    # --- 2. 空間與攝影機規則 (動態模塊) ---
    analysis_view_rules = ""
    if "左" in student_position:
        analysis_view_rules = "*   **分析視角**: 拍攝該學生的【學生個人照片序列】來自教室前方的【左側】攝影機。\n*   **視線基準**: 對於這位學生，【正面朝向鏡頭】僅代表他在看教室的【左前方】。為了看向【教室正前方中心】，他的頭部必須輕微地朝向他自己的【右側】轉動。"
    elif "右" in student_position:
        analysis_view_rules = "*   **分析視角**: 拍攝該學生的【學生個人照片序列】來自教室前方的【右側】攝影機。\n*   **視線基準**: 對於這位學生，【正面朝向鏡頭】僅代表他在看教室的【右前方】。為了看向【教室正前方中心】，他的頭部必須輕微地朝向他自己的【左側】轉動。"
    else: # 預設為中間
        analysis_view_rules = "*   **分析視角**: 拍攝該學生的【學生個人照片序列】來自教室前方的【中間】攝影機。\n*   **視線基準**: 對於這位學生，當他臉部【正面朝向鏡頭】時，即代表其視線正朝向教室的【正前方中心】。"

    camera_and_spatial_rules = f"""
---
**【第一部分：多機位系統與空間推理框架 (最高優先級)】**
你的所有空間判斷都必須基於一個【兩步推理】的框架：首先建立宏觀地圖，然後應用微觀視角。

*   **準則 1A: 宏觀情境視角 (固定中央廣角鏡頭)**
    *   **定義**: 用於情境感知的【班級整體照片】**永遠**來自教室前方的**中央廣角鏡頭**。
    *   **任務**: 你必須使用這張照片來建立一個視覺座標系，將教室劃分為**左側、中央、右側**三個區域，並在其中定位目標學生。

*   **準則 1B: 微觀分析視角 (動態學生鏡頭)**
    *   {analysis_view_rules}

*   **通用空間準則**:
    *   **朝前弧度原則**: 對教室前方的專注行為（`V_TCH`, `V_BRD`）必須發生在一個合理的「朝前弧度」內。一個使學生頭部與其肩膀構成【接近90度】的**大幅度轉頭**，是其注意力【脫離前方】的明確證據，你**必須**將其優先判定為 `V_CLS`。
    *   **影像鏡像轉換**: 所有攝影機畫面都是鏡像的：照片中的**左側** => 教室中的**右側**；照片中的**右側** => 教室中的**左側**。

**【核心執行協議】**:
你的視線分析**必須**結合宏觀與微觀資訊。例如：在班級照片中，你確認學生位於**畫面左側區域** (準則 1A)。同時，`student_position` 文字告訴你拍攝他的鏡頭在**左側** (準則 1B)。綜合這兩點，你才能最準確地判斷他為了看向老師而需要轉動的角度。
"""

    # --- 3. 行為決策協議 (動態模塊) ---
    # 將前後排規則與統一決策協議整合成一個大的、連貫的決策流程
    is_front_row = any(keyword in student_position for keyword in ["第一", "第二"])
    is_rear_row = any(keyword in student_position for keyword in ["第三", "第四"])

    # 3.1 定義前後排獨有的規則
    front_row_rules = """
*   **低頭行為的分診決策樹 (軸線檢查版)**:
    1.  **軸線檢查**: 判斷頭部是【正下方】（判定 `V_BOK`）還是【側下方】（判定 `V_CLS`）。
    2.  **手部動作覆寫**: 檢查是否有隱蔽的非學習動作 (`H_PLAY_HW`) 來覆寫視線判斷。
    3.  **最終備用**: 都不是才使用 `P_DWN`。
*   **前方視線的精確匹配**: 嚴格結合【老師位置情境】進行幾何匹配，判定 `V_TCH` 或 `V_BRD`。
"""
    rear_row_rules = """
*   **低可視度下的姿態優先原則**: 當細節無法辨認時，必須忽略猜測，嚴格依賴姿態鐵則。
*   **低頭行為的「絕對學習推定」原則**: 只要頭部低垂，就**必須、無條件地**判定為 `V_BOK`。
*   **前方視線的「教師優先」原則**: 只要視線朝前，就**優先**標註為 `V_TCH`。
"""

    # 3.2 根據學生位置選擇對應規則
    if is_rear_row:
        position_specific_rules = f"""
**【第二部分：後排學生分析協議 (絕對推斷模式)】**
你必須遵循以下不可動搖的推斷鐵則：
{rear_row_rules}
"""
    else: # 預設為前排
        position_specific_rules = f"""
**【第二部分：前排學生分析協議 (高證據模式)】**
你的所有判斷都必須基於【嚴格的視覺證據】：
{front_row_rules}
"""

    # 3.3 定義統一的、後續的行為決策流程
    universal_decision_flow = """
---
**【第三部分：統一行為決策流程 (全體適用)】**
**【核心原則】**: 你的判斷【必須】基於每一張獨立圖片捕捉到的**瞬時物理證據**。在通過【第二部分】獲得初步的視線判斷後，你必須嚴格遵循以下【單一、統一】的分層決策樹來確定最終的主要行為標籤。

*   **第一層：檢查【壓倒性的身體姿態】(最高優先級)**
    *   **情況 A (向前協作姿態):** 身體【向前或向側前方】探出，進入鄰座同學的桌面空間 -> 主要行為判定為 **`V_BOK`**。**決策結束。**
    *   **情況 B (向後/側面社交姿態):** 身體向【側面或後方】旋轉，與同學進行面對面交流 -> 主要行為判定為 **`V_CLS`**。**決策結束。**

*   **第二層：檢查【定義明確的主動動作】(僅在身體朝前時評估)**
    *   是否在進行**動態的書寫 (`H_NOT`)**？ -> 是 -> `H_NOT` 是主要標籤。**決策結束。**
    *   是否在**玩弄手部/文具 (`H_PLAY_HW`)**？ -> 是 -> `H_PLAY_HW` 是主要標籤。**決策結束。**
    *   手臂是否抬起？ -> 是 -> 執行**【舉手行為過濾器】**（排除 `V_CLS`, `H_TOUCH_H/F` 後，判斷 `I_HND_A/P`）。**決策結束。**

*   **第三層：檢查【其他細微姿態與動作】**
    *   學生是否將頭**枕於**手臂或桌面？ -> 是 -> `P_SLP` (趴睡)。**決策結束。**
    *   學生是否在**托腮 (`P_THK`)**？ -> 是 -> `P_THK` 作為主要標籤。
    *   學生是否在進行**其他非任務相關動作**（如觸摸臉部 `H_TOUCH_F`）？ -> 是 -> 標註對應動作。

*   **第四層：標註【靜態視覺行為】(最終備用選項)**
    *   僅在**所有**上述檢查項都不適用的情況下，才使用在【第二部分】中獲得的**初步視線判斷**（如 `V_TCH`, `V_BOK`）作為最終的主要行為標籤。

*   **第五層：疊加【通用輔助姿態】**
    *   在確定了主要行為後，可疊加一個輔助的姿態標籤（如 `P_LEAN`）。
"""

    # --- 4. 生成最終的、完整的 Prompt ---
    teacher_context_header = "5.  **【老師位置文字情境】**: 用於交叉驗證你視線判斷的輔助數據。" if has_teacher_context else ""

    return f"""
「你是一位世界頂尖的教育分析師，精通電腦視覺、空間幾何推理與心理學。你的任務是綜合所有給定的物理與情境資訊，對學生的學習行為進行最精準、最客觀的標註。」

**【你收到的資訊來源】**
1.  **【班級整體照片】**: 你的【空間座標系】基準。
2.  **【學生個人照片序列】**: 你的主要分析對象。
3.  **【課堂狀態】**: 判斷行為動機的核心上下文。
4.  **【學生座位文字描述】**: `{student_position}`，用於輔助定位。
{teacher_context_header}

{camera_and_spatial_rules}

{position_specific_rules}

{universal_decision_flow}

{behavior_table_for_prompt}

---
**【第四部分：輸出格式要求 - 嚴謹推理與簡明輸出】**
*   你的回答**必須**是一個結構完整的、單一的 JSON 物件。
*   `behavior_category` 的值**必須**是一個包含 1 到 2 個編碼字串的**陣列 (Array)**。
*   `per_image_highlights` 的每個物件中**必須包含** `image_index_in_sequence`, `context_description`, `behavior_category`, 和 `confidence` 這四個鍵。
*   **【關鍵指令】**: `context_description` 內容**必須極度簡潔**，例如："後排協議 -> 低頭推定" 或 "左側區域規則 -> 視線匹配"。

**【輸出 JSON 格式範例】**
```json
{{
  "sequence_analysis_confidence": 0.97,
  "per_image_highlights": [
    {{
      "image_index_in_sequence": 0,
      "context_description": "後排協議：低頭推定。",
      "behavior_category": ["V_BOK", "P_STR"],
      "confidence": 0.95
    }},
    {{
      "image_index_in_sequence": 1,
      "context_description": "左側區域規則：頭部右轉，匹配老師位置。",
      "behavior_category": ["V_TCH"],
      "confidence": 0.99
    }},
    {{
      "image_index_in_sequence": 2,
      "context_description": "社交優先：向前協作，判定為 V_BOK。",
      "behavior_category": ["V_BOK", "P_LEAN"],
      "confidence": 0.98
    }}
  ]
}}
```"""

def extract_json_from_string(text):
    """
    從可能包含額外文字或 Markdown 區塊的字串中，穩健地提取出 JSON 字串。
    """
    # 優先嘗試尋找被 markdown code block 包圍的 JSON
    # re.DOTALL 讓 '.' 可以匹配換行符
    match = re.search(r"```json\s*(\{.*\})\s*```", text, re.DOTALL)
    if match:
        # 如果找到，直接返回第一個捕獲組 (也就是 {...} 的部分)
        return match.group(1)

    # 如果沒有 markdown，就尋找第一個 '{' 和最後一個 '}'
    start_index = text.find('{')
    end_index = text.rfind('}')
    
    if start_index != -1 and end_index != -1 and end_index > start_index:
        # 如果都找到了，返回它們之間的子字串
        return text[start_index : end_index + 1]
        
    # 如果以上方法都失敗，返回原始文本，讓後續的 json.loads() 自然地報錯
    return text

def find_state_for_timestamp(target_timestamp, classroom_states):
    """
    【v4.0 穩定版】根據時間戳，查找對應的課堂狀態標籤。
    使用線性查找以確保最高的準確性和可靠性。
    """
    if not classroom_states:
        return "未知"
    
    # 為了除錯，只打印一次查找的詳細信息
    if not hasattr(find_state_for_timestamp, "has_logged_first_search"):
        print("\n" + "-"*20 + " [除錯日誌 - 首次查找課堂狀態] " + "-"*20)
        print(f"  - 正在用第一張照片的時間戳进行匹配...")
        print(f"  - 目標時間戳 (Target Timestamp): {target_timestamp}")
        if classroom_states:
            first_state = classroom_states[0]
            print(f"  - 正在检查第一個時間區間: 從 {first_state.get('start_time_td')} 到 {first_state.get('end_time_td')}")
        print("-" * 69 + "\n")
        find_state_for_timestamp.has_logged_first_search = True

    # 遍歷每一個課堂狀態的時間區間
    for state in classroom_states:
        start_td = state.get("start_time_td")
        end_td = state.get("end_time_td")
        
        # 確保這個區間的時間數據是有效的
        if start_td and end_td:
            # 檢查目標時間戳是否落在 [開始時間, 結束時間] 這個閉区间内
            if start_td <= target_timestamp <= end_td:
                return state["classroom_state"]
    
    # 如果遍歷完所有區間都沒找到，說明時間戳確實超出了範圍
    return "未知"

def parse_time_offset(time_str):
    """將 "HH:MM:SS" 格式的字串轉換為 timedelta 物件"""
    if not time_str or not isinstance(time_str, str):
        return None
    try:
        h, m, s = map(int, time_str.split(':'))
        return datetime.timedelta(hours=h, minutes=m, seconds=s)
    except (ValueError, TypeError):
        print(f"警告：時間偏移量 '{time_str}' 格式不正確，應為 'HH:MM:SS'。將忽略此設定。")
        return None

def get_valid_input(prompt_message):
    while True:
        user_input = input(prompt_message).strip()
        if user_input: return user_input
        print("錯誤：輸入不能為空，請重新輸入。")

def get_valid_folder_path(prompt_message, is_optional=False): # 新增 is_optional 參數
    while True:
        folder_path = input(prompt_message).strip().strip('"')
        if not folder_path and is_optional:
            return None # 如果是可選的且用戶未輸入，返回 None
        if os.path.isdir(folder_path):
            return folder_path
        print(f"錯誤：路徑 '{folder_path}' 不是一個有效的資料夾，請重新輸入。")

def get_timestamp_from_filename(filename):
    """
    【v3.0 兼容版】从档名解析时间戳。
    能同时处理多种常见格式。
    """
    # 模式一：最优先，精确匹配 HH-MM-SS-ms.jpg 格式 (常见于学生照片)
    match = re.search(r'^(\d{2})-(\d{2})-(\d{2})-(\d{3})\.(jpg|jpeg|png|webp)$', filename, re.IGNORECASE)
    if match:
        try:
            h, m, s, ms, ext = match.groups()
            return datetime.timedelta(hours=int(h), minutes=int(m), seconds=int(s), milliseconds=int(ms))
        except (ValueError, IndexError):
            pass

    # 模式二：匹配包含 ...HH-MM-SS-ms... 的通用格式
    match = re.search(r'(\d{2})-(\d{2})-(\d{2})-(\d{3})', filename)
    if match:
        try:
            h, m, s, ms = map(int, match.groups())
            return datetime.timedelta(hours=h, minutes=m, seconds=s, milliseconds=ms)
        except (ValueError, IndexError):
            pass

    # 模式三：匹配包含 ...h...m...s 的通用格式
    match = re.search(r'(\d+)h(\d{2})m(\d{2})s', filename)
    if match:
        try:
            h, m, s = map(int, match.groups())
            return datetime.timedelta(hours=h, minutes=m, seconds=s)
        except (ValueError, IndexError):
            pass

    # 如果所有格式都沒匹配成功，最终返回 None
    return None

def encode_image_to_base64(image_path, max_size_kb=512, target_quality=75):
    try:
        with Image.open(image_path) as img: img.verify()
        with Image.open(image_path) as img:
            if img.mode == 'RGBA' or img.mode == 'P': img = img.convert('RGB')
            current_quality = target_quality
            output_buffer = io.BytesIO()
            temp_img = img.copy()
            temp_img.save(output_buffer, format="JPEG", quality=current_quality)
            current_size_kb = output_buffer.tell() / 1024

            if current_size_kb > max_size_kb:
                scale_factor = math.sqrt(max_size_kb / current_size_kb)
                new_width = int(temp_img.width * scale_factor * 0.9)
                new_height = int(temp_img.height * scale_factor * 0.9)
                if new_width >= 50 and new_height >= 50: # 最小尺寸限制
                    print(f"    圖片 {os.path.basename(image_path)} ({current_size_kb:.1f} KB) 過大，縮放並調整質量...")
                    temp_img = temp_img.resize((new_width, new_height), Image.Resampling.LANCZOS)
                    output_buffer = io.BytesIO()
                    temp_img.save(output_buffer, format="JPEG", quality=max(current_quality - 15, 40)) # 質量可以降更多
                else:
                    print(f"    警告: 圖片 {os.path.basename(image_path)} 縮放後過小，可能影響質量。使用較低質量。")
                    output_buffer = io.BytesIO()
                    img.save(output_buffer, format="JPEG", quality=max(current_quality // 2, 30) )


            output_buffer.seek(0)
            binary_data = output_buffer.getvalue()
            base64_encoded_data = base64.b64encode(binary_data)
            return f"data:image/jpeg;base64,{base64_encoded_data.decode('utf-8')}"
    except Exception as e: print(f"錯誤：處理圖片 '{image_path}': {e}"); return None

def load_classroom_states(json_path):
    """【全新】讀取預處理好的課堂狀態時間軸 JSON 檔案。"""
    if not json_path or not os.path.isfile(json_path):
        print("提示：未提供或找不到課堂狀態 JSON 檔案。將不使用課堂情境。")
        return None  # 返回 None 以便更明確地判斷失敗
    
    print(f"正在讀取課堂狀態時間軸: {json_path}...")
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f) # <--- 關鍵修正：將 f 作為參數傳入
            
            # 將時間字串預先轉換為 timedelta 物件以便快速比較
            timeline = data.get("timeline", [])
            for state in timeline:
                state["start_time_td"] = parse_time_offset(state["start_time"])
                state["end_time_td"] = parse_time_offset(state["end_time"])
            print(f"成功載入 {len(timeline)} 個課堂狀態階段。")
            return timeline
    except Exception as e:
        print(f"❌ 錯誤：讀取或解析課堂狀態 JSON 時發生問題: {e}")
        return None # 返回 None

def default_error_result_structure():
    return { "error": "分析失敗或無有效數據", "sequence_analysis_confidence": 0.0, "sequence_summary": "未能生成序列總結。", "dominant_sustained_behaviors": [], "significant_behavior_shifts": [], "per_image_highlights": [], "general_sequence_atmosphere_hint": "未知" }

def clean_analysis_json(raw_analysis):
    """
    清理從 API 返回的分析 JSON，只保留我們需要的欄位。
    這是一個防禦性措施，用來處理微調模型的「幻覺」問題。
    """
    if not isinstance(raw_analysis, dict):
        return default_error_result_structure()

    # 定義合法的鍵
    allowed_top_level_keys = {"sequence_analysis_confidence", "per_image_highlights"}
    allowed_highlight_keys = {"image_index_in_sequence", "context_description", "behavior_category", "confidence"}

    cleaned_analysis = {}
    
    # 1. 清理最外層的鍵
    for key, value in raw_analysis.items():
        if key in allowed_top_level_keys:
            cleaned_analysis[key] = value

    # 2. 檢查並清理 per_image_highlights 列表
    if "per_image_highlights" in cleaned_analysis and isinstance(cleaned_analysis["per_image_highlights"], list):
        cleaned_highlights = []
        for raw_highlight in cleaned_analysis["per_image_highlights"]:
            if not isinstance(raw_highlight, dict):
                continue # 如果列表中的元素不是字典，就跳過它
            
            cleaned_highlight = {}
            for key, value in raw_highlight.items():
                if key in allowed_highlight_keys:
                    cleaned_highlight[key] = value
            
            # 確保必要的鍵存在，即使為空
            for required_key in allowed_highlight_keys:
                if required_key not in cleaned_highlight:
                    cleaned_highlight[required_key] = None

            cleaned_highlights.append(cleaned_highlight)
        
        cleaned_analysis["per_image_highlights"] = cleaned_highlights

    # 3. 如果最外層缺少必要的鍵，補上預設值
    if "sequence_analysis_confidence" not in cleaned_analysis:
        cleaned_analysis["sequence_analysis_confidence"] = 0.0
    if "per_image_highlights" not in cleaned_analysis:
        cleaned_analysis["per_image_highlights"] = []

    return cleaned_analysis

def analyze_student_behavior_from_images_sequence(student_image_paths, teacher_position_text, classroom_view_image_path, classroom_state, image_filenames_batch, openai_client, student_id, student_position):
    """
    【v8.2 偵錯增強版】分析單一圖片批次，包含針對 JSON 解析失敗的詳細檢查。
    此版本能明確區分內容篩選、Token 上限和模型格式錯誤。
    """
    if not student_image_paths:
        return default_error_result_structure()
    
    user_message_content = []
    
    # 1. 編碼學生個人圖片序列
    encoded_student_images = []
    for img_path in student_image_paths:
        b64_img = encode_image_to_base64(img_path)
        if b64_img:
            encoded_student_images.append({
                "type": "image_url", 
                "image_url": {"url": b64_img, "detail": IMAGE_DETAIL_LEVEL}
            })

    if not encoded_student_images:
        return default_error_result_structure()
        
    # 2. 編碼班級整體照片
    encoded_classroom_view_image = None
    if classroom_view_image_path:
        b64_img = encode_image_to_base64(classroom_view_image_path)
        if b64_img:
            encoded_classroom_view_image = {
                "type": "image_url", 
                "image_url": {"url": b64_img, "detail": IMAGE_DETAIL_LEVEL}
            }

    # 3. 組合完整的請求體
    user_message_content.append({"type": "text", "text": f"請根據系統提示中的偵探任務，分析學生「{student_id}」的行為。學生照片序列的文件名（供您參考）為: {', '.join(image_filenames_batch)}。"})
    user_message_content.append({"type": "text", "text": f"【課堂狀態】: {classroom_state}"})
    user_message_content.append({"type": "text", "text": f"【老師位置情境】根據預先分析，在此時間段，老師的位置在教室前方的「{teacher_position_text}」。請以此作為判斷『目視教師』的核心依據。"})
    
    if encoded_classroom_view_image:
        user_message_content.append({"type": "text", "text": "【班級整體照片】(用於定位學生和觀察整體氛圍)"})
        user_message_content.append(encoded_classroom_view_image)
    user_message_content.append({"type": "text", "text": "【學生個人照片序列】(主要分析對象)"})
    user_message_content.extend(encoded_student_images)

    # 4. 獲取系統提示
    system_prompt_content = get_behavior_sequence_analysis_system_prompt(student_position, bool(teacher_position_text and teacher_position_text != "未知"))

    # 5. API 呼叫與錯誤處理
    retry_attempts = 2
    for attempt in range(retry_attempts + 1):
        raw_content = ""
        finish_reason = "unknown" # 初始化 finish_reason
        try:
            has_teacher_context = bool(teacher_position_text and teacher_position_text != "未知")
            has_classroom_image = bool(encoded_classroom_view_image)
            has_state = classroom_state != "未知"

            print(f"  正在向 {VISION_DEPLOYMENT_NAME} 發送請求 (學生: {len(encoded_student_images)}, 課堂狀態: {'有' if has_state else '無'}, 老師位置: {'有' if has_teacher_context else '無'}, 班級照片: {'有' if has_classroom_image else '無'})...")
            
            response = openai_client.chat.completions.create(
                model=VISION_DEPLOYMENT_NAME, 
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": system_prompt_content},
                    {"role": "user", "content": user_message_content}
                ],
                # 根據之前的錯誤，已移除 temperature 參數
                # 根據之前的錯誤，已將 max_tokens 改為 max_completion_tokens
                max_completion_tokens=MAX_TOKENS_VISION_COMPLETION 
            )

            # === 【全新】增加詳細的完成原因檢查 ===
            choice = response.choices[0]
            finish_reason = choice.finish_reason

            # 1. 檢查是否被內容篩選器擋掉 (最高優先級)
            if finish_reason == 'content_filter':
                print(f"    警告：API請求因「內容篩選」而終止 (第 {attempt+1} 次嘗試)。這通常是因為輸入的圖片觸發了安全策略。")
                time.sleep(API_RETRY_DELAY_SECONDS)
                continue # 直接跳到下一次 for 迴圈的重試

            # 2. 檢查是否因為達到 Token 上限而被切斷
            if finish_reason == 'length':
                print(f"    警告：API回應因達到 max_completion_tokens ({MAX_TOKENS_VISION_COMPLETION}) 上限而被截斷 (第 {attempt+1} 次嘗試)。")
            
            raw_content = choice.message.content

            # 3. 檢查回傳內容是否為空
            if not raw_content:
                print(f"    錯誤：API返回了空的內容 (第 {attempt+1} 次嘗試)。完成原因: {finish_reason}")
                time.sleep(5)
                continue
            # ==========================================

            # === ✅【核心修改點】在解析前，先呼叫清理函數 ===
            cleaned_content = extract_json_from_string(raw_content)
            # ===============================================
            
            # 解析原始回應
            analysis_result_raw = json.loads(cleaned_content)
            
            # 清理可能包含幻覺的 JSON
            analysis_result = clean_analysis_json(analysis_result_raw)

            # 本地解碼
            if "per_image_highlights" in analysis_result and isinstance(analysis_result["per_image_highlights"], list):
                for hl in analysis_result["per_image_highlights"]:
                    if not hl or not isinstance(hl, dict) or "behavior_category" not in hl:
                        continue
                    codes = hl.get("behavior_category")
                    if not codes: continue
                    if not isinstance(codes, list): codes = [codes]
                    decoded_behaviors = [CODE_TO_BEHAVIOR.get(code, f"未知編碼({code})") for code in codes if code]
                    hl["behavior_category"] = decoded_behaviors
            
            if "sequence_summary" not in analysis_result:
                analysis_result["sequence_summary"] = "已設定為精簡模式，此欄位由本地生成。"

            return analysis_result

        except json.JSONDecodeError:
            # 如果通過了上面的檢查，但仍然解析失敗，表示模型可能產生了格式錯誤的文字
            print(f"    錯誤：無法解析JSON (第 {attempt+1} 次嘗試)。完成原因: '{finish_reason}'。")
            # 【關鍵】打印完整的原始回傳內容，以便我們看到問題所在
            print(f"    --- 模型原始回應 ---")
            print(raw_content)
            print(f"    ----------------------")
            time.sleep(5)
        except RateLimitError:
            print(f"  警告：API速率限制，等待 {API_RETRY_DELAY_SECONDS}s 後重試...");
            time.sleep(API_RETRY_DELAY_SECONDS)
        except APIError as e:
            wait_time = 10 + attempt * 5 
            print(f"  錯誤：API錯誤 (第 {attempt+1} 次嘗試): {e}. 將在 {wait_time} 秒後重試...");
            time.sleep(wait_time)
        except Exception as e:
            wait_time = 5 + attempt * 5
            print(f"  錯誤：未知錯誤 (第 {attempt+1} 次嘗試): {e}. 將在 {wait_time} 秒後重試...");
            time.sleep(wait_time)
    
    print(f"  錯誤：圖片序列分析在多次重試後失敗。")
    return default_error_result_structure()

def generate_personalized_summary_notes(student_id, overall_stats, non_task_highlights, openai_client):
    if not openai_client: return {"error": "OpenAI client not available"}
    # ... (此函數內部的 Prompt 內容完全不需要修改) ...
    stats_summary_for_ai = "\n".join([f"- {s['behavior_category']}: {s['percentage']:.1f}% ({s['count']}次)" for s in overall_stats[:7]]) # 確保百分比格式
    non_task_prompt_part = "該生在本堂課中，未觀察到明顯或頻繁的非任務相關行為。"
    if non_task_highlights:
        highlights_str = "\n".join([f"  - 圖 '{img_fn}' (~{ts}): '{beh}' (判斷依據: {desc})" for img_fn, ts, beh, desc in non_task_highlights[:3]])
        if highlights_str: non_task_prompt_part = f"在本堂課中，觀察到一些非任務相關行為，例如：\n{highlights_str}\n這可能影響了學習專注度。"
    prompt_for_summary = f"""
    「你是一位專業且富有同理心的學習行為教練。你的目標不是批評，而是透過客觀數據，引導學生發現自己的學習模式，並提供能立即實踐的策略，以激發他們『自我反思』的動力。」

    **任務：** 為學生「{student_id}」撰寫一份「AI學習夥伴的觀察與建議」。

    **學生的課堂行為數據：**
    *   **主要行為分佈:**
    {stats_summary_for_ai}
    *   **值得注意的行為片段:**
    {non_task_prompt_part}

    **撰寫指引與 JSON 格式要求：**
    請嚴格遵循以下指引，產生一個結構完整的 JSON 物件。

    *   **`greeting` (問候語):**
        *   用親切、個人化的方式稱呼學生，例如：「嗨，{student_id} 同學，一起來看看這次課堂的學習足跡吧！」

    *   **`positive_feedback` (亮點觀察):**
        *   **必須**從數據中找出最值得肯定的行為（例如「目視教師」或「筆記」佔比最高），並給予具體、真誠的讚美。
        *   **範例**：「我發現你在這堂課有超過一半的時間都在『目視教師』，這代表你非常努力地跟上老師的節奏，非常棒！」

    *   **`observation_points_summary` (行為模式提醒):**
        *   客觀、中性地指出一個或兩個最主要的、可能影響學習的行為模式。避免使用負面詞彙。
        *   **範例**：「數據也顯示，大約有 15% 的時間出現了『玩弄物品』或『目視他處』的狀況，這些時刻可能讓我們不小心錯過了一些重點喔。」

    *   **`reflection_points` (反思引導提問):**
        *   **【此項最為關鍵】** 根據前面的觀察點，設計 2-3 個**開放式問題**，引導學生思考行為背後的原因，而不是直接給答案。
        *   **問題範例 1**：「我們可以一起回想看看，當出現『玩弄物品』的時候，通常是在課程的哪個階段呢？是覺得內容太簡單、太難，還是剛好有點疲倦了呢？」
        *   **問題範例 2**：「當視線看向其他地方時，是想到了什麼有趣的事，還是被教室裡的其他動靜吸引了呢？了解這些原因，能幫助我們找到最適合自己的專注方法。」

    *   **`suggestions` (可實踐的小建議):**
        *   提供 1-2 個**具體、微小、且容易執行**的行動建議。不要說「要專心」，而是給出方法。
        *   **建議範例 1**：「下次當你發現自己開始無意識地轉筆時，可以試著把它輕輕放下，然後做一個深呼吸，再重新將目光移回老師或課本上。」
        *   **建議範例 2**：「如果感覺到疲倦或分心，可以試試看『筆記專注法』：在筆記本上寫下老師說的任何一個關鍵字，這個小動作能幫助我們的大腦重新連線！」

    *   **`encouragement` (鼓勵與結語):**
        *   用一句溫暖、有力的話作結，強調這份報告是幫助他成長的工具。
        *   **範例**：「每一次的觀察都是為了讓我們更了解自己。相信你透過這些小小的調整，一定能發揮出自己最大的潛力，加油！」
    """
    retry_attempts = 2
    for attempt in range(retry_attempts):
        try:
            print(f"  正在為學生 {student_id} 使用 {SUMMARY_DEPLOYMENT_NAME} 生成個性化總結...") # 修改了日誌輸出
            response = client.chat.completions.create(
                # ==========================================================
                # ↓↓↓ 【關鍵修改】使用我們新設定的經濟型模型 ↓↓↓
                # ==========================================================
                model=SUMMARY_DEPLOYMENT_NAME, # 這裡會自動使用 "gpt-4.1"
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": "你是一位富有同理心和洞察力的教育顧問。"},
                          {"role": "user", "content": prompt_for_summary}],
                # --- 【還原 gpt-4.1 參數】 ---
                max_tokens=MAX_TOKENS_SUMMARY_COMPLETION, 
                temperature=0.7        
                
            )
            summary_data = json.loads(response.choices[0].message.content)
            expected_keys = ["greeting", "positive_feedback", "observation_points_summary", "reflection_points", "suggestions", "encouragement"]
            if all(key in summary_data for key in expected_keys): 
                print(f"  成功為學生 {student_id} 生成個性化總結。")
                return summary_data
            else: 
                print(f"    警告：個性化總結JSON缺少鍵。返回: {summary_data}")
                return {key: summary_data.get(key, f"AI未能生成 ({key})") for key in expected_keys}
        except Exception as e: 
            print(f"  生成個性化總結錯誤 ({attempt + 1}): {e}")
            time.sleep(API_RETRY_DELAY_SECONDS if isinstance(e, RateLimitError) else 5)
        if attempt == retry_attempts - 1: 
            print(f"  錯誤：無法為學生 {student_id} 生成個性化總結。")
            return {"greeting": f"親愛的 {student_id},", "positive_feedback": "總結生成遇到問題。", "observation_points_summary": "請參考統計數據。", "reflection_points": "未能生成。", "suggestions": "請自行評估。", "encouragement": "加油！"}
    return {}

def find_closest_image_path(representative_timestamp, sorted_photo_list, max_time_diff_seconds=5):
    """
    一個可重用的輔助函數，用二分查找法在排序好的照片列表中找到時間最接近的照片路徑。
    """
    if not sorted_photo_list:
        return None

    all_timestamps = [item[0] for item in sorted_photo_list]
    # bisect_left 找到應該插入的位置
    insertion_point = bisect.bisect_left(all_timestamps, representative_timestamp)

    closest_path = None
    min_diff = datetime.timedelta.max

    # 只檢查插入點及其前後的幾個候選照片，效率極高
    # 檢查範圍設為 insertion_point-2 到 insertion_point+2 以增加容錯
    start_index = max(0, insertion_point - 2)
    end_index = min(len(sorted_photo_list), insertion_point + 2)

    for i in range(start_index, end_index):
        candidate_ts, candidate_path = sorted_photo_list[i]
        diff = abs(candidate_ts - representative_timestamp)
        if diff < min_diff:
            min_diff = diff
            closest_path = candidate_path

    # 只有在時間差在容許範圍內才返回路徑
    if closest_path and min_diff.total_seconds() <= max_time_diff_seconds:
        return closest_path
    
    return None

def find_closest_position(representative_timestamp, sorted_position_list, max_time_diff_seconds=10):
    """
    用二分查找法在排序好的位置列表中找到時間最接近的老師位置。
    """
    if not sorted_position_list:
        return "未知"

    all_timestamps = [item[0] for item in sorted_position_list]
    insertion_point = bisect.bisect_left(all_timestamps, representative_timestamp)

    closest_position = "未知"
    min_diff = datetime.timedelta.max

    start_index = max(0, insertion_point - 2)
    end_index = min(len(sorted_position_list), insertion_point + 2)

    for i in range(start_index, end_index):
        candidate_ts, candidate_pos = sorted_position_list[i]
        diff = abs(candidate_ts - representative_timestamp)
        if diff < min_diff:
            min_diff = diff
            closest_position = candidate_pos
    
    if min_diff.total_seconds() <= max_time_diff_seconds:
        return closest_position
    
    return "未知"

def process_single_batch(batch_idx, image_batch_info, teacher_positions_data, sorted_classroom_photos, classroom_states, client, student_id, student_position):
    """
    【升級版】處理單一圖片批次，使用老師位置和預處理好的課堂狀態。
    """
    batch_student_paths = [info["path"] for info in image_batch_info]
    batch_image_filenames = [info["filename"] for info in image_batch_info]
    
    representative_timestamp = image_batch_info[0]["timestamp_obj"]

    # --- 為當前批次查找所有情境數據 ---
    teacher_position_text = find_closest_position(representative_timestamp, teacher_positions_data)
    classroom_view_path = find_closest_image_path(representative_timestamp, sorted_classroom_photos)
    # 【關鍵修正】使用正確的變數名稱 classroom_states
    classroom_state = find_state_for_timestamp(representative_timestamp, classroom_states)

    # --- 呼叫【新版】分析函數 ---
    sequence_analysis_data = analyze_student_behavior_from_images_sequence(
        student_image_paths=batch_student_paths,
        teacher_position_text=teacher_position_text,
        classroom_view_image_path=classroom_view_path,
        classroom_state=classroom_state, # <-- 傳入狀態標籤
        image_filenames_batch=batch_image_filenames,
        openai_client=client,
        student_id=student_id,
        student_position=student_position
    )

    return {
        "batch_index": batch_idx,
        "image_batch_info": image_batch_info,
        "matched_teacher_position_text": teacher_position_text,
        "matched_classroom_view_image": os.path.basename(classroom_view_path) if classroom_view_path else None,
        "matched_classroom_state": classroom_state,
        "analysis": sequence_analysis_data
    }

def main():
    print("--- 學生課堂學習行為分析報告生成 (JSON) v5.0 - 成本優化版 ---")
    print(f"視覺模型: {VISION_DEPLOYMENT_NAME}, 文本模型: {VISION_DEPLOYMENT_NAME}, 總結模型: {VISION_DEPLOYMENT_NAME}")
    print(f"圖片批次大小: {IMAGES_PER_API_CALL}, 圖片取樣率: 1/{SAMPLING_RATE}, 圖片解析度: {IMAGE_DETAIL_LEVEL}")
    print("-" * 40)

    # --- 使用者輸入部分 ---
    student_id = get_valid_input("請輸入學生姓名 (可中文): ")
    student_number = get_valid_input("請輸入學生座號 (例如: 1): ") 
    student_images_folder = get_valid_folder_path(f"請輸入 '{student_id}' 的個人影像資料夾: ")
    print("\n--- 請輸入課堂情境資訊 ---")
    student_position = get_valid_input("請輸入學生座位 (例如: '第3排中間', '第1排左側'): ")
    print("-" * 50)

    # --- 準備輸出資料夾 ---
    safe_student_id_for_folder = re.sub(r'[\\/*?:"<>|]',"", student_id.replace(" ", "_"))
    student_specific_folder_base = os.path.join(JSON_OUTPUT_FOLDER, safe_student_id_for_folder)
    os.makedirs(student_specific_folder_base, exist_ok=True)

    # --- 準備學生照片資料 ---
    image_files_with_timestamps = []
    valid_extensions = ('.png', '.jpg', '.jpeg', '.webp')
    print(f"正在掃描學生個人照片資料夾: {student_images_folder}...")
    for filename in os.listdir(student_images_folder):
        if filename.lower().endswith(valid_extensions):
            timestamp = get_timestamp_from_filename(filename)
            if timestamp:
                image_files_with_timestamps.append({
                    "path": os.path.join(student_images_folder, filename), "filename": filename,
                    "timestamp_obj": timestamp, # <-- 直接使用原始時間戳，不做任何校準
                    "timestamp_str": str(timestamp).split('.')[0]
                })
    image_files_with_timestamps.sort(key=lambda x: x["timestamp_obj"])
    
    # ==========================================================
    # ↓↓↓ 【新增】圖片取樣以降低成本 ↓↓↓
    # ==========================================================
    if SAMPLING_RATE > 1:
        original_count = len(image_files_with_timestamps)
        image_files_with_timestamps = image_files_with_timestamps[::SAMPLING_RATE]
        print(f"已執行圖片取樣：從 {original_count} 張原始照片中，每 {SAMPLING_RATE} 張取 1 張，共 {len(image_files_with_timestamps)} 張照片將被分析。")
    else:
        print(f"成功找到 {len(image_files_with_timestamps)} 張學生個人照片 (未取樣)。")

    if not image_files_with_timestamps: 
        print("錯誤：學生資料夾中未找到有效時間格式的圖片，或取樣後為空。")
        return
    # ==========================================================

    # ... 後續的程式碼，從 `load_teacher_positions` 開始，到整個 `main` 函數結束，
    # 都不需要再做任何修改。您可以直接使用您原有的版本。
    # 我將剩餘部分貼在下方以保證完整性。
    
    def load_teacher_positions(json_path, start_time_offset_str=""): # 新增 start_time_offset_str 參數
        if not json_path or not os.path.isfile(json_path):
            print(f"提示：未提供或找不到老師位置 JSON 檔案 ({json_path})。將不使用老師位置情境。")
            return []
        start_offset = parse_time_offset(start_time_offset_str)
        print(f"正在讀取老師位置數據: {json_path}...")
        try:
            with open(json_path, 'r', encoding='utf-8') as f: data = json.load(f)
            position_map, original_count = [], len(data)
            for item in data:
                try:
                    h, m, s = map(int, item['timestamp'].split(':'))
                    td = datetime.timedelta(hours=h, minutes=m, seconds=s)
                    if start_offset and td < start_offset: continue
                    position_map.append((td, item['position']))
                except (ValueError, KeyError): continue
            position_map.sort(key=lambda x: x[0])
            if start_offset: print(f"成功加載老師位置數據。根據起始時間 '{start_time_offset_str}' 進行過濾，保留 {len(position_map)} 筆 (原 {original_count} 筆)。")
            else: print(f"成功加載並索引了 {len(position_map)} 筆老師位置數據。")
            return position_map
        except Exception as e:
            print(f"錯誤：讀取或解析老師位置 JSON 時發生問題: {e}"); return []

    def load_and_sort_photos(folder_path, photo_type_name, start_time_offset_str=""): # 新增 start_time_offset_str 參數
        if not folder_path or not os.path.isdir(folder_path):
            print(f"提示：未提供或找不到 {photo_type_name} 照片資料夾 ({folder_path})。"); return []
        start_offset = parse_time_offset(start_time_offset_str)
        print(f"正在掃描 {photo_type_name} 照片資料夾: {folder_path}...")
        photo_list, original_count = [], 0
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                original_count += 1
                timestamp = get_timestamp_from_filename(filename)
                if timestamp:
                    if start_offset and timestamp < start_offset: continue
                    photo_list.append((timestamp, os.path.join(folder_path, filename)))
        if photo_list:
            sorted_photos = sorted(photo_list)
            if start_offset: print(f"成功加載 {photo_type_name} 照片。根據起始時間 '{start_time_offset_str}' 進行過濾，保留 {len(sorted_photos)} 張 (原 {original_count} 張有效格式照片)。")
            else: print(f"成功加載並索引了 {len(sorted_photos)} 張 {photo_type_name} 照片。")
            return sorted_photos
        else:
            print(f"警告：在 {photo_type_name} 照片資料夾 '{folder_path}' 中未找到符合條件的圖片。"); return []

    teacher_positions_data = load_teacher_positions(TEACHER_POSITION_JSON, TEACHER_JSON_START_TIME_OFFSET)
    sorted_classroom_photos = load_and_sort_photos(CLASSROOM_IMAGES_FOLDER, "班級整體", CLASSROOM_START_TIME_OFFSET)
    classroom_states = load_classroom_states(CLASSROOM_STATE_JSON_PATH)

    image_batches = [image_files_with_timestamps[i:i + IMAGES_PER_API_CALL] for i in range(0, len(image_files_with_timestamps), IMAGES_PER_API_CALL)]
    print(f"學生圖片將被分為 {len(image_batches)} 個批次進行分析。")

    MAX_WORKERS = 4
    print(f"將使用最多 {MAX_WORKERS} 個執行緒進行平行分析...")
    all_results_from_threads = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_batch_idx = { 
            executor.submit(process_single_batch, idx, batch_info, teacher_positions_data, sorted_classroom_photos, classroom_states, client, student_id, student_position): idx 
            for idx, batch_info in enumerate(image_batches) 
        }
        for future in tqdm(as_completed(future_to_batch_idx), total=len(image_batches), desc=f"分析學生 {student_id} 的圖片批次"):
            try:
                result = future.result()
                all_results_from_threads.append(result)
            except Exception as exc:
                batch_idx = future_to_batch_idx[future]
                print(f'\n批次 {batch_idx + 1} 執行時產生錯誤: {exc}')
                all_results_from_threads.append({"batch_index": batch_idx, "error": str(exc)})

    all_results_from_threads.sort(key=lambda x: x['batch_index'])
    print("\n所有批次分析完成，開始匯總數據...")
    all_sequence_analysis_results = []
    overall_behavior_summary = { "total_images_processed_in_batches": 0, "behavior_counts": Counter(), "behavior_confidence_sum": defaultdict(float), "non_task_behavior_examples_for_summary": [] }
    for result in tqdm(all_results_from_threads, desc="匯總分析結果"):
        if "error" in result:
            all_sequence_analysis_results.append({"batch_index": result['batch_index'] + 1, "analysis": default_error_result_structure()})
            continue
        image_batch_info, sequence_analysis_data = result["image_batch_info"], result["analysis"]
        batch_image_filenames = [info["filename"] for info in image_batch_info]
        all_sequence_analysis_results.append({ "batch_index": result['batch_index'] + 1, "image_filenames_in_batch": batch_image_filenames, "matched_teacher_position_text": result.get("matched_teacher_position_text"), "matched_classroom_view_image": result.get("matched_classroom_view_image"), "analysis": sequence_analysis_data })
        if "error" not in sequence_analysis_data:
            overall_behavior_summary["total_images_processed_in_batches"] += len(batch_image_filenames)
            if "per_image_highlights" in sequence_analysis_data:
                for hl_item in sequence_analysis_data.get("per_image_highlights", []):
                    
                    behavior_list_original, conf = hl_item.get("behavior_category"), hl_item.get("confidence", 0.0)
                    
                    # --- ↓↓↓ 【核心修改】低信度與空行為的過濾與覆寫邏輯 ↓↓↓ ---
                    
                    UNKNOWN_LABEL = "被遮擋/無法判斷"
                    
                    # 複製一份原始行為列表，用於後續的非任務行為判斷
                    behavior_list_for_stats = behavior_list_original
                    
                    # 判斷條件 1: 信度是否低於設定的閾值
                    is_low_confidence = conf is not None and conf < BEHAVIOR_CONFIDENCE_THRESHOLD_FILTER
                    # 判斷條件 2: AI 是否沒有返回任何有效的行為標籤
                    is_empty_behavior = not behavior_list_original

                    if is_low_confidence or is_empty_behavior:
                        # 如果觸發任一條件，則用於統計的行為列表將被強制覆寫
                        behavior_list_for_stats = [UNKNOWN_LABEL]
                        
                    # --- ↑↑↑ 過濾邏輯結束 ↑↑↑ ---

                    # 確保用於統計的行為列表是 list 格式
                    if not isinstance(behavior_list_for_stats, list):
                        behavior_list_for_stats = [behavior_list_for_stats]
                    
                    # 遍歷列表中的每一個行為進行統計 (此處使用的是可能被覆寫過的 behavior_list_for_stats)
                    for cat in behavior_list_for_stats:
                        overall_behavior_summary["behavior_counts"][cat] += 1
                        overall_behavior_summary["behavior_confidence_sum"][cat] += float(conf if conf is not None else 0.0) # 確保 conf 是浮點數
                    
                    # 處理非任務行為範例（注意：此處我們使用未經過濾的原始行為 `behavior_list_original`）
                    # 這樣可以確保即使一個 "玩筆" 行為因信度低被歸入 "無法判斷"，我們依然能捕捉到這個 "意圖"
                    if behavior_list_original and isinstance(behavior_list_original, list):
                        primary_behavior_for_example = behavior_list_original[0]
                        non_task_keywords = [
                            "玩弄手部/文具", "觸摸臉部", "觸摸頭髮",
                            "目視他處", "趴睡", "喝水", "飲食"
                        ]
                        if primary_behavior_for_example in non_task_keywords and len(overall_behavior_summary["non_task_behavior_examples_for_summary"]) < 3:
                            try:
                                student_img_idx = hl_item.get("image_index_in_sequence", -1)
                                if 0 <= student_img_idx < len(batch_image_filenames):
                                    hl_filename = batch_image_filenames[student_img_idx]
                                    hl_timestamp = next((info["timestamp_str"] for info in image_batch_info if info["filename"] == hl_filename), "未知時間")
                                    context_desc = hl_item.get("context_description", "")
                                    
                                    # 即使此行為在統計上被歸為"無法判斷"，我們仍記錄其原始識別結果以供參考
                                    overall_behavior_summary["non_task_behavior_examples_for_summary"].append( 
                                        (hl_filename, hl_timestamp, ", ".join(behavior_list_original), context_desc) 
                                    )
                            except Exception as e_idx: print(f"提取非任務示例時出錯: {e_idx}")
    
    overall_behavior_stats_list, total_highlight_instances = [], sum(overall_behavior_summary["behavior_counts"].values())
    valence_summary = {"正向": 0, "負向": 0, "中性": 0}
    for behavior, count in overall_behavior_summary["behavior_counts"].items():
        percentage, avg_confidence = (count / total_highlight_instances * 100) if total_highlight_instances > 0 else 0, (overall_behavior_summary["behavior_confidence_sum"][behavior] / count) if count > 0 else 0
        valence = LABEL_TO_VALENCE.get(behavior, "未分類")
        if valence in valence_summary: valence_summary[valence] += count
        overall_behavior_stats_list.append({ "behavior_category": behavior, "valence": valence, "count": count, "percentage": round(percentage, 1), "average_confidence": round(avg_confidence, 2) })
    overall_behavior_stats_list.sort(key=lambda x: x["count"], reverse=True)
    personalized_notes = generate_personalized_summary_notes(student_id, overall_behavior_stats_list, overall_behavior_summary["non_task_behavior_examples_for_summary"], client)
    behavior_to_images_map = defaultdict(list)
    for result in all_results_from_threads:
        if "error" in result or "analysis" not in result or "error" in result["analysis"]: continue
        sequence_analysis, image_batch_info, filenames_in_batch = result["analysis"], result.get("image_batch_info", []), [info.get("filename") for info in result.get("image_batch_info", [])]
        if "per_image_highlights" in sequence_analysis and isinstance(sequence_analysis["per_image_highlights"], list):
            for highlight in sequence_analysis["per_image_highlights"]:
                # 【修改點 4】修改索引建立邏輯以處理行為列表
                behavior_list, image_index = highlight.get("behavior_category"), highlight.get("image_index_in_sequence")
                
                if not behavior_list or not isinstance(image_index, int) or not (0 <= image_index < len(filenames_in_batch)):
                    continue

                if not isinstance(behavior_list, list):
                    behavior_list = [behavior_list]
                
                image_filename = filenames_in_batch[image_index]
                if image_filename:
                    # 為列表中的每一個行為都建立索引
                    for behavior_category in behavior_list:
                        if image_filename not in behavior_to_images_map[behavior_category]:
                            behavior_to_images_map[behavior_category].append(image_filename)
    for behavior in behavior_to_images_map: behavior_to_images_map[behavior].sort()
    print("行為索引建立完成。")

    total_classified_instances = sum(valence_summary.values())
    valence_summary_with_percentage = { valence: { "count": count, "percentage": round((count / total_classified_instances * 100), 1) if total_classified_instances > 0 else 0 } for valence, count in valence_summary.items() }

    final_json_output = {
        "report_metadata": {
            "student_id": student_id, "student_number": student_number, "report_generation_time": "08/17", "student_image_source_folder": "英文",
            "teacher_position_source_json": os.path.basename(TEACHER_POSITION_JSON) if TEACHER_POSITION_JSON and os.path.isfile(TEACHER_POSITION_JSON) else "N/A",
            "classroom_view_source_folder": os.path.basename(CLASSROOM_IMAGES_FOLDER) if CLASSROOM_IMAGES_FOLDER and os.path.isdir(CLASSROOM_IMAGES_FOLDER) else "N/A",            "classroom_context": { "student_position": student_position },
            "analysis_settings": {
                "vision_model": VISION_DEPLOYMENT_NAME, "text_model": TEXT_DEPLOYMENT_NAME, "images_per_batch": IMAGES_PER_API_CALL, "context_images_per_batch_desc": "動態匹配老師和班級照片各一張", "confidence_threshold": BEHAVIOR_CONFIDENCE_THRESHOLD_FILTER,
                "cost_optimization": { "sampling_rate": SAMPLING_RATE, "image_detail": IMAGE_DETAIL_LEVEL } # 新增成本優化資訊
            }
        },
        "overall_summary": {
            "total_images_found": len(image_files_with_timestamps) * SAMPLING_RATE if SAMPLING_RATE > 1 else len(image_files_with_timestamps), # 顯示原始數量
            "total_images_analyzed": overall_behavior_summary["total_images_processed_in_batches"],
            "total_batches": len(image_batches), "valence_summary": valence_summary_with_percentage, "behavior_statistics": overall_behavior_stats_list, "behavior_to_images_index": behavior_to_images_map, "ai_summary_notes": personalized_notes
        },
        "detailed_sequence_analysis": all_sequence_analysis_results
    }

    current_timestamp_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_student_id_for_filename = re.sub(r'[\\/*?:"<>|]',"", student_id.replace(" ", "_"))
    json_filename = JSON_FILENAME_TEMPLATE.format(student_id=safe_student_id_for_filename, timestamp=current_timestamp_str)
    json_filepath = os.path.join(student_specific_folder_base, json_filename)
    try:
        with open(json_filepath, 'w', encoding='utf-8') as f: json.dump(final_json_output, f, ensure_ascii=False, indent=4)
        print(f"\n✅ 學生 '{student_id}' 的序列行為分析報告已成功儲存至: {json_filepath}")
    except Exception as e: print(f"❌ 錯誤：儲存 JSON 檔案時發生問題：{e}")
    
    print("\n--- 處理完成 ---")

if __name__ == "__main__":
    main()

### test

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import base64
import json
import datetime
import time
from collections import Counter, defaultdict
from openai import AzureOpenAI, APIError, RateLimitError, AuthenticationError 
from PIL import Image, UnidentifiedImageError
import io
from tqdm import tqdm
import math
from dotenv import load_dotenv 
from concurrent.futures import ThreadPoolExecutor, as_completed
import bisect

# --- Configuration ---


# 班級整體照片資料夾路徑 
CLASSROOM_IMAGES_FOLDER = r'C:\Users\User\Desktop\test\student_full_classroom\0810_english_class'

# 【新】老師視角照片資料夾路徑 (從學生視角拍攝老師在黑板前的照片) - 可選
TEACHER_POSITION_JSON = r'C:\Users\User\Desktop\test\teacher_position\0810_position.json'

#課堂情況json
CLASSROOM_STATE_JSON_PATH = r'C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0810\老師上課\TXT\0810.json'

# 【新】時間軸對齊設定 (格式: "HH:MM:SS")
CLASSROOM_START_TIME_OFFSET = "00:00:00"  # 班級照片的起始時間，設定為空字串 "" 表示不篩選
TEACHER_JSON_START_TIME_OFFSET ="00:00:00" # 老師位置資料的起始時間，設定為空字串 "" 表示不篩選

# ... (您原有的其他設定) ...

JSON_OUTPUT_FOLDER = "SynologyDrive\json_behavior"


JSON_FILENAME_TEMPLATE = "student_{student_id}_behavior_report_{timestamp}.json"
SAMPLING_RATE = 3  
IMAGE_DETAIL_LEVEL = "low" # 圖片解析度 ('low' 或 'high')，low 可大幅降低成本
MAX_TOKENS_VISION_COMPLETION = 3500
MAX_TOKENS_SUMMARY_COMPLETION = 1000 # 個性化總結的 token
BEHAVIOR_CONFIDENCE_THRESHOLD_FILTER = 0.97 # 行為信度過濾閾值 (例如 70%)
IMAGES_PER_API_CALL = 6 # 一次API調用處理的圖片數量
BEHAVIOR_SIMILARITY_CONFIDENCE_THRESHOLD = 0.15 # 過濾相似連續行為的信度差異閾值
API_RETRY_DELAY_SECONDS = 10

# ==========================================================

# --- 標準行為分類與定義 (核心辭典，來自圖一) ---
STANDARD_BEHAVIOR_CATEGORIES = {
    "視線": [
        {"label": "目視教師", "definition": "學生的頭部與視線明確聚焦於教室【前方】的【老師所在】區域，且其頭部旋轉角度處於一個【合理的朝前弧度】內（通常不超過45度）。**【絕對排除條款】**: 任何導致學生視線與其身體朝向構成【接近90度或更大角度】的頭部大幅度轉動，【絕對不允許】被標註為『目視教師』。這種姿態應被優先考慮為『目視同學』(`V_CLS`)或『目視他處』(`V_ELS`)。"},
        {"label": "目視黑板", "definition": "學生的頭部與視線明確聚焦於教室【前方】的【非老師所在】的黑板或螢幕區域。當老師位置未知，或學生視線明確未朝向老師時，這是面向前方的預設專注行為。"}, 
        {"label": "目視書本/筆記", "definition": "學生的頭部與視線向量【主要朝下】，且頭部的水平旋轉角度【沒有明顯偏離】其身體所朝向的【個人桌面工作區軸線】。此標籤捕捉的是在個人學習材料上的視覺專注狀態。**【嚴格邊界】**: 一旦頭部/視線明確地、持續地轉向側面（例如，足以與鄰座同學進行眼神交流），即使視線仍然略微朝下，也【必須優先考慮】標註為『目視同學』。**【注意】**：如果學生同時在進行『做筆記』，根據『動作優先』原則，你應將『做筆記』作為主要標籤。"},
        {"label": "目視同學", "definition": "學生的【主要身體朝向與視線】明確脫離前方或個人桌面，轉向側面或後方的同學。**【最高優先級的情境標籤】**: 一旦觀察到這種明確的身體轉向，此標籤的優先級就高於大多數獨立的個人動作。它涵蓋了從純粹的視覺交流到【涉及物件的協作行為】（如共同看書、討論問題）。**【核心情境判斷規則】**: 此行為的性質根據【課堂狀態】決定..."},
        {"label": "目視他處", "definition": "【備用排除性標籤】。當你已確認學生的視線【不】符合『目視教師』、『目視黑板』、『目視書本/筆記』或『目視同學』的任何一項明確定義時，才使用此標籤。它捕捉的是失去焦點的狀態，例如：【抬頭看天花板】、【轉頭看沒有同學及老師的地方】。"}     
    ],
    "肢體(手部)": [
        {"label": "做筆記", "definition": "【核心證據：觀察到一個動態的書寫過程】。你必須能明確看到學生手持筆，且筆尖正在紙張上進行【有意義的移動或書寫/繪製動作】。**【最高優先級】**：這是一個高優先級的動作標籤。**【嚴格排除條款】**：以下情況【絕對不允許】標註為『做筆記』：(1) **靜態持筆**：僅僅手持筆，或筆尖靜止停留在紙上，應標註為『目視書本/筆記』 (`V_BOK`)。(2) **動作中斷**：當學生手部的主要動作變為其他行為（如『觸摸頭髮』、『托腮』、『與同學互動』），即使手中仍持有筆，也必須以該【瞬時動作】為主要標籤。"},
        {"label": "翻書", "definition": "學生手部正在主動翻閱、移動書本。"},
        {"label": "觸摸臉部", "definition": "學生【非支撐性地】用手短暫觸碰或摩擦自己的臉部、鼻子、嘴巴或眼睛。此行為區別於『托腮』的持續性支撐動作。"},
        {"label": "觸摸頭髮", "definition": "學生用手觸摸、撥弄或整理自己的頭髮。注意：即使手臂抬得較高，只要手部的主要動作是與頭髮互動，就【必須】使用此標籤，而不是『舉手』類標籤。"}
    ],
    "身體姿態": [
        {"label": "坐姿直立", "definition": "學生上半身軀幹基本垂直於地面，或輕微前傾。"},
        {"label": "身體前傾", "definition": "學生上半身軀幹明顯向前彎曲，靠近桌面。此行為描述的是一種【清醒狀態下】的姿態。如果學生頭部接觸桌面或手臂，應【優先使用】『趴睡』標籤。"},
        {"label": "身體後靠", "definition": "學生背部倚靠在椅背上。"},
        {"label": "低頭(非學習)", "definition": "【極其嚴格的排除性標籤】。僅在你能夠【極度確信地】觀察到以下【全部】條件時才可使用：1. 學生頭部明顯低垂。2. 其視線【明確沒有】朝向任何學習材料（例如，看向地面、自己的懷中或空無一物的桌面）。3. 其手部沒有在進行任何學習相關操作。"},
        {"label": "趴睡", "definition": "學生將頭部【枕於】手臂或桌面上，呈現明確的休息或睡眠狀態。**【最高優先級】**：只要觀察到頭部接觸桌面或手臂的休息姿態，此標籤的優先級【高於】所有其他學習相關標籤（如『做筆記』、『目視書本』）。"},
        {"label": "托腮", "definition": "【核心定義：手部對頭部提供持續性支撐】。學生使用一隻或兩隻手的手掌、拳頭或手臂，支撐其下巴、臉頰或頭部的重量。這是一個純粹的物理姿態描述，不包含任何意圖推斷。"}
    ],
    "互動": [
        {"label": "主動舉手", "definition": "學生舉起一隻手，意圖提問或回答問題。**【三大核心物理證據，必須同時滿足】**：(1) 手臂向上伸展，手部明顯高於肩膀。(2) 手掌形態為張開朝前或中性放鬆，【嚴禁】手指指向特定方向或揮舞。(3) 該動作具有一定的持續性（非瞬間劃過）。**【情境觸發】**：此行為最常發生在【老師單向授課】的狀態下（如『文法/句型講解』、『閱讀/文章分析』），代表學生的自發性提問或補充。**【絕對排除】**：任何手部接觸頭部、與同學互動的手勢、指向性的動作，都【不允許】標記為此行為。"},
        {"label": "被動舉手", "definition": "學生舉手以回應老師的群體性指令（如投票、調查）。**【核心視覺證據】**：通常是多數學生同時舉手，姿態可能較為放鬆，手臂不必完全伸直。**【情境觸發】**：此行為最常發生在【老師與學生互動】的狀態下（如『課堂問答/互動』、『習題/考卷檢討』），代表學生回應老師的指令或提問。**【關鍵區分】**：此標籤的判斷【高度依賴】課堂情境和群體性動作。如果情境不匹配，應避免使用此標籤。"}
    ],
    "其他狀態": [
        {"label": "喝水", "definition": "【核心證據：清晰可見的容器】。只有當你能夠【明確地】看到學生手持水瓶、杯子或其他容器，並將其送至嘴邊時，才可以使用此標籤。**【嚴格排除】**: 任何僅有低頭姿態、手部靠近臉部但【沒有可見容器】的場景，都【嚴禁】標註為此行為。在此情況下，應優先考慮『目視書本/筆記』(`V_BOK`)或『趴睡』(`P_SLP`)。"},
        {"label": "飲食", "definition": "學生正在食用固體食物。"},
        {"label": "玩弄手部／文具", "definition": "學生手部在進行與學習無關的重複性小動作，例如玩手指、轉筆、玩弄橡皮擦等"},
        # {"label": "整理書包", "definition": "學生正在整理書包、桌面文具。"},
        {"label": "被遮擋/無法判斷", "definition": "【最終備用標籤】。因遮擋、模糊或角度問題，無法清晰識別學生的主要行為時，【必須】使用此標籤。"}
    ]
}
# 自動從新結構生成有效的標籤列表
VALID_BEHAVIOR_LABELS = [item['label'] for category in STANDARD_BEHAVIOR_CATEGORIES.values() for item in category]

BEHAVIOR_CODES = {
    # 視線 (不變)
    "V_TCH": "目視教師", "V_BRD": "目視黑板", "V_BOK": "目視書本/筆記",
    "V_CLS": "目視同學", "V_ELS": "目視他處",
    
    # 肢體(手部) (移除 H_FLIP)
    "H_NOT": "做筆記",
    "H_PLAY_HW": "玩弄手部/文具",
    "H_TOUCH_F": "觸摸臉部",
    "H_TOUCH_H": "觸摸頭髮",
    
    # 身體姿態 (新增 P_THK - P for Posture, THK for Thinking)
    "P_STR": "坐姿直立", "P_LEAN": "身體前傾", "P_BACK": "身體後靠",
    "P_DWN": "低頭(非學習)", "P_SLP": "趴睡",
    "P_THK": "托腮", # <--- 新增
    
    # 互動 (將 I_HND 拆分為主動/被動)
    "I_HND_A": "主動舉手", # A for Active
    "I_HND_P": "被動舉手", # P for Passive
    
    # 其他狀態 (不變)
    "O_DRK_W": "喝水",
    "O_EAT_S": "飲食",
    # "O_TDY": "整理個人物品",
    "O_UNK": "被遮擋/無法判斷"
}

# 自動生成反向查找字典，用於本地解碼
CODE_TO_BEHAVIOR = {code: label for code, label in BEHAVIOR_CODES.items()}
BEHAVIOR_TO_CODE = {label: code for code, label in BEHAVIOR_CODES.items()}

# AI返回標籤到標準標籤的映射規則 (與新標籤對齊)
BEHAVIOR_MAPPING_RULES = {
    "目視桌面/教材": "目視書本", "目視桌面": "目視書本", "看書": "目視書本",
    "書寫/做筆記": "筆記", "動手操作-書寫/做筆記": "筆記",
    "視覺專注-閱讀書本/講義": "目視書本",
    "視覺專注-看老師/黑板方向": "目視教師", "目視黑板/老師": "目視教師",
    "看老師": "目視教師", "看黑板": "目視黑板",
    # --- ↓↓↓ 【修改點】更新映射規則以對應新標籤 ↓↓↓ ---
    "玩弄物品": "玩弄手部/文具", # 將模糊的舊標籤對應到最可能的新標籤
    "非任務相關動作-玩弄物品(筆等)": "玩弄手部/文具",
    # 移除了"非任務相關動作-觸摸臉部/頭髮"，鼓勵AI直接使用更精確的新標籤
    # "非任務相關動作-整理物品": "整理個人物品",
    "社交互動-與同學互動": "目視同學",
    "趴睡/休息": "趴睡",
    "低頭/伏案(非睡)": "低頭",
}

BEHAVIOR_VALENCE_MAP = {
    "正向": [
        "做筆記",
        "主動舉手", # <-- 修改
        "目視教師",
        "目視黑板",
        "目視書本/筆記",
    ],
    "負向": [
        "趴睡",
        "玩弄手部/文具",
        "觸摸臉部",
        "觸摸頭髮",
        "目視他處",
        "目視同學"
    ],
    "中性": [
        "身體前傾",
        "坐姿直立",
        "身體後靠",
        "喝水",
        "飲食",       
        "玩弄手部／文具",
        "翻書",
        "低頭(非學習)",
        # "整理個人物品",     
        "被遮擋/無法判斷",
        "托腮", # <-- 新增
        "被動舉手"  # <-- 新增
    ]
}

LABEL_TO_VALENCE = {
    label: valence 
    for valence, labels in BEHAVIOR_VALENCE_MAP.items() 
    for label in labels
}

# ---------------------------
#  API 金鑰配置
# ---------------------------
load_dotenv()

AZURE_API_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
# 從環境變數讀取部署名稱
VISION_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_VISION_DEPLOYMENT")
SUMMARY_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_SUMMARY_DEPLOYMENT")

# 為了兼容，讓 TEXT_DEPLOYMENT_NAME 跟隨視覺模型 (雖然目前沒用到)
TEXT_DEPLOYMENT_NAME = VISION_DEPLOYMENT_NAME 

# 檢查必要的 Azure 配置是否存在
if not all([AZURE_API_KEY, AZURE_ENDPOINT, VISION_DEPLOYMENT_NAME, SUMMARY_DEPLOYMENT_NAME]):
    print("錯誤：缺少必要的 Azure OpenAI 環境變數。")
    print("請檢查您的 .env 檔案是否包含 AZURE_OPENAI_KEY, AZURE_ENDPOINT, AZURE_OPENAI_VISION_DEPLOYMENT, 和 AZURE_OPENAI_SUMMARY_DEPLOYMENT。")
    exit()

try:
    # ### 【修改 3】: 初始化 AzureOpenAI Client ###
    client = AzureOpenAI(
        api_key=AZURE_API_KEY,
        azure_endpoint=AZURE_ENDPOINT,
        api_version="2024-02-01"  # 使用一個穩定的 API 版本
    )
    client.models.list() # 嘗試調用一個簡單的API來驗證配置
    print("Azure OpenAI client 初始化並驗證成功。")
    print(f"  - 視覺分析將使用部署: '{VISION_DEPLOYMENT_NAME}'")
    print(f"  - 個性化總結將使用部署: '{SUMMARY_DEPLOYMENT_NAME}'")
except AuthenticationError: print("錯誤：Azure OpenAI API 金鑰或端點無效，認證失敗。"); exit()
except Exception as e: print(f"初始化 Azure OpenAI Client 時發生錯誤: {e}"); exit()

# ---------------------------
# 輔助函數
# ---------------------------
def get_behavior_sequence_analysis_system_prompt(student_position, has_teacher_context):
    """
    【v14.1 - 嚴謹推理與簡明輸出混合版】
    結合了 v14.0 的深度分析協議與 v15.0 的成本效益考量。
    核心目標：讓 AI 依據最嚴謹的規則進行思考，但只輸出最核心的推理摘要。
    """
    # --- 1. 預先生成靜態的行為編碼表 (此部分邏輯不變) ---
    behavior_table_for_prompt = "\n\n**【學習行為編碼表】**\n你 **必須** 且 **只能** 從以下列表的「編碼 (Code)」中選擇行為進行標註。...\n"
    for code, label in BEHAVIOR_CODES.items():
        definition = ""
        for category in STANDARD_BEHAVIOR_CATEGORIES.values():
            for item in category:
                if item['label'] == label:
                    definition = item['definition']
                    break
            if definition:
                break
        behavior_table_for_prompt += f"*   **`{code}`**: {label} - {definition}\n"
    behavior_table_for_prompt += "\n*如果行為因任何原因無法清晰判斷，請 **必須** 使用 **`O_UNK`** 編碼。*\n"

    # --- 2. 【核心升級】根據學生位置，動態生成【主攝影機規則】 ---
    primary_camera_rules = ""
    forward_arc_principle = """
    **準則 1A-2: 朝前弧度原則 (The Forward-Facing Arc Principle)**
    *   對教室前方的專注行為（目視教師/黑板）必須發生在一個合理的「朝前弧度」內。微小的頭部轉動是正常的視線調整。
    *   **【硬性邊界】**: 然而，一個使學生頭部與其肩膀平面構成【接近90度】的**大幅度轉頭**，是其注意力【脫離前方】的明確證據。
    *   **【執行指令】**: 一旦你觀察到這種大幅度轉頭，你**必須**將該行為的優先級賦予 `V_CLS` (目視同學)，並**嚴格禁止**將其標註為 `V_TCH` 或 `V_BRD`。
    """

    if "左" in student_position:
        primary_camera_rules = """
**準則 1A: 主要分析視角 (左側攝影機)**
*   【學生個人照片序列】來自教室前方的【左側】攝影機。
*   對於這位坐在左側的學生，【正面朝向鏡頭】僅代表他在看教室的【左前方】。為了看向【教室正前方中心】，他的頭部必須輕微地朝向他自己的【右側】轉動。
{forward_arc_principle}
"""
    elif "右" in student_position:
        primary_camera_rules = """
**準則 1A: 主要分析視角 (右側攝影機)**
*   【學生個人照片序列】來自教室前方的【右側】攝影機。
*   對於這位坐在右側的學生，【正面朝向鏡頭】僅代表他在看教室的【右前方】。為了看向【教室正前方中心】，他的頭部必須輕微地朝向他自己的【左側】轉動。
{forward_arc_principle}
"""
    else: # 預設為中間
        primary_camera_rules = """
**準則 1A: 主要分析視角 (中間攝影機)**
*   【學生個人照片序列】來自教室前方的【中間】攝影機。
*   當這位學生臉部【正面朝向鏡頭】時，即代表其視線正朝向教室的【正前方中心】。
{forward_arc_principle}
"""

    # --- 3. 【全新架構】根據學生【排數】，動態生成【分析協議】 ---
    row_specific_protocol = ""
    is_front_row = any(keyword in student_position for keyword in ["第一", "第二"])
    is_rear_row = any(keyword in student_position for keyword in ["第三", "第四"])

    if is_rear_row:
        # --- 這是為後排學生設計的、基於「邏輯推斷」的協議 ---
        row_specific_protocol = """
**【第二部分：後排學生分析協議 (絕對推斷模式)】**

由於該學生 (`{student_position}`) 位於教室後方，視覺證據極其有限，你必須作為一名頂級偵探，遵循以下不可動搖的推斷鐵則。

*   **鐵則 0：低可視度下的姿態優先原則 (Posture-First under Low Visibility)**
    *   由於後排照片常常模糊或有遮擋，你的首要任務是判斷核心姿態。
    *   當學生的【臉部或手部細節無法辨認】時，你**必須**忽略任何基於模糊形狀的細節猜測（例如，看起來像在喝水或玩手機），並嚴格依賴以下基於身體姿態的鐵則。

*   **鐵則 A：低頭行為的「絕對學習推定」原則 (The Absolute Presumption of Learning)**
    *   對於後排學生，其桌面學習材料【總是】被假定為存在，即使你看不清。
    *   因此，只要你觀察到學生【頭部低垂】，你就**必須、無條件地、最優先地**將其主要行為判定為 `V_BOK` (目視書本/筆記)。
    *   **【疲勞姿態豁免條款】**：學生在低頭看書時，常常會伴隨各種旨在維持專注或對抗疲勞的姿態，例如**『揉眼睛』、『搔頭』、『手扶額頭』、『托腮』**。這些動作【絕對不允許】被用來推翻『目視書本/筆記』的核心判斷。你應將它們視為【學習過程的一部分】，並可將其作為次要標籤疊加。
    *   **【推翻條件】**：只有在你擁有【極其明確的、不可辯駁的】視覺證據，證明他正在進行以下**兩種**活動之一時，才能推翻此推定：(1) **明確的社交互動** (`V_CLS`)；(2) **明確的隱蔽手部動作** (`H_PLAY_HW`)。

*   **鐵則 B：前方視線的「教師優先」原則**
    *   由於後排學生視野開闊，一個朝向教室前方的視線，其目標是老師的概率極高。
    *   因此，只要學生的視線是朝向【教室前方】的任何區域，你就應**優先**將其標註為 `V_TCH` (目視教師)，其次才是 `V_BRD` (目視黑板)。
"""
    else: # 預設為前排規則
        # --- 這是為前排學生設計的、基於「視覺證據」的協議 ---
        row_specific_protocol = """
**【第二部分：前排學生分析協議 (高證據模式)】**

由於該學生 (`{student_position}`) 位於教室前方，視覺證據清晰，你的所有判斷都必須基於【嚴格的視覺證據】。

*   **證據準則 A：低頭行為的分診決策樹 (軸線檢查版)**
    *   當你觀察到學生【頭部低垂】時，你必須遵循以下檢查順序：
        1.  **軸線檢查：** 判斷頭部是【正下方】還是【側下方】。
            *   **A. 如果是【正下方】（視線在個人工作區軸線內）**：直接判定為 `V_BOK` (目視書本/筆記)。
            *   **B. 如果是【側下方】（視線已偏離軸線，轉向側面）**：直接判定為 `V_CLS` (目視同學)。
        2.  **手部動作覆寫：** 在完成上述視線判斷後，檢查手部。
            *   是否有隱蔽的、非學習相關的動作？ -> **是** -> 將主要標籤**覆寫**為 `H_PLAY_HW` (玩弄手部/文具)。
        3.  **最終備用：** 僅在以上情況都明確排除後（例如，低頭但視線看向空無一物的地面），才使用 `P_DWN` (低頭非學習)。

*   **證據準則 B：前方視線的精確匹配**
    *   當學生視線朝向【教室前方】時，你必須嚴格結合【老師位置情境】進行幾何匹配：
        *   視線方向與老師位置**高度匹配**？ -> `V_TCH` (目視教師)。
        *   視線方向在前方，但與老師位置有**偏差**？ -> `V_BRD` (目視黑板)。

"""

    # --- 4. 準備其他通用模塊和資訊 ---
    teacher_context_header = "5.  **【老師位置文字情境】**: 用於交叉驗證你視線判斷的輔助數據。" if has_teacher_context else ""
    universal_rules = """
---
**【第三部分：通用行為判斷準則 (全體適用)】**

在你通過【第二部分】的協議確定了學生的主要視線方向或狀態後，你必須遵循以下通用規則來最終確定標籤組合。

*   **準則 A：【舉手行為的嚴格過-濾器】**
    *   當觀察到【手臂抬起】時，**嚴禁**直接標註為舉手。必須先通過以下排除法檢查：
        1.  手是在【指向】、【揮舞】或與同學手勢互動嗎？ -> **是** -> 標註為 `V_CLS`，過濾結束。
        2.  手是在【觸摸頭部/頭髮】嗎？ -> **是** -> 標註為 `H_TOUCH_H` 或 `H_TOUCH_F`，過濾結束。
    *   只有通過了上述過濾，才能根據【課堂狀態】判斷是 `I_HND_A` (主動) 還是 `I_HND_P` (被動)。

*   **準則 B：【主行為分層決策協議 (社交優先)】**
    *   你必須嚴格遵循以下層級順序來判斷主要行為，**上級條件滿足時，優先級最高**。

    *   **第一層：檢查【壓倒性的社交姿態】**
        *   首先判斷學生的**整體身體軀幹**是否已明確轉向側面或後方，與同學形成一個清晰的互動框架？
        *   **是** -> **主要行為【必須】標註為 `V_CLS` (目視同學)**。在此基礎上，任何手部動作（如操作卡牌、指點書本）都應被理解為是**這次社交互動的組成部分**，而不是獨立的行為。

    *   **第二層：檢查【個人任務動作】(僅在身體朝前時評估)**
        *   如果學生身體基本朝前，未進入社交姿態，則按此順序檢查：
        *   是否在**做筆記 (`H_NOT`)**？ -> **是** -> `H_NOT` 是主要標籤。
        
    
    *   **第三層：檢查【非任務相關動作】(僅在身體朝前時評估)**
        *   是否在**玩弄手部/文具 (`H_PLAY_HW`)**？ -> **是** -> `H_PLAY_HW` 是主要標籤。
        *   是否有其他明確的手部動作（如觸摸臉/頭髮）？ -> **是** -> 標註對應動作。

    *   **第四層：標註【靜態視覺行為】(如果以上皆無)**
        *   如果沒有檢測到任何壓倒性的姿態或主動動作，那麼才把你在【第二部分】中判斷的視線行為（`V_TCH`, `V_BOK` 等）作為主要標籤。

    *   **第五層：疊加一個【通用身體姿態】**
        *   最後，再疊加一個最能描述學生狀態的輔助姿態標籤，如 `P_LEAN`(身體前傾) 等。

*   **準則 C：【瞬時動作的最高優先級原則 (The Principle of Momentary Action Supremacy)】**
    *   你的判斷【必須】基於每一張獨立圖片所捕捉到的**瞬時主要動作**，而不是對整個序列的籠統印象。
    *   **核心規則**：如果一個學生在前一張圖片中正在做筆記 (`H_NOT`)，但在當前圖片中，他的主要物理動作變成了另一件事，你**必須**標註這個新動作。
    *   **範例 1 (動作中斷)**：學生正在寫字，下一張圖他手抬起來觸摸頭髮。那麼這張圖片的主要行為【就是】`H_TOUCH_H`，而不是 `H_NOT` 的延續。
    *   **範例 2 (社交中斷)**：學生正在寫字，下一張圖他轉頭與同學說話或看同學的考卷。那麼這張圖片的主要行為【就是】`V_CLS`，你必須捕捉這個注意力的轉移。
"""
    # --- 5. 生成最終的、完整的 Prompt ---
    return f"""
「你是一位世界頂尖的教育分析師，精通電腦視覺、空間幾何推理與心理學。你的任務是綜合所有給定的物理與情境資訊，對學生的學習行為進行最精準、最客觀的標註。」

**【你收到的資訊來源】**
1.  **【學生個人照片序列】**: 你的主要分析對象。
2.  **【班級整體照片】**: 你的情境驗證工具。
3.  **【課堂狀態】**: 判斷行為動機的核心上下文。
4.  **【學生座位】**: `{student_position}`。
{teacher_context_header}

---
**【第一部分：多機位攝影機系統與物理準則】**
{primary_camera_rules}
**準則 1B: 情境驗證視角 (固定為中間攝影機)**
*   【班級整體照片】**永遠**來自教室前方的【中間】攝影機。
**準則 2: 影像鏡像轉換 (全域適用)**
*   所有攝影機畫面都是鏡像的：照片中的**左側** => 教室中的**右側**；照片中的**右側** => 教室中的**左側**。

---
{row_specific_protocol}
---
{universal_rules}
---
{behavior_table_for_prompt}
---
**【第四部分：輸出格式要求 - 嚴謹推理與簡明輸出】**
*   你的回答**必須**是一個結構完整的、單一的 JSON 物件。
*   `behavior_category` 欄位的值**必須**是一個包含 1 到 2 個編碼字串的**陣列 (Array)**。
*   `per_image_highlights` 的每個物件中，**必須包含** `image_index_in_sequence`, `context_description`, `behavior_category`, 和 `confidence` 這四個鍵。
*   **【關鍵指令】**: `context_description` 欄位的內容**必須極度簡潔**。請只使用**核心推理關鍵字**來描述你的判斷依據，例如："後排協議 -> 低頭推定" 或 "前排協議 -> 視線匹配老師位置"。

**【輸出 JSON 格式範例】**
```json
{{
  "sequence_analysis_confidence": 0.97,
  "per_image_highlights": [
    {{
      "image_index_in_sequence": 0,
      "context_description": "後排協議：低頭，推定為 V_BOK。",
      "behavior_category": ["V_BOK", "P_STR"],
      "confidence": 0.95
    }},
    {{
      "image_index_in_sequence": 1,
      "context_description": "前排協議：視線與老師位置匹配。",
      "behavior_category": ["V_TCH"],
      "confidence": 0.99
    }},
    {{
      "image_index_in_sequence": 2,
      "context_description": "動作優先原則：檢測到書寫動作。",
      "behavior_category": ["H_NOT", "P_LEAN"],
      "confidence": 0.98
    }}
  ]
}}
```"""

def extract_json_from_string(text):
    """
    從可能包含額外文字或 Markdown 區塊的字串中，穩健地提取出 JSON 字串。
    """
    # 優先嘗試尋找被 markdown code block 包圍的 JSON
    # re.DOTALL 讓 '.' 可以匹配換行符
    match = re.search(r"```json\s*(\{.*\})\s*```", text, re.DOTALL)
    if match:
        # 如果找到，直接返回第一個捕獲組 (也就是 {...} 的部分)
        return match.group(1)

    # 如果沒有 markdown，就尋找第一個 '{' 和最後一個 '}'
    start_index = text.find('{')
    end_index = text.rfind('}')
    
    if start_index != -1 and end_index != -1 and end_index > start_index:
        # 如果都找到了，返回它們之間的子字串
        return text[start_index : end_index + 1]
        
    # 如果以上方法都失敗，返回原始文本，讓後續的 json.loads() 自然地報錯
    return text

def find_state_for_timestamp(target_timestamp, classroom_states):
    """
    【v4.0 穩定版】根據時間戳，查找對應的課堂狀態標籤。
    使用線性查找以確保最高的準確性和可靠性。
    """
    if not classroom_states:
        return "未知"
    
    # 為了除錯，只打印一次查找的詳細信息
    if not hasattr(find_state_for_timestamp, "has_logged_first_search"):
        print("\n" + "-"*20 + " [除錯日誌 - 首次查找課堂狀態] " + "-"*20)
        print(f"  - 正在用第一張照片的時間戳进行匹配...")
        print(f"  - 目標時間戳 (Target Timestamp): {target_timestamp}")
        if classroom_states:
            first_state = classroom_states[0]
            print(f"  - 正在检查第一個時間區間: 從 {first_state.get('start_time_td')} 到 {first_state.get('end_time_td')}")
        print("-" * 69 + "\n")
        find_state_for_timestamp.has_logged_first_search = True

    # 遍歷每一個課堂狀態的時間區間
    for state in classroom_states:
        start_td = state.get("start_time_td")
        end_td = state.get("end_time_td")
        
        # 確保這個區間的時間數據是有效的
        if start_td and end_td:
            # 檢查目標時間戳是否落在 [開始時間, 結束時間] 這個閉区间内
            if start_td <= target_timestamp <= end_td:
                return state["classroom_state"]
    
    # 如果遍歷完所有區間都沒找到，說明時間戳確實超出了範圍
    return "未知"

def parse_time_offset(time_str):
    """將 "HH:MM:SS" 格式的字串轉換為 timedelta 物件"""
    if not time_str or not isinstance(time_str, str):
        return None
    try:
        h, m, s = map(int, time_str.split(':'))
        return datetime.timedelta(hours=h, minutes=m, seconds=s)
    except (ValueError, TypeError):
        print(f"警告：時間偏移量 '{time_str}' 格式不正確，應為 'HH:MM:SS'。將忽略此設定。")
        return None

def get_valid_input(prompt_message):
    while True:
        user_input = input(prompt_message).strip()
        if user_input: return user_input
        print("錯誤：輸入不能為空，請重新輸入。")

def get_valid_folder_path(prompt_message, is_optional=False): # 新增 is_optional 參數
    while True:
        folder_path = input(prompt_message).strip().strip('"')
        if not folder_path and is_optional:
            return None # 如果是可選的且用戶未輸入，返回 None
        if os.path.isdir(folder_path):
            return folder_path
        print(f"錯誤：路徑 '{folder_path}' 不是一個有效的資料夾，請重新輸入。")

def get_timestamp_from_filename(filename):
    """
    【v3.0 兼容版】从档名解析时间戳。
    能同时处理多种常见格式。
    """
    # 模式一：最优先，精确匹配 HH-MM-SS-ms.jpg 格式 (常见于学生照片)
    match = re.search(r'^(\d{2})-(\d{2})-(\d{2})-(\d{3})\.(jpg|jpeg|png|webp)$', filename, re.IGNORECASE)
    if match:
        try:
            h, m, s, ms, ext = match.groups()
            return datetime.timedelta(hours=int(h), minutes=int(m), seconds=int(s), milliseconds=int(ms))
        except (ValueError, IndexError):
            pass

    # 模式二：匹配包含 ...HH-MM-SS-ms... 的通用格式
    match = re.search(r'(\d{2})-(\d{2})-(\d{2})-(\d{3})', filename)
    if match:
        try:
            h, m, s, ms = map(int, match.groups())
            return datetime.timedelta(hours=h, minutes=m, seconds=s, milliseconds=ms)
        except (ValueError, IndexError):
            pass

    # 模式三：匹配包含 ...h...m...s 的通用格式
    match = re.search(r'(\d+)h(\d{2})m(\d{2})s', filename)
    if match:
        try:
            h, m, s = map(int, match.groups())
            return datetime.timedelta(hours=h, minutes=m, seconds=s)
        except (ValueError, IndexError):
            pass

    # 如果所有格式都沒匹配成功，最终返回 None
    return None

def encode_image_to_base64(image_path, max_size_kb=512, target_quality=75):
    try:
        with Image.open(image_path) as img: img.verify()
        with Image.open(image_path) as img:
            if img.mode == 'RGBA' or img.mode == 'P': img = img.convert('RGB')
            current_quality = target_quality
            output_buffer = io.BytesIO()
            temp_img = img.copy()
            temp_img.save(output_buffer, format="JPEG", quality=current_quality)
            current_size_kb = output_buffer.tell() / 1024

            if current_size_kb > max_size_kb:
                scale_factor = math.sqrt(max_size_kb / current_size_kb)
                new_width = int(temp_img.width * scale_factor * 0.9)
                new_height = int(temp_img.height * scale_factor * 0.9)
                if new_width >= 50 and new_height >= 50: # 最小尺寸限制
                    print(f"    圖片 {os.path.basename(image_path)} ({current_size_kb:.1f} KB) 過大，縮放並調整質量...")
                    temp_img = temp_img.resize((new_width, new_height), Image.Resampling.LANCZOS)
                    output_buffer = io.BytesIO()
                    temp_img.save(output_buffer, format="JPEG", quality=max(current_quality - 15, 40)) # 質量可以降更多
                else:
                    print(f"    警告: 圖片 {os.path.basename(image_path)} 縮放後過小，可能影響質量。使用較低質量。")
                    output_buffer = io.BytesIO()
                    img.save(output_buffer, format="JPEG", quality=max(current_quality // 2, 30) )


            output_buffer.seek(0)
            binary_data = output_buffer.getvalue()
            base64_encoded_data = base64.b64encode(binary_data)
            return f"data:image/jpeg;base64,{base64_encoded_data.decode('utf-8')}"
    except Exception as e: print(f"錯誤：處理圖片 '{image_path}': {e}"); return None

def load_classroom_states(json_path):
    """【全新】讀取預處理好的課堂狀態時間軸 JSON 檔案。"""
    if not json_path or not os.path.isfile(json_path):
        print("提示：未提供或找不到課堂狀態 JSON 檔案。將不使用課堂情境。")
        return None  # 返回 None 以便更明確地判斷失敗
    
    print(f"正在讀取課堂狀態時間軸: {json_path}...")
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f) # <--- 關鍵修正：將 f 作為參數傳入
            
            # 將時間字串預先轉換為 timedelta 物件以便快速比較
            timeline = data.get("timeline", [])
            for state in timeline:
                state["start_time_td"] = parse_time_offset(state["start_time"])
                state["end_time_td"] = parse_time_offset(state["end_time"])
            print(f"成功載入 {len(timeline)} 個課堂狀態階段。")
            return timeline
    except Exception as e:
        print(f"❌ 錯誤：讀取或解析課堂狀態 JSON 時發生問題: {e}")
        return None # 返回 None

def default_error_result_structure():
    return { "error": "分析失敗或無有效數據", "sequence_analysis_confidence": 0.0, "sequence_summary": "未能生成序列總結。", "dominant_sustained_behaviors": [], "significant_behavior_shifts": [], "per_image_highlights": [], "general_sequence_atmosphere_hint": "未知" }

def clean_analysis_json(raw_analysis):
    """
    清理從 API 返回的分析 JSON，只保留我們需要的欄位。
    這是一個防禦性措施，用來處理微調模型的「幻覺」問題。
    """
    if not isinstance(raw_analysis, dict):
        return default_error_result_structure()

    # 定義合法的鍵
    allowed_top_level_keys = {"sequence_analysis_confidence", "per_image_highlights"}
    allowed_highlight_keys = {"image_index_in_sequence", "context_description", "behavior_category", "confidence"}

    cleaned_analysis = {}
    
    # 1. 清理最外層的鍵
    for key, value in raw_analysis.items():
        if key in allowed_top_level_keys:
            cleaned_analysis[key] = value

    # 2. 檢查並清理 per_image_highlights 列表
    if "per_image_highlights" in cleaned_analysis and isinstance(cleaned_analysis["per_image_highlights"], list):
        cleaned_highlights = []
        for raw_highlight in cleaned_analysis["per_image_highlights"]:
            if not isinstance(raw_highlight, dict):
                continue # 如果列表中的元素不是字典，就跳過它
            
            cleaned_highlight = {}
            for key, value in raw_highlight.items():
                if key in allowed_highlight_keys:
                    cleaned_highlight[key] = value
            
            # 確保必要的鍵存在，即使為空
            for required_key in allowed_highlight_keys:
                if required_key not in cleaned_highlight:
                    cleaned_highlight[required_key] = None

            cleaned_highlights.append(cleaned_highlight)
        
        cleaned_analysis["per_image_highlights"] = cleaned_highlights

    # 3. 如果最外層缺少必要的鍵，補上預設值
    if "sequence_analysis_confidence" not in cleaned_analysis:
        cleaned_analysis["sequence_analysis_confidence"] = 0.0
    if "per_image_highlights" not in cleaned_analysis:
        cleaned_analysis["per_image_highlights"] = []

    return cleaned_analysis

def analyze_student_behavior_from_images_sequence(student_image_paths, teacher_position_text, classroom_view_image_path, classroom_state, image_filenames_batch, openai_client, student_id, student_position):
    """
    【v8.2 偵錯增強版】分析單一圖片批次，包含針對 JSON 解析失敗的詳細檢查。
    此版本能明確區分內容篩選、Token 上限和模型格式錯誤。
    """
    if not student_image_paths:
        return default_error_result_structure()
    
    user_message_content = []
    
    # 1. 編碼學生個人圖片序列
    encoded_student_images = []
    for img_path in student_image_paths:
        b64_img = encode_image_to_base64(img_path)
        if b64_img:
            encoded_student_images.append({
                "type": "image_url", 
                "image_url": {"url": b64_img, "detail": IMAGE_DETAIL_LEVEL}
            })

    if not encoded_student_images:
        return default_error_result_structure()
        
    # 2. 編碼班級整體照片
    encoded_classroom_view_image = None
    if classroom_view_image_path:
        b64_img = encode_image_to_base64(classroom_view_image_path)
        if b64_img:
            encoded_classroom_view_image = {
                "type": "image_url", 
                "image_url": {"url": b64_img, "detail": IMAGE_DETAIL_LEVEL}
            }

    # 3. 組合完整的請求體
    user_message_content.append({"type": "text", "text": f"請根據系統提示中的偵探任務，分析學生「{student_id}」的行為。學生照片序列的文件名（供您參考）為: {', '.join(image_filenames_batch)}。"})
    user_message_content.append({"type": "text", "text": f"【課堂狀態】: {classroom_state}"})
    user_message_content.append({"type": "text", "text": f"【老師位置情境】根據預先分析，在此時間段，老師的位置在教室前方的「{teacher_position_text}」。請以此作為判斷『目視教師』的核心依據。"})
    
    if encoded_classroom_view_image:
        user_message_content.append({"type": "text", "text": "【班級整體照片】(用於定位學生和觀察整體氛圍)"})
        user_message_content.append(encoded_classroom_view_image)
    user_message_content.append({"type": "text", "text": "【學生個人照片序列】(主要分析對象)"})
    user_message_content.extend(encoded_student_images)

    # 4. 獲取系統提示
    system_prompt_content = get_behavior_sequence_analysis_system_prompt(student_position, bool(teacher_position_text and teacher_position_text != "未知"))

    # 5. API 呼叫與錯誤處理
    retry_attempts = 2
    for attempt in range(retry_attempts + 1):
        raw_content = ""
        finish_reason = "unknown" # 初始化 finish_reason
        try:
            has_teacher_context = bool(teacher_position_text and teacher_position_text != "未知")
            has_classroom_image = bool(encoded_classroom_view_image)
            has_state = classroom_state != "未知"

            print(f"  正在向 {VISION_DEPLOYMENT_NAME} 發送請求 (學生: {len(encoded_student_images)}, 課堂狀態: {'有' if has_state else '無'}, 老師位置: {'有' if has_teacher_context else '無'}, 班級照片: {'有' if has_classroom_image else '無'})...")
            
            response = openai_client.chat.completions.create(
                model=VISION_DEPLOYMENT_NAME, 
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": system_prompt_content},
                    {"role": "user", "content": user_message_content}
                ],
                # 根據之前的錯誤，已移除 temperature 參數
                # 根據之前的錯誤，已將 max_tokens 改為 max_completion_tokens
                max_completion_tokens=MAX_TOKENS_VISION_COMPLETION 
            )

            # === 【全新】增加詳細的完成原因檢查 ===
            choice = response.choices[0]
            finish_reason = choice.finish_reason

            # 1. 檢查是否被內容篩選器擋掉 (最高優先級)
            if finish_reason == 'content_filter':
                print(f"    警告：API請求因「內容篩選」而終止 (第 {attempt+1} 次嘗試)。這通常是因為輸入的圖片觸發了安全策略。")
                time.sleep(API_RETRY_DELAY_SECONDS)
                continue # 直接跳到下一次 for 迴圈的重試

            # 2. 檢查是否因為達到 Token 上限而被切斷
            if finish_reason == 'length':
                print(f"    警告：API回應因達到 max_completion_tokens ({MAX_TOKENS_VISION_COMPLETION}) 上限而被截斷 (第 {attempt+1} 次嘗試)。")
            
            raw_content = choice.message.content

            # 3. 檢查回傳內容是否為空
            if not raw_content:
                print(f"    錯誤：API返回了空的內容 (第 {attempt+1} 次嘗試)。完成原因: {finish_reason}")
                time.sleep(5)
                continue
            # ==========================================

            # === ✅【核心修改點】在解析前，先呼叫清理函數 ===
            cleaned_content = extract_json_from_string(raw_content)
            # ===============================================
            
            # 解析原始回應
            analysis_result_raw = json.loads(cleaned_content)
            
            # 清理可能包含幻覺的 JSON
            analysis_result = clean_analysis_json(analysis_result_raw)

            # 本地解碼
            if "per_image_highlights" in analysis_result and isinstance(analysis_result["per_image_highlights"], list):
                for hl in analysis_result["per_image_highlights"]:
                    if not hl or not isinstance(hl, dict) or "behavior_category" not in hl:
                        continue
                    codes = hl.get("behavior_category")
                    if not codes: continue
                    if not isinstance(codes, list): codes = [codes]
                    decoded_behaviors = [CODE_TO_BEHAVIOR.get(code, f"未知編碼({code})") for code in codes if code]
                    hl["behavior_category"] = decoded_behaviors
            
            if "sequence_summary" not in analysis_result:
                analysis_result["sequence_summary"] = "已設定為精簡模式，此欄位由本地生成。"

            return analysis_result

        except json.JSONDecodeError:
            # 如果通過了上面的檢查，但仍然解析失敗，表示模型可能產生了格式錯誤的文字
            print(f"    錯誤：無法解析JSON (第 {attempt+1} 次嘗試)。完成原因: '{finish_reason}'。")
            # 【關鍵】打印完整的原始回傳內容，以便我們看到問題所在
            print(f"    --- 模型原始回應 ---")
            print(raw_content)
            print(f"    ----------------------")
            time.sleep(5)
        except RateLimitError:
            print(f"  警告：API速率限制，等待 {API_RETRY_DELAY_SECONDS}s 後重試...");
            time.sleep(API_RETRY_DELAY_SECONDS)
        except APIError as e:
            wait_time = 10 + attempt * 5 
            print(f"  錯誤：API錯誤 (第 {attempt+1} 次嘗試): {e}. 將在 {wait_time} 秒後重試...");
            time.sleep(wait_time)
        except Exception as e:
            wait_time = 5 + attempt * 5
            print(f"  錯誤：未知錯誤 (第 {attempt+1} 次嘗試): {e}. 將在 {wait_time} 秒後重試...");
            time.sleep(wait_time)
    
    print(f"  錯誤：圖片序列分析在多次重試後失敗。")
    return default_error_result_structure()

def generate_personalized_summary_notes(student_id, overall_stats, non_task_highlights, openai_client):
    if not openai_client: return {"error": "OpenAI client not available"}
    # ... (此函數內部的 Prompt 內容完全不需要修改) ...
    stats_summary_for_ai = "\n".join([f"- {s['behavior_category']}: {s['percentage']:.1f}% ({s['count']}次)" for s in overall_stats[:7]]) # 確保百分比格式
    non_task_prompt_part = "該生在本堂課中，未觀察到明顯或頻繁的非任務相關行為。"
    if non_task_highlights:
        highlights_str = "\n".join([f"  - 圖 '{img_fn}' (~{ts}): '{beh}' (判斷依據: {desc})" for img_fn, ts, beh, desc in non_task_highlights[:3]])
        if highlights_str: non_task_prompt_part = f"在本堂課中，觀察到一些非任務相關行為，例如：\n{highlights_str}\n這可能影響了學習專注度。"
    prompt_for_summary = f"""
    「你是一位專業且富有同理心的學習行為教練。你的目標不是批評，而是透過客觀數據，引導學生發現自己的學習模式，並提供能立即實踐的策略，以激發他們『自我反思』的動力。」

    **任務：** 為學生「{student_id}」撰寫一份「AI學習夥伴的觀察與建議」。

    **學生的課堂行為數據：**
    *   **主要行為分佈:**
    {stats_summary_for_ai}
    *   **值得注意的行為片段:**
    {non_task_prompt_part}

    **撰寫指引與 JSON 格式要求：**
    請嚴格遵循以下指引，產生一個結構完整的 JSON 物件。

    *   **`greeting` (問候語):**
        *   用親切、個人化的方式稱呼學生，例如：「嗨，{student_id} 同學，一起來看看這次課堂的學習足跡吧！」

    *   **`positive_feedback` (亮點觀察):**
        *   **必須**從數據中找出最值得肯定的行為（例如「目視教師」或「筆記」佔比最高），並給予具體、真誠的讚美。
        *   **範例**：「我發現你在這堂課有超過一半的時間都在『目視教師』，這代表你非常努力地跟上老師的節奏，非常棒！」

    *   **`observation_points_summary` (行為模式提醒):**
        *   客觀、中性地指出一個或兩個最主要的、可能影響學習的行為模式。避免使用負面詞彙。
        *   **範例**：「數據也顯示，大約有 15% 的時間出現了『玩弄物品』或『目視他處』的狀況，這些時刻可能讓我們不小心錯過了一些重點喔。」

    *   **`reflection_points` (反思引導提問):**
        *   **【此項最為關鍵】** 根據前面的觀察點，設計 2-3 個**開放式問題**，引導學生思考行為背後的原因，而不是直接給答案。
        *   **問題範例 1**：「我們可以一起回想看看，當出現『玩弄物品』的時候，通常是在課程的哪個階段呢？是覺得內容太簡單、太難，還是剛好有點疲倦了呢？」
        *   **問題範例 2**：「當視線看向其他地方時，是想到了什麼有趣的事，還是被教室裡的其他動靜吸引了呢？了解這些原因，能幫助我們找到最適合自己的專注方法。」

    *   **`suggestions` (可實踐的小建議):**
        *   提供 1-2 個**具體、微小、且容易執行**的行動建議。不要說「要專心」，而是給出方法。
        *   **建議範例 1**：「下次當你發現自己開始無意識地轉筆時，可以試著把它輕輕放下，然後做一個深呼吸，再重新將目光移回老師或課本上。」
        *   **建議範例 2**：「如果感覺到疲倦或分心，可以試試看『筆記專注法』：在筆記本上寫下老師說的任何一個關鍵字，這個小動作能幫助我們的大腦重新連線！」

    *   **`encouragement` (鼓勵與結語):**
        *   用一句溫暖、有力的話作結，強調這份報告是幫助他成長的工具。
        *   **範例**：「每一次的觀察都是為了讓我們更了解自己。相信你透過這些小小的調整，一定能發揮出自己最大的潛力，加油！」
    """
    retry_attempts = 2
    for attempt in range(retry_attempts):
        try:
            print(f"  正在為學生 {student_id} 使用 {SUMMARY_DEPLOYMENT_NAME} 生成個性化總結...") # 修改了日誌輸出
            response = client.chat.completions.create(
                # ==========================================================
                # ↓↓↓ 【關鍵修改】使用我們新設定的經濟型模型 ↓↓↓
                # ==========================================================
                model=SUMMARY_DEPLOYMENT_NAME, # 這裡會自動使用 "gpt-4.1"
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": "你是一位富有同理心和洞察力的教育顧問。"},
                          {"role": "user", "content": prompt_for_summary}],
                # --- 【還原 gpt-4.1 參數】 ---
                max_tokens=MAX_TOKENS_SUMMARY_COMPLETION, 
                temperature=0.7        
                
            )
            summary_data = json.loads(response.choices[0].message.content)
            expected_keys = ["greeting", "positive_feedback", "observation_points_summary", "reflection_points", "suggestions", "encouragement"]
            if all(key in summary_data for key in expected_keys): 
                print(f"  成功為學生 {student_id} 生成個性化總結。")
                return summary_data
            else: 
                print(f"    警告：個性化總結JSON缺少鍵。返回: {summary_data}")
                return {key: summary_data.get(key, f"AI未能生成 ({key})") for key in expected_keys}
        except Exception as e: 
            print(f"  生成個性化總結錯誤 ({attempt + 1}): {e}")
            time.sleep(API_RETRY_DELAY_SECONDS if isinstance(e, RateLimitError) else 5)
        if attempt == retry_attempts - 1: 
            print(f"  錯誤：無法為學生 {student_id} 生成個性化總結。")
            return {"greeting": f"親愛的 {student_id},", "positive_feedback": "總結生成遇到問題。", "observation_points_summary": "請參考統計數據。", "reflection_points": "未能生成。", "suggestions": "請自行評估。", "encouragement": "加油！"}
    return {}

def find_closest_image_path(representative_timestamp, sorted_photo_list, max_time_diff_seconds=5):
    """
    一個可重用的輔助函數，用二分查找法在排序好的照片列表中找到時間最接近的照片路徑。
    """
    if not sorted_photo_list:
        return None

    all_timestamps = [item[0] for item in sorted_photo_list]
    # bisect_left 找到應該插入的位置
    insertion_point = bisect.bisect_left(all_timestamps, representative_timestamp)

    closest_path = None
    min_diff = datetime.timedelta.max

    # 只檢查插入點及其前後的幾個候選照片，效率極高
    # 檢查範圍設為 insertion_point-2 到 insertion_point+2 以增加容錯
    start_index = max(0, insertion_point - 2)
    end_index = min(len(sorted_photo_list), insertion_point + 2)

    for i in range(start_index, end_index):
        candidate_ts, candidate_path = sorted_photo_list[i]
        diff = abs(candidate_ts - representative_timestamp)
        if diff < min_diff:
            min_diff = diff
            closest_path = candidate_path

    # 只有在時間差在容許範圍內才返回路徑
    if closest_path and min_diff.total_seconds() <= max_time_diff_seconds:
        return closest_path
    
    return None

def find_closest_position(representative_timestamp, sorted_position_list, max_time_diff_seconds=10):
    """
    用二分查找法在排序好的位置列表中找到時間最接近的老師位置。
    """
    if not sorted_position_list:
        return "未知"

    all_timestamps = [item[0] for item in sorted_position_list]
    insertion_point = bisect.bisect_left(all_timestamps, representative_timestamp)

    closest_position = "未知"
    min_diff = datetime.timedelta.max

    start_index = max(0, insertion_point - 2)
    end_index = min(len(sorted_position_list), insertion_point + 2)

    for i in range(start_index, end_index):
        candidate_ts, candidate_pos = sorted_position_list[i]
        diff = abs(candidate_ts - representative_timestamp)
        if diff < min_diff:
            min_diff = diff
            closest_position = candidate_pos
    
    if min_diff.total_seconds() <= max_time_diff_seconds:
        return closest_position
    
    return "未知"

def process_single_batch(batch_idx, image_batch_info, teacher_positions_data, sorted_classroom_photos, classroom_states, client, student_id, student_position):
    """
    【升級版】處理單一圖片批次，使用老師位置和預處理好的課堂狀態。
    """
    batch_student_paths = [info["path"] for info in image_batch_info]
    batch_image_filenames = [info["filename"] for info in image_batch_info]
    
    representative_timestamp = image_batch_info[0]["timestamp_obj"]

    # --- 為當前批次查找所有情境數據 ---
    teacher_position_text = find_closest_position(representative_timestamp, teacher_positions_data)
    classroom_view_path = find_closest_image_path(representative_timestamp, sorted_classroom_photos)
    # 【關鍵修正】使用正確的變數名稱 classroom_states
    classroom_state = find_state_for_timestamp(representative_timestamp, classroom_states)

    # --- 呼叫【新版】分析函數 ---
    sequence_analysis_data = analyze_student_behavior_from_images_sequence(
        student_image_paths=batch_student_paths,
        teacher_position_text=teacher_position_text,
        classroom_view_image_path=classroom_view_path,
        classroom_state=classroom_state, # <-- 傳入狀態標籤
        image_filenames_batch=batch_image_filenames,
        openai_client=client,
        student_id=student_id,
        student_position=student_position
    )

    return {
        "batch_index": batch_idx,
        "image_batch_info": image_batch_info,
        "matched_teacher_position_text": teacher_position_text,
        "matched_classroom_view_image": os.path.basename(classroom_view_path) if classroom_view_path else None,
        "matched_classroom_state": classroom_state,
        "analysis": sequence_analysis_data
    }

def main():
    print("--- 學生課堂學習行為分析報告生成 (JSON) v5.0 - 成本優化版 ---")
    print(f"視覺模型: {VISION_DEPLOYMENT_NAME}, 文本模型: {VISION_DEPLOYMENT_NAME}, 總結模型: {VISION_DEPLOYMENT_NAME}")
    print(f"圖片批次大小: {IMAGES_PER_API_CALL}, 圖片取樣率: 1/{SAMPLING_RATE}, 圖片解析度: {IMAGE_DETAIL_LEVEL}")
    print("-" * 40)

    # --- 使用者輸入部分 ---
    student_id = get_valid_input("請輸入學生姓名 (可中文): ")
    student_number = get_valid_input("請輸入學生座號 (例如: 1): ") 
    student_images_folder = get_valid_folder_path(f"請輸入 '{student_id}' 的個人影像資料夾: ")
    print("\n--- 請輸入課堂情境資訊 ---")
    student_position = get_valid_input("請輸入學生座位 (例如: '第3排中間', '第1排左側'): ")
    print("-" * 50)

    # --- 準備輸出資料夾 ---
    safe_student_id_for_folder = re.sub(r'[\\/*?:"<>|]',"", student_id.replace(" ", "_"))
    student_specific_folder_base = os.path.join(JSON_OUTPUT_FOLDER, safe_student_id_for_folder)
    os.makedirs(student_specific_folder_base, exist_ok=True)

    # --- 準備學生照片資料 ---
    image_files_with_timestamps = []
    valid_extensions = ('.png', '.jpg', '.jpeg', '.webp')
    print(f"正在掃描學生個人照片資料夾: {student_images_folder}...")
    for filename in os.listdir(student_images_folder):
        if filename.lower().endswith(valid_extensions):
            timestamp = get_timestamp_from_filename(filename)
            if timestamp:
                image_files_with_timestamps.append({
                    "path": os.path.join(student_images_folder, filename), "filename": filename,
                    "timestamp_obj": timestamp, # <-- 直接使用原始時間戳，不做任何校準
                    "timestamp_str": str(timestamp).split('.')[0]
                })
    image_files_with_timestamps.sort(key=lambda x: x["timestamp_obj"])
    
    # ==========================================================
    # ↓↓↓ 【新增】圖片取樣以降低成本 ↓↓↓
    # ==========================================================
    if SAMPLING_RATE > 1:
        original_count = len(image_files_with_timestamps)
        image_files_with_timestamps = image_files_with_timestamps[::SAMPLING_RATE]
        print(f"已執行圖片取樣：從 {original_count} 張原始照片中，每 {SAMPLING_RATE} 張取 1 張，共 {len(image_files_with_timestamps)} 張照片將被分析。")
    else:
        print(f"成功找到 {len(image_files_with_timestamps)} 張學生個人照片 (未取樣)。")

    if not image_files_with_timestamps: 
        print("錯誤：學生資料夾中未找到有效時間格式的圖片，或取樣後為空。")
        return
    # ==========================================================

    # ... 後續的程式碼，從 `load_teacher_positions` 開始，到整個 `main` 函數結束，
    # 都不需要再做任何修改。您可以直接使用您原有的版本。
    # 我將剩餘部分貼在下方以保證完整性。
    
    def load_teacher_positions(json_path, start_time_offset_str=""): # 新增 start_time_offset_str 參數
        if not json_path or not os.path.isfile(json_path):
            print(f"提示：未提供或找不到老師位置 JSON 檔案 ({json_path})。將不使用老師位置情境。")
            return []
        start_offset = parse_time_offset(start_time_offset_str)
        print(f"正在讀取老師位置數據: {json_path}...")
        try:
            with open(json_path, 'r', encoding='utf-8') as f: data = json.load(f)
            position_map, original_count = [], len(data)
            for item in data:
                try:
                    h, m, s = map(int, item['timestamp'].split(':'))
                    td = datetime.timedelta(hours=h, minutes=m, seconds=s)
                    if start_offset and td < start_offset: continue
                    position_map.append((td, item['position']))
                except (ValueError, KeyError): continue
            position_map.sort(key=lambda x: x[0])
            if start_offset: print(f"成功加載老師位置數據。根據起始時間 '{start_time_offset_str}' 進行過濾，保留 {len(position_map)} 筆 (原 {original_count} 筆)。")
            else: print(f"成功加載並索引了 {len(position_map)} 筆老師位置數據。")
            return position_map
        except Exception as e:
            print(f"錯誤：讀取或解析老師位置 JSON 時發生問題: {e}"); return []

    def load_and_sort_photos(folder_path, photo_type_name, start_time_offset_str=""): # 新增 start_time_offset_str 參數
        if not folder_path or not os.path.isdir(folder_path):
            print(f"提示：未提供或找不到 {photo_type_name} 照片資料夾 ({folder_path})。"); return []
        start_offset = parse_time_offset(start_time_offset_str)
        print(f"正在掃描 {photo_type_name} 照片資料夾: {folder_path}...")
        photo_list, original_count = [], 0
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                original_count += 1
                timestamp = get_timestamp_from_filename(filename)
                if timestamp:
                    if start_offset and timestamp < start_offset: continue
                    photo_list.append((timestamp, os.path.join(folder_path, filename)))
        if photo_list:
            sorted_photos = sorted(photo_list)
            if start_offset: print(f"成功加載 {photo_type_name} 照片。根據起始時間 '{start_time_offset_str}' 進行過濾，保留 {len(sorted_photos)} 張 (原 {original_count} 張有效格式照片)。")
            else: print(f"成功加載並索引了 {len(sorted_photos)} 張 {photo_type_name} 照片。")
            return sorted_photos
        else:
            print(f"警告：在 {photo_type_name} 照片資料夾 '{folder_path}' 中未找到符合條件的圖片。"); return []

    teacher_positions_data = load_teacher_positions(TEACHER_POSITION_JSON, TEACHER_JSON_START_TIME_OFFSET)
    sorted_classroom_photos = load_and_sort_photos(CLASSROOM_IMAGES_FOLDER, "班級整體", CLASSROOM_START_TIME_OFFSET)
    classroom_states = load_classroom_states(CLASSROOM_STATE_JSON_PATH)

    image_batches = [image_files_with_timestamps[i:i + IMAGES_PER_API_CALL] for i in range(0, len(image_files_with_timestamps), IMAGES_PER_API_CALL)]
    print(f"學生圖片將被分為 {len(image_batches)} 個批次進行分析。")

    MAX_WORKERS = 4
    print(f"將使用最多 {MAX_WORKERS} 個執行緒進行平行分析...")
    all_results_from_threads = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_batch_idx = { 
            executor.submit(process_single_batch, idx, batch_info, teacher_positions_data, sorted_classroom_photos, classroom_states, client, student_id, student_position): idx 
            for idx, batch_info in enumerate(image_batches) 
        }
        for future in tqdm(as_completed(future_to_batch_idx), total=len(image_batches), desc=f"分析學生 {student_id} 的圖片批次"):
            try:
                result = future.result()
                all_results_from_threads.append(result)
            except Exception as exc:
                batch_idx = future_to_batch_idx[future]
                print(f'\n批次 {batch_idx + 1} 執行時產生錯誤: {exc}')
                all_results_from_threads.append({"batch_index": batch_idx, "error": str(exc)})

    all_results_from_threads.sort(key=lambda x: x['batch_index'])
    print("\n所有批次分析完成，開始匯總數據...")
    all_sequence_analysis_results = []
    overall_behavior_summary = { "total_images_processed_in_batches": 0, "behavior_counts": Counter(), "behavior_confidence_sum": defaultdict(float), "non_task_behavior_examples_for_summary": [] }
    for result in tqdm(all_results_from_threads, desc="匯總分析結果"):
        if "error" in result:
            all_sequence_analysis_results.append({"batch_index": result['batch_index'] + 1, "analysis": default_error_result_structure()})
            continue
        image_batch_info, sequence_analysis_data = result["image_batch_info"], result["analysis"]
        batch_image_filenames = [info["filename"] for info in image_batch_info]
        all_sequence_analysis_results.append({ "batch_index": result['batch_index'] + 1, "image_filenames_in_batch": batch_image_filenames, "matched_teacher_position_text": result.get("matched_teacher_position_text"), "matched_classroom_view_image": result.get("matched_classroom_view_image"), "analysis": sequence_analysis_data })
        if "error" not in sequence_analysis_data:
            overall_behavior_summary["total_images_processed_in_batches"] += len(batch_image_filenames)
            if "per_image_highlights" in sequence_analysis_data:
                for hl_item in sequence_analysis_data.get("per_image_highlights", []):
                    
                    behavior_list_original, conf = hl_item.get("behavior_category"), hl_item.get("confidence", 0.0)
                    
                    # --- ↓↓↓ 【核心修改】低信度與空行為的過濾與覆寫邏輯 ↓↓↓ ---
                    
                    UNKNOWN_LABEL = "被遮擋/無法判斷"
                    
                    # 複製一份原始行為列表，用於後續的非任務行為判斷
                    behavior_list_for_stats = behavior_list_original
                    
                    # 判斷條件 1: 信度是否低於設定的閾值
                    is_low_confidence = conf is not None and conf < BEHAVIOR_CONFIDENCE_THRESHOLD_FILTER
                    # 判斷條件 2: AI 是否沒有返回任何有效的行為標籤
                    is_empty_behavior = not behavior_list_original

                    if is_low_confidence or is_empty_behavior:
                        # 如果觸發任一條件，則用於統計的行為列表將被強制覆寫
                        behavior_list_for_stats = [UNKNOWN_LABEL]
                        
                    # --- ↑↑↑ 過濾邏輯結束 ↑↑↑ ---

                    # 確保用於統計的行為列表是 list 格式
                    if not isinstance(behavior_list_for_stats, list):
                        behavior_list_for_stats = [behavior_list_for_stats]
                    
                    # 遍歷列表中的每一個行為進行統計 (此處使用的是可能被覆寫過的 behavior_list_for_stats)
                    for cat in behavior_list_for_stats:
                        overall_behavior_summary["behavior_counts"][cat] += 1
                        overall_behavior_summary["behavior_confidence_sum"][cat] += float(conf if conf is not None else 0.0) # 確保 conf 是浮點數
                    
                    # 處理非任務行為範例（注意：此處我們使用未經過濾的原始行為 `behavior_list_original`）
                    # 這樣可以確保即使一個 "玩筆" 行為因信度低被歸入 "無法判斷"，我們依然能捕捉到這個 "意圖"
                    if behavior_list_original and isinstance(behavior_list_original, list):
                        primary_behavior_for_example = behavior_list_original[0]
                        non_task_keywords = [
                            "玩弄手部/文具", "觸摸臉部", "觸摸頭髮",
                            "目視他處", "趴睡", "喝水", "飲食"
                        ]
                        if primary_behavior_for_example in non_task_keywords and len(overall_behavior_summary["non_task_behavior_examples_for_summary"]) < 3:
                            try:
                                student_img_idx = hl_item.get("image_index_in_sequence", -1)
                                if 0 <= student_img_idx < len(batch_image_filenames):
                                    hl_filename = batch_image_filenames[student_img_idx]
                                    hl_timestamp = next((info["timestamp_str"] for info in image_batch_info if info["filename"] == hl_filename), "未知時間")
                                    context_desc = hl_item.get("context_description", "")
                                    
                                    # 即使此行為在統計上被歸為"無法判斷"，我們仍記錄其原始識別結果以供參考
                                    overall_behavior_summary["non_task_behavior_examples_for_summary"].append( 
                                        (hl_filename, hl_timestamp, ", ".join(behavior_list_original), context_desc) 
                                    )
                            except Exception as e_idx: print(f"提取非任務示例時出錯: {e_idx}")
    
    overall_behavior_stats_list, total_highlight_instances = [], sum(overall_behavior_summary["behavior_counts"].values())
    valence_summary = {"正向": 0, "負向": 0, "中性": 0}
    for behavior, count in overall_behavior_summary["behavior_counts"].items():
        percentage, avg_confidence = (count / total_highlight_instances * 100) if total_highlight_instances > 0 else 0, (overall_behavior_summary["behavior_confidence_sum"][behavior] / count) if count > 0 else 0
        valence = LABEL_TO_VALENCE.get(behavior, "未分類")
        if valence in valence_summary: valence_summary[valence] += count
        overall_behavior_stats_list.append({ "behavior_category": behavior, "valence": valence, "count": count, "percentage": round(percentage, 1), "average_confidence": round(avg_confidence, 2) })
    overall_behavior_stats_list.sort(key=lambda x: x["count"], reverse=True)
    personalized_notes = generate_personalized_summary_notes(student_id, overall_behavior_stats_list, overall_behavior_summary["non_task_behavior_examples_for_summary"], client)
    behavior_to_images_map = defaultdict(list)
    for result in all_results_from_threads:
        if "error" in result or "analysis" not in result or "error" in result["analysis"]: continue
        sequence_analysis, image_batch_info, filenames_in_batch = result["analysis"], result.get("image_batch_info", []), [info.get("filename") for info in result.get("image_batch_info", [])]
        if "per_image_highlights" in sequence_analysis and isinstance(sequence_analysis["per_image_highlights"], list):
            for highlight in sequence_analysis["per_image_highlights"]:
                # 【修改點 4】修改索引建立邏輯以處理行為列表
                behavior_list, image_index = highlight.get("behavior_category"), highlight.get("image_index_in_sequence")
                
                if not behavior_list or not isinstance(image_index, int) or not (0 <= image_index < len(filenames_in_batch)):
                    continue

                if not isinstance(behavior_list, list):
                    behavior_list = [behavior_list]
                
                image_filename = filenames_in_batch[image_index]
                if image_filename:
                    # 為列表中的每一個行為都建立索引
                    for behavior_category in behavior_list:
                        if image_filename not in behavior_to_images_map[behavior_category]:
                            behavior_to_images_map[behavior_category].append(image_filename)
    for behavior in behavior_to_images_map: behavior_to_images_map[behavior].sort()
    print("行為索引建立完成。")

    total_classified_instances = sum(valence_summary.values())
    valence_summary_with_percentage = { valence: { "count": count, "percentage": round((count / total_classified_instances * 100), 1) if total_classified_instances > 0 else 0 } for valence, count in valence_summary.items() }

    final_json_output = {
        "report_metadata": {
            "student_id": student_id, "student_number": student_number, "report_generation_time": "08/17", "student_image_source_folder": "英文",
            "teacher_position_source_json": os.path.basename(TEACHER_POSITION_JSON) if TEACHER_POSITION_JSON and os.path.isfile(TEACHER_POSITION_JSON) else "N/A",
            "classroom_view_source_folder": os.path.basename(CLASSROOM_IMAGES_FOLDER) if CLASSROOM_IMAGES_FOLDER and os.path.isdir(CLASSROOM_IMAGES_FOLDER) else "N/A",            "classroom_context": { "student_position": student_position },
            "analysis_settings": {
                "vision_model": VISION_DEPLOYMENT_NAME, "text_model": TEXT_DEPLOYMENT_NAME, "images_per_batch": IMAGES_PER_API_CALL, "context_images_per_batch_desc": "動態匹配老師和班級照片各一張", "confidence_threshold": BEHAVIOR_CONFIDENCE_THRESHOLD_FILTER,
                "cost_optimization": { "sampling_rate": SAMPLING_RATE, "image_detail": IMAGE_DETAIL_LEVEL } # 新增成本優化資訊
            }
        },
        "overall_summary": {
            "total_images_found": len(image_files_with_timestamps) * SAMPLING_RATE if SAMPLING_RATE > 1 else len(image_files_with_timestamps), # 顯示原始數量
            "total_images_analyzed": overall_behavior_summary["total_images_processed_in_batches"],
            "total_batches": len(image_batches), "valence_summary": valence_summary_with_percentage, "behavior_statistics": overall_behavior_stats_list, "behavior_to_images_index": behavior_to_images_map, "ai_summary_notes": personalized_notes
        },
        "detailed_sequence_analysis": all_sequence_analysis_results
    }

    current_timestamp_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_student_id_for_filename = re.sub(r'[\\/*?:"<>|]',"", student_id.replace(" ", "_"))
    json_filename = JSON_FILENAME_TEMPLATE.format(student_id=safe_student_id_for_filename, timestamp=current_timestamp_str)
    json_filepath = os.path.join(student_specific_folder_base, json_filename)
    try:
        with open(json_filepath, 'w', encoding='utf-8') as f: json.dump(final_json_output, f, ensure_ascii=False, indent=4)
        print(f"\n✅ 學生 '{student_id}' 的序列行為分析報告已成功儲存至: {json_filepath}")
    except Exception as e: print(f"❌ 錯誤：儲存 JSON 檔案時發生問題：{e}")
    
    print("\n--- 處理完成 ---")

if __name__ == "__main__":
    main()

### 課堂狀態預處理

In [26]:
# -*- coding: utf-8 -*-
import os
import json
import datetime
import re
from openai import AzureOpenAI, APIError, RateLimitError
from dotenv import load_dotenv

# ==============================================================================
# --- 設定 (請根據您的實際情況修改這三個變數) ---
# ==============================================================================

# 1. 您原始的、包含時間戳的逐字稿 .txt 檔案路徑
SOURCE_TRANSCRIPT_FILE = r'C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0928\老師\0928_3.txt' # 建議使用 0817.txt 來測試效果

# 2. 您希望儲存預處理結果的 .json 檔案路徑
OUTPUT_STATE_JSON_PATH = r'C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0928\老師\0928.json'

# 3. 這次課程的唯一標識符 (可自訂，會寫入JSON中)
CLASS_SESSION_ID = "0921_english_class_processed"

# ==============================================================================
# --- API 金鑰配置 ---
# ==============================================================================
load_dotenv()

AZURE_API_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
TEXT_DEPLOYMENT_NAME = os.getenv("CHAT_COMPLETION_NAME")

# ==============================================================================
# --- ★★★【課堂狀態定義 (v4 - 整合管理版)】★★★ ---
# ==============================================================================

CLASSROOM_STATES_DEFINITIONS = {
    "下課休息": """
【此為最高優先級狀態】。判斷依據如下，滿足任一條件即可成立：
1.  **明確指令**: 老師明確說出「下課」、「休息一下」、「給大家短休」、「五分鐘」等關鍵詞。此指令**立即觸發**此狀態的開始。
2.  **長時無關內容**: 出現【長達數分鐘】的【無教學內容時段】。此狀態**必須包含**那些穿插了如出現長達數分鐘的、明顯的非教學閒聊。等【非課堂錄音】的長段落。請將這些錄音視為教學的「中斷信號」。
""",
    "教師講解": """
老師是主要的發言者，進行任何形式的、連續的知識輸出。這是一個【涵蓋範圍極廣】的類別，它【囊括了所有】的講解活動，無論是【新知識傳授】（如文法、單字、課文分析），還是【舊知識複習】（如系統性地檢討習題、考卷）。只要是老師在主導講話，向學生傳遞教學內容，都應歸於此類。關鍵詞：「第一題」、「對答案」、「我們來看這邊」。
""",
    "學生練習": """
老師在【教師講解狀態】當中，讓學生練習課本、講義、考卷上的題目，課堂進入學生獨立操作時段。氣氛相對輕鬆，老師可能會巡視或回答個別問題。關鍵詞：「大家練習一下」、「給你們十分鐘寫寫看」。
""",
    "學生考試": """
老師宣布進行以【評量】為目的的測驗，【通常發生在課程開始時】。氣氛相對嚴肅，老師會強調紀律與規則。關鍵特徵：指令清晰（如：「現在開始考試」、「不要討論」），隨後是長時間的靜默。
""",
    "師生互動": """
【這是一個涵蓋教學與管理相關的廣泛類別】。它包含了所有非單向教學的、有特定目的的師生交流。包括：
1.  【教學相關】：針對課文或題目的問答與討論、老師為教學舉例而分享的相關故事。
2.  【管理相關】：宣布作業/考試範圍、公布成績、提醒課程規劃、給予學習方法建議等。
""",
    "閒聊": """
【與教學主題或課堂管理完全無關】的非正式交流。其特點是內容比較發散。包括：處理秩序、課前課後的寒暄、以及師生間與當前課程無關的對話。【注意】：此狀態『不包含』老師已宣布的休息時間。
""",
}


if not all([AZURE_API_KEY, AZURE_ENDPOINT, TEXT_DEPLOYMENT_NAME]):
    print("錯誤：缺少必要的 Azure OpenAI 環境變數。")
    print("請檢查您的 .env 檔案是否包含 AZURE_OPENAI_KEY, AZURE_OPENAI_ENDPOINT, 和 CHAT_COMPLETION_NAME。")
    exit()

try:
    client = AzureOpenAI(
        api_key=AZURE_API_KEY,
        azure_endpoint=AZURE_ENDPOINT,
        api_version="2024-02-01"
    )
    client.models.list()
    print("Azure OpenAI client 初始化並驗證成功。")
except Exception as e:
    print(f"初始化 Azure OpenAI Client 時發生錯誤: {e}")
    exit()

def clean_transcript(full_transcript):
    """
    (v2.0 強化版) 清理逐字稿，適應 "HH:MM:SS: [Text]" 新格式。
    1. 智慧推斷每段語音的結束時間。
    2. 根據時間戳間隔，準確標記長時間靜默。
    3. 將新格式轉換回 AI prompt 所需的 "start - end" 格式。
    """
    print("  -> 正在執行逐字稿預清理 (v2.0 - 新格式適配器)...")
    lines = full_transcript.strip().split('\n')
    
    time_format = '%H:%M:%S'
    
    # 步驟 1: 先將所有有效的行解析成一個中間結構
    intermediate_data = []
    for line in lines:
        # 使用新的正規表示式來匹配 "HH:MM:SS: [Text]"
        match = re.match(r'(\d{1,2}:\d{2}:\d{2}):\s*(.*)', line)
        if match:
            start_time_str, text = match.groups()
            text = text.strip()
            
            # 忽略無意義的、由噪音產生的單點行
            if text == '.':
                continue

            try:
                start_time_obj = datetime.datetime.strptime(start_time_str, time_format)
                intermediate_data.append({
                    "start_obj": start_time_obj,
                    "text": text
                })
            except ValueError:
                # 如果時間格式錯誤，跳過此行
                continue

    if not intermediate_data:
        print("  -> 警告：在逐字稿中未解析到任何有效的時間戳行。")
        return ""

    # 步驟 2: 遍歷中間結構，生成最終的、帶有 start-end 的逐字稿
    cleaned_lines = []
    # 處理第一行到倒數第二行
    for i in range(len(intermediate_data) - 1):
        current_entry = intermediate_data[i]
        next_entry = intermediate_data[i+1]
        
        current_start_obj = current_entry["start_obj"]
        next_start_obj = next_entry["start_obj"]
        
        # 推斷當前語句的結束時間，就是下一句的開始時間
        inferred_end_obj = next_start_obj
        
        # 格式化輸出
        start_str = current_start_obj.strftime(time_format)
        end_str = inferred_end_obj.strftime(time_format)
        cleaned_lines.append(f"{start_str} - {end_str}: {current_entry['text']}")
        
        # 檢查並插入長時間靜默標記
        silence_duration = (next_start_obj - inferred_end_obj).total_seconds()
        # 這裡的邏輯稍微改變：我們是檢查 "推斷的結束時間" 和 "下一句的實際開始時間"
        # 但在這個格式下，它們是相同的，所以我們要檢查的是 *兩句* 之間的間隔
        # 正確的邏輯應該是：下一句的開始時間 - 當前句的開始時間
        gap_duration = (next_start_obj - current_start_obj).total_seconds()
        
        # 如果兩句話的開始時間間隔超過 60 秒，就在它們之間插入靜默標記
        # 我們假設一句話的真實持續時間不會太長，所以大的間隔主要是靜默
        # 這裡我們用一個合理的持續時間，比如 30 秒，來計算靜默的起點
        if gap_duration > 60:
            silence_start_obj = current_start_obj + datetime.timedelta(seconds=30) # 假設說話最多持續30秒
            
            # 確保靜默的起點不會晚於下一句的開頭
            if silence_start_obj < next_start_obj:
                silence_minutes = round((next_start_obj - silence_start_obj).total_seconds() / 60)
                if silence_minutes > 0:
                    marker_start = silence_start_obj.strftime(time_format)
                    marker_end = next_start_obj.strftime(time_format)
                    cleaned_lines.append(f"{marker_start} - {marker_end}: [--- 長時間靜默 ({silence_minutes} 分鐘) ---]")


    # 處理最後一行，給它一個預設的持續時間，例如 30 秒
    if intermediate_data:
        last_entry = intermediate_data[-1]
        last_start_obj = last_entry["start_obj"]
        # 給予一個合理的預設持續時間
        last_end_obj = last_start_obj + datetime.timedelta(seconds=30)
        
        start_str = last_start_obj.strftime(time_format)
        end_str = last_end_obj.strftime(time_format)
        cleaned_lines.append(f"{start_str} - {end_str}: {last_entry['text']}")

    print("  -> 預清理完成 (新格式已成功適配)。")
    return "\n".join(cleaned_lines) 

# ==============================================================================
# --- ★★★【新增函式 2：後處理 AI 結果】★★★ ---
# ==============================================================================
def post_process_timeline(timeline, full_transcript_text):
    """
    [v4.1 兼容性增強版]
    在 AI 初步分析後，應用更強大的、基於專家知識的硬性規則來修正時間軸。
    此版本能兼容 AI 可能返回的不同鍵名（如 'start' vs 'start_time'），大幅提升穩健性。
    """
    if not timeline:
        return []

    print("  -> 正在執行AI結果後處理 (v4.1 - 兼容性增強版)...")
    
    time_format = '%H:%M:%S'

    # --- 步驟 1: 數據清理與標準化 (Sanitization) ---
    print("  -> 步驟 1/4: 清理並標準化AI返回的時間軸數據...")
    sanitized_timeline = []
    for entry in timeline:
        # ★★★【【【 核心修正點 】】】★★★
        # 增加兼容性處理：檢查 AI 是用了 'start_time' 還是 'start'
        start_time_str = entry.get('start_time') or entry.get('start')
        end_time_str = entry.get('end_time') or entry.get('end')
        state = entry.get('classroom_state') or entry.get('state')
        summary = entry.get('state_summary') or entry.get('summary', '') # summary 可能沒有，給個預設值

        if not start_time_str or not end_time_str or not state:
            print(f"  -> [警告]: 偵測到一個無效的時間段（缺少必要的時間戳或狀態），已跳過。內容: {entry}")
            continue
        
        try:
            start_seconds = (datetime.datetime.strptime(start_time_str, time_format) - datetime.datetime.strptime("00:00:00", time_format)).total_seconds()
            end_seconds = (datetime.datetime.strptime(end_time_str, time_format) - datetime.datetime.strptime("00:00:00", time_format)).total_seconds()
            
            # 建立一個結構統一的、乾淨的字典，並添加到列表中
            # 這樣可以確保後續所有步驟處理的都是我們期望的格式
            clean_entry = {
                'start_time': start_time_str,
                'end_time': end_time_str,
                'classroom_state': state,
                'state_summary': summary,
                'start_seconds': start_seconds,
                'end_seconds': end_seconds
            }
            sanitized_timeline.append(clean_entry)

        except (ValueError, KeyError) as e:
            print(f"  -> [警告]: 解析時間戳時出錯 '{start_time_str}' 或 '{end_time_str}'，已跳過。錯誤: {e}")
            continue
    
    if not sanitized_timeline:
        print("  -> [錯誤]: 標準化後 timeline 為空，後續處理無法進行。請檢查 AI 返回的原始 JSON 內容。")
        return []

    # --- 步驟 2: 強制修正「學生考試」時段 (後續邏輯不變) ---
    print("  -> 步驟 2/4: 正在應用【強制考試時段】規則...")
    exam_trigger_keywords = ["我們開始做以前測試", "考試", "測驗", "前測"]
    exam_start_index = -1
    
    transcript_lines = full_transcript_text.split('\n')
    for line in transcript_lines:
        if any(keyword in line for keyword in exam_trigger_keywords):
            match = re.search(r'(\d{2}:\d{2}:\d{2})', line)
            if match:
                start_time_str = match.group(1)
                try:
                    start_seconds = (datetime.datetime.strptime(start_time_str, time_format) - datetime.datetime.strptime("00:00:00", time_format)).total_seconds()
                    if start_seconds < 1800:
                        for j, entry in enumerate(sanitized_timeline):
                            if entry.get('start_time') == start_time_str:
                                exam_start_index = j
                                print(f"  -> 在 {entry.get('start_time')} 根據原始逐字稿偵測到考試觸發點。")
                                break
                        if exam_start_index != -1:
                            break
                except ValueError:
                    continue
    
    if exam_start_index != -1:
        sanitized_timeline[exam_start_index]['classroom_state'] = '學生考試'
        for i in range(exam_start_index + 1, len(sanitized_timeline)):
            current_entry = sanitized_timeline[i]
            duration = current_entry.get('end_seconds', 0) - current_entry.get('start_seconds', 0)
            if current_entry.get('classroom_state') in ["教師講解", "師生互動"] and duration > 180:
                print(f"  -> 偵測到明確教學活動於 {current_entry.get('start_time')} 開始，停止考試狀態覆蓋。")
                break
            if current_entry.get('classroom_state') != '學生考試':
                print(f"  -> [規則觸發]: 將 {current_entry.get('start_time')} 的 '{current_entry.get('classroom_state')}' 強制併入考試時段。")
                current_entry['classroom_state'] = '學生考試'

    # --- 步驟 3: 強制尋找並合併「下課休息」時段 (後續邏輯不變) ---
    print("  -> 步驟 3/4: 正在應用【強制下課時段】規則...")
    break_trigger_keywords = ["休息一下", "休息十分鐘", "消化一下", "短休"]
    break_trigger_index = -1
    
    for line in transcript_lines:
        if any(keyword in line for keyword in break_trigger_keywords):
            match = re.search(r'(\d{2}:\d{2}:\d{2})', line)
            if match:
                start_time_str = match.group(1)
                for j, entry in enumerate(sanitized_timeline):
                    if entry.get('start_time') == start_time_str:
                        break_trigger_index = j
                        print(f"  -> 在 {entry.get('start_time')} 根據原始逐字稿偵測到下課休息觸發點。")
                        break
                if break_trigger_index != -1:
                    break
            
    if break_trigger_index != -1:
        sanitized_timeline[break_trigger_index]['classroom_state'] = '下課休息'
        for i in range(break_trigger_index + 1, len(sanitized_timeline)):
            current_entry = sanitized_timeline[i]
            if current_entry.get('classroom_state') in ["閒聊"]:
                print(f"  -> [規則觸發]: 將 {current_entry.get('start_time')} 的 '{current_entry.get('classroom_state')}' 強制併入下課休息時段。")
                current_entry['classroom_state'] = '下課休息'
            elif current_entry.get('classroom_state') in ["教師講解", "師生互動"]:
                print(f"  -> 偵測到教學活動於 {current_entry.get('start_time')} 開始，停止下課休息合併。")
                break
    
    # --- 步驟 4: 合併連續的相同狀態並清理 (後續邏輯不變) ---
    print("  -> 步驟 4/4: 最終合併連續的相同狀態...")
    # 確保 sanitized_timeline 在合併前不為空
    if not sanitized_timeline:
        return []
    
    merged_timeline = [sanitized_timeline[0]]
    for i in range(1, len(sanitized_timeline)):
        last_entry = merged_timeline[-1]
        current_entry = sanitized_timeline[i]
        
        if last_entry.get('classroom_state') == current_entry.get('classroom_state'):
            last_entry['end_time'] = current_entry.get('end_time')
        else:
            merged_timeline.append(current_entry)

    for entry in merged_timeline:
        if 'start_seconds' in entry: del entry['start_seconds']
        if 'end_seconds' in entry: del entry['end_seconds']
            
    print("  -> 後處理與宏觀規則校正完成。")
    return merged_timeline
# ==============================================================================
# --- 核心函數 (已升級) ---
# ==============================================================================
def get_transcript_preprocessing_prompt():
    """定義用於預處理逐字稿的 AI 指令 (v15 - 強制摘要版)"""

    # 這裡我們直接將您另一份程式碼中精煉的定義借鑑過來
    # 這些定義更具體，邊界更清晰
    CLASSROOM_STATES_DEFINITIONS_V2 = {
        "下課休息": """
    【最高優先級與獨佔性規則】判斷依-據：
    1.  **觸發指令**: 當老師明確說出「下課」、「休息一下」、「休息十分鐘」等關鍵詞後，該狀態即啟動。
    2.  **合併原則**: 你必須將此觸發點之後【所有連續的、非教學性質】的片段（如閒聊、長時間靜默）全部合併進來，形成一個【持續至少5分鐘】的單一 `下課休息` 區塊。這是一個宏觀判斷，不要被零碎的對話打斷。
    """,
        "教師講解": """
    【核心定義】：老師作為主要發言者，圍繞【單一教學主題】（如一個文法點、一本書的解析）進行的、連續的知識輸出。
    【包含情境】：為了闡述觀念而引用的**簡短**故事、比喻或個人經驗（**在90秒內能重新連結回教學主題**）。穿插在講解中的簡短問答，如「懂嗎？」、「對不對？」。
    【排除情境】：長時間（**超過2分鐘**）的題外話應被切分為【閒聊】。系統性、逐題式的對答案應歸類為【師生互動】。
    """,
        "學生練習": """
    【觸發條件】：老師下達【明確的、要求學生獨立操作】的指令（如「大家練習一下」、「給你們幾分鐘寫」），且目的是為了鞏固剛教過的知識。
    【確認信號】：指令後通常會伴隨【長時間的教師靜默】或【低音量的同儕討論】。
    """,
        "學生考試": """
    ### ★★★ 核心規則強化 ★★★ ###
    【核心定義】：這是一個**持續性的狀態**，而非單一事件。必須同時滿足以下兩個條件：
    1.  **時間窗口**: **【高優先級規則】** 此狀態**幾乎只會出現在【課程開始的前30分鐘內】**。
    2.  **觸發與持續**:
        a. **觸發**: 老師明確說出「考試」、「前測」、「測驗」等指令性關鍵詞。
        b. **持續與鎖定**: 在該指令之後，出現了【長時間的、以無語音或極零碎話語為主】的時段。你必須將這些時段**宏觀地、統一地**標記為 `學生考試`。在此狀態期間，所有簡短的老師說明或學生問話，都應被視為考試的一部分，**不應中斷**此狀態。
    """,
        "師生互動": """
    【核心定義】：所有非單向教學的、具有【明確教學或管理目的】的雙向或多向交流。
    【明確包含】：結構化問答、**【高優先級】逐題檢討與對答案**、課堂管理（宣布作業、點名等）。
    【明確排除】：老師為了引出概念的自問自答式提問（屬於【教師講解】）。
    """,
        "閒聊": """
    【核心定義】：內容與【當前學科知識點的推進】及【課堂管理】完全無關的非正式交流。
    【明確包含】：長時間的個人故事（**超過2分鐘**或未連結回主題）、延伸的題外話。
    """
    }

    # 動態生成規則描述
    rules_description = ""
    for state, description in CLASSROOM_STATES_DEFINITIONS_V2.items():
        rules_description += f"\n---\n**{state}**\n{description.strip()}\n"

    return f"""
「你是一位頂尖的教育分析師，專長是從課堂對話中精準識別出教學活動的各個階段。你的任務是將一份完整的課堂逐字稿，切分成有意義、連續且分類明確的『課堂狀態』時間段。」

**【你的核心任務】**
分析使用者提供的【完整課堂逐字稿】，並以一個結構化的 JSON 物件作為輸出。你的分析必須涵蓋從頭到尾的整個逐字稿時間範圍。

**【課堂狀態的六個類別與判斷規則】**
你必須從以下六個類別中進行選擇，並嚴格遵循其定義：
{rules_description}
---

**【★★★ 分析策略：絕對優先級決策流程 ★★★】**
為達到最高的分類準確性，你必須嚴格遵循以下決策流程：

**第一步：情境鎖定 (Context Locking)**
- **考試情境**: 掃描課程前30分鐘。一旦偵測到明確的「考試」指令，立即將後續的長時間靜默或零碎對話鎖定為 `學生考試` 狀態，直到一個明確的新教學活動（如「好，我們來對答案」）開始。
- **休息情境**: 掃描整份逐字稿。一旦偵測到明確的「下課/休息」指令，立即將後續的閒聊和靜默時段鎖定並合併為 `下課休息` 狀態。

**第二步：內容相關性篩選 (Content Relevance Filter)**
- 對於未被鎖定的片段，你必須先問自己：「這段對話的核心主題是否與當前的英文教學目標直接相關？」
- 如果答案為【否】，則**必須**歸類為 **`閒聊`**。
- 如果答案為【是】，才進入第三步。

**第三步：教學形式判斷 (Instructional Form Analysis)**
- **單向知識輸出** -> `教師講解`。
- **雙向問答、對答案、課堂管理** -> `師生互動`。
- **要求學生獨立操作的指令後，出現的靜默** -> `學生練習`。

**【★★★ 最終輸出要求 (至關重要) ★★★】**
你的回答**必須**是一個結構完整的 JSON 物件，絕對不能包含任何額外的文字、註解或 Markdown 標記。
- 根物件需包含 `timeline` 鍵。
- `timeline` 是一個列表，其中每個元素都必須包含以下四個鍵：`start_time`, `end_time`, `classroom_state`, `state_summary`。
- `start_time` 和 `end_time` 必須是 "HH:MM:SS" 格式且時間必須連續。
- **`state_summary`**: 你【絕對必須】為每一個時間段，根據其對話內容，提供一句精煉、準確的總結。這個欄位【禁止】留空。
- 在生成最終 JSON 之前，務必再次檢查並合併所有連續且相同的狀態區塊。
"""

def preprocess_transcript():
    """
    [v3.0 強化版] 主執行函數，整合了預處理與後處理流程。
    """
    print("-" * 50)
    print("開始執行課堂逐字稿預處理 (v3.0 強化版)...")
    print(f"  - 來源檔案: {SOURCE_TRANSCRIPT_FILE}")
    print(f"  - 輸出檔案: {OUTPUT_STATE_JSON_PATH}")
    print("-" * 50)

    print("步驟 1/5: 讀取逐字稿檔案...")
    try:
        with open(SOURCE_TRANSCRIPT_FILE, 'r', encoding='utf-8') as f:
            full_transcript = f.read()
    except FileNotFoundError:
        print(f"❌ 錯誤：找不到逐字稿檔案：{SOURCE_TRANSCRIPT_FILE}")
        return
    except Exception as e:
        print(f"❌ 讀取檔案時發生錯誤: {e}")
        return

    if not full_transcript.strip():
        print("❌ 錯誤：逐字稿檔案是空的。")
        return
    print("  -> 讀取成功。")

    print("步驟 2/5: 預處理逐字稿，移除雜訊...")
    cleaned_transcript = clean_transcript(full_transcript)
    
    system_prompt = get_transcript_preprocessing_prompt()
    user_prompt = f"**【完整課堂逐字稿】**\n\n{cleaned_transcript}"

    print(f"步驟 3/5: 正在向 AI ({TEXT_DEPLOYMENT_NAME}) 發送請求，進行分析...")
    raw_content = ""
    try:
        response = client.chat.completions.create(
            model=TEXT_DEPLOYMENT_NAME,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=4096,
            temperature=0.0
        )
        
        raw_content = response.choices[0].message.content
        print("  -> AI 回應接收成功。")

        json_match = re.search(r'\{.*\}', raw_content, re.DOTALL)
        if not json_match:
            raise json.JSONDecodeError("在 AI 的回應中找不到有效的 JSON 物件。", raw_content, 0)
        
        json_string = json_match.group(0)
        parsed_data = json.loads(json_string)

        print("步驟 4/5: 後處理 AI 分析結果，提升連貫性...")
        timeline = parsed_data.get("timeline", [])
        processed_timeline = post_process_timeline(timeline, cleaned_transcript)
        parsed_data["timeline"] = processed_timeline

        print("步驟 5/5: 正在驗證數據的嚴謹性...")
        if not processed_timeline:
            print("  [⚠️ 驗證警告] - 處理後的 timeline 為空。")
        else:
            is_continuous = True
            for i in range(len(processed_timeline) - 1):
                current_end = processed_timeline[i].get("end_time")
                next_start = processed_timeline[i+1].get("start_time")
                if current_end != next_start:
                    print(f"  [❌ 驗證失敗] - 時間不連續！")
                    is_continuous = False
                    break
            
            if is_continuous:
                print("  -> 時間軸驗證通過。")
        
        parsed_data["class_session_id"] = CLASS_SESSION_ID
        parsed_data["processing_date"] = datetime.date.today().isoformat()

        output_dir = os.path.dirname(OUTPUT_STATE_JSON_PATH)
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
            
        with open(OUTPUT_STATE_JSON_PATH, 'w', encoding='utf-8') as f:
            json.dump(parsed_data, f, ensure_ascii=False, indent=2)
        
        print("-" * 50)
        print(f"✅ 逐字稿預處理完成！已儲存至: {OUTPUT_STATE_JSON_PATH}")
        print("-" * 50)

    except json.JSONDecodeError as e:
        print(f"❌ 錯誤：AI 回應的內容不是有效的 JSON 格式。 ({e})")
        print("---------- AI 原始回應 ----------")
        print(raw_content)
        print("---------------------------------")
    except (APIError, RateLimitError) as e:
        print(f"❌ API 錯誤: {e}")
    except Exception as e:
        print(f"❌ 處理過程中發生未知錯誤: {e}")

# ==============================================================================
# --- 執行入口 ---
# ==============================================================================
if __name__ == "__main__":
    preprocess_transcript()

Azure OpenAI client 初始化並驗證成功。
--------------------------------------------------
開始執行課堂逐字稿預處理 (v3.0 強化版)...
  - 來源檔案: C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0928\老師\0928_3.txt
  - 輸出檔案: C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0928\老師\0928.json
--------------------------------------------------
步驟 1/5: 讀取逐字稿檔案...
  -> 讀取成功。
步驟 2/5: 預處理逐字稿，移除雜訊...
  -> 正在執行逐字稿預清理 (v2.0 - 新格式適配器)...
  -> 預清理完成 (新格式已成功適配)。
步驟 3/5: 正在向 AI (gpt-4.1) 發送請求，進行分析...
  -> AI 回應接收成功。
步驟 4/5: 後處理 AI 分析結果，提升連貫性...
  -> 正在執行AI結果後處理 (v4.1 - 兼容性增強版)...
  -> 步驟 1/4: 清理並標準化AI返回的時間軸數據...
  -> 步驟 2/4: 正在應用【強制考試時段】規則...
  -> 在 00:00:00 根據原始逐字稿偵測到考試觸發點。
  -> [規則觸發]: 將 00:17:30 的 '閒聊' 強制併入考試時段。
  -> [規則觸發]: 將 00:21:00 的 '教師講解' 強制併入考試時段。
  -> 偵測到明確教學活動於 00:22:30 開始，停止考試狀態覆蓋。
  -> 步驟 3/4: 正在應用【強制下課時段】規則...
  -> 步驟 4/4: 最終合併連續的相同狀態...
  -> 後處理與宏觀規則校正完成。
步驟 5/5: 正在驗證數據的嚴謹性...
  -> 時間軸驗證通過。
--------------------------------------------------
✅ 逐字稿預處理完成！已儲存至: C:\Users\User\Desktop\test\SynologyDrive\image\上課

### 課程報告(AI課堂筆記)

In [12]:
import os
import re
import json
import base64
import datetime
import time 
# --- 修改點 1: 引入 AzureOpenAI ---
from openai import AzureOpenAI, APIError, RateLimitError 
from dotenv import load_dotenv
from tqdm import tqdm
from collections import defaultdict
import concurrent.futures
from PIL import Image
# ==================================================================
# ⚙️ 設定參數 (這部分不變)
# ==================================================================
LESSON_ID = "0907"
LESSON_SUBJECT = "英文"

# 2. 底下的 CONFIG 字典會根據上面的變數自動生成，無需手動修改
CONFIG = {
    "BLACKBOARD_IMAGES_FOLDER": rf"C:\Users\User\Desktop\test\note_blackboard\{LESSON_ID}_English_filtered",
    "TRANSCRIPT_FILE": rf"C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\{LESSON_ID}\老師\{LESSON_ID}_2.txt",
    "LESSON_DATE": f"{LESSON_ID[:2]}/{LESSON_ID[2:]}",
    "LESSON_SUBJECT": LESSON_SUBJECT,
    "OUTPUT_FOLDER": r"C:\Users\User\Desktop\test\note_json", # 輸出資料夾保持不變
    "CHUNK_MINUTES": 10
}

print("✅ 課程參數已自動設定完成：")
print(f"   - 課程ID: {LESSON_ID}")
print(f"   - 科目: {LESSON_SUBJECT}")
print(f"   - 板書路徑: {CONFIG['BLACKBOARD_IMAGES_FOLDER']}")
print(f"   - 逐字稿路徑: {CONFIG['TRANSCRIPT_FILE']}")
print("-" * 20)

CONTENT_THRESHOLD = 8000 

# --- 核心輔助函數 (有修改) ---

def clean_json_string(json_string):
    """
    清理 AI 回傳的 JSON 字串 (移除 Markdown 標記)
    """
    if not json_string:
        return "{}"
    
    # 移除 Markdown 的 ```json 包裹
    if "```json" in json_string:
        json_string = json_string.replace("```json", "").replace("```", "")
    elif "```" in json_string:
        json_string = json_string.replace("```", "")
    
    return json_string.strip()

def ai_screen_image_relevance(image_path: str, client: AzureOpenAI) -> bool:
    """
    一個高度專業化的 AI 函式，其唯一工作是判斷單張圖片是否包含足夠的教學內容。
    它不讀取任何逐字稿，只進行視覺評估。
    """
    deployment_name = os.getenv("CHAT_COMPLETION_NAME")
    if not deployment_name:
        raise ValueError("錯誤：.env 檔案中缺少 CHAT_COMPLETION_NAME 設定。")

    system_prompt = """
    你是一位嚴格的圖片審查員，你的唯一任務是判斷一張黑板照片是否包含【足夠的教學筆記】。
    你不需要理解筆記的內容，只需要評估其【資訊密度】。

    **判斷標準：**
    - **相關 (relevant):** 黑板上有清晰、足夠數量的文字、圖表或公式，看起來像是一個教學階段的總結。
    - **不相關 (irrelevant):** 黑板符合以下任一情況：空白、幾乎空白、老師正在擦黑板、只有一兩個詞的標題、內容極度稀疏、完全模糊。

    你的回答必須是一個只有單一鍵的 JSON 物件：
    ```json
    {
      "is_relevant": <true 或 false>
    }
    ```
    """
    
    b64_image = encode_image_to_base64(image_path)
    if not b64_image:
        return False # 圖片編碼失敗，視為不相關

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": b64_image}}
        ]}
    ]

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=messages,
            response_format={"type": "json_object"},
            temperature=0.0,
            max_tokens=150
        )
        result = json.loads(response.choices[0].message.content)
        return result.get("is_relevant", False)
    except Exception as e:
        print(f"⚠️ 警告：AI 圖片預審失敗 for {os.path.basename(image_path)}: {e}")
        return False

def is_image_content_sufficient(image_path, threshold):
    """
    使用 Pillow 分析圖片，判斷其內容是否足夠。
    返回 True 如果內容足夠，否則返回 False。
    """
    try:
        with Image.open(image_path).convert('L') as img: # 轉為灰階
            # 設定一個閾值來區分背景和文字。假設黑板是暗色，粉筆是亮色。
            # 像素值 > 50 的被視為文字(白色)，其餘為背景(黑色)。
            binary_img = img.point(lambda p: 255 if p > 50 else 0, '1')
            
            # 計算文字像素（白色像素）的數量
            content_pixels = sum(1 for pixel in binary_img.getdata() if pixel == 255)

            return content_pixels > threshold
    except Exception as e:
        print(f"⚠️ 警告：分析圖片 {os.path.basename(image_path)} 時發生錯誤: {e}")
        return False # 如果圖片損壞或無法讀取，也視為無效

# --- 修改點 2: 重構客戶端初始化函數以使用 Azure ---
def load_openai_client():
    """加載環境變數並初始化 Azure OpenAI 客戶端"""
    load_dotenv()
    
    # 從 .env 檔案讀取您的 Azure 變數
    azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
    api_key = os.getenv("AZURE_OPENAI_KEY")
    # 為 Azure API 設定一個固定的、推薦的版本號
    # 即使 .env 中沒有，程式也能正常運作
    api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2024-02-15-preview")

    if not azure_endpoint or not api_key:
        raise ValueError(
            "錯誤：.env 檔案中缺少 AZURE_OPENAI_ENDPOINT 或 AZURE_OPENAI_KEY。\n"
            "請確保這兩個變數已正確設定。"
        )

    try:
        # 使用 AzureOpenAI 類別來初始化客戶端
        client = AzureOpenAI(
            api_key=api_key,
            azure_endpoint=azure_endpoint,
            api_version=api_version
        )
        client.models.list() 
        print("✅ Azure OpenAI 客戶端初始化並驗證成功。")
        return client
    except APIError as e:
        raise ConnectionError(f"❌ Azure OpenAI API 錯誤: {e}")
    except Exception as e:
        raise Exception(f"❌ 初始化 Azure OpenAI 客戶端時發生未知錯誤: {e}")

def get_timestamp_from_filename(filename, pattern):
    """
    從檔名解析時間戳 (舊版函式，load_blackboard_images 需要用到)
    """
    import re
    match = re.search(pattern, filename)
    if match:
        h, m, s = map(int, match.groups())
        return datetime.timedelta(hours=h, minutes=m, seconds=s)
    return None

def get_timestamp_from_path(path):
    """
    從檔案路徑解析出時間戳 (datetime.timedelta)。
    假設檔名格式包含：01h05m30s
    """
    import re
    match = re.search(r'(\d+)h(\d{2})m(\d{2})s', path)
    if match:
        h, m, s = map(int, match.groups())
        return datetime.timedelta(hours=h, minutes=m, seconds=s)
    return datetime.timedelta(0)

def encode_image_to_base64(image_path):
    try:
        with open(image_path, "rb") as image_file:
            return f"data:image/png;base64,{base64.b64encode(image_file.read()).decode('utf-8')}"
    except Exception as e:
        print(f"警告：無法編碼圖片 {image_path}: {e}")
        return None

def parse_transcript_line(line):
    # 修改點 1: 更新正規表示式，以匹配 "時間戳: 內容" 的格式
    pattern = re.compile(r"(\d{2}:\d{2}:\d{2}): (.*)")
    match = pattern.match(line)
    if match:
        # 修改點 2: 只獲取兩個群組 (時間戳, 內容)
        start_str, content = match.groups()
        h, m, s = map(int, start_str.split(':'))
        start_td = datetime.timedelta(hours=h, minutes=m, seconds=s)

        # 修改點 3: 處理 end_time
        # 由於新格式沒有結束時間，我們將其設定為與開始時間相同。
        # 因為後續分析是以10分鐘為單位的大區塊，單行精確的結束時間影響不大。
        end_td = start_td

        return {'start_time': start_td, 'end_time': end_td, 'content': content.strip()}
    return None

def load_blackboard_images(folder_path):
    images = []
    pattern = r'(\d+)h(\d{2})m(\d{2})s'
    print(f"正在掃描黑板照片資料夾: {folder_path}...")
    if not os.path.isdir(folder_path):
        print(f"警告：找不到資料夾 {folder_path}，將不處理黑板照片。")
        return []
    for filename in sorted(os.listdir(folder_path)):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            timestamp = get_timestamp_from_filename(filename, pattern)
            if timestamp:
                images.append({'timestamp': timestamp, 'path': os.path.join(folder_path, filename)})
    print(f"✔️ 成功加載並索引了 {len(images)} 張黑板照片。")
    return images

def load_and_process_transcripts(file_path):
    transcript_entries = []
    total_duration = datetime.timedelta(0)
    print(f"正在加載逐字稿檔案: {file_path}...")
    if not os.path.isfile(file_path):
        print(f"警告：找不到指定的逐字稿檔案 {file_path}，無法處理逐字稿。")
        return [], total_duration
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        if not lines:
            print(f"警告：逐字稿檔案 {file_path} 是空的。")
            return [], total_duration
        for line in lines:
            entry = parse_transcript_line(line)
            if entry:
                transcript_entries.append(entry)
                total_duration = max(total_duration, entry['end_time'])
    print(f"✔️ 成功加載並處理了 {len(transcript_entries)} 行逐字稿，總時長: {total_duration}。")
    return transcript_entries, total_duration

def filter_redundant_images(images_list, time_threshold_seconds=180, max_images_per_block=2):
    """
    過濾邏輯核心：
    1. 時間相近的圖片 (預設 3 分鐘內)，只保留「較晚」的那一張 (假設越晚黑板字越多)。
    2. 每個區塊最多只保留最後 max_images_per_block 張。
    """
    if not images_list:
        return []

    # 1. 確保按時間排序
    # 我們先為每張圖加上 parsed_time 方便計算
    for img in images_list:
        if 'parsed_time' not in img:
            img['parsed_time'] = get_timestamp_from_path(img['path'])
    
    sorted_imgs = sorted(images_list, key=lambda x: x['parsed_time'])
    
    unique_images = []
    if sorted_imgs:
        current_keeper = sorted_imgs[0]
        
        for i in range(1, len(sorted_imgs)):
            next_img = sorted_imgs[i]
            time_diff = (next_img['parsed_time'] - current_keeper['parsed_time']).total_seconds()
            
            if time_diff < time_threshold_seconds:
                # 兩張圖片太接近 (< 3分鐘)，視為同一組板書
                # 策略：保留「後面」那一張 (next_img)，因為通常板書是累積的
                current_keeper = next_img 
            else:
                # 時間間隔夠久，確認收錄目前的 keeper，並開始新的一組
                unique_images.append(current_keeper)
                current_keeper = next_img
        
        # 別忘了加入最後一張
        unique_images.append(current_keeper)

    # 2. 數量上限控制 (只取最後 N 張，因為通常最後的總結最完整)
    # 當然也可以取前 N 張，視您的偏好。這裡設定取最後 2 張。
    return unique_images[-max_images_per_block:]

def place_images_with_ai(text_blocks: list, images_to_place: list, client: AzureOpenAI) -> list:
    """
    【第二步：插畫師 AI + 邏輯過濾 + Python 組裝】
    修復：在 System Prompt 中明確加入 "JSON" 關鍵字，以符合 API 要求。
    """
    if not images_to_place:
        return text_blocks 

    deployment_name = os.getenv("CHAT_COMPLETION_NAME")
    
    # 1. 準備給 AI 的文字摘要
    formatted_text_for_ai = ""
    for i, block in enumerate(text_blocks):
        # 只取前 200 字給 AI 判斷即可，節省 Token
        content_snippet = block['content'][:200] + "..." if len(block['content']) > 200 else block['content']
        formatted_text_for_ai += f"--- BLOCK {i} ---\n{content_snippet}\n\n"

    # --- 🚨 關鍵修改點：必須包含 "JSON" 這個字 ---
    system_prompt = """
    你是一位圖片排版助手。請判斷黑板照片最適合插入在下方哪個文字區塊的「後面」。
    
    **回傳格式規定 (JSON Only)：**
    請務必回傳一個合法的 JSON 物件。
    格式範例：{"insert_after_block": 數字編號}
    若無適合位置，請回傳：{"insert_after_block": -1}
    """
    
    # 使用 defaultdict 暫存所有 AI 的建議位置
    raw_insertions = defaultdict(list)

    print(f"🎨 (Step 5) 插畫師 AI 正在為 {len(images_to_place)} 張圖片定位...")
    
    # 2. 呼叫 AI 決定圖片位置
    for image_info in tqdm(images_to_place, desc="  └── AI 定位中"):
        b64_image = encode_image_to_base64(image_info['path'])
        if not b64_image: continue

        user_prompt = f"這是一張黑板照片。請判斷它最適合插入到下列哪個文字區塊的後面？\n\n{formatted_text_for_ai}"
        
        try:
            response = client.chat.completions.create(
                model=deployment_name,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": [
                        {"type": "text", "text": user_prompt},
                        {"type": "image_url", "image_url": {"url": b64_image}}
                    ]}
                ],
                response_format={"type": "json_object"},
                temperature=0.0,
                max_tokens=100
            )
            # 這裡也建議加上 clean_json_string 以防萬一
            clean_content = clean_json_string(response.choices[0].message.content)
            result = json.loads(clean_content)
            
            block_index = result.get("insert_after_block", -1)

            if isinstance(block_index, int) and 0 <= block_index < len(text_blocks):
                raw_insertions[block_index].append({
                    "type": "image",
                    "path": image_info["path"],
                    "description": image_info["description"]
                })
        except Exception as e:
            print(f"⚠️ 圖片定位失敗: {e}")

    # 3. 應用過濾邏輯，消除重複圖片
    final_insertions = {}
    for block_idx, img_list in raw_insertions.items():
        # 呼叫過濾函式
        filtered_list = filter_redundant_images(img_list)
        
        # 移除暫存的 parsed_time 欄位，保持 JSON 乾淨
        for img in filtered_list:
            if 'parsed_time' in img:
                del img['parsed_time']
        
        final_insertions[block_idx] = filtered_list

    # 4. 組裝最終列表
    final_content_blocks = []
    for i, text_block in enumerate(text_blocks):
        final_content_blocks.append(text_block) # 加入文字
        if i in final_insertions:
            final_content_blocks.extend(final_insertions[i]) # 加入過濾後的圖片

    return final_content_blocks

def generate_pure_vision_caption(image_path, client):
    """
    【純視覺通道】
    只傳送圖片給 AI，不傳送逐字稿。
    目的：強迫 AI 只能根據視覺像素生成描述，徹底杜絕因閱讀逐字稿而產生的幻覺。
    """
    deployment_name = os.getenv("CHAT_COMPLETION_NAME")
    b64_image = encode_image_to_base64(image_path)
    
    system_prompt = """
    你是一位嚴格的【黑板抄寫員】。你的唯一任務是將黑板上的內容轉述出來。
    
    **規則：**
    1. **所見即所得**：只描述你肉眼能清晰看到的文字、圖表或例句。
    2. **禁止腦補**：如果黑板沒寫，就算你覺得有關聯，也絕對不要寫出來。
    3. **格式**：
       - 先用一句話總結黑板主題（例如：黑板正在講解 Worry 的動詞用法）。
       - 接著列出黑板上出現的關鍵例句（英文）。
       - 如果有文法標記（如 V-ing, adj.），請一併記錄。
    """
    
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": [
                    {"type": "text", "text": "請描述這張黑板圖片的內容。"},
                    {"type": "image_url", "image_url": {"url": b64_image}}
                ]}
            ],
            temperature=0.0, # 溫度歸零，確保客觀
            max_tokens=500
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"⚠️ 圖片描述生成失敗: {e}")
        return "無法辨識黑板內容"

def analyze_content_chunk(transcript_text, image_path, client, max_retries=3, initial_delay=5):
    """
    V8.2 放寬版 (修復 JSON Error)：
    1. 呼叫純視覺描述函式。
    2. 提取主題時，採取「廣納」策略。
    3. [修復] 在 System Prompt 中明確加入 "JSON" 關鍵字以符合 API 要求。
    """
    deployment_name = os.getenv("CHAT_COMPLETION_NAME")
    if not deployment_name: raise ValueError("錯誤：缺少 CHAT_COMPLETION_NAME。")

    # --- 步驟 A: 獲取圖片描述 ---
    image_description = None
    if image_path:
        image_description = generate_pure_vision_caption(image_path, client)

    # --- 步驟 B: 提取主題 ---
    # 修改點：加入 "JSON" 關鍵字與格式範例
    system_prompt = """
    你是一位內容分析師。你的任務是從逐字稿中提取該時段的「主要討論話題」。
    
    請提取 1-3 個主題，主題可以是：
    1. 英文教學內容 (文法、單字、句型) - 這是最重要的。
    2. 課堂活動 (閱讀、討論)。
    3. 老師分享的故事或觀點。

    **輸出格式規定 (JSON Only)：**
    請務必回傳一個合法的 JSON 物件，格式如下：
    ```json
    {
      "topics": [
        {
          "topic_name": "主題名稱 (例如：Worry 的用法)",
          "summary": "該主題的簡短摘要"
        }
      ]
    }
    ```
    """
    
    user_content = f"請分析以下逐字稿片段，提取主要話題：\n\n{transcript_text}"
    if image_description:
        user_content += f"\n\n【黑板圖片內容參考】：\n{image_description}"
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=deployment_name,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_content}
                ],
                response_format={"type": "json_object"},
                temperature=0.1,
            )
            result = json.loads(clean_json_string(response.choices[0].message.content))
            result["image_description"] = image_description
            return result

        except Exception as e:
            print(f"\n❌ 分析區塊時發生錯誤: {e} (第 {attempt + 1}/{max_retries} 次)")
            time.sleep(initial_delay)

    return {"image_description": image_description, "topics": [{"topic_name": "分析失敗", "summary": "API調用失敗"}]}

def cluster_topics_with_ai(topic_list: list[str], client: AzureOpenAI) -> dict | None:
    """
    AI 總編輯 (V9.0 精準命名版)：
    1. 強力過濾雜訊。
    2. 【核心修改】強制使用「具體文法術語」作為標題，拒絕籠統名稱。
    """
    deployment_name = os.getenv("CHAT_COMPLETION_NAME")
    if not deployment_name: raise ValueError("錯誤：缺少 CHAT_COMPLETION_NAME。")

    system_prompt = """
    你是一位台灣國中英文補習班的【王牌教材編輯】。
    你的任務是將一堆零散的教學標籤，整理成 4-6 個「含金量極高」的複習單元。

    **1. 命名規則 (Naming Convention) - 這是最重要的！**
    *   **🚫 絕對禁止籠統標題**：
        - 不可以使用「文法教學」、「句型結構」、「單字解析」、「重要考點」這種廢話當標題。
        - 學生看到標題必須馬上知道今天要考什麼。
    *   **✅ 必須使用「具體文法術語」**：
        - 標題必須包含具體的關鍵字，例如：「情緒分詞 (V-ing/V-ed)」、「名詞子句 (Noun Clauses)」、「連綴動詞」、「關係代名詞」。
    *   **✅ 推薦格式 (主題 + 細節)**：
        - "情緒動詞：Worry 與 Surprise 的用法差異"
        - "分詞構句：形容詞 -ing 與 -ed 的判斷技巧"
        - "名詞子句：That 引導詞的省略規則"

    **2. 過濾規則 (Filter Rules)：**
    - 請無情地刪除所有與「英文學術知識」無關的標籤（如：閒聊、當兵故事、麥克風測試、翻頁、休息時間、人生道理）。

    **3. 歸納邏輯：**
    - 將相關的標籤合併。例如看到 "boring", "excited", "interest" 請合併為 "情緒形容詞與分詞用法"。
    - 最終產出 **4 到 6 個** 單元。

    **輸出格式 (JSON)：**
    ```json
    {
        "具體文法標題 A": ["原始標籤1", "原始標籤2"],
        "具體文法標題 B": ["原始標籤3", "原始標籤4"]
    }
    ```
    """
    
    topics_text = "\n".join(f"- {topic}" for topic in topic_list)
    
    print("\n🧠 (Step 3) 啟動 AI 總編輯 (V9.0 精準命名版)...")
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"請重新編排以下標籤，給出最精準、像講義目錄一樣的標題：\n\n{topics_text}"}
            ],
            response_format={"type": "json_object"},
            temperature=0.1, # 稍微提高一點點溫度，讓它在命名上有點創意，但不要太高
        )
        return json.loads(clean_json_string(response.choices[0].message.content))
    except Exception as e:
        print(f"\n❌ AI 主題聚類錯誤: {e}")
        return None

def refine_knowledge_hub_final(raw_hub_json_str: str, client: AzureOpenAI) -> dict | None:
    """
    【寫手 AI (V5.3 最終修復版)】
    1. 修復 400 Error: 在 Prompt 中明確加入 "JSON" 關鍵字。
    2. 保留圖文穿插優化 (區塊切碎)。
    3. 保留移除 ### 標題的邏輯。
    """
    deployment_name = os.getenv("CHAT_COMPLETION_NAME")
    if not deployment_name: raise ValueError("錯誤：缺少 CHAT_COMPLETION_NAME。")

    try:
        data_temp = json.loads(raw_hub_json_str)
        current_topic = data_temp.get("main_topic", "英文課程")
    except:
        current_topic = "英文課程"

    system_prompt = f"""
    你是一位擁有 20 年經驗的**台灣國中英語王牌教師**。你正在為國三學生編寫一份關於「{current_topic}」的課後補救教學筆記。

    **你的教學標準：**
    1.  **難度控制 (CEFR A2-B1)**：針對台灣國中會考難度，使用學生聽得懂的邏輯。
    2.  **還原例句**：引用老師講過的具體例句。
    3.  **會考陷阱**：指出學生容易錯的地方。

    **【關鍵指令：結構顆粒化 (Granularity)】**
    為了讓黑板照片能穿插在文字中間，**請絕對不要將所有內容寫在同一個 block 裡！**
    請根據「子主題」將內容拆分成 3~5 個獨立的 text blocks。

    **【格式禁令】**
    1. **禁止使用 `###` 或 `##` 標題符號**。請改用 **粗體** (例如：**1. 核心觀念**) 來標示標題。
    2. 內容必須使用 Markdown 排版。

    **【輸出格式規定 (JSON)】**
    請務必回傳一個合法的 **JSON** 物件 (JSON Object)，格式如下：
    {{
        "main_topic": "{current_topic}",
        "content_blocks": [
            {{ "type": "text", "content": "**1. 子標題**\\n內容..." }},
            {{ "type": "text", "content": "**2. 子標題**\\n內容..." }}
        ]
    }}
    """

    # --- Python 處理 JSON ---
    try:
        data_for_refining = json.loads(raw_hub_json_str)
        transcript_text = data_for_refining.get("full_transcript", "")
    except json.JSONDecodeError:
        return None

    print(f"\n✍️ (Step 4) 啟動寫手 AI (圖文穿插優化版)，正在撰寫: {current_topic}...")
    
    try:
        user_prompt = f"""
        請根據以下逐字稿，撰寫教學筆記。
        
        **輸出要求：**
        1. 請以 **JSON** 格式輸出。
        2. 請將內容拆分成 3~5 個獨立的 content_blocks。
        3. **不要**使用 `###`，請用 **粗體** 代替。
        
        ---\n逐字稿內容：\n{transcript_text}\n---
        """

        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.1, 
            max_tokens=3500
        )
        
        raw_response_content = clean_json_string(response.choices[0].message.content)
        refined_data = json.loads(raw_response_content)

        # --- 自動修復邏輯 (保持不變) ---
        if isinstance(refined_data, dict) and "content_blocks" in refined_data:
            fixed_blocks = []
            for block in refined_data["content_blocks"]:
                if "content" in block:
                    # 再次確保沒有 ### 殘留
                    block["content"] = block["content"].replace("### ", "").replace("## ", "")
                    fixed_blocks.append(block)
                else:
                    # 如果 AI 還是產生了分散欄位，手動組合
                    concept = block.get("Concept", block.get("核心觀念", ""))
                    examples = block.get("Examples", block.get("老師的黑板例句", ""))
                    trap = block.get("Exam Trap", block.get("會考陷阱/易錯點", ""))
                    
                    combined_content = ""
                    if concept: combined_content += f"**核心觀念**\n{concept}\n\n"
                    if examples: combined_content += f"**📝 老師例句：**\n{examples}\n\n"
                    if trap: combined_content += f"**⚠️ 會考陷阱：**\n{trap}\n"
                    
                    if combined_content.strip():
                        fixed_blocks.append({"type": "text", "content": combined_content})
            
            refined_data["content_blocks"] = fixed_blocks

        if not isinstance(refined_data, dict) or "content_blocks" not in refined_data:
            return None
        
        return refined_data
        
    except Exception as e:
        print(f"\n❌ AI 筆記生成失敗: {e}")
        return None

def main(config): 
    """主函數， orchestrate 整個流程"""
    
    print("🚀 開始建立主題式知識庫...")
    
    # 1. 初始化 & 2. 加載數據
    client = load_openai_client()
    blackboard_images = load_blackboard_images(config["BLACKBOARD_IMAGES_FOLDER"])
    transcripts, total_duration = load_and_process_transcripts(config["TRANSCRIPT_FILE"])
    
    if not transcripts:
        print("❌ 錯誤：沒有找到任何有效的逐字稿內容，程式終止。")
        return

    # 【新增】步驟 2.5：AI 視覺預審所有圖片
    print(f"\n🤖 啟動 AI 視覺預審員，檢查 {len(blackboard_images)} 張圖片的內容密度...")
    relevant_images = []
    if blackboard_images: # 只有在找到圖片時才執行
        with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
            future_to_img = {executor.submit(ai_screen_image_relevance, img['path'], client): img for img in blackboard_images}
            for future in tqdm(concurrent.futures.as_completed(future_to_img), total=len(blackboard_images), desc="AI 預審圖片中"):
                img = future_to_img[future]
                try:
                    if future.result():
                        relevant_images.append(img)
                except Exception as exc:
                    print(f"預審 {os.path.basename(img['path'])} 產生錯誤: {exc}")
    
    print(f"✔️ AI 預審完成！從 {len(blackboard_images)} 張圖片中篩選出 {len(relevant_images)} 張有效板書。")

    # 3. 內容分塊與分析
    chunk_minutes = config.get("CHUNK_MINUTES", 10) # 恢復為 10 分鐘
    chunk_duration = datetime.timedelta(minutes=chunk_minutes)
    if total_duration.total_seconds() == 0:
        print("❌ 錯誤：逐字稿總時長為 0，無法進行分塊。")
        return
    num_chunks = int(total_duration / chunk_duration) + 1
    print(f"\n🔬 將課程內容分為 {num_chunks} 個時間區塊 (每塊 {chunk_minutes} 分鐘) 進行分析...")

    tasks_to_run = []
    for i in range(num_chunks):
        chunk_start_time = i * chunk_duration
        chunk_end_time = (i + 1) * chunk_duration
        chunk_transcript_lines = [entry['content'] for entry in transcripts if chunk_start_time <= entry['start_time'] < chunk_end_time]
        chunk_transcript_text = "\n".join(chunk_transcript_lines).strip()
        if not chunk_transcript_text: continue

        best_image_path = None
        if relevant_images:
            chunk_mid_point = chunk_start_time + (chunk_duration / 2)
            min_diff = datetime.timedelta.max
            for img in relevant_images:
                diff = abs(img['timestamp'] - chunk_mid_point)
                if diff < min_diff and diff < chunk_duration: 
                    min_diff = diff
                    best_image_path = img['path']
        
        tasks_to_run.append((chunk_transcript_text, best_image_path, client))

    # 平行處理主要分析任務
    all_analysis_results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        future_to_task = {executor.submit(analyze_content_chunk, *task): i for i, task in enumerate(tasks_to_run)}
        for future in tqdm(concurrent.futures.as_completed(future_to_task), total=len(tasks_to_run), desc="分析課堂內容區塊"):
            try:
                result = future.result()
                all_analysis_results.append(result)
            except Exception as exc:
                print(f'\n❌ 一個分析任務產生了錯誤: {exc}')
                all_analysis_results.append({"image_description": None, "topics": [{"topic_name": "分析失敗", "summary": str(exc)}]})
    
    # 4. 將分析結果聚合成初步知識庫
    print("\n🔄 正在將時間序分析結果聚合成【初步】主題知識庫...")
    knowledge_hub_initial = defaultdict(lambda: {"teaching_segments": []})
    
    debug_topics_found = 0 # 計數器

    for i, (transcript_text, image_path, _) in enumerate(tasks_to_run):
        analysis_res = all_analysis_results[i] if i < len(all_analysis_results) else {}
        topics_list = analysis_res.get("topics", [])
        
        if not isinstance(topics_list, list): continue

        for topic in topics_list:
            if not isinstance(topic, dict): continue
            topic_name = topic.get("topic_name", "未分類主題")
            
            # --- 修改點：移除這裡的 Python 硬編碼過濾 ---
            # 我們把過濾的工作完全交給後面的 AI 總編輯 (Step 5)
            # if any(keyword in topic_name for keyword in ["閒聊", "管理", "失敗", "過濾"]):
            #     continue 
            
            # 簡單過濾掉明顯錯誤的分析
            if "分析失敗" in topic_name:
                continue

            image_info = None
            # 只有當 AI 有生成有效的圖片描述時才加入圖片
            if image_path and analysis_res.get("image_description"):
                image_info = {
                    "path": image_path,
                    "description": analysis_res.get("image_description", "無描述")
                }

            knowledge_hub_initial[topic_name]["teaching_segments"].append({
                "summary_of_segment": topic.get("summary", ""),
                "relevant_transcript": transcript_text,
                "relevant_blackboard_image_info": image_info 
            })
            debug_topics_found += 1

    print(f"📊 初步分析共捕捉到 {len(knowledge_hub_initial)} 個原始主題，包含 {debug_topics_found} 個片段。")

    # 5. 進行 AI 主題聚類
    if not knowledge_hub_initial:
        print("⚠️ 初步分析未能生成任何有效主題，跳過後續步驟。")
        clustered_topics = None
    else:
        initial_topic_list = list(knowledge_hub_initial.keys())
        clustered_topics = cluster_topics_with_ai(initial_topic_list, client)

    # 6. 根據聚類結果，重組資料並進行深化精煉
    refined_knowledge_hub_list = []
    used_image_paths = set()

    if clustered_topics:
        print("\n📝 根據主題分類，進行最終圖文整合精煉...")
        initial_hub_dict = dict(knowledge_hub_initial) 

        for new_main_topic, original_topics in tqdm(clustered_topics.items(), desc="最終精煉主題"):
            
            all_transcripts_raw = []
            all_images_for_this_topic = []
            for original_topic in original_topics:
                if original_topic in initial_hub_dict:
                    for segment in initial_hub_dict[original_topic]["teaching_segments"]:
                        all_transcripts_raw.append(segment["relevant_transcript"])
                        img_info = segment.get("relevant_blackboard_image_info")
                        if img_info and img_info.get("path") not in used_image_paths:
                            all_images_for_this_topic.append(img_info)
            
            # --- 【【【 新增：內容預過濾步驟 】】】 ---
            sensitive_keywords = [
                "保險套", "內褲", "長毛象", "廁所幹嘛", "露出來", "褲子拉起來",
                "秦始皇", "中華民國", "大秦帝國", "孫子兵法", "大流氓", "黨的",
                "屌", "幹嘛", "神經病", "智障" # 增加一些常見口語詞
            ]
            
            full_transcript_text = "\n".join(all_transcripts_raw)
            for keyword in sensitive_keywords:
                full_transcript_text = full_transcript_text.replace(keyword, "[...]")

            data_for_text_refining = {
                "main_topic": new_main_topic,
                "full_transcript": full_transcript_text
            }
            
            # 【第 1 步】呼叫寫手 AI，生成純文字筆記
            text_only_result = refine_knowledge_hub_final(json.dumps(data_for_text_refining, ensure_ascii=False), client)

            if not text_only_result:
                continue 

            # 【第 2 步】呼叫插畫師 AI，將圖片插入文字區塊中
            final_content_blocks = place_images_with_ai(
                text_blocks=text_only_result.get("content_blocks", []),
                images_to_place=all_images_for_this_topic,
                client=client
            )
            
            complete_topic = {
                "main_topic": new_main_topic,
                "content_blocks": final_content_blocks
            }

            unwanted_topics = ["健康與生活技能", "家庭與教育", "個人成長", "生活、歷史與社會現象應用", "心理健康"]
            if not any(unwanted in new_main_topic for unwanted in unwanted_topics):
                refined_knowledge_hub_list.append(complete_topic)
                for block in final_content_blocks:
                    if block.get("type") == "image":
                        used_image_paths.add(block["path"])
            else:
                print(f"🗑️ 已過濾掉無關主題：'{new_main_topic}'")

    # 7. 組合並儲存最終報告
    final_report = {
        "metadata": {
            "lesson_date": config["LESSON_DATE"],
            "subject": config["LESSON_SUBJECT"],
            "report_generated_at": datetime.datetime.now().isoformat()
        },
        "knowledge_hub_content": refined_knowledge_hub_list, 
        "knowledge_hub_initial": dict(knowledge_hub_initial) 
    }

    # 8. 組合並儲存最終報告
    print("\n💾 正在儲存最終的知識庫報告...")
    
    lesson_subject = config["LESSON_SUBJECT"]
    lesson_date_formatted = config["LESSON_DATE"].replace('/', '')
    
    output_filename = f"knowledge_hub_{lesson_subject}_{lesson_date_formatted}_final.json"
    output_path = os.path.join(config["OUTPUT_FOLDER"], output_filename)

    os.makedirs(config["OUTPUT_FOLDER"], exist_ok=True)

    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(final_report, f, ensure_ascii=False, indent=4)
        print(f"✔️ 報告成功儲存至: {output_path}")
    except Exception as e:
        print(f"❌ 儲存檔案時發生錯誤: {e}")

if __name__ == "__main__":
    try:
        main(CONFIG)
    except Exception as e:
        print(f"\n❌ 程式執行時發生致命錯誤: {e}")

✅ 課程參數已自動設定完成：
   - 課程ID: 0907
   - 科目: 英文
   - 板書路徑: C:\Users\User\Desktop\test\note_blackboard\0907_English_filtered
   - 逐字稿路徑: C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0907\老師\0907_2.txt
--------------------
🚀 開始建立主題式知識庫...
✅ Azure OpenAI 客戶端初始化並驗證成功。
正在掃描黑板照片資料夾: C:\Users\User\Desktop\test\note_blackboard\0907_English_filtered...
✔️ 成功加載並索引了 108 張黑板照片。
正在加載逐字稿檔案: C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0907\老師\0907_2.txt...
✔️ 成功加載並處理了 438 行逐字稿，總時長: 3:38:30。

🤖 啟動 AI 視覺預審員，檢查 108 張圖片的內容密度...


AI 預審圖片中: 100%|██████████| 108/108 [02:17<00:00,  1.27s/it]


✔️ AI 預審完成！從 108 張圖片中篩選出 62 張有效板書。

🔬 將課程內容分為 22 個時間區塊 (每塊 10 分鐘) 進行分析...


分析課堂內容區塊:  68%|██████▊   | 15/22 [00:14<00:05,  1.25it/s]


❌ 分析區塊時發生錯誤: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': True, 'severity': 'medium'}}}}} (第 1/3 次)


...


✍️ (Step 4) 啟動寫手 AI (圖文穿插優化版)，正在撰寫: 單字、片語與同義詞精選：choice, selection, option, gym, workout, weight lift, abroad, broad, board, aboard 等用法...
🎨 (Step 5) 插畫師 AI 正在為 6 張圖片定位...


最終精煉主題:  80%|████████  | 4/5 [00:59<00:18, 18.71s/it]


✍️ (Step 4) 啟動寫手 AI (圖文穿插優化版)，正在撰寫: 句型結構與文法重點：to 的用法、句首命題句型、連接詞、副詞、名詞用法...
🎨 (Step 5) 插畫師 AI 正在為 3 張圖片定位...


最終精煉主題: 100%|██████████| 5/5 [01:18<00:00, 15.65s/it]


💾 正在儲存最終的知識庫報告...
✔️ 報告成功儲存至: C:\Users\User\Desktop\test\note_json\knowledge_hub_英文_0907_final.json


### 課程報告(AI練習題目)

In [7]:
import os
import re
import json
import base64
import datetime
import time 
from openai import AzureOpenAI, APIError, RateLimitError 
from dotenv import load_dotenv
from tqdm import tqdm
from collections import defaultdict
import concurrent.futures
from PIL import Image

# ==================================================================
# ⚙️ 設定參數 (這部分不變)
# ==================================================================
LESSON_ID = "0928"
LESSON_SUBJECT = "英文"

# 2. 底下的 CONFIG 字典會根據上面的變數自動生成，無需手動修改
CONFIG = {
    "BLACKBOARD_IMAGES_FOLDER": rf"C:\Users\User\Desktop\test\note_blackboard\{LESSON_ID}_English_filtered",
    "TRANSCRIPT_FILE": rf"C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\{LESSON_ID}\老師\{LESSON_ID}_3.txt",
    "LESSON_DATE": f"{LESSON_ID[:2]}/{LESSON_ID[2:]}",
    "LESSON_SUBJECT": LESSON_SUBJECT,
    "OUTPUT_FOLDER": r"C:\Users\User\Desktop\test\note_json", # 輸出資料夾保持不變
    "CHUNK_MINUTES": 10
}

CONTENT_THRESHOLD = 8000 

# --- 核心輔助函數 ---

def safe_json_parse(json_str):
    """
    安全解析 JSON 字串的輔助函式。
    它可以自動去除 OpenAI 有時會輸出的 Markdown 代碼標記 (```json ... ```)，
    避免 json.loads() 發生錯誤。
    """
    if not json_str:
        return {}
    
    # 去除前後空白
    clean_str = json_str.strip()
    
    # 去除 Markdown 的 ```json 開頭
    if clean_str.startswith("```json"):
        clean_str = clean_str[7:]
    elif clean_str.startswith("```"):
        clean_str = clean_str[3:]
    
    # 去除 Markdown 的 ``` 結尾
    if clean_str.endswith("```"):
        clean_str = clean_str[:-3]
        
    # 再次去除可能殘留的空白
    clean_str = clean_str.strip()

    try:
        return json.loads(clean_str)
    except json.JSONDecodeError as e:
        print(f"⚠️ 警告：JSON 解析失敗，原始內容可能不完整或格式錯誤: {e}")
        # 如果解析失敗，返回空字典，避免程式崩潰
        return {}

def ai_screen_image_relevance(image_path: str, client: AzureOpenAI, max_retries: int = 5, initial_delay: int = 5) -> bool:
    """唯一的任務是判斷單張圖片是否包含足夠的教學內容。(已加入重試機制)"""
    deployment_name = os.getenv("CHAT_COMPLETION_NAME")
    if not deployment_name: raise ValueError("錯誤：缺少 CHAT_COMPLETION_NAME。")
    
    system_prompt = """
    你是一位嚴格的圖片審查員，你的唯一任務是判斷一張黑板照片是否包含【足夠的教學筆記】。
    你不需要理解筆記的內容，只需要評估其【資訊密度】。
    - **相關 (relevant):** 黑板上有清晰、足夠數量的文字、圖表或公式。
    - **不相關 (irrelevant):** 黑板符合以下任一情況：空白、幾乎空白、老師正在擦黑板、只有一兩個詞的標題、內容極度稀疏、完全模糊。
    你的回答必須是只有單一鍵的 JSON 物件： `{"is_relevant": <true 或 false>}`
    """
    
    b64_image = encode_image_to_base64(image_path)
    if not b64_image: return False
    
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": [{"type": "image_url", "image_url": {"url": b64_image}}]}]
    
    # --- 重試邏輯 ---
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(model=deployment_name, messages=messages, response_format={"type": "json_object"}, temperature=0.0, max_tokens=150)
            return json.loads(response.choices[0].message.content).get("is_relevant", False)
        
        # 專門捕捉速率限制錯誤
        except RateLimitError as e:
            # 嘗試從錯誤訊息中提取建議的等待時間
            retry_after_match = re.search(r"retry after (\d+)", str(e))
            if retry_after_match:
                wait_time = int(retry_after_match.group(1)) + 1 # 多等一秒增加緩衝
            else:
                wait_time = initial_delay * (2 ** attempt) # 如果沒找到，就使用指數退避策略
            
            print(f"🕒 遇到速率限制 for {os.path.basename(image_path)}。等待 {wait_time} 秒後重試 ({attempt + 1}/{max_retries})...")
            time.sleep(wait_time)
            
        # 捕捉其他 API 錯誤
        except APIError as e:
            print(f"⚠️ 警告：AI 圖片預審時發生 API 錯誤 for {os.path.basename(image_path)}: {e}")
            return False # API 錯誤通常無法透過重試解決，直接返回
            
        # 捕捉其他所有未知錯誤
        except Exception as e:
            print(f"⚠️ 警告：AI 圖片預審時發生未知錯誤 for {os.path.basename(image_path)}: {e}")
            # 根據情況，您也可以選擇在這裡重試
            time.sleep(initial_delay)

    print(f"❌ 警告：AI 圖片預審失敗 for {os.path.basename(image_path)} 在達到最大重試次數後。")
    return False

def is_image_content_sufficient(image_path, threshold):
    """使用 Pillow 分析圖片，判斷其內容是否足夠。"""
    try:
        with Image.open(image_path).convert('L') as img:
            binary_img = img.point(lambda p: 255 if p > 50 else 0, '1')
            content_pixels = sum(1 for pixel in binary_img.getdata() if pixel == 255)
            return content_pixels > threshold
    except Exception as e:
        print(f"⚠️ 警告：分析圖片 {os.path.basename(image_path)} 時發生錯誤: {e}")
        return False

def load_openai_client():
    """加載環境變數並初始化 Azure OpenAI 客戶端"""
    load_dotenv()
    azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
    api_key = os.getenv("AZURE_OPENAI_KEY")
    api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2024-02-15-preview")
    if not azure_endpoint or not api_key:
        raise ValueError("錯誤：.env 檔案中缺少 AZURE_OPENAI_ENDPOINT 或 AZURE_OPENAI_KEY。")
    try:
        client = AzureOpenAI(api_key=api_key, azure_endpoint=azure_endpoint, api_version=api_version)
        client.models.list() 
        print("✅ Azure OpenAI 客戶端初始化並驗證成功。")
        return client
    except APIError as e:
        raise ConnectionError(f"❌ Azure OpenAI API 錯誤: {e}")
    except Exception as e:
        raise Exception(f"❌ 初始化 Azure OpenAI 客戶端時發生未知錯誤: {e}")

def get_timestamp_from_filename(filename, pattern):
    match = re.search(pattern, filename)
    if match:
        h, m, s = map(int, match.groups())
        return datetime.timedelta(hours=h, minutes=m, seconds=s)
    return None

def encode_image_to_base64(image_path):
    try:
        with open(image_path, "rb") as image_file:
            return f"data:image/png;base64,{base64.b64encode(image_file.read()).decode('utf-8')}"
    except Exception as e:
        print(f"警告：無法編碼圖片 {image_path}: {e}")
        return None

def parse_transcript_line(line):
    # 更新正規表示式，以匹配新格式
    pattern = re.compile(r"(\d{2}:\d{2}:\d{2}): (.*)")
    match = pattern.match(line)
    if match:
        # 只獲取時間戳和內容
        start_str, content = match.groups()
        h, m, s = map(int, start_str.split(':'))
        start_td = datetime.timedelta(hours=h, minutes=m, seconds=s)

        # 由於新格式沒有結束時間，我們將其設定為與開始時間相同，
        # 這不影響後續以時間區塊為單位的分析。
        end_td = start_td

        return {'start_time': start_td, 'end_time': end_td, 'content': content.strip()}
    return None

def load_blackboard_images(folder_path):
    images = []
    pattern = r'(\d+)h(\d{2})m(\d{2})s'
    print(f"正在掃描黑板照片資料夾: {folder_path}...")
    if not os.path.isdir(folder_path):
        print(f"警告：找不到資料夾 {folder_path}。")
        return []
    for filename in sorted(os.listdir(folder_path)):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            timestamp = get_timestamp_from_filename(filename, pattern)
            if timestamp:
                images.append({'timestamp': timestamp, 'path': os.path.join(folder_path, filename)})
    print(f"✔️ 成功加載並索引了 {len(images)} 張黑板照片。")
    return images

def fix_quiz_format_programmatically(quiz_list):
    """
    強制檢查並修復練習題格式，特別是填空題的挖空邏輯。
    這是一個「安全網」，防止 AI 偶爾發瘋不挖空。
    """
    fixed_list = []
    for item in quiz_list:
        # 針對填空題進行處理
        if item.get("type") == "fill_in_the_blank":
            question = item.get("question", "")
            answer = item.get("answer", "").strip()
            
            # 檢查 question 中是否包含底線 (至少 3 個連續底線)
            if not re.search(r"_{3,}", question):
                # 如果沒有底線，且答案存在於題目中，則進行替換
                if answer and answer.lower() in question.lower():
                    # 使用正規表達式進行不分大小寫的替換 (只替換第一次出現)
                    pattern = re.compile(re.escape(answer), re.IGNORECASE)
                    new_question = pattern.sub("_______", question, count=1)
                    item["question"] = new_question
                    print(f"🔧 自動修復：已為題目 '{question[:30]}...' 強制挖空。")
                else:
                    # 如果答案不在題目中 (罕見情況)，或者無法匹配
                    print(f"⚠️ 警告：填空題修復失敗，答案 '{answer}' 不在題目 '{question}' 中。")
                    # 選擇性策略：可以選擇丟棄這題，或是保留原樣
        
        fixed_list.append(item)
    return fixed_list

def load_and_process_transcripts(file_path):
    """加載並解析單一的逐字稿檔案"""
    transcript_entries = []
    total_duration = datetime.timedelta(0)
    print(f"正在加載逐字稿檔案: {file_path}...")

    # 檢查檔案是否存在
    if not os.path.isfile(file_path):
        print(f"警告：找不到指定的逐字稿檔案 {file_path}，無法處理逐字稿。")
        return [], total_duration

    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        if not lines:
            print(f"警告：逐字稿檔案 {file_path} 是空的。")
            return [], total_duration
        
        # 直接處理單一檔案，不再需要 cumulative_offset
        for line in lines:
            entry = parse_transcript_line(line)
            if entry:
                transcript_entries.append(entry)
                # 更新總時長
                total_duration = max(total_duration, entry['end_time'])

    print(f"✔️ 成功加載並處理了 {len(transcript_entries)} 行逐字稿，總時長: {total_duration}。")
    return transcript_entries, total_duration

def analyze_content_chunk(transcript_text, image_path, client, max_retries=3, initial_delay=5):
    """
    【新版】階段 1：分析單一時間區塊，【只】提取核心主題名稱。不再生成圖片描述。
    """
    deployment_name = os.getenv("CHAT_COMPLETION_NAME")
    if not deployment_name:
        raise ValueError("錯誤：.env 檔案中缺少 CHAT_COMPLETION_NAME 設定。")

    # --- 新的、極簡的 System Prompt ---
    system_prompt = """
    你是一位高效的教育內容標籤員。你的任務是快速閱讀一段教學內容，並只提取出其中講授的 1 到 2 個核心【學術主題名稱】。

    - 主題名稱應簡潔具體 (例如: "分詞的用法", "名詞子句", "worry 的用法")。
    - 如果主要是閒聊或與學術無關的內容，請返回一個空列表。
    - 不要生成摘要或任何額外描述。

    你的回答必須是只有單一鍵 `topics` 的 JSON 物件，其值為一個【字串列表】。
    ```json
    {
      "topics": ["提取出的第一個主題名稱", "提取出的第二個主題名稱"]
    }
    ```
    """
    
    messages = [{"role": "system", "content": system_prompt}]
    
    user_content = [{"type": "text", "text": f"請分析以下內容並提取主題名稱：\n\n{transcript_text}"}]
    if image_path:
        b64_image = encode_image_to_base64(image_path)
        if b64_image:
            user_content.append({"type": "image_url", "image_url": {"url": b64_image}})
    
    messages.append({"role": "user", "content": user_content})

    # --- API 呼叫邏輯 (保持不變) ---
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=deployment_name,
                messages=messages,
                response_format={"type": "json_object"},
                temperature=0.0,
                max_tokens=200 # 因為任務變簡單，可以減少 token
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            print(f"\n❌ 分析區塊時發生錯誤: {e} (重試 {attempt + 1}/{max_retries})")
            time.sleep(initial_delay)

    # 如果所有重試都失敗，返回一個帶有失敗標記的標準結構
    return {"topics": []}

def cluster_topics_with_ai(topic_list: list[str], client: AzureOpenAI) -> dict | None:
    """階段 2：使用強力互斥合併指令，將初步主題歸納為 5-6 個核心單元。"""
    deployment_name = os.getenv("CHAT_COMPLETION_NAME")
    if not deployment_name: raise ValueError("錯誤：缺少 CHAT_COMPLETION_NAME。")
    
    # --- V3.0 修改：加入互斥性與層級指令 ---
    system_prompt = """
    你是一位極度嚴格的【課程總編輯】。你的任務是將零散的教學主題標籤，重組為 3 到 6 個【絕對互斥】且【標題精確】的教學模組。

    🔥🔥 核心任務：消除任何「概念上的重複」🔥🔥

    **1. 合併規則 (底層邏輯優先)：**
    - 忽略表面例句的差異。**只要涉及相同的「文法規則」，必須強制合併為同一個單元！**
    - 範例：如果標籤 A 是「worry 的用法」，標籤 B 是「分詞形容詞」，標籤 C 是「情緒動詞」，且內容都在講 V-ing vs V-ed。
    - **❌ 錯誤做法：** 拆成 "Unit 1: worry" 和 "Unit 2: 分詞"。
    - **✅ 正確做法：** 合併為 "Unit 1: 情緒形容詞與分詞 (V-ing/V-ed)"。

    **2. 命名規範 (Specific Grammar Terminology)：**
    - 每個主題名稱必須清楚標示該單元測試的「核心文法點」。
    - **❌ 禁用模糊標題：** "進階句型", "核心動詞", "文法解析", "容易混淆的字"。
    - **✅ 使用精確術語：** "名詞子句 (Noun Clauses)", "分詞形容詞 (Participle Adjectives)", "感嘆句 (Exclamatory Sentences)", "附屬子句 (Adverbial Clauses)"。

    **3. 你的工作流程：**
    - 檢視所有標籤。
    - 找出所有關於「分詞/情緒字/Ving/Ved」的標籤 -> 合併成【一個】單元。
    - 找出所有關於「子句/that/wh-」的標籤 -> 合併成【一個】單元。
    - 確保沒有任何兩個單元的文法概念是重疊的。

    你的輸出必須是一個 JSON 物件，`key` 是合併後的主題名稱 (請用精確文法術語)，`value` 是包含在該主題下的原始主題列表。
    """
    
    topics_text = "\n".join(f"- {topic}" for topic in topic_list)
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": f"請將以下主題列表進行「互斥性分類」合併：\n\n{topics_text}"}]
    
    print("\n🧠 正在啟動 AI 總編輯進行互斥性主題聚類...")
    try:
        response = client.chat.completions.create(model=deployment_name, messages=messages, response_format={"type": "json_object"}, temperature=0.0)
        return safe_json_parse(response.choices[0].message.content) # 使用之前建議的 safe_json_parse
    except Exception as e:
        print(f"\n❌ AI 主題聚類過程中發生錯誤: {e}")
        return None

def generate_learning_module_with_quiz(main_topic, segments, client: AzureOpenAI) -> dict:
    """
    【V7.0 - 格式潔癖版 (詳解 + 口說連動 + 題目選項分離)】
    
    修正重點：
    1. 針對選擇題 (multiple_choice) 加入嚴格限制：
       "question" 欄位絕對不能包含 (A)... (B)... 等選項文字，保持題目乾淨。
    """
    deployment_name = os.getenv("CHAT_COMPLETION_NAME")
    if not deployment_name: 
        raise ValueError("錯誤：.env 檔案中缺少 CHAT_COMPLETION_NAME。")

    # --- System Prompt 設計 (V7.0 加入選項分離與詳解要求) ---
    system_prompt = """
    你是一位針對【台灣國中三年級 (Grade 9)】學生的專業英文教材編寫者。
    你的目標受眾程度約為 **CEFR A2 至 B1 等級**。

    **任務指令：**
    請根據提供的教學素材，生成一個結構化的 JSON 學習模組。

    **重點要求：**
    - 本單元的主題是：**「{MAIN_TOPIC}」**。
    - 請確保所有內容（筆記、題目、詳解）都緊扣這個特定的文法主題。
    - **不要** 混入其他不相關的文法點。

    **1. 圖片內容解析 (Image Description):**
    - **視角：** 扮演資深英文老師，使用第一人稱「我」。
    - **內容：** 100% 基於黑板視覺內容，解釋背後的文法規則。

    **2. 內容講解 (Content Blocks):**
    - 圖文穿插，使用繁體中文。

    **3. 互動練習題 (Interactive Quiz) - 【嚴格順序 Strict Sequence】：**
    這個區塊 **必須且只能包含 4 道題目**，並嚴格遵守以下順序與關聯性。
    
    🔥🔥【全域規則 1】：每一道題目都必須包含 "explanation" 欄位，用繁體中文詳細解釋文法原因。
    🔥🔥【全域規則 2 (重要)】：選擇題的 "question" 欄位 **嚴禁** 包含選項內容（如 (A)..., A. ...）。選項只能放在 "options" 物件中。

    *   **第 1 題：選擇題 (`multiple_choice`)**
        - `"question"`: 純粹的題目敘述。**❌絕對不要把選項寫在這裡**。
        - `"options"`: 2~4 個選項，鍵值為 A, B...。
        - `"answer"`: 正確選項代號 (如 "A")。
        - `"explanation"`: 解析為何該選項正確，並說明為何其他選項不適合。

    *   **第 2 題：填空題 (`Fill_in_the_blank`)**
        - `"question"`: 英文句子含底線 `_______` + 中文翻譯。 (請直接在 question 字串中挖好空)
        - `"answer"`: 正確單字。
        - `"explanation"`: 解釋單字詞性、文法規則或搭配詞用法。

    *   **第 3 題：中翻英造句 (`sentence_practice`)**
        - `"question"`: **必須是「請將『(中文句子)』翻譯成英文。」的格式**。
        - `"answer"`: 標準英文完整句子。(這句非常重要，將用於下一題)
        - `"explanation"`: 分析句型結構 (如 S+V+O) 或使用的關鍵片語。

    *   **第 4 題：口說跟讀 (`speaking_practice`) - 【與第 3 題連動】**
        - **目標：** 讓學生朗讀第 3 題寫出的正確答案。
        - `"question"`: 請暫時填寫 "請大聲朗讀上一題的答案"。
        - `"reference_sentence"`: **必須完全等於第 3 題的 `"answer"` 內容**。
        - `"tips"`: 提供發音或語調的小提醒 (繁體中文)。
        - `"explanation"`: 針對這句話的情境或語氣做補充說明。

    **4. 語言強制：**
    - 所有解釋與詳解必須使用**繁體中文**。

    **最終 JSON 輸出格式範例：**
    ```json
    {
        "main_topic": "主題名稱",
        "topic_summary": "總結",
        "content_blocks": [ ... ],
        "interactive_quiz": [
            { 
                "type": "multiple_choice", 
                "question": "下列哪一句正確描述人的感受？", 
                "options": {"A": "I am bored.", "B": "I am boring."}, 
                "answer": "A",
                "explanation": "因為主詞是人，所以形容詞要用 bored (V-ed)。"
            },
            { 
                "type": "Fill_in_the_blank", 
                "question": "The news is very _______.", 
                "answer": "surprising",
                "explanation": "surprising 意為令人驚訝的，用來修飾新聞本身。"
            },
            { 
                "type": "sentence_practice", 
                "question": "請將『這本書很無聊』翻譯成英文。", 
                "answer": "This book is boring.",
                "explanation": "主詞 This book 是物，形容詞用 boring。"
            },
            { 
                "type": "speaking_practice", 
                "question": "請大聲朗讀上一題的答案", 
                "reference_sentence": "This book is boring.",
                "tips": "注意 boring 的重音。",
                "explanation": "請用肯定的語氣讀出這句話。"
            }
        ]
    }
    ```
    """.replace("{MAIN_TOPIC}", main_topic)
    
    # --- 1. 資料準備與圖片去重 ---
    full_transcript = "\n".join([seg.get("relevant_transcript", "") for seg in segments])
    unique_image_paths = []
    seen_paths = set()
    for seg in segments:
        img_info = seg.get("relevant_blackboard_image_info")
        if img_info and img_info.get("path") and img_info["path"] not in seen_paths:
            unique_image_paths.append(img_info["path"])
            seen_paths.add(img_info["path"])
            
    user_prompt_content = [{"type": "text", "text": f"請為主題「{main_topic}」生成學習模組。相關逐字稿如下：\n---\n{full_transcript}\n---"}]
    
    for img_path in unique_image_paths:
        b64_image = encode_image_to_base64(img_path)
        if b64_image:
            user_prompt_content.append({"type": "text", "text": f"--- Analyzing image with path: {img_path} ---"})
            user_prompt_content.append({"type": "image_url", "image_url": {"url": b64_image}})

    print(f"\n🧠 正在為主題 '{main_topic}' 生成學習模組 (V7.0 格式潔癖版)...")
    
    try:
        # --- 2. 呼叫 Azure OpenAI ---
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt_content} 
            ],
            response_format={"type": "json_object"},
            temperature=0.3, 
            max_tokens=4000
        )
        
        # 解析 JSON
        module_data = safe_json_parse(response.choices[0].message.content)
        
        # --- 3. Python 後處理：強制同步第3題與第4題 ---
        if "interactive_quiz" in module_data and isinstance(module_data["interactive_quiz"], list):
            quiz_list = module_data["interactive_quiz"]
            
            # (A) 先執行填空題修復
            quiz_list = fix_quiz_format_programmatically(quiz_list)

            # (B) 處理口說題連動 & 題目文字組裝
            # 尋找第 3 題 (Sentence Practice)
            sentence_q = next((q for q in quiz_list if q.get("type") == "sentence_practice"), None)
            
            if sentence_q and "answer" in sentence_q:
                # 取得第 3 題的標準答案，並去除前後多餘空白
                target_sentence = sentence_q["answer"].strip()
                
                # 尋找第 4 題 (Speaking Practice)
                speaking_q = next((q for q in quiz_list if q.get("type") == "speaking_practice"), None)
                
                # 將英文答案直接組裝進題目敘述中
                new_question_text = f"請大聲朗讀：{target_sentence}"

                if speaking_q:
                    speaking_q["reference_sentence"] = target_sentence
                    speaking_q["question"] = new_question_text
                    if "explanation" not in speaking_q:
                        speaking_q["explanation"] = "請模仿老師的語調，清晰唸出句子。"
                else:
                    new_speaking_q = {
                        "type": "speaking_practice",
                        "question": new_question_text,
                        "reference_sentence": target_sentence,
                        "tips": "請注意語調的起伏，並模仿老師的語氣。",
                        "explanation": "請嘗試流暢地朗讀出這句話。"
                    }
                    quiz_list.append(new_speaking_q)
            
            # 將處理好的題庫存回資料結構
            module_data["interactive_quiz"] = quiz_list

        return module_data

    except Exception as e:
        print(f"\n❌ 為主題 '{main_topic}' 生成學習模組時出錯: {e}")
        return {
            "main_topic": main_topic,
            "topic_summary": "生成失敗",
            "content_blocks": [],
            "interactive_quiz": []
        }

def main(config):
    """
    主執行流程：從原始數據生成包含筆記和練習題的完整學習模組。
    """
    print("🚀 開始建立「AI 互動練習冊」數據...")
    
    # --- 階段 0: 初始化與加載 ---
    client = load_openai_client()
    blackboard_images = load_blackboard_images(config["BLACKBOARD_IMAGES_FOLDER"])
    transcripts, total_duration = load_and_process_transcripts(config["TRANSCRIPT_FILE"])
    
    if not transcripts:
        print("❌ 錯誤：沒有找到任何有效的逐字稿內容，程式終止。")
        return

    # --- AI 視覺預審流程 ---
    print(f"\n🤖 啟動 AI 視覺預審員，檢查 {len(blackboard_images)} 張圖片的內容密度...")
    relevant_images = []
    if blackboard_images:
        with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
            future_to_img = {executor.submit(ai_screen_image_relevance, img['path'], client): img for img in blackboard_images}
            for future in tqdm(concurrent.futures.as_completed(future_to_img), total=len(blackboard_images), desc="AI 預審圖片中"):
                img = future_to_img[future]
                try:
                    if future.result():
                        relevant_images.append(img)
                except Exception as exc:
                    print(f"預審 {os.path.basename(img['path'])} 時產生錯誤: {exc}")
    print(f"✔️ AI 預審完成！從 {len(blackboard_images)} 張圖片中篩選出 {len(relevant_images)} 張有效板書。")
    
    # --- 階段 1: 初步分析與主題提取 ---
    chunk_minutes = config.get("CHUNK_MINUTES", 10) 
    chunk_duration = datetime.timedelta(minutes=chunk_minutes)
    if total_duration.total_seconds() == 0:
        print("❌ 錯誤：逐字稿總時長為 0，無法進行分塊。")
        return
    num_chunks = int(total_duration.total_seconds() / chunk_duration.total_seconds()) + 1
    
    print(f"\n🔬 [階段 1/3] 將課程內容分為 {num_chunks} 個時間區塊進行初步分析...")
    
    tasks_to_run = []
    for i in range(num_chunks):
        chunk_start_time = i * chunk_duration
        chunk_end_time = (i + 1) * chunk_duration
        chunk_transcript_text = "\n".join([e['content'] for e in transcripts if chunk_start_time <= e['start_time'] < chunk_end_time]).strip()
        if not chunk_transcript_text:
            continue
        
        best_image_path = None
        if relevant_images:
            chunk_mid_point = chunk_start_time + (chunk_duration / 2)
            min_diff = datetime.timedelta.max
            for img in relevant_images:
                diff = abs(img['timestamp'] - chunk_mid_point)
                if diff < min_diff and diff < chunk_duration:
                    min_diff = diff
                    best_image_path = img['path']
        
        tasks_to_run.append((chunk_transcript_text, best_image_path, client))
    
    all_analysis_results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        future_to_task = {executor.submit(analyze_content_chunk, *task): i for i, task in enumerate(tasks_to_run)}
        for future in tqdm(concurrent.futures.as_completed(future_to_task), total=len(tasks_to_run), desc="分析內容區塊"):
            try:
                result = future.result()
                all_analysis_results.append(result)
            except Exception as exc:
                print(f'\n❌ 一個分析任務產生了錯誤: {exc}')
                all_analysis_results.append({"topics": []})
    
    knowledge_hub_initial = defaultdict(lambda: {"teaching_segments": []})
    
    for i, (transcript_text, image_path, _) in enumerate(tasks_to_run):
        analysis_res = all_analysis_results[i] if i < len(all_analysis_results) else {}
        topic_names = analysis_res.get("topics", []) 
        if not isinstance(topic_names, list): continue

        for topic_name in topic_names: 
            image_info = {"path": image_path} if image_path else None
            knowledge_hub_initial[topic_name]["teaching_segments"].append({
                "summary_of_segment": "", 
                "relevant_transcript": transcript_text,
                "relevant_blackboard_image_info": image_info 
            })

    # --- 階段 2: 主題聚類與過濾 ---
    print("\n🔄 [階段 2/3] 聚合初步主題並交由 AI 總編輯進行強力合併與過濾...")
    initial_topic_list = list(knowledge_hub_initial.keys())
    
    if not initial_topic_list:
        print("⚠️ 初步分析未能生成任何有效主題，程式終止。")
        return
    
    clustered_topics = cluster_topics_with_ai(initial_topic_list, client)
    if not clustered_topics:
        print("⚠️ AI 主題聚類失敗，程式終止。")
        return
        
    # --- 階段 3: 根據主題生成完整的學習模組 (筆記+練習題) ---
    print("\n📝✍️ [階段 3/3] 根據主題，生成包含筆記與練習題的完整學習模組...")
    
    original_to_main_topic_map = {}
    for main_topic, originals in clustered_topics.items():
        for original in originals:
            original_to_main_topic_map[original] = main_topic
    
    merged_segments = defaultdict(list)
    
    # 🔧 【關鍵修改】：使用全域集合 (Set) 來記錄已經使用過的逐字稿片段
    # 這樣可以確保同一段話絕對不會出現在兩個不同的主題中，徹底解決題目重複問題。
    global_seen_transcripts = set() 
    
    initial_hub_dict = dict(knowledge_hub_initial)
    
    # 為了讓重要的主題先挑選內容，這裡可以考慮對 clustered_topics 做排序，
    # 但為了簡單起見，我們先依序處理。
    for original_topic, data in initial_hub_dict.items():
        main_topic = original_to_main_topic_map.get(original_topic)
        
        if main_topic:
            for segment in data["teaching_segments"]:
                # 使用逐字稿的前 100 個字作為唯一識別碼 (Signature)
                # 增加字數長度可以減少誤判
                transcript_sig = segment["relevant_transcript"][:100]
                
                # 只有當這段逐字稿【從未在整份教材中出現過】時，才加入當前主題
                if transcript_sig not in global_seen_transcripts:
                    merged_segments[main_topic].append(segment)
                    global_seen_transcripts.add(transcript_sig)
                else:
                    # 選擇性 debug：可以看到哪些內容因為重複而被過濾掉了
                    # print(f"  -> 跳過重複內容: {transcript_sig[:20]}... (已存在於其他主題)")
                    pass

    # 平行執行最終的模組生成任務
    final_learning_modules = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        # 使用更新後的 V5.0 生成函式
        future_to_topic = {executor.submit(generate_learning_module_with_quiz, topic, segments, client): topic for topic, segments in merged_segments.items()}
        for future in tqdm(concurrent.futures.as_completed(future_to_topic), total=len(merged_segments), desc="生成學習模組"):
            try:
                module_data = future.result()
                # 只有當 content_blocks 有內容時才加入，避免產生空的主題（因為去重可能導致某些主題變空）
                if module_data.get("content_blocks"):
                    final_learning_modules.append(module_data)
                else:
                    print(f"⚠️ 主題 '{module_data.get('main_topic')}' 因內容重複被過濾，生成結果為空，已略過。")
            except Exception as exc:
                topic_name = future_to_topic[future]
                print(f"\n❌ 為主題 '{topic_name}' 生成學習模組時發生嚴重錯誤: {exc}")

    # --- 最終步驟: 組合並儲存報告 ---
    final_report = {
        "metadata": {
            "lesson_date": config["LESSON_DATE"],
            "subject": config["LESSON_SUBJECT"],
            "report_type": "Interactive Workbook",
            "report_generated_at": datetime.datetime.now().isoformat()
        },
        "workbook_data": {"refined_knowledge_hub": final_learning_modules}
    }

    output_filename = f"interactive_workbook_{config['LESSON_SUBJECT']}_{config['LESSON_DATE'].replace('/', '')}.json"
    output_folder = config.get("OUTPUT_FOLDER", ".")
    os.makedirs(output_folder, exist_ok=True)
    output_path = os.path.join(output_folder, output_filename)

    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(final_report, f, ensure_ascii=False, indent=4)
        print(f"\n🎉 互動練習冊數據建立完成！已成功儲存至: {output_path}")
    except Exception as e:
        print(f"❌ 儲存最終報告時發生錯誤: {e}")

if __name__ == "__main__":
    try:
        main(CONFIG)
    except Exception as e:
        import traceback
        print(f"\n❌ 程式執行時發生致命錯誤: {traceback.format_exc()}")

🚀 開始建立「AI 互動練習冊」數據...
✅ Azure OpenAI 客戶端初始化並驗證成功。
正在掃描黑板照片資料夾: C:\Users\User\Desktop\test\note_blackboard\0928_English_filtered...
✔️ 成功加載並索引了 81 張黑板照片。
正在加載逐字稿檔案: C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0928\老師\0928_3.txt...
✔️ 成功加載並處理了 400 行逐字稿，總時長: 3:19:30。

🤖 啟動 AI 視覺預審員，檢查 81 張圖片的內容密度...


AI 預審圖片中:  99%|█████████▉| 80/81 [01:18<00:00,  1.01it/s]

🕒 遇到速率限制 for blackboard_02h26m10s_f219251.png。等待 5 秒後重試 (1/5)...
🕒 遇到速率限制 for blackboard_02h26m10s_f219251.png。等待 10 秒後重試 (2/5)...
🕒 遇到速率限制 for blackboard_02h26m10s_f219251.png。等待 20 秒後重試 (3/5)...
🕒 遇到速率限制 for blackboard_02h26m10s_f219251.png。等待 40 秒後重試 (4/5)...
🕒 遇到速率限制 for blackboard_02h26m10s_f219251.png。等待 80 秒後重試 (5/5)...


AI 預審圖片中: 100%|██████████| 81/81 [05:28<00:00,  4.06s/it]


❌ 警告：AI 圖片預審失敗 for blackboard_02h26m10s_f219251.png 在達到最大重試次數後。
✔️ AI 預審完成！從 81 張圖片中篩選出 36 張有效板書。

🔬 [階段 1/3] 將課程內容分為 20 個時間區塊進行初步分析...


...


🔄 [階段 2/3] 聚合初步主題並交由 AI 總編輯進行強力合併與過濾...

🧠 正在啟動 AI 總編輯進行互斥性主題聚類...

📝✍️ [階段 3/3] 根據主題，生成包含筆記與練習題的完整學習模組...

🧠 正在為主題 '名詞子句與 that 的用法 (Noun Clauses, that)' 生成學習模組 (V7.0 格式潔癖版)...

🧠 正在為主題 '感嘆句 (Exclamatory Sentences)' 生成學習模組 (V7.0 格式潔癖版)...


生成學習模組:   0%|          | 0/4 [00:00<?, ?it/s]


🧠 正在為主題 '動詞與補語結構 (Linking Verbs, 感覺動詞, 受詞用法)' 生成學習模組 (V7.0 格式潔癖版)...

🧠 正在為主題 '分詞形容詞與感受表達 (Participle Adjectives: V-ing/V-ed, 感受/情緒形容詞)' 生成學習模組 (V7.0 格式潔癖版)...


生成學習模組: 100%|██████████| 4/4 [00:18<00:00,  4.71s/it]


🎉 互動練習冊數據建立完成！已成功儲存至: C:\Users\User\Desktop\test\note_json\interactive_workbook_英文_0928.json


### 班級時間軸狀態

In [ ]:
# -*- coding: utf-8 -*-
import os
import json
import datetime
from collections import Counter, defaultdict

# --- 核心設定 ---

# 1. 將詳細行為映射到宏觀狀態的規則
#    您可以根據教學需求自由調整這裡的分類
BEHAVIOR_STATE_MAPPING = {
    "專注聽講": [
        "目視教師",
        "目視黑板",
        "主動舉手",
        "被動舉手"
    ],
    "專注學習(書寫/閱讀)": [
        "目視書本/筆記",
        "做筆記",
        "翻書"
    ],
    "分心": [
        "目視同學",
        "目視他處",
        "玩弄手部/文具",
        "趴睡",
        "觸摸臉部",
        "觸摸頭髮",
        "低頭(非學習)",
        "飲食"
    ],
    # 其他未被明確歸類的行為將被視為 "中性/其他"
}

# 2. 時間區段的長度（分鐘）
INTERVAL_MINUTES = 15

# --- 輔助函式 ---

def get_timestamp_from_filename(filename):
    """從檔名解析時間戳 (從您提供的程式碼中沿用)"""
    import re
    match = re.search(r'(\d{2})-(\d{2})-(\d{2})-(\d{3})\.(jpg|jpeg|png|webp)$', filename, re.IGNORECASE)
    if match:
        try:
            h, m, s, ms, _ = match.groups()
            return datetime.timedelta(hours=int(h), minutes=int(m), seconds=int(s), milliseconds=int(ms))
        except (ValueError, IndexError):
            pass
    return None

def get_behavior_state(behavior, mapping):
    """根據行為名稱查找其對應的宏觀狀態"""
    for state, behaviors in mapping.items():
        if behavior in behaviors:
            return state
    return "中性/其他" # 如果找不到對應的分類，則歸為此類

# --- 主要邏輯函式 ---

def collect_class_session_data(base_dir, session_id):
    """
    掃描所有學生資料夾，收集指定課堂的所有行為事件。
    """
    full_timeline = []
    students_in_session = set()

    print(f"正在掃描資料夾 '{base_dir}'，尋找課堂ID為 '{session_id}' 的報告...")

    if not os.path.isdir(base_dir):
        print(f"錯誤：找不到指定的資料夾 '{base_dir}'")
        return None, None

    for student_folder in os.listdir(base_dir):
        student_folder_path = os.path.join(base_dir, student_folder)
        if not os.path.isdir(student_folder_path):
            continue

        for report_file in os.listdir(student_folder_path):
            if not report_file.endswith('.json'):
                continue
            
            report_path = os.path.join(student_folder_path, report_file)
            try:
                with open(report_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                # 檢查是否為目標課堂的報告
                if data.get("report_metadata", {}).get("report_generation_time") == session_id:
                    student_id = data.get("report_metadata", {}).get("student_id", student_folder)
                    students_in_session.add(student_id)
                    
                    behavior_index = data.get("overall_summary", {}).get("behavior_to_images_index", {})
                    
                    for behavior, images in behavior_index.items():
                        for image_filename in images:
                            timestamp_td = get_timestamp_from_filename(image_filename)
                            if timestamp_td:
                                full_timeline.append({
                                    "student_id": student_id,
                                    "timestamp_td": timestamp_td,
                                    "behavior": behavior
                                })
            except (json.JSONDecodeError, KeyError) as e:
                print(f"警告：讀取或解析檔案 '{report_file}' 時發生錯誤: {e}")
                continue
    
    if not full_timeline:
        print(f"錯誤：找不到任何關於課堂ID '{session_id}' 的有效數據。")
        return None, None
        
    # 按時間排序
    full_timeline.sort(key=lambda x: x["timestamp_td"])
    
    print(f"成功！找到了 {len(students_in_session)} 位學生的數據，共計 {len(full_timeline)} 筆行為記錄。")
    return full_timeline, list(students_in_session)


def analyze_timeline_by_intervals(timeline, students, interval_minutes):
    """
    將完整的時間軸切割成區段，並分析每個區段的學生狀態比例。
    """
    if not timeline or not students:
        return []

    start_time = timeline[0]['timestamp_td']
    end_time = timeline[-1]['timestamp_td']
    total_students = len(students)
    
    print(f"課堂時間範圍：從 {start_time} 到 {end_time}")
    print(f"將以每 {interval_minutes} 分鐘為一個區段進行分析...")

    interval_delta = datetime.timedelta(minutes=interval_minutes)
    current_interval_start = start_time
    
    class_report = []

    while current_interval_start < end_time:
        current_interval_end = current_interval_start + interval_delta
        
        # 篩選出當前時間區段內的所有事件
        interval_events = [
            event for event in timeline 
            if current_interval_start <= event['timestamp_td'] < current_interval_end
        ]
        
        if not interval_events:
            current_interval_start = current_interval_end
            continue

        # 計算這個區段內，每個學生的主要行為是什麼
        student_dominant_states = {}
        events_by_student = defaultdict(list)
        for event in interval_events:
            events_by_student[event['student_id']].append(event['behavior'])

        for student_id, behaviors in events_by_student.items():
            if behaviors:
                # 找到最頻繁的行為
                most_common_behavior = Counter(behaviors).most_common(1)[0][0]
                # 將其映射到宏觀狀態
                dominant_state = get_behavior_state(most_common_behavior, BEHAVIOR_STATE_MAPPING)
                student_dominant_states[student_id] = dominant_state
        
        # 統計每個宏觀狀態的學生人數
        state_counts = Counter(student_dominant_states.values())
        
        # 計算百分比
        state_percentages = {
            state: round((count / total_students) * 100, 1)
            for state, count in state_counts.items()
        }

        class_report.append({
            "interval_start": str(current_interval_start).split('.')[0],
            "interval_end": str(current_interval_end).split('.')[0],
            "analysis": {
                "total_students_active": len(student_dominant_states),
                "state_counts": dict(state_counts),
                "state_percentages": state_percentages
            }
        })
        
        current_interval_start = current_interval_end
        
    return class_report

# --- 主執行函式 ---

def main():
    """主執行流程"""
    base_dir = r"C:\Users\User\Desktop\test\SynologyDrive\json_behavior"
    session_id = input("請輸入您想分析的課堂ID (例如: 08/24): ")

    if not session_id:
        print("錯誤：課堂ID不能為空。")
        return

    # 1. 收集數據
    timeline, students = collect_class_session_data(base_dir, session_id)
    
    if not timeline:
        return

    # 2. 分析數據
    final_report_data = analyze_timeline_by_intervals(timeline, students, INTERVAL_MINUTES)
    
    if not final_report_data:
        print("分析完成，但未能產生任何時間區段的報告。")
        return
        
    # 3. 輸出報告
    print("\n" + "="*25 + " 班級整體學習狀態報告 " + "="*25)
    print(f"課堂ID: {session_id} | 總人數: {len(students)} | 分析區段: {INTERVAL_MINUTES} 分鐘\n")

    for segment in final_report_data:
        start = segment['interval_start']
        end = segment['interval_end']
        analysis = segment['analysis']
        percentages = analysis['state_percentages']
        
        print(f"--- 時間區段: {start} - {end} ---")
        print(f"  - 專注聽講: {percentages.get('專注聽講', 0.0)}%")
        print(f"  - 專注學習(書寫/閱讀): {percentages.get('專注學習(書寫/閱讀)', 0.0)}%")
        print(f"  - 分心: {percentages.get('分心', 0.0)}%")
        print(f"  - 中性/其他: {percentages.get('中性/其他', 0.0)}%\n")
        
    # 4. 儲存詳細的JSON報告
    output_filename = f"class_report_{session_id.replace('/', '')}_{datetime.datetime.now().strftime('%Y%m%d')}.json"
    output_path = os.path.join(os.path.dirname(base_dir), output_filename) # 存在 json_behavior 的上一層目錄

    report_to_save = {
        "report_metadata": {
            "session_id": session_id,
            "total_students": len(students),
            "interval_minutes": INTERVAL_MINUTES,
            "generation_timestamp": datetime.datetime.now().isoformat()
        },
        "class_behavior_timeline": final_report_data
    }
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(report_to_save, f, ensure_ascii=False, indent=4)
        
    print("="*70)
    print(f"✅ 詳細的JSON報告已成功儲存至: {output_path}")


if __name__ == "__main__":
    main()

### 時間軸序老師逐字稿、板書、學生行為

#### 重新定義行為分類

In [68]:
# -*- coding: utf-8 -*-
import os
import json
import datetime
import re
import asyncio
from openai import AsyncAzureOpenAI , APIError, RateLimitError
from dotenv import load_dotenv
from collections import defaultdict
import time
import random
import numpy as np
from scipy import stats
# ==============================================================================
# --- ★★★【使用者設定區】★★★ ---
# ==============================================================================
TARGET_SESSION_TIME = "09/28"

SESSION_DATE = '0928'

# 統一根目錄（避免重複寫 C:\Users\User\Desktop\test）
BASE_DIR = r'C:\Users\User\Desktop\test'

# 各資料夾與檔案路徑
STUDENT_DATA_DIRECTORY = os.path.join(BASE_DIR, 'SynologyDrive', 'json_behavior')
TEACHER_TRANSCRIPT_PATH = os.path.join(BASE_DIR, 'SynologyDrive', 'image', '上課影片', SESSION_DATE, '老師', f'{SESSION_DATE}_3.txt')
BLACKBOARD_IMAGES_DIRECTORY = os.path.join(BASE_DIR, 'note_blackboard', f'{SESSION_DATE}_English_filtered')

# STUDENT_DATA_DIRECTORY = r'C:\Users\User\Desktop\test\SynologyDrive\json_behavior'
# TEACHER_TRANSCRIPT_PATH = r'C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0928\老師\0928.txt'
# BLACKBOARD_IMAGES_DIRECTORY = r'C:\Users\User\Desktop\test\note_blackboard\0928_English_filtered'

OUTPUT_ANALYSIS_DIRECTORY = r'C:\Users\User\Desktop\test\classroom_analysis_report'
OUTPUT_ANALYSIS_JSON_PATH = os.path.join(OUTPUT_ANALYSIS_DIRECTORY, f'classroom_analysis_report_{TARGET_SESSION_TIME.replace("/", "")}_2.json')
TIME_INTERVAL_SECONDS = 30
CHUNK_SIZE_MINUTES = 20

# 設定 AI 分析任務的最大同時執行數量，避免觸發速率限制
MAX_CONCURRENT_AI_TASKS = 5

# ==============================================================================
# --- ★★★【全域常數設定區 (新增)】★★★ ---
# ==============================================================================
# 1. 主動行為參與 (Active Behavioral Engagement)
# 學生主動執行學習任務，是認知參與的強力指標。
ACTIVE_BEHAVIORAL_ENGAGEMENT = {
    "做筆記", "翻書", "主動舉手"
}

# 2. 被動行為參與 (Passive Behavioral Engagement)
# 學生遵守規範並接收資訊，是聽講時的有效參與形式。
PASSIVE_BEHAVIORAL_ENGAGEMENT = {
    "目視教師", "目視黑板", "坐姿直立"
}

# 3. 行為分心 (Behavioral Disengagement)
# 學生從事與當前學習任務無關的行為 (Off-task)。
BEHAVIORAL_DISENGAGEMENT = {
    "目視同學", "目視他處", "玩弄手部/文具", 
    "低頭(非學習)", "趴睡", "飲食", "喝水"
}

# 4. 程序性/模糊行為 (Procedural / Ambiguous Behavior)
# 行為本身模棱兩可，無法直接推斷其背後的認知狀態，需謹慎解讀。
AMBIGUOUS_BEHAVIORS = {
    "托腮", "觸摸臉部", "觸摸頭髮", 
    "被動舉手", "身體前傾", "身體後靠", 
    "被遮擋/無法判斷"
}

PROFESSIONAL_KEYWORDS = {
    "分詞", "句型", "形容詞", "語法", "時態", "子句", "完成式" 
}

CONTEXTUAL_GLOSSARY = {
    # --- 原有規則 ---
    "數學": ["初學"],
    "橫掃大陸": ["風收大陸"],
    "scare me": ["young me"],
    "吳宋熙": ["地圖鑑美吳宋熙"],

    # --- 新增規則 ---
    # 教學情境與口語
    "D": ["豬"],                   # 對答案時的常見念法
    "打罵教育": ["撒爸教育"],
    "結屎面": ["結一臉"],
    
    # 同音異義詞
    "鑫鑫腸": ["清場", "心情講"],    # 烤肉情境
    "中秋節": ["週週節"],          # 烤肉情境
    "bored": ["board"],             # 文法講解情境
    
    # 專有名詞
    "承恩": ["成恩"],
    
    # 英文詞彙
    "special magic tricks": ["spatial magic tracks"]
}

CLASSROOM_STATES_DEFINITIONS = {
    "下課休息": """
【最高優先級與獨佔性規則】判斷依據：
1.  **核心時間窗口**: 此狀態【僅會出現一次】，且必須發生在課程開始後的【60分鐘至120分鐘之間】。
2.  **觸發與合併**: 在上述時間窗口內，當老師明確說出「下課」、「休息一下」等關鍵詞後，該狀態即啟動。你必須將此觸發點之後【所有連續的、非教學性質】的片段（如`教學靜默`、`閒聊`）全部合併進來，形成一個【持續至少5分鐘，但不超過15分鐘】的單一 `下課休息` 區塊。
3.  **核心任務**: 你的目標是在指定時間窗口內，識別並合併出**唯一一個、連續的、符合時長**的休息時段。這是一個宏觀判斷，不要被零碎的對話打斷。
""",
    "教師講解": """
【核心定義】：老師作為主要發言者，圍繞【單一教學主題】（如一個文法點、一本書的解析）進行的、連續的知識輸出。
### ★★★ 規則強化 v2.1 ★★★ ###
【包含情境】：
1.  **教學性舉例 (Instructional Analogy)**: 為了闡述觀念而引用的**簡短**故事、比喻或個人經驗。**判斷關鍵**：只要其目的是輔助理解，且**在90秒內能重新連結回教學主題**，仍屬於此狀態。
2.  **即時性確認 (Comprehension Checks)**: 穿插在講解中的簡短問答，如「懂嗎？」、「對不對？」、「有沒有問題？」，這是講解流程的一部分，不應切斷此狀態。
3.  **教學框架內的設問與鋪陳 (Rhetorical Framing)**: 老師為了引出一個複雜概念而進行的自問自答或背景鋪陳（例如：「那到底什麼是『完成式』呢？我們得先從時間軸談起...」），只要其最終指向清晰的教學目標，就應被視為 `教師講解` 的框架。
【排除情境】：
1.  **長時間的題外話**: 當話題完全脫離當前教學軌道，且持續佔據主要時間（**超過2分鐘以上**），應被切分為【閒聊】。
2.  **系統性對答案**: 逐題式的、以核對答案為主的環節（例如，老師連續唸出 "C, B, A, C, D..."），應歸類為【師生互動】。
""",
    "學生練習": """
【觸發條件】：老師下達【明確的、要求學生獨立操作】的指令，且目的是為了鞏固剛教過的知識。
【關鍵詞】：「大家練習一下」、「給你們幾分鐘寫」、「現在動筆」、「試試看」、「把背面寫一下」。
【確認信號】：指令後通常會伴隨【長時間的教師靜默】或【低音量的同儕討論】。此狀態應包含整個操作時段，直到老師明確收回主導權。
【注意】：此模式不應出現在課程最開始的30分鐘內，除非有非常明確的非考試練習指令。
""",
    "學生考試": """
### ★★★ 規則強化 v3.0 ★★★ ###
【核心定義】：這是一個持續性的狀態，而非單一事件。必須同時滿足以下兩個條件才能被歸類為此模式：
1.  **時間窗口**: **【高優先級規則】** 此狀態**幾乎只會出現在【課程開始的前30分鐘內】**。
2.  **觸發與持續**:
    a. **觸發**: 老師明確說出「考試」、「前測」、「測驗」、「發考卷」等指令性關鍵詞。
    b. **持續**: 在該指令之後，出現了【長時間的、以無語音或極零碎話語為主】的時段。這些時段即使被微觀分析為 `學生練習` 或 `教學靜默`，你也應該根據宏觀情境將它們統一標記為 `學生考試`。
【注意】：如果老師在課程中段提到「考試」一詞，但其目的是在「講解」考試重點，則**不能**歸類為此模式。判斷的關鍵在於**指令**加上**後續的靜默行為**。
""",
    "師生互動": """
【核心定義】：所有非單向教學的、具有【明確教學或管理目的】的雙向或多向交流。
### ★★★ 規則強化 v2.1 ★★★ ###
【明確包含】：
1.  **結構化問答 (Structured Q&A)**: 老師針對【當前知識點】進行的多回合提問與解答，或學生主動提出的問題。
2.  **逐題檢討與對答案 (Answer Checking)**: 【高優先級規則】老師帶領全班**逐題核對**練習或考卷答案的環節，無論其中是否穿插簡短講解，主體都應被定義為 `師生互動`。
3.  **課堂管理 (Classroom Management)**: 宣布作業、提醒課程規劃、處理學生紀律問題、點名等。
【明確排除】：
1.  **教學設問 (Rhetorical Questions)**: 老師為了引出概念的自問自答式提問，應屬於【教師講解】。
2.  與教學目標推進無關的寒暄或多方閒聊，應屬於【閒聊】。
""",
    "閒聊": """
【核心定義】：內容與【當前學科知識點的推進】及【課堂管理】完全無關的非正式交流，且佔據了主要的對話時間。
【主要功能】：常作為高強度【教師講解】前後的【認知緩衝 (Cognitive Break)】或師生關係的潤滑劑。
### ★★★ 規則強化 v2.1 ★★★ ###
【明確包含】：
1.  **長時間的個人故事**: 老師或學生分享的完整個人經歷（例如軍中故事、旅遊經驗），且**持續超過2分鐘**或**未在短時間內連結回教學主題**。
2.  **延伸的題外話 (Digression)**: 由某個教學點引出，但內容已完全發散到與課程無關的領域（例如，由英文單字延伸到歷史故事、個人政治觀點、影視評論等）。
【判斷準則 v2.1】：
判斷的關鍵在於**「話題是否脫離教學軌道」**。如果一個題外話在1-2分鐘內被老師重新連結回教學主題，則應視為 `教師講解` 中的「教學性舉例」；反之，如果話題持續發散且時間較長，則必須歸類為 `閒聊`。
""",
}

# ==============================================================================
# --- API 金鑰與 Client 初始化 ---
# ==============================================================================
print("--- Classroom Analysis Engine v5.4 (Async Edition) ---")
print("步驟 1/8: 初始化環境與 Azure OpenAI Async Client...")
load_dotenv()
AZURE_API_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
TEXT_DEPLOYMENT_NAME = os.getenv("CHAT_COMPLETION_NAME")

if not all([AZURE_API_KEY, AZURE_ENDPOINT, TEXT_DEPLOYMENT_NAME]):
    print("❌ 錯誤：缺少必要的 Azure OpenAI 環境變數。請檢查 .env 檔案。")
    exit()

try:
    # 【修改】使用 AsyncAzureOpenAI 實例化 client
    async_client = AsyncAzureOpenAI(
        api_key=AZURE_API_KEY, 
        azure_endpoint=AZURE_ENDPOINT, 
        api_version="2024-02-01",
        max_retries=3 # 讓 client 內建一些基礎的重試邏輯
    )
    print("✅ Azure OpenAI Async client 初始化成功。")
except Exception as e:
    print(f"❌ 初始化 Azure OpenAI Async Client 時發生錯誤: {e}")
    exit()

# ==============================================================================
# --- 輔助函式與資料讀取模組 ---
# ==============================================================================
def clean_json_string(json_string: str) -> str:
    """
    (v5.5 新增) 清洗AI返回的字符串，移除其中非法的控制字符，以避免JSON解析錯誤。
    """
    # 建立一個正則表達式，匹配所有不被JSON標準允許的控制字符
    # \x00-\x1F 是主要的控制字符範圍, 但我們需要保留 \b, \f, \n, \r, \t
    # 正則表達式 [^\x20-\x7E\b\f\n\r\t] 匹配所有非可打印ASCII字符且不是合法JSON轉義符的字符
    # 為了更廣泛地處理Unicode，我們使用一個更簡單的方法：只保留 "合法的" 字符。
    
    cleaned_chars = []
    for char in json_string:
        # JSON 規範允許的控制字符是 \b, \f, \n, \r, \t
        if ord(char) < 32 and char not in '\b\f\n\r\t':
            # 如果是其他控制字符，則跳過
            continue
        cleaned_chars.append(char)
        
    return "".join(cleaned_chars)

def parse_timestamp(timestamp_str):
    """
    (v5.11 修正版) 解析多種時間戳格式，包括 H:MM:SS 和 HH:MM:SS。
    """
    h, m, s = 0, 0, 0
    
    # 處理圖片文件名格式
    match = re.match(r'(\d{2})-(\d{2})-(\d{2})-(\d{3})\.jpg', timestamp_str)
    if match: 
        h, m, s, _ = map(int, match.groups())
        return h * 3600 + m * 60 + s
        
    # --- ★★★【核心修正點：讓小時部分可以匹配1位或2位數字】★★★ ---
    # \d{1,2} 表示匹配1到2個數字
    match = re.match(r'(\d{1,2}):(\d{2}):(\d{2})', timestamp_str)
    if match: 
        parts = list(map(int, match.groups()))
        if len(parts) == 3:
            h, m, s = parts
            return h * 3600 + m * 60 + s
        # 有時候 timedelta 可能會輸出 H:MM 這樣的格式，這裡做一個兼容
        elif len(parts) == 2:
            m, s = parts
            return m * 60 + s
    # --- ★★★【修正結束】★★★ ---

    # 處理 WhisperX 的格式
    match = re.search(r'(\d{2})h(\d{2})m(\d{2})s', timestamp_str)
    if match: 
        h, m, s = map(int, match.groups())
        return h * 3600 + m * 60 + s
        
    return None

### --- MODIFIED SECTION --- ###
def load_student_data(directory, session_time):
    """(v4.1) 讀取並整合所有學生的行為數據，並加入數據完整性檢查"""
    print(f"步驟 2/8: 讀取學生行為數據 (目標課堂: {session_time})...")
    all_behaviors = []
    if not os.path.isdir(directory):
        print(f"❌ 錯誤：找不到學生資料夾 '{directory}'。")
        return None, 0
        
    student_folders = [f for f in os.listdir(directory) if os.path.isdir(os.path.join(directory, f))]
    loaded_students_count = 0
    
    for student_name in student_folders:
        student_dir = os.path.join(directory, student_name)
        for filename in os.listdir(student_dir):
            if filename.endswith('.json'):
                filepath = os.path.join(student_dir, filename)
                try:
                    with open(filepath, 'r', encoding='utf-8') as f: data = json.load(f)
                    
                    if data.get("report_metadata", {}).get("report_generation_time") == session_time:
                        student_id = data.get("report_metadata", {}).get("student_id", "未知學生")
                        
                        for behavior_list in data.get('detailed_sequence_analysis', []):
                            image_filenames = behavior_list.get('image_filenames_in_batch', [])
                            image_filenames_len = len(image_filenames)
                            
                            for image_highlight in behavior_list.get('analysis', {}).get('per_image_highlights', []):
                                
                                # ★★★【安全檢查】★★★
                                # 在存取前，先檢查索引是否合法
                                index = image_highlight.get('image_index_in_sequence')
                                if index is not None and index < image_filenames_len:
                                    img_filename = image_filenames[index]
                                    seconds = parse_timestamp(img_filename)
                                    if seconds is not None:
                                        all_behaviors.append({
                                            "seconds": seconds, 
                                            "student_id": student_id, 
                                            "behaviors": image_highlight.get("behavior_category", [])
                                        })
                                else:
                                    # 如果索引不合法，則印出警告並跳過此筆紀錄
                                    print(f"  - ⚠️ 數據不一致警告：在檔案 '{filename}' 中，偵測到無效的 image_index_in_sequence: {index} (列表長度為 {image_filenames_len})。將跳過此筆紀錄。")
                        
                        # print(f"  - 已載入 '{student_name}' 的報告。") # 為減少輸出訊息，可註解此行
                        loaded_students_count +=1
                        
                except Exception as e:
                    # 將 list index out of range 錯誤明確化
                    if isinstance(e, IndexError):
                         print(f"  - ⚠️ 嚴重數據錯誤：在解析 '{filename}' 時發生 'list index out of range'。這通常是源檔案數據不一致導致的。")
                    else:
                         print(f"  - ⚠️ 警告：讀取或解析 '{filename}' 時出錯: {e}")

    if not all_behaviors:
        print(f"❌ 錯誤：在 '{directory}' 中找不到任何符合 '{session_time}' 的學生報告。")
        return None, 0
        
    max_time = max(b['seconds'] for b in all_behaviors) if all_behaviors else 0
    print(f"✅ 成功載入 {loaded_students_count} 位學生的行為數據，課程總時長約 {max_time // 60} 分鐘。")
    return sorted(all_behaviors, key=lambda x: x['seconds']), max_time
### --- END MODIFIED SECTION --- ###

def load_teacher_transcript(filepath):
    """
    (v8.2 格式適應版) 
    讀取新的逐字稿格式 (HH:MM:SS: [Text])，並推斷出每個片段的結束時間。
    """
    print("步驟 3/8: 讀取老師逐字稿 (採用新格式解析器)...")
    if not os.path.exists(filepath):
        print(f"  - ⚠️ 警告：找不到逐字稿檔案 '{filepath}'，將跳過此項分析。")
        return []
    
    intermediate_data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            # --- 核心修改：更新正規表示式以匹配新格式 ---
            # 匹配 "HH:MM:SS: (可選的說話者標籤) [內容]"
            match = re.match(r'(\d{1,2}:\d{2}:\d{2}):\s*(?:[A-Z]:\s*)?(.+)', line)
            if match:
                start_str, text = match.groups()
                start_seconds = parse_timestamp(start_str)
                
                if start_seconds is not None and text.strip():
                    intermediate_data.append({
                        "start_seconds": start_seconds,
                        "text": text.strip()
                    })

    if not intermediate_data:
        print("  - ⚠️ 警告：逐字稿檔案為空或格式不符，無法解析任何內容。")
        return []

    # --- 推斷結束時間 (並加入時間上限防止異常) ---
    transcript_data = []
    MAX_SEGMENT_DURATION_SECONDS = 15  # 設定單句的最長持續時間上限

    for i in range(len(intermediate_data)):
        current_entry = intermediate_data[i]
        
        # 預設結束時間為開始時間 + 上限
        end_seconds = current_entry['start_seconds'] + MAX_SEGMENT_DURATION_SECONDS
        
        # 如果不是最後一個片段，嘗試用下一個片段的開始時間作為結束時間
        if i < len(intermediate_data) - 1:
            next_start_seconds = intermediate_data[i+1]['start_seconds']
            # 取兩者中的較小值，避免因長靜音導致持續時間過長
            end_seconds = min(end_seconds, next_start_seconds)

        if current_entry['start_seconds'] < end_seconds:
            transcript_data.append({
                "start_seconds": current_entry['start_seconds'],
                "end_seconds": end_seconds,
                "text": current_entry['text']
            })

    print(f"✅ 新格式逐字稿載入並處理完成，共 {len(transcript_data)} 筆有效片段。")
    return transcript_data

def load_blackboard_images(directory):
    print("步驟 4/8: 讀取板書圖片列表...")
    if not os.path.isdir(directory): print(f"  - ⚠️ 警告：找不到板書圖片資料夾 '{directory}'，將跳過此項分析。"); return []
    image_data = []
    for filename in os.listdir(directory):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            seconds = parse_timestamp(filename)
            if seconds is not None: image_data.append({"seconds": seconds, "filename": filename})
    print("✅ 板書圖片列表載入完成。")
    return sorted(image_data, key=lambda x: x['seconds'])

def find_state_for_timestamp(seconds: int, state_timeline: list):
    """根據秒數，從預先分析好的狀態時間軸中查詢對應的教學模式"""
    for state in state_timeline:
        # 假設 start_seconds 和 end_seconds 已經在 generate_state_timeline_async 中被計算好
        if state.get('start_seconds') is not None and state.get('end_seconds') is not None:
            if state['start_seconds'] <= seconds < state['end_seconds']:
                return state['classroom_state']
    # 如果課程結束後還有時間戳（例如學生數據比逐字稿長），給一個預設值
    return "課後" 

def force_apply_exam_rule_v2(master_timeline: list, total_seconds: int) -> list:
    """
    (全新後處理規則 v2) 在微觀分類後，強制應用更精準的宏觀考試規則。
    1. 只在課程前/後 30 分鐘生效。
    2. 尋找由 "考試指令關鍵字" 觸發的，且後續是長時間靜默/零碎語音的連續時段。
    3. 如果該時段超過15分鐘，則將其全部覆蓋為 "學生考試"。
    """
    print("  -> 正在執行【強制性 v2.0 - 精準考試規則】後處理...")
    if not master_timeline:
        return []

    # 定義時間窗口和門檻
    FIRST_WINDOW_END = 1800  # 前30分鐘
    LAST_WINDOW_START = total_seconds - 1800 # 後30分鐘
    MIN_EXAM_DURATION = 900 # 連續15分鐘
    LOOK_AHEAD_INTERVALS = 30 # 往後看 15 分鐘來確認是否為安靜時段
    
    # 定義觸發考試的關鍵字
    EXAM_TRIGGER_KEYWORDS = {"考試", "前測", "測驗", "發考卷"}
    
    # 定義可以被視為 "考試中" 的狀態
    QUIET_EXAM_LIKE_MODES = {"教學靜默", "學生練習", "學生考試"}
    FRAGMENTED_SPEECH_THRESHOLD = 10 # 考試期間老師可能會有簡短指令，字數少於10算零碎語音

    exam_start_index = -1

    # --- 掃描時間軸，只尋找 "觸發點" ---
    for i, interval in enumerate(master_timeline):
        seconds = interval["seconds"]
        is_in_start_window = (seconds < FIRST_WINDOW_END)
        
        # 規則：只在課程前30分鐘尋找考試的 "開始"
        if not is_in_start_window:
            continue

        speech = interval.get("teacher_speech", "")
        # 如果在時間窗口內，且老師的話語中包含觸發關鍵字
        if any(keyword in speech for keyword in EXAM_TRIGGER_KEYWORDS):
            
            # 檢查後續時段是否足夠安靜
            quiet_count = 0
            for j in range(i + 1, min(i + 1 + LOOK_AHEAD_INTERVALS, len(master_timeline))):
                next_interval = master_timeline[j]
                is_quiet_mode = next_interval["teaching_mode"] in QUIET_EXAM_LIKE_MODES
                is_fragmented = len(next_interval.get("teacher_speech", "")) < FRAGMENTED_SPEECH_THRESHOLD
                if is_quiet_mode or is_fragmented:
                    quiet_count += 1
            
            # 如果後續15分鐘內，超過80%的時間都是安靜的，我們就認為考試開始了
            if quiet_count / LOOK_AHEAD_INTERVALS > 0.8:
                exam_start_index = i
                print(f"  -> 在 {interval['start_time']} 偵測到考試觸發關鍵字：'{speech}'，並確認後續為長時段靜默。")
                break # 找到第一個就夠了

    # --- 如果找到了開始點，則向後延伸覆蓋，直到遇到明顯的教學活動 ---
    if exam_start_index != -1:
        # 從觸發點的下一個片段開始覆蓋
        for i in range(exam_start_index + 1, len(master_timeline)):
            interval = master_timeline[i]
            is_quiet_mode = interval["teaching_mode"] in QUIET_EXAM_LIKE_MODES
            is_fragmented = len(interval.get("teacher_speech", "")) < FRAGMENTED_SPEECH_THRESHOLD

            if is_quiet_mode or is_fragmented:
                interval["teaching_mode"] = "學生考試"
            else:
                # 一旦遇到明顯的教學活動（例如：老師開始高強度講解），就停止覆蓋
                end_time = interval["start_time"]
                print(f"  -> 考試狀態覆蓋至 {end_time} 結束。")
                break
    else:
        print("  -> 在指定窗口內未找到符合 '指令+長時段靜默' 模式的考試片段。")

    # (您也可以在這裡加入對課程結尾30分鐘的判斷，邏輯類似，但較少見，我們先專注於課前考試)

    return master_timeline

def force_practice_to_exam_in_first_30_mins(master_timeline: list) -> list:
    """
    在微觀分類後，強制將課程前30分鐘內的「學生練習」模式校正為「學生考試」。
    這是一個基於宏觀時間規則的硬性覆蓋，用以解決微觀分析的局限性。
    """
    print("  -> 正在執行【強制性規則】：校正前30分鐘的「學生練習」為「學生考試」...")
    
    # 定義時間窗口 (前30分鐘 = 1800秒)
    TIME_WINDOW_END_SECONDS = 1800 

    corrected_count = 0
    for interval in master_timeline:
        # 檢查是否在時間窗口內
        if interval["seconds"] < TIME_WINDOW_END_SECONDS:
            # 如果模式被微觀分類為「學生練習」，則強制修改
            if interval["teaching_mode"] == "學生練習":
                interval["teaching_mode"] = "學生考試"
                corrected_count += 1
    
    if corrected_count > 0:
        print(f"  -> 校正完成：共 {corrected_count} 個「學生練習」片段被重新標記為「學生考試」。")
    else:
        print("  -> 在前30分鐘內未發現需校正的「學生練習」片段。")
        
    return master_timeline

def force_consolidate_break_time(master_timeline: list) -> list:
    """
    在微觀分類後，強制尋找「下課休息」觸發點，並將其後續的非教學活動
    （如學生練習、閒聊、靜默）合併為一個連續的「下課休息」區塊。
    此版本將確保一個最少5分鐘，最多15分鐘的休息區塊。
    """
    print("  -> 正在執行【強制性規則】：合併與固化「下課休息」時段...")

    BREAK_WINDOW_START_SECONDS = 3600  # 60分鐘
    BREAK_WINDOW_END_SECONDS = 7200    # 120分鐘
    MIN_BREAK_DURATION_SECONDS = 300   # 強制最少休息5分鐘
    MAX_BREAK_DURATION_SECONDS = 900   # 強制最多休息15分鐘
    
    BREAK_TRIGGER_KEYWORDS = {"下課", "休息"}
    # 這些是明確的教學活動，遇到它們就應該停止合併休息時間
    STOP_MERGING_MODES = {"教師講解", "師生互動"}
    
    break_trigger_index = -1

    # 1. 在指定時間窗口內，尋找第一個觸發「下課休息」指令的區間
    for i, interval in enumerate(master_timeline):
        seconds = interval["seconds"]
        if BREAK_WINDOW_START_SECONDS <= seconds < BREAK_WINDOW_END_SECONDS:
            speech = interval.get("teacher_speech", "")
            if any(keyword in speech for keyword in BREAK_TRIGGER_KEYWORDS):
                break_trigger_index = i
                print(f"  -> 在 {interval['start_time']} 偵測到下課休息觸發點。開始向後合併...")
                break

    # 2. 如果找到了觸發點，則開始向後強制覆蓋
    if break_trigger_index != -1:
        break_start_seconds = master_timeline[break_trigger_index]["seconds"]
        
        # 從觸發點開始，一直向後檢查
        for i in range(break_trigger_index, len(master_timeline)):
            current_interval = master_timeline[i]
            current_seconds = current_interval["seconds"]
            duration_from_start = current_seconds - break_start_seconds

            # 停止條件 1: 超過了最大休息時間
            if duration_from_start >= MAX_BREAK_DURATION_SECONDS:
                print(f"  -> 達到15分鐘上限，於 {current_interval['start_time']} 停止合併。")
                break
                
            # 停止條件 2: 遇到了明確的教學活動，且已經滿足了最小休息時間
            if current_interval["teaching_mode"] in STOP_MERGING_MODES and duration_from_start >= MIN_BREAK_DURATION_SECONDS:
                print(f"  -> 偵測到教學活動於 {current_interval['start_time']} 開始，停止合併。")
                break
            
            # 如果不滿足停止條件，就強制覆蓋模式為「下課休息」
            current_interval["teaching_mode"] = "下課休息"
        else: 
            # 如果循環正常結束（沒有被break），說明休息直到課程結束
            print("  -> 下課休息時段合併至課程結束。")

    else:
        print("  -> 在 60-120 分鐘窗口內未找到明確的下課休息觸發點。")

    return master_timeline

def post_process_silent_and_noisy_intervals(master_timeline: list) -> list:
    """
    (全新後處理規則) 在微觀分類後，處理無逐字稿或僅有噪音的片段。
    規則 1 (高優先級): 在課程前30分鐘內，將所有「無逐字稿」或「無法辨識的噪音」的片段強制歸類為「學生考試」。
    規則 2 (一般情況): 在課程30分鐘後，將「無逐字稿」的片段歸類為前一個時間區段的教學模式，以達到合併效果。
    """
    print("  -> 正在執行【後處理規則】：處理無逐字稿與噪音片段...")
    if not master_timeline:
        return []

    # 定義時間窗口 (前30分鐘 = 1800秒)
    TIME_WINDOW_END_SECONDS = 1800
    NOISE_MARKER = "[無法辨識的噪音]"
    
    corrected_exam_count = 0
    merged_silent_count = 0

    # 我們從第二個片段開始循環，因為第一個片段沒有 "前一個" 模式可以參考
    for i in range(1, len(master_timeline)):
        current_interval = master_timeline[i]
        prev_interval = master_timeline[i-1]
        
        speech = current_interval.get("teacher_speech", "").strip()
        is_silent = not speech
        is_noise = NOISE_MARKER in speech
        
        # --- 規則 1: 處理課堂前30分鐘的特殊情況 ---
        if current_interval["seconds"] < TIME_WINDOW_END_SECONDS:
            if is_silent or is_noise:
                if current_interval["teaching_mode"] != "學生考試":
                    current_interval["teaching_mode"] = "學生考試"
                    corrected_exam_count += 1
                # 應用此規則後，跳過後續的通用規則
                continue
        
        # --- 規則 2: 處理30分鐘後的一般無逐字稿情況 ---
        if is_silent:
            prev_mode = prev_interval.get("teaching_mode")
            if prev_mode:
                current_interval["teaching_mode"] = prev_mode
                merged_silent_count += 1
    
    if corrected_exam_count > 0:
        print(f"  -> 校正完成：共 {corrected_exam_count} 個片段被強制歸類為「學生考試」。")
    if merged_silent_count > 0:
         print(f"  -> 合併完成：共 {merged_silent_count} 個無逐字稿片段已併入前一教學模式。")
    if corrected_exam_count == 0 and merged_silent_count == 0:
        print("  -> 未發現需要處理的無逐字稿或噪音片段。")

    return master_timeline

async def create_master_timeline(student_behaviors, transcript, blackboard_images, total_seconds, interval, state_timeline=None):
    """
    (v10.0 職責分離版) 建立整合時間軸，僅專注於數據整合與 AI 微觀分類。
    語速相關的計算將移至後處理函式中。
    """
    print("步驟 5/8: 建立整合時間軸 (採用【微觀教學模式即時分類 v10.0】)...")

    # --- Phase 1: 數據準備 (與之前相同) ---
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_AI_TASKS)
    intervals_to_process = []
    for start_interval in range(0, total_seconds + interval, interval):
        # ... (這整個 for 迴圈的內容完全不變，請保持原樣) ...
        end_interval = start_interval + interval
        interval_data = {
            "seconds": start_interval,
            "start_time": str(datetime.timedelta(seconds=start_interval)), 
            "end_time": str(datetime.timedelta(seconds=end_interval)), 
            "teacher_speech": " ".join([entry['text'] for entry in transcript if start_interval <= entry['start_seconds'] < end_interval]),
            "blackboard_snapshots": [image['filename'] for image in blackboard_images if start_interval <= image['seconds'] < end_interval],
            "student_behavior_summary": defaultdict(list), 
            "focus_distribution": None,
            "cognitive_load_metrics": None, # ★★★ 注意：這裡依然保留，但會在後續步驟中填充
            "teaching_mode": None
        }
        active_engagement_students = set()
        passive_engagement_students = set()
        disengagement_students = set()
        ambiguous_behavior_students = set()
        behaviors_in_interval = [b for b in student_behaviors if start_interval <= b['seconds'] < end_interval]
        for behavior in behaviors_in_interval:
            student_id = behavior['student_id']
            for b_category in behavior['behaviors']:
                interval_data["student_behavior_summary"][b_category].append(student_id)
                if b_category in ACTIVE_BEHAVIORAL_ENGAGEMENT: active_engagement_students.add(student_id)
                elif b_category in PASSIVE_BEHAVIORAL_ENGAGEMENT: passive_engagement_students.add(student_id)
                elif b_category in BEHAVIORAL_DISENGAGEMENT: disengagement_students.add(student_id)
                elif b_category in AMBIGUOUS_BEHAVIORS: ambiguous_behavior_students.add(student_id)
        all_students_in_interval = active_engagement_students.union(passive_engagement_students, disengagement_students, ambiguous_behavior_students)
        total_active_students = len(all_students_in_interval)
        if total_active_students > 0:
            interval_data["focus_distribution"] = {
                "active_behavioral_engagement": round((len(active_engagement_students) / total_active_students) * 100, 1),
                "passive_behavioral_engagement": round((len(passive_engagement_students) / total_active_students) * 100, 1),
                "behavioral_disengagement": round((len(disengagement_students) / total_active_students) * 100, 1),
                "ambiguous_behavior": round((len(ambiguous_behavior_students) / total_active_students) * 100, 1)
            }
        interval_data["student_behavior_summary"] = {k: list(set(v)) for k, v in interval_data["student_behavior_summary"].items()}
        intervals_to_process.append(interval_data)


    # --- Phase 2: 並行AI分類 (與之前相同) ---
    async def classify_and_update(interval_d):
        async with semaphore:
            mode = await classify_interval_teaching_mode_async(
                interval_d["teacher_speech"], 
                interval_d["student_behavior_summary"],
                interval_d["blackboard_snapshots"]
            )
            interval_d["teaching_mode"] = mode
            return interval_d

    print(f"  -> 已將課程分為 {len(intervals_to_process)} 個微觀片段，開始並行 AI 分類...")
    classification_tasks = [classify_and_update(data) for data in intervals_to_process]
    master_timeline = await asyncio.gather(*classification_tasks) # 直接得到 master_timeline
    print("  -> 所有微觀片段分類完成。")
    
    # ★★★【【【 移除 Phase 3 】】】★★★
    # 不再需要第三輪循環，因為語速計算已經被移出

    print("✅ 整合時間軸建立完成。")
    return master_timeline

def merge_consecutive_states(detailed_timeline: list) -> list:
    """
    (全新後處理功能) 將詳細時間軸中連續且相同的教學模式片段合併，並計算持續時長。
    """
    if not detailed_timeline:
        return []

    merged_timeline = []
    # 使用第一個片段初始化
    current_block = {
        "start_time": detailed_timeline[0]["start_time"],
        "end_time": detailed_timeline[0]["end_time"],
        "duration_seconds": TIME_INTERVAL_SECONDS,
        "teaching_mode": detailed_timeline[0]["teaching_mode"],
        "transcript_snippets": [detailed_timeline[0]["teacher_speech"]]
    }

    for i in range(1, len(detailed_timeline)):
        current_interval = detailed_timeline[i]
        
        # 如果當前模式與正在記錄的區塊相同
        if current_interval["teaching_mode"] == current_block["teaching_mode"]:
            # 更新結束時間和持續時長
            current_block["end_time"] = current_interval["end_time"]
            current_block["duration_seconds"] += TIME_INTERVAL_SECONDS
            if current_interval["teacher_speech"]:
                current_block["transcript_snippets"].append(current_interval["teacher_speech"])
        else:
            # 模式發生變化，先將上一個區塊儲存起來
            merged_timeline.append(current_block)
            # 然後開始一個新的區塊
            current_block = {
                "start_time": current_interval["start_time"],
                "end_time": current_interval["end_time"],
                "duration_seconds": TIME_INTERVAL_SECONDS,
                "teaching_mode": current_interval["teaching_mode"],
                "transcript_snippets": [current_interval["teacher_speech"]]
            }

    # 不要忘記儲存最後一個區塊
    merged_timeline.append(current_block)

    # 最後，將秒數轉換為更易讀的分秒格式，並整理逐字稿
    for block in merged_timeline:
        minutes, seconds = divmod(block["duration_seconds"], 60)
        block["duration_formatted"] = f"{int(minutes)} 分 {int(seconds)} 秒"
        block["full_transcript"] = " ".join(filter(None, block["transcript_snippets"])).strip()
        del block["transcript_snippets"] # 刪除臨時用的列表

    return merged_timeline

def merge_and_cleanup_timeline(detailed_timeline: list) -> list:
    """
    (全新後處理功能 v2.0)
    1. 移除 "教學靜默" 模式，將其歸入前一個教學模式。
    2. 將詳細時間軸中連續且相同的教學模式片段合併，並計算持續時長。
    """
    print("步驟 5.9/8: 執行時間軸合併與清理...")
    if not detailed_timeline:
        return []

    # --- 階段 1: 清理 "教學靜默" ---
    cleaned_timeline = list(detailed_timeline) # 創建副本以避免修改原始列表
    for i in range(1, len(cleaned_timeline)):
        if cleaned_timeline[i]["teaching_mode"] == "教學靜默":
            # 將靜默狀態歸入前一個狀態
            cleaned_timeline[i]["teaching_mode"] = cleaned_timeline[i-1]["teaching_mode"]

    # --- 階段 2: 合併連續模式 ---
    merged_timeline = []
    if not cleaned_timeline:
        return []

    # 使用第一個片段初始化
    current_block = {
        "start_time": cleaned_timeline[0]["start_time"],
        "end_time": cleaned_timeline[0]["end_time"],
        "duration_seconds": TIME_INTERVAL_SECONDS,
        "teaching_mode": cleaned_timeline[0]["teaching_mode"],
        "transcript_snippets": [cleaned_timeline[0]["teacher_speech"]]
    }

    for i in range(1, len(cleaned_timeline)):
        current_interval = cleaned_timeline[i]
        
        if current_interval["teaching_mode"] == current_block["teaching_mode"]:
            current_block["end_time"] = current_interval["end_time"]
            current_block["duration_seconds"] += TIME_INTERVAL_SECONDS
            if current_interval["teacher_speech"]:
                current_block["transcript_snippets"].append(current_interval["teacher_speech"])
        else:
            merged_timeline.append(current_block)
            current_block = {
                "start_time": current_interval["start_time"],
                "end_time": current_interval["end_time"],
                "duration_seconds": TIME_INTERVAL_SECONDS,
                "teaching_mode": current_interval["teaching_mode"],
                "transcript_snippets": [current_interval["teacher_speech"]]
            }

    merged_timeline.append(current_block)

    # --- 階段 3: 格式化輸出 ---
    for block in merged_timeline:
        minutes, seconds = divmod(block["duration_seconds"], 60)
        block["duration_formatted"] = f"{int(minutes)} 分 {int(seconds)} 秒"
        block["full_transcript"] = " ".join(filter(None, block["transcript_snippets"])).strip()
        if not block["full_transcript"]:
            block["full_transcript"] = "(無語音)"
        del block["transcript_snippets"]

    print("✅ 時間軸合併與清理完成。")
    return merged_timeline

def find_micro_events(master_timeline, peak_threshold=80.0, trough_threshold=40.0):
    """(v5.1 新增) 在完整時間軸上尋找專注度的高峰、低谷與關鍵轉折點"""
    print("步驟 5.5/8: 尋找微觀事件 (專注度高峰、低谷與轉折點)...")
    micro_events = []
    
    # 用於尋找最大轉折
    max_focus_rise = {"delta": 0, "event": None}
    max_disengagement_rise = {"delta": 0, "event": None}
    
    for i, interval in enumerate(master_timeline):
        dist = interval.get("focus_distribution")
        if not dist:
            continue

        # 檢查高峰和低谷
        if dist.get("task_oriented_focus", 0) >= peak_threshold:
            micro_events.append({
                "type": "專注度高峰 (任務導向)",
                "time": interval["start_time"],
                "details": f"任務導向專注度達到 {dist['task_oriented_focus']}%",
                "teacher_speech_context": interval.get("teacher_speech", "")
            })
        
        if dist.get("disengagement", 0) >= trough_threshold:
            micro_events.append({
                "type": "專注度低谷 (分心)",
                "time": interval["start_time"],
                "details": f"分心狀態佔比達到 {dist['disengagement']}%",
                "teacher_speech_context": interval.get("teacher_speech", "")
            })

        # 比較與前一個時間點的變化，尋找最大轉折
        if i > 0:
            prev_dist = master_timeline[i-1].get("focus_distribution")
            if prev_dist:
                focus_delta = dist.get("task_oriented_focus", 0) - prev_dist.get("task_oriented_focus", 0)
                disengagement_delta = dist.get("disengagement", 0) - prev_dist.get("disengagement", 0)
                
                if focus_delta > max_focus_rise["delta"]:
                    max_focus_rise["delta"] = focus_delta
                    max_focus_rise["event"] = {
                        "type": "關鍵拉升點",
                        "time": interval["start_time"],
                        "details": f"任務導向專注度在30秒內急遽上升 {focus_delta:.1f}%",
                        "teacher_speech_context": interval.get("teacher_speech", "")
                    }
                
                if disengagement_delta > max_disengagement_rise["delta"]:
                    max_disengagement_rise["delta"] = disengagement_delta
                    max_disengagement_rise["event"] = {
                        "type": "關鍵下跌點",
                        "time": interval["start_time"],
                        "details": f"分心狀態佔比在30秒內急遽上升 {disengagement_delta:.1f}%",
                        "teacher_speech_context": interval.get("teacher_speech", "")
                    }

    # 將找到的最大轉折點加入事件列表
    if max_focus_rise["event"]:
        micro_events.insert(0, max_focus_rise["event"]) # 放在最前面，因為很重要
    if max_disengagement_rise["event"]:
        micro_events.insert(0, max_disengagement_rise["event"])
        
    print(f"✅ 微觀事件分析完成，找到 {len(micro_events)} 個值得關注的時間點。")
    return micro_events

def aggregate_timeline(timeline, aggregate_interval_minutes=2):
    """
    將高粒度的時間軸數據聚合成較低粒度的平均值數據。
    (v5.11 修正版 - 增加對 None 值的安全檢查)
    """
    if not timeline:
        return []

    aggregate_interval_seconds = aggregate_interval_minutes * 60
    aggregated = []
    
    current_chunk_start = 0
    # ★★★【修正點 1：確保 timeline 非空，避免 IndexError】★★★
    last_second = timeline[-1]['seconds'] if timeline else 0

    while current_chunk_start <= last_second:
        chunk_end = current_chunk_start + aggregate_interval_seconds
        
        intervals_in_chunk = [
            item for item in timeline 
            if current_chunk_start <= item['seconds'] < chunk_end
        ]
        
        if intervals_in_chunk:
            # --- ★★★【修正點 2：在計算前，過濾掉 focus_distribution 為 None 的數據】★★★ ---
            valid_focus_data = [
                d['focus_distribution'] 
                for d in intervals_in_chunk 
                if d.get('focus_distribution') is not None
            ]
            
            # 只有在存在有效數據時才進行計算
            if valid_focus_data:
                num_valid_intervals = len(valid_focus_data)
                avg_task = sum(d.get('task_oriented_focus', 0) for d in valid_focus_data) / num_valid_intervals
                avg_receptive = sum(d.get('receptive_engagement', 0) for d in valid_focus_data) / num_valid_intervals
                avg_disengagement = sum(d.get('disengagement', 0) for d in valid_focus_data) / num_valid_intervals
                
                focus_distribution_summary = {
                    "task_oriented_focus": round(avg_task, 1),
                    "receptive_engagement": round(avg_receptive, 1),
                    "disengagement": round(avg_disengagement, 1)
                }
            else:
                # 如果這個區塊完全沒有專注度數據，則返回 None 或預設值
                focus_distribution_summary = None
            # --- ★★★【修正結束】★★★ ---

            # 合併老師話語
            combined_speech = " ".join(d.get('teacher_speech', '') for d in intervals_in_chunk if d.get('teacher_speech'))
            
            aggregated.append({
                "start_time": str(datetime.timedelta(seconds=current_chunk_start)),
                "end_time": str(datetime.timedelta(seconds=chunk_end)),
                "focus_distribution": focus_distribution_summary,
                "teacher_speech_summary": combined_speech[:100] + '...' if len(combined_speech) > 100 else combined_speech
            })
            
        current_chunk_start = chunk_end
        
    return aggregated

# ==============================================================================
# --- ★★★【v5.10 新增區塊：進階量化分析模組】★★★ ---
# ==============================================================================

def get_refine_trigger_prompt():
    return """
你是一位精煉語句的專家。你的唯一任務是從使用者提供的一段話中，提取出最核心、最能代表其指令或意圖的關鍵短語。

【規則】
1.  **極度簡潔**：只保留觸發動作或狀態改變的核心詞語。
2.  **移除贅詞**：刪除所有的鋪墊、語氣詞、無關的閒聊（例如 "好不好"、"真的假的"、"我不是要恐懼嚇你們"）。
3.  **保持原意**：提取的短語必須能獨立表達原始句子的核心指令。

【範例】
-   **輸入**: "還會跟我講這些奇奇怪怪的事情嗎?好不好,所以鼓勵大家,我不是要恐懼嚇你們,只是希望鼓勵你們多一個選擇,讓我們休息一下,讓大家消化一下,真的假的,還有嘛,有看過是嗎?還有不會的可以過來握手"
-   **輸出**: "讓我們休息一下, 讓大家消化一下"

-   **輸入**: "好，那我們來看第二個，我們等一下一併講這幾個，我們先看第四題，這一題哦。請問你如果after match, perfect, good news,你可以dance怎樣？"
-   **輸出**: "我們先看第四題"

你的回答只能是精煉後的短語，不要包含任何解釋。
"""

async def refine_single_quote_async(quote: str, semaphore: asyncio.Semaphore):
    """使用AI將單個長句子精煉成關鍵短語"""
    async with semaphore:
        system_prompt = get_refine_trigger_prompt()
        user_prompt = f"請精煉以下句子：\n---\n{quote}\n---"
        
        try:
            response = await async_client.chat.completions.create(
                model=TEXT_DEPLOYMENT_NAME,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                max_tokens=50,  # 精煉後的句子不需要太長
                temperature=0.0
            )
            return response.choices[0].message.content.strip().replace('"', '')
        except Exception as e:
            print(f"  - ⚠️ 精煉句子失敗: {e}")
            return quote # 如果失敗，就返回原始句子

def calculate_mode_durations(master_timeline, total_seconds, interval_seconds):
    """(v5.8) 量化並計算每種教學模式的持續時間"""
    print("步驟 5.6/8: 量化教學模式持續時間...")
    if not master_timeline: return {}
    mode_segments = defaultdict(list)
    if not master_timeline: return {}
    current_mode = master_timeline[0].get("teaching_mode")
    current_segment_duration = 0
    for interval_data in master_timeline:
        mode = interval_data.get("teaching_mode")
        if mode == current_mode:
            current_segment_duration += interval_seconds
        else:
            if current_mode: mode_segments[current_mode].append(current_segment_duration)
            current_mode = mode
            current_segment_duration = interval_seconds
    if current_mode and current_segment_duration > 0:
        mode_segments[current_mode].append(current_segment_duration)
    duration_summary = {}
    for mode, segments in mode_segments.items():
        total_duration = sum(segments)
        number_of_segments = len(segments)
        average_duration = total_duration / number_of_segments if number_of_segments > 0 else 0
        duration_summary[mode] = {
            "total_duration_seconds": total_duration,
            "total_duration_minutes": round(total_duration / 60, 1),
            "percentage_of_class": round((total_duration / total_seconds) * 100, 1) if total_seconds > 0 else 0,
            "number_of_segments": number_of_segments,
            "average_segment_duration_seconds": round(average_duration, 1)
        }
    print("✅ 教學模式持續時間量化完成。")
    return duration_summary

def find_important_keywords(master_timeline):
    """(v5.9) 找出教師提到特定關鍵字的時間點"""
    print("步驟 5.7/8: 標記教學關鍵字時間點...")
    KEYWORDS_TO_MARK = {"重點", "關鍵", "考試", "注意", "小心", "必考", "圈起來", "畫起來", "寫下來", "思考一下"}
    keyword_timestamps = []
    for i, interval in enumerate(master_timeline):
        speech = interval.get("teacher_speech", "")
        if any(keyword in speech for keyword in KEYWORDS_TO_MARK):
            matched_keyword = next((kw for kw in KEYWORDS_TO_MARK if kw in speech), None)
            context_before = master_timeline[i-1].get("teacher_speech", "") if i > 0 else ""
            context_after = master_timeline[i+1].get("teacher_speech", "") if i < len(master_timeline) - 1 else ""
            keyword_timestamps.append({
                "time": interval["start_time"], "keyword": matched_keyword, "full_quote": speech,
                "context_before": context_before, "context_after": context_after
            })
    print(f"✅ 成功標記出 {len(keyword_timestamps)} 個關鍵字時間點。")
    return keyword_timestamps

def find_high_intensity_segments(master_timeline: list, disengagement_threshold=30.0, rise_threshold=15.0) -> list:
    """
    (v1.0 新增) 掃描 master_timeline，找出所有「高強度教學」片段，
    並標記出學生分心度首次顯著上升的時間點。
    """
    print("步驟 5.7/8: 識別高強度教學片段與專注度下降點...")
    segments = []
    in_high_intensity_segment = False
    current_segment = None

    HIGH_INTENSITY_MODES = {"教師講解"}
    
    for i, interval in enumerate(master_timeline):
        mode = interval.get("teaching_mode")
        metrics = interval.get("cognitive_load_metrics")
        focus = interval.get("focus_distribution")

        is_high_intensity = (
            mode in HIGH_INTENSITY_MODES and
            metrics and
            metrics.get("intensity_level") in ["高強度", "中強度"]
        )

        # 偵測到一個新的高強度片段的開始
        if is_high_intensity and not in_high_intensity_segment:
            in_high_intensity_segment = True
            current_segment = {
                "start_time": interval["start_time"],
                "start_seconds": interval["seconds"],
                "breaking_point_time": None,
                "end_time": None,
                "duration_before_break_seconds": None,
                "trigger_reason": None,
                "initial_disengagement": focus.get("behavioral_disengagement", 0) if focus else 0
            }

        # 如果正在一個高強度片段中
        if in_high_intensity_segment:
            current_disengagement = focus.get("behavioral_disengagement", 0) if focus else 0
            
            # 檢查分心度是否觸發「引爆點」
            if current_segment and not current_segment["breaking_point_time"]:
                # 條件1：分心度超過一個絕對閾值
                if current_disengagement > disengagement_threshold:
                    current_segment["breaking_point_time"] = interval["start_time"]
                    current_segment["trigger_reason"] = f"分心度首次超過 {disengagement_threshold}%"
                # 條件2：分心度相比片段開始時，上升超過一個閾值
                elif (current_disengagement - current_segment["initial_disengagement"]) > rise_threshold:
                    current_segment["breaking_point_time"] = interval["start_time"]
                    current_segment["trigger_reason"] = f"分心度上升超過 {rise_threshold}%"

            # 如果高強度狀態結束，則關閉這個片段記錄
            if not is_high_intensity or i == len(master_timeline) - 1:
                in_high_intensity_segment = False
                if current_segment:
                    # 標記整個片段的結束時間
                    current_segment["end_time"] = interval["end_time"]
                    # 如果找到了引爆點，計算其持續時間
                    if current_segment["breaking_point_time"]:
                        break_point_seconds = parse_timestamp(current_segment["breaking_point_time"])
                        duration = break_point_seconds - current_segment["start_seconds"]
                        current_segment["duration_before_break_seconds"] = duration
                    segments.append(current_segment)
                    current_segment = None
                    
    print(f"✅ 高強度片段分析完成，共找到 {len(segments)} 個分析片段。")
    return segments
# ==============================================================================
# --- ★★★【v5.10 新增功能：量化語速變化趨勢】★★★ ---
# ==============================================================================
def get_state_timeline_prompt():
    """定義用於預處理逐字稿、生成宏觀時間軸的 AI 指令 (v13 策略升級版)"""
    # 將 CLASSROOM_STATES_DEFINITIONS 動態轉換為 Prompt 的一部分
    rules_description = ""
    for state, description in CLASSROOM_STATES_DEFINITIONS.items():
        rules_description += f"- **{state}**: {description.strip()}\n"

    ### ★★★ 【核心修改】：全面升級 System Prompt ★★★ ###
    return f"""
「你是一位頂尖的教育分析師，專長是從課堂對話中精準識別出教學活動的各個階段。你的任務是將一份完整的課堂逐字稿，切分成有意義、連續且分類明確的『課堂狀態』時間段。」

**【你的核心任務】**
分析使用者提供的【完整課堂逐字稿】，並以一個結構化的 JSON 物件作為輸出。你的分析必須涵蓋從頭到尾的整個逐字稿時間範圍。

**【課堂狀態的六個類別與判斷規則】**
你必須從以下六個類別中進行選擇，並嚴格遵循其定義：
{rules_description}

**【★★★ 分析策略：絕對優先級決策流程 (Absolute Priority Decision Flow) ★★★】**
為達到最高的分類準確性，你必須嚴格遵循以下決策流程，不得跳躍或違反順序：

**第一步：內容相關性篩選 (Content Relevance Filter - 最高優先級)**
- 這是你的**絕對第一步**。閱讀一段對話時，你必須先問自己：「這段對話的核心主題是否與當前的英文教學目標（例如文法、單字、解題、課堂管理）直接相關？」
- **如果答案為【否】**：無論其形式是獨白、問答還是多方對話，你都**必須**將其歸類為 **`閒聊`**。**此規則的優先級高於一切形式判斷。**
- **如果答案為【是】**：你才可以進入第二步。

**第二步：教學形式判斷 (Instructional Form Analysis)**
- 只有在確認內容與教學相關後，你才需要判斷其形式：
    - **單向知識輸出** -> 歸類為 **`教師講解`**。
    - **雙向問答、討論、對答案、課堂管理** -> 歸類為 **`師生互動`**。
    - **要求學生獨立操作的指令後，出現的長時間靜默** -> 歸類為 **`學生練習`** 或 **`學生考試`**。

**第三步：情境微調 (Contextual Refinement)**
- 在完成上述分類後，做最後的檢查。例如，一個極短（少於90秒）且目的明確是為了引出教學概念的比喻或故事，即使內容稍有偏離，也可以被視為 `教師講解` 的一部分。但這只是微調，不能違背第一步的最高原則。

**【常見錯誤分類警示 (Common Misclassification Alerts)】**
在進行「情境審核」時，請特別警惕以下常見錯誤：
- **錯誤範例 1 (講解誤判為閒聊)**: 將老師為了檢討考卷而逐題唸出答案（如 "CB諸位諸..."）的環節，錯誤地標記為 `閒聊`。
  - **✅ 正確做法**: 根據規則，此環節的核心目的是「對答案」，應歸類為 `師生互動`。
- **錯誤範例 2 (閒聊誤判為講解)**: 將老師分享個人飲食偏好（如「我最近愛上鍋皮辣椒雞」）或講述個人經歷的長篇獨白，僅因其形式是有結構的獨白而錯誤地標記為 `教師講解`。
  - **✅ 正確做法**: 識別出其核心內容與當前教學軌道無關，應歸類為 `閒聊`。
- **錯誤範例 3 (教學鋪陳誤判為互動)**: 將老師為了引出「子句」概念的設問句 `什麼叫子句？`，錯誤標記為 `師生互動`。
  - **✅ 正確做法**: 判斷出這是一個教學鋪陳，其後是連續的知識輸出，應歸類為 `教師講解`。

**【最終輸出要求 (JSON 格式)】**
你的回答**必須**是一個結構完整的 JSON 物件，絕對不能包含任何額外的文字、註解或 Markdown 標記。根物件需包含 `timeline` 鍵，其值為一個列表，每個元素都應包含 `start_time`, `end_time`, `classroom_state`, `state_summary`。時間必須是 "HH:MM:SS" 格式且連續。在你生成最終 JSON 之前，務必再次檢查並合併所有連續且相同的狀態區塊。
"""

def force_find_and_consolidate_break_v2_1(state_timeline: list) -> list:
    """
    (v2.1 精準修正版) 確保在60-120分鐘窗口內有且僅有一個「下課休息」時段。
    此版本只會合併連續的「閒聊」與「下課休息」片段，避免錯誤地吞噬「師生互動」。
    """
    print("  -> 正在執行【強制性 v2.1 - 精準合併模式】單次下課規則後處理...")
    
    BREAK_WINDOW_START_SECONDS = 3600  # 60分鐘
    BREAK_WINDOW_END_SECONDS = 7200    # 120分鐘
    MIN_BREAK_DURATION_SECONDS = 300   # 5分鐘 (可根據需求調整)
    
    # ★★★【核心修正點 1】★★★
    # 重新定義 "休息" 的核心組成，排除「師生互動」，因為它屬於教學活動。
    BREAK_CORE_STATES = {"閒聊", "下課休息"}

    window_states = []
    outside_window_states = []

    # 1. 將時間軸分割為 "窗口內" 與 "窗口外"
    for state in state_timeline:
        start_sec = state.get('start_seconds', 0)
        if BREAK_WINDOW_START_SECONDS <= start_sec < BREAK_WINDOW_END_SECONDS:
            window_states.append(state)
        else:
            outside_window_states.append(state)

    if not window_states:
        print("  -> ⚠️ 警告：在 60-120 分鐘窗口內無任何教學活動，無法尋找下課時間。")
        return state_timeline

    # 2. 在窗口內尋找最長的、僅由「閒聊」或「下課休息」組成的連續片段組
    best_group = []
    current_group = []
    for state in window_states:
        # ★★★【核心修正點 2】★★★
        # 使用更嚴格的 BREAK_CORE_STATES 進行判斷
        if state['classroom_state'] in BREAK_CORE_STATES:
            current_group.append(state)
        else:
            # 當遇到非休息狀態（如教師講解、師生互動）時，就結算前一個 group
            if current_group:
                duration = current_group[-1]['end_seconds'] - current_group[0]['start_seconds']
                best_duration = best_group[-1]['end_seconds'] - best_group[0]['start_seconds'] if best_group else 0
                if duration > best_duration:
                    best_group = list(current_group)
            current_group.clear()
    
    # 處理循環結束後最後一個可能的 group
    if current_group:
        duration = current_group[-1]['end_seconds'] - current_group[0]['start_seconds']
        best_duration = best_group[-1]['end_seconds'] - best_group[0]['start_seconds'] if best_group else 0
        if duration > best_duration:
            best_group = list(current_group)

    # 3. 如果找到了足夠長的連續休息片段，則合併它們
    valid_break = None
    final_timeline = outside_window_states.copy()

    if best_group and (best_group[-1]['end_seconds'] - best_group[0]['start_seconds']) >= MIN_BREAK_DURATION_SECONDS:
        valid_break = {
            "start_time": best_group[0]['start_time'],
            "end_time": best_group[-1]['end_time'],
            "classroom_state": "下課休息",
            "state_summary": "中場休息時間。",
            "start_seconds": best_group[0]['start_seconds'],
            "end_seconds": best_group[-1]['end_seconds']
        }
        final_timeline.append(valid_break)
        print(f"  -> 成功合併窗口內的 {len(best_group)} 個非教學片段，強制生成下課休息時段：({valid_break['start_time']} - {valid_break['end_time']})")
        
        break_start_sec = valid_break['start_seconds']
        break_end_sec = valid_break['end_seconds']
        for state in window_states:
            if not (break_start_sec <= state['start_seconds'] < break_end_sec):
                final_timeline.append(state)
    else:
        final_timeline.extend(window_states)
        print("  -> ⚠️ 警告：在 60-120 分鐘窗口內未找到長度超過5分鐘的連續非教學片段。")

    return post_process_state_timeline(sorted(final_timeline, key=lambda x: x['start_seconds']))

def get_reclassification_prompt():
    """生成一個用於二次分類、高度聚焦的 Prompt"""
    return """
你是一位嚴謹的教育內容審核員。你的唯一任務是判斷一段30秒到數分鐘不等的教師話語，其核心主題是否【直接】與【高中英文課程】的教學目標相關。

【判斷標準】
- 如果內容是關於：英文文法、單字解析、閱讀理解技巧、解題策略、課堂管理指令 -> 回答 `教師講解` 或 `師生互動`。
- 如果內容是關於：老師的個人經歷（軍旅、旅遊、薪資）、與課程無關的歷史故事、動漫、影視評論、時事閒聊 -> 回答 `閒聊`。

【輸出要求】
你的回答【只能是】`教師講解`、`師生互動` 或 `閒聊` 這三個選項之一，絕對不能包含任何其他文字或解釋。
"""

async def refine_chat_vs_lecture_states(state_timeline: list, semaphore: asyncio.Semaphore) -> list:
    """
    (v1.0 新增) 使用輕量級的二次 AI 請求，審核並校正被誤標為「教師講解」的「閒聊」片段。
    """
    print("  -> 正在執行【AI二次審核】以校正「教師講解」與「閒聊」的分類...")
    
    tasks = []
    
    async def reclassify_segment(state):
        async with semaphore:
            # 只審核長度超過60秒的「教師講解」，短片段通常不容易出錯且節省API成本
            if state['classroom_state'] == '教師講解' and (state['end_seconds'] - state['start_seconds']) > 60:
                
                # 提取這段時間內的原始逐字稿內容 (這需要一個輔助邏輯)
                # 為了簡化，我們假設 `state` 物件中已經包含了對應的文本。
                # 如果沒有，您需要從 master_timeline 中提取或重新構建。
                # 這裡我們假設 state_summary 儲存了足夠的上下文
                speech_context = state.get('state_summary', '') 
                
                try:
                    response = await async_client.chat.completions.create(
                        model=TEXT_DEPLOYMENT_NAME,
                        messages=[
                            {"role": "system", "content": get_reclassification_prompt()},
                            {"role": "user", "content": f"請判斷以下內容的核心主題：\n\n---\n{speech_context}\n---"}
                        ],
                        max_tokens=20,
                        temperature=0.0
                    )
                    corrected_mode = response.choices[0].message.content.strip()

                    if corrected_mode in ["閒聊", "師生互動"] and corrected_mode != state['classroom_state']:
                        print(f"    - 校正: 片段 {state['start_time']} 被從 '{state['classroom_state']}' 修改為 '{corrected_mode}'")
                        state['classroom_state'] = corrected_mode
                except Exception as e:
                    print(f"    - ⚠️ 二次審核失敗於片段 {state['start_time']}: {e}")
        return state

    for state in state_timeline:
        tasks.append(reclassify_segment(state))
    
    refined_timeline = await asyncio.gather(*tasks)
    
    # 再次合併可能產生的連續相同狀態
    return post_process_state_timeline(refined_timeline)

def clean_and_prepare_raw_transcript_for_states(filepath):
    """(v8.2 格式適應版) 清理新格式逐字稿，移除雜訊並標記長時間靜默，以供宏觀分析使用。"""
    print("  -> 正在執行逐字稿預清理以進行宏觀分析...")
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = f.read().strip().split('\n')
    except FileNotFoundError:
        return ""

    cleaned_lines = []
    time_format = '%H:%M:%S'
    last_end_time_obj = None

    for line in lines:
        # --- 核心修改：使用與 load_teacher_transcript 相同的解析邏輯 ---
        match = re.match(r'(\d{1,2}:\d{2}:\d{2}):\s*(.+)', line)
        if not match:
            continue
        
        start_time_str, text = match.groups()
        
        try:
            # 由於我們不知道結束時間，這裡的邏輯是基於開始時間的間隔
            start_time_obj = datetime.datetime.strptime(start_time_str, time_format)
            
            if last_end_time_obj:
                silence_duration = (start_time_obj - last_end_time_obj).total_seconds()
                if silence_duration > 60:
                    silence_minutes = round(silence_duration / 60)
                    marker_start = last_end_time_obj.strftime(time_format)
                    marker_end = start_time_obj.strftime(time_format)
                    cleaned_lines.append(f"{marker_start} - {marker_end}: [--- 長時間靜默 ({silence_minutes} 分鐘) ---]")
            
            # 為了讓 AI 能理解，我們模擬一個簡短的結束時間
            # 注意：這裡的結束時間僅用於宏觀分析的 text block，不會影響後續的精準計算
            simulated_end_time_obj = start_time_obj + datetime.timedelta(seconds=5)
            
            cleaned_lines.append(f"{start_time_obj.strftime(time_format)} - {simulated_end_time_obj.strftime(time_format)}: {text}")
            last_end_time_obj = start_time_obj # 下次比較的基準是當前的開始時間
        except ValueError:
            continue

    print("  -> 宏觀分析預清理完成。")
    return "\n".join(cleaned_lines)

def post_process_state_timeline(timeline):
    """後處理時間軸，合併零碎片段，並根據情境規則強制修正狀態。"""
    if not timeline: return []
    print("  -> 正在執行AI宏觀分析結果後處理...")
    
    # 這裡可以保留或簡化您舊程式碼中的後處理邏輯，因為新的Prompt已經很強大。
    # 核心是合併連續的相同狀態。
    merged_timeline = [timeline[0]]
    for i in range(1, len(timeline)):
        last_entry = merged_timeline[-1]
        current_entry = timeline[i]
        
        if last_entry['classroom_state'] == current_entry['classroom_state']:
            last_entry['end_time'] = current_entry['end_time']
        else:
            merged_timeline.append(current_entry)
            
    print("  -> 宏觀分析後處理完成。")
    return merged_timeline

async def generate_state_timeline_async(raw_transcript_path):
    """
    非同步函式，執行宏觀分析，生成權威的教學模式時間軸。
    """
    prepared_transcript = clean_and_prepare_raw_transcript_for_states(raw_transcript_path)
    if not prepared_transcript:
        print("❌ 錯誤：無法讀取或準備用於宏觀分析的逐字稿。")
        return None

    system_prompt = get_state_timeline_prompt()
    user_prompt = f"**【完整課堂逐字稿】**\n\n{prepared_transcript}"

    print(f"  -> 正在向 AI ({TEXT_DEPLOYMENT_NAME}) 發送宏觀分析請求...")
    try:
        response = await async_client.chat.completions.create(
            model=TEXT_DEPLOYMENT_NAME,
            response_format={"type": "json_object"},
            messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
            max_tokens=8192, # 給予足夠的空間
            temperature=0.0
        )
        
        raw_content = response.choices[0].message.content
        parsed_data = json.loads(raw_content)
        
        timeline = parsed_data.get("timeline", [])
        processed_timeline = post_process_state_timeline(timeline)
        
        # 將時間戳轉換為秒數以便後續查詢
        for entry in processed_timeline:
            entry['start_seconds'] = parse_timestamp(entry['start_time'])
            entry['end_seconds'] = parse_timestamp(entry['end_time'])

        return processed_timeline

    except Exception as e:
        print(f"❌ 宏觀教學模式分析失敗: {e}")
        return None

def calculate_block_level_wpm(merged_timeline: list, detailed_timeline: list) -> list:
    """
    (v11.0 升級版 - 動態校準與邊界設定) 
    在合併後的時間軸上，為每一個教學模式區塊計算一個更精準的 WPM。
    - 新增1: 根據區塊內實際有語音的片段比例，動態計算說話時間。
    - 新增2: 增加語速的上下限，防止出現不合理的極端值。
    """
    print("步驟 5.95/8: 執行區塊級語速計算 (採用【動態校準與邊界設定模式】)...")

    # --- ★★★【修改點 1：邊界設定】★★★ ---
    MAX_WPM = 350  # 設定人類說話的合理語速上限，防止因計算導致的數值爆炸
    MIN_WPM_FOR_VALID_SPEECH = 20 # 有效持續說話的最低語速，低於此值可能只是零碎指令

    # 中文「字」到英文「詞」的轉換率保持不變
    CHINESE_CHAR_TO_WORD_RATIO = 1.7

    for block in merged_timeline:
        total_chars = len(block.get("full_transcript", "(無語音)"))
        duration_seconds = block.get("duration_seconds", 0)

        # 如果區塊本身就沒有語音內容或時長，直接設為 0
        if total_chars == 0 or not duration_seconds:
            block["calibrated_wpm"] = 0
            continue
        
        # --- ★★★【修改點 2：動態計算說話佔比】★★★ ---
        start_seconds = parse_timestamp(block["start_time"])
        end_seconds = parse_timestamp(block["end_time"])

        # 1. 從 detailed_timeline 中篩選出此區塊包含的所有 30 秒微觀片段
        intervals_in_block = [
            interval for interval in detailed_timeline 
            if start_seconds is not None and end_seconds is not None and start_seconds <= interval["seconds"] < end_seconds
        ]
        
        if not intervals_in_block:
            block["calibrated_wpm"] = 0
            continue

        # 2. 計算其中真正有老師說話的片段佔了多少比例
        speech_intervals_count = sum(1 for interval in intervals_in_block if interval.get("teacher_speech", "").strip())
        total_intervals_count = len(intervals_in_block)

        # 3. 得到此區塊「真實的」說話時間比例
        dynamic_speech_ratio = (speech_intervals_count / total_intervals_count) if total_intervals_count > 0 else 0
        
        # 為了避免除以零或極小數，設定一個保底的最小說話比例
        if dynamic_speech_ratio < 0.05:
            dynamic_speech_ratio = 0.05

        # 4. 使用這個動態比例來計算估算的有效說話時長
        estimated_speech_duration = duration_seconds * dynamic_speech_ratio

        # --- ★★★【修改點 3：邊界應用與特殊模式處理】★★★ ---

        # 處理特殊模式：如果一個區塊被判定為練習/考試/休息，且總字數很少，
        # 這代表它可能只是一個開頭的指令，其語速沒有參考價值，直接設為 0。
        if block["teaching_mode"] in ["學生練習", "學生考試", "下課休息"] and total_chars < 20:
            block["calibrated_wpm"] = 0
            continue

        # 計算 WPM
        cpm = (total_chars / estimated_speech_duration) * 60
        wpm = cpm / CHINESE_CHAR_TO_WORD_RATIO
        calibrated_wpm = round(wpm, 1)

        # 應用語速上下限，避免出現 729.4 這種極端值
        if calibrated_wpm > MAX_WPM:
            calibrated_wpm = MAX_WPM
        
        # (可選) 如果語速低於閾值但又不為0，可以將其視為零碎指令，或統一到一個最低值
        if 0 < calibrated_wpm < MIN_WPM_FOR_VALID_SPEECH:
            calibrated_wpm = MIN_WPM_FOR_VALID_SPEECH

        block["calibrated_wpm"] = calibrated_wpm

    print("✅ 區塊級語速計算（動態校準版）完成。")
    return merged_timeline

def analyze_speech_rate_trend(merged_timeline_with_wpm: list):
    """
    (v10.0 升級版) 使用【區塊級校準後語速】進行線性迴歸分析，以獲得更穩定、更具代表性的趨勢。
    """
    print("步驟 5.9/8: 分析語速變化趨勢 (基於區塊級校準後語速)...")

    # 1. 提取所有包含【校準後語速】數據的時間點和語速值
    time_points = []
    wpm_points = []
    for block in merged_timeline_with_wpm:
        wpm = block.get("calibrated_wpm")
        if wpm is not None and wpm > 0:
            # 使用每個區塊的中間點作為代表時間點
            start_seconds = parse_timestamp(block["start_time"])
            end_seconds = parse_timestamp(block["end_time"])
            if start_seconds is not None and end_seconds is not None:
                midpoint_seconds = start_seconds + (end_seconds - start_seconds) / 2
                time_points.append(midpoint_seconds)
                wpm_points.append(wpm)
    
    # ... (後續的所有計算邏輯完全保持不變) ...
    if len(time_points) < 5: # 區塊較少，可以降低數據點門檻
        print("  - ⚠️ 語速數據點過少，跳過趨勢分析。")
        return {
            "error": "Not enough data points to perform trend analysis.",
            "trend_description": "數據不足",
            "linear_regression_stats": None,
            "comparative_summary": None
        }

    x = np.array(time_points)
    y = np.array(wpm_points)
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

    trend_description = ""
    if p_value < 0.05:
        if slope < -0.01:
            trend_description = "呈現顯著的下降趨勢 (老師的語速隨時間推移而顯著變慢)。"
        elif slope > 0.01:
            trend_description = "呈現顯著的上升趨勢 (老師的語速隨時間推移而顯著變快)。"
        else:
            trend_description = "語速非常穩定，未觀察到顯著的變化趨勢。"
    else:
        trend_description = "語速整體波動，未觀察到統計上顯著的長期增減趨勢。"

    midpoint_time = time_points[-1] / 2 if time_points else 0
    first_half_wpm = [wpm for t, wpm in zip(time_points, wpm_points) if t <= midpoint_time]
    second_half_wpm = [wpm for t, wpm in zip(time_points, wpm_points) if t > midpoint_time]
    
    avg_first_half = np.mean(first_half_wpm) if first_half_wpm else 0
    avg_second_half = np.mean(second_half_wpm) if second_half_wpm else 0

    analysis_result = {
        "trend_description": trend_description,
        "linear_regression_stats": {
            "slope_per_second": slope,
            "slope_per_minute": slope * 60,
            "p_value": p_value,
            "r_squared": r_value**2
        },
        "comparative_summary": {
            "average_wpm_first_half": round(avg_first_half, 1),
            "average_wpm_second_half": round(avg_second_half, 1),
            "change_percentage": round(((avg_second_half - avg_first_half) / avg_first_half) * 100, 1) if avg_first_half > 0 else 0
        }
    }
    
    print(f"✅ 語速趨勢分析完成: {trend_description}")
    return analysis_result

# ==============================================================================
# --- ★★★【新增區塊：逐字稿前處理過濾器】★★★ ---
# ==============================================================================
def preprocess_transcript(transcript_data: list, wps_threshold=0.8, merge_interval_seconds=1.5) -> list:
    """
    (v6.0 新增) 在 AI 校正前對逐字稿進行預處理。
    1. 過濾掉低語音密度的片段 (可能是靜音或噪音)。
    2. 合併時間上相鄰且語意上可能連續的破碎語句。
    """
    print("步驟 3.2/8: 執行逐字稿前處理過濾器...")
    
    # --- 階段 1: 語音密度過濾器 (Word-Per-Second Filter) ---
    filtered_data = []
    for entry in transcript_data:
        duration = entry['end_seconds'] - entry['start_seconds']
        if duration <= 0:
            continue
        
        # 計算每秒字數 (Words Per Second)
        wps = len(entry['text']) / duration
        
        # 如果 wps 低於閾值，則認為是無效語音，直接跳過
        if wps < wps_threshold:
            continue
            
        filtered_data.append(entry)

    if not filtered_data:
        print("  - ⚠️ 過濾後無有效逐字稿數據。")
        return []

    # --- 階段 2: 碎片化語句合併 (Fragment Merging) ---
    merged_data = []
    i = 0
    while i < len(filtered_data):
        current_entry = filtered_data[i]
        
        # 向後查看，看有多少個片段可以合併
        j = i + 1
        while j < len(filtered_data):
            next_entry = filtered_data[j]
            gap = next_entry['start_seconds'] - current_entry['end_seconds']
            
            # 如果時間間隔小於閾值，就合併它們
            if gap < merge_interval_seconds:
                current_entry['text'] += " " + next_entry['text']
                current_entry['end_seconds'] = next_entry['end_seconds']
                j += 1
            else:
                break
        
        merged_data.append(current_entry)
        i = j

    print(f"✅ 逐字稿前處理完成：原始 {len(transcript_data)} 筆 -> 過濾後 {len(filtered_data)} 筆 -> 合併後 {len(merged_data)} 筆。")
    return merged_data

# ==============================================================================
# --- ★★★【v5.9 新增區塊：關鍵片段微觀分析模組 (全新升級版)】★★★ ---
# ==============================================================================

def get_critical_segment_analysis_prompt():
    """
    【全新升級版】
    生成用於對【單一關鍵高強度教學片段】進行微觀分析的 Prompt。
    - 整合了宏觀功能分類的概念。
    - 新增了量化下滑幅度的要求。
    """
    
    # --- ★ 修改點 1: 引入宏觀功能分類的概念 ---
    # 這裡我們將您另一份程式碼中的 "教學狀態" 概念，轉化為 AI 在分析時可以參考的 "教學活動" 分類
    teaching_activity_definitions = """
**【教學活動參考分類】**
在你的分析中，請隨時參考以下教學活動的分類來理解上下文：
- **教師講解**: 老師主導的、連續的知識輸出，包括新知識傳授或舊知識複習。
- **學生練習/考試**: 老師指令後，學生進行獨立的、靜默的練習或測驗。
- **師生互動**: 針對教學內容的雙向問答、討論，或與課堂管理相關的交流。
- **閒聊/認知緩衝**: 與當前教學主題無直接關聯的個人故事、生活經驗分享或非正式交流。
"""

    # --- ★ 修改點 2: 全面升級 JSON 結構，增加 quantitative_impact ---
    json_structure = """
    {{
      "segment_start_time": "（此片段的開始時間）",
      "segment_end_time": "（此片段的結束時間）",
      "core_teaching_topic": "（總結這 7-12 分鐘內最核心的單一教學主題，例如：'現在完成式的用法解析'）",
      "attention_narrative": "（以敘事方式，詳細描述學生專注度在此片段中的變化軌跡。例如：'片段開始時，學生專注度高...約在 4 分 30 秒後...分心比例達到峰值...'）",
      "breaking_point_analysis": {{
        "time_of_decline": "（找出專注度開始【首次顯著下滑】的具體時間點，例如：'0:28:30'）",
        "teacher_quote_at_breaking_point": "（引用導致專注度下滑的【那一句關鍵話語】）",
        "analysis_of_cause": "（分析下滑的原因，例如：'教師在此處連續講解超過 5 分鐘未進行互動，且引入了一個複雜的語法例外規則，超出了部分學生的工作記憶負荷。'）",
        
        "quantitative_impact": {{
          "focus_metric_analyzed": "（你主要分析的是哪個指標的變化，必須是 'disengagement' 或 'task_oriented_focus'）",
          "value_before_decline_pct": （下滑前的指標數值，僅回傳數字，例如：35.5）,
          "value_after_decline_pct": （下滑後的指標數值，僅回傳數字，例如：55.0）,
          "change_delta_pct": （計算出的變化幅度，僅回傳數字，例如：19.5 或 -20.0）
        }}
      }},
      "segment_summary": "（對這個關鍵片段的教學成效做一個總結性評價。）"
    }}
    """
    
    # --- ★ 修改點 3: 強化 Prompt 指令，使其更明確、更專業 ---
    return f"""
你是一位頂尖的教育心理學家和數據分析師，專長是從課堂數據中進行微觀的因果分析。你的任務是分析一段被標記為【關鍵高強度教學】的課堂數據（長度約 7-12 分鐘）。

{teaching_activity_definitions}

**【你的核心任務】**
1.  **敘述專注度軌跡**: 描述學生專注度從高到低的完整變化過程。
2.  **定位「引爆點」(Breaking Point)**: 精確找出學生專注度開始**首次顯著下滑**的時間點、對應的教師話語，並**量化其影響**。
3.  **分析原因**: 結合認知負荷理論 (Cognitive Load Theory)，解釋為什麼在那個時間點學生的專注度會開始下滑。

**【★★★ V2.0 新增指令：量化引爆點的衝擊 ★★★】**
在 `breaking_point_analysis` 區塊中，你**必須**新增並填寫 `quantitative_impact` 子物件。
- 你需要從輸入數據的 `focus_distribution` 中，找出引爆點時間前後最能反映變化的指標（通常是 `disengagement` 的上升或 `task_oriented_focus` 的下降）。
- 你必須提供變化前的數值、變化後的數值，以及它們之間的差值（delta），所有數值都只要數字本身。

**你的輸入資料格式如下：**
一個JSON列表，代表一段連續的「高強度講解」片段。

**你的輸出格式要求：**
你必須嚴格遵循以下的JSON結構，並填寫所有欄位。你的回答必須是一個結構完整的 JSON 物件，絕對不能包含任何額外的文字、註解或 Markdown 標記。

{json_structure}
"""

async def analyze_critical_segment_with_ai_async(segment, index, total, semaphore, retries=3):
    """(v5.9) 調用 AI 對單個關鍵教學片段進行微觀分析"""
    if not segment: return None
    
    # 使用 async with semaphore 來自動管理並行數量
    async with semaphore:
        print(f"  - (許可已取得) 正在發起關鍵片段 {index + 1}/{total} 的微觀分析請求...")
        system_prompt = get_critical_segment_analysis_prompt()
        user_prompt = f"請對以下這段關鍵高強度教學片段進行微觀分析：\n\n{json.dumps(segment, ensure_ascii=False, indent=2)}"
        
        for attempt in range(retries):
            try:
                # 在請求之間加入一個微小的隨機延遲，進一步錯開請求峰值
                await asyncio.sleep(random.uniform(0.5, 1.5))
                
                response = await async_client.chat.completions.create(
                    model=TEXT_DEPLOYMENT_NAME,
                    response_format={"type": "json_object"},
                    messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                    max_tokens=4096, temperature=0.1,
                )
                return json.loads(response.choices[0].message.content)
            except Exception as e:
                print(f"  - ⚠️ 分析關鍵片段 {index + 1} 時發生錯誤 (嘗試 {attempt + 1}/{retries}): {e}")
                if attempt < retries - 1: await asyncio.sleep(5 * (attempt + 1)) # 增加重試等待
        
        print(f"  - ❌ 關鍵片段 {index + 1}/{total} 在所有重試後分析失敗。")
        return None

# ==============================================================================
# --- ★★★【新增區塊：AI 輔助的教學模式分類器】★★★ ---
# ==============================================================================
async def classify_teaching_mode_with_ai_async(speech_text, wpm, keyword_count, retries=3):
    """
    (v7.1 規則增強版) 使用 AI 根據上下文語義和詳細的判斷規則來判斷教學模式。
    """
    if not speech_text or wpm < 5:
        return "標準互動"

    # --- ★★★【核心修改 1：動態生成規則列表】★★★ ---
    # 這段程式碼會自動將您定義的字典轉換為 AI 容易閱讀的列表格式
    rules_description = "**【課堂狀態的六個類別與判斷規則】**\n"
    for state, description in CLASSROOM_STATES_DEFINITIONS.items():
        rules_description += f"**{state}**: {description.strip()}\n"

    # --- ★★★【核心修改 2：全面升級 System Prompt】★★★ ---
    system_prompt = f"""
你是一位頂尖的教育分析師，專門從簡短的課堂對話中精準識別出教學活動的階段。

**【你的任務】**
根據使用者提供的【30秒逐字稿片段】及其【量化指標】，嚴格遵循下方的判斷規則，從六個類別中選擇一個最符合的 `classroom_state` 作為你的唯一回答。

{rules_description}

**【輸出要求】**
你的回答**只能是**上述六個狀態名稱中的一個（例如，只回答 `教師講解` 或 `閒聊`），絕對不能包含任何額外的文字、解釋或 JSON 標記。
"""
    
    user_prompt = f"""
分析以下課堂片段：

- **逐字稿內容**: "{speech_text}"
- **量化指標**: 語速約 {int(wpm)} WPM, 專業關鍵詞數量 {keyword_count} 個。

最符合的課堂狀態是？
"""

    for attempt in range(retries):
        try:
            response = await async_client.chat.completions.create(
                model=TEXT_DEPLOYMENT_NAME,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                max_tokens=20,
                temperature=0.0
            )
            mode = response.choices[0].message.content.strip().replace('"', '')
            
            if mode in CLASSROOM_STATES_DEFINITIONS.keys():
                return mode
            else:
                print(f"  - ⚠️ AI 分類器返回無效狀態 '{mode}'，將使用 '標準互動' 作為備用。")
                return "標準互動"
                
        except Exception as e:
            print(f"  - ⚠️ AI 模式分類失敗 (嘗試 {attempt + 1}/{retries}): {e}")
            if attempt < retries - 1:
                await asyncio.sleep(2 ** attempt)
    
    print(f"  - ❌ AI 模式分類在所有重試後失敗，將使用 '標準互動' 作為備用。")
    return "標準互動"

async def classify_interval_teaching_mode_async(speech_text: str, student_behavior_summary: dict, blackboard_snapshots: list, retries=3):
    """
    (全新微觀策略) 使用 AI 根據單一時間區間內的綜合資訊，判斷最精確的教學模式。
    """
    # 如果老師完全沒說話，根據是否有板書和學生行為來初步判斷
    if not speech_text.strip():
        # 如果學生行為主要是做筆記或翻書，且有板書，傾向於練習
        if any(b in student_behavior_summary for b in ["做筆記", "翻書"]) and blackboard_snapshots:
            return "學生練習"
        # 如果大量學生處於趴睡或低頭(非學習)，且長時間靜默，可能是考試或練習
        if student_behavior_summary.get("趴睡", []) or student_behavior_summary.get("低頭(非學習)", []):
             return "學生練習" # 或 "學生考試"，這裡需要更複雜的規則，暫定為練習
        return "教學靜默" # 如果沒有足夠線索，定義一個新狀態或歸類為閒聊

    # --- 動態生成規則，與舊函式相同 ---
    rules_description = "**【課堂狀態的六個類別與判斷規則】**\n"
    for state, description in CLASSROOM_STATES_DEFINITIONS.items():
        rules_description += f"**{state}**: {description.strip()}\n"

    # --- 全新設計的 System Prompt，專為微觀分析優化 ---
    system_prompt = f"""
你是一位頂尖的課堂分析師，專門從【極短的】課堂片段中，精準識別教學活動。

**【你的任務】**
根據使用者提供的【單一 30 秒課堂片段】的綜合資訊（包含逐字稿、學生行為），嚴格遵循下方的判斷規則，從六個類別中選擇一個最符合的 `classroom_state` 作為你的唯一回答。

{rules_description}

**【分析提示】**
- **高度關注老師的「指令」**：像「練習一下」、「我們來對答案」這類話語是分類的黃金信號。
- **結合學生行為**：如果老師在講解複雜概念，但學生數據顯示多數人在「做筆記」，這強化了「教師講解」的判斷。如果老師話語零碎，學生多為「目視同學」，則「閒聊」的可能性很高。
- **不要過度推論**：你只分析這 30 秒，不要去猜測之前或之後發生了什麼。

**【輸出要求】**
你的回答**只能是**上述六個狀態名稱中的一個（例如，只回答 `教師講解` 或 `閒聊`），絕對不能包含任何額外的文字、解釋或 JSON 標記。
"""
    
    user_prompt = f"""
分析以下 30 秒課堂片段的教學模式：

- **老師逐字稿**: "{speech_text}"
- **此期間的學生主要行為**: {list(student_behavior_summary.keys())}
- **此期間是否有新板書**: {"有" if blackboard_snapshots else "無"}

最符合的課堂狀態是？
"""

    for attempt in range(retries):
        try:
            response = await async_client.chat.completions.create(
                model=TEXT_DEPLOYMENT_NAME,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                max_tokens=20,
                temperature=0.0
            )
            mode = response.choices[0].message.content.strip().replace('"', '')
            
            if mode in CLASSROOM_STATES_DEFINITIONS.keys():
                return mode
            else:
                # 如果 AI 回傳無效答案，根據是否有老師說話做一個基礎備用判斷
                return "師生互動" if speech_text else "教學靜默"
                
        except Exception as e:
            print(f"  - ⚠️ 微觀模式分類失敗 (嘗試 {attempt + 1}/{retries}): {e}")
            if attempt < retries - 1:
                await asyncio.sleep(2 ** attempt)
    
    # 如果所有重試都失敗，提供一個最終的備用邏輯
    return "師生互動" if speech_text else "教學靜默"

def identify_teaching_cycles(master_timeline):
    """
    (v5.12 修正版) 識別廣義的教學週期 (高認知負荷 -> 低認知負荷)。
    """
    print("步驟 5.8/8: 識別並量化教學週期 (採用廣義模式)...")
    if not master_timeline:
        return {"cycles": [], "summary": {}}

    cycles = []
    current_cycle = None
    in_high_intensity_phase = False

    # 定義高低負荷的狀態集合
    HIGH_LOAD_MODES = {"教師講解", "師生互動"}
    LOW_LOAD_MODES = {"學生練習", "學生考試", "閒聊", "下課休息"}

    for i, interval in enumerate(master_timeline):
        mode = interval.get("teaching_mode")
        metrics = interval.get("cognitive_load_metrics")
        
        # 判斷當前區間是否為高負荷
        is_high_load = False
        if mode in HIGH_LOAD_MODES and metrics:
            intensity = metrics.get("intensity_level")
            if intensity in ["高強度", "中強度"]:
                is_high_load = True

        # 從非高負荷進入高負荷，標記一個新週期的開始
        if is_high_load and not in_high_intensity_phase:
            in_high_intensity_phase = True
            if current_cycle: # 結束上一個週期
                end_sec = parse_timestamp(master_timeline[i-1]["end_time"])
                if current_cycle.get("start_seconds") is not None and end_sec is not None:
                    current_cycle["end_time"] = master_timeline[i-1]["end_time"]
                    current_cycle["total_duration_seconds"] = end_sec - current_cycle["start_seconds"]
                    cycles.append(current_cycle)

            # 開始新週期
            current_cycle = {
                "cycle_index": len(cycles) + 1,
                "start_time": interval["start_time"],
                "start_seconds": interval["seconds"],
                "end_time": None,
                "high_intensity_duration": 0,
                "low_intensity_duration": 0
            }
        
        if current_cycle:
            if is_high_load:
                current_cycle["high_intensity_duration"] += TIME_INTERVAL_SECONDS
            elif mode in LOW_LOAD_MODES:
                current_cycle["low_intensity_duration"] += TIME_INTERVAL_SECONDS
                # 如果從高負荷轉為低負荷，標記高負荷階段結束
                if in_high_intensity_phase:
                    in_high_intensity_phase = False
    
    # 處理最後一個週期
    if current_cycle:
        end_sec = master_timeline[-1]['seconds'] + TIME_INTERVAL_SECONDS
        if current_cycle.get("start_seconds") is not None:
            current_cycle["end_time"] = str(datetime.timedelta(seconds=end_sec))
            current_cycle["total_duration_seconds"] = end_sec - current_cycle["start_seconds"]
            cycles.append(current_cycle)

    # ... (後續的摘要計算不變) ...
    total_cycle_duration = sum(c.get("total_duration_seconds", 0) for c in cycles)
    num_cycles = len(cycles)
    average_cycle_duration = total_cycle_duration / num_cycles if num_cycles > 0 else 0

    summary = {
        "number_of_cycles": num_cycles,
        "average_cycle_duration_seconds": round(average_cycle_duration),
        "average_cycle_duration_minutes": round(average_cycle_duration / 60, 1)
    }
    
    print(f"✅ 成功識別出 {num_cycles} 個教學週期，平均週期時長約 {summary['average_cycle_duration_minutes']} 分鐘。")
    return {"cycles": cycles, "summary": summary}

def find_state_transition_triggers(master_timeline: list) -> list:
    """
    (全新功能) 遍歷完整時間軸，找出教學模式轉變的關鍵時間點，並提取觸發該轉變的關鍵句。
    """
    print("步驟 5.8/8: 分析教學模式轉換的觸發關鍵句...")
    if not master_timeline or len(master_timeline) < 2:
        return []

    triggers = []
    # 從第二個時間區間開始遍歷，以便和前一個進行比較
    for i in range(1, len(master_timeline)):
        prev_interval = master_timeline[i-1]
        current_interval = master_timeline[i]

        prev_mode = prev_interval.get("teaching_mode")
        current_mode = current_interval.get("teaching_mode")

        # 當教學模式發生變化時，記錄下來
        if prev_mode != current_mode:
            
            # 關鍵句是前一個區間老師說的話，因為是這句話導致了下一個區間的模式轉變
            trigger_quote = prev_interval.get("teacher_speech", "").strip()
            
            # 為了提供更豐富的上下文，我們往前多找一句
            context_before = master_timeline[i-2].get("teacher_speech", "").strip() if i > 1 else ""

            # 只記錄那些真正由老師話語觸發的轉變
            if trigger_quote:
                triggers.append({
                    "transition_time": current_interval["start_time"],
                    "from_mode": prev_mode,
                    "to_mode": current_mode,
                    "trigger_quote": trigger_quote,
                    "context_before": context_before
                })

    print(f"✅ 成功識別出 {len(triggers)} 個教學模式轉換觸發點。")
    return triggers

def extract_trigger_keywords_for_states(merged_timeline: list) -> list:
    """
    (全新功能 v2.0) 在合併後的時間軸上，為特定的教學模式找出其內部的觸發關鍵字/句。
    """
    print("  -> 正在提取各教學模式的內部觸發關鍵句...")
    
    # 定義我們關心的模式及其對應的觸發關鍵詞
    TARGET_MODES_KEYWORDS = {
        # --- 核心教學模式 ---
        "教師講解": [
            # 引導與聚焦
            "來看這邊", "專心看", "注意聽", "首先", "再來我們看", 
            "這個地方是", "這個觀念", "解釋一下",
            # 強調重要性
            "重點是", "關鍵在", "一定要記住",
            # 結構性說明
            "句型結構", "分詞用法", "名詞子句"
        ],
        "師生互動": [
            # 開放性提問
            "有沒有問題", "為什麼", "什麼意思", "懂我意思嗎", "懂了嗎",
            "有沒有發現", "有沒有感覺",
            # 邀請參與
            "有沒有人知道", "大家覺得", "誰可以告訴我",
            # 檢查與確認
            "對不對", "是不是", "可以嗎"
        ],

        # --- 任務導向模式 ---
        "學生考試": [
            "考試", "前測", "測驗", "發考卷", "檢討考卷"
        ],
        "學生練習": [
            # 明確指令
            "練習一下", "寫一下", "試試看", "動筆", "給大家幾分鐘", 
            "寫上去", "畫線", "圈起來", "麻煩您",
            # 引導操作
            "給大家思考一下", "先幫我看到"
        ],

        # --- 非教學核心模式 ---
        "下課休息": [
            "下課", "休息一下", "休息"
        ],
        "閒聊": [
            # 故事性開頭
            "我跟你講", "我當年", "我個人", "講個故事", "分享一下",
            # 轉移話題
            "對了", "話說回來"
            # (閒聊模式更多依賴語意，關鍵字較難窮舉，以上為常見信號詞)
        ]
    }
    
    analysis_results = []

    for block in merged_timeline:
        mode = block["teaching_mode"]
        if mode in TARGET_MODES_KEYWORDS:
            transcript = block["full_transcript"]
            keywords_found = []
            
            for keyword in TARGET_MODES_KEYWORDS[mode]:
                if keyword in transcript:
                    # 尋找包含關鍵字的完整句子
                    # 使用正則表達式，匹配包含關鍵字且以句號、問號或空格結尾的句子
                    match = re.search(f"([^。？\s]*{re.escape(keyword)}[^。？\s]*)", transcript)
                    if match and match.group(1) not in keywords_found:
                        keywords_found.append(match.group(1).strip())
            
            if keywords_found:
                analysis_results.append({
                    "mode": mode,
                    "start_time": block["start_time"],
                    "end_time": block["end_time"],
                    "trigger_quotes": keywords_found
                })

    print(f"✅ 成功為 {len(analysis_results)} 個關鍵模式區塊提取了觸發句。")
    return analysis_results

async def correct_single_transcript_chunk(chunk, index, total, context_before, context_after, professional_keywords, contextual_glossary, retries=3):
    """(v7.0 語意增強版) 處理單個逐字稿區塊的校正，並引入上下文知識庫。"""
    
    global PROFESSIONAL_KEYWORDS
    global CONTEXTUAL_GLOSSARY

    professional_terms = ", ".join(PROFESSIONAL_KEYWORDS)
    
    # --- ★★★【核心修改 1：建構知識庫提示】★★★ ---
    glossary_prompt_part = """
**【上下文與專有名詞知識庫】**
在校正時，你必須優先參考以下詞彙對照表來修正常見的同音異義詞錯誤和專有名詞：
- 如果原始文本出現右側的詞 (常見錯誤)，且語意符合左側情境，請大膽將其校正為左側的詞 (正確詞彙)。
"""
    for correct_term, wrong_terms in CONTEXTUAL_GLOSSARY.items():
        if wrong_terms:
            glossary_prompt_part += f"- **{correct_term}** (正確) <--- {', '.join(wrong_terms)} (常見錯誤)\n"
        else:
            glossary_prompt_part += f"- **{correct_term}** (這是一個特殊的專有名詞或黑話，請保留它，不要修改)\n"

    # --- ★★★【核心修改 2：全面升級 System Prompt】★★★ ---
    system_prompt = f"""
你是一位頂尖的中文逐字稿校對專家，專門處理【台灣補習班老師】的上課內容。你不僅懂文法，更懂教育情境和學生的口語文化。

【情境資訊】
這是一堂快節奏的高中英文文法與歷史課。老師語速快，常有口語化的表達、俚語和與學生的玩笑。原始稿件由 ASR 模型生成，可能包含大量錯誤。

{glossary_prompt_part}

**【核心校正規則】**
1.  **語意連貫性優先**: 你的首要任務是將【待校正區塊】的內容修復成**通順且符合邏輯**的中文句子。請務必參考【前文】和【後文】來理解語意，並將破碎的短句合併成完整的語意單元。
2.  **同音異義詞校正**: 這是你的關鍵任務。例如，如果老師在討論學科，但文本出現「初學」，你必須將其修正為「數學」。
3.  **處理無意義的幻覺**: 如果遇到像 "婆婆 check j sock Natalk" 這樣完全無法理解、由噪音產生的亂碼，請直接將其**替換為一個表示無法辨識的標籤**，例如 `"[無法辨識的噪音]"或"[多人同時說話]" `。**絕對不要**試圖去翻譯或解釋這些亂碼。
4.  **保持口語風格**: 只修正錯誤，**不要**將老師口語化的內容（例如 "對啊"、"然後呢"）過度書面化。保留老師的教學風格。
5.  **結構保持一致**: 你的輸出必須是與輸入的【待校正區塊】完全相同的 JSON 列表結構，你只能修改 "text" 欄位的文字。

【你的輸出要求】
請你只返回校正後的【待校正區塊】的 JSON 內容，不要包含任何額外的文字、註解或 Markdown 標記。
"""
    
    user_prompt = f"""
請根據你的專業知識和校正規則，修復以下標示為【待校正區塊】的逐字稿 JSON 資料片段。

---
【前文參考】
{json.dumps(context_before, ensure_ascii=False, indent=2)}
---
【★★★ 待校正區塊 ★★★】
{json.dumps(chunk, ensure_ascii=False, indent=2)}
---
【後文參考】
{json.dumps(context_after, ensure_ascii=False, indent=2)}
---
"""
    for attempt in range(retries):
        corrected_content = "" # 先初始化變數
        try:
            response = await async_client.chat.completions.create(
                model=TEXT_DEPLOYMENT_NAME,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                max_tokens=8192,  # <--- 增加此處的數值
                temperature=0.1
            )
            corrected_content = response.choices[0].message.content
            json_match = re.search(r'```json\s*([\s\S]*?)\s*```', corrected_content, re.DOTALL)
            if json_match:
                corrected_content = json_match.group(1)
            
            corrected_chunk = json.loads(corrected_content)
            print(f"    - ✅ 逐字稿區塊 {index + 1}/{total} 校正成功。")
            return corrected_chunk
            
        except RateLimitError as e:
            wait_time = (2 ** attempt) + random.random()
            print(f"    - ⚠️ 區塊 {index + 1} 觸發速率限制 (嘗試 {attempt + 1}/{retries})，將在 {wait_time:.1f} 秒後重試...")
            await asyncio.sleep(wait_time)
            
        except Exception as e:
            # ★★★【修改點 2：加入詳細的錯誤日誌】★★★
            # 當解析失敗時，印出完整的錯誤訊息和 AI 返回的原始文字，以便偵錯
            print(f"    - ⚠️ 校正區塊 {index + 1} 時發生錯誤 (嘗試 {attempt + 1}/{retries}): {e}")
            print(f"--- AI 返回的原始內容 (區塊 {index + 1}) ---")
            print(corrected_content)
            print("------------------------------------")
            # ★★★【修改結束】★★★

            if attempt < retries - 1:
                await asyncio.sleep(5)
            else:
                print(f"    - ❌ 校正區塊 {index + 1} 失敗，將使用此區塊的原始數據。")
                return chunk

async def correct_transcript_with_ai_async(transcript_data, chunk_size=30):
    """(v6.0 增強版) 使用並行請求校正逐字稿，並為每個區塊提供上下文窗口。"""
    
    # --- ★★★【核心修正點】★★★ ---
    # 在這裡宣告我們要使用全域變數，這樣在下面的 tasks.append 中才能找到它們
    global PROFESSIONAL_KEYWORDS
    global CONTEXTUAL_GLOSSARY
    # --- ★★★【修改結束】★★★ ---

    print("步驟 3.5/8: 調用 AI 進行逐字稿校正 (採用【上下文增強模式】)...")
    
    if not transcript_data:
        print("  - 逐字稿為空，跳過校正。")
        return []
        
    chunks = [transcript_data[i:i + chunk_size] for i in range(0, len(transcript_data), chunk_size)]
    print(f"  - 逐字稿已分為 {len(chunks)} 個區塊，將並行處理。")

    tasks = []
    for i, chunk in enumerate(chunks):
        context_before = chunks[i-1][-2:] if i > 0 else []
        context_after = chunks[i+1][:2] if i < len(chunks) - 1 else []
        
        # 現在這一行可以正常執行了
        tasks.append(correct_single_transcript_chunk(chunk, i, len(chunks), context_before, context_after, PROFESSIONAL_KEYWORDS, CONTEXTUAL_GLOSSARY))
    
    results = await asyncio.gather(*tasks)
    
    corrected_transcript = []
    for result in results:
        if result:
            corrected_transcript.extend(result)
            
    print("✅ AI 逐字稿校正完成。")
    return corrected_transcript

def get_chunk_analysis_prompt(chunk_start_time, chunk_end_time):
    """
    【v8.0 學術化最終版】生成用於分析【單一區塊】的 Prompt。
    - 採用 Fredricks et al. (2004) 和 Chi & Wylie (2014) 的理論框架。
    - 將學生狀態分為四個嚴謹的維度：主動行為參與、被動行為參與、行為分心、模糊行為。
    - 更新 JSON 輸出結構以匹配新的學術術語。
    """
    
    # --- 核心部分 1：學生行為參與狀態的學術解碼指南 ---
    behavior_decoder = """
**【學生行為參與狀態解碼指南 (v8.0)】**
你在分析 `student_behavior_summary` 時，必須嚴格遵循以下基於 Fredricks et al. (2004) 和 Chi & Wylie (2014) 理論的四維分類框架：

**1. 主動行為參與 (Active Behavioral Engagement):**
- **核心行為**: "做筆記", "翻書", "主動舉手"
- **學術解讀**: 學生正在主動地、可見地執行與當前學習任務直接相關的動作。這類行為是推斷**高度認知參與 (High Cognitive Engagement)** 的強力外部指標。

**2. 被動行為參與 (Passive Behavioral Engagement):**
- **核心行為**: "目視教師", "目視黑板", "坐姿直立"
- **學術解讀**: 學生遵守課堂規範並將注意力導向教學來源（老師或黑板），主要以接收資訊為主。這在聽講、觀看示範時是必要且有效的參與形式。它代表學生在任務上 (on-task)，但不涉及主動的物理輸出。

**3. 行為分心 (Behavioral Disengagement):**
- **核心行為**: "目視同學", "目視他處", "玩弄手部/文具", "低頭(非學習)", "趴睡", "飲食", "喝水"
- **學術解讀**: 學生明顯從事與當前學習任務無關的行為 (off-task)。這些是需要關注的負面指標。

**4. 程序性/模糊行為 (Procedural / Ambiguous Behavior):**
- **核心行為**: "托腮", "觸摸臉部", "觸摸頭髮", "被動舉手", "身體前傾", "身體後靠"
- **學術解讀**: 這些行為模棱兩可，單從影像無法直接斷定其背後的認知狀態（例如，「托腮」可能是深度思考，也可能是發呆）。在分析時應保持客觀，**除非有強烈的上下文支持，否則不要輕易將其歸因為專注或分心**。
"""

    # --- 核心部分 2：AI 必須嚴格遵循的 JSON 輸出結構 ---
    json_structure = f"""
{{
  "start_time": "{chunk_start_time}",
  "end_time": "{chunk_end_time}",
  "event_title": "（為這整個時間區塊總結一個最核心的教學活動主題，例如：'分詞句型與形容詞用法的高強度講解與練習'）",
  "event_description": "（客觀、簡潔地描述老師在此區塊的主要教學內容和採用的方法，例如：'本時段教師以高強度講解為主，穿插標準互動與師生互動，重點在於分詞、形容詞用法、時態判斷等語法概念。'）",
  "student_focus_analysis": {{
    "overall_level": "（基於此區塊內行為參與狀態的整體判斷：高 / 中高 / 中 / 低）",
    "average_distribution": {{
      "task_oriented_focus": "（【請注意】為了向下兼容舊版報告，此欄位請填入 active_behavioral_engagement 的平均值）",
      "receptive_engagement": "（【請注意】為了向下兼容舊版報告，此欄位請填入 passive_behavioral_engagement 的平均值）",
      "disengagement": "（【請注意】為了向下兼容舊版報告，此欄位請填入 behavioral_disengagement 的平均值）"
    }},
    "dominant_positive_behaviors": "（列出此區塊最主要的『主動行為參與』或『被動行為參與』的具體行為類別，例如：'目視書本/筆記, 做筆記'）",
    "dominant_negative_behaviors": "（列出此區塊最主要的『行為分心』的具體行為類別，例如：'目視同學, 玩弄手部/文具, 托腮'）"
  }},
  "focus_trigger_analysis": {{
      "positive_trigger": {{
          "trigger_snippet": {{
              "key_quote": "（找出最可能【提升】行為參與度的【一句老師的關鍵話語】）",
              "context_before": "（關鍵話語前的【前一句話】）",
              "context_after": "（關鍵話語後的【後一句話】）",
              "topic": "（為這段對話切片總結一個微主題，如：'明確指令與語法圈選'或'生活化比喻'）"
          }},
          "analysis": "（分析為何這段對話能提升行為參與度，例如：'教師以明確指令引導學生操作（圈選關鍵字），並即時解析語法結構，促使學生從被動行為參與轉化為主動行為參與。'）"
      }},
      "negative_trigger": {{
          "trigger_snippet": {{
              "key_quote": "（找出最可能【導致】行為分心的【一句老師的關鍵話語】）",
              "context_before": "（關鍵話語前的【前一句話】）",
              "context_after": "（關鍵話語後的【後一句話】）",
              "topic": "（為這段對話切片總結一個微主題，如：'長時間個人故事分享'）"
          }},
          "analysis": "（分析為何這段對話導致分心，例如：'教師進行與課程主題脫節的個人故事分享，導致學生分心行為（如目視同學、托腮）顯著增加。'）"
      }}
  }},
  "teaching_pacing_analysis": {{
      "cognitive_load_level": "（基於 `cognitive_load_metrics` 數據，綜合判斷此區塊的認知負荷是：高 / 中 / 低）",
      "analysis": "（分析此區塊的教學節奏。例如：『本區塊教師多次採用高語速、高密度的語法講解，認知負荷明顯偏高。雖然短期內促使學生大量做筆記，主動行為參與顯著，但部分學生也出現分心行為，顯示高負荷下專注度開始波動。』）"
  }},
  "ai_insight": "（你對這【整個時間區塊】的最終專業洞察。**請務必結合『行為參與分佈』、『教學節奏』和『教學模式』**，解釋它們之間的因果關係。例如：『本時段的行為參與波動與教學模式切換高度相關。高強度的「教師講解」模式能顯著提升學生的主動行為參與（如做筆記），但若連續時間過長，行為分心會逐漸累積。老師適時穿插「閒聊」作為認知緩衝，雖然短暫提升了分心比例，但有效幫助學生在後續高強度學習時能重新集中注意力。』）"
}}
"""
    
    # --- 核心部分 3：給予 AI 的系統級指令 ---
    return f"""
你是一位頂尖的教育數據科學家與心理學分析師，專長是從課堂數據中洞察教學模式與學生**行為參與 (Behavioral Engagement)** 之間的動態關聯。

**【★★★ 核心指令 (v8.0) ★★★】**
1.  你必須嚴格使用我提供的 **【學生行為參與狀態解碼指南 (v8.0)】** 作為你所有分析的理論基礎。
2.  你的分析必須包含提供前後文的「對話切片」(trigger_snippet)。
3.  **【高優先級任務】**: 你的輸入資料中新增了 `"teaching_mode"` 欄位。在你的 `ai_insight` 最終洞察中，你必須**深入分析這些【教學模式的切換】與學生行為參與狀態波動之間的因果關係**。例如：
    *   分析一個「閒聊」模式是否成功地在下一個「教師講解」開始前，讓學生的行為從「分心」恢復到「被動參與」？
    *   「學生練習」模式是否對應了「主動行為參與」行為（如"做筆記"）的數據高峰？

{behavior_decoder}

**你的輸入資料格式如下：**
一個JSON列表，每個物件代表一個30秒的時間區間，包含 `teacher_speech`, `student_behavior_summary`, `focus_distribution` (已更新為四維學術化結構), `cognitive_load_metrics`, 以及 `teaching_mode` 欄位。

**你的輸出格式要求：**
你必須嚴格遵循以下的JSON結構，並填寫所有欄位。你的回答必須是一個結構完整的 JSON 物件，絕對不能包含任何額外的文字、註解或 Markdown 標記。

{json_structure}
"""

def get_summary_prompt(pre_summary_str: str):
    """
    生成用於【最終深度匯總分析】的 Prompt (v5.3 - ★★★ 階層式摘要版 ★★★)
    - 輸入的不再是龐大的原始報告數據，而是一份由 AI 預先提煉的精簡摘要。
    """
    
    # 最終輸出的 JSON 結構保持不變，這是我們希望 AI 生成的目標格式
    json_structure = """
{
  "class_narrative": "（根據你收到的預摘要，撰寫一段話總結整堂課的流程與節奏）",
  "quantitative_pacing_analysis": {
    "overall_rhythm": "（分析整堂課的教學節奏。例如：『本堂課呈現約20-30分鐘為一週期的「高-低認知負荷」循環模式。』）",
    "avg_high_load_duration": "（估算一個「高認知負荷」教學片段（如密集語法講解）的平均持續時間，直到學生的分心狀態開始顯著上升為止。例如：『數據顯示，在持續約 7-10 分鐘的高語速、高密度概念講解後，學生的分心比例平均會上升超過15%。』）",
    "attention_reset_patterns": "（分析老師是如何成功「重置」學生注意力的。例如：『在高負荷教學後，老師通常採用平均長度為 1-2 分鐘的「低負荷個人故事」或「明確的互動提問」來作為認知緩衝，這類緩衝有 80% 的機率在隨後的 5 分鐘內將「任務導向專注」拉回高峰。』）"
  },
  "key_pattern_identification": {
    "focus_booster_patterns": [
      {
        "pattern_name": "模式一：指令式任務轉換",
        "description": "（描述這個模式。例如：『在一段發散的閒聊或故事分享後，老師使用明確的、帶有動詞的指令（如「來，寫一下」、「請圈起來」）將學生的注意力強制拉回到具體任務上。』）",
        "evidence": "（從預摘要中引用證據。例如：『此模式在摘要中提到的「關鍵拉升點」最為典型，老師在歷史講解後說出「我們來看這邊」，任務導向專注度急遽上升。』）"
      }
    ],
    "attention_sink_patterns": [
      {
        "pattern_name": "模式一：脫節的長篇敘事",
        "description": "（例如：『老師分享的個人故事或生活經驗，雖然有趣，但與當前的教學主題缺乏明確的關聯，且持續時間過長。』）",
        "evidence": "（例如：『摘要中提到，在分享軍中故事和拇指姑娘故事時，分心狀態比例均顯著提高。』）"
      }
    ]
  },
  "lexical_trigger_highlights": {
      "positive_trigger_words": "（分析摘要中提到的「專注提升點」話語，總結出最常出現的、能觸發專注的詞語類型。例如：『祈使句動詞（如「看」、「寫」、「找」）、疑問詞（如「什麼」、「為什麼」）、強調詞（如「注意」、「關鍵是」）。』）",
      "negative_trigger_words": "（分析摘要中提到的「專注下降點」話語，總結出最常與分心相關的詞語類型。例如：『第一人稱代詞（如「我以前」、「我想說」）、無明確教學目的的重複性詞語、與主題無關的名詞。』）"
  },
  "final_recommendations": [
    "（基於以上所有量化和模式分析，提供 2-3 條更具體、更數據驅動的教學建議。例如：『建議將高認知負荷的純講解控制在 8 分鐘以內，並在其後設計一個包含「祈使句動詞」的互動任務來重置學生專注度。』）",
    "（例如：『在分享個人故事時，嘗試在 1 分鐘內將其與教學要點進行類比或連結，以避免學生的「接收性專注」滑向「分心狀態」。』）"
  ]
}
"""

    # --- ★★★【核心修改點】★★★ ---
    # 修改開頭的說明文字，告知 AI 它現在的輸入是一份預摘要
    return f"""
你是一位頂尖的教育數據科學家與教學設計分析師。你的任務是接收一份已經被 AI 助理**高度濃縮過的課堂分析摘要**，並對其進行深度的、量化的二次分析，以挖掘出可複製的教學模式與細顆粒度的因果關係。

**【你的輸入資料】**
一份文字格式的**課堂分析預摘要**，它提煉了整堂課最關鍵的趨勢、模式與觸發點。

**【★★★ 你的核心任務 ★★★】**
你的輸出**不再是簡單的總結**，而是一份數據驅動的**模式分析報告**。你必須基於這份預摘要，完成以下四個層面的深度分析：

1.  **量化教學節奏分析 (`quantitative_pacing_analysis`)**: 從摘要中推斷老師教學節奏的內在週期性。估算高強度教學的「安全時長」，以及什麼樣的活動能最有效地「重置」學生注意力。
2.  **關鍵教學模式識別 (`key_pattern_identification`)**: 將摘要中提到的孤立事件歸納為可重複的「模式 (Pattern)」。定義出最有效的幾種「專注度助推器」和最需要警惕的「專-注度陷阱」。
3.  **詞彙觸發點分析 (`lexical_trigger_highlights`)**: 根據摘要中引用的關鍵話語，總結出哪些**類型的詞彙或句式**最常與專注度上升或下降相關聯。
4.  **數據驅動的建議 (`final_recommendations`)**: 基於以上所有分析，提出超越通用建議的、具有**量化指標**和**模式指導**的具體教學策略。

**【你的輸出格式要求】**
你必須嚴格遵循以下的JSON結構，不包含任何JSON格式以外的文字或註解。你的分析必須深入、具體，並以輸入的預摘要作為你分析的唯一依據。

{json_structure}

**【以下是提供給你進行深度分析的課堂預摘要】**
{pre_summary_str}
"""

async def analyze_chunk_with_ai_async(chunk, index, total, semaphore, retries=3):
    """
    (v5.4 非同步版本 - ★★★ 速度優化版 ★★★) 
    調用 Azure OpenAI 分析單個區塊。
    - 使用 Semaphore 控制最大並行任務數，避免API速率限制。
    - 增強了重試的等待邏輯。
    """
    if not chunk:
        print(f"  - ⚠️ 區塊 {index + 1}/{total} 為空，跳過分析。")
        return None

    # 使用 async with semaphore 來自動管理並行數量。
    # 在執行這段程式碼之前，它會等待 semaphore 發出許可。
    async with semaphore:
        print(f"  - (許可已取得) 正在發起區塊 {index + 1}/{total} 的分析請求...")
        
        chunk_start_time = chunk[0].get('start_time')
        chunk_end_time = chunk[-1].get('end_time')
        
        # get_chunk_analysis_prompt 函式本身是同步的，無需修改
        system_prompt = get_chunk_analysis_prompt(chunk_start_time, chunk_end_time)
        user_prompt = f"請分析以下課堂數據區塊：\n\n{json.dumps(chunk, ensure_ascii=False, indent=2)}"
        
        raw_content = ""

        for attempt in range(retries):
            try:
                # 在請求之間加入一個微小的隨機延遲，進一步錯開請求峰值，降低速率限制風險
                await asyncio.sleep(random.uniform(0.5, 1.5))
                
                response = await async_client.chat.completions.create(
                    model=TEXT_DEPLOYMENT_NAME,
                    response_format={"type": "json_object"},
                    messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                    max_tokens=8192,
                    temperature=0.1,
                )
                raw_content = response.choices[0].message.content
                return json.loads(raw_content)

            except json.JSONDecodeError as json_err:
                print(f"  - ⚠️ 區塊 {index + 1} 的 AI 返回 JSON 格式有誤 ({json_err})。正在嘗試自動修復...")
                # 建立一個用於修復 JSON 的簡單 Prompt
                fix_prompt = f"""你是一個 JSON 格式修復工具。你的任務是接收一段可能損壞或不完整的 JSON 文字，並盡力將其修復成一個語法正確的 JSON 物件。不要添加任何評論或解釋，只需返回修復後的 JSON 內容。
                
                這是損壞的文字：
                {raw_content}
                """
                try:
                    fix_response = await async_client.chat.completions.create(
                        model=TEXT_DEPLOYMENT_NAME,
                        response_format={"type": "json_object"},
                        messages=[{"role": "system", "content": "你是一個JSON格式修復工具。"}, {"role": "user", "content": fix_prompt}],
                        max_tokens=8192, temperature=0.0,
                    )
                    fixed_content = fix_response.choices[0].message.content
                    print(f"  - ✅ 區塊 {index + 1} JSON 已自動修復。")
                    return json.loads(fixed_content)
                except Exception as fix_e:
                    print(f"  - ❌ 區塊 {index + 1} 自動修復失敗: {fix_e}")
                    # 如果修復也失敗，直接進入下一次重試或結束
                    if attempt >= retries - 1:
                        break 
            
            except RateLimitError as e:
                # 增加重試的基礎等待時間，並加入隨機性
                wait_time = (5 * (attempt + 1)) + random.random() 
                print(f"  - ⚠️ 區塊 {index + 1} 觸發速率限制 (嘗試 {attempt + 1}/{retries})，將在 {wait_time:.1f} 秒後重試...")
                await asyncio.sleep(wait_time)
            
            except Exception as e:
                print(f"❌ 分析區塊 {index + 1} 時發生未知錯誤: {e}")
                if attempt < retries - 1:
                    # 採用指數退避策略增加等待時間
                    await asyncio.sleep(5 * (attempt + 1))
        
    print(f"  - ❌ 區塊 {index + 1}/{total} 在所有重試後分析失敗。")
    return None # 返回 None 表示失敗

async def create_pre_summary_for_final_analysis_async(all_chunk_analyses, micro_events, retries=3):
    """
    (★★★ 全新函式 ★★★)
    在最終深度分析前，先調用 AI 生成一份精簡的摘要。
    此舉旨在巨幅縮減最終請求的 Token 數量，避免觸發 TPM 限制。
    """
    print("步驟 6.8/8: 執行中間步驟 - 生成預摘要以縮減最終分析的 Token 量...")

    # --- 1. 將詳細的分析結果轉換為易於閱讀的文字格式 ---
    summary_input_text = "### 課堂各時間區塊核心洞察 ###\n"
    for i, analysis in enumerate(all_chunk_analyses):
        summary_input_text += f"\n--- 區塊 {i+1} ({analysis.get('start_time')} - {analysis.get('end_time')}) ---\n"
        summary_input_text += f"核心主題: {analysis.get('event_title', 'N/A')}\n"
        summary_input_text += f"AI 洞察: {analysis.get('ai_insight', 'N/A')}\n"
        
        pos_trigger = analysis.get('focus_trigger_analysis', {}).get('positive_trigger', {}).get('trigger_snippet', {}).get('key_quote')
        if pos_trigger:
            summary_input_text += f"專注提升點話語: \"{pos_trigger}\"\n"
            
        neg_trigger = analysis.get('focus_trigger_analysis', {}).get('negative_trigger', {}).get('trigger_snippet', {}).get('key_quote')
        if neg_trigger:
            summary_input_text += f"專注下降點話語: \"{neg_trigger}\"\n"

    summary_input_text += "\n\n### 關鍵微觀事件統計 ###\n"
    event_counts = defaultdict(int)
    for event in micro_events:
        event_counts[event['type']] += 1
    
    for event_type, count in event_counts.items():
        summary_input_text += f"- {event_type}: 共發生 {count} 次。\n"
        
    # --- 2. 設計 Prompt，要求 AI 進行提煉 ---
    system_prompt = """
你是一位高效的數據分析助理。你的任務是接收一份詳細的、分段的課堂分析報告，並將其提煉成一份高度濃縮的摘要。
你的摘要必須專注於以下幾點：
1.  **宏觀趨勢**: 整堂課的專注度起伏是否有一個大致的模式？
2.  **關鍵模式**: 總結有哪些重複出現的、能有效提升專注度的教學行為（助推器）？又有哪些總是導致分心的行為（陷阱）？
3.  **因果關係**: 找出教學節奏（例如，高強度講解後接著故事分享）與學生反應之間最明顯的幾組關聯。
4.  **具體證據**: 引用一兩個最典型的教師話語作為例子。

你的輸出必須是一段精煉的、條理清晰的文字。不要包含任何 JSON 或 Markdown 格式。
"""
    user_prompt = f"請根據以下詳細報告，為我生成一份濃縮的摘要：\n\n{summary_input_text}"

    # --- 3. 調用 AI ---
    for attempt in range(retries):
        try:
            response = await async_client.chat.completions.create(
                model=TEXT_DEPLOYMENT_NAME,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                max_tokens=2048, # 摘要不需要太長
                temperature=0.1
            )
            summary = response.choices[0].message.content
            print("✅ 成功生成預摘要。")
            return summary
        except RateLimitError as e:
            wait_time = 10 * (attempt + 1)
            print(f"  - ⚠️ 預摘要時觸發速率限制 (嘗試 {attempt + 1}/{retries})，將在 {wait_time} 秒後重試...")
            await asyncio.sleep(wait_time)
        except Exception as e:
            print(f"❌ 生成預摘要時發生錯誤: {e}")
            if attempt < retries - 1:
                await asyncio.sleep(5)
            else:
                return f"錯誤：無法生成預摘要。原始數據包含 {len(all_chunk_analyses)} 個區塊分析。" # 返回錯誤訊息

    return None

async def summarize_chunks_with_ai_async(pre_summary_str: str, retries=3):
    """
    (v5.5 穩定版 - ★★★ 階層式摘要版 ★★★) 
    接收一份精簡的預摘要，並調用 AI 進行最終的深度模式分析。
    """
    print("步驟 7/8: 調用 AI 進行最終的深度模式分析 (基於預摘要)...")
    
    # ★★★【核心修改點】★★★
    # get_summary_prompt 現在接收的是預摘要字串，而不是巨大的 JSON
    system_prompt = get_summary_prompt(pre_summary_str)
    
    for attempt in range(retries):
        try:
            start_time = time.time()
            response = await async_client.chat.completions.create(
                model=TEXT_DEPLOYMENT_NAME,
                # 注意：這裡的 prompt 內容大幅縮減，不再需要 response_format
                messages=[{"role": "system", "content": system_prompt}],
                max_tokens=8192,
                temperature=0.2
            )
            end_time = time.time()
            print(f"✅ AI 深度分析完成，耗時 {end_time - start_time:.2f} 秒。")

            raw_content = response.choices[0].message.content
            
            json_match = re.search(r'```json\s*([\s\S]*?)\s*```', raw_content, re.DOTALL)
            json_content_str = json_match.group(1) if json_match else raw_content

            cleaned_json_str = clean_json_string(json_content_str)

            return json.loads(cleaned_json_str)

        except json.JSONDecodeError as e:
            print(f"  - ⚠️ 最終分析返回的 JSON 格式錯誤 (嘗試 {attempt + 1}/{retries})，即使在清洗後仍然解析失敗: {e}")
            if attempt < retries - 1:
                print("     - 61 秒後重試...")
                await asyncio.sleep(61)
            else:
                print(f"❌ JSON 解析失敗，已達最大重試次數。")
                raise e
                
        except RateLimitError as e:
            # 這裡的重試邏輯仍然保留，作為雙重保險
            wait_time = 60 * (attempt + 1) + 1 
            print(f"  - ⚠️ 最終分析觸發速率限制 (嘗試 {attempt + 1}/{retries})，額度恢復中，將在 {wait_time} 秒後重試...")
            await asyncio.sleep(wait_time)
            
        except Exception as e:
            print(f"❌ 匯總分析時發生未知錯誤: {e}")
            if attempt < retries - 1:
                await asyncio.sleep(5 * (attempt + 1))
            else:
                raise e
            
# ==============================================================================
# --- 主執行流程 (v5.10 - 整合語速趨勢分析) ---
# ==============================================================================
async def main():
    start_total_time = time.time()
    
    # --- 階段一 & 二：數據載入與校正 (不變) ---
    student_behaviors, total_seconds = load_student_data(STUDENT_DATA_DIRECTORY, TARGET_SESSION_TIME)
    if student_behaviors is None: 
        exit()
    raw_transcript_for_correction = load_teacher_transcript(TEACHER_TRANSCRIPT_PATH)
    blackboard_images = load_blackboard_images(BLACKBOARD_IMAGES_DIRECTORY)
    preprocessed_transcript = preprocess_transcript(raw_transcript_for_correction)
    corrected_transcript = await correct_transcript_with_ai_async(preprocessed_transcript)

    # --- 階段三 & 四：建立整合時間軸並進行【即時微觀分類】---
    master_timeline = await create_master_timeline(
        student_behaviors, 
        corrected_transcript,
        blackboard_images, 
        total_seconds, 
        TIME_INTERVAL_SECONDS,
        state_timeline=None 
    )

    # 1. 執行無逐字稿與噪音片段的後處理規則
    master_timeline = post_process_silent_and_noisy_intervals(master_timeline)
    
    # 2. 強制合併與固化課堂中段的「下課休息」時段
    master_timeline = force_consolidate_break_time(master_timeline)
    
    # 3. 【核心修正點】：進行模式合併
    merged_timeline_raw = merge_and_cleanup_timeline(master_timeline)
    
    # 4. 【全新步驟】：在合併後的數據上計算 WPM
    merged_timeline_with_wpm = calculate_block_level_wpm(merged_timeline_raw, master_timeline)
    
    # --- 階段五：後續量化與AI洞察分析 (流程不變) ---
    
    # 注意：這裡的 state_trigger_keywords 必須使用帶有 WPM 的版本
    state_trigger_keywords = extract_trigger_keywords_for_states(merged_timeline_with_wpm)

    # ... (AI 精煉觸發關鍵句的邏輯不變，它會使用上面最新的 state_trigger_keywords) ...
    print("\n步驟 5.9/8: 呼叫 AI 精煉觸發關鍵句...")
    refinement_tasks = []
    semaphore_for_refine = asyncio.Semaphore(MAX_CONCURRENT_AI_TASKS) 

    for block in state_trigger_keywords:
        block['refined_quotes'] = []
        for quote in block['trigger_quotes']:
            task = refine_single_quote_async(quote, semaphore_for_refine)
            refinement_tasks.append(task)
            
    refined_quotes_list = await asyncio.gather(*refinement_tasks)

    quote_index = 0
    for block in state_trigger_keywords:
        num_quotes_in_block = len(block['trigger_quotes'])
        # 確保 quote_index 在範圍內
        if quote_index + num_quotes_in_block <= len(refined_quotes_list):
             block['refined_quotes'] = refined_quotes_list[quote_index : quote_index + num_quotes_in_block]
        quote_index += num_quotes_in_block
    
    print("✅ 所有關鍵句精煉完成。")
    
    # --- 階段六：AI 分塊分析 (使用 master_timeline，流程不變) ---
    micro_events = find_micro_events(master_timeline)
    aggregated_timeline = aggregate_timeline(master_timeline, aggregate_interval_minutes=5)
    teaching_mode_summary = calculate_mode_durations(master_timeline, total_seconds, TIME_INTERVAL_SECONDS)
    keyword_timestamps = find_important_keywords(master_timeline)
    teaching_cycle_analysis = identify_teaching_cycles(master_timeline)
    speech_rate_trend_analysis = analyze_speech_rate_trend(merged_timeline_with_wpm)
    high_intensity_segments = find_high_intensity_segments(master_timeline)

    # ... (AI 分塊分析的邏輯不變，使用 master_timeline) ...
    print(f"\n步驟 6/8: 開始分塊處理時間軸 (最大並行數: {MAX_CONCURRENT_AI_TASKS})...")
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_AI_TASKS)
    intervals_per_chunk = (CHUNK_SIZE_MINUTES * 60) // TIME_INTERVAL_SECONDS
    chunks = [master_timeline[i:i + intervals_per_chunk] for i in range(0, len(master_timeline), intervals_per_chunk)]
    analysis_tasks = [analyze_chunk_with_ai_async(chunk, i, len(chunks), semaphore) for i, chunk in enumerate(chunks)]
    
    all_chunk_analyses = await asyncio.gather(*analysis_tasks)
    all_chunk_analyses = [res for res in all_chunk_analyses if res is not None]

    print(f"✅ 所有 {len(all_chunk_analyses)} 個區塊初步分析完成。")
    
    # --- 階段七 & 八：最終匯總與儲存 (修正後的組合) ---
    try:
        if all_chunk_analyses:
            pre_summary = await create_pre_summary_for_final_analysis_async(all_chunk_analyses, micro_events)
            if not pre_summary: return
            final_deep_analysis = await summarize_chunks_with_ai_async(pre_summary)
            
            if final_deep_analysis:
                # final_report 組合時，使用帶有 WPM 的版本
                final_report = {
                    "class_session_id": f"{TARGET_SESSION_TIME.replace('/', '')}_english_class",
                    "overall_summary": final_deep_analysis,
                    "key_state_trigger_analysis": state_trigger_keywords,
                    "quantitative_summary": {
                        "teaching_mode_distribution": teaching_mode_summary,
                        "teaching_cycle_analysis": teaching_cycle_analysis,
                        "speech_rate_trend_analysis": speech_rate_trend_analysis,
                        "high_intensity_segment_analysis": high_intensity_segments
                    },
                    "keyword_timestamps": keyword_timestamps,
                    "micro_event_summary": micro_events,
                    "timeline_analysis": all_chunk_analyses,
                    "merged_timeline": merged_timeline_with_wpm, # ★★★ 最終報告使用這個變數！ ★★★
                    "detailed_timeline": master_timeline 
                }
                
                print("\n步驟 8/8: 儲存最終深度分析報告...")
                os.makedirs(OUTPUT_ANALYSIS_DIRECTORY, exist_ok=True)
                with open(OUTPUT_ANALYSIS_JSON_PATH, 'w', encoding='utf-8') as f:
                    json.dump(final_report, f, ensure_ascii=False, indent=2)
                
                end_total_time = time.time()
                total_duration = end_total_time - start_total_time

                print("-" * 50)
                print(f"🎉🎉🎉 全部分析完成！(宏觀模式整合版) 🎉🎉🎉")
                print(f"⏱️ 總耗時: {total_duration // 60:.0f} 分 {total_duration % 60:.2f} 秒。")
                print(f"✅ 綜合分析報告已成功儲存至: {OUTPUT_ANALYSIS_JSON_PATH}")
                print("-" * 50)
            else:
                 print("❌ AI 最終深度分析失敗，未生成報告檔案。")
        else:
            print("❌ 所有區塊均分析失敗，未生成任何報告檔案。")

    except Exception as e:
        print(f"\n❌ 在主流程中發生嚴重錯誤: {e}")
        import traceback
        traceback.print_exc()
        print("❌ 程式已終止。")

await main()

--- Classroom Analysis Engine v5.4 (Async Edition) ---
步驟 1/8: 初始化環境與 Azure OpenAI Async Client...
✅ Azure OpenAI Async client 初始化成功。
步驟 2/8: 讀取學生行為數據 (目標課堂: 09/28)...
✅ 成功載入 24 位學生的行為數據，課程總時長約 184 分鐘。
步驟 3/8: 讀取老師逐字稿 (採用新格式解析器)...
✅ 新格式逐字稿載入並處理完成，共 400 筆有效片段。
步驟 4/8: 讀取板書圖片列表...
✅ 板書圖片列表載入完成。
步驟 3.2/8: 執行逐字稿前處理過濾器...
✅ 逐字稿前處理完成：原始 400 筆 -> 過濾後 290 筆 -> 合併後 290 筆。
步驟 3.5/8: 調用 AI 進行逐字稿校正 (採用【上下文增強模式】)...
  - 逐字稿已分為 10 個區塊，將並行處理。
    - ✅ 逐字稿區塊 4/10 校正成功。
    - ✅ 逐字稿區塊 10/10 校正成功。
    - ✅ 逐字稿區塊 2/10 校正成功。
    - ✅ 逐字稿區塊 3/10 校正成功。
    - ✅ 逐字稿區塊 1/10 校正成功。
    - ✅ 逐字稿區塊 7/10 校正成功。
    - ✅ 逐字稿區塊 6/10 校正成功。
    - ✅ 逐字稿區塊 9/10 校正成功。
    - ✅ 逐字稿區塊 5/10 校正成功。
    - ✅ 逐字稿區塊 8/10 校正成功。
✅ AI 逐字稿校正完成。
步驟 5/8: 建立整合時間軸 (採用【微觀教學模式即時分類 v10.0】)...
  -> 已將課程分為 370 個微觀片段，開始並行 AI 分類...
  -> 所有微觀片段分類完成。
✅ 整合時間軸建立完成。
  -> 正在執行【後處理規則】：處理無逐字稿與噪音片段...
  -> 校正完成：共 31 個片段被強制歸類為「學生考試」。
  -> 合併完成：共 88 個無逐字稿片段已併入前一教學模式。
  -> 正在執行【強制性規則】：合併與固化「下課休息」時段...
  -> 在 1:00:00 偵測到下課休息觸發點。開始向後合併...
  -> 偵測到教學活動於 1:05:00 開始，停止

#### 測試

In [16]:
# -*- coding: utf-8 -*-
import os
import json
import datetime
import re
import asyncio
from openai import AsyncAzureOpenAI , APIError, RateLimitError
from dotenv import load_dotenv
from collections import defaultdict
import time
import random
import numpy as np
from scipy import stats
# ==============================================================================
# --- ★★★【使用者設定區】★★★ ---
# ==============================================================================
TARGET_SESSION_TIME = "09/28"

SESSION_DATE = '0928'

# 統一根目錄（避免重複寫 C:\Users\User\Desktop\test）
BASE_DIR = r'C:\Users\User\Desktop\test'

# 各資料夾與檔案路徑
STUDENT_DATA_DIRECTORY = os.path.join(BASE_DIR, 'SynologyDrive', 'json_behavior')
TEACHER_TRANSCRIPT_PATH = os.path.join(BASE_DIR, 'SynologyDrive', 'image', '上課影片', SESSION_DATE, '老師', f'{SESSION_DATE}_3.txt')
BLACKBOARD_IMAGES_DIRECTORY = os.path.join(BASE_DIR, 'note_blackboard', f'{SESSION_DATE}_English_filtered')

# STUDENT_DATA_DIRECTORY = r'C:\Users\User\Desktop\test\SynologyDrive\json_behavior'
# TEACHER_TRANSCRIPT_PATH = r'C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0928\老師\0928.txt'
# BLACKBOARD_IMAGES_DIRECTORY = r'C:\Users\User\Desktop\test\note_blackboard\0928_English_filtered'

OUTPUT_ANALYSIS_DIRECTORY = r'C:\Users\User\Desktop\test\classroom_analysis_report'
OUTPUT_ANALYSIS_JSON_PATH = os.path.join(OUTPUT_ANALYSIS_DIRECTORY, f'classroom_analysis_report_{TARGET_SESSION_TIME.replace("/", "")}.json')
TIME_INTERVAL_SECONDS = 30
CHUNK_SIZE_MINUTES = 20

# 設定 AI 分析任務的最大同時執行數量，避免觸發速率限制
MAX_CONCURRENT_AI_TASKS = 5

# ==============================================================================
# --- ★★★【全域常數設定區 (新增)】★★★ ---
# ==============================================================================
# 1. 主動行為參與 (Active Behavioral Engagement)
# 學生主動執行學習任務，是認知參與的強力指標。
ACTIVE_BEHAVIORAL_ENGAGEMENT = {
    "做筆記", "翻書", "主動舉手"
}

# 2. 被動行為參與 (Passive Behavioral Engagement)
# 學生遵守規範並接收資訊，是聽講時的有效參與形式。
PASSIVE_BEHAVIORAL_ENGAGEMENT = {
    "目視教師", "目視黑板", "坐姿直立"
}

# 3. 行為分心 (Behavioral Disengagement)
# 學生從事與當前學習任務無關的行為 (Off-task)。
BEHAVIORAL_DISENGAGEMENT = {
    "目視同學", "目視他處", "玩弄手部/文具", 
    "低頭(非學習)", "趴睡", "飲食", "喝水"
}

# 4. 程序性/模糊行為 (Procedural / Ambiguous Behavior)
# 行為本身模棱兩可，無法直接推斷其背後的認知狀態，需謹慎解讀。
AMBIGUOUS_BEHAVIORS = {
    "托腮", "觸摸臉部", "觸摸頭髮", 
    "被動舉手", "身體前傾", "身體後靠", 
    "被遮擋/無法判斷"
}

PROFESSIONAL_KEYWORDS = {
    "分詞", "句型", "形容詞", "語法", "時態", "子句", "完成式" 
}

CONTEXTUAL_GLOSSARY = {
    # --- 原有規則 ---
    "數學": ["初學"],
    "橫掃大陸": ["風收大陸"],
    "scare me": ["young me"],
    "吳宋熙": ["地圖鑑美吳宋熙"],

    # --- 新增規則 ---
    # 教學情境與口語
    "D": ["豬"],                   # 對答案時的常見念法
    "打罵教育": ["撒爸教育"],
    "結屎面": ["結一臉"],
    
    # 同音異義詞
    "鑫鑫腸": ["清場", "心情講"],    # 烤肉情境
    "中秋節": ["週週節"],          # 烤肉情境
    "bored": ["board"],             # 文法講解情境
    
    # 專有名詞
    "承恩": ["成恩"],
    
    # 英文詞彙
    "special magic tricks": ["spatial magic tracks"]
}

CLASSROOM_STATES_DEFINITIONS = {
    "下課休息": """
【最高優先級與獨佔性規則】判斷依據：
1.  **核心時間窗口**: 此狀態【僅會出現一次】，且必須發生在課程開始後的【60分鐘至120分鐘之間】。
2.  **觸發與合併**: 在上述時間窗口內，當老師明確說出「下課」、「休息一下」等關鍵詞後，該狀態即啟動。你必須將此觸發點之後【所有連續的、非教學性質】的片段（如`教學靜默`、`閒聊`）全部合併進來，形成一個【持續至少5分鐘，但不超過15分鐘】的單一 `下課休息` 區塊。
3.  **核心任務**: 你的目標是在指定時間窗口內，識別並合併出**唯一一個、連續的、符合時長**的休息時段。這是一個宏觀判斷，不要被零碎的對話打斷。
""",
    "教師講解": """
【核心定義】：老師作為主要發言者，圍繞【單一教學主題】（如一個文法點、一本書的解析）進行的、連續的知識輸出。
### ★★★ 規則強化 v2.1 ★★★ ###
【包含情境】：
1.  **教學性舉例 (Instructional Analogy)**: 為了闡述觀念而引用的**簡短**故事、比喻或個人經驗。**判斷關鍵**：只要其目的是輔助理解，且**在90秒內能重新連結回教學主題**，仍屬於此狀態。
2.  **即時性確認 (Comprehension Checks)**: 穿插在講解中的簡短問答，如「懂嗎？」、「對不對？」、「有沒有問題？」，這是講解流程的一部分，不應切斷此狀態。
3.  **教學框架內的設問與鋪陳 (Rhetorical Framing)**: 老師為了引出一個複雜概念而進行的自問自答或背景鋪陳（例如：「那到底什麼是『完成式』呢？我們得先從時間軸談起...」），只要其最終指向清晰的教學目標，就應被視為 `教師講解` 的框架。
【排除情境】：
1.  **長時間的題外話**: 當話題完全脫離當前教學軌道，且持續佔據主要時間（**超過2分鐘以上**），應被切分為【閒聊】。
2.  **系統性對答案**: 逐題式的、以核對答案為主的環節（例如，老師連續唸出 "C, B, A, C, D..."），應歸類為【師生互動】。
""",
    "學生練習": """
【觸發條件】：老師下達【明確的、要求學生獨立操作】的指令，且目的是為了鞏固剛教過的知識。
【關鍵詞】：「大家練習一下」、「給你們幾分鐘寫」、「現在動筆」、「試試看」、「把背面寫一下」。
【確認信號】：指令後通常會伴隨【長時間的教師靜默】或【低音量的同儕討論】。此狀態應包含整個操作時段，直到老師明確收回主導權。
【注意】：此模式不應出現在課程最開始的30分鐘內，除非有非常明確的非考試練習指令。
""",
    "學生考試": """
### ★★★ 規則強化 v3.0 ★★★ ###
【核心定義】：這是一個持續性的狀態，而非單一事件。必須同時滿足以下兩個條件才能被歸類為此模式：
1.  **時間窗口**: **【高優先級規則】** 此狀態**幾乎只會出現在【課程開始的前30分鐘內】**。
2.  **觸發與持續**:
    a. **觸發**: 老師明確說出「考試」、「前測」、「測驗」、「發考卷」等指令性關鍵詞。
    b. **持續**: 在該指令之後，出現了【長時間的、以無語音或極零碎話語為主】的時段。這些時段即使被微觀分析為 `學生練習` 或 `教學靜默`，你也應該根據宏觀情境將它們統一標記為 `學生考試`。
【注意】：如果老師在課程中段提到「考試」一詞，但其目的是在「講解」考試重點，則**不能**歸類為此模式。判斷的關鍵在於**指令**加上**後續的靜默行為**。
""",
    "師生互動": """
【核心定義】：所有非單向教學的、具有【明確教學或管理目的】的雙向或多向交流。
### ★★★ 規則強化 v2.1 ★★★ ###
【明確包含】：
1.  **結構化問答 (Structured Q&A)**: 老師針對【當前知識點】進行的多回合提問與解答，或學生主動提出的問題。
2.  **逐題檢討與對答案 (Answer Checking)**: 【高優先級規則】老師帶領全班**逐題核對**練習或考卷答案的環節，無論其中是否穿插簡短講解，主體都應被定義為 `師生互動`。
3.  **課堂管理 (Classroom Management)**: 宣布作業、提醒課程規劃、處理學生紀律問題、點名等。
【明確排除】：
1.  **教學設問 (Rhetorical Questions)**: 老師為了引出概念的自問自答式提問，應屬於【教師講解】。
2.  與教學目標推進無關的寒暄或多方閒聊，應屬於【閒聊】。
""",
    "閒聊": """
【核心定義】：內容與【當前學科知識點的推進】及【課堂管理】完全無關的非正式交流，且佔據了主要的對話時間。
【主要功能】：常作為高強度【教師講解】前後的【認知緩衝 (Cognitive Break)】或師生關係的潤滑劑。
### ★★★ 規則強化 v2.1 ★★★ ###
【明確包含】：
1.  **長時間的個人故事**: 老師或學生分享的完整個人經歷（例如軍中故事、旅遊經驗），且**持續超過2分鐘**或**未在短時間內連結回教學主題**。
2.  **延伸的題外話 (Digression)**: 由某個教學點引出，但內容已完全發散到與課程無關的領域（例如，由英文單字延伸到歷史故事、個人政治觀點、影視評論等）。
【判斷準則 v2.1】：
判斷的關鍵在於**「話題是否脫離教學軌道」**。如果一個題外話在1-2分鐘內被老師重新連結回教學主題，則應視為 `教師講解` 中的「教學性舉例」；反之，如果話題持續發散且時間較長，則必須歸類為 `閒聊`。
""",
}

# ==============================================================================
# --- API 金鑰與 Client 初始化 ---
# ==============================================================================
print("--- Classroom Analysis Engine v5.4 (Async Edition) ---")
print("步驟 1/8: 初始化環境與 Azure OpenAI Async Client...")
load_dotenv()
AZURE_API_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
TEXT_DEPLOYMENT_NAME = os.getenv("CHAT_COMPLETION_NAME")

if not all([AZURE_API_KEY, AZURE_ENDPOINT, TEXT_DEPLOYMENT_NAME]):
    print("❌ 錯誤：缺少必要的 Azure OpenAI 環境變數。請檢查 .env 檔案。")
    exit()

try:
    # 【修改】使用 AsyncAzureOpenAI 實例化 client
    async_client = AsyncAzureOpenAI(
        api_key=AZURE_API_KEY, 
        azure_endpoint=AZURE_ENDPOINT, 
        api_version="2024-02-01",
        max_retries=3 # 讓 client 內建一些基礎的重試邏輯
    )
    print("✅ Azure OpenAI Async client 初始化成功。")
except Exception as e:
    print(f"❌ 初始化 Azure OpenAI Async Client 時發生錯誤: {e}")
    exit()

# ==============================================================================
# --- 輔助函式與資料讀取模組 ---
# ==============================================================================
def clean_json_string(json_string: str) -> str:
    """
    (v5.5 新增) 清洗AI返回的字符串，移除其中非法的控制字符，以避免JSON解析錯誤。
    """
    # 建立一個正則表達式，匹配所有不被JSON標準允許的控制字符
    # \x00-\x1F 是主要的控制字符範圍, 但我們需要保留 \b, \f, \n, \r, \t
    # 正則表達式 [^\x20-\x7E\b\f\n\r\t] 匹配所有非可打印ASCII字符且不是合法JSON轉義符的字符
    # 為了更廣泛地處理Unicode，我們使用一個更簡單的方法：只保留 "合法的" 字符。
    
    cleaned_chars = []
    for char in json_string:
        # JSON 規範允許的控制字符是 \b, \f, \n, \r, \t
        if ord(char) < 32 and char not in '\b\f\n\r\t':
            # 如果是其他控制字符，則跳過
            continue
        cleaned_chars.append(char)
        
    return "".join(cleaned_chars)

def parse_timestamp(timestamp_str):
    """
    (v5.11 修正版) 解析多種時間戳格式，包括 H:MM:SS 和 HH:MM:SS。
    """
    h, m, s = 0, 0, 0
    
    # 處理圖片文件名格式
    match = re.match(r'(\d{2})-(\d{2})-(\d{2})-(\d{3})\.jpg', timestamp_str)
    if match: 
        h, m, s, _ = map(int, match.groups())
        return h * 3600 + m * 60 + s
        
    # --- ★★★【核心修正點：讓小時部分可以匹配1位或2位數字】★★★ ---
    # \d{1,2} 表示匹配1到2個數字
    match = re.match(r'(\d{1,2}):(\d{2}):(\d{2})', timestamp_str)
    if match: 
        parts = list(map(int, match.groups()))
        if len(parts) == 3:
            h, m, s = parts
            return h * 3600 + m * 60 + s
        # 有時候 timedelta 可能會輸出 H:MM 這樣的格式，這裡做一個兼容
        elif len(parts) == 2:
            m, s = parts
            return m * 60 + s
    # --- ★★★【修正結束】★★★ ---

    # 處理 WhisperX 的格式
    match = re.search(r'(\d{2})h(\d{2})m(\d{2})s', timestamp_str)
    if match: 
        h, m, s = map(int, match.groups())
        return h * 3600 + m * 60 + s
        
    return None

### --- MODIFIED SECTION --- ###
def load_student_data(directory, session_time):
    """(v4.1) 讀取並整合所有學生的行為數據，並加入數據完整性檢查"""
    print(f"步驟 2/8: 讀取學生行為數據 (目標課堂: {session_time})...")
    all_behaviors = []
    if not os.path.isdir(directory):
        print(f"❌ 錯誤：找不到學生資料夾 '{directory}'。")
        return None, 0
        
    student_folders = [f for f in os.listdir(directory) if os.path.isdir(os.path.join(directory, f))]
    loaded_students_count = 0
    
    for student_name in student_folders:
        student_dir = os.path.join(directory, student_name)
        for filename in os.listdir(student_dir):
            if filename.endswith('.json'):
                filepath = os.path.join(student_dir, filename)
                try:
                    with open(filepath, 'r', encoding='utf-8') as f: data = json.load(f)
                    
                    if data.get("report_metadata", {}).get("report_generation_time") == session_time:
                        student_id = data.get("report_metadata", {}).get("student_id", "未知學生")
                        
                        for behavior_list in data.get('detailed_sequence_analysis', []):
                            image_filenames = behavior_list.get('image_filenames_in_batch', [])
                            image_filenames_len = len(image_filenames)
                            
                            for image_highlight in behavior_list.get('analysis', {}).get('per_image_highlights', []):
                                
                                # ★★★【安全檢查】★★★
                                # 在存取前，先檢查索引是否合法
                                index = image_highlight.get('image_index_in_sequence')
                                if index is not None and index < image_filenames_len:
                                    img_filename = image_filenames[index]
                                    seconds = parse_timestamp(img_filename)
                                    if seconds is not None:
                                        all_behaviors.append({
                                            "seconds": seconds, 
                                            "student_id": student_id, 
                                            "behaviors": image_highlight.get("behavior_category", [])
                                        })
                                else:
                                    # 如果索引不合法，則印出警告並跳過此筆紀錄
                                    print(f"  - ⚠️ 數據不一致警告：在檔案 '{filename}' 中，偵測到無效的 image_index_in_sequence: {index} (列表長度為 {image_filenames_len})。將跳過此筆紀錄。")
                        
                        # print(f"  - 已載入 '{student_name}' 的報告。") # 為減少輸出訊息，可註解此行
                        loaded_students_count +=1
                        
                except Exception as e:
                    # 將 list index out of range 錯誤明確化
                    if isinstance(e, IndexError):
                         print(f"  - ⚠️ 嚴重數據錯誤：在解析 '{filename}' 時發生 'list index out of range'。這通常是源檔案數據不一致導致的。")
                    else:
                         print(f"  - ⚠️ 警告：讀取或解析 '{filename}' 時出錯: {e}")

    if not all_behaviors:
        print(f"❌ 錯誤：在 '{directory}' 中找不到任何符合 '{session_time}' 的學生報告。")
        return None, 0
        
    max_time = max(b['seconds'] for b in all_behaviors) if all_behaviors else 0
    print(f"✅ 成功載入 {loaded_students_count} 位學生的行為數據，課程總時長約 {max_time // 60} 分鐘。")
    return sorted(all_behaviors, key=lambda x: x['seconds']), max_time
### --- END MODIFIED SECTION --- ###

def load_teacher_transcript(filepath):
    """
    (v8.2 格式適應版) 
    讀取新的逐字稿格式 (HH:MM:SS: [Text])，並推斷出每個片段的結束時間。
    """
    print("步驟 3/8: 讀取老師逐字稿 (採用新格式解析器)...")
    if not os.path.exists(filepath):
        print(f"  - ⚠️ 警告：找不到逐字稿檔案 '{filepath}'，將跳過此項分析。")
        return []
    
    intermediate_data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            # --- 核心修改：更新正規表示式以匹配新格式 ---
            # 匹配 "HH:MM:SS: (可選的說話者標籤) [內容]"
            match = re.match(r'(\d{1,2}:\d{2}:\d{2}):\s*(?:[A-Z]:\s*)?(.+)', line)
            if match:
                start_str, text = match.groups()
                start_seconds = parse_timestamp(start_str)
                
                if start_seconds is not None and text.strip():
                    intermediate_data.append({
                        "start_seconds": start_seconds,
                        "text": text.strip()
                    })

    if not intermediate_data:
        print("  - ⚠️ 警告：逐字稿檔案為空或格式不符，無法解析任何內容。")
        return []

    # --- 推斷結束時間 (並加入時間上限防止異常) ---
    transcript_data = []
    MAX_SEGMENT_DURATION_SECONDS = 15  # 設定單句的最長持續時間上限

    for i in range(len(intermediate_data)):
        current_entry = intermediate_data[i]
        
        # 預設結束時間為開始時間 + 上限
        end_seconds = current_entry['start_seconds'] + MAX_SEGMENT_DURATION_SECONDS
        
        # 如果不是最後一個片段，嘗試用下一個片段的開始時間作為結束時間
        if i < len(intermediate_data) - 1:
            next_start_seconds = intermediate_data[i+1]['start_seconds']
            # 取兩者中的較小值，避免因長靜音導致持續時間過長
            end_seconds = min(end_seconds, next_start_seconds)

        if current_entry['start_seconds'] < end_seconds:
            transcript_data.append({
                "start_seconds": current_entry['start_seconds'],
                "end_seconds": end_seconds,
                "text": current_entry['text']
            })

    print(f"✅ 新格式逐字稿載入並處理完成，共 {len(transcript_data)} 筆有效片段。")
    return transcript_data

def load_blackboard_images(directory):
    print("步驟 4/8: 讀取板書圖片列表...")
    if not os.path.isdir(directory): print(f"  - ⚠️ 警告：找不到板書圖片資料夾 '{directory}'，將跳過此項分析。"); return []
    image_data = []
    for filename in os.listdir(directory):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            seconds = parse_timestamp(filename)
            if seconds is not None: image_data.append({"seconds": seconds, "filename": filename})
    print("✅ 板書圖片列表載入完成。")
    return sorted(image_data, key=lambda x: x['seconds'])

def find_state_for_timestamp(seconds: int, state_timeline: list):
    """根據秒數，從預先分析好的狀態時間軸中查詢對應的教學模式"""
    for state in state_timeline:
        # 假設 start_seconds 和 end_seconds 已經在 generate_state_timeline_async 中被計算好
        if state.get('start_seconds') is not None and state.get('end_seconds') is not None:
            if state['start_seconds'] <= seconds < state['end_seconds']:
                return state['classroom_state']
    # 如果課程結束後還有時間戳（例如學生數據比逐字稿長），給一個預設值
    return "課後" 

def force_apply_exam_rule_v2(master_timeline: list, total_seconds: int) -> list:
    """
    (全新後處理規則 v2) 在微觀分類後，強制應用更精準的宏觀考試規則。
    1. 只在課程前/後 30 分鐘生效。
    2. 尋找由 "考試指令關鍵字" 觸發的，且後續是長時間靜默/零碎語音的連續時段。
    3. 如果該時段超過15分鐘，則將其全部覆蓋為 "學生考試"。
    """
    print("  -> 正在執行【強制性 v2.0 - 精準考試規則】後處理...")
    if not master_timeline:
        return []

    # 定義時間窗口和門檻
    FIRST_WINDOW_END = 1800  # 前30分鐘
    LAST_WINDOW_START = total_seconds - 1800 # 後30分鐘
    MIN_EXAM_DURATION = 900 # 連續15分鐘
    LOOK_AHEAD_INTERVALS = 30 # 往後看 15 分鐘來確認是否為安靜時段
    
    # 定義觸發考試的關鍵字
    EXAM_TRIGGER_KEYWORDS = {"考試", "前測", "測驗", "發考卷"}
    
    # 定義可以被視為 "考試中" 的狀態
    QUIET_EXAM_LIKE_MODES = {"教學靜默", "學生練習", "學生考試"}
    FRAGMENTED_SPEECH_THRESHOLD = 10 # 考試期間老師可能會有簡短指令，字數少於10算零碎語音

    exam_start_index = -1

    # --- 掃描時間軸，只尋找 "觸發點" ---
    for i, interval in enumerate(master_timeline):
        seconds = interval["seconds"]
        is_in_start_window = (seconds < FIRST_WINDOW_END)
        
        # 規則：只在課程前30分鐘尋找考試的 "開始"
        if not is_in_start_window:
            continue

        speech = interval.get("teacher_speech", "")
        # 如果在時間窗口內，且老師的話語中包含觸發關鍵字
        if any(keyword in speech for keyword in EXAM_TRIGGER_KEYWORDS):
            
            # 檢查後續時段是否足夠安靜
            quiet_count = 0
            for j in range(i + 1, min(i + 1 + LOOK_AHEAD_INTERVALS, len(master_timeline))):
                next_interval = master_timeline[j]
                is_quiet_mode = next_interval["teaching_mode"] in QUIET_EXAM_LIKE_MODES
                is_fragmented = len(next_interval.get("teacher_speech", "")) < FRAGMENTED_SPEECH_THRESHOLD
                if is_quiet_mode or is_fragmented:
                    quiet_count += 1
            
            # 如果後續15分鐘內，超過80%的時間都是安靜的，我們就認為考試開始了
            if quiet_count / LOOK_AHEAD_INTERVALS > 0.8:
                exam_start_index = i
                print(f"  -> 在 {interval['start_time']} 偵測到考試觸發關鍵字：'{speech}'，並確認後續為長時段靜默。")
                break # 找到第一個就夠了

    # --- 如果找到了開始點，則向後延伸覆蓋，直到遇到明顯的教學活動 ---
    if exam_start_index != -1:
        # 從觸發點的下一個片段開始覆蓋
        for i in range(exam_start_index + 1, len(master_timeline)):
            interval = master_timeline[i]
            is_quiet_mode = interval["teaching_mode"] in QUIET_EXAM_LIKE_MODES
            is_fragmented = len(interval.get("teacher_speech", "")) < FRAGMENTED_SPEECH_THRESHOLD

            if is_quiet_mode or is_fragmented:
                interval["teaching_mode"] = "學生考試"
            else:
                # 一旦遇到明顯的教學活動（例如：老師開始高強度講解），就停止覆蓋
                end_time = interval["start_time"]
                print(f"  -> 考試狀態覆蓋至 {end_time} 結束。")
                break
    else:
        print("  -> 在指定窗口內未找到符合 '指令+長時段靜默' 模式的考試片段。")

    # (您也可以在這裡加入對課程結尾30分鐘的判斷，邏輯類似，但較少見，我們先專注於課前考試)

    return master_timeline

def force_practice_to_exam_in_first_30_mins(master_timeline: list) -> list:
    """
    在微觀分類後，強制將課程前30分鐘內的「學生練習」模式校正為「學生考試」。
    這是一個基於宏觀時間規則的硬性覆蓋，用以解決微觀分析的局限性。
    """
    print("  -> 正在執行【強制性規則】：校正前30分鐘的「學生練習」為「學生考試」...")
    
    # 定義時間窗口 (前30分鐘 = 1800秒)
    TIME_WINDOW_END_SECONDS = 1800 

    corrected_count = 0
    for interval in master_timeline:
        # 檢查是否在時間窗口內
        if interval["seconds"] < TIME_WINDOW_END_SECONDS:
            # 如果模式被微觀分類為「學生練習」，則強制修改
            if interval["teaching_mode"] == "學生練習":
                interval["teaching_mode"] = "學生考試"
                corrected_count += 1
    
    if corrected_count > 0:
        print(f"  -> 校正完成：共 {corrected_count} 個「學生練習」片段被重新標記為「學生考試」。")
    else:
        print("  -> 在前30分鐘內未發現需校正的「學生練習」片段。")
        
    return master_timeline

def force_consolidate_break_time(master_timeline: list) -> list:
    """
    在微觀分類後，強制尋找「下課休息」觸發點，並將其後續的非教學活動
    （如學生練習、閒聊、靜默）合併為一個連續的「下課休息」區塊。
    此版本將確保一個最少5分鐘，最多15分鐘的休息區塊。
    """
    print("  -> 正在執行【強制性規則】：合併與固化「下課休息」時段...")

    BREAK_WINDOW_START_SECONDS = 3600  # 60分鐘
    BREAK_WINDOW_END_SECONDS = 7200    # 120分鐘
    MIN_BREAK_DURATION_SECONDS = 300   # 強制最少休息5分鐘
    MAX_BREAK_DURATION_SECONDS = 900   # 強制最多休息15分鐘
    
    BREAK_TRIGGER_KEYWORDS = {"下課", "休息"}
    # 這些是明確的教學活動，遇到它們就應該停止合併休息時間
    STOP_MERGING_MODES = {"教師講解", "師生互動"}
    
    break_trigger_index = -1

    # 1. 在指定時間窗口內，尋找第一個觸發「下課休息」指令的區間
    for i, interval in enumerate(master_timeline):
        seconds = interval["seconds"]
        if BREAK_WINDOW_START_SECONDS <= seconds < BREAK_WINDOW_END_SECONDS:
            speech = interval.get("teacher_speech", "")
            if any(keyword in speech for keyword in BREAK_TRIGGER_KEYWORDS):
                break_trigger_index = i
                print(f"  -> 在 {interval['start_time']} 偵測到下課休息觸發點。開始向後合併...")
                break

    # 2. 如果找到了觸發點，則開始向後強制覆蓋
    if break_trigger_index != -1:
        break_start_seconds = master_timeline[break_trigger_index]["seconds"]
        
        # 從觸發點開始，一直向後檢查
        for i in range(break_trigger_index, len(master_timeline)):
            current_interval = master_timeline[i]
            current_seconds = current_interval["seconds"]
            duration_from_start = current_seconds - break_start_seconds

            # 停止條件 1: 超過了最大休息時間
            if duration_from_start >= MAX_BREAK_DURATION_SECONDS:
                print(f"  -> 達到15分鐘上限，於 {current_interval['start_time']} 停止合併。")
                break
                
            # 停止條件 2: 遇到了明確的教學活動，且已經滿足了最小休息時間
            if current_interval["teaching_mode"] in STOP_MERGING_MODES and duration_from_start >= MIN_BREAK_DURATION_SECONDS:
                print(f"  -> 偵測到教學活動於 {current_interval['start_time']} 開始，停止合併。")
                break
            
            # 如果不滿足停止條件，就強制覆蓋模式為「下課休息」
            current_interval["teaching_mode"] = "下課休息"
        else: 
            # 如果循環正常結束（沒有被break），說明休息直到課程結束
            print("  -> 下課休息時段合併至課程結束。")

    else:
        print("  -> 在 60-120 分鐘窗口內未找到明確的下課休息觸發點。")

    return master_timeline

def post_process_silent_and_noisy_intervals(master_timeline: list) -> list:
    """
    (全新後處理規則) 在微觀分類後，處理無逐字稿或僅有噪音的片段。
    規則 1 (高優先級): 在課程前30分鐘內，將所有「無逐字稿」或「無法辨識的噪音」的片段強制歸類為「學生考試」。
    規則 2 (一般情況): 在課程30分鐘後，將「無逐字稿」的片段歸類為前一個時間區段的教學模式，以達到合併效果。
    """
    print("  -> 正在執行【後處理規則】：處理無逐字稿與噪音片段...")
    if not master_timeline:
        return []

    # 定義時間窗口 (前30分鐘 = 1800秒)
    TIME_WINDOW_END_SECONDS = 1800
    NOISE_MARKER = "[無法辨識的噪音]"
    
    corrected_exam_count = 0
    merged_silent_count = 0

    # 我們從第二個片段開始循環，因為第一個片段沒有 "前一個" 模式可以參考
    for i in range(1, len(master_timeline)):
        current_interval = master_timeline[i]
        prev_interval = master_timeline[i-1]
        
        speech = current_interval.get("teacher_speech", "").strip()
        is_silent = not speech
        is_noise = NOISE_MARKER in speech
        
        # --- 規則 1: 處理課堂前30分鐘的特殊情況 ---
        if current_interval["seconds"] < TIME_WINDOW_END_SECONDS:
            if is_silent or is_noise:
                if current_interval["teaching_mode"] != "學生考試":
                    current_interval["teaching_mode"] = "學生考試"
                    corrected_exam_count += 1
                # 應用此規則後，跳過後續的通用規則
                continue
        
        # --- 規則 2: 處理30分鐘後的一般無逐字稿情況 ---
        if is_silent:
            prev_mode = prev_interval.get("teaching_mode")
            if prev_mode:
                current_interval["teaching_mode"] = prev_mode
                merged_silent_count += 1
    
    if corrected_exam_count > 0:
        print(f"  -> 校正完成：共 {corrected_exam_count} 個片段被強制歸類為「學生考試」。")
    if merged_silent_count > 0:
         print(f"  -> 合併完成：共 {merged_silent_count} 個無逐字稿片段已併入前一教學模式。")
    if corrected_exam_count == 0 and merged_silent_count == 0:
        print("  -> 未發現需要處理的無逐字稿或噪音片段。")

    return master_timeline

async def create_master_timeline(student_behaviors, transcript, blackboard_images, total_seconds, interval, state_timeline=None):
    """
    (v10.0 職責分離版) 建立整合時間軸，僅專注於數據整合與 AI 微觀分類。
    語速相關的計算將移至後處理函式中。
    """
    print("步驟 5/8: 建立整合時間軸 (採用【微觀教學模式即時分類 v10.0】)...")

    # --- Phase 1: 數據準備 (與之前相同) ---
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_AI_TASKS)
    intervals_to_process = []
    for start_interval in range(0, total_seconds + interval, interval):
        # ... (這整個 for 迴圈的內容完全不變，請保持原樣) ...
        end_interval = start_interval + interval
        interval_data = {
            "seconds": start_interval,
            "start_time": str(datetime.timedelta(seconds=start_interval)), 
            "end_time": str(datetime.timedelta(seconds=end_interval)), 
            "teacher_speech": " ".join([entry['text'] for entry in transcript if start_interval <= entry['start_seconds'] < end_interval]),
            "blackboard_snapshots": [image['filename'] for image in blackboard_images if start_interval <= image['seconds'] < end_interval],
            "student_behavior_summary": defaultdict(list), 
            "focus_distribution": None,
            "cognitive_load_metrics": None, # ★★★ 注意：這裡依然保留，但會在後續步驟中填充
            "teaching_mode": None
        }
        active_engagement_students = set()
        passive_engagement_students = set()
        disengagement_students = set()
        ambiguous_behavior_students = set()
        behaviors_in_interval = [b for b in student_behaviors if start_interval <= b['seconds'] < end_interval]
        for behavior in behaviors_in_interval:
            student_id = behavior['student_id']
            for b_category in behavior['behaviors']:
                interval_data["student_behavior_summary"][b_category].append(student_id)
                if b_category in ACTIVE_BEHAVIORAL_ENGAGEMENT: active_engagement_students.add(student_id)
                elif b_category in PASSIVE_BEHAVIORAL_ENGAGEMENT: passive_engagement_students.add(student_id)
                elif b_category in BEHAVIORAL_DISENGAGEMENT: disengagement_students.add(student_id)
                elif b_category in AMBIGUOUS_BEHAVIORS: ambiguous_behavior_students.add(student_id)
        all_students_in_interval = active_engagement_students.union(passive_engagement_students, disengagement_students, ambiguous_behavior_students)
        total_active_students = len(all_students_in_interval)
        if total_active_students > 0:
            interval_data["focus_distribution"] = {
                "active_behavioral_engagement": round((len(active_engagement_students) / total_active_students) * 100, 1),
                "passive_behavioral_engagement": round((len(passive_engagement_students) / total_active_students) * 100, 1),
                "behavioral_disengagement": round((len(disengagement_students) / total_active_students) * 100, 1),
                "ambiguous_behavior": round((len(ambiguous_behavior_students) / total_active_students) * 100, 1)
            }
        interval_data["student_behavior_summary"] = {k: list(set(v)) for k, v in interval_data["student_behavior_summary"].items()}
        intervals_to_process.append(interval_data)


    # --- Phase 2: 並行AI分類 (與之前相同) ---
    async def classify_and_update(interval_d):
        async with semaphore:
            classification_result = await classify_interval_teaching_mode_async(
                interval_d["teacher_speech"], 
                interval_d["student_behavior_summary"],
                interval_d["blackboard_snapshots"]
            )
            # --- ★★★【修改點：將 reason 改為 key_quote】★★★ ---
            interval_d["teaching_mode"] = classification_result.get("mode", "未知")
            interval_d["key_quote"] = classification_result.get("key_quote", "N/A") # 將 mode_reason 替換為 key_quote
            return interval_d

    print(f"  -> 已將課程分為 {len(intervals_to_process)} 個微觀片段，開始並行 AI 分類...")
    classification_tasks = [classify_and_update(data) for data in intervals_to_process]
    master_timeline = await asyncio.gather(*classification_tasks)
    print("  -> 所有微觀片段分類與溯因完成。")
    
    print("✅ 整合時間軸建立完成。")
    return master_timeline

def merge_consecutive_states(detailed_timeline: list) -> list:
    """
    (全新後處理功能) 將詳細時間軸中連續且相同的教學模式片段合併，並計算持續時長。
    """
    if not detailed_timeline:
        return []

    merged_timeline = []
    # 使用第一個片段初始化
    current_block = {
        "start_time": detailed_timeline[0]["start_time"],
        "end_time": detailed_timeline[0]["end_time"],
        "duration_seconds": TIME_INTERVAL_SECONDS,
        "teaching_mode": detailed_timeline[0]["teaching_mode"],
        "transcript_snippets": [detailed_timeline[0]["teacher_speech"]]
    }

    for i in range(1, len(detailed_timeline)):
        current_interval = detailed_timeline[i]
        
        # 如果當前模式與正在記錄的區塊相同
        if current_interval["teaching_mode"] == current_block["teaching_mode"]:
            # 更新結束時間和持續時長
            current_block["end_time"] = current_interval["end_time"]
            current_block["duration_seconds"] += TIME_INTERVAL_SECONDS
            if current_interval["teacher_speech"]:
                current_block["transcript_snippets"].append(current_interval["teacher_speech"])
        else:
            # 模式發生變化，先將上一個區塊儲存起來
            merged_timeline.append(current_block)
            # 然後開始一個新的區塊
            current_block = {
                "start_time": current_interval["start_time"],
                "end_time": current_interval["end_time"],
                "duration_seconds": TIME_INTERVAL_SECONDS,
                "teaching_mode": current_interval["teaching_mode"],
                "transcript_snippets": [current_interval["teacher_speech"]]
            }

    # 不要忘記儲存最後一個區塊
    merged_timeline.append(current_block)

    # 最後，將秒數轉換為更易讀的分秒格式，並整理逐字稿
    for block in merged_timeline:
        minutes, seconds = divmod(block["duration_seconds"], 60)
        block["duration_formatted"] = f"{int(minutes)} 分 {int(seconds)} 秒"
        block["full_transcript"] = " ".join(filter(None, block["transcript_snippets"])).strip()
        del block["transcript_snippets"] # 刪除臨時用的列表

    return merged_timeline

def merge_and_cleanup_timeline(detailed_timeline: list) -> list:
    """
    (v2.2 證據匯總版)
    1. 移除 "教學靜默" 模式。
    2. 合併連續相同的教學模式片段，並【匯總】所有原始的 "key_quote" 作為證據。
    """
    print("步驟 5.9/8: 執行時間軸合併與清理 (採用【證據匯總模式】)...")
    if not detailed_timeline:
        return []

    # --- 階段 1: 清理 "教學靜默" ---
    cleaned_timeline = list(detailed_timeline) 
    for i in range(1, len(cleaned_timeline)):
        if cleaned_timeline[i]["teaching_mode"] == "教學靜默":
            cleaned_timeline[i]["teaching_mode"] = cleaned_timeline[i-1]["teaching_mode"]

    # --- 階段 2: 合併連續模式並匯總證據 ---
    merged_timeline = []
    if not cleaned_timeline:
        return []

    # <--- 修改點 1: 初始化 current_block，並建立一個 trigger_quotes 列表 ---
    current_block = {
        "start_time": cleaned_timeline[0]["start_time"],
        "end_time": cleaned_timeline[0]["end_time"],
        "duration_seconds": TIME_INTERVAL_SECONDS,
        "teaching_mode": cleaned_timeline[0]["teaching_mode"],
        "transcript_snippets": [cleaned_timeline[0]["teacher_speech"]],
        "trigger_quotes": [] # 初始化為空列表
    }
    # 將第一個有效的 key_quote 加入
    if cleaned_timeline[0].get("key_quote") and cleaned_timeline[0].get("key_quote") != "N/A":
        current_block["trigger_quotes"].append(cleaned_timeline[0].get("key_quote"))


    for i in range(1, len(cleaned_timeline)):
        current_interval = cleaned_timeline[i]
        
        if current_interval["teaching_mode"] == current_block["teaching_mode"]:
            current_block["end_time"] = current_interval["end_time"]
            current_block["duration_seconds"] += TIME_INTERVAL_SECONDS
            if current_interval["teacher_speech"]:
                current_block["transcript_snippets"].append(current_interval["teacher_speech"])
            
            # <--- 修改點 2: 收集新的、不重複的、非 "N/A" 的 key_quote ---
            new_quote = current_interval.get("key_quote")
            if new_quote and new_quote != "N/A" and new_quote not in current_block["trigger_quotes"]:
                current_block["trigger_quotes"].append(new_quote)

        else:
            merged_timeline.append(current_block)
            current_block = {
                "start_time": current_interval["start_time"],
                "end_time": current_interval["end_time"],
                "duration_seconds": TIME_INTERVAL_SECONDS,
                "teaching_mode": current_interval["teaching_mode"],
                "transcript_snippets": [current_interval["teacher_speech"]],
                "trigger_quotes": [] # 為新區塊初始化
            }
            if current_interval.get("key_quote") and current_interval.get("key_quote") != "N/A":
                current_block["trigger_quotes"].append(current_interval.get("key_quote"))


    merged_timeline.append(current_block)

    # --- 階段 3: 格式化輸出 ---
    for block in merged_timeline:
        minutes, seconds = divmod(block["duration_seconds"], 60)
        block["duration_formatted"] = f"{int(minutes)} 分 {int(seconds)} 秒"
        block["full_transcript"] = " ".join(filter(None, block["transcript_snippets"])).strip()
        if not block["full_transcript"]:
            block["full_transcript"] = "(無語音)"
        del block["transcript_snippets"]

        # <--- 修改點 3: 如果一個區塊最終沒有收集到任何有效引文，則填入 "N/A" ---
        if not block["trigger_quotes"]:
            block["trigger_quotes"] = ["N/A"]


    print("✅ 時間軸合併與清理（證據匯總模式）完成。")
    return merged_timeline

def find_micro_events(master_timeline, peak_threshold=80.0, trough_threshold=40.0):
    """(v5.1 新增) 在完整時間軸上尋找專注度的高峰、低谷與關鍵轉折點"""
    print("步驟 5.5/8: 尋找微觀事件 (專注度高峰、低谷與轉折點)...")
    micro_events = []
    
    # 用於尋找最大轉折
    max_focus_rise = {"delta": 0, "event": None}
    max_disengagement_rise = {"delta": 0, "event": None}
    
    for i, interval in enumerate(master_timeline):
        dist = interval.get("focus_distribution")
        if not dist:
            continue

        # 檢查高峰和低谷
        if dist.get("task_oriented_focus", 0) >= peak_threshold:
            micro_events.append({
                "type": "專注度高峰 (任務導向)",
                "time": interval["start_time"],
                "details": f"任務導向專注度達到 {dist['task_oriented_focus']}%",
                "teacher_speech_context": interval.get("teacher_speech", "")
            })
        
        if dist.get("disengagement", 0) >= trough_threshold:
            micro_events.append({
                "type": "專注度低谷 (分心)",
                "time": interval["start_time"],
                "details": f"分心狀態佔比達到 {dist['disengagement']}%",
                "teacher_speech_context": interval.get("teacher_speech", "")
            })

        # 比較與前一個時間點的變化，尋找最大轉折
        if i > 0:
            prev_dist = master_timeline[i-1].get("focus_distribution")
            if prev_dist:
                focus_delta = dist.get("task_oriented_focus", 0) - prev_dist.get("task_oriented_focus", 0)
                disengagement_delta = dist.get("disengagement", 0) - prev_dist.get("disengagement", 0)
                
                if focus_delta > max_focus_rise["delta"]:
                    max_focus_rise["delta"] = focus_delta
                    max_focus_rise["event"] = {
                        "type": "關鍵拉升點",
                        "time": interval["start_time"],
                        "details": f"任務導向專注度在30秒內急遽上升 {focus_delta:.1f}%",
                        "teacher_speech_context": interval.get("teacher_speech", "")
                    }
                
                if disengagement_delta > max_disengagement_rise["delta"]:
                    max_disengagement_rise["delta"] = disengagement_delta
                    max_disengagement_rise["event"] = {
                        "type": "關鍵下跌點",
                        "time": interval["start_time"],
                        "details": f"分心狀態佔比在30秒內急遽上升 {disengagement_delta:.1f}%",
                        "teacher_speech_context": interval.get("teacher_speech", "")
                    }

    # 將找到的最大轉折點加入事件列表
    if max_focus_rise["event"]:
        micro_events.insert(0, max_focus_rise["event"]) # 放在最前面，因為很重要
    if max_disengagement_rise["event"]:
        micro_events.insert(0, max_disengagement_rise["event"])
        
    print(f"✅ 微觀事件分析完成，找到 {len(micro_events)} 個值得關注的時間點。")
    return micro_events

def aggregate_timeline(timeline, aggregate_interval_minutes=2):
    """
    將高粒度的時間軸數據聚合成較低粒度的平均值數據。
    (v5.11 修正版 - 增加對 None 值的安全檢查)
    """
    if not timeline:
        return []

    aggregate_interval_seconds = aggregate_interval_minutes * 60
    aggregated = []
    
    current_chunk_start = 0
    # ★★★【修正點 1：確保 timeline 非空，避免 IndexError】★★★
    last_second = timeline[-1]['seconds'] if timeline else 0

    while current_chunk_start <= last_second:
        chunk_end = current_chunk_start + aggregate_interval_seconds
        
        intervals_in_chunk = [
            item for item in timeline 
            if current_chunk_start <= item['seconds'] < chunk_end
        ]
        
        if intervals_in_chunk:
            # --- ★★★【修正點 2：在計算前，過濾掉 focus_distribution 為 None 的數據】★★★ ---
            valid_focus_data = [
                d['focus_distribution'] 
                for d in intervals_in_chunk 
                if d.get('focus_distribution') is not None
            ]
            
            # 只有在存在有效數據時才進行計算
            if valid_focus_data:
                num_valid_intervals = len(valid_focus_data)
                avg_task = sum(d.get('task_oriented_focus', 0) for d in valid_focus_data) / num_valid_intervals
                avg_receptive = sum(d.get('receptive_engagement', 0) for d in valid_focus_data) / num_valid_intervals
                avg_disengagement = sum(d.get('disengagement', 0) for d in valid_focus_data) / num_valid_intervals
                
                focus_distribution_summary = {
                    "task_oriented_focus": round(avg_task, 1),
                    "receptive_engagement": round(avg_receptive, 1),
                    "disengagement": round(avg_disengagement, 1)
                }
            else:
                # 如果這個區塊完全沒有專注度數據，則返回 None 或預設值
                focus_distribution_summary = None
            # --- ★★★【修正結束】★★★ ---

            # 合併老師話語
            combined_speech = " ".join(d.get('teacher_speech', '') for d in intervals_in_chunk if d.get('teacher_speech'))
            
            aggregated.append({
                "start_time": str(datetime.timedelta(seconds=current_chunk_start)),
                "end_time": str(datetime.timedelta(seconds=chunk_end)),
                "focus_distribution": focus_distribution_summary,
                "teacher_speech_summary": combined_speech[:100] + '...' if len(combined_speech) > 100 else combined_speech
            })
            
        current_chunk_start = chunk_end
        
    return aggregated

# ==============================================================================
# --- ★★★【v5.10 新增區塊：進階量化分析模組】★★★ ---
# ==============================================================================

def get_refine_trigger_prompt():
    return """
你是一位精煉語句的專家。你的唯一任務是從使用者提供的一段話中，提取出最核心、最能代表其指令或意圖的關鍵短語。

【規則】
1.  **極度簡潔**：只保留觸發動作或狀態改變的核心詞語。
2.  **移除贅詞**：刪除所有的鋪墊、語氣詞、無關的閒聊（例如 "好不好"、"真的假的"、"我不是要恐懼嚇你們"）。
3.  **保持原意**：提取的短語必須能獨立表達原始句子的核心指令。

【範例】
-   **輸入**: "還會跟我講這些奇奇怪怪的事情嗎?好不好,所以鼓勵大家,我不是要恐懼嚇你們,只是希望鼓勵你們多一個選擇,讓我們休息一下,讓大家消化一下,真的假的,還有嘛,有看過是嗎?還有不會的可以過來握手"
-   **輸出**: "讓我們休息一下, 讓大家消化一下"

-   **輸入**: "好，那我們來看第二個，我們等一下一併講這幾個，我們先看第四題，這一題哦。請問你如果after match, perfect, good news,你可以dance怎樣？"
-   **輸出**: "我們先看第四題"

你的回答只能是精煉後的短語，不要包含任何解釋。
"""

async def refine_single_quote_async(quote: str, semaphore: asyncio.Semaphore):
    """使用AI將單個長句子精煉成關鍵短語"""
    async with semaphore:
        system_prompt = get_refine_trigger_prompt()
        user_prompt = f"請精煉以下句子：\n---\n{quote}\n---"
        
        try:
            response = await async_client.chat.completions.create(
                model=TEXT_DEPLOYMENT_NAME,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                max_tokens=50,  # 精煉後的句子不需要太長
                temperature=0.0
            )
            return response.choices[0].message.content.strip().replace('"', '')
        except Exception as e:
            print(f"  - ⚠️ 精煉句子失敗: {e}")
            return quote # 如果失敗，就返回原始句子

def calculate_mode_durations(master_timeline, total_seconds, interval_seconds):
    """(v5.8) 量化並計算每種教學模式的持續時間"""
    print("步驟 5.6/8: 量化教學模式持續時間...")
    if not master_timeline: return {}
    mode_segments = defaultdict(list)
    if not master_timeline: return {}
    current_mode = master_timeline[0].get("teaching_mode")
    current_segment_duration = 0
    for interval_data in master_timeline:
        mode = interval_data.get("teaching_mode")
        if mode == current_mode:
            current_segment_duration += interval_seconds
        else:
            if current_mode: mode_segments[current_mode].append(current_segment_duration)
            current_mode = mode
            current_segment_duration = interval_seconds
    if current_mode and current_segment_duration > 0:
        mode_segments[current_mode].append(current_segment_duration)
    duration_summary = {}
    for mode, segments in mode_segments.items():
        total_duration = sum(segments)
        number_of_segments = len(segments)
        average_duration = total_duration / number_of_segments if number_of_segments > 0 else 0
        duration_summary[mode] = {
            "total_duration_seconds": total_duration,
            "total_duration_minutes": round(total_duration / 60, 1),
            "percentage_of_class": round((total_duration / total_seconds) * 100, 1) if total_seconds > 0 else 0,
            "number_of_segments": number_of_segments,
            "average_segment_duration_seconds": round(average_duration, 1)
        }
    print("✅ 教學模式持續時間量化完成。")
    return duration_summary

def find_important_keywords(master_timeline):
    """(v5.9) 找出教師提到特定關鍵字的時間點"""
    print("步驟 5.7/8: 標記教學關鍵字時間點...")
    KEYWORDS_TO_MARK = {"重點", "關鍵", "考試", "注意", "小心", "必考", "圈起來", "畫起來", "寫下來", "思考一下"}
    keyword_timestamps = []
    for i, interval in enumerate(master_timeline):
        speech = interval.get("teacher_speech", "")
        if any(keyword in speech for keyword in KEYWORDS_TO_MARK):
            matched_keyword = next((kw for kw in KEYWORDS_TO_MARK if kw in speech), None)
            context_before = master_timeline[i-1].get("teacher_speech", "") if i > 0 else ""
            context_after = master_timeline[i+1].get("teacher_speech", "") if i < len(master_timeline) - 1 else ""
            keyword_timestamps.append({
                "time": interval["start_time"], "keyword": matched_keyword, "full_quote": speech,
                "context_before": context_before, "context_after": context_after
            })
    print(f"✅ 成功標記出 {len(keyword_timestamps)} 個關鍵字時間點。")
    return keyword_timestamps

def find_high_intensity_segments(master_timeline: list, disengagement_threshold=30.0, rise_threshold=15.0) -> list:
    """
    (v1.0 新增) 掃描 master_timeline，找出所有「高強度教學」片段，
    並標記出學生分心度首次顯著上升的時間點。
    """
    print("步驟 5.7/8: 識別高強度教學片段與專注度下降點...")
    segments = []
    in_high_intensity_segment = False
    current_segment = None

    HIGH_INTENSITY_MODES = {"教師講解"}
    
    for i, interval in enumerate(master_timeline):
        mode = interval.get("teaching_mode")
        metrics = interval.get("cognitive_load_metrics")
        focus = interval.get("focus_distribution")

        is_high_intensity = (
            mode in HIGH_INTENSITY_MODES and
            metrics and
            metrics.get("intensity_level") in ["高強度", "中強度"]
        )

        # 偵測到一個新的高強度片段的開始
        if is_high_intensity and not in_high_intensity_segment:
            in_high_intensity_segment = True
            current_segment = {
                "start_time": interval["start_time"],
                "start_seconds": interval["seconds"],
                "breaking_point_time": None,
                "end_time": None,
                "duration_before_break_seconds": None,
                "trigger_reason": None,
                "initial_disengagement": focus.get("behavioral_disengagement", 0) if focus else 0
            }

        # 如果正在一個高強度片段中
        if in_high_intensity_segment:
            current_disengagement = focus.get("behavioral_disengagement", 0) if focus else 0
            
            # 檢查分心度是否觸發「引爆點」
            if current_segment and not current_segment["breaking_point_time"]:
                # 條件1：分心度超過一個絕對閾值
                if current_disengagement > disengagement_threshold:
                    current_segment["breaking_point_time"] = interval["start_time"]
                    current_segment["trigger_reason"] = f"分心度首次超過 {disengagement_threshold}%"
                # 條件2：分心度相比片段開始時，上升超過一個閾值
                elif (current_disengagement - current_segment["initial_disengagement"]) > rise_threshold:
                    current_segment["breaking_point_time"] = interval["start_time"]
                    current_segment["trigger_reason"] = f"分心度上升超過 {rise_threshold}%"

            # 如果高強度狀態結束，則關閉這個片段記錄
            if not is_high_intensity or i == len(master_timeline) - 1:
                in_high_intensity_segment = False
                if current_segment:
                    # 標記整個片段的結束時間
                    current_segment["end_time"] = interval["end_time"]
                    # 如果找到了引爆點，計算其持續時間
                    if current_segment["breaking_point_time"]:
                        break_point_seconds = parse_timestamp(current_segment["breaking_point_time"])
                        duration = break_point_seconds - current_segment["start_seconds"]
                        current_segment["duration_before_break_seconds"] = duration
                    segments.append(current_segment)
                    current_segment = None
                    
    print(f"✅ 高強度片段分析完成，共找到 {len(segments)} 個分析片段。")
    return segments
# ==============================================================================
# --- ★★★【v5.10 新增功能：量化語速變化趨勢】★★★ ---
# ==============================================================================
def get_state_timeline_prompt():
    """定義用於預處理逐字稿、生成宏觀時間軸的 AI 指令 (v13 策略升級版)"""
    # 將 CLASSROOM_STATES_DEFINITIONS 動態轉換為 Prompt 的一部分
    rules_description = ""
    for state, description in CLASSROOM_STATES_DEFINITIONS.items():
        rules_description += f"- **{state}**: {description.strip()}\n"

    ### ★★★ 【核心修改】：全面升級 System Prompt ★★★ ###
    return f"""
「你是一位頂尖的教育分析師，專長是從課堂對話中精準識別出教學活動的各個階段。你的任務是將一份完整的課堂逐字稿，切分成有意義、連續且分類明確的『課堂狀態』時間段。」

**【你的核心任務】**
分析使用者提供的【完整課堂逐字稿】，並以一個結構化的 JSON 物件作為輸出。你的分析必須涵蓋從頭到尾的整個逐字稿時間範圍。

**【課堂狀態的六個類別與判斷規則】**
你必須從以下六個類別中進行選擇，並嚴格遵循其定義：
{rules_description}

**【★★★ 分析策略：絕對優先級決策流程 (Absolute Priority Decision Flow) ★★★】**
為達到最高的分類準確性，你必須嚴格遵循以下決策流程，不得跳躍或違反順序：

**第一步：內容相關性篩選 (Content Relevance Filter - 最高優先級)**
- 這是你的**絕對第一步**。閱讀一段對話時，你必須先問自己：「這段對話的核心主題是否與當前的英文教學目標（例如文法、單字、解題、課堂管理）直接相關？」
- **如果答案為【否】**：無論其形式是獨白、問答還是多方對話，你都**必須**將其歸類為 **`閒聊`**。**此規則的優先級高於一切形式判斷。**
- **如果答案為【是】**：你才可以進入第二步。

**第二步：教學形式判斷 (Instructional Form Analysis)**
- 只有在確認內容與教學相關後，你才需要判斷其形式：
    - **單向知識輸出** -> 歸類為 **`教師講解`**。
    - **雙向問答、討論、對答案、課堂管理** -> 歸類為 **`師生互動`**。
    - **要求學生獨立操作的指令後，出現的長時間靜默** -> 歸類為 **`學生練習`** 或 **`學生考試`**。

**第三步：情境微調 (Contextual Refinement)**
- 在完成上述分類後，做最後的檢查。例如，一個極短（少於90秒）且目的明確是為了引出教學概念的比喻或故事，即使內容稍有偏離，也可以被視為 `教師講解` 的一部分。但這只是微調，不能違背第一步的最高原則。

**【常見錯誤分類警示 (Common Misclassification Alerts)】**
在進行「情境審核」時，請特別警惕以下常見錯誤：
- **錯誤範例 1 (講解誤判為閒聊)**: 將老師為了檢討考卷而逐題唸出答案（如 "CB諸位諸..."）的環節，錯誤地標記為 `閒聊`。
  - **✅ 正確做法**: 根據規則，此環節的核心目的是「對答案」，應歸類為 `師生互動`。
- **錯誤範例 2 (閒聊誤判為講解)**: 將老師分享個人飲食偏好（如「我最近愛上鍋皮辣椒雞」）或講述個人經歷的長篇獨白，僅因其形式是有結構的獨白而錯誤地標記為 `教師講解`。
  - **✅ 正確做法**: 識別出其核心內容與當前教學軌道無關，應歸類為 `閒聊`。
- **錯誤範例 3 (教學鋪陳誤判為互動)**: 將老師為了引出「子句」概念的設問句 `什麼叫子句？`，錯誤標記為 `師生互動`。
  - **✅ 正確做法**: 判斷出這是一個教學鋪陳，其後是連續的知識輸出，應歸類為 `教師講解`。

**【最終輸出要求 (JSON 格式)】**
你的回答**必須**是一個結構完整的 JSON 物件，絕對不能包含任何額外的文字、註解或 Markdown 標記。根物件需包含 `timeline` 鍵，其值為一個列表，每個元素都應包含 `start_time`, `end_time`, `classroom_state`, `state_summary`。時間必須是 "HH:MM:SS" 格式且連續。在你生成最終 JSON 之前，務必再次檢查並合併所有連續且相同的狀態區塊。
"""

def force_find_and_consolidate_break_v2_1(state_timeline: list) -> list:
    """
    (v2.1 精準修正版) 確保在60-120分鐘窗口內有且僅有一個「下課休息」時段。
    此版本只會合併連續的「閒聊」與「下課休息」片段，避免錯誤地吞噬「師生互動」。
    """
    print("  -> 正在執行【強制性 v2.1 - 精準合併模式】單次下課規則後處理...")
    
    BREAK_WINDOW_START_SECONDS = 3600  # 60分鐘
    BREAK_WINDOW_END_SECONDS = 7200    # 120分鐘
    MIN_BREAK_DURATION_SECONDS = 300   # 5分鐘 (可根據需求調整)
    
    # ★★★【核心修正點 1】★★★
    # 重新定義 "休息" 的核心組成，排除「師生互動」，因為它屬於教學活動。
    BREAK_CORE_STATES = {"閒聊", "下課休息"}

    window_states = []
    outside_window_states = []

    # 1. 將時間軸分割為 "窗口內" 與 "窗口外"
    for state in state_timeline:
        start_sec = state.get('start_seconds', 0)
        if BREAK_WINDOW_START_SECONDS <= start_sec < BREAK_WINDOW_END_SECONDS:
            window_states.append(state)
        else:
            outside_window_states.append(state)

    if not window_states:
        print("  -> ⚠️ 警告：在 60-120 分鐘窗口內無任何教學活動，無法尋找下課時間。")
        return state_timeline

    # 2. 在窗口內尋找最長的、僅由「閒聊」或「下課休息」組成的連續片段組
    best_group = []
    current_group = []
    for state in window_states:
        # ★★★【核心修正點 2】★★★
        # 使用更嚴格的 BREAK_CORE_STATES 進行判斷
        if state['classroom_state'] in BREAK_CORE_STATES:
            current_group.append(state)
        else:
            # 當遇到非休息狀態（如教師講解、師生互動）時，就結算前一個 group
            if current_group:
                duration = current_group[-1]['end_seconds'] - current_group[0]['start_seconds']
                best_duration = best_group[-1]['end_seconds'] - best_group[0]['start_seconds'] if best_group else 0
                if duration > best_duration:
                    best_group = list(current_group)
            current_group.clear()
    
    # 處理循環結束後最後一個可能的 group
    if current_group:
        duration = current_group[-1]['end_seconds'] - current_group[0]['start_seconds']
        best_duration = best_group[-1]['end_seconds'] - best_group[0]['start_seconds'] if best_group else 0
        if duration > best_duration:
            best_group = list(current_group)

    # 3. 如果找到了足夠長的連續休息片段，則合併它們
    valid_break = None
    final_timeline = outside_window_states.copy()

    if best_group and (best_group[-1]['end_seconds'] - best_group[0]['start_seconds']) >= MIN_BREAK_DURATION_SECONDS:
        valid_break = {
            "start_time": best_group[0]['start_time'],
            "end_time": best_group[-1]['end_time'],
            "classroom_state": "下課休息",
            "state_summary": "中場休息時間。",
            "start_seconds": best_group[0]['start_seconds'],
            "end_seconds": best_group[-1]['end_seconds']
        }
        final_timeline.append(valid_break)
        print(f"  -> 成功合併窗口內的 {len(best_group)} 個非教學片段，強制生成下課休息時段：({valid_break['start_time']} - {valid_break['end_time']})")
        
        break_start_sec = valid_break['start_seconds']
        break_end_sec = valid_break['end_seconds']
        for state in window_states:
            if not (break_start_sec <= state['start_seconds'] < break_end_sec):
                final_timeline.append(state)
    else:
        final_timeline.extend(window_states)
        print("  -> ⚠️ 警告：在 60-120 分鐘窗口內未找到長度超過5分鐘的連續非教學片段。")

    return post_process_state_timeline(sorted(final_timeline, key=lambda x: x['start_seconds']))

def get_reclassification_prompt():
    """生成一個用於二次分類、高度聚焦的 Prompt"""
    return """
你是一位嚴謹的教育內容審核員。你的唯一任務是判斷一段30秒到數分鐘不等的教師話語，其核心主題是否【直接】與【高中英文課程】的教學目標相關。

【判斷標準】
- 如果內容是關於：英文文法、單字解析、閱讀理解技巧、解題策略、課堂管理指令 -> 回答 `教師講解` 或 `師生互動`。
- 如果內容是關於：老師的個人經歷（軍旅、旅遊、薪資）、與課程無關的歷史故事、動漫、影視評論、時事閒聊 -> 回答 `閒聊`。

【輸出要求】
你的回答【只能是】`教師講解`、`師生互動` 或 `閒聊` 這三個選項之一，絕對不能包含任何其他文字或解釋。
"""

async def refine_chat_vs_lecture_states(state_timeline: list, semaphore: asyncio.Semaphore) -> list:
    """
    (v1.0 新增) 使用輕量級的二次 AI 請求，審核並校正被誤標為「教師講解」的「閒聊」片段。
    """
    print("  -> 正在執行【AI二次審核】以校正「教師講解」與「閒聊」的分類...")
    
    tasks = []
    
    async def reclassify_segment(state):
        async with semaphore:
            # 只審核長度超過60秒的「教師講解」，短片段通常不容易出錯且節省API成本
            if state['classroom_state'] == '教師講解' and (state['end_seconds'] - state['start_seconds']) > 60:
                
                # 提取這段時間內的原始逐字稿內容 (這需要一個輔助邏輯)
                # 為了簡化，我們假設 `state` 物件中已經包含了對應的文本。
                # 如果沒有，您需要從 master_timeline 中提取或重新構建。
                # 這裡我們假設 state_summary 儲存了足夠的上下文
                speech_context = state.get('state_summary', '') 
                
                try:
                    response = await async_client.chat.completions.create(
                        model=TEXT_DEPLOYMENT_NAME,
                        messages=[
                            {"role": "system", "content": get_reclassification_prompt()},
                            {"role": "user", "content": f"請判斷以下內容的核心主題：\n\n---\n{speech_context}\n---"}
                        ],
                        max_tokens=20,
                        temperature=0.0
                    )
                    corrected_mode = response.choices[0].message.content.strip()

                    if corrected_mode in ["閒聊", "師生互動"] and corrected_mode != state['classroom_state']:
                        print(f"    - 校正: 片段 {state['start_time']} 被從 '{state['classroom_state']}' 修改為 '{corrected_mode}'")
                        state['classroom_state'] = corrected_mode
                except Exception as e:
                    print(f"    - ⚠️ 二次審核失敗於片段 {state['start_time']}: {e}")
        return state

    for state in state_timeline:
        tasks.append(reclassify_segment(state))
    
    refined_timeline = await asyncio.gather(*tasks)
    
    # 再次合併可能產生的連續相同狀態
    return post_process_state_timeline(refined_timeline)

def clean_and_prepare_raw_transcript_for_states(filepath):
    """(v8.2 格式適應版) 清理新格式逐字稿，移除雜訊並標記長時間靜默，以供宏觀分析使用。"""
    print("  -> 正在執行逐字稿預清理以進行宏觀分析...")
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = f.read().strip().split('\n')
    except FileNotFoundError:
        return ""

    cleaned_lines = []
    time_format = '%H:%M:%S'
    last_end_time_obj = None

    for line in lines:
        # --- 核心修改：使用與 load_teacher_transcript 相同的解析邏輯 ---
        match = re.match(r'(\d{1,2}:\d{2}:\d{2}):\s*(.+)', line)
        if not match:
            continue
        
        start_time_str, text = match.groups()
        
        try:
            # 由於我們不知道結束時間，這裡的邏輯是基於開始時間的間隔
            start_time_obj = datetime.datetime.strptime(start_time_str, time_format)
            
            if last_end_time_obj:
                silence_duration = (start_time_obj - last_end_time_obj).total_seconds()
                if silence_duration > 60:
                    silence_minutes = round(silence_duration / 60)
                    marker_start = last_end_time_obj.strftime(time_format)
                    marker_end = start_time_obj.strftime(time_format)
                    cleaned_lines.append(f"{marker_start} - {marker_end}: [--- 長時間靜默 ({silence_minutes} 分鐘) ---]")
            
            # 為了讓 AI 能理解，我們模擬一個簡短的結束時間
            # 注意：這裡的結束時間僅用於宏觀分析的 text block，不會影響後續的精準計算
            simulated_end_time_obj = start_time_obj + datetime.timedelta(seconds=5)
            
            cleaned_lines.append(f"{start_time_obj.strftime(time_format)} - {simulated_end_time_obj.strftime(time_format)}: {text}")
            last_end_time_obj = start_time_obj # 下次比較的基準是當前的開始時間
        except ValueError:
            continue

    print("  -> 宏觀分析預清理完成。")
    return "\n".join(cleaned_lines)

def post_process_state_timeline(timeline):
    """後處理時間軸，合併零碎片段，並根據情境規則強制修正狀態。"""
    if not timeline: return []
    print("  -> 正在執行AI宏觀分析結果後處理...")
    
    # 這裡可以保留或簡化您舊程式碼中的後處理邏輯，因為新的Prompt已經很強大。
    # 核心是合併連續的相同狀態。
    merged_timeline = [timeline[0]]
    for i in range(1, len(timeline)):
        last_entry = merged_timeline[-1]
        current_entry = timeline[i]
        
        if last_entry['classroom_state'] == current_entry['classroom_state']:
            last_entry['end_time'] = current_entry['end_time']
        else:
            merged_timeline.append(current_entry)
            
    print("  -> 宏觀分析後處理完成。")
    return merged_timeline

async def generate_state_timeline_async(raw_transcript_path):
    """
    非同步函式，執行宏觀分析，生成權威的教學模式時間軸。
    """
    prepared_transcript = clean_and_prepare_raw_transcript_for_states(raw_transcript_path)
    if not prepared_transcript:
        print("❌ 錯誤：無法讀取或準備用於宏觀分析的逐字稿。")
        return None

    system_prompt = get_state_timeline_prompt()
    user_prompt = f"**【完整課堂逐字稿】**\n\n{prepared_transcript}"

    print(f"  -> 正在向 AI ({TEXT_DEPLOYMENT_NAME}) 發送宏觀分析請求...")
    try:
        response = await async_client.chat.completions.create(
            model=TEXT_DEPLOYMENT_NAME,
            response_format={"type": "json_object"},
            messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
            max_tokens=8192, # 給予足夠的空間
            temperature=0.0
        )
        
        raw_content = response.choices[0].message.content
        parsed_data = json.loads(raw_content)
        
        timeline = parsed_data.get("timeline", [])
        processed_timeline = post_process_state_timeline(timeline)
        
        # 將時間戳轉換為秒數以便後續查詢
        for entry in processed_timeline:
            entry['start_seconds'] = parse_timestamp(entry['start_time'])
            entry['end_seconds'] = parse_timestamp(entry['end_time'])

        return processed_timeline

    except Exception as e:
        print(f"❌ 宏觀教學模式分析失敗: {e}")
        return None

def calculate_block_level_wpm(merged_timeline: list, detailed_timeline: list) -> list:
    """
    (v11.0 升級版 - 動態校準與邊界設定) 
    在合併後的時間軸上，為每一個教學模式區塊計算一個更精準的 WPM。
    - 新增1: 根據區塊內實際有語音的片段比例，動態計算說話時間。
    - 新增2: 增加語速的上下限，防止出現不合理的極端值。
    """
    print("步驟 5.95/8: 執行區塊級語速計算 (採用【動態校準與邊界設定模式】)...")

    # --- ★★★【修改點 1：邊界設定】★★★ ---
    MAX_WPM = 350  # 設定人類說話的合理語速上限，防止因計算導致的數值爆炸
    MIN_WPM_FOR_VALID_SPEECH = 20 # 有效持續說話的最低語速，低於此值可能只是零碎指令

    # 中文「字」到英文「詞」的轉換率保持不變
    CHINESE_CHAR_TO_WORD_RATIO = 1.7

    for block in merged_timeline:
        total_chars = len(block.get("full_transcript", "(無語音)"))
        duration_seconds = block.get("duration_seconds", 0)

        # 如果區塊本身就沒有語音內容或時長，直接設為 0
        if total_chars == 0 or not duration_seconds:
            block["calibrated_wpm"] = 0
            continue
        
        # --- ★★★【修改點 2：動態計算說話佔比】★★★ ---
        start_seconds = parse_timestamp(block["start_time"])
        end_seconds = parse_timestamp(block["end_time"])

        # 1. 從 detailed_timeline 中篩選出此區塊包含的所有 30 秒微觀片段
        intervals_in_block = [
            interval for interval in detailed_timeline 
            if start_seconds is not None and end_seconds is not None and start_seconds <= interval["seconds"] < end_seconds
        ]
        
        if not intervals_in_block:
            block["calibrated_wpm"] = 0
            continue

        # 2. 計算其中真正有老師說話的片段佔了多少比例
        speech_intervals_count = sum(1 for interval in intervals_in_block if interval.get("teacher_speech", "").strip())
        total_intervals_count = len(intervals_in_block)

        # 3. 得到此區塊「真實的」說話時間比例
        dynamic_speech_ratio = (speech_intervals_count / total_intervals_count) if total_intervals_count > 0 else 0
        
        # 為了避免除以零或極小數，設定一個保底的最小說話比例
        if dynamic_speech_ratio < 0.05:
            dynamic_speech_ratio = 0.05

        # 4. 使用這個動態比例來計算估算的有效說話時長
        estimated_speech_duration = duration_seconds * dynamic_speech_ratio

        # --- ★★★【修改點 3：邊界應用與特殊模式處理】★★★ ---

        # 處理特殊模式：如果一個區塊被判定為練習/考試/休息，且總字數很少，
        # 這代表它可能只是一個開頭的指令，其語速沒有參考價值，直接設為 0。
        if block["teaching_mode"] in ["學生練習", "學生考試", "下課休息"] and total_chars < 20:
            block["calibrated_wpm"] = 0
            continue

        # 計算 WPM
        cpm = (total_chars / estimated_speech_duration) * 60
        wpm = cpm / CHINESE_CHAR_TO_WORD_RATIO
        calibrated_wpm = round(wpm, 1)

        # 應用語速上下限，避免出現 729.4 這種極端值
        if calibrated_wpm > MAX_WPM:
            calibrated_wpm = MAX_WPM
        
        # (可選) 如果語速低於閾值但又不為0，可以將其視為零碎指令，或統一到一個最低值
        if 0 < calibrated_wpm < MIN_WPM_FOR_VALID_SPEECH:
            calibrated_wpm = MIN_WPM_FOR_VALID_SPEECH

        block["calibrated_wpm"] = calibrated_wpm

    print("✅ 區塊級語速計算（動態校準版）完成。")
    return merged_timeline

def analyze_speech_rate_trend(merged_timeline_with_wpm: list):
    """
    (v10.0 升級版) 使用【區塊級校準後語速】進行線性迴歸分析，以獲得更穩定、更具代表性的趨勢。
    """
    print("步驟 5.9/8: 分析語速變化趨勢 (基於區塊級校準後語速)...")

    # 1. 提取所有包含【校準後語速】數據的時間點和語速值
    time_points = []
    wpm_points = []
    for block in merged_timeline_with_wpm:
        wpm = block.get("calibrated_wpm")
        if wpm is not None and wpm > 0:
            # 使用每個區塊的中間點作為代表時間點
            start_seconds = parse_timestamp(block["start_time"])
            end_seconds = parse_timestamp(block["end_time"])
            if start_seconds is not None and end_seconds is not None:
                midpoint_seconds = start_seconds + (end_seconds - start_seconds) / 2
                time_points.append(midpoint_seconds)
                wpm_points.append(wpm)
    
    # ... (後續的所有計算邏輯完全保持不變) ...
    if len(time_points) < 5: # 區塊較少，可以降低數據點門檻
        print("  - ⚠️ 語速數據點過少，跳過趨勢分析。")
        return {
            "error": "Not enough data points to perform trend analysis.",
            "trend_description": "數據不足",
            "linear_regression_stats": None,
            "comparative_summary": None
        }

    x = np.array(time_points)
    y = np.array(wpm_points)
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

    trend_description = ""
    if p_value < 0.05:
        if slope < -0.01:
            trend_description = "呈現顯著的下降趨勢 (老師的語速隨時間推移而顯著變慢)。"
        elif slope > 0.01:
            trend_description = "呈現顯著的上升趨勢 (老師的語速隨時間推移而顯著變快)。"
        else:
            trend_description = "語速非常穩定，未觀察到顯著的變化趨勢。"
    else:
        trend_description = "語速整體波動，未觀察到統計上顯著的長期增減趨勢。"

    midpoint_time = time_points[-1] / 2 if time_points else 0
    first_half_wpm = [wpm for t, wpm in zip(time_points, wpm_points) if t <= midpoint_time]
    second_half_wpm = [wpm for t, wpm in zip(time_points, wpm_points) if t > midpoint_time]
    
    avg_first_half = np.mean(first_half_wpm) if first_half_wpm else 0
    avg_second_half = np.mean(second_half_wpm) if second_half_wpm else 0

    analysis_result = {
        "trend_description": trend_description,
        "linear_regression_stats": {
            "slope_per_second": slope,
            "slope_per_minute": slope * 60,
            "p_value": p_value,
            "r_squared": r_value**2
        },
        "comparative_summary": {
            "average_wpm_first_half": round(avg_first_half, 1),
            "average_wpm_second_half": round(avg_second_half, 1),
            "change_percentage": round(((avg_second_half - avg_first_half) / avg_first_half) * 100, 1) if avg_first_half > 0 else 0
        }
    }
    
    print(f"✅ 語速趨勢分析完成: {trend_description}")
    return analysis_result

# ==============================================================================
# --- ★★★【新增區塊：逐字稿前處理過濾器】★★★ ---
# ==============================================================================
def preprocess_transcript(transcript_data: list, wps_threshold=0.8, merge_interval_seconds=1.5) -> list:
    """
    (v6.0 新增) 在 AI 校正前對逐字稿進行預處理。
    1. 過濾掉低語音密度的片段 (可能是靜音或噪音)。
    2. 合併時間上相鄰且語意上可能連續的破碎語句。
    """
    print("步驟 3.2/8: 執行逐字稿前處理過濾器...")
    
    # --- 階段 1: 語音密度過濾器 (Word-Per-Second Filter) ---
    filtered_data = []
    for entry in transcript_data:
        duration = entry['end_seconds'] - entry['start_seconds']
        if duration <= 0:
            continue
        
        # 計算每秒字數 (Words Per Second)
        wps = len(entry['text']) / duration
        
        # 如果 wps 低於閾值，則認為是無效語音，直接跳過
        if wps < wps_threshold:
            continue
            
        filtered_data.append(entry)

    if not filtered_data:
        print("  - ⚠️ 過濾後無有效逐字稿數據。")
        return []

    # --- 階段 2: 碎片化語句合併 (Fragment Merging) ---
    merged_data = []
    i = 0
    while i < len(filtered_data):
        current_entry = filtered_data[i]
        
        # 向後查看，看有多少個片段可以合併
        j = i + 1
        while j < len(filtered_data):
            next_entry = filtered_data[j]
            gap = next_entry['start_seconds'] - current_entry['end_seconds']
            
            # 如果時間間隔小於閾值，就合併它們
            if gap < merge_interval_seconds:
                current_entry['text'] += " " + next_entry['text']
                current_entry['end_seconds'] = next_entry['end_seconds']
                j += 1
            else:
                break
        
        merged_data.append(current_entry)
        i = j

    print(f"✅ 逐字稿前處理完成：原始 {len(transcript_data)} 筆 -> 過濾後 {len(filtered_data)} 筆 -> 合併後 {len(merged_data)} 筆。")
    return merged_data

# ==============================================================================
# --- ★★★【v5.9 新增區塊：關鍵片段微觀分析模組 (全新升級版)】★★★ ---
# ==============================================================================

def get_critical_segment_analysis_prompt():
    """
    【全新升級版】
    生成用於對【單一關鍵高強度教學片段】進行微觀分析的 Prompt。
    - 整合了宏觀功能分類的概念。
    - 新增了量化下滑幅度的要求。
    """
    
    # --- ★ 修改點 1: 引入宏觀功能分類的概念 ---
    # 這裡我們將您另一份程式碼中的 "教學狀態" 概念，轉化為 AI 在分析時可以參考的 "教學活動" 分類
    teaching_activity_definitions = """
**【教學活動參考分類】**
在你的分析中，請隨時參考以下教學活動的分類來理解上下文：
- **教師講解**: 老師主導的、連續的知識輸出，包括新知識傳授或舊知識複習。
- **學生練習/考試**: 老師指令後，學生進行獨立的、靜默的練習或測驗。
- **師生互動**: 針對教學內容的雙向問答、討論，或與課堂管理相關的交流。
- **閒聊/認知緩衝**: 與當前教學主題無直接關聯的個人故事、生活經驗分享或非正式交流。
"""

    # --- ★ 修改點 2: 全面升級 JSON 結構，增加 quantitative_impact ---
    json_structure = """
    {{
      "segment_start_time": "（此片段的開始時間）",
      "segment_end_time": "（此片段的結束時間）",
      "core_teaching_topic": "（總結這 7-12 分鐘內最核心的單一教學主題，例如：'現在完成式的用法解析'）",
      "attention_narrative": "（以敘事方式，詳細描述學生專注度在此片段中的變化軌跡。例如：'片段開始時，學生專注度高...約在 4 分 30 秒後...分心比例達到峰值...'）",
      "breaking_point_analysis": {{
        "time_of_decline": "（找出專注度開始【首次顯著下滑】的具體時間點，例如：'0:28:30'）",
        "teacher_quote_at_breaking_point": "（引用導致專注度下滑的【那一句關鍵話語】）",
        "analysis_of_cause": "（分析下滑的原因，例如：'教師在此處連續講解超過 5 分鐘未進行互動，且引入了一個複雜的語法例外規則，超出了部分學生的工作記憶負荷。'）",
        
        "quantitative_impact": {{
          "focus_metric_analyzed": "（你主要分析的是哪個指標的變化，必須是 'disengagement' 或 'task_oriented_focus'）",
          "value_before_decline_pct": （下滑前的指標數值，僅回傳數字，例如：35.5）,
          "value_after_decline_pct": （下滑後的指標數值，僅回傳數字，例如：55.0）,
          "change_delta_pct": （計算出的變化幅度，僅回傳數字，例如：19.5 或 -20.0）
        }}
      }},
      "segment_summary": "（對這個關鍵片段的教學成效做一個總結性評價。）"
    }}
    """
    
    # --- ★ 修改點 3: 強化 Prompt 指令，使其更明確、更專業 ---
    return f"""
你是一位頂尖的教育心理學家和數據分析師，專長是從課堂數據中進行微觀的因果分析。你的任務是分析一段被標記為【關鍵高強度教學】的課堂數據（長度約 7-12 分鐘）。

{teaching_activity_definitions}

**【你的核心任務】**
1.  **敘述專注度軌跡**: 描述學生專注度從高到低的完整變化過程。
2.  **定位「引爆點」(Breaking Point)**: 精確找出學生專注度開始**首次顯著下滑**的時間點、對應的教師話語，並**量化其影響**。
3.  **分析原因**: 結合認知負荷理論 (Cognitive Load Theory)，解釋為什麼在那個時間點學生的專注度會開始下滑。

**【★★★ V2.0 新增指令：量化引爆點的衝擊 ★★★】**
在 `breaking_point_analysis` 區塊中，你**必須**新增並填寫 `quantitative_impact` 子物件。
- 你需要從輸入數據的 `focus_distribution` 中，找出引爆點時間前後最能反映變化的指標（通常是 `disengagement` 的上升或 `task_oriented_focus` 的下降）。
- 你必須提供變化前的數值、變化後的數值，以及它們之間的差值（delta），所有數值都只要數字本身。

**你的輸入資料格式如下：**
一個JSON列表，代表一段連續的「高強度講解」片段。

**你的輸出格式要求：**
你必須嚴格遵循以下的JSON結構，並填寫所有欄位。你的回答必須是一個結構完整的 JSON 物件，絕對不能包含任何額外的文字、註解或 Markdown 標記。

{json_structure}
"""

def get_trigger_refinement_prompt():
    """
    【全新功能】生成用於對【單一合併後區塊】進行觸發語句精煉的 Prompt。
    """
    return """
你是一位頂尖的教學分析專家和資訊架構師。你的核心任務是從一段完整的教學逐字稿中，提取出 3-5 個最能代表該教學片段核心「活動」或「主題」的關鍵短語。

【你的核心任務】
分析使用者提供的【教學區塊逐字稿】，並以一個 JSON 列表的形式，返回 3-5 個關鍵判斷語句。

【規則】
1.  **高度精煉**: 每個語句都應該是簡潔的短語，最好在 10 個字以內。
2.  **動詞開頭**: 盡量使用動詞開頭，使其看起來像一個「行動」或「主題」，例如「解釋分詞用法」、「複習名詞子句」、「進行課堂問答」。
3.  **基於原文**: 提煉的短語必須忠於原文的核心內容。
4.  **JSON 列表輸出**: 你的回答必須是一個結構完整的 JSON 物件，根鍵為 "refined_quotes"，其值為一個字串列表。

【範例 1】
- **輸入逐字稿**: "好，那我們現在要來講這個重點了啊...我們來看名詞子句的部分...名詞子句具有名詞的性質...這才是本課的重點...我們在這個情況下當中，常常會搭配that去引導一個事件..."
- **輸出 JSON**:
  {
    "refined_quotes": [
      "複習名詞子句",
      "解釋句型結構",
      "說明 that 的用法"
    ]
  }

【範例 2】
- **輸入逐字稿**: "好，那我們來看第七題，來，一樣的題目，來，請大家思考一下...專心看，看到那個...甚至還有surprised、worried、angry...我們之前教的情緒分詞..."
- **輸出 JSON**:
  {
    "refined_quotes": [
      "引導思考第七題",
      "要求學生專心看",
      "講解情緒分詞"
    ]
  }
"""

async def refine_trigger_quotes_async(block: dict, semaphore: asyncio.Semaphore):
    """
    使用 AI 對單個合併後區塊的逐字稿進行分析，提煉出主題式的關鍵判斷語句。
    """
    async with semaphore:
        # 只對包含實質內容的「教師講解」和「師生互動」區塊進行精煉，節省成本
        if block["teaching_mode"] not in ["教師講解", "師生互動"] or len(block.get("full_transcript", "")) < 50:
            return # 對於閒聊、考試等模式，保持原始 trigger_quotes

        system_prompt = get_trigger_refinement_prompt()
        user_prompt = f"請根據以下【教學區塊逐字稿】，為我提煉出關鍵判斷語句：\n\n---\n{block['full_transcript']}\n---"
        
        try:
            response = await async_client.chat.completions.create(
                model=TEXT_DEPLOYMENT_NAME,
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                max_tokens=1024,
                temperature=0.0
            )
            result = json.loads(response.choices[0].message.content)
            refined_quotes = result.get("refined_quotes")
            
            # 如果 AI 成功返回了列表，就更新 block 中的 trigger_quotes
            if refined_quotes and isinstance(refined_quotes, list):
                block["trigger_quotes"] = refined_quotes
            
        except Exception as e:
            print(f"  - ⚠️ 在精煉 {block['start_time']} 區塊的 trigger_quotes 時發生錯誤: {e}")
            # 如果發生錯誤，就保持原始的 trigger_quotes 不變
    return # 函式直接修改傳入的 block 物件，無需返回值

async def analyze_critical_segment_with_ai_async(segment, index, total, semaphore, retries=3):
    """(v5.9) 調用 AI 對單個關鍵教學片段進行微觀分析"""
    if not segment: return None
    
    # 使用 async with semaphore 來自動管理並行數量
    async with semaphore:
        print(f"  - (許可已取得) 正在發起關鍵片段 {index + 1}/{total} 的微觀分析請求...")
        system_prompt = get_critical_segment_analysis_prompt()
        user_prompt = f"請對以下這段關鍵高強度教學片段進行微觀分析：\n\n{json.dumps(segment, ensure_ascii=False, indent=2)}"
        
        for attempt in range(retries):
            try:
                # 在請求之間加入一個微小的隨機延遲，進一步錯開請求峰值
                await asyncio.sleep(random.uniform(0.5, 1.5))
                
                response = await async_client.chat.completions.create(
                    model=TEXT_DEPLOYMENT_NAME,
                    response_format={"type": "json_object"},
                    messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                    max_tokens=4096, temperature=0.1,
                )
                return json.loads(response.choices[0].message.content)
            except Exception as e:
                print(f"  - ⚠️ 分析關鍵片段 {index + 1} 時發生錯誤 (嘗試 {attempt + 1}/{retries}): {e}")
                if attempt < retries - 1: await asyncio.sleep(5 * (attempt + 1)) # 增加重試等待
        
        print(f"  - ❌ 關鍵片段 {index + 1}/{total} 在所有重試後分析失敗。")
        return None

# ==============================================================================
# --- ★★★【新增區塊：AI 輔助的教學模式分類器】★★★ ---
# ==============================================================================
async def classify_interval_teaching_mode_async(speech_text: str, student_behavior_summary: dict, blackboard_snapshots: list, retries=3):
    """
    【v12.2 最終健壯版】
    使用 AI 根據單一時間區間內的綜合資訊，判斷教學模式，並【直接從逐字稿中擷取關鍵引文作為證據】。
    - 增強了 Prompt，強制 AI 在不確定時也需返回格式完整的預設值。
    - 增加了對 Markdown 格式污染的自動清洗功能。
    - 採用寬容的 .get() 解析，大幅提高成功率。
    - 修正了 API 回應的存取方式。
    """
    if not speech_text.strip():
        # --- 無語音時的邏輯保持不變 ---
        if any(b in student_behavior_summary for b in ["做筆記", "翻書"]) and blackboard_snapshots:
            return {"mode": "學生練習", "key_quote": "偵測到學生筆記/翻書行為且有新板書。"}
        if student_behavior_summary.get("趴睡", []) or student_behavior_summary.get("低頭(非學習)", []):
             return {"mode": "學生練習", "key_quote": "偵測到長時間靜默且有學生趴睡/低頭。"}
        return {"mode": "教學靜默", "key_quote": "無教師語音且無明顯學生學習任務。"}

    # --- 定義 JSON 輸出結構與範例 ---
    json_structure_example = """
{
  "mode": "（從六個類別中選擇一個）",
  "key_quote": "（【直接從逐字稿中擷取】最能證明此分類的一小段關鍵原文，例如：'我們在教室休息五分鐘' 或 '來講解一些蠻重要的文法'）"
}
"""

    rules_description = "\n".join([f"- **{state}**: {desc.strip()}" for state, desc in CLASSROOM_STATES_DEFINITIONS.items()])

    # --- 全面升級 Prompt 指令 ---
    system_prompt = f"""
你是一位精準的課堂片段分析師。你的任務是分析一段30秒的課堂逐字稿，並完成兩項任務：
1.  **分類 (Classify)**: 從提供的六個教學模式中，選擇最符合的一個。
2.  **取證 (Extract Evidence)**: 從逐字稿原文中，【直接擷取】一句話或一個短語作為你分類判斷的【直接證據】。

**【教學模式分類規則】**
{rules_description}

**【★★★ 核心指令：取證而非總結 & 絕不回傳空值 ★★★】**
- 你的回答中 `key_quote` 欄位的值，【必須是】從使用者提供的逐字稿中複製貼上的原文片段。
- **絕對禁止**自己創造、總結或解釋原因。例如，如果逐字稿是「我們休息五分鐘」，你的 `key_quote` 就應該是「休息五分鐘」，而不是「老師宣布休息」。
- 如果逐字稿很長，只擷取最關鍵、最能代表該模式啟動的句子。
- **【高優先級規則】**: 即使你對分類沒有把握，也必須從提供的六個模式中猜測一個最可能的，並從原文中提取一句話作為 `key_quote`。**絕對禁止**返回空的或缺少鍵的 JSON 物件。如果逐字稿為空，`key_quote` 可以是 "無教師語音"。


**【輸出格式要求】**
你的回答必須是結構完整的 JSON 物件，包含 "mode" 和 "key_quote" 兩個鍵，絕對不能包含任何額外的文字或註解。範例如下：
{json_structure_example}
"""
    
    user_prompt = f"""
分析以下 30 秒課堂片段：

- **老師逐字稿**: "{speech_text}"
- **此期間的學生主要行為**: {list(student_behavior_summary.keys())}
- **此期間是否有新板書**: {"有" if blackboard_snapshots else "無"}
"""

    for attempt in range(retries):
        try:
            response = await async_client.chat.completions.create(
                model=TEXT_DEPLOYMENT_NAME,
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                max_tokens=200, 
                temperature=0.0
            )
            
            # 修正 1: 正確存取 API 回應內容
            content = response.choices[0].message.content
            
            # 修正 2: 清洗可能存在的 Markdown 格式
            json_match = re.search(r'```json\s*([\s\S]*?)\s*```', content, re.DOTALL)
            if json_match:
                content = json_match.group(1)

            parsed_json = json.loads(content)
            
            # 修正 3: 使用 .get() 進行寬容解析
            mode = parsed_json.get("mode")
            key_quote = parsed_json.get("key_quote")

            # 只要 mode 存在且合法，就認為這次解析是成功的
            if mode in CLASSROOM_STATES_DEFINITIONS:
                # 即使 key_quote 是 None，也給它一個預設值
                return {"mode": mode, "key_quote": key_quote if key_quote is not None else "AI 未提供引文"}
            else:
                # 如果連 mode 都沒有或不合法，才拋出錯誤觸發重試
                raise ValueError(f"AI returned a JSON without a valid 'mode'. Got: {parsed_json}")
                
        except Exception as e:
            print(f"  - ⚠️ 微觀模式分類（證據擷取版）失敗 (嘗試 {attempt + 1}/{retries}): {e}")
            if attempt < retries - 1:
                await asyncio.sleep(2 ** attempt)
    
    # 如果所有重試都失敗，提供一個備用返回值
    return {"mode": "師生互動" if speech_text else "教學靜默", "key_quote": "AI 分析失敗，使用備用規則判斷。"}

def aggregate_behavior_data_for_chart(detailed_timeline, interval_minutes=10):
    """
    在後端預先聚合行為數據，以供前端圖表直接使用。
    """
    if not detailed_timeline:
        return {}

    interval_seconds = interval_minutes * 60
    aggregated = defaultdict(lambda: {"sums": defaultdict(float), "count": 0})

    for d in detailed_timeline:
        if d.get("focus_distribution"):
            chunk_index = d['seconds'] // interval_seconds
            dist = d["focus_distribution"]
            aggregated[chunk_index]["sums"]["active"] += dist.get("active_behavioral_engagement", 0)
            aggregated[chunk_index]["sums"]["passive"] += dist.get("passive_behavioral_engagement", 0)
            aggregated[chunk_index]["sums"]["disengaged"] += dist.get("behavioral_disengagement", 0)
            aggregated[chunk_index]["sums"]["ambiguous"] += dist.get("ambiguous_behavior", 0)
            aggregated[chunk_index]["count"] += 1
            
    chart_data = {"labels": [], "datasets": defaultdict(list)}
    for index in sorted(aggregated.keys()):
        chunk = aggregated[index]
        count = chunk["count"]
        if count > 0:
            chart_data["labels"].append((index + 1) * interval_minutes)
            chart_data["datasets"]["active"].append(round(chunk["sums"]["active"] / count, 1))
            chart_data["datasets"]["passive"].append(round(chunk["sums"]["passive"] / count, 1))
            chart_data["datasets"]["disengaged"].append(round(chunk["sums"]["disengaged"] / count, 1))
            chart_data["datasets"]["ambiguous"].append(round(chunk["sums"]["ambiguous"] / count, 1))
            
    return chart_data

def identify_teaching_cycles(master_timeline):
    """
    (v5.12 修正版) 識別廣義的教學週期 (高認知負荷 -> 低認知負荷)。
    """
    print("步驟 5.8/8: 識別並量化教學週期 (採用廣義模式)...")
    if not master_timeline:
        return {"cycles": [], "summary": {}}

    cycles = []
    current_cycle = None
    in_high_intensity_phase = False

    # 定義高低負荷的狀態集合
    HIGH_LOAD_MODES = {"教師講解", "師生互動"}
    LOW_LOAD_MODES = {"學生練習", "學生考試", "閒聊", "下課休息"}

    for i, interval in enumerate(master_timeline):
        mode = interval.get("teaching_mode")
        metrics = interval.get("cognitive_load_metrics")
        
        # 判斷當前區間是否為高負荷
        is_high_load = False
        if mode in HIGH_LOAD_MODES and metrics:
            intensity = metrics.get("intensity_level")
            if intensity in ["高強度", "中強度"]:
                is_high_load = True

        # 從非高負荷進入高負荷，標記一個新週期的開始
        if is_high_load and not in_high_intensity_phase:
            in_high_intensity_phase = True
            if current_cycle: # 結束上一個週期
                end_sec = parse_timestamp(master_timeline[i-1]["end_time"])
                if current_cycle.get("start_seconds") is not None and end_sec is not None:
                    current_cycle["end_time"] = master_timeline[i-1]["end_time"]
                    current_cycle["total_duration_seconds"] = end_sec - current_cycle["start_seconds"]
                    cycles.append(current_cycle)

            # 開始新週期
            current_cycle = {
                "cycle_index": len(cycles) + 1,
                "start_time": interval["start_time"],
                "start_seconds": interval["seconds"],
                "end_time": None,
                "high_intensity_duration": 0,
                "low_intensity_duration": 0
            }
        
        if current_cycle:
            if is_high_load:
                current_cycle["high_intensity_duration"] += TIME_INTERVAL_SECONDS
            elif mode in LOW_LOAD_MODES:
                current_cycle["low_intensity_duration"] += TIME_INTERVAL_SECONDS
                # 如果從高負荷轉為低負荷，標記高負荷階段結束
                if in_high_intensity_phase:
                    in_high_intensity_phase = False
    
    # 處理最後一個週期
    if current_cycle:
        end_sec = master_timeline[-1]['seconds'] + TIME_INTERVAL_SECONDS
        if current_cycle.get("start_seconds") is not None:
            current_cycle["end_time"] = str(datetime.timedelta(seconds=end_sec))
            current_cycle["total_duration_seconds"] = end_sec - current_cycle["start_seconds"]
            cycles.append(current_cycle)

    # ... (後續的摘要計算不變) ...
    total_cycle_duration = sum(c.get("total_duration_seconds", 0) for c in cycles)
    num_cycles = len(cycles)
    average_cycle_duration = total_cycle_duration / num_cycles if num_cycles > 0 else 0

    summary = {
        "number_of_cycles": num_cycles,
        "average_cycle_duration_seconds": round(average_cycle_duration),
        "average_cycle_duration_minutes": round(average_cycle_duration / 60, 1)
    }
    
    print(f"✅ 成功識別出 {num_cycles} 個教學週期，平均週期時長約 {summary['average_cycle_duration_minutes']} 分鐘。")
    return {"cycles": cycles, "summary": summary}

def find_state_transition_triggers(master_timeline: list) -> list:
    """
    (全新功能) 遍歷完整時間軸，找出教學模式轉變的關鍵時間點，並提取觸發該轉變的關鍵句。
    """
    print("步驟 5.8/8: 分析教學模式轉換的觸發關鍵句...")
    if not master_timeline or len(master_timeline) < 2:
        return []

    triggers = []
    # 從第二個時間區間開始遍歷，以便和前一個進行比較
    for i in range(1, len(master_timeline)):
        prev_interval = master_timeline[i-1]
        current_interval = master_timeline[i]

        prev_mode = prev_interval.get("teaching_mode")
        current_mode = current_interval.get("teaching_mode")

        # 當教學模式發生變化時，記錄下來
        if prev_mode != current_mode:
            
            # 關鍵句是前一個區間老師說的話，因為是這句話導致了下一個區間的模式轉變
            trigger_quote = prev_interval.get("teacher_speech", "").strip()
            
            # 為了提供更豐富的上下文，我們往前多找一句
            context_before = master_timeline[i-2].get("teacher_speech", "").strip() if i > 1 else ""

            # 只記錄那些真正由老師話語觸發的轉變
            if trigger_quote:
                triggers.append({
                    "transition_time": current_interval["start_time"],
                    "from_mode": prev_mode,
                    "to_mode": current_mode,
                    "trigger_quote": trigger_quote,
                    "context_before": context_before
                })

    print(f"✅ 成功識別出 {len(triggers)} 個教學模式轉換觸發點。")
    return triggers

def extract_trigger_keywords_for_states(merged_timeline: list) -> list:
    """
    (全新功能 v2.0) 在合併後的時間軸上，為特定的教學模式找出其內部的觸發關鍵字/句。
    """
    print("  -> 正在提取各教學模式的內部觸發關鍵句...")
    
    # 定義我們關心的模式及其對應的觸發關鍵詞
    TARGET_MODES_KEYWORDS = {
        # --- 核心教學模式 ---
        "教師講解": [
            # 引導與聚焦
            "來看這邊", "專心看", "注意聽", "首先", "再來我們看", 
            "這個地方是", "這個觀念", "解釋一下",
            # 強調重要性
            "重點是", "關鍵在", "一定要記住",
            # 結構性說明
            "句型結構", "分詞用法", "名詞子句"
        ],
        "師生互動": [
            # 開放性提問
            "有沒有問題", "為什麼", "什麼意思", "懂我意思嗎", "懂了嗎",
            "有沒有發現", "有沒有感覺",
            # 邀請參與
            "有沒有人知道", "大家覺得", "誰可以告訴我",
            # 檢查與確認
            "對不對", "是不是", "可以嗎"
        ],

        # --- 任務導向模式 ---
        "學生考試": [
            "考試", "前測", "測驗", "發考卷", "檢討考卷"
        ],
        "學生練習": [
            # 明確指令
            "練習一下", "寫一下", "試試看", "動筆", "給大家幾分鐘", 
            "寫上去", "畫線", "圈起來", "麻煩您",
            # 引導操作
            "給大家思考一下", "先幫我看到"
        ],

        # --- 非教學核心模式 ---
        "下課休息": [
            "下課", "休息一下", "休息"
        ],
        "閒聊": [
            # 故事性開頭
            "我跟你講", "我當年", "我個人", "講個故事", "分享一下",
            # 轉移話題
            "對了", "話說回來"
            # (閒聊模式更多依賴語意，關鍵字較難窮舉，以上為常見信號詞)
        ]
    }
    
    analysis_results = []

    for block in merged_timeline:
        mode = block["teaching_mode"]
        if mode in TARGET_MODES_KEYWORDS:
            transcript = block["full_transcript"]
            keywords_found = []
            
            for keyword in TARGET_MODES_KEYWORDS[mode]:
                if keyword in transcript:
                    # 尋找包含關鍵字的完整句子
                    # 使用正則表達式，匹配包含關鍵字且以句號、問號或空格結尾的句子
                    match = re.search(f"([^。？\s]*{re.escape(keyword)}[^。？\s]*)", transcript)
                    if match and match.group(1) not in keywords_found:
                        keywords_found.append(match.group(1).strip())
            
            if keywords_found:
                analysis_results.append({
                    "mode": mode,
                    "start_time": block["start_time"],
                    "end_time": block["end_time"],
                    "trigger_quotes": keywords_found
                })

    print(f"✅ 成功為 {len(analysis_results)} 個關鍵模式區塊提取了觸發句。")
    return analysis_results

async def correct_single_transcript_chunk(chunk, index, total, context_before, context_after, professional_keywords, contextual_glossary, retries=3):
    """(v7.0 語意增強版) 處理單個逐字稿區塊的校正，並引入上下文知識庫。"""
    
    global PROFESSIONAL_KEYWORDS
    global CONTEXTUAL_GLOSSARY

    professional_terms = ", ".join(PROFESSIONAL_KEYWORDS)
    
    # --- ★★★【核心修改 1：建構知識庫提示】★★★ ---
    glossary_prompt_part = """
**【上下文與專有名詞知識庫】**
在校正時，你必須優先參考以下詞彙對照表來修正常見的同音異義詞錯誤和專有名詞：
- 如果原始文本出現右側的詞 (常見錯誤)，且語意符合左側情境，請大膽將其校正為左側的詞 (正確詞彙)。
"""
    for correct_term, wrong_terms in CONTEXTUAL_GLOSSARY.items():
        if wrong_terms:
            glossary_prompt_part += f"- **{correct_term}** (正確) <--- {', '.join(wrong_terms)} (常見錯誤)\n"
        else:
            glossary_prompt_part += f"- **{correct_term}** (這是一個特殊的專有名詞或黑話，請保留它，不要修改)\n"

    # --- ★★★【核心修改 2：全面升級 System Prompt】★★★ ---
    system_prompt = f"""
你是一位頂尖的中文逐字稿校對專家，專門處理【台灣補習班老師】的上課內容。你不僅懂文法，更懂教育情境和學生的口語文化。

【情境資訊】
這是一堂快節奏的高中英文文法與歷史課。老師語速快，常有口語化的表達、俚語和與學生的玩笑。原始稿件由 ASR 模型生成，可能包含大量錯誤。

{glossary_prompt_part}

**【核心校正規則】**
1.  **語意連貫性優先**: 你的首要任務是將【待校正區塊】的內容修復成**通順且符合邏輯**的中文句子。請務必參考【前文】和【後文】來理解語意，並將破碎的短句合併成完整的語意單元。
2.  **同音異義詞校正**: 這是你的關鍵任務。例如，如果老師在討論學科，但文本出現「初學」，你必須將其修正為「數學」。
3.  **處理無意義的幻覺**: 如果遇到像 "婆婆 check j sock Natalk" 這樣完全無法理解、由噪音產生的亂碼，請直接將其**替換為一個表示無法辨識的標籤**，例如 `"[無法辨識的噪音]"或"[多人同時說話]" `。**絕對不要**試圖去翻譯或解釋這些亂碼。
4.  **保持口語風格**: 只修正錯誤，**不要**將老師口語化的內容（例如 "對啊"、"然後呢"）過度書面化。保留老師的教學風格。
5.  **結構保持一致**: 你的輸出必須是與輸入的【待校正區塊】完全相同的 JSON 列表結構，你只能修改 "text" 欄位的文字。

【你的輸出要求】
請你只返回校正後的【待校正區塊】的 JSON 內容，不要包含任何額外的文字、註解或 Markdown 標記。
"""
    
    user_prompt = f"""
請根據你的專業知識和校正規則，修復以下標示為【待校正區塊】的逐字稿 JSON 資料片段。

---
【前文參考】
{json.dumps(context_before, ensure_ascii=False, indent=2)}
---
【★★★ 待校正區塊 ★★★】
{json.dumps(chunk, ensure_ascii=False, indent=2)}
---
【後文參考】
{json.dumps(context_after, ensure_ascii=False, indent=2)}
---
"""
    for attempt in range(retries):
        corrected_content = "" # 先初始化變數
        try:
            response = await async_client.chat.completions.create(
                model=TEXT_DEPLOYMENT_NAME,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                max_tokens=8192,  # <--- 增加此處的數值
                temperature=0.1
            )
            corrected_content = response.choices[0].message.content
            json_match = re.search(r'```json\s*([\s\S]*?)\s*```', corrected_content, re.DOTALL)
            if json_match:
                corrected_content = json_match.group(1)
            
            corrected_chunk = json.loads(corrected_content)
            print(f"    - ✅ 逐字稿區塊 {index + 1}/{total} 校正成功。")
            return corrected_chunk
            
        except RateLimitError as e:
            wait_time = (2 ** attempt) + random.random()
            print(f"    - ⚠️ 區塊 {index + 1} 觸發速率限制 (嘗試 {attempt + 1}/{retries})，將在 {wait_time:.1f} 秒後重試...")
            await asyncio.sleep(wait_time)
            
        except Exception as e:
            # ★★★【修改點 2：加入詳細的錯誤日誌】★★★
            # 當解析失敗時，印出完整的錯誤訊息和 AI 返回的原始文字，以便偵錯
            print(f"    - ⚠️ 校正區塊 {index + 1} 時發生錯誤 (嘗試 {attempt + 1}/{retries}): {e}")
            print(f"--- AI 返回的原始內容 (區塊 {index + 1}) ---")
            print(corrected_content)
            print("------------------------------------")
            # ★★★【修改結束】★★★

            if attempt < retries - 1:
                await asyncio.sleep(5)
            else:
                print(f"    - ❌ 校正區塊 {index + 1} 失敗，將使用此區塊的原始數據。")
                return chunk

async def correct_transcript_with_ai_async(transcript_data, chunk_size=30):
    """(v6.0 增強版) 使用並行請求校正逐字稿，並為每個區塊提供上下文窗口。"""
    
    # --- ★★★【核心修正點】★★★ ---
    # 在這裡宣告我們要使用全域變數，這樣在下面的 tasks.append 中才能找到它們
    global PROFESSIONAL_KEYWORDS
    global CONTEXTUAL_GLOSSARY
    # --- ★★★【修改結束】★★★ ---

    print("步驟 3.5/8: 調用 AI 進行逐字稿校正 (採用【上下文增強模式】)...")
    
    if not transcript_data:
        print("  - 逐字稿為空，跳過校正。")
        return []
        
    chunks = [transcript_data[i:i + chunk_size] for i in range(0, len(transcript_data), chunk_size)]
    print(f"  - 逐字稿已分為 {len(chunks)} 個區塊，將並行處理。")

    tasks = []
    for i, chunk in enumerate(chunks):
        context_before = chunks[i-1][-2:] if i > 0 else []
        context_after = chunks[i+1][:2] if i < len(chunks) - 1 else []
        
        # 現在這一行可以正常執行了
        tasks.append(correct_single_transcript_chunk(chunk, i, len(chunks), context_before, context_after, PROFESSIONAL_KEYWORDS, CONTEXTUAL_GLOSSARY))
    
    results = await asyncio.gather(*tasks)
    
    corrected_transcript = []
    for result in results:
        if result:
            corrected_transcript.extend(result)
            
    print("✅ AI 逐字稿校正完成。")
    return corrected_transcript

def get_chunk_analysis_prompt(chunk_start_time, chunk_end_time):
    """
    【v8.0 學術化最終版】生成用於分析【單一區塊】的 Prompt。
    - 採用 Fredricks et al. (2004) 和 Chi & Wylie (2014) 的理論框架。
    - 將學生狀態分為四個嚴謹的維度：主動行為參與、被動行為參與、行為分心、模糊行為。
    - 更新 JSON 輸出結構以匹配新的學術術語。
    """


    # --- 核心部分 2：AI 必須嚴格遵循的 JSON 輸出結構 ---
    json_structure = f"""
{{
  "start_time": "{chunk_start_time}",
  "end_time": "{chunk_end_time}",
  "student_focus_analysis": {{
    "overall_level": "（基於此區塊內行為參與狀態的整體判斷：高 / 中高 / 中 / 低）",
    "average_distribution": {{
      "task_oriented_focus": "（計算並填入 active_behavioral_engagement 的平均百分比，僅數字）",
      "receptive_engagement": "（計算並填入 passive_behavioral_engagement 的平均百分比，僅數字）",
      "disengagement": "（計算並填入 behavioral_disengagement 的平均百分比，僅數字）"
    }}
  }}
}}
"""
    
    # --- 核心修改 2：大幅簡化 AI 的系統指令 ---
    return f"""
你是一位專注的數據處理助理。你的唯一任務是分析一段課堂數據區塊，並計算出其中學生不同行為參與狀態的平均百分比。

**【你的核心任務】**
1.  **計算平均值**: 根據輸入數據中每個時間點的 `focus_distribution`，計算出 `active_behavioral_engagement`、`passive_behavioral_engagement` 和 `behavioral_disengagement` 在整個區塊內的平均值。
2.  **綜合評級**: 根據計算出的平均值，給出一個整體的參與度評級 (`overall_level`)。
3.  **嚴格格式輸出**: 將結果填入指定的 JSON 結構中。

**你的輸入資料格式如下：**
一個JSON列表，每個物件代表一個30秒的時間區間，包含 `focus_distribution` 物件。

**你的輸出格式要求：**
你必須嚴格遵循以下的JSON結構，並填寫所有欄位。你的回答必須是一個結構完整的 JSON 物件，絕對不能包含任何額外的文字、註解或 Markdown 標記。

{json_structure}
"""

def get_summary_prompt(pre_summary_str: str):
    return f"""
你是一位擁有豐富現場經驗的**「台灣國中英文補習班老師」**。你的溝通對象是一位**經驗豐富的老師**。
你的任務是寫一封給老師的**「觀課回饋信」**。目標是透過數據提供靈感，而不是進行學術診斷。

**【★★★ 絕對關鍵指令：去術語化 (No Jargon) ★★★】**
1.  **禁止使用學術名詞**：請絕對避免使用「認知負荷」、「後設認知」、「鷹架」、「圖式」、「工作記憶」等教育心理學術語。
2.  **使用白話文**：請用生活化的比喻。例如：
    *   不要說「降低認知負荷」，請說「讓學生的大腦休息一下」。
    *   不要說「建立行為制約」，請說「建立默契」或「儀式感」。
    *   不要說「提升後設認知」，請說「幫助學生想一想」。
3.  **語氣溫暖尊重**：請站在「幫老師省力」、「讓上課更開心」的角度給予建議，而不是「為了提升績效」。

**【輸出格式要求】**
請嚴格遵循以下的 Markdown 格式，建議部分請保持單純的條列式（第一點、第二點），並清楚分出「具體作法」與「好處」。

---
### **您的課堂洞察報告：核心發現與教學靈感**

老師您好，
我是您的 AI 觀課夥伴。這份報告整理自課堂的客觀數據，希望能成為一面鏡子，映照出您課堂中許多精彩的細節，並提供一些不同的視角供您參考。
---
#### **📈 發現一：[填寫第一個核心發現的標題]**
[詳細分析文字...請描述數據呈現的現象，記得使用白話文。]

> **💡 給您的教學反思：**
> *   [運用開放式問題，引導老師思考]
> *   [運用開放式問題，引導老師思考]
---
#### **📉 發現二：[填寫第二個核心發現的標題]**
[詳細分析文字...]

> **💡 給您的教學反思：**
> *   [引導反思問題]
> *   [引導反思問題]
---
#### **🎯 發現三：[填寫第三個核心發現的標題]**
[詳細分析文字...]

> **💡 給您的教學反思：**
> *   [引導反思問題]
> *   [引導反思問題]
---
### **未來課堂的行動靈感**
基於數據顯示的契機，我們為您整理了兩個或許能讓課堂更省力、更高效的嘗試方向：

1.  **[靈感一標題：請用動詞開頭]**
    *   **具體作法**：[請提供一個簡單、可立即執行的動作。例如：「試著在講解10分鐘後，插入一個讓學生舉手的小活動。」]
    *   **好處**：[請強調對老師的好處。例如：「這能讓您稍微喝口水，也能讓學生重新醒過來。」]

2.  **[靈感二標題：請用動詞開頭]**
    *   **具體作法**：[請提供簡單明確的動作。]
    *   **好處**：[請強調對老師的好處。]

**總結來說**，[撰寫一段溫暖的結語，肯定老師的付出。]

---

**【以下是課堂預摘要 (請忽略其中關於下課、考試的數據，專注於教學互動)】**
{pre_summary_str}
"""

async def analyze_chunk_with_ai_async(chunk, index, total, semaphore, retries=3):
    """
    (v5.4 非同步版本 - ★★★ 速度優化版 ★★★) 
    調用 Azure OpenAI 分析單個區塊。
    - 使用 Semaphore 控制最大並行任務數，避免API速率限制。
    - 增強了重試的等待邏輯。
    """
    if not chunk:
        print(f"  - ⚠️ 區塊 {index + 1}/{total} 為空，跳過分析。")
        return None

    # 使用 async with semaphore 來自動管理並行數量。
    # 在執行這段程式碼之前，它會等待 semaphore 發出許可。
    async with semaphore:
        print(f"  - (許可已取得) 正在發起區塊 {index + 1}/{total} 的分析請求...")
        
        chunk_start_time = chunk[0].get('start_time')
        chunk_end_time = chunk[-1].get('end_time')
        
        # get_chunk_analysis_prompt 函式本身是同步的，無需修改
        system_prompt = get_chunk_analysis_prompt(chunk_start_time, chunk_end_time)
        user_prompt = f"請分析以下課堂數據區塊：\n\n{json.dumps(chunk, ensure_ascii=False, indent=2)}"
        
        raw_content = ""

        for attempt in range(retries):
            try:
                # 在請求之間加入一個微小的隨機延遲，進一步錯開請求峰值，降低速率限制風險
                await asyncio.sleep(random.uniform(0.5, 1.5))
                
                response = await async_client.chat.completions.create(
                    model=TEXT_DEPLOYMENT_NAME,
                    response_format={"type": "json_object"},
                    messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                    max_tokens=8192,
                    temperature=0.1,
                )
                raw_content = response.choices[0].message.content
                return json.loads(raw_content)

            except json.JSONDecodeError as json_err:
                print(f"  - ⚠️ 區塊 {index + 1} 的 AI 返回 JSON 格式有誤 ({json_err})。正在嘗試自動修復...")
                # 建立一個用於修復 JSON 的簡單 Prompt
                fix_prompt = f"""你是一個 JSON 格式修復工具。你的任務是接收一段可能損壞或不完整的 JSON 文字，並盡力將其修復成一個語法正確的 JSON 物件。不要添加任何評論或解釋，只需返回修復後的 JSON 內容。
                
                這是損壞的文字：
                {raw_content}
                """
                try:
                    fix_response = await async_client.chat.completions.create(
                        model=TEXT_DEPLOYMENT_NAME,
                        response_format={"type": "json_object"},
                        messages=[{"role": "system", "content": "你是一個JSON格式修復工具。"}, {"role": "user", "content": fix_prompt}],
                        max_tokens=8192, temperature=0.0,
                    )
                    fixed_content = fix_response.choices[0].message.content
                    print(f"  - ✅ 區塊 {index + 1} JSON 已自動修復。")
                    return json.loads(fixed_content)
                except Exception as fix_e:
                    print(f"  - ❌ 區塊 {index + 1} 自動修復失敗: {fix_e}")
                    # 如果修復也失敗，直接進入下一次重試或結束
                    if attempt >= retries - 1:
                        break 
            
            except RateLimitError as e:
                # 增加重試的基礎等待時間，並加入隨機性
                wait_time = (5 * (attempt + 1)) + random.random() 
                print(f"  - ⚠️ 區塊 {index + 1} 觸發速率限制 (嘗試 {attempt + 1}/{retries})，將在 {wait_time:.1f} 秒後重試...")
                await asyncio.sleep(wait_time)
            
            except Exception as e:
                print(f"❌ 分析區塊 {index + 1} 時發生未知錯誤: {e}")
                if attempt < retries - 1:
                    # 採用指數退避策略增加等待時間
                    await asyncio.sleep(5 * (attempt + 1))
        
    print(f"  - ❌ 區塊 {index + 1}/{total} 在所有重試後分析失敗。")
    return None # 返回 None 表示失敗

async def create_pre_summary_for_final_analysis_async(
    teaching_mode_summary: dict,
    behavior_chart_data: dict,
    keyword_timestamps: list,
    retries=3
):
    """
    【v2.0 精簡報告版】
    將核心的圖表數據和關鍵詞數據轉化為文字摘要，以供最終分析使用。
    """
    print("步驟 6.8/8: 執行中間步驟 - 將核心數據轉換為預摘要...")

    # --- 1. 將核心數據轉換為易於閱讀的文字格式 ---
    summary_input_text = "### 課堂核心數據摘要 ###\n\n"
    
    # 摘要教學模式分佈
    summary_input_text += "#### 教學模式時間佔比 ####\n"
    for mode, data in teaching_mode_summary.items():
        summary_input_text += f"- {mode}: 佔比 {data['percentage_of_class']}%，共持續 {data['total_duration_minutes']} 分鐘。\n"
    
    # 摘要行為參與趨勢
    summary_input_text += "\n#### 學生行為參與趨勢（每10分鐘平均值） ####\n"
    labels = behavior_chart_data.get("labels", [])
    active_data = behavior_chart_data.get("datasets", {}).get("active", [])
    disengaged_data = behavior_chart_data.get("datasets", {}).get("disengaged", [])
    
    for i, label in enumerate(labels):
        if i < len(active_data) and i < len(disengaged_data):
            summary_input_text += f"- 在第 {label} 分鐘時：主動參與平均為 {active_data[i]}%，行為分心平均為 {disengaged_data[i]}%。\n"
    
    # 摘要關鍵詞使用情況
    summary_input_text += "\n#### 教師使用的關鍵指令詞 ####\n"
    keyword_groups = defaultdict(list)
    for item in keyword_timestamps:
        keyword_groups[item['keyword']].append(item['time'])
    for keyword, times in keyword_groups.items():
        summary_input_text += f"- 關鍵詞 \"{keyword}\" 在 {', '.join(times)} 等時間點被使用。\n"

    # --- 2. 設計 Prompt，要求 AI 進行提煉 (保持不變) ---
    system_prompt = """
你是一位高效的數據分析助理。你的任務是接收一份詳細的、分段的課堂分析報告，並將其提煉成一份高度濃縮的摘要。
你的摘要必須專注於以下幾點：
1.  **宏觀趨勢**: 整堂課的專注度起伏是否有一個大致的模式？
2.  **關鍵模式**: 總結有哪些重複出現的、能有效提升專注度的教學行為（助推器）？又有哪些總是導致分心的行為（陷阱）？
3.  **因果關係**: 找出教學節奏（例如，高強度講解後接著故事分享）與學生反應之間最明顯的幾組關聯。
4.  **具體證據**: 引用一兩個最典型的教師話語作為例子。

你的輸出必須是一段精煉的、條理清晰的文字。不要包含任何 JSON 或 Markdown 格式。
"""
    user_prompt = f"請根據以下詳細報告，為我生成一份濃縮的摘要：\n\n{summary_input_text}"

    # --- 3. 調用 AI (保持不變) ---
    # ... (此部分程式碼邏輯與您現有的版本相同，無需修改) ...
    for attempt in range(retries):
        try:
            response = await async_client.chat.completions.create(
                model=TEXT_DEPLOYMENT_NAME,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                max_tokens=2048,
                temperature=0.1
            )
            summary = response.choices[0].message.content
            print("✅ 成功生成預摘要。")
            return summary
        except RateLimitError as e:
            wait_time = 10 * (attempt + 1)
            print(f"  - ⚠️ 預摘要時觸發速率限制 (嘗試 {attempt + 1}/{retries})，將在 {wait_time} 秒後重試...")
            await asyncio.sleep(wait_time)
        except Exception as e:
            print(f"❌ 生成預摘要時發生錯誤: {e}")
            if attempt < retries - 1:
                await asyncio.sleep(5)
            else:
                return "錯誤：無法生成預摘要。"

    return None

async def summarize_chunks_with_ai_async(pre_summary_str: str, retries=3):
    """
    (v10.0 精簡報告版) 
    接收預摘要，並調用 AI 生成最終的【文字版】核心發現與建議。
    """
    print("步驟 7/8: 調用 AI 生成最終的文字版核心發現與教學建議...")
    
    # get_summary_prompt 現在會返回一個要求 Markdown 文本輸出的 Prompt
    system_prompt = get_summary_prompt(pre_summary_str)
    
    for attempt in range(retries):
        try:
            start_time = time.time()
            response = await async_client.chat.completions.create(
                model=TEXT_DEPLOYMENT_NAME,
                # 注意：我們不再需要 response_format={"type": "json_object"}
                messages=[{"role": "system", "content": system_prompt}],
                max_tokens=4096,  # 文字報告通常不需要 8192 token
                temperature=0.2
            )
            end_time = time.time()
            
            # 直接獲取並返回 AI 生成的文字內容
            report_text = response.choices[0].message.content.strip()
            
            # 增加一個健全性檢查，確保返回的不是空字串
            if report_text:
                print(f"✅ AI 文字報告生成完成，耗時 {end_time - start_time:.2f} 秒。")
                return report_text
            else:
                print(f"  - ⚠️ AI 返回了空的報告內容 (嘗試 {attempt + 1}/{retries})，將重試...")
                # 觸發重試
                raise ValueError("AI returned empty content")

        except RateLimitError as e:
            wait_time = 60 * (attempt + 1) + 1 
            print(f"  - ⚠️ 最終分析觸發速率限制 (嘗試 {attempt + 1}/{retries})，將在 {wait_time} 秒後重試...")
            await asyncio.sleep(wait_time)
            
        except Exception as e:
            print(f"❌ 生成最終文字報告時發生錯誤: {e}")
            if attempt < retries - 1:
                await asyncio.sleep(5 * (attempt + 1))
            else:
                # 如果所有重試都失敗，返回 None
                print("❌ 最終文字報告生成失敗，已達最大重試次數。")
                return None
            
# ==============================================================================
# --- 主執行流程 (v5.10 - 整合語速趨勢分析) ---
# ==============================================================================
async def main():
    start_total_time = time.time()

    semaphore = asyncio.Semaphore(MAX_CONCURRENT_AI_TASKS)
    
    # --- 階段一 & 二：數據載入與校正 (不變) ---
    print("--- Classroom Analysis Engine v9.0 (Minimalist Report Edition) ---")
    student_behaviors, total_seconds = load_student_data(STUDENT_DATA_DIRECTORY, TARGET_SESSION_TIME)
    if student_behaviors is None: 
        exit()
    raw_transcript_for_correction = load_teacher_transcript(TEACHER_TRANSCRIPT_PATH)
    blackboard_images = load_blackboard_images(BLACKBOARD_IMAGES_DIRECTORY)
    preprocessed_transcript = preprocess_transcript(raw_transcript_for_correction)
    corrected_transcript = await correct_transcript_with_ai_async(preprocessed_transcript)

    # --- 階段三 & 四：建立整合時間軸並進行微觀分類 ---
    master_timeline = await create_master_timeline(
        student_behaviors, 
        corrected_transcript,
        blackboard_images, 
        total_seconds, 
        TIME_INTERVAL_SECONDS,
        state_timeline=None 
    )

    # --- 階段五：數據後處理與核心數據提取 ---
    # 執行一系列後處理規則來校準教學模式
    master_timeline = post_process_silent_and_noisy_intervals(master_timeline)
    master_timeline = force_consolidate_break_time(master_timeline)
    
    # 提取前端需要的核心數據
    # 1. 教學模式分佈圖的數據
    teaching_mode_summary = calculate_mode_durations(master_timeline, total_seconds, TIME_INTERVAL_SECONDS)
    
    # 2. 關鍵詞彙實例分析的數據
    keyword_timestamps = find_important_keywords(master_timeline)
    
    # 3. 雙層互動時間軸的數據 (需要先合併，再計算WPM)
    merged_timeline_raw = merge_and_cleanup_timeline(master_timeline)
    merged_timeline_with_wpm = calculate_block_level_wpm(merged_timeline_raw, master_timeline)

    print("\n步驟 5.99/8: 開始對關鍵教學區塊的 Trigger Quotes 進行 AI 二次精煉...")
    refinement_tasks = [refine_trigger_quotes_async(block, semaphore) for block in merged_timeline_with_wpm]
    await asyncio.gather(*refinement_tasks)
    print("✅ Trigger Quotes AI 二次精煉完成。")
    
    # 4. 【新增】行為參與分佈圖的數據 (預先在後端聚合)
    behavior_chart_data = aggregate_behavior_data_for_chart(master_timeline, interval_minutes=10)

    # --- 階段六：AI 分塊分析 (使用極簡Prompt，只獲取渲染卡片所需數據) ---
    print(f"\n步驟 6/8: 開始分塊處理時間軸以獲取量化數據 (最大並行數: {MAX_CONCURRENT_AI_TASKS})...")
    # semaphore = asyncio.Semaphore(MAX_CONCURRENT_AI_TASKS)
    intervals_per_chunk = (CHUNK_SIZE_MINUTES * 60) // TIME_INTERVAL_SECONDS
    chunks = [master_timeline[i:i + intervals_per_chunk] for i in range(0, len(master_timeline), intervals_per_chunk)]
    # 此處調用的 analyze_chunk_with_ai_async 應使用您修改過的極簡 Prompt
    analysis_tasks = [analyze_chunk_with_ai_async(chunk, i, len(chunks), semaphore) for i, chunk in enumerate(chunks)]
    
    # all_chunk_analyses 現在將會是一個非常精簡的列表
    all_chunk_analyses = await asyncio.gather(*analysis_tasks)
    all_chunk_analyses = [res for res in all_chunk_analyses if res is not None]

    print(f"✅ 所有 {len(all_chunk_analyses)} 個區塊的量化數據提取完成。")

    # --- 階段七 & 八：生成最終 AI 建議文字並儲存極簡報告 ---
    try:
        # 即使 all_chunk_analyses 變得很精簡，它仍然是生成最終建議的基礎
        if all_chunk_analyses:
            # 預摘要的生成依然是必要的，它將所有區塊的數據濃縮成文字
            pre_summary = await create_pre_summary_for_final_analysis_async(
                teaching_mode_summary,
                behavior_chart_data,
                keyword_timestamps
            )
            
            if not pre_summary: 
                print("❌ 無法生成預摘要，終止流程。")
                return

            # 根據預摘要，生成最終的文字版報告 (使用您修改過的新Prompt)
            final_teacher_report_text = await summarize_chunks_with_ai_async(pre_summary)
            
            if final_teacher_report_text:
                # 組裝最終的、極度精簡的 JSON 報告
                final_report = {
                    # 這是給老師看的文字報告
                    "teacher_report_text": final_teacher_report_text,
                    
                    # 這是給前端圖表用的數據
                    "charts_data": {
                        "teaching_mode_distribution": teaching_mode_summary,
                        "behavior_participation_chart": behavior_chart_data
                    },
                    
                    # 這是給前端互動元件用的數據
                    "interactive_data": {
                        "content_timeline": merged_timeline_with_wpm,
                        "keyword_timestamps": keyword_timestamps
                    },
                    
                    # 這是渲染逐段洞察卡片所需的精簡數據
                    "timeline_analysis": all_chunk_analyses
                }
                
                print("\n步驟 8/8: 儲存最終精簡分析報告...")
                os.makedirs(OUTPUT_ANALYSIS_DIRECTORY, exist_ok=True)
                with open(OUTPUT_ANALYSIS_JSON_PATH, 'w', encoding='utf-8') as f:
                    json.dump(final_report, f, ensure_ascii=False, indent=2)
                
                end_total_time = time.time()
                total_duration = end_total_time - start_total_time

                print("-" * 50)
                print(f"🎉🎉🎉 全部分析完成！(極簡報告版) 🎉🎉🎉")
                print(f"⏱️ 總耗時: {total_duration // 60:.0f} 分 {total_duration % 60:.2f} 秒。")
                print(f"✅ 綜合分析報告已成功儲存至: {OUTPUT_ANALYSIS_JSON_PATH}")
                print("-" * 50)
            else:
                 print("❌ AI 最終文字報告生成失敗，未生成報告檔案。")
        else:
            print("❌ 所有區塊均分析失敗，未生成任何報告檔案。")

    except Exception as e:
        print(f"\n❌ 在主流程中發生嚴重錯誤: {e}")
        import traceback
        traceback.print_exc()
        print("❌ 程式已終止。")

await main()

--- Classroom Analysis Engine v5.4 (Async Edition) ---
步驟 1/8: 初始化環境與 Azure OpenAI Async Client...
✅ Azure OpenAI Async client 初始化成功。
--- Classroom Analysis Engine v9.0 (Minimalist Report Edition) ---
步驟 2/8: 讀取學生行為數據 (目標課堂: 09/28)...
✅ 成功載入 24 位學生的行為數據，課程總時長約 184 分鐘。
步驟 3/8: 讀取老師逐字稿 (採用新格式解析器)...
✅ 新格式逐字稿載入並處理完成，共 400 筆有效片段。
步驟 4/8: 讀取板書圖片列表...
✅ 板書圖片列表載入完成。
步驟 3.2/8: 執行逐字稿前處理過濾器...
✅ 逐字稿前處理完成：原始 400 筆 -> 過濾後 290 筆 -> 合併後 290 筆。
步驟 3.5/8: 調用 AI 進行逐字稿校正 (採用【上下文增強模式】)...
  - 逐字稿已分為 10 個區塊，將並行處理。
    - ✅ 逐字稿區塊 10/10 校正成功。
    - ✅ 逐字稿區塊 6/10 校正成功。
    - ✅ 逐字稿區塊 1/10 校正成功。
    - ✅ 逐字稿區塊 5/10 校正成功。
    - ✅ 逐字稿區塊 2/10 校正成功。
    - ✅ 逐字稿區塊 8/10 校正成功。
    - ✅ 逐字稿區塊 3/10 校正成功。
    - ✅ 逐字稿區塊 7/10 校正成功。
    - ✅ 逐字稿區塊 9/10 校正成功。
    - ✅ 逐字稿區塊 4/10 校正成功。
✅ AI 逐字稿校正完成。
步驟 5/8: 建立整合時間軸 (採用【微觀教學模式即時分類 v10.0】)...
  -> 已將課程分為 370 個微觀片段，開始並行 AI 分類...
  -> 所有微觀片段分類與溯因完成。
✅ 整合時間軸建立完成。
  -> 正在執行【後處理規則】：處理無逐字稿與噪音片段...
  -> 校正完成：共 30 個片段被強制歸類為「學生考試」。
  -> 合併完成：共 59 個無逐字稿片段已併入前一教學模式。
  -> 正在執行【強制性規則】：合併與固化「下課休息

#### 舊版

In [ ]:
# -*- coding: utf-8 -*-
import os
import json
import datetime
import re
import asyncio
from openai import AsyncAzureOpenAI , APIError, RateLimitError
from dotenv import load_dotenv
from collections import defaultdict
import time
import random
import numpy as np
from scipy import stats
# ==============================================================================
# --- ★★★【使用者設定區】★★★ ---
# ==============================================================================
TARGET_SESSION_TIME = "09/28"
STUDENT_DATA_DIRECTORY = r'C:\Users\User\Desktop\test\SynologyDrive\json_behavior'
TEACHER_TRANSCRIPT_PATH = r'C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0928\老師\0928.txt'
BLACKBOARD_IMAGES_DIRECTORY = r'C:\Users\User\Desktop\test\note_blackboard\0928_English_filtered'
OUTPUT_ANALYSIS_DIRECTORY = r'C:\Users\User\Desktop\test\classroom_analysis_report'
OUTPUT_ANALYSIS_JSON_PATH = os.path.join(OUTPUT_ANALYSIS_DIRECTORY, f'classroom_analysis_report_{TARGET_SESSION_TIME.replace("/", "")}.json')
TIME_INTERVAL_SECONDS = 30
CHUNK_SIZE_MINUTES = 20

# ==============================================================================
# --- API 金鑰與 Client 初始化 ---
# ==============================================================================
print("--- Classroom Analysis Engine v5.4 (Async Edition) ---")
print("步驟 1/8: 初始化環境與 Azure OpenAI Async Client...")
load_dotenv()
AZURE_API_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
TEXT_DEPLOYMENT_NAME = os.getenv("CHAT_COMPLETION_NAME")

if not all([AZURE_API_KEY, AZURE_ENDPOINT, TEXT_DEPLOYMENT_NAME]):
    print("❌ 錯誤：缺少必要的 Azure OpenAI 環境變數。請檢查 .env 檔案。")
    exit()

try:
    # 【修改】使用 AsyncAzureOpenAI 實例化 client
    async_client = AsyncAzureOpenAI(
        api_key=AZURE_API_KEY, 
        azure_endpoint=AZURE_ENDPOINT, 
        api_version="2024-02-01",
        max_retries=3 # 讓 client 內建一些基礎的重試邏輯
    )
    print("✅ Azure OpenAI Async client 初始化成功。")
except Exception as e:
    print(f"❌ 初始化 Azure OpenAI Async Client 時發生錯誤: {e}")
    exit()

# ==============================================================================
# --- 輔助函式與資料讀取模組 ---
# ==============================================================================
def clean_json_string(json_string: str) -> str:
    """
    (v5.5 新增) 清洗AI返回的字符串，移除其中非法的控制字符，以避免JSON解析錯誤。
    """
    # 建立一個正則表達式，匹配所有不被JSON標準允許的控制字符
    # \x00-\x1F 是主要的控制字符範圍, 但我們需要保留 \b, \f, \n, \r, \t
    # 正則表達式 [^\x20-\x7E\b\f\n\r\t] 匹配所有非可打印ASCII字符且不是合法JSON轉義符的字符
    # 為了更廣泛地處理Unicode，我們使用一個更簡單的方法：只保留 "合法的" 字符。
    
    cleaned_chars = []
    for char in json_string:
        # JSON 規範允許的控制字符是 \b, \f, \n, \r, \t
        if ord(char) < 32 and char not in '\b\f\n\r\t':
            # 如果是其他控制字符，則跳過
            continue
        cleaned_chars.append(char)
        
    return "".join(cleaned_chars)

def parse_timestamp(timestamp_str):
    """
    (v5.11 修正版) 解析多種時間戳格式，包括 H:MM:SS 和 HH:MM:SS。
    """
    h, m, s = 0, 0, 0
    
    # 處理圖片文件名格式
    match = re.match(r'(\d{2})-(\d{2})-(\d{2})-(\d{3})\.jpg', timestamp_str)
    if match: 
        h, m, s, _ = map(int, match.groups())
        return h * 3600 + m * 60 + s
        
    # --- ★★★【核心修正點：讓小時部分可以匹配1位或2位數字】★★★ ---
    # \d{1,2} 表示匹配1到2個數字
    match = re.match(r'(\d{1,2}):(\d{2}):(\d{2})', timestamp_str)
    if match: 
        parts = list(map(int, match.groups()))
        if len(parts) == 3:
            h, m, s = parts
            return h * 3600 + m * 60 + s
        # 有時候 timedelta 可能會輸出 H:MM 這樣的格式，這裡做一個兼容
        elif len(parts) == 2:
            m, s = parts
            return m * 60 + s
    # --- ★★★【修正結束】★★★ ---

    # 處理 WhisperX 的格式
    match = re.search(r'(\d{2})h(\d{2})m(\d{2})s', timestamp_str)
    if match: 
        h, m, s = map(int, match.groups())
        return h * 3600 + m * 60 + s
        
    return None

### --- MODIFIED SECTION --- ###
def load_student_data(directory, session_time):
    """(v4.1) 讀取並整合所有學生的行為數據，並加入數據完整性檢查"""
    print(f"步驟 2/8: 讀取學生行為數據 (目標課堂: {session_time})...")
    all_behaviors = []
    if not os.path.isdir(directory):
        print(f"❌ 錯誤：找不到學生資料夾 '{directory}'。")
        return None, 0
        
    student_folders = [f for f in os.listdir(directory) if os.path.isdir(os.path.join(directory, f))]
    loaded_students_count = 0
    
    for student_name in student_folders:
        student_dir = os.path.join(directory, student_name)
        for filename in os.listdir(student_dir):
            if filename.endswith('.json'):
                filepath = os.path.join(student_dir, filename)
                try:
                    with open(filepath, 'r', encoding='utf-8') as f: data = json.load(f)
                    
                    if data.get("report_metadata", {}).get("report_generation_time") == session_time:
                        student_id = data.get("report_metadata", {}).get("student_id", "未知學生")
                        
                        for behavior_list in data.get('detailed_sequence_analysis', []):
                            image_filenames = behavior_list.get('image_filenames_in_batch', [])
                            image_filenames_len = len(image_filenames)
                            
                            for image_highlight in behavior_list.get('analysis', {}).get('per_image_highlights', []):
                                
                                # ★★★【安全檢查】★★★
                                # 在存取前，先檢查索引是否合法
                                index = image_highlight.get('image_index_in_sequence')
                                if index is not None and index < image_filenames_len:
                                    img_filename = image_filenames[index]
                                    seconds = parse_timestamp(img_filename)
                                    if seconds is not None:
                                        all_behaviors.append({
                                            "seconds": seconds, 
                                            "student_id": student_id, 
                                            "behaviors": image_highlight.get("behavior_category", [])
                                        })
                                else:
                                    # 如果索引不合法，則印出警告並跳過此筆紀錄
                                    print(f"  - ⚠️ 數據不一致警告：在檔案 '{filename}' 中，偵測到無效的 image_index_in_sequence: {index} (列表長度為 {image_filenames_len})。將跳過此筆紀錄。")
                        
                        # print(f"  - 已載入 '{student_name}' 的報告。") # 為減少輸出訊息，可註解此行
                        loaded_students_count +=1
                        
                except Exception as e:
                    # 將 list index out of range 錯誤明確化
                    if isinstance(e, IndexError):
                         print(f"  - ⚠️ 嚴重數據錯誤：在解析 '{filename}' 時發生 'list index out of range'。這通常是源檔案數據不一致導致的。")
                    else:
                         print(f"  - ⚠️ 警告：讀取或解析 '{filename}' 時出錯: {e}")

    if not all_behaviors:
        print(f"❌ 錯誤：在 '{directory}' 中找不到任何符合 '{session_time}' 的學生報告。")
        return None, 0
        
    max_time = max(b['seconds'] for b in all_behaviors) if all_behaviors else 0
    print(f"✅ 成功載入 {loaded_students_count} 位學生的行為數據，課程總時長約 {max_time // 60} 分鐘。")
    return sorted(all_behaviors, key=lambda x: x['seconds']), max_time
### --- END MODIFIED SECTION --- ###

def load_teacher_transcript(filepath):
    print("步驟 3/8: 讀取老師逐字稿...")
    if not os.path.exists(filepath): print(f"  - ⚠️ 警告：找不到逐字稿檔案 '{filepath}'，將跳過此項分析。"); return []
    transcript_data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            match = re.match(r'(\d{2}:\d{2}:\d{2}) - (\d{2}:\d{2}:\d{2}):\s*(.+)', line)
            if match:
                start_str, end_str, text = match.groups()
                start_seconds = parse_timestamp(start_str); end_seconds = parse_timestamp(end_str)
                if start_seconds is not None:
                    
                    # --- ★★★【核心修改點】★★★ ---
                    # 使用 re.sub 移除開頭的 "大寫字母 + 冒號 + 空格" 模式
                    # 例如 "A: 你好" -> "你好"
                    #      "C: 來來來" -> "來來來"
                    cleaned_text = re.sub(r'^[A-Z]:\s+', '', text)
                    # --- ★★★【修改結束】★★★ ---

                    # 使用清洗後的 cleaned_text 來儲存
                    transcript_data.append({"start_seconds": start_seconds, "end_seconds": end_seconds, "text": cleaned_text})

    print("✅ 逐字稿載入完成 (並已進行自動清洗)。") # 您可以修改 print 訊息
    return transcript_data

def load_blackboard_images(directory):
    print("步驟 4/8: 讀取板書圖片列表...")
    if not os.path.isdir(directory): print(f"  - ⚠️ 警告：找不到板書圖片資料夾 '{directory}'，將跳過此項分析。"); return []
    image_data = []
    for filename in os.listdir(directory):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            seconds = parse_timestamp(filename)
            if seconds is not None: image_data.append({"seconds": seconds, "filename": filename})
    print("✅ 板書圖片列表載入完成。")
    return sorted(image_data, key=lambda x: x['seconds'])

def create_master_timeline(student_behaviors, transcript, blackboard_images, total_seconds, interval):
    # 更新版本號和說明
    print("步驟 5/8: 建立整合時間軸 (v5.7 - 新增教學模式自動標籤)...") 

    # --- ★★★【修改點 1：定義教學模式的量化標準】★★★ ---
    # 這些閾值是專為補習班情境設計的，可靈活調整
    HIGH_INTENSITY_WPM = 180
    HIGH_LEXICAL_DENSITY = 2  # 補習班快節奏，2個術語可能已構成高強度

    # 用於識別「學生練習時間」的觸發詞
    PRACTICE_TRIGGER_WORDS = {
        "練習一下", "寫一下", "試試看", "給大家", "分鐘", 
        "檢查一下", "對一下答案", "先寫", "完成"
    }
    PRACTICE_WPM_THRESHOLD = 60 # 學生練習時，老師的語速通常很低

    # 用於識別「認知緩衝時間」的標準
    COGNITIVE_BREAK_WPM_UPPER_BOUND = 170 # 故事分享通常語速適中偏快
    IDLE_WPM_THRESHOLD = 40 # 低於此語速可能為無效語音

    # --- ★★★【補全部分：確保行為和關鍵詞定義完整】★★★ ---
    TASK_ORIENTED_BEHAVIORS = {
        "目視書本/筆記", "做筆記", "翻書",
        "主動舉手", "被動舉手", "身體前傾"
    }
    RECEPTIVE_ENGAGEMENT_BEHAVIORS = {
        "目視教師", "目視黑板", "坐姿直立", "身體後靠" 
    }
    DISENGAGEMENT_BEHAVIORS = {
        "目視同學", "目視他處", "托腮", "觸摸臉部", "觸摸頭髮", 
        "玩弄手部/文具", "低頭(非學習)", "趴睡", "飲食"
    }
    PROFESSIONAL_KEYWORDS = {
        "分詞", "動詞", "形容詞", "介系詞", "主詞", "受詞", "補語",
        "完成式", "進行式", "過去式", "未來式", "時態", "語法", "句型",
        "附加問句", "共和", "民主", "獨裁", "憲法", "政黨", "理論"
    }
    # --- ★★★【補全結束】★★★ ---

    master_timeline = []
    # 使用 enumerate 以便在迴圈中獲取索引 i
    for i, start_interval in enumerate(range(0, total_seconds + interval, interval)):
        end_interval = start_interval + interval
        
        interval_data = {
            "seconds": start_interval,
            "start_time": str(datetime.timedelta(seconds=start_interval)), 
            "end_time": str(datetime.timedelta(seconds=end_interval)), 
            "teacher_speech": "", 
            "student_behavior_summary": defaultdict(list), 
            "blackboard_snapshots": [],
            "focus_distribution": None,
            "cognitive_load_metrics": None,
            # --- ★★★【修改點 2：新增 teaching_mode 欄位】★★★ ---
            "teaching_mode": "標準互動" # 預設模式
            # --- ★★★【修改結束】★★★ ---
        }

        # --- 整合老師逐字稿 ---
        current_speech_parts = [
            entry['text'] for entry in transcript 
            if entry['start_seconds'] < end_interval and entry['end_seconds'] > start_interval
        ]
        interval_data["teacher_speech"] = " ".join(current_speech_parts)

        # --- 整合板書圖片 ---
        interval_data["blackboard_snapshots"] = [
            image['filename'] for image in blackboard_images 
            if start_interval <= image['seconds'] < end_interval
        ]

        # --- 計算專注度分佈 ---
        task_focused_students = set()
        receptive_students = set()
        disengaged_students = set()
        
        behaviors_in_interval = [
            b for b in student_behaviors 
            if start_interval <= b['seconds'] < end_interval
        ]

        for behavior in behaviors_in_interval:
            student_id = behavior['student_id']
            for b_category in behavior['behaviors']:
                interval_data["student_behavior_summary"][b_category].append(student_id)
                if b_category in TASK_ORIENTED_BEHAVIORS:
                    task_focused_students.add(student_id)
                elif b_category in RECEPTIVE_ENGAGEMENT_BEHAVIORS:
                    receptive_students.add(student_id)
                elif b_category in DISENGAGEMENT_BEHAVIORS:
                    disengaged_students.add(student_id)
        
        all_students_in_interval = task_focused_students.union(receptive_students, disengaged_students)
        total_active_students = len(all_students_in_interval)

        if total_active_students > 0:
            interval_data["focus_distribution"] = {
                "task_oriented_focus": round((len(task_focused_students) / total_active_students) * 100, 1),
                "receptive_engagement": round((len(receptive_students) / total_active_students) * 100, 1),
                "disengagement": round((len(disengaged_students) / total_active_students) * 100, 1)
            }
        
        # --- 計算認知負荷指標 ---
        speech_text = interval_data["teacher_speech"]
        wpm, question_count, keyword_count = 0, 0, 0 # 先初始化
        if speech_text:
            char_count = len(speech_text)
            wpm = (char_count / interval) * 60
            question_count = speech_text.count('？') + speech_text.count('?')
            keyword_count = sum(1 for keyword in PROFESSIONAL_KEYWORDS if keyword in speech_text)
            
            interval_data["cognitive_load_metrics"] = {
                "speech_rate_wpm": round(wpm, 1),
                "question_frequency": question_count,
                "lexical_density": keyword_count
            }

        # --- ★★★【修改點 3：根據新規則，自動標記教學模式】★★★ ---
        # 判斷邏輯具有優先級順序：高強度 -> 學生練習 -> 認知緩衝
        
        # 規則 1: 高強度講解 (High-Intensity Lecture)
        if (wpm >= HIGH_INTENSITY_WPM) or (keyword_count >= HIGH_LEXICAL_DENSITY):
            interval_data["teaching_mode"] = "高強度講解"
        
        # 規則 2: 學生練習時間 (Student Practice Time)
        # 條件：前一個時間區間包含練習觸發詞，且當前區間老師語速很低
        elif master_timeline: # ★★★【核心修正點】★★★ 直接檢查 master_timeline 是否有內容
            # 使用 -1 索引來安全地獲取最後一個（也就是前一個剛被加入的）元素
            prev_speech = master_timeline[-1].get("teacher_speech", "") 
            if any(word in prev_speech for word in PRACTICE_TRIGGER_WORDS) and wpm < PRACTICE_WPM_THRESHOLD:
                interval_data["teaching_mode"] = "學生練習時間"

        # 規則 3: 認知緩衝時間 (Cognitive Break Time)
        # 條件：語速在正常說話範圍，但沒有專業術語，且不是高強度講解
        elif IDLE_WPM_THRESHOLD < wpm < COGNITIVE_BREAK_WPM_UPPER_BOUND and keyword_count == 0:
             interval_data["teaching_mode"] = "認知緩衝時間"
        
        # 如果以上都不是，則維持預設的 "標準互動"
        # --- ★★★【修改結束】★★★ ---

        # --- 清理並加入最終列表 ---
        interval_data["student_behavior_summary"] = {k: list(set(v)) for k, v in interval_data["student_behavior_summary"].items()}
        
        if interval_data["teacher_speech"] or interval_data["student_behavior_summary"]: 
            master_timeline.append(interval_data)

    print("✅ 整合時間軸建立完成，並已自動標記教學模式。")
    return master_timeline

def find_micro_events(master_timeline, peak_threshold=80.0, trough_threshold=40.0):
    """(v5.1 新增) 在完整時間軸上尋找專注度的高峰、低谷與關鍵轉折點"""
    print("步驟 5.5/8: 尋找微觀事件 (專注度高峰、低谷與轉折點)...")
    micro_events = []
    
    # 用於尋找最大轉折
    max_focus_rise = {"delta": 0, "event": None}
    max_disengagement_rise = {"delta": 0, "event": None}
    
    for i, interval in enumerate(master_timeline):
        dist = interval.get("focus_distribution")
        if not dist:
            continue

        # 檢查高峰和低谷
        if dist.get("task_oriented_focus", 0) >= peak_threshold:
            micro_events.append({
                "type": "專注度高峰 (任務導向)",
                "time": interval["start_time"],
                "details": f"任務導向專注度達到 {dist['task_oriented_focus']}%",
                "teacher_speech_context": interval.get("teacher_speech", "")
            })
        
        if dist.get("disengagement", 0) >= trough_threshold:
            micro_events.append({
                "type": "專注度低谷 (分心)",
                "time": interval["start_time"],
                "details": f"分心狀態佔比達到 {dist['disengagement']}%",
                "teacher_speech_context": interval.get("teacher_speech", "")
            })

        # 比較與前一個時間點的變化，尋找最大轉折
        if i > 0:
            prev_dist = master_timeline[i-1].get("focus_distribution")
            if prev_dist:
                focus_delta = dist.get("task_oriented_focus", 0) - prev_dist.get("task_oriented_focus", 0)
                disengagement_delta = dist.get("disengagement", 0) - prev_dist.get("disengagement", 0)
                
                if focus_delta > max_focus_rise["delta"]:
                    max_focus_rise["delta"] = focus_delta
                    max_focus_rise["event"] = {
                        "type": "關鍵拉升點",
                        "time": interval["start_time"],
                        "details": f"任務導向專注度在30秒內急遽上升 {focus_delta:.1f}%",
                        "teacher_speech_context": interval.get("teacher_speech", "")
                    }
                
                if disengagement_delta > max_disengagement_rise["delta"]:
                    max_disengagement_rise["delta"] = disengagement_delta
                    max_disengagement_rise["event"] = {
                        "type": "關鍵下跌點",
                        "time": interval["start_time"],
                        "details": f"分心狀態佔比在30秒內急遽上升 {disengagement_delta:.1f}%",
                        "teacher_speech_context": interval.get("teacher_speech", "")
                    }

    # 將找到的最大轉折點加入事件列表
    if max_focus_rise["event"]:
        micro_events.insert(0, max_focus_rise["event"]) # 放在最前面，因為很重要
    if max_disengagement_rise["event"]:
        micro_events.insert(0, max_disengagement_rise["event"])
        
    print(f"✅ 微觀事件分析完成，找到 {len(micro_events)} 個值得關注的時間點。")
    return micro_events

def aggregate_timeline(timeline, aggregate_interval_minutes=2):
    """
    將高粒度的時間軸數據聚合成較低粒度的平均值數據。
    (v5.11 修正版 - 增加對 None 值的安全檢查)
    """
    if not timeline:
        return []

    aggregate_interval_seconds = aggregate_interval_minutes * 60
    aggregated = []
    
    current_chunk_start = 0
    # ★★★【修正點 1：確保 timeline 非空，避免 IndexError】★★★
    last_second = timeline[-1]['seconds'] if timeline else 0

    while current_chunk_start <= last_second:
        chunk_end = current_chunk_start + aggregate_interval_seconds
        
        intervals_in_chunk = [
            item for item in timeline 
            if current_chunk_start <= item['seconds'] < chunk_end
        ]
        
        if intervals_in_chunk:
            # --- ★★★【修正點 2：在計算前，過濾掉 focus_distribution 為 None 的數據】★★★ ---
            valid_focus_data = [
                d['focus_distribution'] 
                for d in intervals_in_chunk 
                if d.get('focus_distribution') is not None
            ]
            
            # 只有在存在有效數據時才進行計算
            if valid_focus_data:
                num_valid_intervals = len(valid_focus_data)
                avg_task = sum(d.get('task_oriented_focus', 0) for d in valid_focus_data) / num_valid_intervals
                avg_receptive = sum(d.get('receptive_engagement', 0) for d in valid_focus_data) / num_valid_intervals
                avg_disengagement = sum(d.get('disengagement', 0) for d in valid_focus_data) / num_valid_intervals
                
                focus_distribution_summary = {
                    "task_oriented_focus": round(avg_task, 1),
                    "receptive_engagement": round(avg_receptive, 1),
                    "disengagement": round(avg_disengagement, 1)
                }
            else:
                # 如果這個區塊完全沒有專注度數據，則返回 None 或預設值
                focus_distribution_summary = None
            # --- ★★★【修正結束】★★★ ---

            # 合併老師話語
            combined_speech = " ".join(d.get('teacher_speech', '') for d in intervals_in_chunk if d.get('teacher_speech'))
            
            aggregated.append({
                "start_time": str(datetime.timedelta(seconds=current_chunk_start)),
                "end_time": str(datetime.timedelta(seconds=chunk_end)),
                "focus_distribution": focus_distribution_summary,
                "teacher_speech_summary": combined_speech[:100] + '...' if len(combined_speech) > 100 else combined_speech
            })
            
        current_chunk_start = chunk_end
        
    return aggregated

# ==============================================================================
# --- ★★★【v5.10 新增區塊：進階量化分析模組】★★★ ---
# ==============================================================================

def calculate_mode_durations(master_timeline, total_seconds, interval_seconds):
    """(v5.8) 量化並計算每種教學模式的持續時間"""
    print("步驟 5.6/8: 量化教學模式持續時間...")
    if not master_timeline: return {}
    mode_segments = defaultdict(list)
    if not master_timeline: return {}
    current_mode = master_timeline[0].get("teaching_mode")
    current_segment_duration = 0
    for interval_data in master_timeline:
        mode = interval_data.get("teaching_mode")
        if mode == current_mode:
            current_segment_duration += interval_seconds
        else:
            if current_mode: mode_segments[current_mode].append(current_segment_duration)
            current_mode = mode
            current_segment_duration = interval_seconds
    if current_mode and current_segment_duration > 0:
        mode_segments[current_mode].append(current_segment_duration)
    duration_summary = {}
    for mode, segments in mode_segments.items():
        total_duration = sum(segments)
        number_of_segments = len(segments)
        average_duration = total_duration / number_of_segments if number_of_segments > 0 else 0
        duration_summary[mode] = {
            "total_duration_seconds": total_duration,
            "total_duration_minutes": round(total_duration / 60, 1),
            "percentage_of_class": round((total_duration / total_seconds) * 100, 1) if total_seconds > 0 else 0,
            "number_of_segments": number_of_segments,
            "average_segment_duration_seconds": round(average_duration, 1)
        }
    print("✅ 教學模式持續時間量化完成。")
    return duration_summary

def find_important_keywords(master_timeline):
    """(v5.9) 找出教師提到特定關鍵字的時間點"""
    print("步驟 5.7/8: 標記教學關鍵字時間點...")
    KEYWORDS_TO_MARK = {"重點", "關鍵", "考試", "注意", "小心", "必考", "圈起來", "畫起來", "寫下來", "思考一下"}
    keyword_timestamps = []
    for i, interval in enumerate(master_timeline):
        speech = interval.get("teacher_speech", "")
        if any(keyword in speech for keyword in KEYWORDS_TO_MARK):
            matched_keyword = next((kw for kw in KEYWORDS_TO_MARK if kw in speech), None)
            context_before = master_timeline[i-1].get("teacher_speech", "") if i > 0 else ""
            context_after = master_timeline[i+1].get("teacher_speech", "") if i < len(master_timeline) - 1 else ""
            keyword_timestamps.append({
                "time": interval["start_time"], "keyword": matched_keyword, "full_quote": speech,
                "context_before": context_before, "context_after": context_after
            })
    print(f"✅ 成功標記出 {len(keyword_timestamps)} 個關鍵字時間點。")
    return keyword_timestamps


# ==============================================================================
# --- ★★★【v5.10 新增功能：量化語速變化趨勢】★★★ ---
# ==============================================================================
def analyze_speech_rate_trend(master_timeline):
    """
    (v5.10 新增) 使用線性迴歸分析語速隨時間的變化趨勢，並比較課堂前後半段的平均語速。
    """
    print("步驟 5.9/8: 分析語速變化趨勢...")

    # 1. 提取所有包含語速數據的時間點和語速值
    time_points = []
    wpm_points = []
    for interval in master_timeline:
        # 確保該時間區間有認知負荷指標且語速不為 None
        if interval.get("cognitive_load_metrics") and interval["cognitive_load_metrics"].get("speech_rate_wpm") is not None:
            # 只分析老師有在說話的時間點 (語速 > 0)
            if interval["cognitive_load_metrics"]["speech_rate_wpm"] > 0:
                time_points.append(interval["seconds"])
                wpm_points.append(interval["cognitive_load_metrics"]["speech_rate_wpm"])
    
    # 如果數據點太少，則無法進行有意義的分析
    if len(time_points) < 10:
        print("  - ⚠️ 語速數據點過少，跳過趨勢分析。")
        return {
            "error": "Not enough data points to perform trend analysis.",
            "trend_description": "數據不足",
            "linear_regression_stats": None,
            "comparative_summary": None
        }

    # 2. 執行線性迴歸分析
    # 將列表轉換為 NumPy 陣列以便計算
    x = np.array(time_points)
    y = np.array(wpm_points)
    
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

    # 3. 解讀迴歸結果
    trend_description = ""
    # p-value < 0.05 表示趨勢在統計上是顯著的
    if p_value < 0.05:
        if slope < -0.01: # 斜率為負，且有一定幅度，代表下降趨勢
            trend_description = "呈現顯著的下降趨勢 (老師的語速隨時間推移而顯著變慢)。"
        elif slope > 0.01: # 斜率為正，代表上升趨勢
            trend_description = "呈現顯著的上升趨勢 (老師的語速隨時間推移而顯著變快)。"
        else:
            trend_description = "語速非常穩定，未觀察到顯著的變化趨勢。"
    else:
        # p-value >= 0.05 表示觀察到的趨勢不顯著，可能只是隨機波動
        trend_description = "語速整體波動，未觀察到統計上顯著的長期增減趨勢。"

    # 4. 計算課堂前後半段的平均語速作為輔助參考
    midpoint_time = time_points[-1] / 2 if time_points else 0
    first_half_wpm = [wpm for t, wpm in zip(time_points, wpm_points) if t <= midpoint_time]
    second_half_wpm = [wpm for t, wpm in zip(time_points, wpm_points) if t > midpoint_time]
    
    avg_first_half = np.mean(first_half_wpm) if first_half_wpm else 0
    avg_second_half = np.mean(second_half_wpm) if second_half_wpm else 0

    # 5. 組織並返回分析結果
    analysis_result = {
        "trend_description": trend_description,
        "linear_regression_stats": {
            "slope_per_second": slope,
            "slope_per_minute": slope * 60, # 每分鐘語速的變化量，更直觀
            "p_value": p_value,
            "r_squared": r_value**2 # R平方值，表示趨勢線對數據的解釋程度
        },
        "comparative_summary": {
            "average_wpm_first_half": round(avg_first_half, 1),
            "average_wpm_second_half": round(avg_second_half, 1),
            "change_percentage": round(((avg_second_half - avg_first_half) / avg_first_half) * 100, 1) if avg_first_half > 0 else 0
        }
    }
    
    print(f"✅ 語速趨勢分析完成: {trend_description}")
    return analysis_result

# ==============================================================================
# --- ★★★【v5.9 新增區塊：關鍵片段微觀分析模組】★★★ ---
# ==============================================================================

# ==============================================================================
# --- ★★★【v5.9 新增區塊：關鍵片段微觀分析模組 (全新升級版)】★★★ ---
# ==============================================================================

def get_critical_segment_analysis_prompt():
    """
    【全新升級版】
    生成用於對【單一關鍵高強度教學片段】進行微觀分析的 Prompt。
    - 整合了宏觀功能分類的概念。
    - 新增了量化下滑幅度的要求。
    """
    
    # --- ★ 修改點 1: 引入宏觀功能分類的概念 ---
    # 這裡我們將您另一份程式碼中的 "教學狀態" 概念，轉化為 AI 在分析時可以參考的 "教學活動" 分類
    teaching_activity_definitions = """
**【教學活動參考分類】**
在你的分析中，請隨時參考以下教學活動的分類來理解上下文：
- **教師講解**: 老師主導的、連續的知識輸出，包括新知識傳授或舊知識複習。
- **學生練習/考試**: 老師指令後，學生進行獨立的、靜默的練習或測驗。
- **師生互動**: 針對教學內容的雙向問答、討論，或與課堂管理相關的交流。
- **閒聊/認知緩衝**: 與當前教學主題無直接關聯的個人故事、生活經驗分享或非正式交流。
"""

    # --- ★ 修改點 2: 全面升級 JSON 結構，增加 quantitative_impact ---
    json_structure = """
    {{
      "segment_start_time": "（此片段的開始時間）",
      "segment_end_time": "（此片段的結束時間）",
      "core_teaching_topic": "（總結這 7-12 分鐘內最核心的單一教學主題，例如：'現在完成式的用法解析'）",
      "attention_narrative": "（以敘事方式，詳細描述學生專注度在此片段中的變化軌跡。例如：'片段開始時，學生專注度高...約在 4 分 30 秒後...分心比例達到峰值...'）",
      "breaking_point_analysis": {{
        "time_of_decline": "（找出專注度開始【首次顯著下滑】的具體時間點，例如：'0:28:30'）",
        "teacher_quote_at_breaking_point": "（引用導致專注度下滑的【那一句關鍵話語】）",
        "analysis_of_cause": "（分析下滑的原因，例如：'教師在此處連續講解超過 5 分鐘未進行互動，且引入了一個複雜的語法例外規則，超出了部分學生的工作記憶負荷。'）",
        
        "quantitative_impact": {{
          "focus_metric_analyzed": "（你主要分析的是哪個指標的變化，必須是 'disengagement' 或 'task_oriented_focus'）",
          "value_before_decline_pct": （下滑前的指標數值，僅回傳數字，例如：35.5）,
          "value_after_decline_pct": （下滑後的指標數值，僅回傳數字，例如：55.0）,
          "change_delta_pct": （計算出的變化幅度，僅回傳數字，例如：19.5 或 -20.0）
        }}
      }},
      "segment_summary": "（對這個關鍵片段的教學成效做一個總結性評價。）"
    }}
    """
    
    # --- ★ 修改點 3: 強化 Prompt 指令，使其更明確、更專業 ---
    return f"""
你是一位頂尖的教育心理學家和數據分析師，專長是從課堂數據中進行微觀的因果分析。你的任務是分析一段被標記為【關鍵高強度教學】的課堂數據（長度約 7-12 分鐘）。

{teaching_activity_definitions}

**【你的核心任務】**
1.  **敘述專注度軌跡**: 描述學生專注度從高到低的完整變化過程。
2.  **定位「引爆點」(Breaking Point)**: 精確找出學生專注度開始**首次顯著下滑**的時間點、對應的教師話語，並**量化其影響**。
3.  **分析原因**: 結合認知負荷理論 (Cognitive Load Theory)，解釋為什麼在那個時間點學生的專注度會開始下滑。

**【★★★ V2.0 新增指令：量化引爆點的衝擊 ★★★】**
在 `breaking_point_analysis` 區塊中，你**必須**新增並填寫 `quantitative_impact` 子物件。
- 你需要從輸入數據的 `focus_distribution` 中，找出引爆點時間前後最能反映變化的指標（通常是 `disengagement` 的上升或 `task_oriented_focus` 的下降）。
- 你必須提供變化前的數值、變化後的數值，以及它們之間的差值（delta），所有數值都只要數字本身。

**你的輸入資料格式如下：**
一個JSON列表，代表一段連續的「高強度講解」片段。

**你的輸出格式要求：**
你必須嚴格遵循以下的JSON結構，並填寫所有欄位。你的回答必須是一個結構完整的 JSON 物件，絕對不能包含任何額外的文字、註解或 Markdown 標記。

{json_structure}
"""

async def analyze_critical_segment_with_ai_async(segment, index, total, retries=3):
    """(v5.9) 調用 AI 對單個關鍵教學片段進行微觀分析"""
    if not segment: return None
    system_prompt = get_critical_segment_analysis_prompt()
    user_prompt = f"請對以下這段關鍵高強度教學片段進行微觀分析：\n\n{json.dumps(segment, ensure_ascii=False, indent=2)}"
    print(f"  - 正在發起關鍵片段 {index + 1}/{total} 的微觀分析請求...")
    for attempt in range(retries):
        try:
            response = await async_client.chat.completions.create(
                model=TEXT_DEPLOYMENT_NAME,
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                max_tokens=4096, temperature=0.1,
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            print(f"  - ⚠️ 分析關鍵片段 {index + 1} 時發生錯誤 (嘗試 {attempt + 1}/{retries}): {e}")
            if attempt < retries - 1: await asyncio.sleep(5)
    print(f"  - ❌ 關鍵片段 {index + 1}/{total} 在所有重試後分析失敗。")
    return None

# ==============================================================================
# --- AI 分析模組 (v4.0 - 包含完整 Prompt) ---
# ==============================================================================
def identify_teaching_cycles(master_timeline):
    """
    (v5.9 新增, v5.11 修正版) 識別完整的教學週期（高強度 -> 低強度），並進行量化統計，
    增加對 None 值的安全檢查。
    """
    print("步驟 5.8/8: 識別並量化教學週期...")
    if not master_timeline:
        return {"cycles": [], "summary": {}}

    cycles = []
    current_cycle = None
    in_high_intensity = False

    for i, interval in enumerate(master_timeline):
        mode = interval.get("teaching_mode")

        if mode == "高強度講解" and not in_high_intensity:
            in_high_intensity = True
            
            if current_cycle:
                start_sec = parse_timestamp(current_cycle["start_time"])
                end_sec = parse_timestamp(master_timeline[i-1]["end_time"])

                # --- ★★★【修正點：在計算前進行檢查】★★★ ---
                if start_sec is not None and end_sec is not None:
                    current_cycle["end_time"] = master_timeline[i-1]["end_time"]
                    current_cycle["total_duration_seconds"] = end_sec - start_sec
                    cycles.append(current_cycle)
                # --- ★★★【修正結束】★★★ ---

            current_cycle = {
                "cycle_index": len(cycles) + 1,
                "start_time": interval["start_time"],
                "end_time": None,
                "high_intensity_duration": 0,
                "low_intensity_duration": 0
            }

        if current_cycle:
            if mode == "高強度講解":
                current_cycle["high_intensity_duration"] += TIME_INTERVAL_SECONDS
            elif mode in ["學生練習時間", "認知緩衝時間"]:
                current_cycle["low_intensity_duration"] += TIME_INTERVAL_SECONDS
                in_high_intensity = False

    if current_cycle:
        start_sec = parse_timestamp(current_cycle["start_time"])
        end_sec = parse_timestamp(master_timeline[-1]["end_time"])
        
        # --- ★★★【修正點：在計算前進行檢查】★★★ ---
        if start_sec is not None and end_sec is not None:
            current_cycle["end_time"] = master_timeline[-1]["end_time"]
            current_cycle["total_duration_seconds"] = end_sec - start_sec
            cycles.append(current_cycle)
        # --- ★★★【修正結束】★★★ ---

    # 計算統計摘要
    total_cycle_duration = sum(c.get("total_duration_seconds", 0) for c in cycles)
    num_cycles = len(cycles)
    average_cycle_duration = total_cycle_duration / num_cycles if num_cycles > 0 else 0

    summary = {
        "number_of_cycles": num_cycles,
        "average_cycle_duration_seconds": round(average_cycle_duration),
        "average_cycle_duration_minutes": round(average_cycle_duration / 60, 1)
    }

    print(f"✅ 成功識別出 {num_cycles} 個教學週期，平均週期時長約 {summary['average_cycle_duration_minutes']} 分鐘。")
    return {"cycles": cycles, "summary": summary}

async def correct_single_transcript_chunk(chunk, index, total, retries=3):
    """(非同步輔助函式) 處理單個逐字稿區塊的校正"""
    input_text = json.dumps(chunk, ensure_ascii=False, indent=2)
    system_prompt = """
你是一位專業的中文逐字稿校對員，擁有豐富的教育領域知識，專門處理課堂教學的內容。
你的任務是讀取一份帶有時間戳的 JSON 格式逐字稿片段，並修正其中由語音辨識錯誤導致的錯字、同音異字、以及不通順的語句。

【情境資訊】
這份逐字稿來自一堂**高中英文文法課**。內容主要圍繞英文文法、單字、考試重點，偶爾會穿插生活化例子、時事或歷史典故。

【校正規則】
1.  **語意優先**: 根據上下文，修正明顯不合理的詞彙。例如，如果看到「請大家在重點上打上心砲」，你應該理解這很可能是「打上星號」的誤辨。
2.  **保持原意**: 只修正錯誤，不要添加、刪減或改變老師原本的語意。
3.  **保持結構**: 你的輸出必須是與輸入完全相同的 JSON 結構（一個包含多個物件的列表），只是修正了 "text" 欄位中的文字。不要添加任何額外的解釋。
4.  **流暢通順**: 修正後的語句應該通順自然。
"""
    user_prompt = f"請根據你的專業知識和校正規則，修復以下這份逐字稿 JSON 資料片段：\n\n{input_text}"

    for attempt in range(retries):
        try:
            response = await async_client.chat.completions.create(
                model=TEXT_DEPLOYMENT_NAME,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                max_tokens=4096,
                temperature=0.1
            )
            corrected_content = response.choices[0].message.content
            json_match = re.search(r'```json\s*([\s\S]*?)\s*```', corrected_content, re.DOTALL)
            if json_match:
                corrected_content = json_match.group(1)
            
            corrected_chunk = json.loads(corrected_content)
            print(f"    - ✅ 逐字稿區塊 {index + 1}/{total} 校正成功。")
            return corrected_chunk
            
        except RateLimitError as e:
            wait_time = (2 ** attempt) + random.random() # 指數退避策略
            print(f"    - ⚠️ 區塊 {index + 1} 觸發速率限制 (嘗試 {attempt + 1}/{retries})，將在 {wait_time:.1f} 秒後重試...")
            await asyncio.sleep(wait_time) # 使用 asyncio.sleep
        except Exception as e:
            print(f"    - ⚠️ 校正區塊 {index + 1} 時發生錯誤 (嘗試 {attempt + 1}/{retries}): {e}")
            if attempt < retries - 1:
                await asyncio.sleep(5)
            else:
                print(f"    - ❌ 校正區塊 {index + 1} 失敗，將使用此區塊的原始數據。")
                return chunk # 如果所有重試都失敗，返回原始 chunk

async def correct_transcript_with_ai_async(transcript_data, chunk_size=30):
    """(v5.4 非同步版本) 使用並行請求校正逐字稿"""
    print("步驟 3.5/8: 調用 AI 進行逐字稿校正 (採用【非同步並行模式】)...")
    
    if not transcript_data:
        print("  - 逐字稿為空，跳過校正。")
        return []
        
    chunks = [transcript_data[i:i + chunk_size] for i in range(0, len(transcript_data), chunk_size)]
    print(f"  - 逐字稿已分為 {len(chunks)} 個區塊，將並行處理。")

    # 創建所有並行任務
    tasks = [correct_single_transcript_chunk(chunk, i, len(chunks)) for i, chunk in enumerate(chunks)]
    
    # 並行執行所有任務並等待結果
    # asyncio.gather 會按順序返回結果
    results = await asyncio.gather(*tasks)
    
    # 將所有結果（已校正或原始的 chunk）按順序合併
    corrected_transcript = []
    for result in results:
        if result:
            corrected_transcript.extend(result)
            
    print("✅ AI 逐字稿校正完成。")
    return corrected_transcript

def get_chunk_analysis_prompt(chunk_start_time, chunk_end_time):
    """生成用於分析【單一區塊】的 Prompt (v5.2 - 整合認知負荷分析)"""
    
    behavior_decoder = """
**【學生專注狀態解碼指南 (v5.1)】**
你在分析 `student_behavior_summary` 時，必須嚴格遵循以下分類來判斷學生的專注狀態：

**🟢 任務導向專注 (Task-Oriented Focus):**
- **核心行為**: "目視書本/筆記", "做筆記", "翻書", "主動舉手", "被動舉手", "身體前傾"
- **解讀**: 學生正在【主動執行】學習任務。這是最高品質的專注。

**🟡 接收性專注 (Receptive Engagement):**
- **核心行為**: "目視教師", "目視黑板", "坐姿直立", "身體後靠"
- **解讀**: 學生正在【被動接收】老師傳遞的資訊。這是一種有效的專注，尤其在聽講、看示範時。當老師講故事或笑話時，學生從「任務導向」切換到此狀態是正常的，**不應輕易判斷為專注度下降**。

**🔴 分心狀態 (Disengagement):**
- **核心行為**: "目視同學", "目視他處", "托腮", "觸摸臉部", "觸摸頭髮", "玩弄手部/文具", "低頭(非學習)", "趴睡"
- **解讀**: 這些是分心的強烈信號。當這些行為集中出現時，構成需要關注的「專注度低谷事件」。
"""

    json_structure = f"""
{{
  "start_time": "{chunk_start_time}",
  "end_time": "{chunk_end_time}",
  "event_title": "（為這整個時間區塊總結一個最核心的教學活動主題）",
  "event_description": "（客觀描述老師在此區塊的主要教學內容和方法）",
  "student_focus_analysis": {{
    "overall_level": "（基於此區塊內專注度分佈的整體判斷：高 / 中 / 低）",
    "average_distribution": {{
      "task_oriented_focus": "（計算此區塊內所有 task_oriented_focus 的【平均值】，四捨五入到小數點後一位）",
      "receptive_engagement": "（計算此區塊內所有 receptive_engagement 的【平均值】，四捨五入到小數點後一位）",
      "disengagement": "（計算此區塊內所有 disengagement 的【平均值】，四捨五入到小數點後一位）"
    }},
    "dominant_positive_behaviors": "（列出此區塊最主要的『🟢任務導向』或『🟡接收性』行為）",
    "dominant_negative_behaviors": "（列出此區塊最主要的『🔴分心』行為）"
  }},
  "focus_trigger_analysis": {{
      "positive_trigger": {{
          "trigger_snippet": {{
              "key_quote": "（找出最可能【提升】專注度的【一句老師的關鍵話語】）",
              "context_before": "（關鍵話語前的【前一句話】）",
              "context_after": "（關鍵話語後的【後一句話】）",
              "topic": "（為這段對話切片總結一個微主題，如：'強調考試重點'或'生活化比喻'）"
          }},
          "analysis": "（分析為何這段對話能提升專注度，例如：'老師用明確指令引導學生操作，將接收性專注轉化為任務導向專注。'）"
      }},
      "negative_trigger": {{
          "trigger_snippet": {{
              "key_quote": "（找出最可能【導致】專注度【下降】的【一句老師的關鍵話語】）",
              "context_before": "（關鍵話語前的【前一句話】）",
              "context_after": "（關鍵話語後的【後一句話】）",
              "topic": "（為這段對話切片總結一個微主題，如：'長時間個人故事分享'）"
          }},
          "analysis": "（分析為何這段對話導致分心，例如：'與課程主題脫節，導致學生從專注狀態轉為分心。'）"
      }}
  }},
  "teaching_pacing_analysis": {{
      "cognitive_load_level": "（基於 `cognitive_load_metrics` 數據，綜合判斷此區塊的認知負荷是：高 / 中 / 低）",
      "analysis": "（分析此區塊的教學節奏。例如：『教師採用高語速、高密度的概念講解，導致認知負荷偏高，雖然短期內提升了接收性專注，但也可能造成部分學生分心。』或『教師透過頻繁提問和較慢的語速，有效控制了認知負荷，讓學生有時間消化，促進了任務導向專注。』）"
  }},
  "ai_insight": "（你對這【整個時間區塊】的最終專業洞察。**請務必結合『專注度分佈』和『教學節奏分析』**，解釋兩者之間的關聯。例如：『本區塊的專注度波動與教學節奏的變化高度相關。在高認知負荷的語法講解後，老師穿插的低認知負荷的故事分享，雖然導致了短暫分心，但可能起到了必要的認知緩衝作用，為後續的高效學習做了鋪墊。』）"
}}
"""
    
    return f"""
你是一位頂尖的教育數據與心理學分析師，擅長分析在**高效率學習環境（如補習班）**中，教學模式與學生認知狀態的關聯。

**【★★★ 核心指令 (v5.7) ★★★】**
1.  你必須使用我提供的 **【學生專注狀態解碼指南】** 作為判斷學生專注度的黃金準則。
2.  你的分析必須包含**前後文**的「對話切片」。
3.  **【V5.7 高優先級任務】**: 你的輸入資料中新增了 `"teaching_mode"` 欄位，它已被預先標記為以下四種模式之一：
    *   `"高強度講解"`: 老師進行高密度、快節奏的單向知識傳遞。
    *   `"學生練習時間"`: 老師指令後，學生獨立操作練習的時段。
    *   `"認知緩衝時間"`: 老師分享故事、笑話等低負荷內容。
    *   `"標準互動"`: 其他一般性互動。

    在你的 `ai_insight` 欄位中，你必須**深入分析這些【教學模式的切換】與學生專注度波動之間的因果關係**。例如：
    *   分析連續多個「高強度講解」區間是否導致了分心比例的上升。
    *   一個「認知緩衝時間」是否成功地在下一個「高強度講解」開始前重置了學生的注意力？
    *   「學生練習時間」標籤是否對應了學生「🟢 任務導向專注」行為（如"做筆記"）的高峰？

{behavior_decoder}

**你的輸入資料格式如下：**
一個JSON列表，每個物件代表一個30秒的時間區間，包含 `teacher_speech`, `student_behavior_summary`, `focus_distribution`, `cognitive_load_metrics` 以及**新增的 `teaching_mode`** 欄位。

**你的輸出格式要求：**
你必須嚴格遵循以下的JSON結構，並填寫所有欄位...
{json_structure}
"""

def get_summary_prompt(report_in_progress_str):
    """
    生成用於【最終深度匯總分析】的 Prompt (v5.3 - 引入模式識別與量化分析)
    - 輸入的不再只是區塊摘要，而是到目前為止生成的完整報告數據。
    """
    
    # ★★★ V5.3 核心修改點：全新的、要求進行深度分析的 JSON 結構 ★★★
    json_structure = """
{
  "class_narrative": "（根據所有 event_title 和 event_description，撰寫一段話總結整堂課的流程與節奏，這部分保持不變）",
  "quantitative_pacing_analysis": {
    "overall_rhythm": "（分析整堂課的教學節奏。例如：『本堂課呈現約20-30分鐘為一週期的「高-低認知負荷」循環模式。』）",
    "avg_high_load_duration": "（估算一個「高認知負荷」教學片段（如密集語法講解）的平均持續時間，直到學生的分心狀態開始顯著上升為止。例如：『數據顯示，在持續約 7-10 分鐘的高語速、高密度概念講解後，學生的分心比例平均會上升超過15%。』）",
    "attention_reset_patterns": "（分析老師是如何成功「重置」學生注意力的。例如：『在高負荷教學後，老師通常採用平均長度為 1-2 分鐘的「低負荷個人故事」或「明確的互動提問」來作為認知緩衝，這類緩衝有 80% 的機率在隨後的 5 分鐘內將「任務導向專注」拉回高峰。』）"
  },
  "key_pattern_identification": {
    "focus_booster_patterns": [
      {
        "pattern_name": "模式一：指令式任務轉換",
        "description": "（描述這個模式。例如：『在一段發散的閒聊或故事分享後，老師使用明確的、帶有動詞的指令（如「來，寫一下」、「請圈起來」）將學生的注意力強制拉回到具體任務上。』）",
        "evidence": "（從 `micro_event_summary` 和 `timeline_analysis` 中引用1-2個具體證據。例如：『此模式在 2:14:00 的「關鍵拉升點」最為典型，老師在歷史講解後說出「我們來看這邊」，任務導向專注度在30秒內急遽上升 48.5%。』）"
      },
      {
        "pattern_name": "模式二：結構化提問檢查",
        "description": "（例如：『在進行一段概念講解後，老師會立即提出一個封閉式或半開放式的問題（如「...是什麼時候？」、「...是什麼意思？」）來檢查學生的理解程度並促進思考。』）",
        "evidence": "（例如：『在 2:01:00，老師提問「憲法會保障什麼？」，隨後學生的任務導向專注（查找課本、筆記）顯著增加。』）"
      }
    ],
    "attention_sink_patterns": [
      {
        "pattern_name": "模式一：脫節的長篇敘事",
        "description": "（例如：『老師分享的個人故事或生活經驗，雖然有趣，但與當前的教學主題（如語法、歷史事件）缺乏明確的關聯，且持續時間超過 2 分鐘。』）",
        "evidence": "（例如：『此模式在 0:28:00（軍中CPR故事）和 1:09:00（拇指姑娘故事）最為明顯，這兩個時段的分心狀態比例均超過了 60%。』）"
      }
    ]
  },
  "lexical_trigger_highlights": {
      "positive_trigger_words": "（分析所有「專注提升點」的教師話語，總結出最常出現的、能觸發專注的詞語類型。例如：『祈使句動詞（如「看」、「寫」、「找」）、疑問詞（如「什麼」、「為什麼」）、強調詞（如「注意」、「關鍵是」）。』）",
      "negative_trigger_words": "（分析所有「專注下降點」的教師話語，總結出最常與分心相關的詞語類型。例如：『第一人稱代詞（如「我以前」、「我想說」）、無明確教學目的的重複性詞語、與主題無關的名詞。』）"
  },
  "final_recommendations": [
    "（基於以上所有量化和模式分析，提供 2-3 條更具體、更數據驅動的教學建議。例如：『建議將高認知負荷的純講解控制在 8 分鐘以內，並在其後設計一個包含「祈使句動詞」的互動任務來重置學生專注度。』）",
    "（例如：『在分享個人故事時，嘗試在 1 分鐘內將其與教學要點進行類比或連結，以避免學生的「接收性專注」滑向「分心狀態」。』）"
  ]
}
"""

    return f"""
你是一位頂尖的教育數據科學家與教學設計分析師。你的任務是接收一份已經初步處理好的課堂綜合分析報告，並對其進行深度的、量化的二次分析，以挖掘出可複製的教學模式與細顆粒度的因果關係。

**【你的輸入資料】**
一個 JSON 物件，包含了整堂課的 `overall_summary` (初步總結), `micro_event_summary` (所有專注度高峰、低谷、轉折點), 以及 `timeline_analysis` (包含每個時間區塊詳細的專注度分佈和認知負荷指標)。

**【★★★ 你的核心任務 ★★★】**
你的輸出**不再是簡單的總結**，而是一份數據驅動的**模式分析報告**。你必須完成以下四個層面的深度分析：

1.  **量化教學節奏分析 (`quantitative_pacing_analysis`)**:
    -   分析整堂課的「認知負荷」與「專注度分佈」時間序列數據。
    -   找出老師教學節奏的內在週期性。估算高強度教學的「安全時長」，以及什麼樣的活動能最有效地「重置」學生注意力。

2.  **關鍵教學模式識別 (`key_pattern_identification`)**:
    -   聚焦於 `micro_event_summary` 中標記的「關鍵拉升點」和「關鍵下跌點」。
    -   將這些孤立的事件歸納為可重複的「模式 (Pattern)」。定義出最有效的幾種「專注度助推器 (Focus Booster)」模式和最需要警惕的「專注度陷阱 (Attention Sink)」模式，並用數據證據支撐你的結論。

3.  **詞彙觸發點分析 (`lexical_trigger_highlights`)**:
    -   這是一個細顆粒度的語言分析。研究所有關鍵轉折點的 `teacher_speech_context`。
    -   總結出哪些**類型的詞彙或句式**最常與專注度上升或下降相關聯。

4.  **數據驅動的建議 (`final_recommendations`)**:
    -   基於以上所有分析，提出超越「多互動、少閒聊」這種通用建議的、具有**量化指標**和**模式指導**的具體教學策略。

**【你的輸出格式要求】**
你必須嚴格遵循以下的JSON結構，不包含任何JSON格式以外的文字或註解。你的分析必須深入、具體，並大量引用輸入數據作為證據。

{json_structure}

**【以下是提供給你進行深度分析的初步報告數據】**
{report_in_progress_str}
"""

async def analyze_chunk_with_ai_async(chunk, index, total, retries=3):
    """(v5.4 非同步版本) 調用 Azure OpenAI 分析單個區塊"""
    if not chunk:
        print(f"  - ⚠️ 區塊 {index + 1}/{total} 為空，跳過分析。")
        return None

    chunk_start_time = chunk[0].get('start_time')
    chunk_end_time = chunk[-1].get('end_time')
    
    # get_chunk_analysis_prompt 函式本身是同步的，無需修改
    system_prompt = get_chunk_analysis_prompt(chunk_start_time, chunk_end_time)
    user_prompt = f"請分析以下課堂數據區塊：\n\n{json.dumps(chunk, ensure_ascii=False, indent=2)}"
    
    print(f"  - 正在發起區塊 {index + 1}/{total} 的分析請求...")
    raw_content = ""

    for attempt in range(retries):
        try:
            response = await async_client.chat.completions.create(
                model=TEXT_DEPLOYMENT_NAME,
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                max_tokens=8192,
                temperature=0.1,
            )
            raw_content = response.choices[0].message.content
            return json.loads(raw_content)

        except json.JSONDecodeError as json_err:
            print(f"  - ⚠️ 區塊 {index + 1} 的 AI 返回 JSON 格式有誤 ({json_err})。正在嘗試自動修復...")
            fix_prompt = f"你是一個 JSON 格式修復工具...（省略以保持簡潔）...這是損壞的文字：\n{raw_content}"
            try:
                fix_response = await async_client.chat.completions.create(
                    model=TEXT_DEPLOYMENT_NAME,
                    response_format={"type": "json_object"},
                    messages=[{"role": "system", "content": "你是一個JSON格式修復工具。"}, {"role": "user", "content": fix_prompt}],
                    max_tokens=8192, temperature=0.0,
                )
                fixed_content = fix_response.choices[0].message.content
                print(f"  - ✅ 區塊 {index + 1} JSON 已自動修復。")
                return json.loads(fixed_content)
            except Exception as fix_e:
                print(f"  - ❌ 區塊 {index + 1} 自動修復失敗: {fix_e}")
                if attempt >= retries - 1: break # 如果修復也失敗，直接進入下一次重試或結束
        
        except RateLimitError as e:
            wait_time = (2 ** attempt) + random.random()
            print(f"  - ⚠️ 區塊 {index + 1} 觸發速率限制 (嘗試 {attempt + 1}/{retries})，將在 {wait_time:.1f} 秒後重試...")
            await asyncio.sleep(wait_time)
        
        except Exception as e:
            print(f"❌ 分析區塊 {index + 1} 時發生未知錯誤: {e}")
            if attempt < retries - 1: await asyncio.sleep(5)
    
    print(f"  - ❌ 區塊 {index + 1}/{total} 在所有重試後分析失敗。")
    return None # 返回 None 表示失敗

async def summarize_chunks_with_ai_async(report_in_progress, retries=3):
    """(v5.5 穩定版) 調用 AI 進行最終的深度模式分析，並加入JSON清洗邏輯"""
    print("步驟 7/8: 調用 AI 進行最終的深度模式分析...")
    report_in_progress_str = json.dumps(report_in_progress, ensure_ascii=False, indent=2)
    
    # get_summary_prompt 函式本身是同步的，無需修改
    system_prompt = get_summary_prompt(report_in_progress_str)
    
    for attempt in range(retries):
        try:
            start_time = time.time()
            response = await async_client.chat.completions.create(
                model=TEXT_DEPLOYMENT_NAME,
                messages=[{"role": "system", "content": system_prompt}],
                max_tokens=8192,
                temperature=0.2
            )
            end_time = time.time()
            print(f"✅ AI 深度分析完成，耗時 {end_time - start_time:.2f} 秒。")

            raw_content = response.choices[0].message.content
            
            # 從 markdown 中提取 JSON (保持不變)
            json_match = re.search(r'```json\s*([\s\S]*?)\s*```', raw_content, re.DOTALL)
            json_content_str = json_match.group(1) if json_match else raw_content

            # ★★★ V5.5 核心修改點：在解析前進行清洗 ★★★
            cleaned_json_str = clean_json_string(json_content_str)

            # 使用清洗後的字符串進行解析
            return json.loads(cleaned_json_str)

        except json.JSONDecodeError as e:
            # 這裡的錯誤日誌現在更有價值，因為我們可以知道清洗是否失敗
            print(f"  - ⚠️ 最終分析返回的 JSON 格式錯誤 (嘗試 {attempt + 1}/{retries})，即使在清洗後仍然解析失敗: {e}")
            if attempt < retries - 1:
                print("     - 61 秒後重試...")
                await asyncio.sleep(61)
            else:
                print(f"❌ JSON 解析失敗，已達最大重試次數。")
                raise e # 向上拋出錯誤
        except RateLimitError as e:
            wait_time = (2 ** attempt) + random.random()
            print(f"  - ⚠️ 觸發速率限制，將在 {wait_time:.1f} 秒後重試...")
            await asyncio.sleep(wait_time)
        except Exception as e:
            print(f"❌ 匯總分析時發生未知錯誤: {e}")
            if attempt < retries - 1:
                await asyncio.sleep(5)
            else:
                raise e # 如果所有重試都失敗，則拋出異常
            

# ==============================================================================
# --- 主執行流程 (v5.10 - 整合語速趨勢分析) ---
# ==============================================================================
async def main():
    start_total_time = time.time()
    
    # 步驟 1-4: 載入與預處理數據
    student_behaviors, total_seconds = load_student_data(STUDENT_DATA_DIRECTORY, TARGET_SESSION_TIME)
    if student_behaviors is None: exit()
    raw_transcript = load_teacher_transcript(TEACHER_TRANSCRIPT_PATH)
    blackboard_images = load_blackboard_images(BLACKBOARD_IMAGES_DIRECTORY)
    transcript = await correct_transcript_with_ai_async(raw_transcript)

    # 步驟 5: 核心數據處理與標記
    master_timeline = create_master_timeline(student_behaviors, transcript, blackboard_images, total_seconds, TIME_INTERVAL_SECONDS)
    micro_events = find_micro_events(master_timeline)
    aggregated_timeline = aggregate_timeline(master_timeline, aggregate_interval_minutes=5)
    
    # (v5.8) 量化教學模式
    teaching_mode_summary = calculate_mode_durations(master_timeline, total_seconds, TIME_INTERVAL_SECONDS)
    
    # (v5.9) 標記關鍵字
    keyword_timestamps = find_important_keywords(master_timeline)

    # (v5.9) 識別教學週期
    teaching_cycle_analysis = identify_teaching_cycles(master_timeline)

    # (v5.10) 分析語速趨勢
    speech_rate_trend_analysis = analyze_speech_rate_trend(master_timeline)
    
    # 步驟 6: AI 分塊分析
    print(f"步驟 6/8: 開始分塊處理時間軸...")
    intervals_per_chunk = (CHUNK_SIZE_MINUTES * 60) // TIME_INTERVAL_SECONDS
    chunks = [master_timeline[i:i + intervals_per_chunk] for i in range(0, len(master_timeline), intervals_per_chunk)]
    analysis_tasks = [analyze_chunk_with_ai_async(chunk, i, len(chunks)) for i, chunk in enumerate(chunks)]
    
    # (v5.9) 找出並準備分析關鍵高強度片段 (7-12分鐘)
    critical_segments_for_analysis = []
    print("步驟 6.5/8: 尋找關鍵高強度教學片段 (7-12分鐘) 進行微觀分析...")
    current_hi_streak = 0
    segment_start_index = -1
    for i, interval in enumerate(master_timeline):
        if interval.get("teaching_mode") == "高強度講解":
            if current_hi_streak == 0:
                segment_start_index = i
            current_hi_streak += 1
        else:
            duration_seconds = current_hi_streak * TIME_INTERVAL_SECONDS
            if 420 <= duration_seconds <= 720: # 7-12 分鐘
                critical_segments_for_analysis.append(master_timeline[segment_start_index:i])
            current_hi_streak = 0
    if current_hi_streak > 0: # 處理結尾的片段
        duration_seconds = current_hi_streak * TIME_INTERVAL_SECONDS
        if 420 <= duration_seconds <= 720:
            critical_segments_for_analysis.append(master_timeline[segment_start_index:])
    
    critical_analysis_tasks = []
    if critical_segments_for_analysis:
        print(f"  - 找到 {len(critical_segments_for_analysis)} 個關鍵片段，加入並行分析任務。")
        critical_analysis_tasks = [analyze_critical_segment_with_ai_async(seg, i, len(critical_segments_for_analysis)) for i, seg in enumerate(critical_segments_for_analysis)]
        
    # 並行執行所有分析 (包括分塊分析和關鍵片段分析)
    all_tasks = analysis_tasks + critical_analysis_tasks
    all_analyses_results = await asyncio.gather(*all_tasks)
    
    # 分離出不同類型的分析結果
    all_chunk_analyses = [res for res in all_analyses_results[:len(chunks)] if res is not None]
    critical_segment_analyses = [res for res in all_analyses_results[len(chunks):] if res is not None]

    print(f"✅ 所有 {len(all_chunk_analyses)} 個區塊初步分析完成。")
    if critical_segment_analyses:
        print(f"✅ 所有 {len(critical_segment_analyses)} 個關鍵片段微觀分析完成。")

    # 步驟 7 & 8: 最終匯總與儲存
    try:
        if all_chunk_analyses:
            report_in_progress = {
                "micro_event_summary": micro_events,
                "timeline_analysis": all_chunk_analyses
            }
            final_deep_analysis = await summarize_chunks_with_ai_async(report_in_progress)
            
            if final_deep_analysis:
                final_report = {
                    "class_session_id": f"{TARGET_SESSION_TIME.replace('/', '')}_english_class",
                    "overall_summary": final_deep_analysis,
                    "quantitative_summary": {
                        "teaching_mode_distribution": teaching_mode_summary,
                        "teaching_cycle_analysis": teaching_cycle_analysis,
                        "speech_rate_trend_analysis": speech_rate_trend_analysis,
                    },
                    "keyword_timestamps": keyword_timestamps,
                    "critical_segment_analysis": critical_segment_analyses,
                    "micro_event_summary": micro_events,
                    "timeline_analysis": all_chunk_analyses,
                    "detailed_timeline": master_timeline,
                    "aggregated_timeline": aggregated_timeline 
                }
                
                print("步驟 8/8: 儲存最終深度分析報告...")
                os.makedirs(OUTPUT_ANALYSIS_DIRECTORY, exist_ok=True)
                output_path_v5_10 = os.path.join(OUTPUT_ANALYSIS_DIRECTORY, f'classroom_analysis_report_{TARGET_SESSION_TIME.replace("/", "")}.json')

                with open(output_path_v5_10, 'w', encoding='utf-8') as f:
                    json.dump(final_report, f, ensure_ascii=False, indent=2)
                
                end_total_time = time.time()
                total_duration = end_total_time - start_total_time

                print("-" * 50)
                print(f"🎉🎉🎉 全部分析完成！(v5.10 語速趨勢分析版) 🎉🎉🎉")
                print(f"⏱️ 總耗時: {total_duration // 60:.0f} 分 {total_duration % 60:.2f} 秒。")
                print(f"✅ 綜合分析報告已成功儲存至: {output_path_v5_10}")
                print("-" * 50)
            else:
                 print("❌ AI 最終深度分析失敗，未生成報告檔案。")
        else:
            print("❌ 所有區塊均分析失敗，未生成任何報告檔案。")

    except Exception as e:
        print(f"\n❌ 在主流程中發生嚴重錯誤: {e}")
        print("❌ 程式已終止。")

await main()

### 教室專注度量化數據提取

In [ ]:
# -*- coding: utf-8 -*-
import os
import json
import pandas as pd
import numpy as np
import glob

# ==============================================================================
# --- ★★★【輔助函式區塊 (v3.2 整合版)】★★★ ---
# ==============================================================================

def safe_get(data_dict, key_list, default=None):
    """安全地從巢狀字典中取值。"""
    for key in key_list:
        if isinstance(data_dict, dict):
            data_dict = data_dict.get(key)
        else:
            return default
    return data_dict if data_dict is not None else default

def seconds_to_time_str(seconds):
    """將總秒數轉換回 'HH:MM:SS' 格式的字串。"""
    if seconds is None: return ''
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02}:{minutes:02}:{secs:02}"

def time_str_to_seconds(time_str):
    """將 'HH:MM:SS' 格式轉換為總秒數。"""
    if not time_str: return 0
    try:
        parts = list(map(int, time_str.split(':')))
        if len(parts) == 3: return parts[0] * 3600 + parts[1] * 60 + parts[2]
        if len(parts) == 2: return parts[0] * 60 + parts[1]
        return 0
    except (ValueError, TypeError):
        return 0

# --- 【新增】輔助函式，用於計算單一時間點的專注度分數 ---
def get_focus_score(item):
    """計算單一30秒區間的總專注度分數 (主動+被動)。"""
    dist = safe_get(item, ['focus_distribution'], {})
    active = dist.get('active_behavioral_engagement', dist.get('task_oriented_focus', 0))
    passive = dist.get('passive_behavioral_engagement', dist.get('receptive_engagement', 0))
    return (active or 0) + (passive or 0)

def get_focus_trend(segment_intervals):
    """計算並判斷一個區段內專注度的趨勢。"""
    if len(segment_intervals) < 2: return '平穩'

    start_focus = get_focus_score(segment_intervals[0])
    end_focus = get_focus_score(segment_intervals[-1])
    change = end_focus - start_focus

    if change > 15: return '顯著上升'
    elif change > 5: return '微幅上升'
    elif change < -15: return '顯著下降'
    elif change < -5: return '微幅下降'
    else: return '平穩'

# ==============================================================================
# --- ★★★【核心邏輯區塊 (v3.2)】★★★ ---
# ==============================================================================

def analyze_dynamic_segment(segment_intervals, class_id, positive_triggers, timeline_analysis):
    """
    【修改】對動態區段進行詳細指標計算，並加入新指標。
    """
    start_interval = segment_intervals[0]
    end_interval = segment_intervals[-1]
    
    start_sec = start_interval.get('seconds', 0)
    end_sec = end_interval.get('seconds', 0) + 30
    duration_sec = end_sec - start_sec

    # --- 1. 教學行為量化指標 (原有) ---
    mode = start_interval.get('teaching_mode', '未知')
    wpm_scores = [safe_get(item, ['cognitive_load_metrics', 'speech_rate_wpm']) for item in segment_intervals if item.get('cognitive_load_metrics')]
    wpm_valid = [wpm for wpm in wpm_scores if wpm is not None and wpm > 0]
    avg_wpm = np.mean(wpm_valid) if wpm_valid else 0

    duration_min = duration_sec / 60.0
    directive_count = sum(sum(item.get('teacher_speech', '').count(word) for word in positive_triggers) for item in segment_intervals)
    question_count = sum(safe_get(item, ['cognitive_load_metrics', 'question_frequency'], 0) for item in segment_intervals)
    
    directive_density = (directive_count / duration_min) if duration_min > 0 else 0
    question_frequency = (question_count / duration_min) if duration_min > 0 else 0

    # --- 2. 學生狀態量化指標 (原有) ---
    def get_avg_focus(key1, key2):
        scores = [safe_get(item, ['focus_distribution', key1], safe_get(item, ['focus_distribution', key2], 0)) for item in segment_intervals]
        valid_scores = [s for s in scores if s is not None]
        return np.mean(valid_scores) if valid_scores else 0

    avg_active = get_avg_focus('active_behavioral_engagement', 'task_oriented_focus')
    avg_passive = get_avg_focus('passive_behavioral_engagement', 'receptive_engagement')
    avg_disengaged = get_avg_focus('behavioral_disengagement', 'disengagement')
    focus_trend = get_focus_trend(segment_intervals)
    
    # --- 3. 質化描述繼承 (原有) ---
    parent_segment = None
    for summary_segment in timeline_analysis:
        summary_start_sec = time_str_to_seconds(summary_segment.get('start_time'))
        summary_end_sec = time_str_to_seconds(summary_segment.get('end_time'))
        if summary_start_sec <= start_sec < summary_end_sec:
            parent_segment = summary_segment
            break
            
    if parent_segment:
        event_title = parent_segment.get('event_title', 'N/A')
        pos_trigger = safe_get(parent_segment, ['focus_trigger_analysis', 'positive_trigger'], {})
        neg_trigger = safe_get(parent_segment, ['focus_trigger_analysis', 'negative_trigger'], {})
    else:
        event_title, pos_trigger, neg_trigger = '未找到對應摘要', {}, {}

    # --- 4. 【新增指標】閒聊關聯度分析 ---
    chat_relevance = '非閒聊區段'
    if mode == '閒聊':
        neg_topic = safe_get(neg_trigger, ['trigger_snippet', 'topic'], '')
        if '個人故事' in neg_topic or '脫節' in neg_topic:
            chat_relevance = '無關閒聊'
        else:
            chat_relevance = '相關或中性閒聊'

    # --- 5. 【新增指標】學生具體行為追蹤 ---
    sloucher_counts = []
    fidgeter_counts = []
    behavior_diversities = []
    for item in segment_intervals:
        behaviors = item.get('student_behavior_summary', {})
        sloucher_counts.append(len(behaviors.get('趴睡', [])))
        fidgeter_counts.append(len(behaviors.get('玩弄手部/文具', [])))
        behavior_diversities.append(len(behaviors.keys()))

    avg_slouchers_count = np.mean(sloucher_counts) if sloucher_counts else 0
    avg_fidgeters_count = np.mean(fidgeter_counts) if fidgeter_counts else 0
    avg_behavior_diversity = np.mean(behavior_diversities) if behavior_diversities else 0

    # --- 6. 組裝所有數據 ---
    return {
        '課堂ID': class_id,
        '開始時間': seconds_to_time_str(start_sec),
        '結束時間': seconds_to_time_str(end_sec),
        '區段主要教學模式': mode,
        '區段時長(秒)': duration_sec,
        '區段主要教學活動': event_title,
        '區段平均語速(WPM)': round(avg_wpm, 2),
        '指令詞彙密度(次/分)': round(directive_density, 2),
        '平均提問頻率(次/分)': round(question_frequency, 2),
        '平均主動參與度(%)': round(avg_active, 2),
        '平均被動參與度(%)': round(avg_passive, 2),
        '平均行為分心度(%)': round(avg_disengaged, 2),
        '專注度趨勢': focus_trend,
        '閒聊內容關聯度': chat_relevance,  # 新增
        '平均趴睡學生數': round(avg_slouchers_count, 2), # 新增
        '平均玩筆學生數': round(avg_fidgeters_count, 2), # 新增
        '平均行為多樣性': round(avg_behavior_diversity, 2), # 新增
        '包含專注提升點': '是' if pos_trigger else '否',
        '提升點觸發模式': safe_get(pos_trigger, ['trigger_snippet', 'topic'], ''),
        '包含專注下降點': '是' if neg_trigger else '否',
        '下降點觸發模式': safe_get(neg_trigger, ['trigger_snippet', 'topic'], '')
    }

def process_timeline_dynamically(class_id, detailed_timeline, timeline_analysis, positive_triggers):
    """(原有函式) 將 detailed_timeline 動態切分並分析。"""
    if not detailed_timeline: return []
    segments, current_segment_intervals, current_mode = [], [], None
    for interval in detailed_timeline:
        mode = interval.get('teaching_mode', '未知')
        if current_mode is not None and mode != current_mode:
            segments.append(analyze_dynamic_segment(current_segment_intervals, class_id, positive_triggers, timeline_analysis))
            current_segment_intervals = []
        current_segment_intervals.append(interval)
        current_mode = mode
    if current_segment_intervals:
        segments.append(analyze_dynamic_segment(current_segment_intervals, class_id, positive_triggers, timeline_analysis))
    return segments

# --- 【新增函式】宏觀課堂結構分析 ---
def analyze_class_summary(data):
    """從 quantitative_summary 提取全局數據。"""
    class_id = safe_get(data, ['class_session_id'], 'unknown_class')
    summary_data = { '課堂ID': class_id }

    # 1. 教學模式分佈
    dist = safe_get(data, ['quantitative_summary', 'teaching_mode_distribution'], {})
    for mode, values in dist.items():
        summary_data[f'模式_{mode}_佔比(%)'] = values.get('percentage_of_class')
        summary_data[f'模式_{mode}_平均時長(秒)'] = values.get('average_segment_duration_seconds')

    # 2. 語速趨勢
    trend = safe_get(data, ['quantitative_summary', 'speech_rate_trend_analysis'], {})
    summary_data['語速趨勢_斜率(WPM/分)'] = round(safe_get(trend, ['linear_regression_stats', 'slope_per_minute'], 0), 2)
    summary_data['語速趨勢_下半場變化(%)'] = safe_get(trend, ['comparative_summary', 'change_percentage'])

    return summary_data

# --- 【新增函式】關鍵詞即時影響分析 ---
def analyze_keyword_impact(class_id, detailed_timeline, keyword_timestamps):
    """分析關鍵詞觸發前後的專注度變化。"""
    if not keyword_timestamps or not detailed_timeline:
        return []

    # 為了快速查找，先將 timeline 轉為字典
    timeline_map = {item['seconds']: item for item in detailed_timeline}
    impact_results = []

    for kw_event in keyword_timestamps:
        kw_time_str = kw_event.get('time')
        kw_seconds = time_str_to_seconds(kw_time_str)
        
        # 找到關鍵詞所在以及之前的區間
        current_interval = timeline_map.get(kw_seconds)
        previous_interval = timeline_map.get(kw_seconds - 30)
        
        if not current_interval or not previous_interval:
            continue # 如果找不到前後區間，則跳過

        focus_before = get_focus_score(previous_interval)
        focus_after = get_focus_score(current_interval)
        
        impact_results.append({
            '課堂ID': class_id,
            '觸發詞': kw_event.get('keyword'),
            '觸發時間點': kw_time_str,
            '觸發前專注度': round(focus_before, 2),
            '觸發後專注度': round(focus_after, 2),
            '專注度變化': round(focus_after - focus_before, 2)
        })
    return impact_results

def generate_reports():
    """【主函數修改】執行所有操作並生成多個報告。"""
    input_directory = r"C:\Users\User\Desktop\test\classroom_analysis_report"
    output_directory = r"C:\Users\User\Desktop\test\classroom_analysis_report"
    
    # --- 【修改】定義多個輸出檔案名稱 ---
    output_dynamic_segments_file = "report_1_dynamic_segments.csv"
    output_class_summary_file = "report_2_class_summary.csv"
    output_keyword_impact_file = "report_3_keyword_impact.csv"
    
    print("--- 課堂洞察報告生成器 v3.2 (增強版) ---")
    print(f"1. 開始掃描資料夾：{input_directory}")
    
    json_files = glob.glob(os.path.join(input_directory, "classroom_analysis_report_*.json"))
    if not json_files:
        print(f"❌ 錯誤：找不到任何 'classroom_analysis_report_*.json' 檔案。")
        return

    print(f"✅ 找到 {len(json_files)} 份報告，準備進行多維度數據提取...")
    
    all_dynamic_segments = []
    all_class_summaries = []
    all_keyword_impacts = []

    for file_path in json_files:
        filename = os.path.basename(file_path)
        print(f"\n... 正在處理檔案: {filename}")
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            class_id = safe_get(data, ['class_session_id'], 'unknown_class')
            detailed_timeline = data.get('detailed_timeline', [])
            timeline_analysis = data.get('timeline_analysis', [])
            keyword_timestamps = data.get('keyword_timestamps', []) # 新增讀取
            
            positive_triggers_str = safe_get(data, ['overall_summary', 'lexical_trigger_highlights', 'positive_trigger_words'], '')
            positive_triggers = [word.strip().replace("』", "").replace("『", "") for word in positive_triggers_str.split('、')]
            
            if not detailed_timeline:
                print(f"⚠️ 警告：檔案 '{filename}' 缺少 'detailed_timeline' 數據，部分分析將跳過。")
                continue

            # --- 執行三種不同的分析 ---
            all_dynamic_segments.extend(process_timeline_dynamically(class_id, detailed_timeline, timeline_analysis, positive_triggers))
            all_class_summaries.append(analyze_class_summary(data))
            all_keyword_impacts.extend(analyze_keyword_impact(class_id, detailed_timeline, keyword_timestamps))
            
        except Exception as e:
            print(f"⚠️ 警告：處理檔案 '{filename}' 時發生嚴重錯誤: {e}。")

    # --- 儲存第一個報告：動態區段分析 ---
    if all_dynamic_segments:
        print("\n2. [報告1] 動態區段分析完成，正在生成 CSV...")
        df_segments = pd.DataFrame(all_dynamic_segments)
        column_order = [
            '課堂ID', '開始時間', '結束時間', '區段主要教學模式', '區段時長(秒)',
            '區段主要教學活動', '區段平均語速(WPM)', '指令詞彙密度(次/分)', '平均提問頻率(次/分)',
            '平均主動參與度(%)', '平均被動參與度(%)', '平均行為分心度(%)', '專注度趨勢',
            '閒聊內容關聯度', '平均趴睡學生數', '平均玩筆學生數', '平均行為多樣性',
            '包含專注提升點', '提升點觸發模式', '包含專注下降點', '下降點觸發模式'
        ]
        df_segments = df_segments.reindex(columns=column_order)
        output_path = os.path.join(output_directory, output_dynamic_segments_file)
        df_segments.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"✅ 報告1 已儲存至：{output_path}")

    # --- 儲存第二個報告：宏觀結構總結 ---
    if all_class_summaries:
        print("\n3. [報告2] 宏觀結構分析完成，正在生成 CSV...")
        df_summary = pd.DataFrame(all_class_summaries)
        output_path = os.path.join(output_directory, output_class_summary_file)
        df_summary.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"✅ 報告2 已儲存至：{output_path}")

    # --- 儲存第三個報告：關鍵詞影響分析 ---
    if all_keyword_impacts:
        print("\n4. [報告3] 關鍵詞影響分析完成，正在生成 CSV...")
        df_keywords = pd.DataFrame(all_keyword_impacts)
        output_path = os.path.join(output_directory, output_keyword_impact_file)
        df_keywords.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"✅ 報告3 已儲存至：{output_path}")

    print("-" * 50)
    print(f"🎉🎉🎉 所有報告生成成功！ 🎉🎉🎉")
    print("-" * 50)

if __name__ == '__main__':
    generate_reports()

In [ ]:
# -*- coding: utf-8 -*-
import os
import json
import pandas as pd
import numpy as np
import glob

# ==============================================================================
# --- ★★★【輔助函式區塊 (v3.3 整合版)】★★★ ---
# ==============================================================================

def safe_get(data_dict, key_list, default=None):
    """安全地從巢狀字典中取值。"""
    for key in key_list:
        if isinstance(data_dict, dict):
            data_dict = data_dict.get(key)
        else:
            return default
    return data_dict if data_dict is not None else default

def seconds_to_time_str(seconds):
    """將總秒數轉換回 'HH:MM:SS' 格式的字串。"""
    if seconds is None or np.isnan(seconds): return '' # --- 【修改】增加對 NaN 值的處理 ---
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02}:{minutes:02}:{secs:02}"

def time_str_to_seconds(time_str):
    """將 'HH:MM:SS' 格式轉換為總秒數。"""
    if not time_str: return 0
    try:
        parts = list(map(int, time_str.split(':')))
        if len(parts) == 3: return parts[0] * 3600 + parts[1] * 60 + parts[2]
        if len(parts) == 2: return parts[0] * 60 + parts[1]
        return 0
    except (ValueError, TypeError):
        return 0

def get_focus_score(item):
    """計算單一30秒區間的總專注度分數 (主動+被動)。"""
    dist = safe_get(item, ['focus_distribution'], {})
    active = dist.get('active_behavioral_engagement', dist.get('task_oriented_focus', 0))
    passive = dist.get('passive_behavioral_engagement', dist.get('receptive_engagement', 0))
    return (active or 0) + (passive or 0)

def get_avg_focus_from_intervals(intervals, key1, key2): # --- 【新增】將平均專注度計算獨立出來以便複用 ---
    """從給定的時間區間列表中計算平均專注度。"""
    scores = [safe_get(item, ['focus_distribution', key1], safe_get(item, ['focus_distribution', key2], 0)) for item in intervals]
    valid_scores = [s for s in scores if s is not None]
    return np.mean(valid_scores) if valid_scores else 0

def get_focus_trend(segment_intervals):
    """計算並判斷一個區段內專注度的趨勢。"""
    if len(segment_intervals) < 2: return '平穩'
    start_focus = get_focus_score(segment_intervals[0])
    end_focus = get_focus_score(segment_intervals[-1])
    change = end_focus - start_focus
    if change > 15: return '顯著上升'
    elif change > 5: return '微幅上升'
    elif change < -15: return '顯著下降'
    elif change < -5: return '微幅下降'
    else: return '平穩'

# ==============================================================================
# --- ★★★【核心邏輯區塊 (v3.3)】★★★ ---
# ==============================================================================

def analyze_dynamic_segment(segment_intervals, class_id, positive_triggers, negative_triggers, timeline_analysis): # --- 【修改】增加 negative_triggers 參數 ---
    """
    【修改】對動態區段進行詳細指標計算，並加入負面詞彙與板書指標。
    """
    start_interval = segment_intervals[0]
    end_interval = segment_intervals[-1]
    
    start_sec = start_interval.get('seconds', 0)
    end_sec = end_interval.get('seconds', 0) + 30
    duration_sec = end_sec - start_sec

    # --- 1. 教學行為量化指標 (原有+新增) ---
    mode = start_interval.get('teaching_mode', '未知')
    wpm_scores = [safe_get(item, ['cognitive_load_metrics', 'speech_rate_wpm']) for item in segment_intervals if item.get('cognitive_load_metrics')]
    wpm_valid = [wpm for wpm in wpm_scores if wpm is not None and wpm > 0]
    avg_wpm = np.mean(wpm_valid) if wpm_valid else 0
    duration_min = duration_sec / 60.0

    # 指令詞與提問
    positive_trigger_count = sum(sum(item.get('teacher_speech', '').count(word) for word in positive_triggers) for item in segment_intervals)
    question_count = sum(safe_get(item, ['cognitive_load_metrics', 'question_frequency'], 0) for item in segment_intervals)
    positive_trigger_density = (positive_trigger_count / duration_min) if duration_min > 0 else 0
    question_frequency = (question_count / duration_min) if duration_min > 0 else 0

    # --- 【新增】負面詞彙密度 ---
    negative_trigger_count = sum(sum(item.get('teacher_speech', '').count(word) for word in negative_triggers) for item in segment_intervals)
    negative_trigger_density = (negative_trigger_count / duration_min) if duration_min > 0 else 0

    # --- 2. 學生狀態量化指標 (原有+新增) ---
    avg_active = get_avg_focus_from_intervals(segment_intervals, 'active_behavioral_engagement', 'task_oriented_focus')
    avg_passive = get_avg_focus_from_intervals(segment_intervals, 'passive_behavioral_engagement', 'receptive_engagement')
    avg_disengaged = get_avg_focus_from_intervals(segment_intervals, 'behavioral_disengagement', 'disengagement')
    focus_trend = get_focus_trend(segment_intervals)
    
    # --- 3. 質化描述繼承 (原有) ---
    parent_segment = next((seg for seg in timeline_analysis if time_str_to_seconds(seg.get('start_time')) <= start_sec < time_str_to_seconds(seg.get('end_time'))), None)
    event_title = parent_segment.get('event_title', '未找到對應摘要') if parent_segment else '未找到對應摘要'
    pos_trigger = safe_get(parent_segment, ['focus_trigger_analysis', 'positive_trigger'], {}) if parent_segment else {}
    neg_trigger = safe_get(parent_segment, ['focus_trigger_analysis', 'negative_trigger'], {}) if parent_segment else {}

    # --- 4. 閒聊關聯度分析 (原有) ---
    chat_relevance = '非閒聊區段'
    if mode == '閒聊':
        neg_topic = safe_get(neg_trigger, ['trigger_snippet', 'topic'], '')
        chat_relevance = '無關閒聊' if '個人故事' in neg_topic or '脫節' in neg_topic else '相關或中性閒聊'

    # --- 5. 學生具體行為追蹤 (原有+新增) ---
    sloucher_counts, fidgeter_counts, behavior_diversities, blackboard_counts = [], [], [], []
    for item in segment_intervals:
        behaviors = item.get('student_behavior_summary', {})
        sloucher_counts.append(len(behaviors.get('趴睡', [])))
        fidgeter_counts.append(len(behaviors.get('玩弄手部/文具', [])))
        behavior_diversities.append(len(behaviors.keys()))
        blackboard_counts.append(len(item.get('blackboard_snapshots', []))) # --- 【新增】板書使用次數 ---

    avg_slouchers_count = np.mean(sloucher_counts) if sloucher_counts else 0
    avg_fidgeters_count = np.mean(fidgeter_counts) if fidgeter_counts else 0
    avg_behavior_diversity = np.mean(behavior_diversities) if behavior_diversities else 0
    avg_blackboard_usage = np.mean(blackboard_counts) if blackboard_counts else 0 # --- 【新增】平均板書使用 ---

    # --- 6. 組裝所有數據 ---
    return {
        '課堂ID': class_id,
        '開始時間': seconds_to_time_str(start_sec),
        '結束時間': seconds_to_time_str(end_sec),
        '區段主要教學模式': mode,
        '區段時長(秒)': duration_sec,
        '區段主要教學活動': event_title,
        '區段平均語速(WPM)': round(avg_wpm, 2),
        '正面指令密度(次/分)': round(positive_trigger_density, 2), # --- 【修改】欄位名稱更精確 ---
        '負面詞彙密度(次/分)': round(negative_trigger_density, 2), # --- 【新增】---
        '平均提問頻率(次/分)': round(question_frequency, 2),
        '板書使用頻率(次/30秒)': round(avg_blackboard_usage, 2), # --- 【新增】---
        '平均主動參與度(%)': round(avg_active, 2),
        '平均被動參與度(%)': round(avg_passive, 2),
        '平均行為分心度(%)': round(avg_disengaged, 2),
        '專注度趨勢': focus_trend,
        '閒聊內容關聯度': chat_relevance,
        '平均趴睡學生數': round(avg_slouchers_count, 2),
        '平均玩筆學生數': round(avg_fidgeters_count, 2),
        '平均行為多樣性': round(avg_behavior_diversity, 2),
        '包含專注提升點': '是' if pos_trigger else '否',
        '提升點觸發模式': safe_get(pos_trigger, ['trigger_snippet', 'topic'], ''),
        '包含專注下降點': '是' if neg_trigger else '否',
        '下降點觸發模式': safe_get(neg_trigger, ['trigger_snippet', 'topic'], '')
    }

def process_timeline_dynamically(class_id, detailed_timeline, timeline_analysis, positive_triggers, negative_triggers): # --- 【修改】增加 negative_triggers 參數 ---
    """(原有函式修改) 將 detailed_timeline 動態切分並分析。"""
    if not detailed_timeline: return []
    segments, current_segment_intervals, current_mode = [], [], None
    for interval in detailed_timeline:
        mode = interval.get('teaching_mode', '未知')
        if current_mode is not None and mode != current_mode:
            segments.append(analyze_dynamic_segment(current_segment_intervals, class_id, positive_triggers, negative_triggers, timeline_analysis)) # --- 【修改】傳入 negative_triggers ---
            current_segment_intervals = []
        current_segment_intervals.append(interval)
        current_mode = mode
    if current_segment_intervals:
        segments.append(analyze_dynamic_segment(current_segment_intervals, class_id, positive_triggers, negative_triggers, timeline_analysis)) # --- 【修改】傳入 negative_triggers ---
    return segments

def analyze_class_summary(data):
    """(原有函式) 從 quantitative_summary 提取全局數據。"""
    class_id = safe_get(data, ['class_session_id'], 'unknown_class')
    summary_data = { '課堂ID': class_id }
    dist = safe_get(data, ['quantitative_summary', 'teaching_mode_distribution'], {})
    for mode, values in dist.items():
        summary_data[f'模式_{mode}_佔比(%)'] = values.get('percentage_of_class')
        summary_data[f'模式_{mode}_平均時長(秒)'] = values.get('average_segment_duration_seconds')
    trend = safe_get(data, ['quantitative_summary', 'speech_rate_trend_analysis'], {})
    summary_data['語速趨勢_斜率(WPM/分)'] = round(safe_get(trend, ['linear_regression_stats', 'slope_per_minute'], 0), 2)
    summary_data['語速趨勢_下半場變化(%)'] = safe_get(trend, ['comparative_summary', 'change_percentage'])
    return summary_data

def analyze_keyword_impact(class_id, detailed_timeline, keyword_timestamps):
    """(原有函式) 分析關鍵詞觸發前後的專注度變化。"""
    if not keyword_timestamps or not detailed_timeline: return []
    timeline_map = {item['seconds']: item for item in detailed_timeline}
    impact_results = []
    for kw_event in keyword_timestamps:
        kw_time_str = kw_event.get('time')
        kw_seconds = time_str_to_seconds(kw_time_str)
        current_interval = timeline_map.get(kw_seconds)
        previous_interval = timeline_map.get(kw_seconds - 30)
        if not current_interval or not previous_interval: continue
        focus_before = get_focus_score(previous_interval)
        focus_after = get_focus_score(current_interval)
        impact_results.append({
            '課堂ID': class_id, '觸發詞': kw_event.get('keyword'), '觸發時間點': kw_time_str,
            '觸發前專注度': round(focus_before, 2), '觸發後專注度': round(focus_after, 2),
            '專注度變化': round(focus_after - focus_before, 2)
        })
    return impact_results

# --- 【新增函式】專注力臨界點分析 ---
def analyze_breaking_points(data):
    """分析高強度教學區段的專注力臨界點。"""
    class_id = safe_get(data, ['class_session_id'], 'unknown_class')
    breaking_point_data = safe_get(data, ['quantitative_summary', 'high_intensity_segment_analysis'], [])
    if not breaking_point_data: return []
    
    results = []
    for segment in breaking_point_data:
        if segment.get('breaking_point_time'): # 只分析有出現臨界點的區段
            results.append({
                '課堂ID': class_id,
                '高強度區段開始時間': segment.get('start_time'),
                '專注臨界點時間': segment.get('breaking_point_time'),
                '可持續時長(秒)': segment.get('duration_before_break_seconds'),
                '觸發原因': segment.get('trigger_reason'),
                '初始分心度(%)': segment.get('initial_disengagement')
            })
    return results

# --- 【新增函式】教學循環結構分析 ---
def analyze_teaching_cycles(data, detailed_timeline):
    """分析高低強度教學循環的結構與成效。"""
    class_id = safe_get(data, ['class_session_id'], 'unknown_class')
    cycles_data = safe_get(data, ['quantitative_summary', 'teaching_cycle_analysis', 'cycles'], [])
    if not cycles_data or not detailed_timeline: return []

    results = []
    for cycle in cycles_data:
        start_sec = cycle.get('start_seconds')
        end_time_str = cycle.get('end_time')
        end_sec = time_str_to_seconds(end_time_str)
        
        # 篩選出屬於該循環的時間區間
        cycle_intervals = [item for item in detailed_timeline if start_sec <= item['seconds'] < end_sec]
        if not cycle_intervals: continue

        # 計算該循環內的平均專注度
        avg_active = get_avg_focus_from_intervals(cycle_intervals, 'active_behavioral_engagement', 'task_oriented_focus')
        avg_disengaged = get_avg_focus_from_intervals(cycle_intervals, 'behavioral_disengagement', 'disengagement')

        low_duration = cycle.get('low_intensity_duration', 0)
        high_low_ratio = cycle.get('high_intensity_duration', 0) / low_duration if low_duration > 0 else float('inf')

        results.append({
            '課堂ID': class_id,
            '循環索引': cycle.get('cycle_index'),
            '開始時間': cycle.get('start_time'),
            '結束時間': end_time_str,
            '總時長(秒)': cycle.get('total_duration_seconds'),
            '高強度時長(秒)': cycle.get('high_intensity_duration'),
            '低強度時長(秒)': low_duration,
            '高低強度時長比': round(high_low_ratio, 2),
            '循環內平均主動參與度(%)': round(avg_active, 2),
            '循環內平均分心度(%)': round(avg_disengaged, 2)
        })
    return results

def generate_reports():
    """【主函數修改】執行所有操作並生成多個報告。"""
    input_directory = r"C:\Users\User\Desktop\test\classroom_analysis_report"
    output_directory = r"C:\Users\User\Desktop\test\classroom_analysis_report"
    
    # --- 【修改】定義多個輸出檔案名稱 ---
    output_files = {
        "dynamic_segments": "report_1_dynamic_segments.csv",
        "class_summary": "report_2_class_summary.csv",
        "keyword_impact": "report_3_keyword_impact.csv",
        "breaking_points": "report_4_breaking_points.csv", # --- 【新增】---
        "teaching_cycles": "report_5_teaching_cycles.csv"  # --- 【新增】---
    }
    
    print("--- 課堂洞察報告生成器 v3.3 (最終版) ---")
    print(f"1. 開始掃描資料夾：{input_directory}")
    
    json_files = glob.glob(os.path.join(input_directory, "classroom_analysis_report_*.json"))
    if not json_files:
        print(f"❌ 錯誤：找不到任何 'classroom_analysis_report_*.json' 檔案。")
        return

    print(f"✅ 找到 {len(json_files)} 份報告，準備進行多維度數據提取...")
    
    # --- 【修改】初始化所有報告的列表 ---
    all_data = {
        "dynamic_segments": [], "class_summaries": [], "keyword_impacts": [],
        "breaking_points": [], "teaching_cycles": []
    }

    for file_path in json_files:
        filename = os.path.basename(file_path)
        print(f"\n... 正在處理檔案: {filename}")
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            class_id = safe_get(data, ['class_session_id'], 'unknown_class')
            detailed_timeline = data.get('detailed_timeline', [])
            timeline_analysis = data.get('timeline_analysis', [])
            keyword_timestamps = data.get('keyword_timestamps', [])
            
            # --- 【修改】同時讀取正面與負面觸發詞 ---
            pos_triggers_str = safe_get(data, ['overall_summary', 'lexical_trigger_highlights', 'positive_trigger_words'], '')
            neg_triggers_str = safe_get(data, ['overall_summary', 'lexical_trigger_highlights', 'negative_trigger_words'], '')
            positive_triggers = [word.strip().replace("』", "").replace("『", "") for word in pos_triggers_str.split('、')]
            negative_triggers = [word.strip().replace("』", "").replace("『", "") for word in neg_triggers_str.split('、')]
            
            if not detailed_timeline:
                print(f"⚠️ 警告：檔案 '{filename}' 缺少 'detailed_timeline' 數據，部分分析將跳過。")
            
            # --- 【修改】執行所有五種分析 ---
            all_data["dynamic_segments"].extend(process_timeline_dynamically(class_id, detailed_timeline, timeline_analysis, positive_triggers, negative_triggers))
            all_data["class_summaries"].append(analyze_class_summary(data))
            all_data["keyword_impacts"].extend(analyze_keyword_impact(class_id, detailed_timeline, keyword_timestamps))
            all_data["breaking_points"].extend(analyze_breaking_points(data))
            all_data["teaching_cycles"].extend(analyze_teaching_cycles(data, detailed_timeline))
            
        except Exception as e:
            print(f"⚠️ 警告：處理檔案 '{filename}' 時發生嚴重錯誤: {e}。")

    # --- 儲存報告1：動態區段分析 ---
    if all_data["dynamic_segments"]:
        print("\n2. [報告1] 動態區段分析完成，正在生成 CSV...")
        df = pd.DataFrame(all_data["dynamic_segments"])
        column_order = [
            '課堂ID', '開始時間', '結束時間', '區段主要教學模式', '區段時長(秒)',
            '區段主要教學活動', '區段平均語速(WPM)', '正面指令密度(次/分)', '負面詞彙密度(次/分)',
            '平均提問頻率(次/分)', '板書使用頻率(次/30秒)', '平均主動參與度(%)', '平均被動參與度(%)', '平均行為分心度(%)',
            '專注度趨勢', '閒聊內容關聯度', '平均趴睡學生數', '平均玩筆學生數', '平均行為多樣性',
            '包含專注提升點', '提升點觸發模式', '包含專注下降點', '下降點觸發模式'
        ]
        df = df.reindex(columns=[col for col in column_order if col in df.columns])
        output_path = os.path.join(output_directory, output_files["dynamic_segments"])
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"✅ 報告1 已儲存至：{output_path}")

    # --- 儲存報告2：宏觀結構總結 ---
    if all_data["class_summaries"]:
        print("\n3. [報告2] 宏觀結構分析完成，正在生成 CSV...")
        df = pd.DataFrame(all_data["class_summaries"])
        output_path = os.path.join(output_directory, output_files["class_summary"])
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"✅ 報告2 已儲存至：{output_path}")

    # --- 儲存報告3：關鍵詞影響分析 ---
    if all_data["keyword_impacts"]:
        print("\n4. [報告3] 關鍵詞影響分析完成，正在生成 CSV...")
        df = pd.DataFrame(all_data["keyword_impacts"])
        output_path = os.path.join(output_directory, output_files["keyword_impact"])
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"✅ 報告3 已儲存至：{output_path}")

    # --- 【新增】儲存報告4：專注力臨界點分析 ---
    if all_data["breaking_points"]:
        print("\n5. [報告4] 專注力臨界點分析完成，正在生成 CSV...")
        df = pd.DataFrame(all_data["breaking_points"])
        output_path = os.path.join(output_directory, output_files["breaking_points"])
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"✅ 報告4 已儲存至：{output_path}")

    # --- 【新增】儲存報告5：教學循環分析 ---
    if all_data["teaching_cycles"]:
        print("\n6. [報告5] 教學循環分析完成，正在生成 CSV...")
        df = pd.DataFrame(all_data["teaching_cycles"])
        output_path = os.path.join(output_directory, output_files["teaching_cycles"])
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"✅ 報告5 已儲存至：{output_path}")

    print("-" * 50)
    print(f"🎉🎉🎉 所有報告生成成功！ 🎉🎉🎉")
    print("-" * 50)

if __name__ == '__main__':
    generate_reports()

# Re-Group

In [ ]:
import os
import shutil
import torch
from PIL import Image
import open_clip # 使用 open_clip_torch
import numpy as np
from tqdm import tqdm # 用於顯示進度條
from sklearn.cluster import DBSCAN # 可選的聚類算法
from sklearn.metrics.pairwise import cosine_similarity # 計算餘弦相似度

# --- 配置參數 ---
SOURCE_CROPS_PARENT_DIR = r"C:\Users\User\Desktop\test\test_regroup" # 包含所有 MaintainedID_X 資料夾的路徑
OUTPUT_REGROUPED_DIR = r"C:\Users\User\Desktop\test\test" # 重新分組後圖像存放的目錄
CLIP_MODEL_NAME = 'ViT-B-32' # 常用的 CLIP 模型之一
CLIP_PRETRAINED_DATASET = 'laion2b_s34b_b79k' # 常用的預訓練數據集 for open_clip

# 相似度閾值，用於判斷兩張圖片是否為同一個人 (0.0 - 1.0，越高越相似)
# 這個閾值非常關鍵，需要根據你的數據集進行調整。
# CLIP 的餘弦相似度通常比較高，可以從 0.8 或 0.85 開始嘗試。
# 如果希望更嚴格（不容易把不同人分為一組），可以提高此值。
# 如果希望更寬鬆（更容易把相似的人分為一組，但可能引入錯誤），可以降低此值。
SIMILARITY_THRESHOLD_CLIP = 0.81 # <--- 關鍵參數，需要仔細調整！

# 是否使用聚類算法 (DBSCAN) 而不是簡單的閾值合併
USE_CLUSTERING = False # 設為 True 則使用 DBSCAN，False 則使用基於閾值的迭代合併
DBSCAN_EPS = 0.19 # DBSCAN 的鄰域半徑 (1 - cosine_similarity)，所以值越小，要求越相似。對應上面的 0.88。
DBSCAN_MIN_SAMPLES = 1 # DBSCAN 成為核心點的最小樣本數

# --- 初始化 CLIP 模型 ---
print(f"正在載入 CLIP 模型: {CLIP_MODEL_NAME} (預訓練於 {CLIP_PRETRAINED_DATASET})...")
try:
    clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
        CLIP_MODEL_NAME,
        pretrained=CLIP_PRETRAINED_DATASET,
        device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )
    clip_model.eval() # 設置為評估模式
    tokenizer = open_clip.get_tokenizer(CLIP_MODEL_NAME)
    print("CLIP 模型載入完成！")
except Exception as e:
    print(f"載入 CLIP 模型失敗: {e}")
    exit()

def get_image_paths(parent_dir):
    """遍歷所有 MaintainedID_X 資料夾，獲取所有 jpg 圖像的路徑及其原始 ID。"""
    image_paths = []
    original_ids = []
    for id_folder in tqdm(sorted(os.listdir(parent_dir)), desc="掃描資料夾"):
        if os.path.isdir(os.path.join(parent_dir, id_folder)) and id_folder.startswith("MaintainedID_"):
            try:
                original_id_num = int(id_folder.split("_")[1]) # 提取原始 MaintainedID 數字
            except:
                continue # 跳過不符合格式的資料夾名

            for img_file in os.listdir(os.path.join(parent_dir, id_folder)):
                if img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    image_paths.append(os.path.join(parent_dir, id_folder, img_file))
                    original_ids.append(original_id_num) # 記錄原始ID
    return image_paths, original_ids

def extract_clip_features(image_paths_list):
    """為圖像列表提取 CLIP 特徵向量。"""
    features_list = []
    device = next(clip_model.parameters()).device # 獲取模型所在的設備

    for img_path in tqdm(image_paths_list, desc="提取 CLIP 特徵"):
        try:
            image = Image.open(img_path).convert("RGB")
            image_input = clip_preprocess(image).unsqueeze(0).to(device)
            with torch.no_grad():
                image_features = clip_model.encode_image(image_input)
                image_features /= image_features.norm(dim=-1, keepdim=True) # L2 正規化
            features_list.append(image_features.cpu().numpy().flatten())
        except Exception as e:
            print(f"處理圖像 {img_path} 時出錯: {e}")
            features_list.append(None) # 如果出錯，添加 None 作為佔位符
    return features_list

def regroup_images_by_threshold(image_paths, features, original_ids):
    """基於相似度閾值將圖像重新分組。"""
    if not os.path.exists(OUTPUT_REGROUPED_DIR):
        os.makedirs(OUTPUT_REGROUPED_DIR)

    # 過濾掉提取特徵失敗的圖像
    valid_indices = [i for i, f in enumerate(features) if f is not None]
    if not valid_indices:
        print("沒有有效的圖像特徵被提取。")
        return
    
    image_paths = [image_paths[i] for i in valid_indices]
    features = np.array([features[i] for i in valid_indices])
    original_ids = [original_ids[i] for i in valid_indices]

    num_images = len(image_paths)
    assigned_group = [-1] * num_images # -1 表示尚未分配組別
    current_group_id = 0

    print(f"\n開始基於閾值 {SIMILARITY_THRESHOLD_CLIP} 重新分組圖像...")
    for i in tqdm(range(num_images), desc="圖像分組中"):
        if assigned_group[i] == -1: # 如果當前圖像尚未分配組別
            current_group_id += 1
            assigned_group[i] = current_group_id
            # 創建該組的輸出資料夾
            group_folder = os.path.join(OUTPUT_REGROUPED_DIR, f"Person_{current_group_id}")
            if not os.path.exists(group_folder):
                os.makedirs(group_folder)
            
            # 將當前圖像複製到新組別
            shutil.copy(image_paths[i], os.path.join(group_folder, os.path.basename(image_paths[i])))

            # 尋找與當前圖像相似的其他未分配圖像
            for j in range(i + 1, num_images):
                if assigned_group[j] == -1: # 只考慮未分配的
                    # 計算餘弦相似度
                    similarity = np.dot(features[i], features[j]) / (np.linalg.norm(features[i]) * np.linalg.norm(features[j]))
                    if similarity >= SIMILARITY_THRESHOLD_CLIP:
                        assigned_group[j] = current_group_id
                        shutil.copy(image_paths[j], os.path.join(group_folder, os.path.basename(image_paths[j])))
    
    print(f"\n圖像重新分組完成！共創建了 {current_group_id} 個組別。")
    print(f"結果已保存到: {OUTPUT_REGROUPED_DIR}")

def regroup_images_by_clustering(image_paths, features, original_ids):
    """使用 DBSCAN 聚類算法將圖像重新分組。"""
    if not os.path.exists(OUTPUT_REGROUPED_DIR):
        os.makedirs(OUTPUT_REGROUPED_DIR)

    valid_indices = [i for i, f in enumerate(features) if f is not None]
    if not valid_indices:
        print("沒有有效的圖像特徵被提取。")
        return
        
    image_paths = [image_paths[i] for i in valid_indices]
    features_np = np.array([features[i] for i in valid_indices]) # 確保是 NumPy 陣列
    original_ids = [original_ids[i] for i in valid_indices]

    print(f"\n開始使用 DBSCAN 進行聚類 (eps={DBSCAN_EPS}, min_samples={DBSCAN_MIN_SAMPLES})...")
    # DBSCAN 使用距離矩陣，我們有特徵向量，可以直接用 cosine 距離
    # sklearn 的 DBSCAN 接受特徵矩陣和 metric='cosine'
    # 由於 DBSCAN 的 eps 是距離，而我們通常用相似度，需要轉換
    # cosine_distance = 1 - cosine_similarity
    # 所以 eps 應該是一個較小的值，如果 SIMILARITY_THRESHOLD_CLIP 是 0.88，那麼 eps 大約是 1 - 0.88 = 0.12
    
    clustering = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES, metric='cosine', n_jobs=-1).fit(features_np)
    labels = clustering.labels_ # 每個圖像的簇標籤，-1 代表噪聲點（未被分到任何簇）

    num_clusters = len(set(labels)) - (1 if -1 in labels else 0) # 計算有效簇的數量
    print(f"DBSCAN 聚類完成！找到了 {num_clusters} 個簇 (不包括噪聲點)。")

    for i, label in enumerate(tqdm(labels, desc="複製圖像到聚類資料夾")):
        if label == -1: # 噪聲點，可以選擇單獨存放或忽略
            group_folder = os.path.join(OUTPUT_REGROUPED_DIR, "Noise_or_Unassigned")
        else:
            group_folder = os.path.join(OUTPUT_REGROUPED_DIR, f"Person_Cluster_{label}")
        
        if not os.path.exists(group_folder):
            os.makedirs(group_folder)
        
        try:
            shutil.copy(image_paths[i], os.path.join(group_folder, os.path.basename(image_paths[i])))
        except Exception as e_copy:
            print(f"複製圖像 {image_paths[i]} 到 {group_folder} 失敗: {e_copy}")
            
    print(f"\n圖像重新分組 (聚類) 完成！")
    print(f"結果已保存到: {OUTPUT_REGROUPED_DIR}")


def main_regroup():
    if not os.path.isdir(SOURCE_CROPS_PARENT_DIR):
        print(f"錯誤: 源裁切圖像父目錄 '{SOURCE_CROPS_PARENT_DIR}' 不存在或不是一個目錄。")
        return

    all_image_paths, all_original_ids = get_image_paths(SOURCE_CROPS_PARENT_DIR)
    if not all_image_paths:
        print("在源目錄中沒有找到任何圖像。")
        return
    
    print(f"共找到 {len(all_image_paths)} 張圖像進行處理。")

    all_features = extract_clip_features(all_image_paths)
    
    # 過濾掉提取特徵失敗的圖像 (再次確認)
    valid_indices = [i for i, f in enumerate(all_features) if f is not None]
    if not valid_indices:
        print("特徵提取後沒有有效的圖像。")
        return
        
    filtered_image_paths = [all_image_paths[i] for i in valid_indices]
    filtered_features = [all_features[i] for i in valid_indices]
    filtered_original_ids = [all_original_ids[i] for i in valid_indices]

    if USE_CLUSTERING:
        regroup_images_by_clustering(filtered_image_paths, filtered_features, filtered_original_ids)
    else:
        regroup_images_by_threshold(filtered_image_paths, filtered_features, filtered_original_ids)

if __name__ == "__main__":
    # 創建一個虛假的 SOURCE_CROPS_PARENT_DIR 結構以供測試 (如果它不存在)
    # 你應該將 SOURCE_CROPS_PARENT_DIR 指向你實際的包含 MaintainedID_X 資料夾的路徑
    if not os.path.exists(SOURCE_CROPS_PARENT_DIR) or not any(f.startswith("MaintainedID_") for f in os.listdir(SOURCE_CROPS_PARENT_DIR)):
        print(f"警告: '{SOURCE_CROPS_PARENT_DIR}' 看起來不是一個有效的源目錄，或者其中沒有 MaintainedID_X 資料夾。")
        print("為了演示，將創建一些虛擬的資料夾和圖像。在實際使用時，請確保路徑正確。")
        if not os.path.exists(SOURCE_CROPS_PARENT_DIR): os.makedirs(SOURCE_CROPS_PARENT_DIR)
        for i in range(1, 6): # 創建 5 個虛擬 ID 資料夾
            id_folder = os.path.join(SOURCE_CROPS_PARENT_DIR, f"MaintainedID_{i}")
            if not os.path.exists(id_folder): os.makedirs(id_folder)
            for j in range(2): # 每個資料夾放 2 張虛擬圖片
                try:
                    dummy_img_name = f"dummy_img_{i}_{j}.jpg"
                    dummy_pil_img = Image.fromarray(np.random.randint(0, 256, (128, 128, 3), dtype=np.uint8))
                    dummy_pil_img.save(os.path.join(id_folder, dummy_img_name))
                except Exception as e_dummy:
                    print(f"創建虛擬圖像失敗: {e_dummy}")
        print("已創建虛擬資料夾和圖像結構。")

    main_regroup()

# CLIP 找出關鍵幀

In [15]:
# -*- coding: utf-8 -*-
import sys
import os

# --- 新增的診斷碼 ---
try:
    import torch
    print("--- DIAGNOSTIC INFORMATION ---")
    print(f"Python Executable: {sys.executable}")
    print(f"PyTorch Version: {torch.__version__}")
    if torch.cuda.is_available():
        print(f"PyTorch CUDA Version: {torch.version.cuda}")
        print(f"Is CUDA Available: Yes")
    else:
        print("Is CUDA Available: No")
    print("----------------------------\n")
except ImportError:
    print("\n--- ERROR ---")
    print("PyTorch is NOT installed or cannot be found in this environment.")
    print(f"This script is running from: {sys.executable}")
    print("---------------\n")
    exit() # 如果連 torch 都找不到，就直接退出
import re
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch # 確保 torch 已導入
import shutil
from tqdm import tqdm # 導入 tqdm

# ---------------------------------------------------------------
# CONFIGURATION (參數化配置區)
# ---------------------------------------------------------------
# 您可以在此處調整程式的行為，無需修改下方的主要邏輯

# --- 模型與處理設定 ---
MODEL_ID = "openai/clip-vit-base-patch32"  # 要使用的 CLIP 模型 ID
FOLDER_PREFIX_TO_PROCESS = "ID_"          # 指定要處理的資料夾名稱前綴
KEYFRAMES_SUBFOLDER_NAME = "Keyframes"    # 用於儲存關鍵幀的子資料夾名稱

# --- 動態閾值計算設定 ---
PERCENTILE_TO_USE = 10           # 使用相似度分數的第 n 百分位數作為基準 (越低越嚴格)
MIN_ABSOLUTE_THRESHOLD = 0.85     # 即使百分位數很低，閾值也不會低於此值

# --- 繪圖與輸出設定 ---
MAX_KEYFRAME_PAIRS_DISPLAY = 6   # 在報告圖中最多顯示幾對關鍵幀縮圖
THUMBNAIL_SIZE = (150, 150)      # 縮圖的尺寸 (寬, 高)
PLOT_FILENAME_PREFIX = "similarity_plot_" # 儲存的報告圖檔名前綴
PLOT_DPI = 150                   # 儲存的報告圖的解析度 (dots per inch)
PLOT_MAX_XTICKS = 15             # 圖表中 X 軸最多顯示的刻度數量
PLOT_MAX_ANNOTATIONS = 20        # 圖表中最多標註的相似度數值數量

# ---------------------------------------------------------------
# GPU 設備檢查與設定
# ---------------------------------------------------------------
if torch.cuda.is_available():
    device = torch.device("cuda")
    try:
        gpu_name = torch.cuda.get_device_name(0)
        print(f"GPU is available: {gpu_name}")
        print("Running calculations on GPU.")
    except Exception as e:
        print(f"GPU is available, but couldn't get device name: {e}")
        print("Running calculations on GPU.")
else:
    device = torch.device("cpu")
    print("GPU not available, running calculations on CPU.")

# ---------------------------------------------------------------
# CLIP 模型加載 (移至指定設備)
# ---------------------------------------------------------------
try:
    print(f"Loading CLIP model '{MODEL_ID}' (forcing re-download and using safetensors)...")
    
    # 在加載模型時，同時加入 force_download=True 和 use_safetensors=True
    model = CLIPModel.from_pretrained(
        MODEL_ID, 
        force_download=True, 
        use_safetensors=True  # <--- 加入這一行是關鍵！
    )
    
    # Processor 的加載保持不變即可
    processor = CLIPProcessor.from_pretrained(
        MODEL_ID, 
        force_download=True
    )

    # 將模型移到檢測到的設備 (GPU 或 CPU)
    model.to(device)
    # 將模型設置為評估模式 (不計算梯度，節省資源)
    model.eval()
    print("CLIP model loaded successfully and moved to device:", device)
except Exception as e:
    print(f"Error loading CLIP model: {e}")
    print("Please check your internet connection and transformers installation.")
    exit()

# ---------------------------------------------------------------
# 輔助函數 (修改以使用 GPU)
# ---------------------------------------------------------------
def get_image_embedding(image_path):
    """計算單張圖片的 CLIP 嵌入並進行 L2 歸一化 (在指定設備上運行)"""
    try:
        image = Image.open(image_path).convert("RGB")
        if 'model' not in globals() or 'processor' not in globals():
             raise RuntimeError("CLIP model or processor not properly initialized.")
        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            image_features = model.get_image_features(**inputs)
        norm_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
        return norm_features
    except FileNotFoundError:
        return None
    except Exception as e:
        print(f"Error processing image {image_path}: {e}")
        return None

def compute_similarity(img1_path, img2_path):
    """計算兩張圖片嵌入的餘弦相似度 (在指定設備上計算)"""
    emb1 = get_image_embedding(img1_path)
    emb2 = get_image_embedding(img2_path)
    if emb1 is None or emb2 is None:
        return 0.0
    similarity = torch.nn.functional.cosine_similarity(emb1, emb2)
    return similarity.cpu().item()

def load_images_from_folder(folder_path, extensions=(".jpg", ".jpeg", ".png", ".bmp")):
    """從資料夾加載圖片路徑並按檔名中的數字排序"""
    try:
        if not os.path.isdir(folder_path):
             return []
        files = [f for f in os.listdir(folder_path)
                 if f.lower().endswith(extensions) and os.path.isfile(os.path.join(folder_path, f))]
    except Exception as e:
        print(f"Error reading folder {folder_path}: {e}")
        return []
    if not files:
        return []

    def get_sort_key(filename):
        """提取檔名中的數字用於排序，優先考慮時間戳格式或幀號"""
        time_match = re.search(r'(\d{2})h(\d{2})m(\d{2})s', filename)
        if time_match:
            h, m, s = map(int, time_match.groups())
            return h * 3600 + m * 60 + s
        frame_match = re.search(r'(?:f|frame)?(\d+)', filename)
        if frame_match:
            try:
                return int(frame_match.group(1))
            except (ValueError, IndexError):
                pass
        print(f"Warning: Could not extract time/frame number from '{filename}'. Using alphabetical sort key.")
        return filename.lower()

    try:
        files.sort(key=get_sort_key)
    except Exception as e:
        print(f"Warning: Error during sorting ({e}). Files might be out of order.")
        files.sort()

    full_paths = [os.path.join(folder_path, f) for f in files]
    return full_paths

# ---------------------------------------------------------------
# 繪圖函數
# ---------------------------------------------------------------
def plot_similarity_and_key_thumbnails(
    image_paths,
    similarities,
    threshold,
    original_folder_path,
    max_pairs_to_show=5,
    keyframes_subfolder="Keyframes",
    save_plot_filename=None
    ):
    """
    繪製相似度，顯示/保存關鍵幀縮圖，並保存所有相關的關鍵幀。
    如果提供了 save_plot_filename，則保存圖表而不是顯示它。
    """
    num_images = len(image_paths)
    if num_images < 2: return None
    valid_similarities = [s for s in similarities if isinstance(s, (int, float))]
    if len(valid_similarities) != num_images - 1:
         print(f"Error: Mismatch in similarity count ({len(valid_similarities)}) vs image count ({num_images})")
         if len(similarities) != num_images -1:
              return None

    # --- 準備數據 ---
    sim_x = list(range(2, num_images + 1))
    low_sim_indices = [i + 2 for i, sim in enumerate(similarities) if isinstance(sim, (int, float)) and sim < threshold]
    low_sim_values = [sim for sim in similarities if isinstance(sim, (int, float)) and sim < threshold]

    # --- 創建關鍵幀資料夾 ---
    keyframes_folder_path = os.path.join(original_folder_path, keyframes_subfolder)
    saved_keyframes_paths = set()
    can_save_keyframes = False
    if low_sim_indices:
        try:
            os.makedirs(keyframes_folder_path, exist_ok=True)
            print(f"  Keyframes subfolder: {keyframes_folder_path}")
            can_save_keyframes = True
        except OSError as e:
            print(f"  Error creating directory {keyframes_folder_path}: {e}")

    # --- 設置圖形佈局 ---
    num_pairs_to_show_display = min(len(low_sim_indices), max_pairs_to_show)
    base_height = 6
    height_per_pair = 2.5
    total_height = max(base_height, base_height + num_pairs_to_show_display * height_per_pair if num_pairs_to_show_display > 0 else base_height)
    fig = plt.figure(figsize=(15, total_height))
    ax_sim, ax_thumbs_container = None, None

    if num_pairs_to_show_display > 0:
        height_ratios = [max(1, base_height), max(1, num_pairs_to_show_display * height_per_pair)]
        if sum(height_ratios) <= 0: height_ratios = [2, 3]
        try:
            gs_main = gridspec.GridSpec(2, 1, height_ratios=height_ratios, figure=fig)
            ax_sim = fig.add_subplot(gs_main[0, 0])
            ax_thumbs_container = fig.add_subplot(gs_main[1, 0])
            ax_thumbs_container.set_title(f"Key Frame Pairs (Similarity < {threshold:.3f}) (Showing first {num_pairs_to_show_display})")
            ax_thumbs_container.axis("off")
        except ValueError as e:
            print(f"  Error creating main GridSpec: {e}. Attempting single plot.")
            gs_main = gridspec.GridSpec(1, 1, figure=fig)
            ax_sim = fig.add_subplot(gs_main[0, 0])
            num_pairs_to_show_display = 0
    else:
        gs_main = gridspec.GridSpec(1, 1, figure=fig)
        ax_sim = fig.add_subplot(gs_main[0, 0])

    if ax_sim is None:
        print("Error: Could not create Matplotlib axes. Skipping plot.")
        plt.close(fig)
        return None

    # --- 1. 繪製上方相似度折線圖 ---
    folder_name = os.path.basename(original_folder_path)
    ax_sim.plot(sim_x, similarities, marker='.', linestyle='-', color='blue', markersize=5, label='Similarity')
    ax_sim.set_xlabel("Frame Index (Comparison: Previous Frame -> Current Frame)")
    ax_sim.set_ylabel("Cosine Similarity")
    ax_sim.set_title(f"Consecutive Frame Similarity ({folder_name}) (CLIP Embedding)")
    ax_sim.grid(True, linestyle='--', alpha=0.6)
    ax_sim.axhline(threshold, color='red', linestyle='--', linewidth=1, label=f'Threshold ({threshold:.3f})')
    if low_sim_indices:
        ax_sim.plot(low_sim_indices, low_sim_values, 'ro', markersize=6, label=f'Below Threshold')
        num_annotations_to_show = PLOT_MAX_ANNOTATIONS # 使用配置參數
        annotation_step = max(1, len(low_sim_values) // num_annotations_to_show)
        annotated_count = 0
        for i, txt in enumerate(low_sim_values):
             if i % annotation_step == 0 and annotated_count < num_annotations_to_show:
                 ax_index = low_sim_indices[i]
                 if isinstance(txt, (int, float)):
                     ax_sim.annotate(f"{txt:.2f}", (ax_index, txt), textcoords="offset points", xytext=(0, -15), ha='center', color='red', fontsize=8)
                     annotated_count += 1

    num_xticks = PLOT_MAX_XTICKS # 使用配置參數
    if num_images > 1 and len(sim_x)>0:
        if len(sim_x) <= num_xticks:
             xtick_step = 1
        else:
             xtick_step = max(1, (len(sim_x) // num_xticks))
        xtick_indices_calc = list(range(sim_x[0], sim_x[-1] + 1, xtick_step))
        if not xtick_indices_calc or xtick_indices_calc[-1] < sim_x[-1]: xtick_indices_calc.append(sim_x[-1])
        if xtick_indices_calc[0] > sim_x[0]: xtick_indices_calc.insert(0, sim_x[0])
        if len(xtick_indices_calc) > num_xticks + 5:
             xtick_indices = np.linspace(sim_x[0], sim_x[-1], num_xticks, dtype=int).tolist()
             if sim_x[0] not in xtick_indices: xtick_indices.insert(0, sim_x[0])
             if sim_x[-1] not in xtick_indices: xtick_indices.append(sim_x[-1])
             xtick_indices = sorted(list(set(xtick_indices)))
        else:
             xtick_indices = xtick_indices_calc
        ax_sim.set_xticks(xtick_indices)
        ax_sim.set_xticklabels(xtick_indices, rotation=45, ha='right', fontsize=8)
    else:
        ax_sim.set_xticks([])
    ax_sim.legend()

    # --- 2. 繪製下方 *部分* 關鍵幀縮圖 ---
    if num_pairs_to_show_display > 0 and ax_thumbs_container is not None:
        try:
            gs_thumbs = gridspec.GridSpecFromSubplotSpec(num_pairs_to_show_display, 2, subplot_spec=gs_main[1, 0], wspace=0.1, hspace=0.5)
            thumb_size = THUMBNAIL_SIZE # 使用配置參數
            for i in range(num_pairs_to_show_display):
                img_idx1, img_idx2 = low_sim_indices[i] - 2, low_sim_indices[i] - 1
                if not (0 <= img_idx1 < len(image_paths) and 0 <= img_idx2 < len(image_paths)): continue
                img_path1_disp, img_path2_disp = image_paths[img_idx1], image_paths[img_idx2]
                similarity_value_disp = low_sim_values[i]
                
                # 左縮圖
                ax_left = fig.add_subplot(gs_thumbs[i, 0])
                try:
                    img1 = cv2.cvtColor(cv2.imread(img_path1_disp), cv2.COLOR_BGR2RGB)
                    ax_left.imshow(cv2.resize(img1, thumb_size))
                except Exception:
                    ax_left.text(0.5, 0.5, f"Err\n{os.path.basename(img_path1_disp)}", ha='center', va='center', color='red', fontsize=8)
                ax_left.set_title(f"Frame {img_idx1 + 1}", fontsize=9)
                ax_left.axis("off")
                
                # 右縮圖
                ax_right = fig.add_subplot(gs_thumbs[i, 1])
                try:
                    img2 = cv2.cvtColor(cv2.imread(img_path2_disp), cv2.COLOR_BGR2RGB)
                    ax_right.imshow(cv2.resize(img2, thumb_size))
                except Exception:
                    ax_right.text(0.5, 0.5, f"Err\n{os.path.basename(img_path2_disp)}", ha='center', va='center', color='red', fontsize=8)
                sim_text = f"{similarity_value_disp:.3f}" if isinstance(similarity_value_disp, (int, float)) else "N/A"
                title_text = f"Frame {img_idx2 + 1}\n(Sim({img_idx1+1}→{img_idx2+1}): {sim_text})"
                ax_right.set_title(title_text, fontsize=9)
                ax_right.axis("off")

        except Exception as e:
             print(f"  Error during thumbnail generation: {e}")
             if ax_thumbs_container is not None: ax_thumbs_container.set_title("Failed to generate thumbnails", color='red')

    # --- 3. 保存 *所有* 相關的關鍵幀 ---
    if low_sim_indices and can_save_keyframes:
        saved_count_this_run = 0
        for i in range(len(low_sim_indices)):
            img_idx1_save, img_idx2_save = low_sim_indices[i] - 2, low_sim_indices[i] - 1
            if not (0 <= img_idx1_save < len(image_paths) and 0 <= img_idx2_save < len(image_paths)): continue
            for img_path_to_save in [image_paths[img_idx1_save], image_paths[img_idx2_save]]:
                if img_path_to_save not in saved_keyframes_paths:
                    try:
                        dest_path = os.path.join(keyframes_folder_path, os.path.basename(img_path_to_save))
                        if not os.path.exists(dest_path):
                             shutil.copy2(img_path_to_save, dest_path)
                             saved_count_this_run += 1
                        saved_keyframes_paths.add(img_path_to_save)
                    except Exception as e:
                        print(f"  Error saving keyframe {os.path.basename(img_path_to_save)}: {e}")
        if saved_count_this_run > 0: print(f"  Saved {saved_count_this_run} new unique keyframes.")

    # --- 最終調整 & 保存/顯示 ---
    try:
        fig.tight_layout(rect=[0, 0.03, 1, 0.95], h_pad=3.0 if num_pairs_to_show_display > 0 else None)
    except Exception as e:
        print(f"  Warning: Error adjusting layout: {e}")

    if save_plot_filename:
        try:
            full_save_path = os.path.join(original_folder_path, save_plot_filename)
            fig.savefig(full_save_path, dpi=PLOT_DPI) # 使用配置參數
            print(f"  Plot saved to: {full_save_path}")
        except Exception as e:
            print(f"  Error saving plot to {full_save_path}: {e}")
    else:
        plt.show()
    plt.close(fig)

    return len(saved_keyframes_paths)

# ---------------------------------------------------------------
# 處理單個 ID 資料夾的函數
# ---------------------------------------------------------------
def process_id_folder(id_folder_path):
    """處理單個 ID 資料夾：加載圖片，計算相似度，計算閾值，繪圖，保存關鍵幀"""
    print("-" * 50)
    folder_name = os.path.basename(id_folder_path)
    print(f"Processing folder: {folder_name}")

    image_paths = load_images_from_folder(id_folder_path)
    if len(image_paths) < 2:
        print(f"  Skipping folder '{folder_name}': Needs at least 2 images, found {len(image_paths)}.")
        return

    print(f"  Found {len(image_paths)} images for '{folder_name}'.")

    # --- 使用 tqdm 顯示進度條 ---
    try:
        # 將相似度計算包在 tqdm 中
        desc = f"  Calculating similarities for '{folder_name}'"
        similarities = [compute_similarity(image_paths[i], image_paths[i+1]) for i in tqdm(range(len(image_paths) - 1), desc=desc)]
    except Exception as sim_e:
         print(f"  Error during bulk similarity calculation for '{folder_name}': {sim_e}")
         similarities = []
         # 備用方案也加入進度條
         desc_fallback = f"  Calculating (fallback) for '{folder_name}'"
         for i in tqdm(range(len(image_paths) - 1), desc=desc_fallback):
             try:
                 similarities.append(compute_similarity(image_paths[i], image_paths[i+1]))
             except Exception as single_sim_e:
                 print(f"   Error calculating similarity between frame {i+1} and {i+2}: {single_sim_e}")
                 similarities.append(0.0)
    
    # --- 動態閾值計算 (使用配置參數) ---
    valid_similarities = [s for s in similarities if isinstance(s, (int, float))]
    if not valid_similarities:
        print(f"  Error: No valid similarities calculated for '{folder_name}'. Skipping plot/save.")
        return
        
    try:
        calculated_percentile_value = np.percentile(np.array(valid_similarities), PERCENTILE_TO_USE)
        dynamic_threshold = max(calculated_percentile_value, MIN_ABSOLUTE_THRESHOLD)
        print(f"  Dynamic threshold for '{folder_name}': {dynamic_threshold:.4f} (Based on {PERCENTILE_TO_USE}th percentile, min {MIN_ABSOLUTE_THRESHOLD})")
    except IndexError:
        print(f"  Error calculating percentile for '{folder_name}'. Using default threshold {MIN_ABSOLUTE_THRESHOLD}.")
        dynamic_threshold = MIN_ABSOLUTE_THRESHOLD

    # --- 繪製圖表並保存關鍵幀 (使用配置參數) ---
    plot_filename = f"{PLOT_FILENAME_PREFIX}{folder_name}.png"

    saved_count = plot_similarity_and_key_thumbnails(
        image_paths,
        similarities,
        threshold=dynamic_threshold,
        original_folder_path=id_folder_path,
        max_pairs_to_show=MAX_KEYFRAME_PAIRS_DISPLAY,
        keyframes_subfolder=KEYFRAMES_SUBFOLDER_NAME,
        save_plot_filename=plot_filename
    )

    # 打印該資料夾的保存總結
    if saved_count is not None:
         low_sim_count = len([s for s in similarities if isinstance(s, (float, int)) and s < dynamic_threshold])
         if saved_count > 0 :
             print(f"  Finished processing '{folder_name}'. Saved {saved_count} unique keyframes.")
         elif low_sim_count > 0:
             print(f"  Finished processing '{folder_name}'. Identified low similarity pairs, but no new unique keyframes were saved (likely already exist).")
         else:
             print(f"  Finished processing '{folder_name}'. No significant changes detected (above threshold). No keyframes saved.")
    else:
         print(f"  Finished processing '{folder_name}', but encountered issues during plotting/saving.")


# ---------------------------------------------------------------
# 主函數
# ---------------------------------------------------------------
def main():
    """主執行函數，引導使用者輸入路徑並遍歷處理"""
    parent_folder_path = input("Please enter the parent folder path containing ID subfolders (e.g., person_crops): ").strip().strip('"')

    if not os.path.isdir(parent_folder_path):
        print(f"Error: The entered path is not a valid directory: {parent_folder_path}")
        return

    print(f"Starting batch processing for folders within: {parent_folder_path}")
    processed_folder_count = 0
    folder_names = sorted([d for d in os.listdir(parent_folder_path)
                           if os.path.isdir(os.path.join(parent_folder_path, d))])

    for item_name in folder_names:
        item_path = os.path.join(parent_folder_path, item_name)

        # 使用配置參數來決定處理和跳過哪些資料夾
        if item_name.startswith(FOLDER_PREFIX_TO_PROCESS):
            process_id_folder(item_path)
            processed_folder_count += 1
        elif item_name == KEYFRAMES_SUBFOLDER_NAME:
             print(f"Skipping '{item_name}' directory (as it is the output folder).")

    print("-" * 50)
    if processed_folder_count == 0:
        print(f"No valid ID subfolders (starting with '{FOLDER_PREFIX_TO_PROCESS}') found in {parent_folder_path}.")
    else:
        print(f"Batch processing finished. Processed {processed_folder_count} ID folders.")

if __name__ == "__main__":
    # 如果在沒有圖形介面的伺服器上運行，可以取消下一行的註釋
    # import matplotlib; matplotlib.use('Agg')
    main()

--- DIAGNOSTIC INFORMATION ---
Python Executable: c:\Users\User\AppData\Local\Programs\Python\Python310\python.exe
PyTorch Version: 2.5.1+cu121
PyTorch CUDA Version: 12.1
Is CUDA Available: Yes
----------------------------



c:\Users\User\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU is available: NVIDIA GeForce RTX 4080 SUPER
Running calculations on GPU.
Loading CLIP model 'openai/clip-vit-base-patch32' (forcing re-download and using safetensors)...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00,  4.37it/s]


CLIP model loaded successfully and moved to device: cuda
Starting batch processing for folders within: student_week_photo\1019
--------------------------------------------------
Processing folder: ID_1
  Skipping folder 'ID_1': Needs at least 2 images, found 0.
--------------------------------------------------
Processing folder: ID_10
  Found 11470 images for 'ID_10'.


...

  Dynamic threshold for 'ID_7': 0.8889 (Based on 10th percentile, min 0.85)
  Keyframes subfolder: student_week_photo\1019\ID_7\Keyframes
  Saved 1772 new unique keyframes.
  Plot saved to: student_week_photo\1019\ID_7\similarity_plot_ID_7.png
  Finished processing 'ID_7'. Saved 1772 unique keyframes.
--------------------------------------------------
Processing folder: ID_8
  Found 12821 images for 'ID_8'.


  Calculating similarities for 'ID_8': 100%|██████████| 12820/12820 [03:14<00:00, 66.08it/s]


  Dynamic threshold for 'ID_8': 0.8500 (Based on 10th percentile, min 0.85)
  Keyframes subfolder: student_week_photo\1019\ID_8\Keyframes
  Saved 3344 new unique keyframes.
  Plot saved to: student_week_photo\1019\ID_8\similarity_plot_ID_8.png
  Finished processing 'ID_8'. Saved 3344 unique keyframes.
--------------------------------------------------
Processing folder: ID_9
  Found 12202 images for 'ID_9'.


  Calculating similarities for 'ID_9': 100%|██████████| 12201/12201 [03:05<00:00, 65.82it/s]


  Dynamic threshold for 'ID_9': 0.8936 (Based on 10th percentile, min 0.85)
  Keyframes subfolder: student_week_photo\1019\ID_9\Keyframes
  Saved 2051 new unique keyframes.
  Plot saved to: student_week_photo\1019\ID_9\similarity_plot_ID_9.png
  Finished processing 'ID_9'. Saved 2051 unique keyframes.
--------------------------------------------------
Batch processing finished. Processed 29 ID folders.


# 刪除多餘資料夾

In [ ]:
# -*- coding: utf-8 -*-
import os
import shutil
from tqdm import tqdm # 用於顯示漂亮的進度條

# --- 參數設定 ---

# 1. 輸入目錄：指向包含所有待處理 ID_X 資料夾的路徑
#    請將此路徑替換為您實際的資料夾路徑
SOURCE_DIR = r"C:\Users\User\Desktop\test\output_crops_individual_55_NEW_Fil"

# 2. 輸出目錄：存放篩選和重新編號後結果的新資料夾路徑
#    如果此資料夾不存在，程式會自動建立
OUTPUT_DIR = r"C:\Users\User\Desktop\test\DEMO_output_crops_individual_55_NEW_Fil"

# 3. 圖片數量閾值：資料夾內的圖片數量必須 "大於或等於" 此數值才會被保留
MIN_IMAGE_COUNT_THRESHOLD = 150

# 4. 新資料夾的命名範本
NEW_FOLDER_PREFIX = "ID_" # 您可以改成 "ID_" 或其他喜歡的前綴

# --------------------

def filter_and_renumber_folders():
    """
    過濾掉圖片數量過少的資料夾，並將合格的資料夾複製到新目錄並重新編號。
    """
    # 檢查來源目錄是否存在
    if not os.path.isdir(SOURCE_DIR):
        print(f"錯誤：來源目錄 '{SOURCE_DIR}' 不存在或不是一個有效的資料夾。")
        return

    # 準備輸出目錄，如果已存在則先清空，確保結果是乾淨的
    if os.path.exists(OUTPUT_DIR):
        print(f"警告：輸出目錄 '{OUTPUT_DIR}' 已存在，將會被清空以確保結果乾淨。")
        shutil.rmtree(OUTPUT_DIR)
    os.makedirs(OUTPUT_DIR)
    print(f"將把結果儲存到新的目錄: {OUTPUT_DIR}")

    # --- 第一步：掃描所有資料夾並篩選出合格的 ---
    
    # 獲取所有符合 'ID_' 開頭的子資料夾
    try:
        all_id_folders = [d for d in os.listdir(SOURCE_DIR) if os.path.isdir(os.path.join(SOURCE_DIR, d)) and d.startswith("ID_")]
    except FileNotFoundError:
        print(f"錯誤：無法訪問來源目錄 '{SOURCE_DIR}'。")
        return
        
    if not all_id_folders:
        print("在來源目錄中沒有找到任何 'ID_' 開頭的資料夾。")
        return

    print(f"\n找到 {len(all_id_folders)} 個待處理的資料夾，開始進行篩選...")
    
    qualified_folders = []
    for folder_name in tqdm(all_id_folders, desc="掃描與計數"):
        folder_path = os.path.join(SOURCE_DIR, folder_name)
        
        # 計算資料夾內的圖片數量
        try:
            image_count = len([f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        except Exception as e:
            print(f"\n讀取資料夾 '{folder_name}' 時發生錯誤: {e}")
            continue

        # 判斷是否滿足閾值條件
        if image_count >= MIN_IMAGE_COUNT_THRESHOLD:
            qualified_folders.append(folder_path) # 將完整路徑加入合格列表
        else:
            print(f"\n已過濾資料夾: '{folder_name}' (圖片數量: {image_count} < {MIN_IMAGE_COUNT_THRESHOLD})")

    if not qualified_folders:
        print("\n篩選完畢，沒有任何資料夾滿足條件。")
        return
        
    print(f"\n篩選完畢！共有 {len(qualified_folders)} 個資料夾滿足條件 (圖片數量 >= {MIN_IMAGE_COUNT_THRESHOLD})。")

    # --- 第二步：複製並重新編號 ---
    
    print("\n開始複製檔案並重新命名...")
    
    # 為了讓輸出資料夾名稱排序好看 (ID_1, ID_2 ... ID_10 而不是 ID_1, ID_10...)
    # 雖然這裡不需要，但保留這個好習慣
    # qualified_folders.sort() 
    
    new_id_counter = 1
    for src_folder_path in tqdm(qualified_folders, desc="複製與重編號"):
        # 組合新的資料夾名稱
        new_folder_name = f"{NEW_FOLDER_PREFIX}{new_id_counter}"
        dest_folder_path = os.path.join(OUTPUT_DIR, new_folder_name)
        
        # 使用 shutil.copytree 完整地複製整個資料夾
        try:
            shutil.copytree(src_folder_path, dest_folder_path)
        except Exception as e:
            print(f"\n從 '{src_folder_path}' 複製到 '{dest_folder_path}' 時發生錯誤: {e}")
            continue
            
        new_id_counter += 1

    print("\n處理完成！所有合格的資料夾已被複製並重新編號。")

if __name__ == "__main__":
    filter_and_renumber_folders()

### 印出每個資料夾檔案數量

In [ ]:
# -*- coding: utf-8 -*-
import os

# --- 參數設定 ---

# 1. 目標目錄：指向包含所有 Person_X 資料夾的路徑
#    請將此路徑替換為您實際的資料夾路徑
TARGET_DIR = r"C:\Users\User\Desktop\test\0604__individual_full_class_mid"

# 2. 資料夾前綴：指定您要搜尋的資料夾名稱開頭
FOLDER_PREFIX = "ID_"

# --------------------

def count_files_in_prefixed_folders():
    """
    計算指定目錄下，所有特定前綴資料夾中的檔案數量。
    """
    # 檢查目標目錄是否存在
    if not os.path.isdir(TARGET_DIR):
        print(f"錯誤：目錄 '{TARGET_DIR}' 不存在或不是一個有效的資料夾。")
        return

    print(f"正在掃描目錄: '{TARGET_DIR}'")
    print(f"尋找前綴為 '{FOLDER_PREFIX}' 的資料夾...\n")

    # 尋找所有符合前綴的資料夾
    try:
        # 使用 os.scandir() 效率比 os.listdir() 稍高，且可以直接判斷是否為目錄
        person_folders = [entry.name for entry in os.scandir(TARGET_DIR) if entry.is_dir() and entry.name.startswith(FOLDER_PREFIX)]
    except FileNotFoundError:
        print(f"錯誤：無法訪問目錄 '{TARGET_DIR}'。")
        return

    # 如果沒有找到任何符合條件的資料夾
    if not person_folders:
        print("在指定目錄中沒有找到任何符合條件的資料夾。")
        return

    # 為了讓輸出結果更整齊，可以對資料夾名稱進行自然排序
    # 例如，讓 Person_10 出現在 Person_9 之後，而不是 Person_1 之後
    person_folders.sort(key=lambda name: int(name.split('_')[-1]))

    total_folders_counted = 0
    total_files_counted = 0

    # 遍歷每一個找到的資料夾並計數
    for folder_name in person_folders:
        folder_path = os.path.join(TARGET_DIR, folder_name)
        
        try:
            # os.listdir() 返回目錄下的檔案和資料夾列表
            files_in_folder = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
            file_count = len(files_in_folder)
            
            # 印出結果
            print(f"資料夾: {folder_name:<15} | 檔案數量: {file_count}")
            
            total_folders_counted += 1
            total_files_counted += file_count

        except Exception as e:
            print(f"讀取資料夾 '{folder_name}' 時發生錯誤: {e}")
    
    # 打印總結信息
    print("\n------------------------------------")
    print(f"統計完成！")
    print(f"總共掃描了 {total_folders_counted} 個資料夾。")
    print(f"所有資料夾中的檔案總數為: {total_files_counted}")
    print("------------------------------------")


if __name__ == "__main__":
    count_files_in_prefixed_folders()

# 合併影片

### GUI & GPU 加速版本

In [ ]:
# -*- coding: utf-8 -*-

import os
import tkinter as tk
from tkinter import filedialog
import subprocess

# ==============================================================================
# --- 設定區 ---
# 這個設定現在對新的合併方法有效了！
# ==============================================================================
GPU_BRAND = "NVIDIA" 
# ==============================================================================


def check_ffmpeg():
    """檢查系統中是否能找到 ffmpeg 命令。"""
    print("正在檢查 FFmpeg 環境...")
    try:
        subprocess.run(['ffmpeg', '-version'], check=True, capture_output=True)
        print("FFmpeg 環境正常！")
        return True
    except FileNotFoundError:
        print("\n錯誤：找不到 'ffmpeg' 命令。請安裝 FFmpeg 並設定環境變數。")
        return False
    except Exception as e:
        print(f"檢查 FFmpeg 時發生未知錯誤: {e}")
        return False

def merge_videos_robust(video_paths, output_path):
    """
    使用 FFmpeg 的 concat filter 進行穩健的影片合併。
    此方法會重新編碼，可以解決時間戳不連續的問題，並支援 GPU 加速。
    """
    print("開始使用 FFmpeg 的 concat filter 進行穩健合併...")

    # 1. 根據 GPU_BRAND 選擇編碼器
    codec_map = {
        "NVIDIA": "h264_nvenc",
        "AMD": "h264_amf",
        "INTEL": "h264_qsv",
        "CPU": "libx264"
    }
    selected_codec = codec_map.get(GPU_BRAND.upper(), "libx264")
    print(f"將使用編碼器: {selected_codec}")
    
    try:
        # 2. 建立 FFmpeg 命令
        # 為每個輸入檔案加上 -i 參數
        command = ['ffmpeg', '-y']
        filter_complex_str = ""
        
        for i, path in enumerate(video_paths):
            command.extend(['-i', path])
            # 建立濾鏡字串，例如 [0:v][0:a][1:v][1:a]...
            filter_complex_str += f"[{i}:v:0][{i}:a:0]"

        # 組合完整的 filter_complex 參數
        num_videos = len(video_paths)
        filter_complex_str += f"concat=n={num_videos}:v=1:a=1[outv][outa]"
        
        command.extend([
            '-filter_complex', filter_complex_str,
            '-map', '[outv]',
            '-map', '[outa]',
            '-c:v', selected_codec,  # 使用 GPU 或 CPU 進行影片編碼
            '-c:a', 'aac',           # 音訊編碼
            output_path
        ])

        print(f"\n正在執行 FFmpeg 命令:\n{' '.join(command)}\n")

        # 3. 執行命令
        # 注意：對於長時間運行的進程，我們不再捕獲輸出，而是讓它直接顯示在終端機上
        # 這樣您可以看到即時的進度條
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True, encoding='utf-8')
        
        for line in process.stdout:
            print(line, end='') # 即時輸出 FFmpeg 的進度

        process.wait() # 等待進程結束

        if process.returncode == 0:
            print("\n影片合併成功！")
        else:
            print(f"\n錯誤：FFmpeg 執行失敗，返回碼: {process.returncode}")

    except Exception as e:
        print(f"發生未知錯誤: {e}")


def main():
    """主函式，使用 GUI 讓使用者選擇檔案並執行合併。"""
    if not check_ffmpeg():
        input("\n請按 Enter 鍵結束程式...")
        return

    root = tk.Tk()
    root.withdraw()

    print("\n====================================")
    print("  MP4 影片合併工具 (穩健版)       ")
    print(f"  (加速設定: {GPU_BRAND})")
    print("====================================")
    print("此版本能處理多數影片，並使用GPU加速。")
    print("程式將會彈出一個視窗，請選擇您想合併的影片檔案。")

    file_types = [("影片檔案", "*.mp4 *.mov *.avi *.mkv"), ("所有檔案", "*.*")]
    video_paths = filedialog.askopenfilenames(title="請選擇要合併的影片檔案", filetypes=file_types)

    if not video_paths:
        print("\n您沒有選擇任何檔案。程式已結束。")
        return
    if len(video_paths) < 2:
        print("\n至少需要兩個影片才能進行合併。程式已結束。")
        return

    print(f"\n您已選擇 {len(video_paths)} 個影片。")
    print("接下來，請選擇合併後影片的儲存位置與檔名。")

    output_path = filedialog.asksaveasfilename(
        title="請選擇儲存位置與檔名", filetypes=[("MP4 檔案", "*.mp4")],
        defaultextension=".mp4", initialfile="merged_video.mp4"
    )
    
    if not output_path:
        print("\n您沒有選擇儲存位置。程式已結束。")
        return

    # 使用新的穩健合併函式
    merge_videos_robust(video_paths, output_path)


if __name__ == "__main__":
    main()
    input("\n按 Enter 鍵結束程式...")

# 影像轉檔

### .mp4 轉 ,mp3

In [1]:
import os
from moviepy import VideoFileClip

def convert_mp4_to_api_friendly_mp3(mp4_input_path: str, mp3_output_path: str) -> bool:
    """
    將指定的 .mp4 檔案轉換為對 API 友善的 .mp3 格式。

    Args:
        mp4_input_path (str): 輸入的 .mp4 檔案完整路徑。
        mp3_output_path (str): 輸出的 .mp3 檔案完整路徑。

    Returns:
        bool: 如果轉換成功，返回 True，否則返回 False。
    """
    try:
        print(f"正在讀取影片檔案：{mp4_input_path}")
        # 讀取影片檔案
        video_clip = VideoFileClip(mp4_input_path)
        
        print("正在轉換音訊，請稍候...")
        # 提取音訊並寫入檔案，同時指定編碼器和位元速率
        audio_clip = video_clip.audio
        # audio_clip.write_audiofile(
        #     mp3_output_path, 
        #     codec='libmp3lame',  # <--- [關鍵] 使用標準的 LAME MP3 編碼器
        #     bitrate="192k"      # <--- [關鍵] 設定標準的位元速率
        # )

        audio_clip.write_audiofile(
            mp3_output_path, 
            codec='libmp3lame',
            bitrate="128k",           # <--- [修改] 從 192k 降為 128k
            ffmpeg_params=["-ac", "1"], # <--- [新增] 強制轉為單聲道 (1 channel)，提升低流量下的人聲清晰度
            logger='bar'              # 保持進度條
        )
        
        # 關閉 clip 以釋放資源
        audio_clip.close()
        video_clip.close()

        print(f"✅ 轉檔成功！已將音訊儲存為：{mp3_output_path}")
        return True

    except Exception as e:
        print(f"❌ 轉檔過程中發生錯誤：{e}")
        # 如果有 clip 物件存在，也嘗試關閉
        if 'video_clip' in locals() and video_clip:
            video_clip.close()
        if 'audio_clip' in locals() and audio_clip:
            audio_clip.close()
        return False

# --- 主程式執行區 ---
# 這段程式碼只有在直接執行這個 .py 檔案時才會被觸發
if __name__ == "__main__":
    # 讓使用者輸入 .mp4 檔案路徑
    user_video_path = input("請輸入 .mp4 檔案的完整路徑：").strip().strip('"') # 去除可能的多餘引號
    
    # 檢查檔案是否存在
    if not os.path.exists(user_video_path):
        print("錯誤：檔案不存在。請檢查路徑是否正確。")
    # 確認檔案副檔名是否為 .mp4
    elif not user_video_path.lower().endswith('.mp4'):
        print("錯誤：請提供一個有效的 .mp4 檔案。")
    else:
        # 自動生成輸出的 .mp3 檔案路徑
        base_name = os.path.splitext(user_video_path)[0]
        output_mp3_path = base_name + "_converted.mp3"
        
        # 呼叫我們封裝好的轉換函式
        convert_mp4_to_api_friendly_mp3(user_video_path, output_mp3_path)

正在讀取影片檔案：SynologyDrive\image\上課影片\0928\老師\0928.mp4
正在轉換音訊，請稍候...
MoviePy - Writing audio in SynologyDrive\image\上課影片\0928\老師\0928_converted.mp3


MoviePy - Done.
✅ 轉檔成功！已將音訊儲存為：SynologyDrive\image\上課影片\0928\老師\0928_converted.mp3


### .mkv轉.mp4


In [ ]:
import os
import subprocess
import sys

def check_ffmpeg():
    """檢查 FFmpeg 是否可執行"""
    try:
        # 嘗試執行 ffmpeg -version 並抑制輸出
        subprocess.run(['ffmpeg', '-version'], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        print("成功找到 FFmpeg。")
        return True
    except FileNotFoundError:
        print("\n錯誤：找不到 'ffmpeg' 指令。")
        print("請確認您已經安裝 FFmpeg 並且將其執行檔路徑加入到系統的環境變數 PATH 中。")
        print("您可以從 https://ffmpeg.org/download.html 下載 FFmpeg。")
        return False
    except subprocess.CalledProcessError:
        # 即使找到 ffmpeg，如果 -version 命令本身出錯，也視為不可用
        print("\n錯誤：執行 'ffmpeg -version' 時發生錯誤，請檢查您的 FFmpeg 安裝。")
        return False
    except Exception as e:
        print(f"\n檢查 FFmpeg 時發生未預期的錯誤: {e}")
        return False

def convert_mkv_to_mp4(mkv_filepath):
    """
    使用 ffmpeg 將指定的 MKV 檔案轉換為 MP4 檔案。
    檔案將儲存在與來源檔案相同的目錄下，名稱相同但副檔名為 .mp4。
    此版本進行重新編碼以確保相容性。

    Args:
        mkv_filepath (str): 輸入的 .mkv 檔案完整路徑。

    Returns:
        bool: 轉換成功返回 True，失敗返回 False。
    """
    # --- 1. 驗證輸入檔案 ---
    if not os.path.isfile(mkv_filepath):
        print(f"錯誤：找不到檔案 '{mkv_filepath}'")
        return False
    if not mkv_filepath.lower().endswith('.mkv'):
        print(f"錯誤：輸入檔案 '{os.path.basename(mkv_filepath)}' 不是 .mkv 檔案。")
        return False

    # --- 2. 建立輸出檔案路徑 ---
    try:
        directory = os.path.dirname(mkv_filepath)
        filename_without_ext = os.path.splitext(os.path.basename(mkv_filepath))[0]
        mp4_filepath = os.path.join(directory, filename_without_ext + ".mp4")
    except Exception as e:
        print(f"錯誤：無法建立輸出檔案路徑。 {e}")
        return False

    print(f"\n準備轉換 (進行重新編碼)：") # 提示用戶模式改變
    print(f"  來源檔案：{mkv_filepath}")
    print(f"  目標檔案：{mp4_filepath}")

    # --- 3. 建立 FFmpeg 指令 ---
    # 現在不再使用 -c copy，而是指定重新編碼
    command = [
        'ffmpeg',
        '-y',                     # 自動覆蓋輸出檔案
        '-i', mkv_filepath,       # 輸入檔案
        # --- 使用建議的重新編碼參數 ---
        '-c:v', 'libx264',        # 使用 H.264 視訊編碼器
        '-crf', '23',             # 設定視訊品質 (數值越低越好，23是個好平衡)
        '-preset', 'medium',      # 編碼速度預設 (可改 fast, faster, slow, slower...)
        '-c:a', 'aac',            # 使用 AAC 音訊編碼器
        '-b:a', '128k',           # 設定音訊位元率 (可改 192k, 256k...)
        # --- 重新編碼參數結束 ---
        '-map_metadata', '0',     # 複製檔案元數據
        '-movflags', '+faststart',# 優化 MP4 結構
        mp4_filepath              # 輸出檔案
    ]


    print(f"\n執行 FFmpeg 指令：\n{' '.join(command)}")
    print("\n轉換中 (重新編碼可能需要較長時間)，請稍候...") # 提示時間可能較長

    # --- 4. 執行 FFmpeg 指令 ---
    try:
        # 使用 subprocess.run 執行指令
        result = subprocess.run(command, check=True, capture_output=True, text=True, encoding='utf-8', errors='replace')
        print("\n-----------------------------------")
        print(f"成功！轉換完成。")
        print(f"輸出檔案儲存於： '{mp4_filepath}'")
        print("-----------------------------------")
        # print("\nFFmpeg 詳細輸出訊息 (偵錯用):")
        # print(result.stdout)
        # print(result.stderr) # 可以查看 stderr 了解編碼過程細節
        return True

    except subprocess.CalledProcessError as e:
        # FFmpeg 指令執行失敗
        print("\n!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
        print(f"錯誤：FFmpeg 轉換失敗 (返回碼 {e.returncode})。")
        print("即使進行了重新編碼，轉換仍然失敗。請檢查 FFmpeg 的錯誤訊息以獲取詳細資訊。")
        print("\n----- FFmpeg 錯誤訊息 -----")
        print(e.stderr) # 顯示 FFmpeg 的錯誤輸出以幫助診斷
        print("---------------------------")
        print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
        # 轉換失敗時，嘗試刪除可能已產生但不完整的輸出檔案
        if os.path.exists(mp4_filepath):
            try:
                os.remove(mp4_filepath)
                print(f"\n已刪除不完整的輸出檔案：'{mp4_filepath}'")
            except OSError as remove_err:
                print(f"\n警告：無法刪除不完整的輸出檔案 '{mp4_filepath}': {remove_err}")
        return False
    except Exception as e:
        # 其他可能的錯誤
        print("\n!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
        print(f"執行 FFmpeg 時發生未預期的錯誤：{e}")
        print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
        return False

# --- 主程式碼的其他部分保持不變 ---
# if __name__ == "__main__":
#    ... (檢查 ffmpeg, 獲取輸入等)
# --- 程式主執行區塊 ---
if __name__ == "__main__":
    print("--- MKV 轉 MP4 轉換器 (使用 FFmpeg) ---")

    # 檢查 FFmpeg 是否存在且可用
    if not check_ffmpeg():
        print("\n腳本無法繼續執行，因為找不到或無法執行 FFmpeg。")
        input("請按 Enter 鍵結束...")
        sys.exit(1) # 退出程式

    # 迴圈直到使用者輸入有效的檔案路徑
    while True:
        mkv_file_input = input("\n請輸入要轉換的 .mkv 檔案完整路徑\n(或將檔案拖曳到此視窗後按 Enter)：").strip()

        # 處理 Windows 拖曳檔案時可能產生的引號
        if mkv_file_input.startswith('"') and mkv_file_input.endswith('"'):
            mkv_file_input = mkv_file_input[1:-1]
        if mkv_file_input.startswith("'") and mkv_file_input.endswith("'"):
             mkv_file_input = mkv_file_input[1:-1]

        # 檢查檔案是否存在且是 .mkv 檔
        if os.path.isfile(mkv_file_input) and mkv_file_input.lower().endswith('.mkv'):
            break # 輸入有效，跳出迴圈
        elif not os.path.exists(mkv_file_input):
             print(f"錯誤：找不到路徑 '{mkv_file_input}'。請檢查路徑是否正確。")
        elif not os.path.isfile(mkv_file_input):
             print(f"錯誤：'{mkv_file_input}' 不是一個檔案。請輸入檔案路徑。")
        else: # 存在但不是 .mkv
             print(f"錯誤：檔案 '{os.path.basename(mkv_file_input)}' 的副檔名不是 .mkv。")

    # 執行轉換
    success = convert_mkv_to_mp4(mkv_file_input)

    if success:
        print("\n轉換操作已完成。")
    else:
        print("\n轉換操作失敗。")

    # 在 Windows 上防止視窗直接關閉
    if sys.platform == "win32":
       input("\n按 Enter 鍵結束程式...")

# 黑板影像

### 影像重整

In [ ]:
import os
import shutil
import re
from datetime import timedelta

# --- 設定區 (已根據您的最新截圖修正) ---

# 1. 來源資料夾路徑 (已修正為 seience 和 Fll)
#    注意：Windows 路徑中的 Fll 和 FIl 可能會被視為相同，但最好保持一致
SOURCE_FOLDERS = [
    r'C:\Users\User\Desktop\test\0706_English_1',
    r'C:\Users\User\Desktop\test\0706_English_2',
    r'C:\Users\User\Desktop\test\0706_English_3'
]

# 2. 輸出資料夾路徑
OUTPUT_FOLDER = 'C:/Users/User/Desktop/test/0706_English'

# 3. 影片之間的間距 (分鐘)
GAP_MINUTES = 3

# 4. 影片的幀率 (Frame Rate)
FRAME_RATE = 30

# --- 程式碼主體 (已修正檔名比對邏輯) ---

def parse_filename(filename):
    """從檔案名稱中解析出時間和幀數"""
    # *** 主要修正點 ***
    # 在 blackboard_0 後面的底線 _ 加上 ?，代表這個底線可有可無
    # 這樣就能同時匹配 "blackboard_00h..." 和 "blackboard_0_0h..."
    match = re.match(r"blackboard_0_?(\d+)h(\d+)m(\d+)s_f(\d+)\.png", filename, re.IGNORECASE)
    if not match:
        return None
    
    h, m, s, f = map(int, match.groups())
    return {
        "time": timedelta(hours=h, minutes=m, seconds=s),
        "frame": f
    }

def format_timedelta_to_hms(td):
    """將 timedelta 物件格式化為 HhMmSs 字串"""
    total_seconds = int(td.total_seconds())
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60
    return f"{hours:02d}h{minutes:02d}m{seconds:02d}s"

def process_folders():
    """主處理函數"""
    if not os.path.exists(OUTPUT_FOLDER):
        os.makedirs(OUTPUT_FOLDER)
        print(f"已建立輸出資料夾: {OUTPUT_FOLDER}")

    cumulative_time_offset = timedelta(0)
    gap_timedelta = timedelta(minutes=GAP_MINUTES)
    
    print("開始處理圖片整合...")

    for i, folder_path in enumerate(SOURCE_FOLDERS):
        print(f"\n--- 正在處理資料夾 {i+1}/{len(SOURCE_FOLDERS)}: {folder_path} ---")

        if not os.path.isdir(folder_path):
            print(f"!!!!!! 警告：找不到此資料夾，請檢查上面的路徑設定是否正確。已跳過。")
            continue

        files_to_process = []
        all_files_in_folder = os.listdir(folder_path)
        if not all_files_in_folder:
            print("  此資料夾是空的。")
        
        for filename in all_files_in_folder:
            if filename.lower().endswith('.png'):
                parsed_data = parse_filename(filename)
                if parsed_data:
                    full_path = os.path.join(folder_path, filename)
                    files_to_process.append((full_path, parsed_data['time']))
        
        if not files_to_process:
            print("\n  處理完成：此資料夾中沒有找到任何符合格式的圖片。")
            continue
            
        files_to_process.sort(key=lambda x: x[1])

        # 處理該資料夾中的每一個檔案
        for original_path, original_time in files_to_process:
            new_time = original_time + cumulative_time_offset
            new_total_seconds = new_time.total_seconds()
            new_frame_number = int(new_total_seconds * FRAME_RATE) + 1
            new_time_str = format_timedelta_to_hms(new_time)
            # 統一輸出成沒有底線的格式
            new_filename = f"blackboard_0{new_time_str}_f{new_frame_number:07d}.png"
            destination_path = os.path.join(OUTPUT_FOLDER, new_filename)
            shutil.copy2(original_path, destination_path)
        
        print(f"\n  處理完成：已成功處理 {len(files_to_process)} 個檔案。")

        last_file_original_time = files_to_process[-1][1]
        end_of_current_sequence = last_file_original_time + cumulative_time_offset
        cumulative_time_offset = end_of_current_sequence + gap_timedelta
        
        print(f"  目前累計時間結束於: {format_timedelta_to_hms(end_of_current_sequence)}")
        print(f"  下一個影片的起始時間將加上偏移: {format_timedelta_to_hms(cumulative_time_offset)}")

    print("\n--- 所有資料夾處理完畢！---")
    print(f"整合後的檔案已儲存至: {OUTPUT_FOLDER}")

if __name__ == "__main__":
    process_folders()

### 黑板擷取

In [ ]:
# -*- coding: utf-8 -*-
import cv2
import os
import numpy as np

# ---------------------------
# 1. 設定 (Configuration)
# ---------------------------
# --- 使用者需要設定 ---
CAPTURE_INTERVAL_SECONDS = 10
SAVE_FOLDER_BLACKBOARD = "0824_English"
# --- 設定結束 ---

# 全域變數
roi_points = []
roi_defined = False
final_roi = None

os.makedirs(SAVE_FOLDER_BLACKBOARD, exist_ok=True)

# ---------------------------
# 2. 滑鼠點擊回調函數 (不變)
# ---------------------------
def select_roi(event, x, y, flags, param):
    global roi_points, roi_defined, final_roi
    frame_copy = param['frame'].copy()

    if event == cv2.EVENT_LBUTTONDOWN and not roi_defined:
        if len(roi_points) < 4:
            roi_points.append((x, y))
            print(f"點擊: {len(roi_points)}/4 - 座標: ({x}, {y})")
            for pt in roi_points:
                cv2.circle(frame_copy, pt, 5, (0, 255, 0), -1)
            if len(roi_points) == 4:
                pts = np.array(roi_points, dtype=np.int32)
                x_roi, y_roi, w_roi, h_roi = cv2.boundingRect(pts)
                final_roi = (x_roi, y_roi, x_roi + w_roi, y_roi + h_roi)
                roi_defined = True
                print(f"ROI 區域已定義: {final_roi}")
                print("按任意鍵繼續...")
                cv2.rectangle(frame_copy, (final_roi[0], final_roi[1]), (final_roi[2], final_roi[3]), (0, 0, 255), 2)
            elif len(roi_points) > 1:
                 cv2.polylines(frame_copy, [np.array(roi_points)], isClosed=False, color=(0,255,0), thickness=1)

    cv2.imshow("Select Blackboard ROI", frame_copy)

# ---------------------------
# 3. 毫秒轉 HHhMMmSSs 格式函數 (不變)
# ---------------------------
def format_milliseconds(milliseconds):
    """將毫秒轉換為 HHhMMmSSs 格式的字串"""
    if milliseconds < 0:
        milliseconds = 0
    total_seconds = int(milliseconds / 1000)
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60
    return f"{hours:02d}h{minutes:02d}m{seconds:02d}s"

# ---------------------------
# 4. 主程式 (使用 cap.set() 實現高效跳轉)
# ---------------------------
def main():
    global roi_points, roi_defined, final_roi

    video_path = input("請輸入影片路徑 (Enter video path): ").strip().strip('"')
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"無法開啟影片 (Cannot open video): {video_path}")
        return

    # --- 步驟 1: 選取 ROI (不變) ---
    ret, first_frame = cap.read()
    if not ret:
        print("無法讀取影片的第一幀。")
        cap.release()
        return

    print("\n" + "="*30)
    print("請在彈出的視窗中，用滑鼠左鍵依序點擊黑板的四個角落。")
    print("確認無誤後，請按鍵盤上的任意鍵繼續擷取。")
    print("="*30 + "\n")

    cv2.namedWindow("Select Blackboard ROI")
    param_dict = {'frame': first_frame}
    cv2.setMouseCallback("Select Blackboard ROI", select_roi, param_dict)
    cv2.imshow("Select Blackboard ROI", first_frame)
    while not roi_defined:
         key_select = cv2.waitKey(20) & 0xFF
         if key_select != 255 and roi_defined: break
         elif key_select == ord('q'):
              print("在選取 ROI 時退出。")
              cv2.destroyAllWindows(); cap.release(); return
    cv2.destroyWindow("Select Blackboard ROI")
    if not roi_defined: return
    
    print(f"將擷取此區域: {final_roi}")
    print(f"開始高效處理，每隔 {CAPTURE_INTERVAL_SECONDS} 秒影片時間跳轉並擷取一次...")

    # --- 步驟 2: 高效跳轉與擷取 ---
    roi_x1, roi_y1, roi_x2, roi_y2 = final_roi
    
    # *** 獲取影片總長度以便設定迴圈邊界 ***
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_msec = (total_frames / fps) * 1000 if fps > 0 else 0
    
    # *** 處理第 0 秒 (使用已讀取的第一幀) ***
    print("--- Capturing (Video Time: 00h00m00s) ---")
    blackboard_roi = first_frame[roi_y1:roi_y2, roi_x1:roi_x2]
    save_filename = f"blackboard_00h00m00s_f1.png"
    save_path = os.path.join(SAVE_FOLDER_BLACKBOARD, save_filename)
    cv2.imwrite(save_path, blackboard_roi)
    print(f"  Saved: {save_filename}")

    # *** 使用迴圈處理後續的時間點 ***
    capture_interval_msec = CAPTURE_INTERVAL_SECONDS * 1000
    current_msec = capture_interval_msec

    while current_msec < duration_msec:
        print(f"\nJumping to: {format_milliseconds(current_msec)}")
        
        # *** 核心：跳轉到指定毫秒 ***
        cap.set(cv2.CAP_PROP_POS_MSEC, current_msec)
        
        ret, frame = cap.read()
        if not ret:
            print(f"  Warning: 無法在時間點 {format_milliseconds(current_msec)} 讀取到畫面。")
            break

        # 讀取成功後，獲取精確的時間和幀號
        actual_msec = cap.get(cv2.CAP_PROP_POS_MSEC)
        actual_frame_num = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
        print(f"--- Capturing (Video Time: {format_milliseconds(actual_msec)}) ---")
        
        # 擷取並儲存
        blackboard_roi = frame[roi_y1:roi_y2, roi_x1:roi_x2]
        video_time_str = format_milliseconds(actual_msec)
        
        # *** 這裡已經修正錯誤 ***
        save_filename = f"blackboard_{video_time_str}_f{actual_frame_num}.png"
        
        save_path = os.path.join(SAVE_FOLDER_BLACKBOARD, save_filename)
        cv2.imwrite(save_path, blackboard_roi)
        print(f"  Saved: {save_filename}")

        # 更新到下一個目標時間點
        current_msec += capture_interval_msec

    cap.release()
    print("\nProcessing finished.")

if __name__ == "__main__":
    main()

### 黑板影像過濾

In [ ]:
import os
import shutil
from PIL import Image
import imagehash # 導入 imagehash
from tqdm import tqdm # 用於顯示進度條 (pip install tqdm)
import re # 用於更精確的排序

# ---------------------------
# 設定 (Configuration)
# ---------------------------
SOURCE_FOLDER = r"C:\Users\User\Desktop\test\teacher_blackboard\0817_English"  # 包含原始擷圖的資料夾
FILTERED_FOLDER = r"C:\Users\User\Desktop\test\note_blackboard\0817_English_filtered" # 儲存過濾後圖片的新資料夾
HASH_DISTANCE_THRESHOLD = 20      # 感知哈希距離閾值。值越小，要求圖片越相似才算重複。
                                  # 建議從 3-8 開始測試。值越大，過濾掉的圖片越多。
HASH_FUNC = imagehash.phash       # 使用哪種哈希算法 (phash 通常效果不錯，其他還有 ahash, dhash, whash)

# ---------------------------
# 輔助函數：從檔名提取數字用於排序
# ---------------------------
def get_sort_key(filename):
    """
    從檔名 (例如 blackboard_20231027_103005_f123.png) 提取數字部分，
    優先使用幀號 fxxx，其次是時間戳，用於精確排序。
    """
    # 嘗試提取幀號 fxxx
    match_f = re.search(r'_f(\d+)\.', filename)
    if match_f:
        return int(match_f.group(1))

    # 如果沒有幀號，嘗試提取時間戳 YYYYMMDD_HHMMSS
    match_ts = re.search(r'(\d{8}_\d{6})', filename)
    if match_ts:
        # 將時間戳轉換為數字以便排序
        try:
            # 將 YYYYMMDD_HHMMSS 轉為一個大整數
            return int(match_ts.group(1).replace('_', ''))
        except ValueError:
             # 如果轉換失敗，返回檔名本身
             print(f"Warning: Could not parse timestamp in {filename}")
             return filename

    # 如果都找不到，返回原始檔名
    return filename

# ---------------------------
# 主過濾邏輯
# ---------------------------
def filter_images():
    # 檢查來源資料夾是否存在
    if not os.path.isdir(SOURCE_FOLDER):
        print(f"錯誤：來源資料夾 '{SOURCE_FOLDER}' 不存在。")
        return

    # 創建過濾後的資料夾
    os.makedirs(FILTERED_FOLDER, exist_ok=True)
    print(f"過濾後的圖片將儲存到: '{FILTERED_FOLDER}'")

    # 獲取所有圖片檔案，並按檔名中的數字排序
    try:
        image_files = [f for f in os.listdir(SOURCE_FOLDER) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        # 使用自訂的 key 函數排序
        image_files.sort(key=get_sort_key)
    except Exception as e:
        print(f"錯誤：讀取或排序資料夾 '{SOURCE_FOLDER}' 中的檔案時出錯: {e}")
        return

    if not image_files:
        print(f"在 '{SOURCE_FOLDER}' 中找不到任何圖片檔案。")
        return

    print(f"找到 {len(image_files)} 張圖片，開始進行過濾...")

    last_kept_hash = None # 上一張保留圖片的哈希
    kept_count = 0
    skipped_count = 0
    error_count = 0

    # 使用 tqdm 顯示進度
    for filename in tqdm(image_files, desc="Filtering Images"):
        source_path = os.path.join(SOURCE_FOLDER, filename)
        dest_path = os.path.join(FILTERED_FOLDER, filename)

        try:
            # 打開圖片並計算哈希
            with Image.open(source_path) as img:
                current_hash = HASH_FUNC(img)

            # 如果是第一張圖片，或者與上一張保留的圖片差異足夠大
            if last_kept_hash is None or (current_hash - last_kept_hash) > HASH_DISTANCE_THRESHOLD:
                # 保留這張圖片：複製到目標資料夾
                shutil.copy2(source_path, dest_path) # copy2 會盡量保留元數據
                last_kept_hash = current_hash # 更新哈希
                kept_count += 1
            else:
                # 圖片與上一張保留的太相似，跳過
                skipped_count += 1
                # print(f"Skipping {filename} (Hash distance: {current_hash - last_kept_hash})") # 可選：打印跳過信息

        except Exception as e:
            print(f"\n處理檔案 '{filename}' 時發生錯誤: {e}")
            error_count += 1

    print("\n" + "="*30)
    print("過濾完成！")
    print(f"保留圖片數量: {kept_count}")
    print(f"跳過相似圖片數量: {skipped_count}")
    if error_count > 0:
        print(f"處理錯誤數量: {error_count}")
    print(f"結果已存儲在資料夾: '{FILTERED_FOLDER}'")
    print("="*30)

# ---------------------------
# 執行主邏輯
# ---------------------------
if __name__ == "__main__":
    filter_images()

# 語音轉文字

In [2]:
import os
import requests
import json
import time
import re
from dotenv import load_dotenv
from pydub import AudioSegment
from opencc import OpenCC
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from thefuzz import process as fuzz_process

# ==============================================================================
# --- 設定區 ---
# ==============================================================================
# 1. 您的長音訊檔案路徑
LONG_AUDIO_FILE_PATH = r"C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0928\老師\0928.mp3"

# 2. 最終輸出的文字稿檔案路徑
TXT_OUTPUT_FILE_PATH = r"C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0928\老師\0928_3.txt"

# 3. 切割音訊的臨時儲存資料夾
TEMP_CHUNK_FOLDER = r"C:\Users\User\Desktop\test\temp_audio_chunks"

# 4. 每個音訊片段的長度（單位：秒）
CHUNK_LENGTH_IN_SECONDS = 30

# 5. 同時發送給 API 的請求數量
MAX_CONCURRENT_WORKERS = 10

# 6. 學生姓名名單
STUDENT_NAMES = [
    "張玹寧", "許勻綺", "林沛潔", "張勛恆", "田騏民", "陳華銘", "王子誠",
    "邱立堯", "謝竣翔", "林程善", "烏鴻佾", "黃麗安", "邱珈翔", "徐品漢",
    "許宥琳", "李璿", "楊昀臻", "胡承恩", "莊雅筑", "羅妍溱", "吳文宣",
    "傅煜昊", "蕭凱億", "林弈萱", "徐芙暄", "陳昱蓁", "蘇亦淮"
]

# 7. 姓名校正的相似度分數門檻 (0-100)
NAME_CORRECTION_SCORE_CUTOFF = 85

# ===【修改點 1】=== 新增 PROMPT 參數，為模型提供上下文線索
PROMPT_TEXT = (
    "這是一段在台灣進行的國中英文補習班課堂錄音。老師的教學內容圍繞著英文文法、單字、和克漏字測驗。"
    "以下是可能出現的專有名詞、術語和學生姓名，請在辨識時優先參考："
    
    "【核心文法概念】: "
    "分詞, 介系詞, 動詞用法, 過去式, 現在完成式, 附加問句, "
    "主詞, 受詞, 補語, 時態, 句型, 不定詞, 動名詞, 助動詞, 感官動詞, 連綴動詞, 授與動詞。"
    
    "【關鍵歷史名詞】: "
    "凡爾賽合約, 威瑪共和, 納粹, 希特勒, 第一次世界大戰, 第二次世界大戰, 民主, 共和, 獨裁, 憲法, 政黨。"
    
    "【學生姓名】: "
    "張玹寧, 許勻綺, 林沛潔, 張勛恆, 田騏民, 陳華銘, 王子誠, 邱立堯, 謝竣翔, "
    "林程善, 烏鴻佾, 黃麗安, 邱珈翔, 徐品漢, 許宥琳, 李璿, 楊昀臻, 胡承恩, "
    "莊雅筑, 羅妍溱, 吳文宣, 傅煜昊, 蕭凱億, 林弈萱, 徐芙暄, 陳昱蓁, 蘇亦淮。"
    
    "【課堂管理與口語】: "
    "好來, OK, 可以嗎, 對不對, 懂我意思嗎, 專心看, 圈起來, 畫起來。"
)
# ==============================================================================

# --- 全域工具 ---
cc = OpenCC('s2twp')

def format_time(seconds: float) -> str:
    """將秒數轉換為 HH:MM:SS 格式"""
    m, s = divmod(seconds, 60)
    h, m = divmod(m, 60)
    return f"{int(h):02d}:{int(m):02d}:{int(round(s)):02d}"

def filter_allowed_chars(text: str) -> str:
    """只保留繁體中文、英文、數字和基礎標點符號"""
    pattern = re.compile(r'[^\u4e00-\u9fffA-Za-z0-9\s.,?!:\[\]\'"()]')
    return pattern.sub('', text)

def correct_names_in_text(text: str, name_list: list, cutoff: int) -> str:
    """在文本中尋找並校正與名單相似的姓名"""
    corrected_text = text
    for length in [2, 3]:
        for i in range(len(text) - length + 1):
            substring = text[i:i+length]
            if all('\u4e00' <= char <= '\u9fff' for char in substring):
                best_match, score = fuzz_process.extractOne(substring, name_list)
                if score >= cutoff:
                    corrected_text = corrected_text.replace(substring, best_match)
    return corrected_text

# ===【修改點 2】=== 在 API 請求中加入 prompt 參數
def transcribe_chunk(args):
    """轉錄單一音訊片段，並進行姓名校正與字元過濾"""
    endpoint_url, api_key, file_path, start_time_s = args
    
    headers = {'api-key': api_key}
    
    # 將 PROMPT_TEXT 加入到 API 請求的參數中
    params = {
        'language': 'zh-TW', 
        'response_format': 'verbose_json',
        'prompt': PROMPT_TEXT  # <-- 在這裡加入 prompt
    }
    
    try:
        with open(file_path, 'rb') as audio_file:
            files = {'file': (os.path.basename(file_path), audio_file, 'audio/mpeg')}
            response = requests.post(endpoint_url, headers=headers, params=params, files=files, timeout=300)
            response.raise_for_status()
            
            result_dict = response.json()
            if result_dict and 'text' in result_dict:
                raw_text = result_dict['text']
                traditional_text = cc.convert(raw_text).strip()
                corrected_text = correct_names_in_text(traditional_text, STUDENT_NAMES, NAME_CORRECTION_SCORE_CUTOFF)
                final_text = filter_allowed_chars(corrected_text)
                return start_time_s, final_text
    except Exception:
        return start_time_s, None
    return start_time_s, None

def main():
    """主執行函式：切割、並行轉錄、排序合併、清理"""
    print("--- 增強版長音訊並行轉錄程式 (已加入 Prompt 優化) ---")

    # (載入環境變數部分... 維持不變)
    load_dotenv()
    endpoint = os.getenv("AZURE_TRANSCRIBE_ENDPOINT")
    deployment_name = os.getenv("AZURE_TRANSCRIBE_DEPLOYMENT_NAME")
    api_key = os.getenv("AZURE_TRANSCRIBE_API_KEY")
    api_version = os.getenv("API_VERSION")
    if not all([endpoint, deployment_name, api_key, api_version]):
        print("❌ 錯誤：.env 設定不完整。")
        return
    deployment_name = deployment_name.strip()
    full_endpoint_url = f"{endpoint}/openai/deployments/{deployment_name}/audio/transcriptions?api-version={api_version}"

    # (準備與切割音訊部分... 維持不變)
    if not os.path.exists(LONG_AUDIO_FILE_PATH):
        print(f"❌ 錯誤：找不到音訊檔案 '{LONG_AUDIO_FILE_PATH}'")
        return
    chunk_files_with_time = []
    try:
        print(f"切割音訊中 (每段 {CHUNK_LENGTH_IN_SECONDS} 秒)...")
        os.makedirs(TEMP_CHUNK_FOLDER, exist_ok=True)
        audio = AudioSegment.from_mp3(LONG_AUDIO_FILE_PATH)
        chunk_length_ms = CHUNK_LENGTH_IN_SECONDS * 1000
        for i, start_ms in enumerate(range(0, len(audio), chunk_length_ms)):
            chunk = audio[start_ms : start_ms + chunk_length_ms]
            start_time_s = start_ms / 1000
            chunk_name = os.path.join(TEMP_CHUNK_FOLDER, f"chunk_{i+1:04d}.mp3")
            chunk.export(chunk_name, format="mp3")
            chunk_files_with_time.append((chunk_name, start_time_s))
        print(f"✅ 音訊切割完成，共 {len(chunk_files_with_time)} 個片段。")
    except Exception as e:
        print(f"❌ 錯誤：切割音訊時發生問題 - {e}")
        return

    # (建立並行轉錄任務部分... 維持不變)
    tasks = [(full_endpoint_url, api_key, file_path, start_time) for file_path, start_time in chunk_files_with_time]
    results = []
    print(f"\n🚀 開始並行轉錄 (同時處理 {MAX_CONCURRENT_WORKERS} 個檔案)...")
    with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_WORKERS) as executor:
        future_to_task = {executor.submit(transcribe_chunk, task): task for task in tasks}
        for future in tqdm(as_completed(future_to_task), total=len(tasks), desc="轉錄進度"):
            result = future.result()
            if result:
                results.append(result)
    print("✅ 所有片段處理完畢。")

    # (排序、合併並寫入最終檔案部分... 維持不變)
    print("\n整理並寫入最終檔案...")
    results.sort(key=lambda x: x[0])
    final_transcript_lines = []
    failed_chunks = 0
    for start_time_s, text in results:
        if text:
            timestamp = format_time(start_time_s)
            final_transcript_lines.append(f"{timestamp}: {text}")
        else:
            failed_chunks += 1
    try:
        with open(TXT_OUTPUT_FILE_PATH, 'w', encoding='utf-8') as f:
            f.write('\n'.join(final_transcript_lines))
        print(f"\n✅✅✅ 全部轉錄完成！")
        print(f"最終結果已儲存至: {TXT_OUTPUT_FILE_PATH}")
        if failed_chunks > 0:
            print(f"⚠️ 警告：有 {failed_chunks} 個片段轉錄失敗。")
    except IOError as e:
        print(f"❌ 錯誤：寫入最終檔案時失敗 - {e}")

    # (清理臨時檔案部分... 維持不變)
    print("\n清理臨時檔案...")
    try:
        for file_path, _ in chunk_files_with_time:
            os.remove(file_path)
        os.rmdir(TEMP_CHUNK_FOLDER)
        print("✅ 清理完成。")
    except Exception as e:
        print(f"⚠️ 警告：清理臨時檔案時發生錯誤 - {e}")

if __name__ == "__main__":
    main()

--- 增強版長音訊並行轉錄程式 (已加入 Prompt 優化) ---
切割音訊中 (每段 30 秒)...
✅ 音訊切割完成，共 400 個片段。

🚀 開始並行轉錄 (同時處理 10 個檔案)...


轉錄進度: 100%|██████████| 400/400 [02:05<00:00,  3.19it/s]


✅ 所有片段處理完畢。

整理並寫入最終檔案...

✅✅✅ 全部轉錄完成！
最終結果已儲存至: C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0928\老師\0928_3.txt
⚠️ 警告：有 1 個片段轉錄失敗。

清理臨時檔案...
✅ 清理完成。


In [11]:
import os
import requests
import json
import time
import re
from dotenv import load_dotenv
from pydub import AudioSegment
from opencc import OpenCC
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from thefuzz import process as fuzz_process

# ==============================================================================
# --- 設定區 ---
# ==============================================================================
# 1. 您的長音訊檔案路徑
LONG_AUDIO_FILE_PATH = r"C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0928\老師\0928.mp3"

# 2. 最終輸出的文字稿檔案路徑
TXT_OUTPUT_FILE_PATH = r"C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0928\老師\0928_3.txt"

# 3. 切割音訊的臨時儲存資料夾
TEMP_CHUNK_FOLDER = r"C:\Users\User\Desktop\test\temp_audio_chunks"

# 4. 每個音訊片段的長度（單位：秒）
CHUNK_LENGTH_IN_SECONDS = 30

# 5. 同時發送給 API 的請求數量
MAX_CONCURRENT_WORKERS = 10

# 6. 學生姓名名單
STUDENT_NAMES = [
    "張玹寧", "許勻綺", "林沛潔", "張勛恆", "田騏民", "陳華銘", "王子誠",
    "邱立堯", "謝竣翔", "林程善", "烏鴻佾", "黃麗安", "邱珈翔", "徐品漢",
    "許宥琳", "李璿", "楊昀臻", "胡承恩", "莊雅筑", "羅妍溱", "吳文宣",
    "傅煜昊", "蕭凱億", "林弈萱", "徐芙暄", "陳昱蓁", "蘇亦淮"
]

# 7. 姓名校正的相似度分數門檻 (0-100)
NAME_CORRECTION_SCORE_CUTOFF = 85

# 8. 無語音機率門檻值 (0.0 - 1.0)
#    AI 模型認為這段音訊「沒有語音」的機率。如果高於此值，就代表可能是噪音或靜音，應被過濾。
NO_SPEECH_PROB_THRESHOLD = 0.75

# 9. 平均信賴度門檻值 (通常為負數, 越接近 0 越好)
#    AI 模型對其辨識結果的信心程度。如果低於此值，代表辨識結果可能不準確，應被過濾。
AVG_LOGPROB_THRESHOLD = -0.8

# ===【修改點 1】=== 新增 PROMPT 參數，為模型提供上下文線索
PROMPT_TEXT = (
    "這是一段在台灣進行的國中英文補習班課堂錄音。老師的教學內容圍繞著英文文法、單字、和克漏字測驗。"
    "以下是可能出現的專有名詞、術語和學生姓名，請在辨識時優先參考："
    
    "【核心文法概念】: "
    "分詞, 介系詞, 動詞用法, 過去式, 現在完成式, 附加問句, "
    "主詞, 受詞, 補語, 時態, 句型, 不定詞, 動名詞, 助動詞, 感官動詞, 連綴動詞, 授與動詞。"
    
    "【關鍵歷史名詞】: "
    "凡爾賽合約, 威瑪共和, 納粹, 希特勒, 第一次世界大戰, 第二次世界大戰, 民主, 共和, 獨裁, 憲法, 政黨。"
    
    "【學生姓名】: "
    "張玹寧, 許勻綺, 林沛潔, 張勛恆, 田騏民, 陳華銘, 王子誠, 邱立堯, 謝竣翔, "
    "林程善, 烏鴻佾, 黃麗安, 邱珈翔, 徐品漢, 許宥琳, 李璿, 楊昀臻, 胡承恩, "
    "莊雅筑, 羅妍溱, 吳文宣, 傅煜昊, 蕭凱億, 林弈萱, 徐芙暄, 陳昱蓁, 蘇亦淮。"
    
    "【課堂管理與口語】: "
    "好來, OK, 可以嗎, 對不對, 懂我意思嗎, 專心看, 圈起來, 畫起來。"
)
# ==============================================================================

# --- 全域工具 ---
cc = OpenCC('s2twp')

def format_time(seconds: float) -> str:
    """將秒數轉換為 HH:MM:SS 格式"""
    m, s = divmod(seconds, 60)
    h, m = divmod(m, 60)
    return f"{int(h):02d}:{int(m):02d}:{int(round(s)):02d}"

def filter_allowed_chars(text: str) -> str:
    """只保留繁體中文、英文、數字和基礎標點符號"""
    pattern = re.compile(r'[^\u4e00-\u9fffA-Za-z0-9\s.,?!:\[\]\'"()]')
    return pattern.sub('', text)

def correct_names_in_text(text: str, name_list: list, cutoff: int) -> str:
    """在文本中尋找並校正與名單相似的姓名"""
    corrected_text = text
    for length in [2, 3]:
        for i in range(len(text) - length + 1):
            substring = text[i:i+length]
            if all('\u4e00' <= char <= '\u9fff' for char in substring):
                best_match, score = fuzz_process.extractOne(substring, name_list)
                if score >= cutoff:
                    corrected_text = corrected_text.replace(substring, best_match)
    return corrected_text

# ===【修改點 2】=== 在 API 請求中加入 prompt 參數
def transcribe_chunk(args):
    """轉錄單一音訊片段，並進行姓名校正與字元過濾"""
    endpoint_url, api_key, file_path, start_time_s = args
    
    headers = {'api-key': api_key}
    
    # 將 PROMPT_TEXT 加入到 API 請求的參數中
    params = {
        'language': 'zh-TW', 
        'response_format': 'verbose_json',
        'prompt': PROMPT_TEXT  # <-- 在這裡加入 prompt
    }
    
    try:
        with open(file_path, 'rb') as audio_file:
            files = {'file': (os.path.basename(file_path), audio_file, 'audio/mpeg')}
            response = requests.post(endpoint_url, headers=headers, params=params, files=files, timeout=300)
            response.raise_for_status()
            
            result_dict = response.json()
            # 直接回傳 API 的原始結果字典
            return start_time_s, result_dict 
    except Exception as e:
        # 在捕獲錯誤時，將錯誤 "e" 的內容印出來！
        # print(f"❌ 檔案 {os.path.basename(file_path)} 轉錄失敗，詳細錯誤: {e}")
        return start_time_s, e

def main():
    """主執行函式：切割、並行轉錄、過濾、排序合併、清理"""
    print("--- 增強版長音訊並行轉錄程式 (已加入雜訊過濾) ---")

    # --- 1. 載入環境變數與設定 API 端點 (維持不變) ---
    load_dotenv()
    endpoint = os.getenv("AZURE_TRANSCRIBE_ENDPOINT")
    deployment_name = os.getenv("AZURE_TRANSCRIBE_DEPLOYMENT_NAME")
    api_key = os.getenv("AZURE_TRANSCRIBE_API_KEY")
    api_version = os.getenv("API_VERSION")
    if not all([endpoint, deployment_name, api_key, api_version]):
        print("❌ 錯誤：.env 設定不完整。")
        return
    deployment_name = deployment_name.strip()
    full_endpoint_url = f"{endpoint}/openai/deployments/{deployment_name}/audio/transcriptions?api-version={api_version}"

    # --- 2. 準備與切割音訊 (維持不變) ---
    if not os.path.exists(LONG_AUDIO_FILE_PATH):
        print(f"❌ 錯誤：找不到音訊檔案 '{LONG_AUDIO_FILE_PATH}'")
        return
    chunk_files_with_time = []
    try:
        print(f"切割音訊中 (每段 {CHUNK_LENGTH_IN_SECONDS} 秒)...")
        os.makedirs(TEMP_CHUNK_FOLDER, exist_ok=True)
        audio = AudioSegment.from_mp3(LONG_AUDIO_FILE_PATH)
        chunk_length_ms = CHUNK_LENGTH_IN_SECONDS * 1000
        for i, start_ms in enumerate(range(0, len(audio), chunk_length_ms)):
            chunk = audio[start_ms : start_ms + chunk_length_ms]
            start_time_s = start_ms / 1000
            chunk_name = os.path.join(TEMP_CHUNK_FOLDER, f"chunk_{i+1:04d}.mp3")
            chunk.export(chunk_name, format="mp3")
            chunk_files_with_time.append((chunk_name, start_time_s))
        print(f"✅ 音訊切割完成，共 {len(chunk_files_with_time)} 個片段。")
    except Exception as e:
        print(f"❌ 錯誤：切割音訊時發生問題 - {e}")
        return

    # --- 3. 建立並行轉錄任務 (維持不變) ---
    tasks = [(full_endpoint_url, api_key, file_path, start_time) for file_path, start_time in chunk_files_with_time]
    success_results = []
    error_messages = []

    print(f"\n🚀 開始並行轉錄 (同時處理 {MAX_CONCURRENT_WORKERS} 個檔案)...")
    with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_WORKERS) as executor:
        future_to_task = {executor.submit(transcribe_chunk, task): task for task in tasks}
        for future in tqdm(as_completed(future_to_task), total=len(tasks), desc="轉錄進度"):
            start_time, result = future.result()
            
            # 判斷回傳的 result 是不是一個 Exception 物件
            if isinstance(result, Exception):
                error_messages.append(result)
            elif result:
                success_results.append((start_time, result))
            else: # 處理其他可能的失敗情況
                error_messages.append(Exception("未知的轉錄錯誤"))

    print("✅ 所有片段處理完畢。")

    # 【修改點】在處理結果前，先印出收集到的錯誤訊息
    if error_messages:
        print("\n" + "="*30)
        print("🚨 轉錄過程中發生錯誤：")
        # 為了避免洗版，只印出前 5 個不重複的錯誤訊息
        unique_errors = list(set(str(e) for e in error_messages))
        for i, error in enumerate(unique_errors[:5]):
            print(f"  錯誤 {i+1}: {error}")
        if len(unique_errors) > 5:
            print(f"  ... (還有 {len(unique_errors) - 5} 種類似的錯誤未顯示)")
        print("="*30 + "\n")

    # --- 4. 排序、過濾、處理並寫入最終檔案 (核心修改處) ---
    print("\n整理、過濾並寫入最終檔案...")
    final_transcript_lines = []
    filtered_segments_count = 0 # 為了避免命名衝突，改個名字
    
    success_results.sort(key=lambda x: x[0])

    for chunk_start_time, result_dict in success_results:
        
        # --- 【核心修改：判斷 'segments' 是否存在】 ---
        if 'segments' in result_dict and result_dict['segments']:
            #【情況 A：API 回傳了詳細的 segments 列表】
            for segment in result_dict['segments']:
                no_speech_prob = segment.get('no_speech_prob', 0)
                avg_logprob = segment.get('avg_logprob', 0)
                
                # 執行我們的過濾邏輯
                if no_speech_prob > NO_SPEECH_PROB_THRESHOLD or avg_logprob < AVG_LOGPROB_THRESHOLD:
                    filtered_segments_count += 1
                    continue

                raw_text = segment.get('text', '')
                if not raw_text.strip():
                    continue

                traditional_text = cc.convert(raw_text).strip()
                corrected_text = correct_names_in_text(traditional_text, STUDENT_NAMES, NAME_CORRECTION_SCORE_CUTOFF)
                final_text = filter_allowed_chars(corrected_text)
                
                segment_start_time = chunk_start_time + segment['start']
                timestamp = format_time(segment_start_time)
                
                final_transcript_lines.append(f"{timestamp}: {final_text}")
        
        elif 'text' in result_dict and result_dict['text'].strip():
            #【情況 B：API 只回傳了簡單的 'text'】
            # 在這種情況下，我們無法進行基於信賴度的過濾，只能退回到原始的處理方式
            raw_text = result_dict['text']
            
            traditional_text = cc.convert(raw_text).strip()
            corrected_text = correct_names_in_text(traditional_text, STUDENT_NAMES, NAME_CORRECTION_SCORE_CUTOFF)
            final_text = filter_allowed_chars(corrected_text)
            
            timestamp = format_time(chunk_start_time)
            final_transcript_lines.append(f"{timestamp}: {final_text}")

    # --- 【修改 try...except 區塊以適應新邏輯】 ---
    try:
        with open(TXT_OUTPUT_FILE_PATH, 'w', encoding='utf-8') as f:
            f.write('\n'.join(final_transcript_lines))
        print(f"\n✅✅✅ 全部轉錄完成！")
        print(f"最終結果已儲存至: {TXT_OUTPUT_FILE_PATH}")
        
        # 只有在 API 支援 segments 的情況下，這個訊息才有意義
        if filtered_segments_count > 0:
            print(f"ℹ️ 根據設定，共過濾掉 {filtered_segments_count} 個噪音或辨識不清的片段。")
        
        if len(error_messages) > 0:
            print(f"⚠️ 警告：有 {len(error_messages)} 個音訊區塊轉錄失敗。")

    except IOError as e:
        print(f"❌ 錯誤：寫入最終檔案時失敗 - {e}")

    # --- 5. 清理臨時檔案 (維持不變) ---
    print("\n清理臨時檔案...")
    try:
        for file_path, _ in chunk_files_with_time:
            os.remove(file_path)
        os.rmdir(TEMP_CHUNK_FOLDER)
        print("✅ 清理完成。")
    except Exception as e:
        print(f"⚠️ 警告：清理臨時檔案時發生錯誤 - {e}")

if __name__ == "__main__":
    main()

--- 增強版長音訊並行轉錄程式 (已加入雜訊過濾) ---
切割音訊中 (每段 30 秒)...
✅ 音訊切割完成，共 400 個片段。

🚀 開始並行轉錄 (同時處理 10 個檔案)...


轉錄進度: 100%|██████████| 400/400 [02:07<00:00,  3.13it/s]


✅ 所有片段處理完畢。

整理、過濾並寫入最終檔案...

✅✅✅ 全部轉錄完成！
最終結果已儲存至: C:\Users\User\Desktop\test\SynologyDrive\image\上課影片\0928\老師\0928_3.txt

清理臨時檔案...
✅ 清理完成。


# 老師位置(左、中、右)

### 測試不儲存

In [ ]:
# -*- coding: utf-8 -*-

import cv2
import os
import torch
from ultralytics import YOLO
from tkinter import Tk, filedialog

# --- 設定 ---

# 我們要使用的 YOLOv8 模型。'yolov8n.pt' 是最小最快的模型，對於這個任務已經足夠。
# 第一次執行時，程式會自動下載此模型。
MODEL_NAME = 'yolov12x.pt' 

# 在 COCO 數據集中，'person' (人) 的類別 ID 是 0。
PERSON_CLASS_ID = 0

# 位置標籤定義
POSITION_LABELS = {
    0: "Left",
    1: "Center",
    2: "Right"
}
# 顯示文字與方框的視覺設定
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 1.2
FONT_THICKNESS = 3
TEXT_COLOR_BG = (0, 0, 0) # 文字背景，黑色
TEXT_COLOR_FG = (255, 255, 255) # 文字顏色，白色
BOX_COLOR = (0, 255, 0) # 偵測方框顏色，綠色
BOX_THICKNESS = 2

def select_folder():
    """打開一個對話框讓使用者選擇資料夾"""
    root = Tk()
    root.withdraw()  # 隱藏主視窗
    folder_path = filedialog.askdirectory(title="請選擇存放老師照片的資料夾")
    return folder_path

def get_position_from_box(box_coords, image_width):
    """
    根據偵測框的中心點，判斷其在圖片中的位置 (左/中/右)。
    
    Args:
        box_coords (list): [x1, y1, x2, y2] 偵測框的左上角和右下角座標。
        image_width (int): 圖片的總寬度。
        
    Returns:
        int: 0 代表左側, 1 代表中間, 2 代表右側。
    """
    # 計算偵測框的中心 x 座標
    center_x = (box_coords[0] + box_coords[2]) / 2
    
    # 判斷中心點在哪個三分之一的區域
    if center_x < image_width / 3:
        return 0  # 左側
    elif center_x > image_width * 2 / 3:
        return 2  # 右側
    else:
        return 1  # 中間

def main():
    """主執行函數"""
    print("--- 老師位置偵測程式 (YOLOv8) ---")
    
    # 檢查是否有可用的 GPU，並告知使用者
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"正在使用的計算裝置: {device}")

    # 載入預訓練的 YOLOv8 模型
    try:
        print(f"正在載入模型 '{MODEL_NAME}'...")
        model = YOLO(MODEL_NAME)
        print("模型載入成功！")
    except Exception as e:
        print(f"錯誤：無法載入模型。請確認網路連線或套件是否安裝正確。 {e}")
        return

    # 讓使用者選擇資料夾
    image_folder = select_folder()
    if not image_folder:
        print("未選擇資料夾，程式結束。")
        return
        
    print(f"已選擇資料夾: {image_folder}")

    # 獲取資料夾內所有圖片檔案，並依照檔名排序 (您的檔名已含時間序)
    valid_extensions = ('.png', '.jpg', '.jpeg', '.webp')
    image_files = sorted([f for f in os.listdir(image_folder) if f.lower().endswith(valid_extensions)])
    
    if not image_files:
        print("錯誤：在選定資料夾中找不到任何圖片檔案。")
        return

    print(f"找到 {len(image_files)} 張圖片，開始進行分析...")

    # 逐一處理每張圖片
    for filename in image_files:
        image_path = os.path.join(image_folder, filename)
        
        # 讀取圖片
        image = cv2.imread(image_path)
        if image is None:
            print(f"警告：無法讀取圖片 {filename}，跳過。")
            continue
        
        # 獲取圖片尺寸
        h, w, _ = image.shape
        
        # 使用 YOLO 模型進行物件偵測
        # verbose=False 讓終端機保持乾淨，不顯示每次偵測的詳細日誌
        results = model(image, verbose=False)
        
        # 從偵測結果中找出「人」
        teacher_box = None
        largest_area = 0
        
        # results[0].boxes 包含了所有偵測到的物件
        for box in results[0].boxes:
            # 檢查類別是否為 'person'
            if int(box.cls[0]) == PERSON_CLASS_ID:
                # 如果偵測到多個人，我們假設老師是畫面中最大的人
                current_box_coords = box.xyxy[0].tolist()
                box_w = current_box_coords[2] - current_box_coords[0]
                box_h = current_box_coords[3] - current_box_coords[1]
                area = box_w * box_h
                
                if area > largest_area:
                    largest_area = area
                    teacher_box = current_box_coords
        
        position_text = ""
        # 如果有找到老師
        if teacher_box:
            # 判斷位置
            position_index = get_position_from_box(teacher_box, w)
            position_text = POSITION_LABELS[position_index]
            
            # 將偵測框畫在圖片上
            x1, y1, x2, y2 = [int(coord) for coord in teacher_box]
            cv2.rectangle(image, (x1, y1), (x2, y2), BOX_COLOR, BOX_THICKNESS)
        else:
            position_text = "Not detected"

        # 將位置文字寫在圖片左上角
        # 為了讓文字更清晰，我們先畫一個黑色背景，再寫上白色文字
        (text_w, text_h), _ = cv2.getTextSize(position_text, FONT, FONT_SCALE, FONT_THICKNESS)
        cv2.rectangle(image, (5, 5), (15 + text_w, 15 + text_h), TEXT_COLOR_BG, -1)
        cv2.putText(image, position_text, (10, 10 + text_h), FONT, FONT_SCALE, TEXT_COLOR_FG, FONT_THICKNESS, cv2.LINE_AA)
        
        # 顯示結果圖片
        # 為了避免圖片太大，可以縮放視窗
        display_image = cv2.resize(image, (w // 2, h // 2)) # 縮小為一半尺寸
        cv2.imshow('Teacher Position Detection - Press any key for next image', display_image)
        
        # 等待使用者按下任意鍵後，再處理下一張圖片
        key = cv2.waitKey(0) 
        
        # 如果使用者按下 'q' 或 ESC 鍵，則提早結束
        if key == ord('q') or key == 27:
            print("使用者手動中斷，程式結束。")
            break
            
    # 關閉所有 OpenCV 視窗
    cv2.destroyAllWindows()
    print("--- 所有圖片處理完畢 ---")

if __name__ == "__main__":
    main()

### 儲存成 json 檔案

In [ ]:
# -*- coding: utf-8 -*-

import cv2
import os
import torch
import json
import re
import datetime
from ultralytics import YOLO
from tqdm import tqdm  # 引入 tqdm 來顯示進度條

# --- 設定 ---
# 註：YOLOv12x.pt 不是一個標準的 YOLOv8 模型名稱。
# 常用的模型包括 yolov8n.pt (最小), yolov8s.pt, yolov8m.pt, yolov8l.pt, yolov8x.pt (最大)。
# 這裡我先使用 yolov8n.pt，您可以根據需要更換成更大的模型以獲取更高精度。
MODEL_NAME = 'yolov12x.pt' 
PERSON_CLASS_ID = 0

# 【修改點 1】將位置標籤擴展為五個，並更新索引
POSITION_LABELS = {
    0: "左側",
    1: "中間偏左",
    2: "中間",
    3: "中間偏右",
    4: "右側",
    -1: "未偵測到" 
}

def get_valid_input(prompt_message):
    """通用函數，用於獲取有效的非空使用者輸入"""
    while True:
        user_input = input(prompt_message).strip()
        if user_input:
            return user_input
        print("錯誤：輸入不能為空，請重新輸入。")

def get_valid_folder_path(prompt_message):
    """獲取有效資料夾路徑"""
    while True:
        folder_path = input(prompt_message).strip().strip('"') # 移除可能的引號
        if os.path.isdir(folder_path):
            return folder_path
        print(f"錯誤：路徑 '{folder_path}' 不是一個有效的資料夾，請重新輸入。")

def get_timestamp_from_filename(filename):
    """
    從 'blackboard_0_00h01m29s_f002691.png' 這種格式的檔名中解析出時間。
    返回一個 timedelta 物件以便排序。
    """
    match = re.search(r'(\d+)h(\d{2})m(\d{2})s', filename)
    if match:
        try:
            h, m, s = map(int, match.groups())
            return datetime.timedelta(hours=h, minutes=m, seconds=s)
        except (ValueError, IndexError):
            return None
    return None

def get_position_from_box_five_sections(box_coords, image_width):
    """
    【修改點 2】根據偵測框的中心點，判斷其在圖片中的五段式位置。
    """
    center_x = (box_coords[0] + box_coords[2]) / 2
    
    section_width = image_width / 5
    
    if center_x < section_width:
        return 0  # 左側
    elif center_x < section_width * 2:
        return 1  # 中間偏左
    elif center_x < section_width * 3:
        return 2  # 中間
    elif center_x < section_width * 4:
        return 3  # 中間偏右
    else:
        return 4  # 右側

def main():
    """主執行函數"""
    print("--- 老師位置自動化分析與 JSON 生成工具 (YOLOv8 - 五段式位置) ---")
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"正在使用的計算裝置: {device}")

    try:
        print(f"正在載入模型 '{MODEL_NAME}'... (若為首次執行，將會自動下載)")
        # 附註：您程式碼中的 'yolov12x.pt' 並非標準 YOLOv8 模型名稱。
        # 如果您有自訂模型，請確保檔案存在。否則請使用標準名稱如 'yolov8x.pt'。
        # 此處使用 'yolov8n.pt' 作為範例。
        model = YOLO(MODEL_NAME)
        print("模型載入成功！")
    except Exception as e:
        print(f"錯誤：無法載入模型 '{MODEL_NAME}'。請檢查模型名稱是否正確，或確認網路連線。 {e}")
        return

    # --- 使用者輸入部分 ---
    image_folder = get_valid_folder_path("請輸入老師照片資料夾的路徑: ")
    output_filename = get_valid_input("請輸入要導出的 JSON 檔案名稱 (例如: teacher_positions_5_sections.json): ")
    if not output_filename.lower().endswith('.json'):
        output_filename += '.json'
    
    print("-" * 40)
    print(f"開始處理資料夾: {image_folder}")
    print(f"結果將儲存至: {output_filename}")
    
    valid_extensions = ('.png', '.jpg', '.jpeg', '.webp')
    all_files = os.listdir(image_folder)
    
    images_with_timestamps = []
    for filename in all_files:
        if filename.lower().endswith(valid_extensions):
            timestamp = get_timestamp_from_filename(filename)
            if timestamp:
                images_with_timestamps.append({
                    "filename": filename,
                    "path": os.path.join(image_folder, filename),
                    "timestamp_obj": timestamp
                })
            else:
                print(f"警告：檔名 '{filename}' 格式不符，無法解析時間，將跳過此檔案。")

    images_with_timestamps.sort(key=lambda x: x["timestamp_obj"])
    
    if not images_with_timestamps:
        print("錯誤：在選定資料夾中找不到任何符合時間格式的圖片檔案。")
        return

    print(f"共找到 {len(images_with_timestamps)} 張有效圖片，開始進行分析...")
    
    results_data = []

    for image_info in tqdm(images_with_timestamps, desc="分析進度"):
        image_path = image_info["path"]
        
        image = cv2.imread(image_path)
        if image is None:
            continue
        
        h, w, _ = image.shape
        
        results = model(image, device=device, verbose=False)
        
        teacher_box = None
        largest_area = 0
        
        for box in results[0].boxes:
            if int(box.cls[0]) == PERSON_CLASS_ID:
                current_box_coords = box.xyxy[0].tolist()
                area = (current_box_coords[2] - current_box_coords[0]) * (current_box_coords[3] - current_box_coords[1])
                if area > largest_area:
                    largest_area = area
                    teacher_box = current_box_coords
        
        position_index = -1
        if teacher_box:
            # 【修改點 3】呼叫新的五段式判斷函數
            position_index = get_position_from_box_five_sections(teacher_box, w)
        
        total_seconds = int(image_info["timestamp_obj"].total_seconds())
        hours, remainder = divmod(total_seconds, 3600)
        minutes, seconds = divmod(remainder, 60)
        timestamp_str = f"{hours:02}:{minutes:02}:{seconds:02}"

        results_data.append({
            "timestamp": timestamp_str,
            "position": POSITION_LABELS[position_index],
            "source_filename": image_info["filename"]
        })
            
    try:
        with open(output_filename, 'w', encoding='utf-8') as f:
            json.dump(results_data, f, ensure_ascii=False, indent=4)
        print("\n✅ 分析完成！")
        print(f"結果已成功儲存至檔案: {os.path.abspath(output_filename)}")
    except Exception as e:
        print(f"\n❌ 錯誤：儲存 JSON 檔案時發生問題：{e}")

if __name__ == "__main__":
    main()

# 提取json 數據至xlsx

### 行為次數

In [ ]:
import os
import json
import pandas as pd
from collections import defaultdict

# --- 全域設定：更新後的行為類別中英對照表 & 輸出欄位順序 ---

# 更新此對照表，使其與您 JSON 檔案中的 "behavior_category" 完全一致
BEHAVIOR_MAP = {
    # --- 更新的標籤 ---
    '做筆記': 'Note_taking',
    '目視書本/筆記': 'Looking_at_book',
    '低頭(非學習)': 'Looking_down',
    # --- 新增的標籤 ---
    '坐姿直立': 'Sitting_upright',
    # --- 維持不變的標籤 ---
    '目視教師': 'Looking_at_teacher',
    '翻閱書本': 'Flipping_through_books',
    '玩弄物品': 'Fidgeting_with_objects',
    '目視同學': 'Looking_at_classmates',
    '無明顯特定行為': 'No_specific_behavior',
    '目視黑板': 'Looking_at_blackboard',
    '整理個人物品': 'Organizing_personal_items',
    '目視他處': 'Looking_elsewhere',
    '喝水/飲食': 'Drinking_Eating',
    '被遮擋/無法判斷': 'Obstructed_Undetermined',
    '舉手': 'Raising_hand',
    '趴睡': 'Lying_prone',
    '身體後靠': 'Leaning_back',
    '目視前方': 'Looking_forward', # 保留以相容舊檔案
    '身體前傾': 'Leaning_forward'
}

# 根據您提供的英文欄位名稱，定義輸出的欄位順序
COLUMN_ORDER = [
    '姓名',
    'behavior_positive',
    'behavior_native', # 根據您的範例圖，使用 native
    'behavior_natural',
    'Note_taking',
    'Looking_at_teacher',
    'Flipping_through_books',
    'Fidgeting_with_objects',
    'Looking_at_book',
    'Looking_down',
    'Looking_at_classmates',
    'No_specific_behavior',
    'Looking_at_blackboard',
    'Organizing_personal_items',
    'Looking_elsewhere',
    'Drinking_Eating',
    'Obstructed_Undetermined',
    'Raising_hand',
    'Lying_prone',
    'Leaning_back',
    'Looking_forward',
    'Sitting_upright',
    'Leaning_forward'
]

# --- 新增的部分：定義學生的固定排序順序 ---
STUDENT_ORDER_LIST = [
    '張玹寧', '許勻綺', '林沛潔', '張劼恆', '田軉民', '陳華銘', 
    '王予誠', '邱立堯', '謝竣翔', '林程善', '烏鴻佾', '黃麗安', 
    '邱珈翔', '徐品漢', '許宥琳', '李璿', '楊昀臻', '胡承恩', 
    '莊雅筑', '羅妍溱', '吳文宣', '傅煜旻', '蕭凱億', '林羿萱', 
    '徐芙暄', '陳昱蓁'
]


def process_behavior_reports_wide(root_path, output_filename):
    """
    遍歷指定路徑下的學生資料夾，提取 JSON 數據，並將每個學生彙總成一列（寬資料），
    最後匯出到一個 Excel 檔案中。每個工作表對應一個特定的日期和課程。
    """
    data_by_sheet = defaultdict(list)

    print(f"開始遍歷資料夾：{root_path}")
    if not os.path.isdir(root_path):
        print(f"錯誤：找不到指定的資料夾路徑 '{root_path}'。請檢查路徑是否正確。")
        return

    for dirpath, _, filenames in os.walk(root_path):
        for filename in filenames:
            if filename.endswith('.json'):
                json_file_path = os.path.join(dirpath, filename)
                print(f"  正在處理檔案：{os.path.relpath(json_file_path)}")

                try:
                    with open(json_file_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)

                    metadata = data.get('report_metadata', {})
                    summary = data.get('overall_summary', {})
                    
                    student_name = metadata.get('student_id')
                    if not student_name:
                        print(f"    -> 警告：檔案 {filename} 缺少 'student_id'，已跳過。")
                        continue

                    # 初始化一個學生的資料列
                    row_data = {'姓名': student_name}
                    
                    # 1. 提取正、負、中性行為的次數
                    valence_summary = summary.get('valence_summary', {})
                    row_data['behavior_positive'] = valence_summary.get('正向', {}).get('count')
                    row_data['behavior_native'] = valence_summary.get('負向', {}).get('count')
                    row_data['behavior_natural'] = valence_summary.get('中性', {}).get('count')

                    # 2. 提取各種行為的次數
                    behavior_stats = summary.get('behavior_statistics', [])
                    for behavior in behavior_stats:
                        chinese_name = behavior.get('behavior_category')
                        english_name = BEHAVIOR_MAP.get(chinese_name)
                        
                        if english_name:
                            row_data[english_name] = behavior.get('count')
                        else:
                            print(f"    -> 注意：在檔案 {filename} 中發現未知的行為類別 '{chinese_name}'，此欄位將被忽略。")
                    
                    # 根據日期和課程分類
                    gen_time = metadata.get('report_generation_time', '未知日期')
                    source_folder = metadata.get('student_image_source_folder', '未知課程')
                    sheet_name = f"{gen_time.replace('/', '')}_{source_folder}"
                    
                    data_by_sheet[sheet_name].append(row_data)

                except Exception as e:
                    print(f"    -> 處理檔案 {filename} 時發生錯誤: {e}，已跳過。")

    if not data_by_sheet:
        print("沒有找到任何可處理的數據，程式結束。")
        return

    # --- 將整理好的數據寫入 Excel ---
    print(f"\n數據處理完成，正在寫入 Excel 檔案：{output_filename}")
    with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        for sheet_name in sorted(data_by_sheet.keys()):
            df = pd.DataFrame(data_by_sheet[sheet_name])
            
            # --- 修改的部分：依照指定的學生名單排序 ---
            # 1. 將 '姓名' 欄位轉換為有特定順序的類別型別
            df['姓名'] = pd.Categorical(df['姓名'], categories=STUDENT_ORDER_LIST, ordered=True)
            # 2. 根據 '姓名' 的自訂順序進行排序，並重設索引
            df = df.sort_values('姓名').reset_index(drop=True)

            # 使用 reindex 確保欄位順序與 COLUMN_ORDER 一致，這比原先的迴圈更簡潔高效
            final_df = df.reindex(columns=COLUMN_ORDER)
            
            # 將整理好的 DataFrame 寫入工作表
            final_df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"  已建立工作表：'{sheet_name}'，包含 {len(final_df)} 位學生的數據。")

    print(f"\n成功建立 Excel 檔案 '{output_filename}'！")


# --- 主程式執行區塊 ---
if __name__ == '__main__':
    # 將 .py 檔案放置在與 'SynologyDrive' 同一層目錄下
    target_path = os.path.join('SynologyDrive', 'json_behavior')
    
    # 輸出的 Excel 檔案名稱
    excel_output_file = 'behavior_summary_wide_format.xlsx' # 更新了檔案名以避免覆蓋舊檔
    
    # 執行主程式
    process_behavior_reports_wide(target_path, excel_output_file)

### 行為次數(New)

In [12]:
import os
import json
import pandas as pd
from collections import defaultdict

# --- 全域設定：與您的標準完全同步的行為類別中英對照表 & 輸出欄位順序 ---

# 此對照表已根據您的 STANDARD_BEHAVIOR_CATEGORIES 進行最終更新
BEHAVIOR_MAP = {
    # --- 視線 ---
    '目視教師': 'Looking_at_teacher',
    '目視黑板': 'Looking_at_blackboard',
    '目視書本/筆記': 'Looking_at_book',
    '目視同學': 'Looking_at_classmates',
    '目視他處': 'Looking_elsewhere',
    
    # --- 肢體(手部) ---
    '做筆記': 'Note_taking',
    '翻閱書本': 'Flipping_through_books',
    '玩弄手部/文具': 'Fidgeting',
    '觸摸臉部': 'Touching_face',
    '觸摸頭髮': 'Touching_hair',
    
    # --- 身體姿態 ---
    '坐姿直立': 'Sitting_upright',
    '身體前傾': 'Leaning_forward',
    '身體後靠': 'Leaning_back',
    '低頭(非學習)': 'Looking_down',
    '趴睡': 'Lying_prone',
    
    # --- 互動 ---
    '舉手': 'Raising_hand',
    
    # --- 其他狀態 ---
    '喝水': 'Drinking',  # [修改] 拆分為獨立項
    '飲食': 'Eating',    # [修改] 拆分為獨立項
    '整理個人物品': 'Organizing_personal_items',
    '被遮擋/無法判斷': 'Obstructed_Undetermined',
    
    # --- 為相容舊檔案或潛在錯字而保留的對應 ---
    '翻閱書書本': 'Flipping_through_books',
    '玩弄物品': 'Fidgeting',
    '喝水/飲食': 'Drinking', # 將舊的合併項指向 'Drinking'，避免數據遺失
    '無明顯特定行為': 'No_specific_behavior',
    '目視前方': 'Looking_forward',
}


# 根據最終的行為類別，更新輸出的欄位順序
COLUMN_ORDER = [
    '姓名',
    # 整體統計
    'behavior_positive',
    'behavior_native',
    'behavior_natural',
    # 核心學習行為
    'Note_taking',
    'Looking_at_teacher',
    'Looking_at_book',
    'Looking_at_blackboard',
    'Raising_hand',
    'Flipping_through_books',
    # 非學習/分心行為
    'Looking_down',
    'Looking_at_classmates',
    'Looking_elsewhere',
    'Fidgeting',
    'Touching_face',
    'Touching_hair',
    'Lying_prone',
    # 姿勢/中性行為
    'Sitting_upright',
    'Leaning_forward',
    'Leaning_back',
    # 其他狀態
    'Drinking', # [修改]
    'Eating',   # [修改]
    'Organizing_personal_items',
    'Obstructed_Undetermined',
    # 為了相容性保留
    'No_specific_behavior',
    'Looking_forward'
]

# --- 學生固定排序順序 (維持不變) ---
STUDENT_ORDER_LIST = [
    '張玹寧', '許勻綺', '林沛潔', '張劼恆', '田軉民', '陳華銘', 
    '王予誠', '邱立堯', '謝竣翔', '林程善', '烏鴻佾', '黃麗安', 
    '邱珈翔', '徐品漢', '許宥琳', '李璿', '楊昀臻', '胡承恩', 
    '莊雅筑', '羅妍溱', '吳文宣', '傅煜旻', '蕭凱億', '林羿萱', 
    '徐芙暄', '陳昱蓁'
]


def process_behavior_reports_wide(root_path, output_filename):
    """
    遍歷指定路徑下的學生資料夾，提取 JSON 數據，並將每個學生彙總成一列（寬資料），
    最後匯出到一個 Excel 檔案中。每個工作表對應一個特定的日期和課程。
    """
    data_by_sheet = defaultdict(list)

    print(f"開始遍歷資料夾：{root_path}")
    if not os.path.isdir(root_path):
        print(f"錯誤：找不到指定的資料夾路徑 '{root_path}'。請檢查路徑是否正確。")
        return

    for dirpath, _, filenames in os.walk(root_path):
        for filename in filenames:
            if filename.endswith('.json'):
                json_file_path = os.path.join(dirpath, filename)
                print(f"  正在處理檔案：{os.path.relpath(json_file_path, root_path)}")

                try:
                    with open(json_file_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)

                    metadata = data.get('report_metadata', {})
                    summary = data.get('overall_summary', {})
                    
                    student_name = metadata.get('student_id')
                    if not student_name:
                        print(f"    -> 警告：檔案 {filename} 缺少 'student_id'，已跳過。")
                        continue

                    # 初始化一個學生的資料列，並將所有可能的行為欄位預設為 0
                    row_data = {key: 0 for key in COLUMN_ORDER}
                    row_data['姓名'] = student_name
                    
                    # 1. 提取正、負、中性行為的次數
                    valence_summary = summary.get('valence_summary', {})
                    row_data['behavior_positive'] = valence_summary.get('正向', {}).get('count', 0)
                    row_data['behavior_native'] = valence_summary.get('負向', {}).get('count', 0)
                    row_data['behavior_natural'] = valence_summary.get('中性', {}).get('count', 0)

                    # 2. 提取各種行為的次數
                    behavior_stats = summary.get('behavior_statistics', [])
                    for behavior in behavior_stats:
                        chinese_name = behavior.get('behavior_category')
                        english_name = BEHAVIOR_MAP.get(chinese_name)
                        
                        if english_name:
                            row_data[english_name] = behavior.get('count', 0)
                        else:
                            print(f"    -> 注意：在檔案 {filename} 中發現未知的行為類別 '{chinese_name}'，此欄位將被忽略。")
                    
                    # 根據日期和課程分類
                    gen_time = metadata.get('report_generation_time', '未知日期')
                    source_folder = metadata.get('student_image_source_folder', '未知課程')
                    sheet_name = f"{gen_time.replace('/', '')}_{source_folder}"
                    
                    data_by_sheet[sheet_name].append(row_data)

                except Exception as e:
                    print(f"    -> 處理檔案 {filename} 時發生錯誤: {e}，已跳過。")

    if not data_by_sheet:
        print("沒有找到任何可處理的數據，程式結束。")
        return

    # --- 將整理好的數據寫入 Excel ---
    print(f"\n數據處理完成，正在寫入 Excel 檔案：{output_filename}")
    with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        for sheet_name in sorted(data_by_sheet.keys()):
            df = pd.DataFrame(data_by_sheet[sheet_name])
            
            if df.empty:
                print(f"  工作表 '{sheet_name}' 沒有數據，已跳過。")
                continue
                
            # --- 依照指定的學生名單排序 ---
            df['姓名'] = pd.Categorical(df['姓名'], categories=STUDENT_ORDER_LIST, ordered=True)
            df = df.sort_values('姓名').reset_index(drop=True)

            # 使用 reindex 確保欄位順序與 COLUMN_ORDER 一致，並處理不存在的欄位（填充為0）
            final_df = df.reindex(columns=COLUMN_ORDER, fill_value=0)
            
            # 將整理好的 DataFrame 寫入工作表
            final_df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"  已建立工作表：'{sheet_name}'，包含 {len(final_df)} 位學生的數據。")

    print(f"\n成功建立 Excel 檔案 '{output_filename}'！")


# --- 主程式執行區塊 ---
if __name__ == '__main__':
    target_path = os.path.join('SynologyDrive', 'json_behavior')
    excel_output_file = 'behavior_summary_final.xlsx'
    process_behavior_reports_wide(target_path, excel_output_file)

開始遍歷資料夾：SynologyDrive\json_behavior
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250727_100633.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250802_210935.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250804_110148.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250804_140123.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250810_134555.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250819_121508.json
    -> 注意：在檔案 student_傅煜旻_behavior_report_20250819_121508.json 中發現未知的行為類別 '托腮'，此欄位將被忽略。
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250823_190436.json
    -> 注意：在檔案 student_傅煜旻_behavior_report_20250823_190436.json 中發現未知的行為類別 '托腮'，此欄位將被忽略。
    -> 注意：在檔案 student_傅煜旻_behavior_report_20250823_190436.json 中發現未知的行為類別 '主動舉手'，此欄位將被忽略。
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250902_100850.json
    -> 注意：在檔案 student_傅煜旻_behavior_report_20250902_100850.json 中發現未知的行為類別 '托腮'，此欄位將被忽略。
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250908_115153.json
    -> 注意：在檔案 student_傅煜旻_behavior_report_20250908_1

### 行為提取(復合)

In [33]:
# -*- coding: utf-8 -*-
import os
import json
import pandas as pd
from collections import defaultdict

# --- 全域設定：行為類別中英對照表 & 輸出欄位順序 ---

# 與您的主分析腳本完全同步的 BEHAVIOR_MAP
BEHAVIOR_MAP = {
    # --- 視線 ---
    '目視教師': 'Looking_at_teacher',
    '目視黑板': 'Looking_at_blackboard',
    '目視書本/筆記': 'Looking_at_book',
    '目視同學': 'Looking_at_classmates',
    '目視他處': 'Looking_elsewhere',
    # --- 肢體(手部) ---
    '做筆記': 'Note_taking',
    '玩弄手部/文具': 'Fidgeting',
    '觸摸臉部': 'Touching_face',
    '觸摸頭髮': 'Touching_hair',
    # --- 身體姿態 ---
    '坐姿直立': 'Sitting_upright',
    '身體前傾': 'Leaning_forward',
    '身體後靠': 'Leaning_back',
    '低頭(非學習)': 'Looking_down',
    '趴睡': 'Lying_prone',
    '托腮': 'Chin_rest',
    # --- 互動 ---
    '主動舉手': 'Raising_hand_Active',
    '被動舉手': 'Raising_hand_Passive',
    # --- 其他狀態 ---
    '喝水': 'Drinking',
    '飲食': 'Eating',
    '被遮擋/無法判斷': 'Obstructed_Undetermined',
}

# 定義您最關心的複合行為組合
COMPOSITE_BEHAVIORS_TO_TRACK = {
    ('Leaning_forward', 'Note_taking'): 'Composite_Forward_NoteTaking',
    ('Leaning_forward', 'Looking_at_book'): 'Composite_Forward_LookBook',
    ('Sitting_upright', 'Note_taking'): 'Composite_Upright_NoteTaking',
    ('Sitting_upright', 'Looking_at_teacher'): 'Composite_Upright_LookTeacher',
    ('Touching_face', 'Looking_at_book'): 'Composite_TouchFace_LookBook',
    ('Touching_hair', 'Looking_at_book'): 'Composite_TouchHair_LookBook',
}

# 智能合併欄位順序 (此部分邏輯不變，確保欄位完整)
USER_SPECIFIED_ORDER = [
    '姓名',
    'behavior_positive', 'behavior_negative', 'behavior_neutral',
    'Note_taking', 'Looking_at_teacher', 'Looking_at_book', 'Looking_at_blackboard',
    'Looking_down', 'Looking_at_classmates',
    'Looking_elsewhere', 'Fidgeting', 'Touching_face', 'Touching_hair', 'Lying_prone',
    'Sitting_upright', 'Leaning_forward', 'Leaning_back', 'Drinking', 'Eating',
    'Obstructed_Undetermined'
]
all_current_behaviors = set(BEHAVIOR_MAP.values())
base_columns = [col for col in USER_SPECIFIED_ORDER if col in all_current_behaviors or col in ['姓名', 'behavior_positive', 'behavior_negative', 'behavior_neutral']]
new_behaviors = sorted([b for b in all_current_behaviors if b not in base_columns])
BASE_COLUMN_ORDER = base_columns + new_behaviors
COMPOSITE_COLUMN_NAMES = sorted(list(COMPOSITE_BEHAVIORS_TO_TRACK.values()))
COLUMN_ORDER = BASE_COLUMN_ORDER + COMPOSITE_COLUMN_NAMES

# --- ★★★【核心修改點 1：將學生名單變為全域常量】★★★ ---
# 這是我們班級的「完整花名冊」
FULL_STUDENT_ROSTER = [
    '張玹寧', '許勻綺', '林沛潔', '張劼恆', '田軉民', '陳華銘', 
    '王予誠', '邱立堯', '謝竣翔', '林程善', '烏鴻佾', '黃麗安', 
    '邱珈翔', '徐品漢', '許宥琳', '李璿', '楊昀臻', '胡承恩', 
    '莊雅筑', '羅妍溱', '吳文宣', '傅煜旻', '蕭凱億', '林羿萱', 
    '徐芙暄', '陳昱蓁'
]


def process_behavior_reports_wide(root_path, output_filename):
    """
    [v5.0 完整名單版] 遍歷 JSON 檔案，確保 Excel 報告中包含所有學生，即使某學生當天沒有報告。
    """
    data_by_sheet = defaultdict(list)

    print(f"開始遍歷資料夾：{root_path}")
    if not os.path.isdir(root_path):
        print(f"錯誤：找不到指定的資料夾路徑 '{root_path}'。")
        return

    # --- 數據收集部分 (邏輯不變) ---
    for dirpath, _, filenames in os.walk(root_path):
        for filename in filenames:
            if filename.endswith('.json'):
                json_file_path = os.path.join(dirpath, filename)
                print(f"  正在處理檔案：{os.path.relpath(json_file_path, root_path)}")

                try:
                    with open(json_file_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)

                    metadata = data.get('report_metadata', {})
                    summary = data.get('overall_summary', {})
                    
                    student_name = metadata.get('student_id')
                    if not student_name:
                        print(f"    -> 警告：檔案 {filename} 缺少 'student_id'，已跳過。")
                        continue

                    row_data = {key: 0 for key in COLUMN_ORDER}
                    row_data['姓名'] = student_name
                    
                    valence_summary = summary.get('valence_summary', {})
                    row_data['behavior_positive'] = valence_summary.get('正向', {}).get('count', 0)
                    row_data['behavior_negative'] = valence_summary.get('負向', {}).get('count', 0)
                    row_data['behavior_neutral'] = valence_summary.get('中性', {}).get('count', 0)

                    behavior_stats = summary.get('behavior_statistics', [])
                    for behavior in behavior_stats:
                        chinese_name = behavior.get('behavior_category')
                        english_name = BEHAVIOR_MAP.get(chinese_name)
                        if english_name in row_data:
                            row_data[english_name] = behavior.get('count', 0)

                    detailed_analysis = data.get('detailed_sequence_analysis', [])
                    for batch in detailed_analysis:
                        analysis = batch.get('analysis', {})
                        highlights = analysis.get('per_image_highlights', [])
                        
                        for image_data in highlights:
                            behaviors = image_data.get('behavior_category', [])
                            if isinstance(behaviors, list) and len(behaviors) > 1:
                                english_behaviors_set = {BEHAVIOR_MAP.get(b) for b in behaviors if BEHAVIOR_MAP.get(b)}
                                for combo_to_track, new_col_name in COMPOSITE_BEHAVIORS_TO_TRACK.items():
                                    if all(item in english_behaviors_set for item in combo_to_track):
                                        row_data[new_col_name] += 1
                    
                    gen_time = metadata.get('report_generation_time', '未知日期')
                    source_folder = metadata.get('student_image_source_folder', '未知課程')
                    sheet_name = f"{gen_time.replace('/', '')}_{source_folder}"
                    
                    data_by_sheet[sheet_name].append(row_data)

                except Exception as e:
                    print(f"    -> 處理檔案 {filename} 時發生錯誤: {e}，已跳過。")

    if not data_by_sheet:
        print("沒有找到任何可處理的數據，程式結束。")
        return

    print(f"\n數據處理完成，正在寫入 Excel 檔案：{output_filename}")
    with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        for sheet_name in sorted(data_by_sheet.keys()):
            # 創建一個只包含當天有數據的學生的 DataFrame
            df_present_students = pd.DataFrame(data_by_sheet[sheet_name])
            
            if df_present_students.empty:
                print(f"  工作表 '{sheet_name}' 沒有數據，已跳過。")
                continue

            # --- ★★★【核心修改點 2：合併完整名單】★★★ ---
            # 1. 創建一個包含所有學生的「基礎」DataFrame
            df_all_students = pd.DataFrame({'姓名': FULL_STUDENT_ROSTER})

            # 2. 使用 left merge，將有數據的學生合併到完整名單上
            #    這會保留完整名單中的所有學生，沒有數據的學生其欄位會是 NaN
            df_merged = pd.merge(df_all_students, df_present_students, on='姓名', how='left')

            # 3. 將所有 NaN (缺席學生的數據) 填充為 0
            df_merged = df_merged.fillna(0)

            # 確保學生順序是按照我們的完整名單來的
            df_merged['姓名'] = pd.Categorical(df_merged['姓名'], categories=FULL_STUDENT_ROSTER, ordered=True)
            df_merged = df_merged.sort_values('姓名').reset_index(drop=True)

            # 將數值欄位轉換為整數，避免出現 .0
            for col in df_merged.columns:
                if col != '姓名':
                    df_merged[col] = df_merged[col].astype(int)

            # 使用最終的欄位順序來格式化 DataFrame
            final_df = df_merged.reindex(columns=COLUMN_ORDER, fill_value=0)
            
            final_df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"  已建立工作表：'{sheet_name}'，包含 {len(final_df)} 位學生的數據 (含缺席者)。")

    print(f"\n成功建立 Excel 檔案 '{output_filename}'！")


if __name__ == '__main__':
    target_path = os.path.join('SynologyDrive', 'json_behavior')
    excel_output_file = 'behavior_summary_full_roster.xlsx' # 建議使用新檔名
    process_behavior_reports_wide(target_path, excel_output_file)

開始遍歷資料夾：SynologyDrive\json_behavior
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250727_100633.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250802_210935.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250804_110148.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250804_140123.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250810_134555.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250819_121508.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250823_190436.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250902_100850.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250908_115153.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250918_173724.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250928_014129.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20251006_154650.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20251015_041632.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20251101_041801.json
  正在處理檔案：吳文宣\student_吳文宣_behavior_report_20250727_100634.json
  正在處理檔案：吳文宣\student_吳文宣_behavior_

In [13]:
# -*- coding: utf-8 -*-
import os
import json
import pandas as pd
from collections import defaultdict, Counter
from itertools import combinations

# --- 全域設定 ---

# 與您的主分析腳本完全同步的 BEHAVIOR_MAP
BEHAVIOR_MAP = {
    '目視教師': 'Looking_at_teacher', '目視黑板': 'Looking_at_blackboard',
    '目視書本/筆記': 'Looking_at_book', '目視同學': 'Looking_at_classmates',
    '目視他處': 'Looking_elsewhere', '做筆記': 'Note_taking',
    '玩弄手部/文具': 'Fidgeting', '觸摸臉部': 'Touching_face',
    '觸摸頭髮': 'Touching_hair', '坐姿直立': 'Sitting_upright',
    '身體前傾': 'Leaning_forward', '身體後靠': 'Leaning_back',
    '低頭(非學習)': 'Looking_down', '趴睡': 'Lying_prone',
    '托腮': 'Chin_rest', '主動舉手': 'Raising_hand_Active',
    '被動舉手': 'Raising_hand_Passive', '喝水': 'Drinking',
    '飲食': 'Eating', '被遮擋/無法判斷': 'Obstructed_Undetermined',
}

# 完整學生名單
FULL_STUDENT_ROSTER = [
    '張玹寧', '許勻綺', '林沛潔', '張劼恆', '田軉民', '陳華銘', 
    '王予誠', '邱立堯', '謝竣翔', '林程善', '烏鴻佾', '黃麗安', 
    '邱珈翔', '徐品漢', '許宥琳', '李璿', '楊昀臻', '胡承恩', 
    '莊雅筑', '羅妍溱', '吳文宣', '傅煜旻', '蕭凱億', '林羿萱', 
    '徐芙暄', '陳昱蓁'
]

def create_composite_name(combo_tuple):
    """根據行為組合元組，創建一個標準化的複合行為欄位名"""
    # 例如：('Looking_at_book', 'Sitting_upright') -> 'Comp_Looking_at_book_&_Sitting_upright'
    return 'Comp_' + '_&_'.join(combo_tuple)

def process_behavior_reports_dynamic(root_path, output_filename):
    """
    [v5.0 動態發現版] 自動發現並統計所有單一及複合行為，確保數據完整性。
    """
    data_by_sheet = defaultdict(list)
    
    # --- ★★★【核心升級 1：動態發現所有欄位】★★★ ---
    all_discovered_composite_columns = set()

    print(f"--- Pass 1: Discovering all behaviors ---")
    print(f"開始遍歷資料夾進行學習與發現：{root_path}")
    if not os.path.isdir(root_path):
        print(f"錯誤：找不到指定的資料夾路徑 '{root_path}'。")
        return

    # 第一次遍歷：僅為了發現所有複合行為的組合
    for dirpath, _, filenames in os.walk(root_path):
        for filename in filenames:
            if filename.endswith('.json'):
                json_file_path = os.path.join(dirpath, filename)
                try:
                    with open(json_file_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                    
                    detailed_analysis = data.get('detailed_sequence_analysis', [])
                    for batch in detailed_analysis:
                        highlights = batch.get('analysis', {}).get('per_image_highlights', [])
                        for image_data in highlights:
                            behaviors = image_data.get('behavior_category', [])
                            if isinstance(behaviors, list) and len(behaviors) >= 2:
                                english_behaviors = sorted([BEHAVIOR_MAP[b] for b in behaviors if b in BEHAVIOR_MAP])
                                # 我們只統計兩個行為的組合，這是最常見且最有意義的
                                for combo in combinations(english_behaviors, 2):
                                    all_discovered_composite_columns.add(create_composite_name(combo))
                except Exception:
                    # 在發現階段，忽略處理錯誤的檔案
                    continue
    
    print(f"發現階段完成，共找到 {len(all_discovered_composite_columns)} 種獨特的複合行為組合。")

    # --- 建立最終的欄位順序 ---
    base_columns = ['姓名', 'behavior_positive', 'behavior_negative', 'behavior_neutral'] + sorted(list(BEHAVIOR_MAP.values()))
    final_column_order = base_columns + sorted(list(all_discovered_composite_columns))
    
    print("\n--- Pass 2: Extracting and Counting Data ---")
    # 第二次遍歷：提取數據並計數
    for dirpath, _, filenames in os.walk(root_path):
        for filename in filenames:
            if filename.endswith('.json'):
                json_file_path = os.path.join(dirpath, filename)
                print(f"  正在處理檔案：{os.path.relpath(json_file_path, root_path)}")

                try:
                    with open(json_file_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)

                    metadata = data.get('report_metadata', {})
                    summary = data.get('overall_summary', {})
                    student_name = metadata.get('student_id')
                    if not student_name: continue

                    row_data = {key: 0 for key in final_column_order}
                    row_data['姓名'] = student_name
                    
                    # 提取正負向統計
                    valence_summary = summary.get('valence_summary', {})
                    row_data['behavior_positive'] = valence_summary.get('正向', {}).get('count', 0)
                    row_data['behavior_negative'] = valence_summary.get('負向', {}).get('count', 0)
                    row_data['behavior_neutral'] = valence_summary.get('中性', {}).get('count', 0)

                    # 提取單一行為統計
                    behavior_stats = summary.get('behavior_statistics', [])
                    for behavior in behavior_stats:
                        english_name = BEHAVIOR_MAP.get(behavior.get('behavior_category'))
                        if english_name in row_data:
                            row_data[english_name] = behavior.get('count', 0)

                    # --- ★★★【核心升級 2：動態統計複合行為】★★★ ---
                    detailed_analysis = data.get('detailed_sequence_analysis', [])
                    for batch in detailed_analysis:
                        highlights = batch.get('analysis', {}).get('per_image_highlights', [])
                        for image_data in highlights:
                            behaviors = image_data.get('behavior_category', [])
                            if isinstance(behaviors, list) and len(behaviors) >= 2:
                                english_behaviors = sorted([BEHAVIOR_MAP[b] for b in behaviors if b in BEHAVIOR_MAP])
                                for combo in combinations(english_behaviors, 2):
                                    composite_name = create_composite_name(combo)
                                    if composite_name in row_data:
                                        row_data[composite_name] += 1
                    
                    gen_time = metadata.get('report_generation_time', '未知日期')
                    source_folder = metadata.get('student_image_source_folder', '未知課程')
                    sheet_name = f"{gen_time.replace('/', '')}_{source_folder}"
                    
                    data_by_sheet[sheet_name].append(row_data)

                except Exception as e:
                    print(f"    -> 處理檔案 {filename} 時發生錯誤: {e}，已跳過。")

    if not data_by_sheet:
        print("沒有找到任何可處理的數據，程式結束。")
        return

    print(f"\n數據處理完成，正在寫入 Excel 檔案：{output_filename}")
    with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        for sheet_name in sorted(data_by_sheet.keys()):
            df_present_students = pd.DataFrame(data_by_sheet[sheet_name])
            if df_present_students.empty: continue

            df_all_students = pd.DataFrame({'姓名': FULL_STUDENT_ROSTER})
            df_merged = pd.merge(df_all_students, df_present_students, on='姓名', how='left')
            df_merged = df_merged.fillna(0)

            df_merged['姓名'] = pd.Categorical(df_merged['姓名'], categories=FULL_STUDENT_ROSTER, ordered=True)
            df_merged = df_merged.sort_values('姓名').reset_index(drop=True)

            for col in df_merged.columns:
                if col != '姓名':
                    df_merged[col] = df_merged[col].astype(int)

            # 使用我們動態發現並排序好的最終欄位順序
            final_df = df_merged.reindex(columns=final_column_order, fill_value=0)
            
            final_df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"  已建立工作表：'{sheet_name}'，包含 {len(final_df)} 位學生的數據。")

    print(f"\n成功建立 Excel 檔案 '{output_filename}'！")

if __name__ == '__main__':
    target_path = os.path.join('SynologyDrive', 'json_behavior')
    excel_output_file = 'behavior_summary_DYNAMIC.xlsx'
    process_behavior_reports_dynamic(target_path, excel_output_file)

--- Pass 1: Discovering all behaviors ---
開始遍歷資料夾進行學習與發現：SynologyDrive\json_behavior
發現階段完成，共找到 124 種獨特的複合行為組合。

--- Pass 2: Extracting and Counting Data ---
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250727_100633.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250802_210935.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250804_110148.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250804_140123.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250810_134555.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250819_121508.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250823_190436.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250902_100850.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250908_115153.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250918_173724.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250928_014129.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20251006_154650.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20251015_041632.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_re

In [8]:
# -*- coding: utf-8 -*-
import os
import json
import re
import pandas as pd
from collections import defaultdict

# --- 全域設定：行為類別中英對照表 ---
# 用來將 JSON 裡的中文原始標籤轉換為內部英文代碼 (方便程式處理)
BEHAVIOR_MAP = {
    # --- 視線 ---
    '目視教師': 'Looking_at_teacher',
    '目視黑板': 'Looking_at_blackboard',
    '目視書本/筆記': 'Looking_at_book',
    '目視同學': 'Looking_at_classmates',
    '目視他處': 'Looking_elsewhere',
    # --- 肢體(手部) ---
    '做筆記': 'Note_taking',
    '玩弄手部/文具': 'Fidgeting',
    '觸摸臉部': 'Touching_face',
    '觸摸頭髮': 'Touching_hair',
    # --- 身體姿態 ---
    '坐姿直立': 'Sitting_upright',
    '身體前傾': 'Leaning_forward',
    '身體後靠': 'Leaning_back',
    '低頭(非學習)': 'Looking_down',
    '趴睡': 'Lying_prone',
    '托腮': 'Chin_rest',
    # --- 互動 ---
    '主動舉手': 'Raising_hand_Active',
    '被動舉手': 'Raising_hand_Passive',
    # --- 其他狀態 ---
    '喝水': 'Drinking',
    '飲食': 'Eating',
    '被遮擋/無法判斷': 'Obstructed_Undetermined',
}

# --- ★★★ 核心修改：定義新的 12 個分類群組 ★★★ ---
# 格式： '顯示在 Excel 的欄位名稱': ['需要加總的內部英文代碼', ...]
GROUP_MAPPING = {
    '1目視教師、目視黑板': ['Looking_at_teacher', 'Looking_at_blackboard'],
    '2目視書本/筆記': ['Looking_at_book'],
    '3目視同學、目視他處': ['Looking_at_classmates', 'Looking_elsewhere'],
    '4做筆記': ['Note_taking'],
    '6觸摸臉部、觸摸頭髮、托腮': ['Touching_face', 'Touching_hair', 'Chin_rest'],
    '7坐姿直立、身體前傾': ['Sitting_upright', 'Leaning_forward'],
    '8身體後靠': ['Leaning_back'],
    '9低頭(非學習)、趴睡': ['Looking_down', 'Lying_prone'],
    '10主動舉手': ['Raising_hand_Active'],
    '12飲食、喝水': ['Eating', 'Drinking'],
    '13玩弄手部/文具': ['Fidgeting'],
    '14被遮擋/無法判斷': ['Obstructed_Undetermined']
}

# 定義 Excel 的最終欄位順序
# 包含：姓名 + 正負向情緒 + 上述定義的分類群組
OUTPUT_COLUMN_ORDER = [
    '姓名',
    'behavior_positive', 'behavior_negative', 'behavior_neutral'
] + list(GROUP_MAPPING.keys())

# --- 班級完整名單 ---
FULL_STUDENT_ROSTER = [
    '張玹寧', '許勻綺', '林沛潔', '張劼恆', '田軉民', '陳華銘', 
    '王予誠', '邱立堯', '謝竣翔', '林程善', '烏鴻佾', '黃麗安', 
    '邱珈翔', '徐品漢', '許宥琳', '李璿', '楊昀臻', '胡承恩', 
    '莊雅筑', '羅妍溱', '吳文宣', '傅煜旻', '蕭凱億', '林羿萱', 
    '徐芙暄', '陳昱蓁'
]

def process_behavior_reports_grouped(root_path, output_filename):
    """
    遍歷 JSON 檔案，依照使用者指定的新分組邏輯 (GROUP_MAPPING) 進行加總統計，並匯出 Excel。
    """
    data_by_sheet = defaultdict(list)

    print(f"開始遍歷資料夾：{root_path}")
    if not os.path.isdir(root_path):
        print(f"錯誤：找不到指定的資料夾路徑 '{root_path}'。")
        return

    # 為了快速查找，建立一個 "英文代碼 -> 群組名稱" 的反向索引
    # 例如: 'Looking_at_teacher' -> '1目視教師、目視黑板'
    code_to_group = {}
    for group_name, codes in GROUP_MAPPING.items():
        for code in codes:
            code_to_group[code] = group_name

    # --- 數據收集 ---
    for dirpath, _, filenames in os.walk(root_path):
        for filename in filenames:
            if filename.endswith('.json'):
                json_file_path = os.path.join(dirpath, filename)
                print(f"  正在處理檔案：{os.path.relpath(json_file_path, root_path)}")

                try:
                    with open(json_file_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)

                    metadata = data.get('report_metadata', {})
                    summary = data.get('overall_summary', {})
                    
                    # 取得學生姓名 (加上 .strip() 去除可能存在的空白)
                    student_name = metadata.get('student_id', '').strip()
                    if not student_name:
                        print(f"    -> 警告：檔案 {filename} 缺少 'student_id'，已跳過。")
                        continue

                    # 初始化該行數據，所有欄位預設為 0
                    row_data = {key: 0 for key in OUTPUT_COLUMN_ORDER}
                    row_data['姓名'] = student_name
                    
                    # 1. 讀取情緒數據
                    valence_summary = summary.get('valence_summary', {})
                    row_data['behavior_positive'] = valence_summary.get('正向', {}).get('count', 0)
                    row_data['behavior_negative'] = valence_summary.get('負向', {}).get('count', 0)
                    row_data['behavior_neutral'] = valence_summary.get('中性', {}).get('count', 0)

                    # 2. 讀取並加總行為數據
                    behavior_stats = summary.get('behavior_statistics', [])
                    for behavior in behavior_stats:
                        chinese_name = behavior.get('behavior_category')
                        count = behavior.get('count', 0)
                        
                        # 找出對應的英文代碼
                        english_code = BEHAVIOR_MAP.get(chinese_name)
                        
                        # 如果這個行為代碼屬於我們定義的群組，就加總到該群組
                        if english_code in code_to_group:
                            target_group = code_to_group[english_code]
                            row_data[target_group] += count

                    # 3. 產生 Excel 工作表名稱 (加上安全處理)
                    gen_time = metadata.get('report_generation_time', '未知日期')
                    source_folder = metadata.get('student_image_source_folder', '未知課程')
                    
                    raw_sheet_name = f"{gen_time.replace('/', '')}_{source_folder}"
                    # 替換不合法符號，並截斷長度為 31 (Excel 限制)
                    safe_name = re.sub(r'[:\\/?*\[\]]', '_', raw_sheet_name)
                    sheet_name = safe_name[:31]
                    
                    data_by_sheet[sheet_name].append(row_data)

                except Exception as e:
                    print(f"    -> 處理檔案 {filename} 時發生錯誤: {e}，已跳過。")

    if not data_by_sheet:
        print("沒有找到任何可處理的數據，程式結束。")
        return

    print(f"\n數據處理完成，正在寫入 Excel 檔案：{output_filename}")
    with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        for sheet_name in sorted(data_by_sheet.keys()):
            # 該工作表當天有資料的學生
            df_present = pd.DataFrame(data_by_sheet[sheet_name])
            
            if df_present.empty:
                continue

            # --- 合併完整名單 (確保缺席者也在表上) ---
            df_roster = pd.DataFrame({'姓名': FULL_STUDENT_ROSTER})
            
            # Left Merge: 保留名單上所有人
            df_merged = pd.merge(df_roster, df_present, on='姓名', how='left')
            
            # 填充 NaN 為 0
            df_merged = df_merged.fillna(0)

            # 依照名單順序排序
            df_merged['姓名'] = pd.Categorical(df_merged['姓名'], categories=FULL_STUDENT_ROSTER, ordered=True)
            df_merged = df_merged.sort_values('姓名').reset_index(drop=True)

            # 轉為整數 (美觀)
            for col in df_merged.columns:
                if col != '姓名':
                    df_merged[col] = df_merged[col].astype(int)

            # 依照指定的欄位順序輸出
            final_df = df_merged.reindex(columns=OUTPUT_COLUMN_ORDER, fill_value=0)
            
            final_df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"  已建立工作表：'{sheet_name}'，包含 {len(final_df)} 筆資料。")

    print(f"\n成功建立 Excel 檔案 '{output_filename}'！")

if __name__ == '__main__':
    # 設定資料夾路徑與輸出檔名
    target_path = os.path.join('SynologyDrive', 'json_behavior')
    excel_output_file = 'behavior_summary_grouped.xlsx' 
    
    process_behavior_reports_grouped(target_path, excel_output_file)

開始遍歷資料夾：SynologyDrive\json_behavior
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250727_100633.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250802_210935.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250804_110148.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250804_140123.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250810_134555.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250819_121508.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250823_190436.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250908_115153.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250918_173724.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20250928_014129.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20251006_154650.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20251101_041801.json
  正在處理檔案：傅煜旻\student_傅煜旻_behavior_report_20251125_033439.json
  正在處理檔案：吳文宣\student_吳文宣_behavior_report_20250727_100634.json
  正在處理檔案：吳文宣\student_吳文宣_behavior_report_20250802_205638.json
  正在處理檔案：吳文宣\student_吳文宣_behavior_

### 行為比例

In [ ]:
# -*- coding: utf-8 -*-
import os
import json
import pandas as pd
from collections import defaultdict

# --- 全域設定 ---

# 1. 學生姓名排序列表 (依照您提供的名單)
# 如果有學生不在名單中，他們會被排在最後。
STUDENT_ORDER_LIST = [
    '張玹寧', '許勻綺', '林沛潔', '張劼恆', '田軉民', '陳華銘', 
    '王予誠', '邱立堯', '謝竣翔', '林程善', '烏鴻佾', '黃麗安', 
    '邱珈翔', '徐品漢', '許宥琳', '李璿', '楊昀臻', '胡承恩', 
    '莊雅筑', '羅妍溱', '吳文宣', '傅煜旻', '蕭凱億', '林羿萱', 
    '徐芙暄', '陳昱蓁'
]

# 2. 行為類別中英對照表
# 請確保此對照表與您 JSON 檔案中的 "behavior_category" 完全一致
BEHAVIOR_MAP = {
    '做筆記': 'Note_taking',
    '目視書本/筆記': 'Looking_at_book',
    '低頭(非學習)': 'Looking_down',
    '坐姿直立': 'Sitting_upright',
    '目視教師': 'Looking_at_teacher',
    '翻閱書本': 'Flipping_through_books',
    '玩弄物品': 'Fidgeting_with_objects',
    '目視同學': 'Looking_at_classmates',
    '無明顯特定行為': 'No_specific_behavior',
    '目視黑板': 'Looking_at_blackboard',
    '整理個人物品': 'Organizing_personal_items',
    '目視他處': 'Looking_elsewhere',
    '喝水/飲食': 'Drinking_Eating',
    '被遮擋/無法判斷': 'Obstructed_Undetermined',
    '舉手': 'Raising_hand',
    '趴睡': 'Lying_prone',
    '身體後靠': 'Leaning_back',
    '目視前方': 'Looking_forward', # 保留以相容舊檔案
    '身體前傾': 'Leaning_forward'
}

# 3. 輸出 Excel 的欄位順序
COLUMN_ORDER = [
    '姓名',
    'behavior_positive',
    'behavior_native', # 根據您的範例圖，使用 native
    'behavior_natural',
    'Note_taking',
    'Looking_at_teacher',
    'Flipping_through_books',
    'Fidgeting_with_objects',
    'Looking_at_book',
    'Looking_down',
    'Looking_at_classmates',
    'No_specific_behavior',
    'Looking_at_blackboard',
    'Organizing_personal_items',
    'Looking_elsewhere',
    'Drinking_Eating',
    'Obstructed_Undetermined',
    'Raising_hand',
    'Lying_prone',
    'Leaning_back',
    'Looking_forward',
    'Sitting_upright',
    'Leaning_forward'
]

def process_behavior_reports_wide(root_path, output_filename):
    """
    遍歷指定路徑下的學生資料夾，提取 JSON 數據中的「百分比」，並將每個學生彙總成一列（寬資料），
    最後按照指定的學生順序和欄位順序，匯出到一個 Excel 檔案中。
    """
    data_by_sheet = defaultdict(list)

    print(f"開始遍歷資料夾：{root_path}")
    if not os.path.isdir(root_path):
        print(f"錯誤：找不到指定的資料夾路徑 '{root_path}'。請檢查路徑是否正確。")
        return

    for dirpath, _, filenames in os.walk(root_path):
        for filename in filenames:
            if filename.endswith('.json'):
                json_file_path = os.path.join(dirpath, filename)
                print(f"  正在處理檔案：{os.path.relpath(json_file_path)}")

                try:
                    with open(json_file_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)

                    metadata = data.get('report_metadata', {})
                    summary = data.get('overall_summary', {})
                    
                    student_name = metadata.get('student_id')
                    if not student_name:
                        print(f"    -> 警告：檔案 {filename} 缺少 'student_id'，已跳過。")
                        continue

                    # 初始化一個學生的資料列
                    row_data = {'姓名': student_name}
                    
                    # --- 【修改點 1】改為提取「百分比 (percentage)」 ---
                    
                    # 1. 提取正、負、中性行為的「百分比」
                    valence_summary = summary.get('valence_summary', {})
                    row_data['behavior_positive'] = valence_summary.get('正向', {}).get('percentage')
                    row_data['behavior_native'] = valence_summary.get('負向', {}).get('percentage')
                    row_data['behavior_natural'] = valence_summary.get('中性', {}).get('percentage')

                    # 2. 提取各種具體行為的「百分比」
                    behavior_stats = summary.get('behavior_statistics', [])
                    for behavior in behavior_stats:
                        chinese_name = behavior.get('behavior_category')
                        english_name = BEHAVIOR_MAP.get(chinese_name)
                        
                        if english_name:
                            row_data[english_name] = behavior.get('percentage') # 提取 percentage
                        else:
                            print(f"    -> 注意：在檔案 {filename} 中發現未知的行為類別 '{chinese_name}'，此欄位將被忽略。")
                    
                    # 根據日期和課程分類
                    gen_time = metadata.get('report_generation_time', '未知日期')
                    source_folder = metadata.get('student_image_source_folder', '未知課程')
                    sheet_name = f"{gen_time.replace('/', '')}_{source_folder}"
                    
                    data_by_sheet[sheet_name].append(row_data)

                except Exception as e:
                    print(f"    -> 處理檔案 {filename} 時發生錯誤: {e}，已跳過。")

    if not data_by_sheet:
        print("沒有找到任何可處理的數據，程式結束。")
        return

    # --- 將整理好的數據寫入 Excel ---
    print(f"\n數據處理完成，正在寫入 Excel 檔案：{output_filename}")
    with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        for sheet_name in sorted(data_by_sheet.keys()):
            df = pd.DataFrame(data_by_sheet[sheet_name])
            
            # --- 【修改點 2】依照指定的學生名單排序 ---
            
            # 1. 將 '姓名' 欄位轉換為 Categorical Type，這樣 pandas 才認得指定的順序
            df['姓名'] = pd.Categorical(df['姓名'], categories=STUDENT_ORDER_LIST, ordered=True)
            
            # 2. 根據 '姓名' 欄位排序。不在名單中的學生會被排在後面(如果有)
            df.sort_values('姓名', inplace=True)
            
            # 3. 使用 reindex 確保欄位順序和完整性
            # 這一步會自動新增所有在 COLUMN_ORDER 中但 df 中不存在的欄位（值為NaN），並確保順序完全正確
            final_df = df.reindex(columns=COLUMN_ORDER)
            
            # 將整理好的 DataFrame 寫入工作表
            final_df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"  已建立工作表：'{sheet_name}'，包含 {len(final_df)} 位學生的數據。")

    print(f"\n成功建立 Excel 檔案 '{output_filename}'！")


# --- 主程式執行區塊 ---
if __name__ == '__main__':
    # 將此 .py 檔案放置在與 'SynologyDrive' 資料夾同一個目錄層級下執行
    # 或者，您也可以提供絕對路徑，例如：r'C:\Users\YourUser\SynologyDrive\json_behavior'
    target_path = os.path.join('SynologyDrive', 'json_behavior')
    
    # 定義輸出的 Excel 檔案名稱
    excel_output_file = 'behavior_summary_percentage.xlsx'
    
    # 執行主程式
    process_behavior_reports_wide(target_path, excel_output_file)

### 刪除特定日期報告

In [5]:
# -*- coding: utf-8 -*-
import os
import json

# --- 設定 ---

# 1. 請將此路徑替換為您包含所有學生報告的根資料夾
#    使用 'r' 前綴可以確保 Windows 路徑的反斜線被正確處理
ROOT_FOLDER = r"SynologyDrive\json_behavior"

# 2. 這是您要用來識別舊報告的【鍵】
TARGET_KEY = "report_generation_time"

# 3. 這是您要匹配的【值】
TARGET_VALUE = "09/28"

# --- 主程式 ---

def find_and_delete_old_reports():
    """
    遍歷指定資料夾，找到並刪除所有符合條件的舊 JSON 報告。
    """
    print("--- 舊版 JSON 報告清理工具 ---")
    print(f"目標資料夾: {os.path.abspath(ROOT_FOLDER)}")
    print(f"刪除條件: JSON 內部 \"{TARGET_KEY}\" 的值為 \"{TARGET_VALUE}\"")
    print("-" * 40)

    if not os.path.isdir(ROOT_FOLDER):
        print(f"❌ 錯誤：找不到指定的資料夾 '{ROOT_FOLDER}'。請檢查路徑是否正確。")
        return

    files_to_delete = []
    total_files_checked = 0

    print("🔍 正在掃描所有學生資料夾，請稍候...")

    # os.walk 會遍歷所有子資料夾和檔案
    for dirpath, _, filenames in os.walk(ROOT_FOLDER):
        for filename in filenames:
            # 只處理 .json 檔案
            if filename.lower().endswith(".json"):
                total_files_checked += 1
                filepath = os.path.join(dirpath, filename)
                
                try:
                    with open(filepath, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                    
                    # 安全地檢查嵌套的鍵是否存在，並比對值
                    # data.get("report_metadata", {})確保即使沒有"report_metadata"也不會報錯
                    if data.get("report_metadata", {}).get(TARGET_KEY) == TARGET_VALUE:
                        print(f"  [🎯 找到目標] - {filepath}")
                        files_to_delete.append(filepath)

                except json.JSONDecodeError:
                    print(f"  [⚠️ 警告] - 無法解析 JSON 檔案，已跳過: {filepath}")
                except Exception as e:
                    print(f"  [❌ 錯誤] - 讀取檔案時發生未知錯誤 {filepath}: {e}")

    print("\n掃描完成。")
    print("-" * 40)

    # 如果沒有找到任何符合條件的檔案
    if not files_to_delete:
        print(f"🎉 檢查了 {total_files_checked} 個 JSON 檔案，沒有找到符合條件的舊報告。無需執行任何操作。")
        return

    # --- 安全確認步驟 ---
    print(f"📊 總共檢查了 {total_files_checked} 個 JSON 檔案，找到 {len(files_to_delete)} 個符合條件的檔案將被刪除：")
    for file_path in files_to_delete:
        print(f"  - {file_path}")

    print("\n🚨🚨🚨【高風險操作警告】🚨🚨🚨")
    print("檔案刪除後將無法恢復。強烈建議您在繼續前，先備份整個資料夾。")
    
    # 請求使用者最終確認
    try:
        confirm = input("您確定要永久刪除以上列出的所有檔案嗎？ (請輸入 'y' 確認, 或輸入其他任意鍵取消): ").lower()
    except KeyboardInterrupt:
        print("\n操作被使用者中斷。")
        confirm = 'n'


    if confirm == 'y':
        print("\n正在執行刪除操作...")
        deleted_count = 0
        for file_path in files_to_delete:
            try:
                os.remove(file_path)
                print(f"  [✅ 已刪除] {file_path}")
                deleted_count += 1
            except Exception as e:
                print(f"  [❌ 刪除失敗] {file_path} - 原因: {e}")
        
        print("-" * 40)
        print(f"👍 操作完成！成功刪除了 {deleted_count} / {len(files_to_delete)} 個檔案。")
    else:
        print("\n操作已取消。沒有任何檔案被刪除。")

if __name__ == "__main__":
    find_and_delete_old_reports()

--- 舊版 JSON 報告清理工具 ---
目標資料夾: c:\Users\User\Desktop\test\SynologyDrive\json_behavior
刪除條件: JSON 內部 "report_generation_time" 的值為 "09/28"
----------------------------------------
🔍 正在掃描所有學生資料夾，請稍候...
  [🎯 找到目標] - SynologyDrive\json_behavior\傅煜旻\student_傅煜旻_behavior_report_20251116_041723.json
  [🎯 找到目標] - SynologyDrive\json_behavior\吳文宣\student_吳文宣_behavior_report_20251116_041449.json
  [🎯 找到目標] - SynologyDrive\json_behavior\張玹寧\student_張玹寧_behavior_report_20251116_021416.json
  [🎯 找到目標] - SynologyDrive\json_behavior\徐品漢\student_徐品漢_behavior_report_20251116_033934.json
  [🎯 找到目標] - SynologyDrive\json_behavior\徐芙暄\student_徐芙暄_behavior_report_20251116_042500.json
  [🎯 找到目標] - SynologyDrive\json_behavior\李璿\student_李璿_behavior_report_20251116_035320.json
  [🎯 找到目標] - SynologyDrive\json_behavior\林沛潔\student_林沛潔_behavior_report_20251116_024809.json
  [🎯 找到目標] - SynologyDrive\json_behavior\林程善\student_林程善_behavior_report_20251116_031832.json
  [🎯 找到目標] - SynologyDrive\json_behavior\烏鴻佾\student

### 移動指定日期報告至其他資料夾

In [4]:
# -*- coding: utf-8 -*-
import os
import json
import shutil

# --- 1. 設定區 ---

# 1. 根據您提供的最新資訊，設定正確的根資料夾
ROOT_FOLDER = r"C:\Users\User\Desktop\test\SynologyDrive\json_behavior" 

# 2. 用於存放提取出的報告的基礎資料夾
DESTINATION_BASE_FOLDER = r"extracted_reports"

# 3. 根據您提供的 JSON 結構，設定正確的鍵
NESTING_KEY = "report_metadata"
TARGET_KEY = "report_generation_time"


# --- 2. 主程式 ---

def extract_reports_by_date_v3():
    """
    【版本三 - 最終修正版】
    遍歷指定資料夾，通過讀取並解析每個 JSON 檔案的【內部內容】，
    來找到符合指定日期的報告，並將它們複製到一個新的資料夾。
    """
    print("--- JSON 報告提取工具 v3 (基於 JSON 內部內容) ---")

    # --- 步驟 1: 獲取使用者輸入 ---
    try:
        target_date_str = input("请输入您要提取的報告日期 (格式如 08/24): ").strip()
        if not target_date_str:
            print("\n❌ 未輸入任何日期，程式已終止。")
            return
    except KeyboardInterrupt:
        print("\n操作被使用者中斷。")
        return

    # --- 步驟 2: 準備目標路徑和資料夾 ---
    safe_date_folder_name = f"extracted_{target_date_str.replace('/', '_')}"
    destination_folder = os.path.join(DESTINATION_BASE_FOLDER, safe_date_folder_name)

    try:
        os.makedirs(destination_folder, exist_ok=True)
    except Exception as e:
        print(f"❌ 錯誤：無法創建目標資料夾 '{destination_folder}'。原因: {e}")
        return

    # --- 步驟 3: 顯示設定並掃描檔案 ---
    print("-" * 50)
    print(f"源資料夾: {os.path.abspath(ROOT_FOLDER)}")
    print(f"目標資料夾: {os.path.abspath(destination_folder)}")
    print(f"提取條件: JSON 內部 \"{NESTING_KEY}\" -> \"{TARGET_KEY}\" 的值為 \"{target_date_str}\"")
    print("-" * 50)

    if not os.path.isdir(ROOT_FOLDER):
        print(f"❌ 錯誤：找不到指定的源資料夾 '{ROOT_FOLDER}'。請檢查路徑是否正確。")
        return

    files_to_copy = []
    total_files_checked = 0

    print("🔍 正在掃描並讀取所有 JSON 檔案，請稍候...")

    # 使用 os.walk 遍歷所有子資料夾和檔案
    for dirpath, _, filenames in os.walk(ROOT_FOLDER):
        for filename in filenames:
            if filename.lower().endswith(".json"):
                total_files_checked += 1
                filepath = os.path.join(dirpath, filename)
                
                try:
                    with open(filepath, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                    
                    # --- 【核心修正點】: 精確地從JSON內部獲取日期值 ---
                    # 使用 .get() 來安全地訪問，避免因缺少鍵而引發錯誤
                    metadata = data.get(NESTING_KEY)
                    if metadata and isinstance(metadata, dict):
                        value_found = metadata.get(TARGET_KEY)
                        
                        # 檢查值是否匹配
                        if value_found == target_date_str:
                            print(f"  [🎯 找到目標] - {filepath}")
                            files_to_copy.append(filepath)

                except json.JSONDecodeError:
                    print(f"  [⚠️ 警告] - 無法解析 JSON，已跳過: {filepath}")
                except Exception as e:
                    print(f"  [❌ 錯誤] - 處理檔案時發生未知錯誤 {filepath}: {e}")

    print("\n掃描完成。")
    print("-" * 50)

    # --- 步驟 4: 執行複製操作 ---
    if not files_to_copy:
        print(f"🎉 檢查了 {total_files_checked} 個 JSON 檔案，沒有找到日期為 '{target_date_str}' 的報告。")
        print(f"   請確認您的 JSON 檔案中 \"{TARGET_KEY}\" 欄位的值是否正確。")
        return

    print(f"📊 總共找到 {len(files_to_copy)} 個符合條件的檔案，準備開始複製...")
    
    copied_count = 0
    failed_count = 0

    for source_path in files_to_copy:
        try:
            # 將檔案複製到目標資料夾，保持原檔名
            shutil.copy2(source_path, destination_folder)
            print(f"  [✅ 已複製] {os.path.basename(source_path)}")
            copied_count += 1
        except Exception as e:
            print(f"  [❌ 複製失敗] {os.path.basename(source_path)} - 原因: {e}")
            failed_count += 1
            
    print("-" * 50)
    print("👍 操作完成！")
    print(f"成功複製了 {copied_count} 個檔案。")
    if failed_count > 0:
        print(f"有 {failed_count} 個檔案複製失敗，請檢查上面的錯誤訊息。")
    print(f"所有檔案均已儲存至: {os.path.abspath(destination_folder)}")


if __name__ == "__main__":
    extract_reports_by_date_v3()

--- JSON 報告提取工具 v3 (基於 JSON 內部內容) ---
--------------------------------------------------
源資料夾: C:\Users\User\Desktop\test\SynologyDrive\json_behavior
目標資料夾: c:\Users\User\Desktop\test\extracted_reports\extracted_09_28
提取條件: JSON 內部 "report_metadata" -> "report_generation_time" 的值為 "09/28"
--------------------------------------------------
🔍 正在掃描並讀取所有 JSON 檔案，請稍候...
  [🎯 找到目標] - C:\Users\User\Desktop\test\SynologyDrive\json_behavior\傅煜旻\student_傅煜旻_behavior_report_20251116_041723.json
  [🎯 找到目標] - C:\Users\User\Desktop\test\SynologyDrive\json_behavior\吳文宣\student_吳文宣_behavior_report_20251116_041449.json
  [🎯 找到目標] - C:\Users\User\Desktop\test\SynologyDrive\json_behavior\張玹寧\student_張玹寧_behavior_report_20251116_021416.json
  [🎯 找到目標] - C:\Users\User\Desktop\test\SynologyDrive\json_behavior\徐品漢\student_徐品漢_behavior_report_20251116_033934.json
  [🎯 找到目標] - C:\Users\User\Desktop\test\SynologyDrive\json_behavior\徐芙暄\student_徐芙暄_behavior_report_20251116_042500.json
  [🎯 找到目標] - C:\Users\User\

In [57]:
import os
import json
from collections import defaultdict

def find_duplicate_report_times(root_folder_path):
    """
    遍歷指定的根資料夾，找出每個學生底下 report_generation_time 重複的 JSON 檔案。

    Args:
        root_folder_path (str): 包含所有學生資料夾的根目錄路徑。

    Returns:
        dict: 一個字典，儲存了有重複報告時間的學生及其對應的檔案資訊。
    """
    # 用於儲存結果的結構：{ '學生姓名': { '報告時間': ['檔案路徑1', '檔案路徑2'] } }
    student_reports = defaultdict(lambda: defaultdict(list))

    # 檢查路徑是否存在
    if not os.path.exists(root_folder_path):
        print(f"錯誤：找不到指定的路徑 '{root_folder_path}'")
        return {}

    # 遍歷根目錄下的所有學生資料夾
    for student_name in os.listdir(root_folder_path):
        student_folder_path = os.path.join(root_folder_path, student_name)

        # 確認這是一個資料夾
        if os.path.isdir(student_folder_path):
            # 遍歷學生資料夾中的所有檔案
            for filename in os.listdir(student_folder_path):
                # 確保檔案是 .json 結尾
                if filename.endswith(".json"):
                    file_path = os.path.join(student_folder_path, filename)
                    
                    try:
                        with open(file_path, 'r', encoding='utf-8') as f:
                            data = json.load(f)
                            
                            # 提取 report_generation_time
                            # 根據您提供的結構，它在 report_metadata 裡面
                            metadata = data.get("report_metadata", {})
                            report_time = metadata.get("report_generation_time")
                            
                            if report_time:
                                student_reports[student_name][report_time].append(filename)
                                
                    except json.JSONDecodeError:
                        print(f"警告：無法解析檔案 '{file_path}'，可能不是有效的 JSON 格式。")
                    except Exception as e:
                        print(f"處理檔案 '{file_path}' 時發生錯誤：{e}")

    # 找出重複的項目
    duplicates = {}
    for student, reports in student_reports.items():
        for time, files in reports.items():
            # 如果同一個時間點有超過一個檔案，就是重複的
            if len(files) > 1:
                if student not in duplicates:
                    duplicates[student] = {}
                duplicates[student][time] = files
                
    return duplicates

def print_duplicates_table(duplicates):
    """
    將重複的報告以表格形式印出。
    """
    if not duplicates:
        print("太好了！檢查完畢，沒有發現任何重複的 report_generation_time。")
        return

    print("發現以下重複的 report_generation_time 項目：")
    print("-" * 80)
    print(f"{'學生姓名':<15} | {'重複的報告時間':<20} | {'對應的 JSON 檔案列表'}")
    print("-" * 80)

    for student, times in duplicates.items():
        for time, files in times.items():
            # 為了讓排版整齊，將檔案列表轉換成字串
            files_str = ", ".join(files)
            print(f"{student:<15} | {time:<20} | {files_str}")

    print("-" * 80)

# --- 主程式執行區 ---
if __name__ == "__main__":
    # !!! 請將此路徑修改為您 'json_behavior' 資料夾的實際路徑 !!!
    # 例如： "D:\\SynologyDrive\\json_behavior"
    # 注意：在 Windows 中，路徑的分隔符號建議使用 \\ 或 /
    root_folder_path = "SynologyDrive/json_behavior"
    
    # 執行查找
    duplicate_data = find_duplicate_report_times(root_folder_path)
    
    # 印出結果
    print_duplicates_table(duplicate_data)

太好了！檢查完畢，沒有發現任何重複的 report_generation_time。


In [3]:
import os
import json
from collections import defaultdict
import shutil

def find_all_duplicate_reports(root_folder_path):
    """
    遍歷根資料夾，找出所有學生的、有重複 report_generation_time 的 JSON 檔案。
    這個函數是前置作業，會找出所有的重複項以供後續篩選。

    Args:
        root_folder_path (str): 包含所有學生資料夾的根目錄路徑。

    Returns:
        dict: 一個字典，結構為 { '學生姓名': { '報告時間': ['檔案路徑1', '檔案路徑2', ...] } }
    """
    student_reports = defaultdict(lambda: defaultdict(list))
    
    if not os.path.isdir(root_folder_path):
        print(f"錯誤：找不到指定的資料夾 '{root_folder_path}'")
        return None

    print("正在分析所有 JSON 檔案，請稍候...")
    
    # 遍歷所有學生資料夾
    for student_name in os.listdir(root_folder_path):
        student_folder_path = os.path.join(root_folder_path, student_name)
        if os.path.isdir(student_folder_path):
            for filename in os.listdir(student_folder_path):
                if filename.endswith(".json"):
                    file_path = os.path.join(student_folder_path, filename)
                    try:
                        with open(file_path, 'r', encoding='utf-8') as f:
                            data = json.load(f)
                            metadata = data.get("report_metadata", {})
                            report_time = metadata.get("report_generation_time")
                            if report_time:
                                student_reports[student_name][report_time].append(file_path)
                    except Exception as e:
                        print(f"處理檔案 '{file_path}' 時發生錯誤：{e}")
    
    # 篩選出有重複的項目
    duplicates = {}
    for student, reports in student_reports.items():
        for time, files in reports.items():
            if len(files) > 1:
                if student not in duplicates:
                    duplicates[student] = {}
                # 關鍵：根據檔案路徑（內含時間戳）排序，確保最後一個永遠是最新產生的
                duplicates[student][time] = sorted(files)
                
    print("分析完成！")
    return duplicates

def extract_and_move_older_files(all_duplicates, target_report_time, destination_folder):
    """
    根據使用者指定的 report_time，找出較舊的檔案並移動它們。

    Args:
        all_duplicates (dict): 包含所有重複項的字典。
        target_report_time (str): 使用者想要處理的報告時間，例如 "09/28"。
        destination_folder (str): 要將舊檔案移至的目標資料夾。
    """
    if not os.path.exists(destination_folder):
        print(f"目標資料夾不存在，正在為您建立：'{destination_folder}'")
        os.makedirs(destination_folder)

    moved_files_count = 0
    found_match = False

    print("-" * 50)
    print(f"開始處理報告時間為 '{target_report_time}' 的重複檔案...")

    for student, times in all_duplicates.items():
        # 檢查這位學生是否有符合目標時間的重複報告
        if target_report_time in times:
            found_match = True
            file_paths = times[target_report_time]
            
            # 排序後，除了最後一個檔案之外，都是較舊的檔案
            older_files_to_move = file_paths[:-1]
            newest_file = os.path.basename(file_paths[-1])

            print(f"\n找到學生 '{student}' 的重複報告：")
            print(f"  - 較新的檔案 (將保留): {newest_file}")
            
            if not older_files_to_move:
                print("  - 沒有找到比它更舊的檔案可移動。")
                continue

            for old_file_path in older_files_to_move:
                filename = os.path.basename(old_file_path)
                destination_path = os.path.join(destination_folder, filename)
                try:
                    print(f"  - 正在移動較舊的檔案: {filename}")
                    shutil.move(old_file_path, destination_path)
                    moved_files_count += 1
                except Exception as e:
                    print(f"    -> 移動失敗！錯誤訊息: {e}")

    print("-" * 50)
    if not found_match:
        print(f"處理完成。在所有資料中，沒有找到任何報告時間為 '{target_report_time}' 的重複項目。")
    else:
        print(f"處理完成！總共移動了 {moved_files_count} 個較舊的檔案到 '{destination_folder}'。")


# --- 主程式執行區 ---
if __name__ == "__main__":
    # 1. 讓使用者輸入必要的資訊
    print("--- JSON 重複報告整理工具 ---")
    root_folder = input("請輸入您的 'json_behavior' 資料夾完整路徑：").strip()
    report_time_to_process = input("請輸入您想要處理的報告時間 (格式為 MM/DD, 例如 09/28)：").strip()
    archive_folder = input("請輸入一個用來存放舊報告的資料夾路徑 (如果不存在將會自動建立)：").strip()

    # 2. 檢查輸入是否為空
    if not root_folder or not report_time_to_process or not archive_folder:
        print("\n錯誤：所有欄位都必須輸入！程式已終止。")
    else:
        # 3. 執行分析，找出所有重複的報告
        all_duplicate_data = find_all_duplicate_reports(root_folder)

        # 4. 如果分析成功，則執行移動操作
        if all_duplicate_data is not None:
            extract_and_move_older_files(all_duplicate_data, report_time_to_process, archive_folder)

--- JSON 重複報告整理工具 ---
正在分析所有 JSON 檔案，請稍候...
分析完成！
--------------------------------------------------
開始處理報告時間為 '09/28' 的重複檔案...
--------------------------------------------------
處理完成。在所有資料中，沒有找到任何報告時間為 '09/28' 的重複項目。


### 提取 Rater 數據

In [9]:
# create_dated_consolidated_report.py (v2.0)
import pandas as pd
import os
import json
import re

# =========================================================================
# --- 1. 配置區 ---
# =========================================================================

BASE_PATH = 'training_json' 
OUTPUT_PATH = r"C:\Users\User\Desktop\test\training_json"
OUTPUT_FILENAME = "consolidated_annotation_report_v2.xlsx" # 更新檔名以區分

RATER_FILES = {
    'c123': os.path.join(BASE_PATH, 'human_annotation_c123_v2.json'),
    'd123': os.path.join(BASE_PATH, 'human_annotation_d123_v2.json')
}

BEHAVIOR_CATEGORIES_GROUPED = {
    "視線": ["目視教師", "目視黑板", "目視書本/筆記", "目視同學", "目視他處"],
    "肢體(手部)": ["做筆記", "翻書", "觸摸臉部", "觸摸頭髮", "托腮"],
    "身體姿態": ["坐姿直立", "身體前傾", "身體後靠", "低頭(非學習)", "趴睡"],
    "互動": ["主動舉手", "被動舉手"],
    "其他狀態": ["喝水", "飲食", "玩弄手部/文具", "被遮擋/無法判斷"]
}
ALL_CATEGORIES = list(BEHAVIOR_CATEGORIES_GROUPED.keys())
BEHAVIOR_TO_CATEGORY = {behavior: category for category, behaviors in BEHAVIOR_CATEGORIES_GROUPED.items() for behavior in behaviors}

# =========================================================================
# --- 2. 輔助函數 ---
# =========================================================================

def extract_date_from_path(image_path):
    match = re.search(r'student_week_photo[\\/](\d{4})[\\/]', image_path)
    return match.group(1) if match else "unknown"

def normalize_to_list(behavior_data):
    if isinstance(behavior_data, list): return behavior_data
    if isinstance(behavior_data, str): return [behavior_data]
    return []

def get_ai_label_for_category(ai_label_list, target_category):
    for label in ai_label_list:
        if BEHAVIOR_TO_CATEGORY.get(label) == target_category:
            return label
    return None

# =========================================================================
# --- 3. 主執行流程 ---
# =========================================================================

def create_dated_consolidated_excel():
    print("--- 開始生成按日期分頁的合併式 Excel 報告 (v2.0) ---")

    # --- 步驟 1 & 2: 讀取所有數據 (與之前相同) ---
    if not os.path.exists(OUTPUT_PATH):
        os.makedirs(OUTPUT_PATH)
        print(f"✅ 已創建輸出資料夾: {OUTPUT_PATH}")

    active_raters = list(RATER_FILES.keys())
    rater_data, all_image_paths = {}, set()
    for rater_id, file_path in RATER_FILES.items():
        if not os.path.isfile(file_path):
            active_raters.remove(rater_id)
            continue
        with open(file_path, 'r', encoding='utf-8') as f: data = json.load(f)
        processed_data = {item['image_path']: item.get('corrected_behaviors', {}) for item in data if item.get('image_path')}
        rater_data[rater_id] = processed_data
        all_image_paths.update(processed_data.keys())
        print(f"✅ 成功讀取標註員 {rater_id} 的數據。")

    with open(RATER_FILES[active_raters[0]], 'r', encoding='utf-8') as f: raw_data = json.load(f)
    ai_data = {item['image_path']: normalize_to_list(item.get('original_behavior')) for item in raw_data if item.get('image_path')}
    all_image_paths.update(ai_data.keys())
    print(f"✅ 成功提取 AI 的數據。")

    # --- 步驟 3: 創建基礎 DataFrame (與之前相同) ---
    df_records = []
    for img in sorted(list(all_image_paths)):
        record = {'image_path': img}
        for rater_id in active_raters:
            record[f'behavior_{rater_id}'] = rater_data.get(rater_id, {}).get(img, {})
        record['behavior_AI'] = ai_data.get(img, [])
        df_records.append(record)
    
    base_df = pd.DataFrame(df_records)
    base_df['date'] = base_df['image_path'].apply(extract_date_from_path)
    print("✅ 已將所有數據對齊並整合至基礎 DataFrame。")

    # --- 步驟 4: 準備【總體】的 DataFrame (與之前相同) ---
    print("\n--- 正在準備【總體】數據視圖 ---")
    
    # 4.1: Aligned_Comparison_View (寬格式)
    wide_df_overall = base_df[['image_path', 'date']].copy()
    for category in ALL_CATEGORIES:
        for rater_id in active_raters:
            wide_df_overall[f'{rater_id}_{category}'] = base_df[f'behavior_{rater_id}'].apply(lambda d: d.get(category))
        wide_df_overall[f'AI_{category}'] = base_df['behavior_AI'].apply(get_ai_label_for_category, target_category=category)

    # 4.2: Raw_Data_Long_Format (長格式)
    long_format_records = []
    for _, row in base_df.iterrows():
        for rater_id in active_raters:
            for category, label in row[f'behavior_{rater_id}'].items():
                long_format_records.append({'image_path': row['image_path'], 'date': row['date'], 'rater_id': rater_id, 'category': category, 'behavior_label': label})
        for label in row['behavior_AI']:
            category = BEHAVIOR_TO_CATEGORY.get(label, '未知類別')
            long_format_records.append({'image_path': row['image_path'], 'date': row['date'], 'rater_id': 'AI', 'category': category, 'behavior_label': label})
    long_df_overall = pd.DataFrame(long_format_records)

    # 4.3: SPSS_Kappa_Ready (SPSS適用格式)
    spss_df_overall = long_df_overall.pivot_table(
        index=['image_path', 'date', 'category'], columns='rater_id', values='behavior_label', aggfunc='first'
    ).reset_index()
    spss_df_overall.rename(columns=lambda c: f'rater_{c}' if c in active_raters + ['AI'] else c, inplace=True)
    print("✅ 已生成三種總體數據視圖。")

    # --- 步驟 5: 【v2.0 修改點】將所有數據按日期和總結寫入 Excel ---
    output_filepath = os.path.join(OUTPUT_PATH, OUTPUT_FILENAME)
    print(f"\n--- 正在將所有數據寫入 Excel 檔案: {output_filepath} ---")
    
    with pd.ExcelWriter(output_filepath, engine='openpyxl') as writer:
        # 首先，循環遍歷每個日期，寫入單日的工作表
        unique_dates = sorted([d for d in base_df['date'].unique() if d != "unknown"])
        for date in unique_dates:
            print(f"\n--- 正在處理日期: {date} ---")
            
            # 從總體 DataFrame 中篩選出當日數據
            daily_wide_df = wide_df_overall[wide_df_overall['date'] == date]
            daily_spss_df = spss_df_overall[spss_df_overall['date'] == date]
            daily_long_df = long_df_overall[long_df_overall['date'] == date]
            
            # 寫入工作表，並在名稱中加入日期
            daily_wide_df.to_excel(writer, sheet_name=f'Aligned_{date}', index=False)
            print(f"   - ✅ 已寫入工作表: Aligned_{date}")
            daily_spss_df.to_excel(writer, sheet_name=f'SPSS_{date}', index=False)
            print(f"   - ✅ 已寫入工作表: SPSS_{date}")
            daily_long_df.to_excel(writer, sheet_name=f'LongFormat_{date}', index=False)
            print(f"   - ✅ 已寫入工作表: LongFormat_{date}")

        # 最後，寫入包含所有數據的總結工作表
        print("\n--- 正在寫入總結工作表 ---")
        wide_df_overall.to_excel(writer, sheet_name='Aligned_Overall', index=False)
        print("   - ✅ 已寫入工作表: Aligned_Overall")
        
        spss_df_overall.to_excel(writer, sheet_name='SPSS_Overall', index=False)
        print("   - ✅ 已寫入工作表: SPSS_Overall")
        
        long_df_overall.to_excel(writer, sheet_name='LongFormat_Overall', index=False)
        print("   - ✅ 已寫入工作表: LongFormat_Overall")

    print("\n🎉 全部處理完成！按日期分頁的合併式報告已成功生成。")

# =========================================================================
# --- 4. 執行程式 ---
# =========================================================================
if __name__ == "__main__":
    create_dated_consolidated_excel()

--- 開始生成按日期分頁的合併式 Excel 報告 (v2.0) ---
✅ 成功讀取標註員 c123 的數據。
✅ 成功讀取標註員 d123 的數據。
✅ 成功提取 AI 的數據。
✅ 已將所有數據對齊並整合至基礎 DataFrame。

--- 正在準備【總體】數據視圖 ---
✅ 已生成三種總體數據視圖。

--- 正在將所有數據寫入 Excel 檔案: C:\Users\User\Desktop\test\training_json\consolidated_annotation_report_v2.xlsx ---

--- 正在處理日期: 0824 ---
   - ✅ 已寫入工作表: Aligned_0824
   - ✅ 已寫入工作表: SPSS_0824
   - ✅ 已寫入工作表: LongFormat_0824

--- 正在處理日期: 0907 ---
   - ✅ 已寫入工作表: Aligned_0907
   - ✅ 已寫入工作表: SPSS_0907
   - ✅ 已寫入工作表: LongFormat_0907

--- 正在處理日期: 0914 ---
   - ✅ 已寫入工作表: Aligned_0914
   - ✅ 已寫入工作表: SPSS_0914
   - ✅ 已寫入工作表: LongFormat_0914

--- 正在處理日期: 0918 ---
   - ✅ 已寫入工作表: Aligned_0918
   - ✅ 已寫入工作表: SPSS_0918
   - ✅ 已寫入工作表: LongFormat_0918

--- 正在處理日期: 0921 ---
   - ✅ 已寫入工作表: Aligned_0921
   - ✅ 已寫入工作表: SPSS_0921
   - ✅ 已寫入工作表: LongFormat_0921

--- 正在處理日期: 0928 ---
   - ✅ 已寫入工作表: Aligned_0928
   - ✅ 已寫入工作表: SPSS_0928
   - ✅ 已寫入工作表: LongFormat_0928

--- 正在寫入總結工作表 ---
   - ✅ 已寫入工作表: Aligned_Overall
   - ✅ 已寫入工作表: SPSS_Overall
   - ✅ 已寫入工作表: Long

# ICC

In [ ]:
# calculate_consistency.py (v2.0 - 整合了 HTML 審閱頁面生成器)
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score
import pingouin as pg
import os
import webbrowser

# =========================================================================
# --- 1. 配置區 ---
# =========================================================================
# 定義標註員的檔案路徑
BASE_PATH = 'training_json' # 假設此腳本與 training_json 資料夾在同一目錄
RATER_FILES = {
    'c123': os.path.join(BASE_PATH, 'human_annotation_c123.json'),
    'd123': os.path.join(BASE_PATH, 'human_annotation_d123.json')
    # 如果有第三位標註員，可以在這裡加上
    # 'e123': os.path.join(BASE_PATH, 'human_annotation_e123.json')
}
# 生成的審閱頁面檔名
REVIEW_PAGE_FILENAME = 'review_disagreements.html'

# =========================================================================
# --- 【全新函數】生成審閱頁面 ---
# =========================================================================
def generate_review_page(disagreements_df, output_filename, rater_names):
    """生成一個簡單的 HTML 頁面，用於視覺化審閱不一致的案例。"""
    if disagreements_df.empty:
        return

    # 從 rater_names 動態生成 CSS 樣式和 HTML 標題
    rater_css = ""
    rater_headers = ""
    rater_columns = ""
    colors = ['#007bff', '#dc3545', '#28a745', '#ffc107', '#17a2b8'] # 為多個標註員準備顏色
    
    for i, name in enumerate(rater_names):
        color = colors[i % len(colors)]
        rater_css += f".rater_{name} {{ color: {color}; font-weight: bold; }}\n"
        rater_headers += f"<th>標註員 {name}</th>\n"
        rater_columns += f'<td><span class="rater_{name}">{{row["behavior_{name}"]}}</span></td>\n'


    html_template = """
    <!DOCTYPE html>
    <html lang="zh-Hant">
    <head>
        <meta charset="UTF-8">
        <title>標註分歧審閱</title>
        <style>
            body {{ font-family: 'Segoe UI', 'Microsoft JhengHei', sans-serif; margin: 20px; background-color: #f8f9fa; color: #343a40; }}
            h1, h2 {{ color: #0056b3; border-bottom: 2px solid #dee2e6; padding-bottom: 10px; }}
            .container {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(350px, 1fr)); gap: 25px; }}
            .case {{ background-color: white; border: 1px solid #ddd; border-radius: 8px; padding: 15px; box-shadow: 0 4px 8px rgba(0,0,0,0.05); transition: box-shadow 0.3s ease; }}
            .case:hover {{ box-shadow: 0 8px 16px rgba(0,0,0,0.1); }}
            .case img {{ max-width: 100%; height: auto; border-radius: 4px; margin-bottom: 15px; }}
            .case table {{ width: 100%; border-collapse: collapse; margin-bottom: 10px; }}
            .case th, .case td {{ padding: 8px; text-align: left; border-bottom: 1px solid #eee; }}
            .case th {{ font-weight: 600; color: #495057; }}
            .case .path {{ font-family: monospace; font-size: 0.8em; color: #6c757d; word-wrap: break-word; margin-top: 10px; }}
            {rater_css}
        </style>
    </head>
    <body>
        <h1>標註分歧審閱</h1>
        <h2>共找到 {num_disagreements} 個不一致的案例</h2>
        <div class="container">
            {content}
        </div>
    </body>
    </html>
    """

    content = ""
    for _, row in disagreements_df.iterrows():
        image_path = row['image_path']
        file_uri = 'file:///' + os.path.abspath(image_path).replace('\\', '/')
        
        # 動態生成每個標註員的評分列
        rater_tds = ""
        for name in rater_names:
            rater_tds += f'<td><span class="rater_{name}">{row[f"behavior_{name}"]}</span></td>'
        
        content += f"""
        <div class="case">
            <a href="{file_uri}" target="_blank"><img src="{file_uri}" alt="圖片讀取失敗"></a>
            <table>
                <tr>
                    <th>標註員</th>
                    <th>行為標籤</th>
                </tr>
                {''.join([f'<tr><td>{name}</td>{rater_tds.split("</td>")[i]}</td></tr>' for i, name in enumerate(rater_names)])}
            </table>
            <div class="path">{image_path}</div>
        </div>
        """
        # A bit of a hack to re-create the rows properly
        row_html = ""
        for name in rater_names:
            row_html += f"""
            <tr>
                <td><strong>{name}</strong></td>
                <td><span class="rater_{name}">{row[f'behavior_{name}']}</span></td>
            </tr>
            """
        content += f"""
        <div class="case">
            <a href="{file_uri}" target="_blank" title="點擊在新分頁中打開原始圖片"><img src="{file_uri}" alt="圖片讀取失敗"></a>
            <table>
                {row_html}
            </table>
            <div class="path">{image_path}</div>
        </div>
        """

    # 填充模板
    final_html = html_template.format(
        num_disagreements=len(disagreements_df),
        content=content,
        rater_css=rater_css
    )

    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write(final_html)
    print(f"\n✅ 已生成視覺化審閱頁面: {output_filename}")
    # 嘗試自動在瀏覽器中打開
    try:
        webbrowser.open('file://' + os.path.realpath(output_filename))
    except Exception as e:
        print(f"   (無法自動打開瀏覽器，請手動打開檔案。錯誤: {e})")

# =========================================================================
# --- 主分析函數 (無修改) ---
# =========================================================================
def analyze_inter_rater_reliability(rater_files):
    print("--- 標註者間信度分析報告 ---\n")

    all_dfs = []
    rater_names = []
    for rater_id, file_path in rater_files.items():
        rater_names.append(rater_id)
        if not os.path.isfile(file_path):
            print(f"❌ 錯誤：找不到檔案 {file_path}，已跳過。")
            continue
        try:
            df = pd.read_json(file_path)
            df = df[['image_path', 'corrected_behavior', 'calibration_confidence']]
            df.rename(columns={
                'corrected_behavior': f'behavior_{rater_id}',
                'calibration_confidence': f'confidence_{rater_id}'
            }, inplace=True)
            all_dfs.append(df)
            print(f"✅ 成功讀取標註員 {rater_id} 的 {len(df)} 筆紀錄。")
        except Exception as e:
            print(f"❌ 讀取檔案 {file_path} 時發生錯誤: {e}")
            
    if len(all_dfs) < 2:
        print("\n需要至少兩份有效的標註檔案才能進行比較。程式終止。")
        return

    merged_df = all_dfs[0]
    for i in range(1, len(all_dfs)):
        merged_df = pd.merge(merged_df, all_dfs[i], on='image_path', how='inner')

    common_annotations_count = len(merged_df)
    if common_annotations_count == 0:
        print("\n❌ 兩位標註員沒有標註任何共同的圖片，無法進行分析。")
        return

    print(f"\n在所有檔案中共找到 {common_annotations_count} 筆共同的標註圖片。\n")
    print("-" * 50)

    print("\n【分析一：行為分類 (corrected_behavior) 的一致性】\n")
    
    rater1_behaviors = merged_df[f'behavior_{rater_names[0]}']
    rater2_behaviors = merged_df[f'behavior_{rater_names[1]}']

    percent_agreement = np.mean(rater1_behaviors == rater2_behaviors) * 100
    print(f"1. 簡單百分比一致性: {percent_agreement:.2f}%")
    
    kappa = cohen_kappa_score(rater1_behaviors, rater2_behaviors)
    print(f"2. Cohen's Kappa 係數: {kappa:.4f}")

    kappa_interpretation = "幾乎完全一致" if kappa > 0.80 else \
                           "高度一致" if kappa > 0.60 else \
                           "中度一致" if kappa > 0.40 else \
                           "尚可一致" if kappa > 0.20 else "微弱一致"
    print(f"   解讀: {kappa_interpretation}")
    print("-" * 50)

    print("\n【分析二：信度評分 (calibration_confidence) 的一致性】\n")
    
    long_format_df = pd.melt(merged_df, 
                             id_vars=['image_path'], 
                             value_vars=[f'confidence_{name}' for name in rater_names],
                             var_name='rater', 
                             value_name='rating')
    
    icc_results = pg.intraclass_corr(data=long_format_df, 
                                     targets='image_path', 
                                     raters='rater', 
                                     ratings='rating').set_index('Type')
    
    icc3k = icc_results.loc['ICC3k']
    print("組內相關係數 (Intraclass Correlation Coefficient, ICC):")
    print(f"   ICC 值: {icc3k['ICC']:.4f}")
    print(f"   95% 信賴區間: [{icc3k['CI95%'][0]:.4f}, {icc3k['CI95%'][1]:.4f}]")

    icc_interpretation = "極好" if icc3k['ICC'] > 0.90 else \
                         "好" if icc3k['ICC'] > 0.75 else \
                         "中等" if icc3k['ICC'] > 0.50 else "差"
    print(f"   解讀: {icc_interpretation}")
    print("-" * 50)

    print("\n【附錄：行為分類存在分歧的案例】\n")
    disagreements = merged_df[rater1_behaviors != rater2_behaviors]
    if disagreements.empty:
        print("恭喜！在所有共同標註的圖片中，行為分類完全一致！")
    else:
        print(f"共找到 {len(disagreements)} 筆不一致的標註：")
        
        # --- 【修正 SettingWithCopyWarning 的寫法】 ---
        # 創建一個副本來進行修改，避免警告
        disagreements_to_show = disagreements[['image_path'] + [f'behavior_{name}' for name in rater_names]].copy()
        disagreements_to_show['image_path'] = disagreements_to_show['image_path'].apply(lambda x: '...' + x[-50:])
        
        print(disagreements_to_show.to_string(index=False))
        
        # --- 【呼叫新函數來生成 HTML 頁面】 ---
        generate_review_page(disagreements, REVIEW_PAGE_FILENAME, rater_names)

# =========================================================================
# --- 執行分析 ---
# =========================================================================
if __name__ == "__main__":
    analyze_inter_rater_reliability(RATER_FILES)